In [ ]:
"""B1 baseline (own pipeline) -- label-free retrieval + FPNet + library similarity + GLACIER/ICEBERG + fixed fusion.
Sources embedded from experiments/b1_baseline (sha256 b1_common 9585a88d285b, b1_pipeline 2dc30c5c5954)."""
import os, sys, glob, subprocess, time, json
T0 = time.time()
os.makedirs('/kaggle/working/b1', exist_ok=True)
SOURCES = {"b1_common.py": "\"\"\"B1 shared definitions: adduct -> neutral mass, electron-mass convention, scorer identity, entropy similarity.\n\nIon m/z = M + sum(adduct atoms) -/+ electron (positive: minus m_e, negative: plus m_e). enveda-180 positive-mode m/z carry a +m_e bias\n(no electron removed) and are corrected by the caller (subtract ELECTRON before using these formulas).\n\"\"\"\nimport numpy as np\n\nELECTRON = 0.00054858\nH, C, N, O, Na, K, Cl = 1.00782503, 12.0, 14.0030740, 15.9949146, 22.9897693, 38.9637067, 34.9688527\nH2O = 2 * H + O\n# neutral mass M = prec + ADDUCT_SHIFT[adduct]\nADDUCT_SHIFT = {\n    \"[M+H]+\": -(H - ELECTRON),\n    \"[M+NH4]+\": -(N + 4 * H - ELECTRON),\n    \"[M+Na]+\": -(Na - ELECTRON),\n    \"[M+K]+\": -(K - ELECTRON),\n    \"[M-H2O+H]+\": H2O - (H - ELECTRON),\n    \"[M-2H2O+H]+\": 2 * H2O - (H - ELECTRON),\n    \"[M-H]-\": (H - ELECTRON),\n    \"[M-H2O-H]-\": H2O + (H - ELECTRON),\n    \"[M+CH2O2-H]-\": -(C + 2 * H + 2 * O) + (H - ELECTRON),\n    \"[M+Cl]-\": -(Cl + ELECTRON),\n}\nTEST_ADDUCTS = list(ADDUCT_SHIFT)\nPOSITIVE = {a for a in ADDUCT_SHIFT if a.endswith(\"+\")}\n\n\ndef neutral_mass(prec, adduct):\n    s = ADDUCT_SHIFT.get(adduct)\n    return None if s is None else float(prec) + s\n\n\n# ----------------------------------------------------------------------------------------------- spectral entropy similarity\ndef clean_peaks(mz, it, prec=None, floor=0.001, top=512, prec_margin=1.5):\n    \"\"\"relative-intensity filter, drop peaks at/above the precursor (minus margin), keep the top-N, L1-normalise.\"\"\"\n    mz = np.asarray(mz, np.float64); it = np.asarray(it, np.float64)\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\n    if prec is not None: ok &= mz < float(prec) - prec_margin\n    mz, it = mz[ok], it[ok]\n    if len(it) == 0: return mz, it\n    it = it / it.max(); keep = it >= floor; mz, it = mz[keep], it[keep]\n    if len(it) > top:\n        idx = np.argpartition(-it, top - 1)[:top]; mz, it = mz[idx], it[idx]\n    o = np.argsort(mz); mz, it = mz[o], it[o]\n    return mz, it / it.sum()\n\n\ndef entropy_weight(it):\n    \"\"\"Li et al. 2021 entropy weighting: S < 3 -> intensity^(0.25 + 0.25 S), renormalised.\"\"\"\n    p = it / it.sum(); s = float(-(p[p > 0] * np.log(p[p > 0])).sum())\n    if s < 3.0:\n        w = 0.25 + 0.25 * s; p = p ** w; p = p / p.sum()\n    return p\n\n\ndef merge_tolerance(mz_a, ia, mz_b, ib, tol_da=0.01, tol_ppm=20.0):\n    \"\"\"align two sorted peak lists: greedy nearest matching within max(tol_da, tol_ppm) -> aligned intensity vectors.\"\"\"\n    i = j = 0; a_out, b_out = [], []\n    while i < len(mz_a) and j < len(mz_b):\n        tol = max(tol_da, mz_a[i] * tol_ppm * 1e-6); d = mz_a[i] - mz_b[j]\n        if abs(d) <= tol:\n            a_out.append(ia[i]); b_out.append(ib[j]); i += 1; j += 1\n        elif d < 0:\n            a_out.append(ia[i]); b_out.append(0.0); i += 1\n        else:\n            a_out.append(0.0); b_out.append(ib[j]); j += 1\n    while i < len(mz_a): a_out.append(ia[i]); b_out.append(0.0); i += 1\n    while j < len(mz_b): a_out.append(0.0); b_out.append(ib[j]); j += 1\n    return np.asarray(a_out), np.asarray(b_out)\n\n\ndef entropy_similarity(mz_a, ia, mz_b, ib, tol_da=0.01, tol_ppm=20.0):\n    \"\"\"spectral entropy similarity in [0, 1] of two cleaned (sorted, L1-normalised) spectra.\"\"\"\n    if len(ia) == 0 or len(ib) == 0: return 0.0\n    pa, pb = entropy_weight(ia), entropy_weight(ib)\n    a, b = merge_tolerance(mz_a, pa, mz_b, pb, tol_da, tol_ppm)\n    m = (a + b) / 2.0\n    def ent(p):\n        p = p[p > 0]; return float(-(p * np.log(p)).sum())\n    return max(0.0, min(1.0, 1.0 - (2 * ent(m) - ent(a) - ent(b)) / np.log(4)))\n\n\n# ---------------------------------------------------------------------------------------------------------------------------------\n# Forward-model adduct proxies (2026-10-25). The MassSpecGym ICEBERG/GLACIER checkpoints cover positive [M+H]+ / [M+Na]+ only, so a\n# molecule without such a spectrum gets no forward evidence (panel enveda-np-examples: 34/250; 16 of them have an [M+NH4]+ spectrum,\n# 18 are negative-only). The public lineage's N1A arm A (LB-validated, +0.027 [0.010, 0.047] on its NP bench) scores [M+NH4]+ and\n# [M-H2O+H]+ spectra of such molecules as [M+H]+ proxies; proxies where a real [M+H]+ exists HURT (-0.047) and are never made.\n#   [M+NH4]+   : M = prec - 18.033823 (NH4+), drop peaks >= M + 1.007276 - 1.5 (the adduct/precursor region), precursor -> M + 1.007276\n#   [M-H2O+H]+ : M = prec + 18.010565 - 1.007276, keep peaks < prec - 1.5, precursor -> M + 1.007276\n#   [M-2H2O+H]+, [M+K]+, [M+Na]+ : never (as in the public lineage)\n# neg=True (EXPERIMENTAL, not LB-validated): negative-mode spectra mapped onto the positive axis (deprotonated fragment F-H -> F+H:\n# m/z + 2*1.007276), precursor -> M + 1.007276. Fragmentation chemistry differs between modes; use only after a paired test.\nPROTON_MASS = 1.007276; NH4_ION = 18.033823; H2O_MASS = 18.010565\nNEG_NEUTRAL = {\"[M-H]-\": lambda p: p + PROTON_MASS, \"[M-H2O-H]-\": lambda p: p + PROTON_MASS + H2O_MASS, \"[M+CH2O2-H]-\": lambda p: p - (46.005480 - PROTON_MASS),\n               \"[M+Cl]-\": lambda p: p - 34.969402}\n\n\ndef _is_positive(s):\n    md = s.get(\"mode\")\n    return md in (1, \"1\", \"positive\", \"pos\", \"+\")\n\n\ndef forward_proxies(spectra, covered=(\"[M+H]+\", \"[M+Na]+\"), neg=False):\n    \"\"\"spectra: list of dicts with mz, it, prec, adduct, mode (+ any other keys, copied). Returns the list of proxy spectra (possibly empty).\"\"\"\n    if any(_is_positive(s) and s.get(\"adduct\") in covered for s in spectra): return []\n    out = []\n    for s in spectra:\n        mz = np.asarray(s[\"mz\"], float); it = np.asarray(s[\"it\"], float); add = s.get(\"adduct\"); prec = float(s[\"prec\"])\n        if _is_positive(s) and add == \"[M+NH4]+\":\n            newp = prec - NH4_ION + PROTON_MASS; keep = mz < newp - 1.5; shift = 0.0; kind = \"nh4\"\n        elif _is_positive(s) and add == \"[M-H2O+H]+\":\n            newp = prec + H2O_MASS; keep = mz < prec - 1.5; shift = 0.0; kind = \"h2o\"\n        elif neg and (not _is_positive(s)) and add in NEG_NEUTRAL:\n            newp = NEG_NEUTRAL[add](prec) + PROTON_MASS; shift = 2 * PROTON_MASS; keep = (mz + shift) < newp - 1.5; kind = \"neg\"\n        else: continue\n        if keep.sum() < 2: continue\n        p = dict(s); p.update(mz=(mz[keep] + shift).tolist(), it=it[keep].tolist(), prec=float(newp), adduct=\"[M+H]+\", mode=\"positive\", proxy=kind, proxy_from=add)\n        out.append(p)\n    return out\n", "b1_pipeline.py": "\"\"\"B1 pipeline -- label-free MS/MS -> structure identification for a test parquet, end to end, locally or on Kaggle.\n\nStages (all label-free; see docs/b1_baseline_design.md):\n  1. test spectra per molecule (10 adducts, both polarities; electron-mass convention of b1_common)\n  2. train spectra store built from train.parquet (library search references) -- in memory on Kaggle, mmap locally\n  3. candidates: neutral-mass windows (+-10 ppm per observed adduct) over the PubChem tier + the v2 pool (train structures + COCONUT),\n     parsed once in worker processes (raw InChIKey14 + FPNet f.z over the fold-model ensemble), merged on the raw key with provenance\n  4. library similarity (spectral entropy, same polarity) for candidates that are train structures\n  5. documentation priors (popularity dataset, tier / pool rows)\n  6. shortlist = top-K by FP, top-K by z(FP)+0.15 pop, top-20 by library similarity; official scorer keys; duplicates merged\n  7. GLACIER / ICEBERG on the shortlists (budgeted; molecules with the weakest library match first)\n  8. fusion: S = z(FP) + W_GL z(GL) + W_ICE z(ICE) + LAM pop; library-priority when the best match >= TAU and beats the runner-up by MARGIN\n  9. top-25 by official identity -> submission.csv; stage statistics -> b1_run_manifest.json\nUsage (local): python -X utf8 b1_pipeline.py --test data/raw/test.parquet --out <dir> [--limit N] [--no_forward]\nOn Kaggle the paths are resolved under /kaggle/input/** (same datasets as the public E0 kernel) and /kaggle/working.\n\"\"\"\nimport os as _os\nfor _v in (\"OMP_NUM_THREADS\", \"OPENBLAS_NUM_THREADS\", \"MKL_NUM_THREADS\", \"NUMBA_NUM_THREADS\"): _os.environ.setdefault(_v, \"1\")   # one BLAS thread per process: 16 workers x 64 OpenBLAS threads made the Spark 30x slower\nimport argparse, glob, json, os, subprocess, sys, time\nfrom collections import Counter\nfrom multiprocessing import Pool\nimport numpy as np, pandas as pd, polars as pl\n\nHERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)\nfrom b1_common import ADDUCT_SHIFT, ELECTRON, clean_peaks, entropy_similarity, forward_proxies  # noqa: E402\n\nKAGGLE = os.path.exists(\"/kaggle/input\")\nPPM, K_SHORT, K_LIB, LIB_MIN, MAX_REF, MAX_Q = 10.0, 100, 20, 0.2, 24, 8      # MAX_Q 8 with adduct round-robin = the evaluation's query policy\nSHORTLIST_LAM = 0.15   # prior weight for the shortlist (as evaluated); LAM below is the fusion weight\nMAX_HEAVY_FORWARD = 100   # structures above this are not sent to the forward models\nDUP_SIM = 0.98            # a reference at the same precursor this similar to a query spectrum is treated as the same acquisition (re-deposit) and not used\nW_GL, W_ICE, LAM, TAU, MARGIN = 2.0, 1.0, 0.5, 2.0, 0.2   # LAM 0.5 (B1 eval: better than 0.15 on A, B and the C1 set); TAU 2.0 disables the library-priority rule until the deduplicated C1 evaluation supports it\nICE_BUDGET, GL_BUDGET = 5400, 4000\n\n\ndef find(pattern, root=\"/kaggle/input\"):\n    hits = sorted(glob.glob(os.path.join(root, \"**\", pattern), recursive=True))\n    if not hits: raise FileNotFoundError(pattern)\n    return hits[0]\n\n\ndef paths(args):\n    if KAGGLE:\n        def first(*pats):\n            for pat in pats:\n                try: return find(pat)\n                except FileNotFoundError: pass\n            raise FileNotFoundError(pats)\n        pop = os.path.dirname(first(\"pc_lsid.npy\"))\n        tier = os.path.dirname(first(\"casmi26-pubchem-tier/pc_mass.npy\", \"pubchem_tier/pc_mass.npy\", \"pc_mass.npy\"))\n        return dict(train=\"/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra/train.parquet\", tier=tier, pop=pop,\n                    pool=first(\"casmi26-v2-pool/pool_meta.parquet\", \"pool_meta.parquet\"), fold_safe=os.path.dirname(first(\"fpnet_fold0.pt\")),\n                    v3=os.path.dirname(os.path.dirname(first(\"casmi26-v3-models/code/casmi/fpnet.py\", \"v3_models/code/casmi/fpnet.py\"))),\n                    glacier=os.path.dirname(first(\"gl_runner.py\")), iceberg=os.path.dirname(first(\"ice_runner.py\")), work=\"/kaggle/working\", store=None, runner_python=None)\n    R = os.environ.get(\"CASMI_ROOT\", r\"C:\\kaggle\\CASMI\" if os.name == \"nt\" else os.path.expanduser(\"~/CASMI2026\"))\n    runner_py = f\"{R}/.venv-ice/Scripts/python.exe\" if os.name == \"nt\" else f\"{R}/.venv-ice/bin/python\"\n    return dict(train=f\"{R}/data/raw/train.parquet\", tier=f\"{R}/data/external/pubchem_tier\", pop=f\"{R}/data/external/pubchem_popularity\", pool=f\"{R}/data/external/v2_pool/pool_meta.parquet\",\n                fold_safe=f\"{R}/data/external/fold_safe_fpnet\", v3=f\"{R}/data/external/v3_models/code\", glacier=f\"{R}/data/external/glacier\", iceberg=f\"{R}/data/external/iceberg\",\n                work=args.out, store=f\"{R}/data/processed/spectra_store\", runner_python=runner_py)\n\n\n# ------------------------------------------------------------------------------------------------------------ workers\ndef _init(v3, fs):\n    global chem_w, bits_w, Chem, TAUT\n    sys.path.insert(0, v3)\n    from casmi import chem as chem_w\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog(\"rdApp.*\"); TAUT = rdMolStandardize.TautomerEnumerator(); bits_w = np.load(f\"{fs}/fp_bits.npy\")\n\n\ndef _feat(args):\n    smiles, zlog = args\n    keys = [None] * len(smiles); sc = np.full(len(smiles), np.nan, np.float32)\n    for i, s in enumerate(smiles):\n        m = Chem.MolFromSmiles(s)\n        if m is None: continue\n        try: keys[i] = Chem.MolToInchiKey(m)[:14]\n        except Exception: continue\n        f = chem_w.raw_fingerprint(s)\n        if f is not None: sc[i] = float(f[bits_w].astype(np.float32) @ zlog)\n    return keys, sc\n\n\ndef _rawkey(smiles):\n    out = []\n    for s in smiles:\n        try:\n            m = Chem.MolFromSmiles(s); out.append(Chem.MolToInchiKey(m)[:14] if m is not None else None)\n        except Exception:\n            out.append(None)\n    return out\n\n\ndef _okey(smiles):\n    out = []\n    for s in smiles:\n        try:\n            m = Chem.MolFromSmiles(s); Chem.RemoveStereochemistry(m); out.append(Chem.MolToInchiKey(TAUT.Canonicalize(m))[:14])\n        except Exception:\n            out.append(None)\n    return out\n\n\n# ------------------------------------------------------------------------------------------------------------ forward-model site (Kaggle)\ndef _compatible_rdkit_wheel(pkg):\n    \"\"\"RDKit 2025.3.6 wheel compatible with this interpreter (bundled in the ICEBERG package or attached as a dataset).\"\"\"\n    from pip._vendor.packaging.tags import sys_tags\n    from pip._vendor.packaging.utils import parse_wheel_filename\n    compatible = set(sys_tags())\n    for path in sorted(glob.glob(os.path.join(pkg, \"wheels\", \"rdkit-2025.3.6-*.whl*\"))) + sorted(glob.glob(\"/kaggle/input/**/rdkit-2025.3.6-*.whl\", recursive=True)):\n        name = os.path.basename(path).removesuffix(\".ice\")\n        _, version, _, tags = parse_wheel_filename(name)\n        if str(version) == \"2025.3.6\" and compatible.intersection(tags): return path\n    raise FileNotFoundError(\"no compatible RDKit 2025.3.6 wheel for the ICEBERG/GLACIER site\")\n\n\ndef preseed_site(pkg, site):\n    \"\"\"install the ICEBERG wheels (+ a compatible RDKit) into `site` with pip --target and write the runner's marker (same procedure as E0).\"\"\"\n    import shutil\n    srcs = sorted(glob.glob(os.path.join(pkg, \"wheels\", \"*.whl.ice\")) + glob.glob(os.path.join(pkg, \"wheels\", \"*.whl\")))\n    keep = [w for w in srcs if not os.path.basename(w).lower().startswith(\"rdkit\")] + [_compatible_rdkit_wheel(pkg)]\n    os.makedirs(site, exist_ok=True); wdir = os.path.join(site, \".wheels\"); os.makedirs(wdir, exist_ok=True); whl = []\n    for w in keep:\n        dst = os.path.join(wdir, os.path.basename(w)[:-4] if w.endswith(\".ice\") else os.path.basename(w)); shutil.copyfile(w, dst); whl.append(dst)\n    r = subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"--no-index\", \"--no-deps\", \"--no-compile\", \"--disable-pip-version-check\", \"--upgrade\", \"--target\", site] + whl,\n                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n    print(\"[b1] site preseed rc\", r.returncode, (r.stdout or \"\")[-300:], flush=True)\n    if r.returncode != 0: raise RuntimeError(\"site preseed failed: \" + (r.stdout or \"\")[-500:])\n    shutil.rmtree(wdir, ignore_errors=True)\n    open(os.path.join(site, \".ice_site_ok\"), \"w\").write(chr(10).join(os.path.basename(w) for w in srcs))\n\n\ndef heavy_ok(smiles):\n    try:\n        from rdkit import Chem\n        m = Chem.MolFromSmiles(smiles); return m is not None and m.GetNumHeavyAtoms() <= MAX_HEAVY_FORWARD\n    except Exception:\n        return False\n\n\n# ------------------------------------------------------------------------------------------------------------ data\ndef load_test(path, limit=0):\n    t = pl.read_parquet(path)\n    cols = [\"molecule_id\", \"ms2_mzs\", \"ms2_normalized_intensities\", \"precursor_mz\", \"adduct\", \"ionization_mode\", \"instrument_type\", \"collision_energy_ev\", \"collision_energy_orig\", \"collision_energy_orig_units\"]\n    t = t.select([c for c in cols if c in t.columns])\n    mols = {}\n    for r in t.iter_rows(named=True):\n        ce = r.get(\"collision_energy_ev\"); ce = [float(x) for x in ce] if ce is not None and len(ce) else None\n        mols.setdefault(str(r[\"molecule_id\"]), []).append(dict(mz=np.asarray(r[\"ms2_mzs\"], np.float64), it=np.asarray(r[\"ms2_normalized_intensities\"], np.float64), prec=float(r[\"precursor_mz\"]),\n                                                                 adduct=r[\"adduct\"], mode=r[\"ionization_mode\"], instrument=r.get(\"instrument_type\") or \"\", ce=float(np.mean(ce)) if ce else None,\n                                                                 ce_ev=ce, ce_orig=r.get(\"collision_energy_orig\"), ce_units=r.get(\"collision_energy_orig_units\")))\n    ids = list(mols)\n    if limit: ids = ids[:limit]\n    return {m: mols[m] for m in ids}, t[\"molecule_id\"].unique().to_list()\n\n\ndef build_store(train_parquet, scorer_keys_parquet=None):\n    \"\"\"flat spectra arrays + meta from train.parquet (in memory); returns dict.\"\"\"\n    import pyarrow.parquet as pq\n    pf = pq.ParquetFile(train_parquet); mz_parts, it_parts, lens, metas = [], [], [], []\n    for b in pf.iter_batches(batch_size=100_000, columns=[\"inchikey14\", \"ingest_lib\", \"adduct\", \"ionization_mode\", \"precursor_mz\", \"collision_energy_ev\", \"ms2_mzs\", \"ms2_normalized_intensities\"]):\n        mz = b.column(\"ms2_mzs\"); it = b.column(\"ms2_normalized_intensities\")\n        for cm, ci in zip(mz.chunks if hasattr(mz, \"chunks\") else [mz], it.chunks if hasattr(it, \"chunks\") else [it]):\n            lens.append(np.diff(np.asarray(cm.offsets)).astype(np.int64)); mz_parts.append(np.asarray(cm.flatten(), np.float32)); it_parts.append(np.asarray(ci.flatten(), np.float32))\n        tb = pl.from_arrow(b.select([\"inchikey14\", \"ingest_lib\", \"adduct\", \"ionization_mode\", \"precursor_mz\", \"collision_energy_ev\"]))\n        metas.append(tb.with_columns(pl.col(\"collision_energy_ev\").list.mean().alias(\"ce_mean\")).drop(\"collision_energy_ev\"))\n    lens = np.concatenate(lens); off = np.zeros(len(lens) + 1, np.int64); np.cumsum(lens, out=off[1:])\n    meta = pl.concat(metas).with_row_index(\"row\")\n    return dict(mz=np.concatenate(mz_parts), it=np.concatenate(it_parts), off=off, meta=meta)\n\n\ndef load_store(store_dir):\n    return dict(mz=np.load(f\"{store_dir}/spec_mz.npy\", mmap_mode=\"r\"), it=np.load(f\"{store_dir}/spec_int.npy\", mmap_mode=\"r\"), off=np.load(f\"{store_dir}/spec_off.npy\"),\n                meta=pl.read_parquet(f\"{store_dir}/spec_meta.parquet\"))\n\n\ndef official_keys_of_train(store, pool):\n    \"\"\"map train structures (inchikey14) -> official scorer key through the pool (pool keys are official; src 1/3 rows carry train_sid) --\n    falls back to canonicalising structures absent from the pool.\"\"\"\n    return None\n\n\n# ------------------------------------------------------------------------------------------------------------ main\ndef main():\n    ap = argparse.ArgumentParser(); ap.add_argument(\"--test\", default=None); ap.add_argument(\"--out\", default=\"experiments/b1_baseline/pipeline_local\"); ap.add_argument(\"--limit\", type=int, default=0)\n    ap.add_argument(\"--no_forward\", action=\"store_true\"); ap.add_argument(\"--workers\", type=int, default=0); ap.add_argument(\"--folds\", default=\"0\")\n    ap.add_argument(\"--k_short\", type=int, default=K_SHORT, help=\"FP branch size of the shortlist (default 100)\")\n    ap.add_argument(\"--k_fpp\", type=int, default=None, help=\"FP+prior branch size (default = --k_short)\")\n    ap.add_argument(\"--k_pop\", type=int, default=0, help=\"prior-only branch size added to the shortlist (default 0 = off)\")\n    ap.add_argument(\"--w_ice\", type=float, default=W_ICE, help=\"ICE weight in the fixed rule; 0 skips the ICEBERG runner entirely\")\n    ap.add_argument(\"--tau\", type=float, default=TAU, help=\"library-priority threshold of the fixed rule (default 2.0 = off)\")\n    ap.add_argument(\"--margin\", type=float, default=MARGIN, help=\"library-priority margin over the runner-up (default 0.2)\")\n    ap.add_argument(\"--rank_model\", default=None, help=\"LightGBM LambdaRank model file (default: lambdarank.txt next to this script; absent -> fixed rule)\")\n    ap.add_argument(\"--exclude_rows\", default=None, help=\"evaluation: JSON list of spectra-store rows never used as references (the query acquisitions) -> class-1 proxy\")\n    ap.add_argument(\"--exclude_identities\", default=None, help=\"evaluation: JSON list of official keys whose references are removed and whose train membership is hidden -> class-2 proxy\")\n    ap.add_argument(\"--solution\", default=None, help=\"evaluation: CSV with molecule_id,smiles; scores the submission with the official scorer replica\")\n    ap.add_argument(\"--mask_json\", default=None, help=\"evaluation: per-molecule masks {molecule_id: {ids: [...], rows: [...], same_lib: {identity, lib}}} (same conventions as b1_features)\")\n    ap.add_argument(\"--gl_budget\", type=int, default=GL_BUDGET); ap.add_argument(\"--ice_budget\", type=int, default=ICE_BUDGET)\n    ap.add_argument(\"--deadline_s\", type=int, default=0, help=\"wall-clock deadline since start; runner budgets are capped at deadline - elapsed - 900 s (0 = off)\")\n    ap.add_argument(\"--min_gl_cover\", type=float, default=0.5, help=\"learned fusion only when GL scored at least this fraction of a molecule's candidates (else the fixed rule)\")\n    ap.add_argument(\"--main\", default=\"learned\", choices=[\"learned\", \"blend21\", \"blend11\"], help=\"main list: learned model, or the blend hedge w*z(learned)+z(fixed rule) with w = 2 or 1 (GL-cover fallback unchanged)\")\n    ap.add_argument(\"--adduct_proxy\", action=\"store_true\", help=\"forward-model proxies for molecules without a covered positive spectrum: [M+NH4]+ / [M-H2O+H]+ scored as [M+H]+ (public N1A arm A)\")\n    ap.add_argument(\"--neg_proxy\", action=\"store_true\", help=\"EXPERIMENTAL: also map negative-mode spectra of such molecules onto the positive axis (b1_common.forward_proxies)\")\n    args = ap.parse_args(); P = paths(args); os.makedirs(P[\"work\"], exist_ok=True); T0 = time.time(); log = lambda *x: print(f\"[b1 {time.time()-T0:7.0f}s]\", *x, flush=True)\n    test_path = args.test or (\"/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra/test.parquet\" if KAGGLE else os.path.join(os.path.dirname(P[\"train\"]), \"test.parquet\"))\n    workers = args.workers or max(2, (os.cpu_count() or 4) - 1)\n    import torch\n    sys.path.insert(0, P[\"v3\"])\n    from casmi import fpnet, chem\n    from casmi.spectra import merge_spectra\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog(\"rdApp.*\"); TAUT = rdMolStandardize.TautomerEnumerator()\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    mols, all_ids = load_test(test_path, args.limit); log(f\"test: {len(mols)} molecules of {len(all_ids)}; {sum(len(v) for v in mols.values())} spectra\")\n    # ---- stores\n    store = load_store(P[\"store\"]) if P[\"store\"] and os.path.exists(P[\"store\"]) else build_store(P[\"train\"])\n    meta = store[\"meta\"]; S_MZ, S_IT, S_OFF = store[\"mz\"], store[\"it\"], store[\"off\"]\n    lib_of_row = meta[\"ingest_lib\"].to_numpy(); add_of_row = meta[\"adduct\"].to_numpy(); mode_of_row = meta[\"ionization_mode\"].to_numpy(); prec_of_row = meta[\"precursor_mz\"].to_numpy()\n    ik_rows = {k: np.asarray(r, np.int64) for k, r in meta.group_by(\"inchikey14\").agg(pl.col(\"row\")).iter_rows()}\n    log(f\"store: {meta.height:,} spectra, {len(ik_rows):,} train structures\")\n    pool = pl.read_parquet(P[\"pool\"]).select(\"key\", \"smiles\", \"mass\", \"src\", \"pid\")\n    tmass = np.load(f\"{P['tier']}/pc_mass.npy\", mmap_mode=\"r\"); toff = np.load(f\"{P['tier']}/pc_off.npy\", mmap_mode=\"r\"); tbuf = np.load(f\"{P['tier']}/pc_smiles.npy\", mmap_mode=\"r\")\n    PC = {n: np.load(f\"{P['pop']}/pc_{n}.npy\", mmap_mode=\"r\") for n in (\"lsid\", \"lpmid\", \"lsynonyms\", \"np\", \"src_mask\", \"lotus_refs\")}\n    PL = {n: np.load(f\"{P['pop']}/pool_{n}.npy\") for n in (\"lsid\", \"lpmid\", \"lsynonyms\", \"np\", \"src_mask\", \"lotus_refs\")}\n    assert all(v.shape[0] == tmass.shape[0] for v in PC.values()), 'popularity arrays are not aligned to the tier'\n    assert all(v.shape[0] == pool.height for v in PL.values()), 'pool popularity arrays are not aligned to the pool'\n    log(f\"paths: {json.dumps({k: str(v) for k, v in P.items()})}\")\n    CMASS = CSMI = None\n    try:\n        import pickle\n        cdir = os.path.dirname(find(\"coco_mass.npy\")) if KAGGLE else os.path.join(os.path.dirname(os.path.dirname(P[\"pool\"])), \"coconut\")\n        if os.path.exists(f\"{cdir}/coco_mass.npy\"):\n            CMASS = np.load(f\"{cdir}/coco_mass.npy\"); CSMI = np.asarray(pickle.load(open(f\"{cdir}/coco_meta.pkl\", \"rb\"))[\"smiles\"]); log(f\"COCONUT candidate store: {len(CMASS):,} structures\")\n    except Exception as e:\n        CMASS = CSMI = None; log(\"COCONUT store not used:\", repr(e))\n    order = np.argsort(pool[\"mass\"].to_numpy(), kind=\"stable\")\n    pmass = pool[\"mass\"].to_numpy()[order]; psmi = pool[\"smiles\"].to_numpy()[order]; psrc = pool[\"src\"].to_numpy()[order]; ppid = pool[\"pid\"].to_numpy()[order]; pkey = pool[\"key\"].to_numpy()[order]\n    folds = [int(x) for x in args.folds.split(\",\")]\n    models = [fpnet.load_model(f\"{P['fold_safe']}/fpnet_fold{k}.pt\", dev) for k in folds]; dtype = fpnet.autocast_dtype(dev)\n    log(f\"FPNet folds {folds} loaded on {dev}; tier rows {tmass.shape[0]:,}; pool {pool.height:,}\")\n\n    def spectrum(row, shift=0.0):\n        s, e = int(S_OFF[row]), int(S_OFF[row + 1]); return np.asarray(S_MZ[s:e], np.float64) - shift, np.asarray(S_IT[s:e], np.float64)\n\n    def tier_window(lo_m, hi_m):\n        lo = int(np.searchsorted(tmass, lo_m, \"left\")); hi = int(np.searchsorted(tmass, hi_m, \"right\"))\n        if hi <= lo: return [], np.zeros(0, np.int64), np.zeros(0)\n        o = np.asarray(toff[lo:hi + 1]).astype(np.int64); raw = bytes(tbuf[o[0]:o[-1]]); base = o[0]\n        return [raw[o[i] - base:o[i + 1] - base].decode(\"ascii\") for i in range(hi - lo)], np.arange(lo, hi), np.asarray(tmass[lo:hi], np.float64)\n\n    stats, feats, runner, ref_cache, molinfo = [], {}, [], {}, {}\n    with Pool(workers, initializer=_init, initargs=(P[\"v3\"], P[\"fold_safe\"])) as wp, torch.no_grad():\n        # (official key -> reference rows) for train structures: scorer keys from the store meta when available (local), else raw\n        # InChIKey14 of the pool's train SMILES computed in the workers and matched to the store's inchikey14\n        if \"scorer_ik14\" in meta.columns:\n            key_rows = {k: np.asarray(r, np.int64) for k, r in meta.filter(pl.col(\"scorer_ik14\").is_not_null()).group_by(\"scorer_ik14\").agg(pl.col(\"row\")).iter_rows()}\n        else:\n            tr = pool.filter(pl.col(\"src\").is_in([1, 3])); tsmi = tr[\"smiles\"].to_list(); tkey = tr[\"key\"].to_list()\n            rk = [k for rr_ in wp.map(_rawkey, [tsmi[i:i + 2000] for i in range(0, len(tsmi), 2000)]) for k in rr_]\n            ik_to_key = {ik: key for ik, key in zip(rk, tkey) if ik}\n            key_rows = {}\n            for ik, rows in ik_rows.items():\n                k = ik_to_key.get(ik)\n                if k: key_rows.setdefault(k, []).append(rows)\n            key_rows = {k: np.concatenate(v) for k, v in key_rows.items()}\n        log(f\"library index: {len(key_rows):,} official identities with reference spectra\")\n        EXCL_ROWS = set(json.load(open(args.exclude_rows))) if args.exclude_rows else set()\n        EXCL_IDS = set(json.load(open(args.exclude_identities))) if args.exclude_identities else set()\n        MASK = json.load(open(args.mask_json)) if args.mask_json else {}\n        if EXCL_ROWS or EXCL_IDS or MASK: log(f\"EVALUATION MODE: global masks {len(EXCL_ROWS)} rows / {len(EXCL_IDS)} identities; per-molecule masks for {len(MASK)} molecules\")\n        for n_i, (mid, spectra) in enumerate(mols.items()):\n            tm0 = time.time()\n            known = [s for s in spectra if s[\"adduct\"] in ADDUCT_SHIFT]\n            by_add = {}\n            for s in sorted(known, key=lambda s: (s[\"adduct\"] != \"[M+H]+\", s[\"adduct\"], s[\"ce\"] if s[\"ce\"] is not None else 0.0)):\n                by_add.setdefault(s[\"adduct\"], []).append(s)\n            q = []\n            while len(q) < MAX_Q and any(by_add.values()):          # adduct round-robin (same policy as b1_features.py)\n                for ad in list(by_add):\n                    if by_add[ad] and len(q) < MAX_Q: q.append(by_add[ad].pop(0))\n            q = [dict(s, clean=clean_peaks(s[\"mz\"], s[\"it\"], s[\"prec\"])) for s in q]\n            if not q:                                         # unknown adducts only: fall back to [M+H]+ / [M-H]- by polarity\n                q = [dict(s, adduct=(\"[M+H]+\" if s[\"mode\"] == \"positive\" else \"[M-H]-\"), clean=clean_peaks(s[\"mz\"], s[\"it\"], s[\"prec\"])) for s in spectra][:MAX_Q]\n            M_by_add = {}\n            for s in q: M_by_add.setdefault(s[\"adduct\"], []).append(s[\"prec\"] + ADDUCT_SHIFT[s[\"adduct\"]])\n            M_by_add = {k: float(np.median(v)) for k, v in M_by_add.items()}\n            src = {}   # smiles -> provenance (de-duplicated across the adduct windows before fingerprinting)\n            for M in M_by_add.values():\n                tol = M * PPM * 1e-6; ts, trows, tms = tier_window(M - tol, M + tol)\n                for s_, tr_, m_ in zip(ts, trows, tms):\n                    d = src.setdefault(s_, dict(train=False, coco=False, trow=-1, pid=-1, pkey=None, mass=float(m_)))\n                    if d[\"trow\"] < 0: d[\"trow\"] = int(tr_)\n                lo = int(np.searchsorted(pmass, M - tol, \"left\")); hi = int(np.searchsorted(pmass, M + tol, \"right\"))\n                for s_, sr_, pi_, pk_, m_ in zip(psmi[lo:hi], psrc[lo:hi], ppid[lo:hi], pkey[lo:hi], pmass[lo:hi]):\n                    d = src.setdefault(str(s_), dict(train=False, coco=False, trow=-1, pid=-1, pkey=None, mass=float(m_)))\n                    d[\"train\"] |= int(sr_) in (1, 3); d[\"coco\"] |= int(sr_) in (2, 3)\n                    if d[\"pid\"] < 0: d[\"pid\"] = int(pi_); d[\"pkey\"] = str(pk_)\n                if CMASS is not None:\n                    clo = int(np.searchsorted(CMASS, M - tol, \"left\")); chi = int(np.searchsorted(CMASS, M + tol, \"right\"))\n                    for s_, m_ in zip(CSMI[clo:chi], CMASS[clo:chi]):\n                        d = src.setdefault(str(s_), dict(train=False, coco=False, trow=-1, pid=-1, pkey=None, mass=float(m_))); d[\"coco\"] = True\n            smiles = list(src)\n            src_train = [src[s_][\"train\"] for s_ in smiles]; src_coco = [src[s_][\"coco\"] for s_ in smiles]; tier_row = [src[s_][\"trow\"] for s_ in smiles]\n            pool_pid = [src[s_][\"pid\"] for s_ in smiles]; pool_key = [src[s_][\"pkey\"] for s_ in smiles]; cmass = [src[s_][\"mass\"] for s_ in smiles]\n            # FPNet logits (polarity groups, fold ensemble)\n            zl, wsum = None, 0.0\n            for mode in (\"positive\", \"negative\"):\n                grp = [s for s in q if s[\"mode\"] == mode]\n                if not grp: continue\n                pairs = [(s[\"mz\"], s[\"it\"]) for s in grp]; mz, it = pairs[0] if len(pairs) == 1 else merge_spectra(pairs)\n                prec = float(np.median([s[\"prec\"] for s in grp])); ces = [s[\"ce\"] for s in grp if s[\"ce\"] is not None]\n                ad = Counter(s[\"adduct\"] for s in grp).most_common(1)[0][0]; pm, pi = fpnet.prep_peaks(mz, it, prec)\n                inp = fpnet.collate([dict(mz=pm, it=pi, prec=prec, adduct_ix=chem.adduct_index(ad), ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\n                                          n_merged=min(len(grp), 8), mode=1.0 if mode == \"positive\" else -1.0)], dev)\n                with torch.autocast(dev, dtype=dtype, enabled=(dev == \"cuda\")):\n                    zz = torch.stack([m(*inp)[0].float()[0] for m in models]).mean(0).cpu().numpy().astype(np.float32)\n                zl = zz * len(grp) if zl is None else zl + zz * len(grp); wsum += len(grp)\n            zlog = (zl / wsum).astype(np.float32)\n            chunks = [smiles[i:i + 400] for i in range(0, len(smiles), 400)]\n            res = wp.map(_feat, [(c, zlog) for c in chunks]) if chunks else []\n            raw_keys = [k for rr_ in res for k in rr_[0]]; fp = np.concatenate([rr_[1] for rr_ in res]) if res else np.zeros(0, np.float32)\n            rep = {}\n            for i, k in enumerate(raw_keys):\n                if k is None: continue\n                if k not in rep: rep[k] = dict(i=i, train=src_train[i], coco=src_coco[i], trow=tier_row[i], pid=pool_pid[i], pkey=pool_key[i], mass=cmass[i], fp=float(fp[i]) if np.isfinite(fp[i]) else -np.inf)\n                else:\n                    p = rep[k]; p[\"train\"] |= src_train[i]; p[\"coco\"] |= src_coco[i]\n                    if p[\"trow\"] < 0: p[\"trow\"] = tier_row[i]\n                    if p[\"pid\"] < 0: p[\"pid\"] = pool_pid[i]; p[\"pkey\"] = pool_key[i]\n                    if np.isfinite(fp[i]) and fp[i] > p[\"fp\"]: p[\"fp\"] = float(fp[i])\n            mk = MASK.get(str(mid), {}); excl_ids = EXCL_IDS | set(mk.get(\"ids\", [])); excl_rows = EXCL_ROWS | set(int(x) for x in mk.get(\"rows\", [])); same_lib = mk.get(\"same_lib\")\n            if excl_ids:\n                for k in list(rep):\n                    if rep[k][\"pkey\"] in excl_ids:\n                        rep[k][\"train\"] = False\n                        if rep[k][\"trow\"] < 0 and not rep[k][\"coco\"]: rep[k][\"excluded\"] = True   # library-only structure of a hidden identity: stays in the window statistics (as the evaluator) but never becomes a candidate\n            keys = list(rep); n_u = len(keys)\n            if n_u == 0:\n                stats.append(dict(molecule=mid, n_query=len(q), n_unique=0, seconds=time.time() - tm0)); feats[mid] = []; continue\n            fpu = np.array([rep[k][\"fp\"] for k in keys]); fpu[~np.isfinite(fpu)] = np.nan\n            trow = np.array([rep[k][\"trow\"] for k in keys]); pid = np.array([rep[k][\"pid\"] for k in keys]); in_train = np.array([rep[k][\"train\"] for k in keys]); in_coco = np.array([rep[k][\"coco\"] for k in keys])\n            cm_ = np.array([rep[k][\"mass\"] for k in keys], float)\n            comp = {n: np.zeros(n_u, np.float32) for n in PC}; ht = trow >= 0; hp = (~ht) & (pid >= 0)\n            for n in comp:\n                if ht.any(): comp[n][ht] = np.asarray(PC[n][np.sort(trow[ht])], dtype=np.float32)[np.argsort(np.argsort(trow[ht]))]\n                if hp.any(): comp[n][hp] = np.asarray(PL[n][pid[hp]], dtype=np.float32)\n            comp[\"np\"][in_coco] = np.maximum(comp[\"np\"][in_coco], 1.0)\n            pop = comp[\"lsid\"] + comp[\"lpmid\"] + 0.5 * comp[\"lsynonyms\"] + 1.0 * (comp[\"np\"] != 0) + 0.5 * ((comp[\"src_mask\"].astype(np.int64) & 1) != 0) + 0.5 * np.log1p(np.maximum(comp[\"lotus_refs\"], 0))\n            # library similarity\n            lib = np.zeros(n_u); lib_same = np.zeros(n_u); nref = np.zeros(n_u, int)\n            for ci, k in enumerate(keys):\n                pk = rep[k][\"pkey\"]\n                if not rep[k][\"train\"] or pk is None or pk not in key_rows or pk in excl_ids: continue\n                rrows = [int(x) for x in key_rows[pk] if x not in excl_rows]\n                if same_lib and pk == same_lib.get(\"identity\"): rrows = [x for x in rrows if lib_of_row[x] != same_lib.get(\"lib\")]   # class-1 view: references from other libraries only (as b1_features)\n                kept = []\n                for x in rrows:                                       # duplicate acquisitions of a query spectrum are not independent references\n                    if x not in ref_cache:\n                        sh = ELECTRON if (lib_of_row[x] == \"enveda-180\" and mode_of_row[x] == \"positive\") else 0.0\n                        mz, it = spectrum(x, sh); ref_cache[x] = (clean_peaks(mz, it, float(prec_of_row[x]) - sh), float(prec_of_row[x]) - sh)\n                    (rmz, rit), rprec = ref_cache[x]\n                    dup = False\n                    for s in q:\n                        if abs(s[\"prec\"] - rprec) < 0.005 and s[\"mode\"] == mode_of_row[x]:\n                            qmz, qit = s[\"clean\"]\n                            if (s[\"adduct\"] == add_of_row[x] and len(qmz) == len(rmz) and len(qmz) > 0 and np.allclose(qmz, rmz, atol=2e-4) and np.allclose(qit, rit, atol=1e-3)) or \\\n                               entropy_similarity(qmz, qit, rmz, rit) >= DUP_SIM: dup = True; break\n                    if not dup: kept.append(x)\n                if len(kept) > MAX_REF:\n                    pri = np.array([0 if add_of_row[x] in M_by_add else 1 for x in kept]); kept = [kept[i] for i in np.argsort(pri, kind=\"stable\")[:MAX_REF]]\n                best = same = 0.0\n                for x in kept:\n                    (rmz, rit), rprec = ref_cache[x]\n                    for s in q:\n                        if s[\"mode\"] != mode_of_row[x]: continue\n                        sim = entropy_similarity(s[\"clean\"][0], s[\"clean\"][1], rmz, rit); best = max(best, sim)\n                        if add_of_row[x] == s[\"adduct\"]: same = max(same, sim)\n                lib[ci], lib_same[ci], nref[ci] = best, same, len(kept)\n            if len(ref_cache) > 200_000: ref_cache.clear()\n            # shortlist + official keys\n            fpz = np.nan_to_num(fpu, nan=np.nanmin(fpu) if np.isfinite(fpu).any() else 0.0); zf = (fpz - fpz.mean()) / (fpz.std() + 1e-9)\n            K = min(args.k_short, n_u); K2 = min(args.k_fpp if args.k_fpp is not None else args.k_short, n_u)\n            top_fp = np.argsort(-np.nan_to_num(fpu, nan=-1e18), kind=\"stable\")[:K]; top_fpp = np.argsort(-(zf + SHORTLIST_LAM * pop), kind=\"stable\")[:K2]\n            top_lib = [i for i in np.argsort(-lib, kind=\"stable\")[:K_LIB] if lib[i] > LIB_MIN]\n            short = np.union1d(np.union1d(top_fp, top_fpp), np.array(top_lib, dtype=np.int64))\n            if args.k_pop > 0: short = np.union1d(short, np.argsort(-(pop + 1e-6 * zf), kind=\"stable\")[:min(args.k_pop, n_u)])   # prior-only branch (Codex 008 Q2)\n            short = np.array([i for i in short if not rep[keys[i]].get(\"excluded\")], dtype=np.int64)\n            s_smiles = [smiles[rep[keys[i]][\"i\"]] for i in short]\n            okeys = [k for rr_ in wp.map(_okey, [s_smiles[i:i + 25] for i in range(0, len(s_smiles), 25)]) for k in rr_]\n            best = {}\n            for j, i in enumerate(short):\n                if okeys[j] is not None: best.setdefault(okeys[j], []).append(i)\n            ppm_err = np.min(np.abs(np.array([[(M - m_) / m_ * 1e6 for m_ in cm_] for M in M_by_add.values()])), axis=0) if M_by_add else np.full(n_u, np.nan)\n            fp_sorted = np.sort(np.nan_to_num(fpu, nan=-1e18))\n            rows = []\n            for ok, idxs in best.items():\n                i = max(idxs, key=lambda j: np.nan_to_num(fpu[j], nan=-1e18))\n                rows.append(dict(key=ok, smiles=smiles[rep[keys[i]][\"i\"]], fp=float(fpu[i]) if np.isfinite(fpu[i]) else None, pop=float(pop[i]), lib=float(max(lib[j] for j in idxs)),\n                                 lib_same=float(max(lib_same[j] for j in idxs)), lib_nref=int(max(nref[j] for j in idxs)), in_train=bool(any(in_train[j] for j in idxs)),\n                                 in_tier=bool(any(trow[j] >= 0 for j in idxs)), in_coconut=bool(any(in_coco[j] for j in idxs)), ppm_err=float(np.nanmin([ppm_err[j] for j in idxs])),\n                                 fp_rank_window=int(len(fp_sorted) - np.searchsorted(fp_sorted, np.nan_to_num(fpu[i], nan=-1e18), side=\"left\")), **{n: float(comp[n][i]) for n in comp}))\n            feats[mid] = rows\n            molinfo[mid] = dict(n_query=len(q), ce_known=float(np.mean([s[\"ce\"] is not None for s in q])), n_unique=float(n_u))\n            rspec = [dict(mz=s[\"mz\"].tolist(), it=s[\"it\"].tolist(), prec=s[\"prec\"], adduct=s[\"adduct\"], mode=s[\"mode\"], instrument=s[\"instrument\"],\n                          ce_ev=s.get(\"ce_ev\"), ce_orig=s.get(\"ce_orig\"), ce_units=s.get(\"ce_units\")) for s in q]\n            prox = forward_proxies(rspec, neg=args.neg_proxy) if args.adduct_proxy else []; rspec += prox   # proxies only when no covered positive spectrum\n            runner.append(dict(mid=mid, lib_max=float(lib.max()), spectra=rspec, cands=[r[\"smiles\"] for r in rows if heavy_ok(r[\"smiles\"])]))\n            stats.append(dict(molecule=mid, n_query=len(q), adducts=\"|\".join(sorted(M_by_add)), n_window=len(smiles), n_unique=n_u, n_train=int(in_train.sum()), n_shortlist=len(rows), lib_max=float(lib.max()), n_proxy=len(prox), seconds=time.time() - tm0))\n            if (n_i + 1) % 20 == 0 or n_i == len(mols) - 1:\n                d = pd.DataFrame(stats); log(f\"{n_i+1}/{len(mols)} | window median {d.n_unique.median():.0f} | shortlist median {d.n_shortlist.median():.0f} | lib_max>=0.6: {int((d.lib_max >= TAU).sum())} | {d.seconds.mean():.1f}s/mol\")\n    pd.DataFrame(stats).to_csv(f\"{P['work']}/b1_stage_stats.csv\", index=False)\n    try:   # per-candidate table (parity tests / debugging): molecule, official key, scores, provenance\n        pd.DataFrame([dict(molecule=m, **{k_: v_ for k_, v_ in r_.items()}) for m, rows_ in feats.items() for r_ in rows_]).to_parquet(f\"{P['work']}/b1_candidates.parquet\", index=False)\n    except Exception as e: log('candidate dump failed:', repr(e))\n    # ---- forward models: weakest library match first\n    runner.sort(key=lambda it: it[\"lib_max\"]); items = [dict(mid=it[\"mid\"], spectra=it[\"spectra\"], cands=it[\"cands\"]) for it in runner]\n    json.dump(items, open(f\"{P['work']}/b1_forward_in.json\", \"w\"))\n    def _budget(b):\n        if args.deadline_s > 0: b = min(b, int(args.deadline_s - (time.time() - T0) - 900))\n        return max(300, int(b))\n    GL, ICE = {}, {}\n    if not args.no_forward and items:\n        if KAGGLE:\n            sys.path.append(P[\"iceberg\"]); sys.path.append(P[\"glacier\"])\n            import fuse as ice_fuse, gl_fuse\n            try: preseed_site(P[\"iceberg\"], f\"{P['work']}/ice_site\")\n            except Exception as e: log(\"site preseed failed (runner will try its own install):\", repr(e))\n            if args.w_ice == 0: log(\"ICE skipped (w_ice = 0)\")\n            else:\n              try: ICE = ice_fuse.run_ice(P[\"iceberg\"], items, workdir=f\"{P['work']}/ice_work\", device=dev, budget_s=_budget(args.ice_budget), site=f\"{P['work']}/ice_site\", extra_args=[\"--any-rdkit\"]); log(\"ICE done\")\n              except Exception as e: log(\"ICE failed:\", repr(e))\n            try: GL = gl_fuse.run_gl(P[\"glacier\"], items, workdir=f\"{P['work']}/gl_work\", device=dev, budget_s=_budget(args.gl_budget), site=f\"{P['work']}/ice_site\", wheels=os.path.join(P[\"iceberg\"], \"wheels\"), extra_args=[\"--any-rdkit\"]); log(\"GL done\")\n            except Exception as e: log(\"GL failed:\", repr(e))\n        else:\n            for name, pkg, script in ((\"gl\", P[\"glacier\"], \"gl_runner.py\"), (\"ice\", P[\"iceberg\"], \"ice_runner.py\")):\n                if name == \"ice\" and args.w_ice == 0: log(\"ICE skipped (w_ice = 0)\"); continue\n                out = f\"{P['work']}/{name}_out.json\"\n                driver = os.path.join(os.path.dirname(os.path.dirname(HERE)), \"spark\", \"jobs\", \"ice_resume.py\")   # GB10: restart the runner after a CUDA crash and merge outputs\n                cmd_head = [P[\"runner_python\"], driver, \"--python\", P[\"runner_python\"], f\"{pkg}/{script}\", pkg, f\"{P['work']}/b1_forward_in.json\", out, \"--\"] if os.path.exists(driver) else [P[\"runner_python\"], f\"{pkg}/{script}\", pkg, f\"{P['work']}/b1_forward_in.json\", out]\n                subprocess.run(cmd_head + [\"--device\", dev, \"--site\", \"none\", \"--budget\", str(_budget(args.gl_budget if name == \"gl\" else args.ice_budget))],\n                               stdout=open(f\"{P['work']}/{name}.log\", \"w\"), stderr=subprocess.STDOUT)\n                try: (GL if name == \"gl\" else ICE).update(json.load(open(out))); log(f\"{name} done\")\n                except Exception as e: log(f\"{name} output missing:\", repr(e))\n    # ---- fusion + submission\n    def z(x):\n        x = np.asarray(x, float); ok = np.isfinite(x); out = np.full_like(x, np.nan)\n        if ok.sum() >= 2 and x[ok].std(ddof=1) > 0: out[ok] = (x[ok] - x[ok].mean()) / x[ok].std(ddof=1)\n        elif ok.sum() >= 1: out[ok] = 0.0\n        out[~ok] = np.nanmin(out[ok]) if ok.any() else 0.0\n        return out\n    booster = None\n    model_path = args.rank_model or os.path.join(HERE, \"lambdarank.txt\")\n    if os.path.exists(model_path):\n        try:\n            import lightgbm as lgb\n            from b1_rankfeats import feature_matrix\n            booster = lgb.Booster(model_file=model_path)\n            model_feats = booster.feature_name(); model_feats = None if any(f.startswith(\"Column_\") for f in model_feats) else model_feats\n            log(f\"learned fusion model loaded: {model_path} ({booster.num_trees()} trees, features {len(model_feats) if model_feats else 'default'})\")\n        except Exception as e:\n            booster = None; log(\"learned fusion unavailable -> fixed rule:\", repr(e))\n    sub, sub_fixed, fused_stats = [], [], Counter()\n    for mid in all_ids:\n        rows = feats.get(str(mid), [])\n        if not rows: sub.append((mid, \"CCO\")); sub_fixed.append((mid, \"CCO\")); fused_stats[\"empty\"] += 1; continue\n        fp = np.array([r[\"fp\"] if r[\"fp\"] is not None else np.nan for r in rows]); pop = np.array([r[\"pop\"] for r in rows]); lib = np.array([r[\"lib\"] for r in rows])\n        g = np.array([(GL.get(str(mid)) or {}).get(r[\"smiles\"], np.nan) if (GL.get(str(mid)) or {}).get(r[\"smiles\"]) is not None else np.nan for r in rows], float)\n        ic = np.array([(ICE.get(str(mid)) or {}).get(r[\"smiles\"], np.nan) if (ICE.get(str(mid)) or {}).get(r[\"smiles\"]) is not None else np.nan for r in rows], float)\n        # fixed rule first: the insurance candidate, and the fallback when the learned fusion is unavailable\n        S = z(fp) + LAM * pop\n        if np.isfinite(g).sum() >= 2: S = S + W_GL * z(g); fused_stats[\"gl_used\"] += 1\n        if args.w_ice != 0 and np.isfinite(ic).sum() >= 2: S = S + args.w_ice * z(ic); fused_stats[\"ice_used\"] += 1\n        top = np.sort(lib)[::-1]; second = top[1] if len(top) > 1 else 0.0\n        S_plain = S.copy()   # the fixed rule without library priority (used by the blend hedges)\n        if lib.max() >= args.tau and lib.max() - second >= args.margin: S = np.where(lib >= args.tau, 100.0 + lib, S); fused_stats[\"lib_priority\"] += 1\n        order = np.argsort(-S, kind=\"stable\"); sub_fixed.append((mid, \";\".join(rows[i][\"smiles\"] for i in order[:25])))\n        gl_cover_ok = np.isfinite(g).sum() >= max(2, args.min_gl_cover * len(rows))\n        if booster is not None and not gl_cover_ok: fused_stats[\"fallback_gl_cover\"] += 1\n        if booster is not None and gl_cover_ok:\n            try:\n                mi = molinfo.get(str(mid), dict(n_query=1, ce_known=0.0))\n                X = feature_matrix(fp, g, ic, pop, [r[\"lsid\"] for r in rows], [r[\"lpmid\"] for r in rows], [r[\"lsynonyms\"] for r in rows], [r[\"np\"] for r in rows], [r[\"src_mask\"] for r in rows],\n                                   [r[\"lotus_refs\"] for r in rows], lib, [r[\"lib_same\"] for r in rows], [r[\"lib_nref\"] for r in rows], [r[\"in_tier\"] for r in rows], [r[\"in_train\"] for r in rows],\n                                   [r[\"in_coconut\"] for r in rows], [r[\"ppm_err\"] for r in rows], [r[\"fp_rank_window\"] for r in rows], mi[\"n_query\"], mi[\"ce_known\"],\n                                   n_window=mi.get(\"n_unique\", np.nan), feats=model_feats)\n                pred = booster.predict(X)\n                if args.main == \"learned\": sc = pred\n                else: sc = (2.0 if args.main == \"blend21\" else 1.0) * z(pred) + z(S_plain)   # blend hedge (2026-10-25): z(learned) + z(fixed rule), 2:1 or 1:1\n                order = np.argsort(-sc, kind=\"stable\"); fused_stats[\"learned\"] += 1\n            except Exception as e:\n                fused_stats[\"learned_failed\"] += 1; log(\"learned fusion failed for\", mid, repr(e))\n        sub.append((mid, \";\".join(rows[i][\"smiles\"] for i in order[:25])))\n    df = pd.DataFrame(sub, columns=[\"molecule_id\", \"smiles\"]); df.to_csv(f\"{P['work']}/submission.csv\", index=False)\n    df_fixed = pd.DataFrame(sub_fixed, columns=[\"molecule_id\", \"smiles\"]); df_fixed.to_csv(f\"{P['work']}/submission_fixed.csv\", index=False)\n    json.dump(dict(seconds=round(time.time() - T0), molecules=len(all_ids), scored=len(feats), fusion=dict(fused_stats), gl_mols=len(GL), ice_mols=len(ICE), learned_model=(model_path if booster is not None else None), params=dict(K=args.k_short, K_fpp=args.k_fpp, K_pop=args.k_pop, w_ice=args.w_ice, tau=args.tau, margin=args.margin, gl_budget=args.gl_budget, ice_budget=args.ice_budget, deadline_s=args.deadline_s, min_gl_cover=args.min_gl_cover, main=args.main, adduct_proxy=bool(args.adduct_proxy), neg_proxy=bool(args.neg_proxy), W_GL=W_GL, W_ICE=W_ICE, LAM=LAM, TAU=TAU, MARGIN=MARGIN, folds=folds)),\n              open(f\"{P['work']}/b1_run_manifest.json\", \"w\"), indent=1)\n    log(f\"submission written: {df.shape}, fusion {dict(fused_stats)}\")\n    if args.solution:\n        sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(HERE)), \"src\"))\n        import metric\n        sol = pd.read_csv(args.solution); sol[\"molecule_id\"] = sol[\"molecule_id\"].astype(str)\n\n        def _score(sub_df):\n            s_ = sub_df.copy(); s_[\"molecule_id\"] = s_[\"molecule_id\"].astype(str); so = sol[sol.molecule_id.isin(set(s_.molecule_id))]\n            official = float(metric.score(so[[\"molecule_id\", \"smiles\"]].copy(), s_[s_.molecule_id.isin(set(so.molecule_id))].copy(), \"molecule_id\"))\n            pred = s_.set_index(\"molecule_id\").loc[so.molecule_id, \"smiles\"]\n            return official, so.assign(rr=[metric.per_molecule_reciprocal_rank(t_, str(p_).split(\";\")) for t_, p_ in zip(so.smiles, pred)])\n        official, per = _score(df); official_fixed, per_fixed = _score(df_fixed)\n        per = per.merge(per_fixed[[\"molecule_id\", \"rr\"]].rename(columns={\"rr\": \"rr_fixed\"}), on=\"molecule_id\", how=\"left\"); per.to_csv(f\"{P['work']}/eval_per_molecule.csv\", index=False)\n        out_ = dict(official_mrr25=official, official_mrr25_fixed=official_fixed, learned_model=(model_path if booster is not None else None), n=len(per), masks=dict(rows=len(EXCL_ROWS), identities=len(EXCL_IDS), per_molecule=len(MASK)))\n        for col in (\"set\", \"group\", \"lib\"):\n            if col in per.columns: out_[f\"by_{col}\"] = {str(k): dict(main=round(float(g_.rr.mean()), 4), fixed=round(float(g_.rr_fixed.mean()), 4)) for k, g_ in per.groupby(col)}\n        json.dump(out_, open(f\"{P['work']}/eval_scores.json\", \"w\"), indent=1)\n        log(f\"OFFICIAL MRR@25 = {official:.4f} (main{' = learned' if booster is not None else ' = fixed'}) / {official_fixed:.4f} (fixed rule) on {len(per)} molecules | {out_.get('by_set')}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "b1_rankfeats.py": "\"\"\"Shared candidate-feature construction for the learned fusion (training and inference use exactly this function).\n\nAll features are label-free and molecule-relative where it matters (z within the molecule's shortlist). Missing model scores are\nencoded explicitly (gl_missing / ice_missing) and the z-term is set to the list minimum (never 'average'); when a model has < 2 finite\nvalues on the list its z-term is 0 for every candidate.\n\"\"\"\nimport numpy as np\n\nFEATS = [\"z_fp\", \"z_gl\", \"z_ice\", \"fp\", \"gl\", \"ice\", \"pop_public\", \"lsid\", \"lpmid\", \"lsynonyms\", \"np_bit\", \"src_bit\", \"lotus\", \"lib_sim\", \"lib_same\", \"lib_nref\", \"lib_margin\",\n         \"lib_max_mol\", \"in_tier\", \"in_train\", \"in_coconut\", \"ppm_err\", \"fp_rank_window\", \"n_cands\", \"gl_missing\", \"ice_missing\", \"n_query\", \"ce_known\"]\n# v3 additions (Codex 008 / diagnostic D1): each model's confidence on this list (top-1 minus top-2 of its z), the candidate's rank within the\n# list under each model, its distance to the list's best, the library-similarity rank, the in_train x lib_sim interaction and the window size.\nEXTRA_V3 = [\"rank_fp\", \"rank_gl\", \"rank_ice\", \"gap_fp\", \"gap_gl\", \"gap_ice\", \"margin_fp_mol\", \"margin_gl_mol\", \"margin_ice_mol\", \"lib_rank\", \"in_train_x_lib\", \"log_n_window\"]\n# v3s = the small block to compare first (Codex 008): per-model rank + per-model confidence + window size\nEXTRA_V3S = [\"rank_fp\", \"rank_gl\", \"rank_ice\", \"margin_fp_mol\", \"margin_gl_mol\", \"margin_ice_mol\", \"log_n_window\"]\n# v2_noice: ICE removed entirely (the fixed rule is better without the ICE term on all development sets, 2026-10-24 16:50)\nFEATSETS = {\"v2\": FEATS, \"v2_noice\": [f for f in FEATS if f not in (\"z_ice\", \"ice\", \"ice_missing\")], \"v3s\": FEATS + EXTRA_V3S, \"v3\": FEATS + EXTRA_V3}\n# monotone directions for the constrained variant (0 = unconstrained): only the model scores (up) and the mass error (down), Codex 008\nMONO = dict(z_fp=1, z_gl=1, z_ice=1, fp=1, gl=1, ice=1, ppm_err=-1)\n# 2026-10-25 (panel probe): reference evidence may only raise a candidate -- in the C2 training view the truth has no references by construction,\n# so an unconstrained ranker learns 'has references -> decoy' (lib_same / lib_nref) and collapses on class-1 molecules with many references\nMONO_LIB = dict(MONO, lib_sim=1, lib_same=1, lib_margin=1, in_train_x_lib=1)\n\n\ndef rank_desc(v):\n    \"\"\"1 = best; ties share the best rank of their group (independent of the row order); candidates without a value get the worst rank (len).\"\"\"\n    v = np.asarray(v, float); ok = np.isfinite(v); r = np.full(len(v), float(len(v)))\n    if ok.any():\n        vals = v[ok]; srt = np.sort(vals)[::-1]\n        r[np.flatnonzero(ok)] = np.searchsorted(-srt, -vals, side=\"left\") + 1\n    return r\n\n\ndef zlist(x):\n    x = np.asarray(x, float); ok = np.isfinite(x); out = np.full_like(x, np.nan)\n    if ok.sum() >= 2 and x[ok].std(ddof=1) > 0: out[ok] = (x[ok] - x[ok].mean()) / x[ok].std(ddof=1)\n    elif ok.sum() >= 1: out[ok] = 0.0\n    out[~ok] = np.nanmin(out[ok]) if ok.any() else 0.0\n    return out\n\n\ndef term(vals):\n    v = np.asarray(vals, float)\n    return zlist(v) if np.isfinite(v).sum() >= 2 else np.zeros(len(v))\n\n\ndef feature_matrix(fp, gl, ice, pop_public, lsid, lpmid, lsynonyms, np_bits, src_mask, lotus_refs, lib_sim, lib_same, lib_nref, in_tier, in_train, in_coconut,\n                   ppm_err, fp_rank_window, n_query, ce_known, n_window=np.nan, feats=None):\n    \"\"\"arrays of equal length (one molecule's candidates) -> (n, len(FEATS)) float matrix in FEATS order.\"\"\"\n    fp = np.asarray(fp, float); gl = np.asarray(gl, float); ice = np.asarray(ice, float); lib = np.asarray(lib_sim, float)\n    top = np.sort(lib)[::-1]; second = top[1] if len(top) > 1 else 0.0\n    cols = dict(z_fp=zlist(fp), z_gl=term(gl), z_ice=term(ice), fp=np.nan_to_num(fp, nan=np.nanmin(fp) if np.isfinite(fp).any() else 0.0),\n                gl=np.nan_to_num(gl, nan=-1.0), ice=np.nan_to_num(ice, nan=-1.0), pop_public=np.asarray(pop_public, float), lsid=np.asarray(lsid, float), lpmid=np.asarray(lpmid, float),\n                lsynonyms=np.asarray(lsynonyms, float), np_bit=(np.asarray(np_bits, float) != 0).astype(float), src_bit=((np.asarray(src_mask).astype(np.int64) & 1) != 0).astype(float),\n                lotus=np.log1p(np.maximum(np.asarray(lotus_refs, float), 0)), lib_sim=lib, lib_same=np.asarray(lib_same, float), lib_nref=np.asarray(lib_nref, float), lib_margin=lib - second,\n                lib_max_mol=np.full(len(fp), float(lib.max()) if len(lib) else 0.0), in_tier=np.asarray(in_tier, float), in_train=np.asarray(in_train, float), in_coconut=np.asarray(in_coconut, float),\n                ppm_err=np.nan_to_num(np.asarray(ppm_err, float), nan=10.0), fp_rank_window=np.log1p(np.asarray(fp_rank_window, float)), n_cands=np.full(len(fp), float(len(fp))),\n                gl_missing=(~np.isfinite(gl)).astype(float), ice_missing=(~np.isfinite(ice)).astype(float), n_query=np.full(len(fp), float(n_query)), ce_known=np.full(len(fp), float(ce_known)))\n    feats = list(feats) if feats is not None else FEATS\n    if any(c in EXTRA_V3 for c in feats):\n        def gap(zv): return (float(np.max(zv)) - zv) if len(zv) else zv\n        def margin(zv):\n            srt = np.sort(zv)[::-1]; return np.full(len(zv), float(srt[0] - srt[1]) if len(srt) > 1 else 0.0)\n        cols.update(rank_fp=np.log1p(rank_desc(fp)), rank_gl=np.log1p(rank_desc(gl)), rank_ice=np.log1p(rank_desc(ice)),\n                    gap_fp=gap(cols[\"z_fp\"]), gap_gl=gap(cols[\"z_gl\"]), gap_ice=gap(cols[\"z_ice\"]),\n                    margin_fp_mol=margin(cols[\"z_fp\"]), margin_gl_mol=margin(cols[\"z_gl\"]), margin_ice_mol=margin(cols[\"z_ice\"]),\n                    lib_rank=np.log1p(rank_desc(lib)), in_train_x_lib=cols[\"in_train\"] * lib,\n                    log_n_window=np.full(len(fp), np.log1p(float(n_window)) if np.isfinite(float(n_window)) else np.nan))\n    return np.column_stack([cols[c] for c in feats])\n", "lambdarank.txt": "tree\nversion=v4\nnum_class=1\nnum_tree_per_iteration=1\nlabel_index=0\nmax_feature_idx=22\nobjective=lambdarank\nfeature_names=z_fp z_gl fp gl pop_public lsid lpmid lsynonyms np_bit src_bit lotus lib_sim lib_same lib_margin lib_max_mol in_tier in_coconut ppm_err fp_rank_window n_cands gl_missing n_query ce_known\nfeature_infos=[-6.0377807586954999:3.8192807495727994] [-7.7204244955041368:14.05101656247404] [-352.42535400390625:285.84283447265625] [-1:0.80725974388421873] [0:28.562094688415527] [0:10.0703125] [0:12.8515625] [0:6.30078125] [0:1] [0:1] [0:6.7345916599729483] [0:1] [0:1] [-0.96696845071513815:1] [0:1] [0:1] [0:1] [0.0014625007251234602:9.9987461574987453] [0.69314718055994529:10.601000190605713] [1:270] none [1:8] [0:1]\ntree_sizes=3471 3469 3475 3475 3451 3492 3484 3473 3480 3458 3473 3458 3465 3481 3479 3469 3470 3479 3467 3483 3453 3474 3469 3478 3483 3478 3458 3464 3443 3498 3483 3510 3478 3474 3472 3457 3456 3496 3480 3470 3487 3477 3475 3487 3478 3476 3467 3493 3467 3466 3489 3488 3486 3502 3475 3490 3499 3476 3489 3496 3467 3475 3486 3492 3490 3483 3508 3485 3496 3483 3471 3474 3478 3471 3489 3474 3491 3473 3478 3482 3506 3489 3485 3478 3497 3491 3483 3493 3481 3481 3499 3486 3475 3484 3483 3471 3515 3476 3474 3474 3508 3497 3488 3503 3488 3499 3520 3519 3506 3490 3493 3485 3502 3490 3475 3488 3486 3491 3463 3500 3503 3504 3457 3492 3504 3472 3519 3488 3486 3482 3492 3488 3492 3504 3474 3504 3498 3490 3491 3482 3463 3485 3475 3511 3492 3488 3500 3484 3501 3491 3466 3488 3517 3505 3497 3496 3492 3502 3486 3503 3511 3512 3527 3486 3515 3481 3493 3491 3497 3472 3498 3516 3490 3493 3495 3495 3500 3510 3505 3503 3488 3495 3490 3518 3480 3495 3489 3515 3490 3512 3514 3481 3499 3490 3517 3512 3491 3496 3491 3470 3502 3485 3482 3510 3507 3518 3505 3505 3472 3499 3501 3511 3520 3518 3509 3515 3517 3500 3514 3487 3493 3521 3485 3526 3512 3485 3488 3527 3485 3510 3500 3507 3491 3519 3495 3474 3491 3524 3491 3539 3496 3514 3511 3476 3512 3497 3480 3499 3519 3492 3511 3521 3524 3498 3494 3510 3539 3516 3504 3514 3486 3511 3506 3513 3519 3528 3539 3486 3463 3503 3477 3533 3516 3503 3505 3496 3512 3528 3544 3530 3494 3542 3524 3524 3533 3525 3536 3516 3488 3518 3510 3506 3495 3523 3550 3518 3506 3514 3510 3504\n\nTree=0\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 17 4 2 11 1 0 1 2 17 7 10 2 17 3 18 17 0 18 0 2 19 5 4 5 0 1 7 3\nsplit_gain=3601.98 518.933 306.523 164.469 119.555 48.159 40.1951 37.6631 35.9385 32.7176 22.9178 20.115 16.1422 19.4633 15.7908 13.686 12.1029 9.83243 9.30907 8.95 8.03384 7.65852 7.47806 6.96606 6.19205 5.29232 4.90097 4.86301 16.1889 7.32843\nthreshold=0.95935769349451683 0.62517033806334033 4.5240028322656851 11.379394531250002 -13.633526802062987 0.034261801747426719 1.4130033503388202 1.4460303622695712 3.3033179733375269 -18.836229324340817 1.8807230532061787 1.2426757812500002 1.8688348091416842 -14.08206272125244 0.45020544458187189 0.38657275885365588 3.6242520361853052 2.775573723725536 0.94290057586474996 1.4978661367769956 0.65747518837589958 -61.090599060058587 235.50000000000003 1.0000000180025095e-35 8.5990149974823016 3.9609375000000004 0.96046990436194013 1.9634439907166827 1.0000000180025095e-35 0.122919973249338\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 6 7 20 -6 9 12 24 -2 -9 14 15 -14 22 19 26 -13 -10 -1 -5 -8 -11 -21 -4 -18 -17 28 29 -23\nright_child=2 -3 8 4 5 -7 21 10 18 11 -12 17 13 -15 -16 16 25 -19 -20 23 -22 27 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.010799489925363925 -0.047869031318403914 0.056693298453382418 -0.057667126557488603 -0.05728867240037009 0.051265040100502861 -0.048722359416657458 -0.022420565817069714 0.037361642687514733 -0.031132592257459429 -0.029110454412551268 -0.047715875802963194 0.049438092934963609 -0.049028889904664356 0.036324597822410494 -0.046828710853631987 0.035433057962924591 -0.052489373712988521 -0.028564825359090027 0.047589554631723922 0.012612278583882424 0.010445712886002177 0.031831062590407022 0.043204957648513363 -0.058772098092067661 0.011329103672124873 0.00071204238428764627 -0.026642363893407305 0.059007308034336708 0.051411062100402138 -0.040601041806504735\nleaf_weight=4.0448140699882051 3.9460188691300564 41.0508877527609 24.719354897970341 21.129380458558447 26.692382058739899 4.3202617654460473 0.5966179721581274 5.8436487535363977 1.0784685807302583 0.94240474759135207 3.8843530916492455 28.993385471723744 4.4689087502192706 3.3525131765636607 3.5553627540357411 2.7953569060191508 6.9886813305783999 0.90868988912552495 3.9381521679461002 0.91012364253401656 1.3931016186252225 1.1933897184208029 4.1179286777041852 608.63647790518735 0.91276237228885193 2.0983415606897324 0.79868860635906447 490.45169781985169 83.582545716519235 2.0929496007738635\nleaf_count=661 5472 309 4391 1058 147 181 1217 889 161 391 829 1420 438 98 4094 410 3694 345 18 299 17 98 1418 200792 50 508 379 8141 5790 2529\ninternal_value=-0.000624647 -0.0459507 0.0507765 -0.0520655 -0.00150967 0.0362493 0.0553589 -0.0562604 -0.0407977 0.0262929 0.00210932 0.0349096 -0.0571578 -0.0124732 -0.00506944 -0.0576951 -0.0241615 0.04622 0.0283043 -0.0583477 -0.0528331 0.0574455 0.0271562 -0.058645 -0.054877 -0.0413516 0.0188417 0.0575666 0.0488537 -0.0130085\ninternal_weight=1389.44 738.408 651.03 697.357 53.5351 31.0126 620.381 643.822 30.6487 42.4638 9.728 38.5178 634.094 7.82142 8.6157 626.272 12.6811 29.9021 5.01662 613.591 22.5225 577.917 5.06033 609.547 25.6321 9.08702 3.59405 577.321 86.8689 3.28634\ninternal_count=246244 210709 35535 210400 1403 328 30915 208997 4620 13140 1718 7668 207279 536 5903 206743 4991 1765 179 201752 1075 17775 1809 201091 4441 4202 789 16558 8417 2627\nis_linear=0\nshrinkage=0.03\n\n\nTree=1\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 18 5 7 7 2 11 0 13 7 18 1 10 3 1 0 19 2 1 14 7 1 14 2 18 14 17 14 14\nsplit_gain=5922.94 2873.59 1334.81 617.943 345.955 307.436 256.984 237.055 169.207 250.427 188.779 139.369 132.033 126.016 109.732 100.579 95.7516 87.8756 87.5818 77.0832 73.1464 52.9674 61.1328 52.2968 47.7732 41.2091 39.8423 37.3379 36.331 35.8964\nthreshold=2.5144622974930591 1.0000000180025095e-35 2.9174053685313028 4.2109375000000009 0.89599609375000011 1.0000000180025095e-35 -17.667204856872555 0.30499968189655369 0.76735450420087303 1.0000000180025095e-35 2.5244140625000004 1.4978661367769956 4.0016611463566001 1.2424533248940002 0.44825707527455844 0.7474413184595442 1.8987378677022575 261.50000000000006 57.827308654785163 1.3779212206518803 0.95830763268103369 3.7011718750000004 0.73033385732376643 0.51224307552198323 -29.210144996643063 6.0438190738591979 0.80236194689084439 1.6068337556467065 0.27639659288032453 0.69882437336200276\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 13 11 7 -5 16 9 10 12 19 17 14 -4 27 -2 28 23 -1 -17 22 -8 -16 -14 -24 -15 -6 -7 -19\nright_child=5 -3 3 6 15 8 21 -9 -10 -11 -12 -13 24 26 18 20 -18 29 -20 -21 -22 -23 25 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.026516168973746725 -0.046539110160250967 0.054518050943799981 -0.054519584418965504 -0.050992148889722831 0.0050787615309202975 -0.014594475898028394 -0.035472433067468778 0.05512830690436147 0.050478093382268413 0.056641999787936582 0.04904235887275251 -0.037819055580419231 -0.050194867892268548 0.030631695197771973 -0.0089715477640704248 0.038602786076419274 0.047246715096555786 0.035166122904133401 0.052428071062936565 0.034568306664079472 -0.052514032731815569 0.045476379327209056 0.035115934217026523 -0.045550416404046709 0.039691708378243165 -0.049851718340274916 -0.050057972774303308 -0.052994499242918894 -0.058902144543576583 -0.049553715586031749\nleaf_weight=28.946217499178601 69.609340674243867 353.27757815693622 1805.9707048371201 51.759977352805436 33.074458175338805 24.86118128686212 19.640502168389503 39.461878969101235 1338.1833131518215 154.4663271708414 65.562818725884426 149.51987988778274 5.1844730367884031 21.737879799620718 50.317765963845886 241.54657259662054 10.548956000478936 20.579389256774451 10.130231857299803 48.289594171801582 7.3609548928216091 67.924845891306177 44.509124401956797 120.52746077789925 43.192241365381051 4.9065804040292269 6.3014760413207105 13.168947181780821 55.04618318611756 4.7569450461305669\nleaf_count=1420 1071 625 208040 6949 1416 309 1593 23 1817 88 90 9945 78 570 1197 1302 24 85 4 349 165 539 418 5080 139 114 592 826 1000 83\ninternal_value=-0.000469293 -0.0258784 -0.0363151 -0.0465279 0.00681453 0.0422376 0.00535246 -0.00422691 0.0454851 0.027516 0.0068259 -0.0209993 -0.0113921 -0.0513877 -0.052279 0.0281819 -0.0337667 -0.0301673 -0.029743 0.0116289 0.0357088 0.0268582 0.00815736 -0.034892 0.0292372 0.0258579 0.0113984 -0.0122307 -0.0454677 0.0179822\ninternal_weight=4910.36 3078.91 2725.63 2203.73 521.907 1831.45 188.741 119.62 1711.83 373.65 219.183 226.756 153.62 2014.99 1986.95 295.151 80.1583 105.244 180.975 77.2358 248.908 136.981 69.0562 170.845 48.3767 49.4157 28.0394 46.2434 79.9074 25.3363\ninternal_count=245951 241144 240519 225096 15423 4807 9613 1118 3689 1872 1784 11714 1694 215483 214321 3709 1095 1477 6281 1769 1467 2664 2125 6277 217 532 1162 2242 1309 168\nis_linear=0\nshrinkage=0.03\n\n\nTree=2\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 6 13 18 7 1 5 1 0 2 8 4 14 3 4 8 14 3 4 18 3 14 4 19 18 14 18 22 2\nsplit_gain=8327.17 1849.83 454.687 372.23 583.015 393.463 210.759 137.405 166.019 125.179 81.0083 59.9139 57.4548 55.7599 88.6569 49.8923 44.8427 70.228 44.0276 71.278 65.6444 40.8273 55.9802 36.1311 35.4796 34.8399 33.7653 32.2401 31.1538 71.3503\nthreshold=1.3779212206518803 1.0000000180025095e-35 3.2382812500000004 1.0000000180025095e-35 2.249904835165133 0.89599609375000011 3.3033179733375269 3.7255859375000004 2.1468275719512682 1.2225960941547209 -18.836229324340817 1.0000000180025095e-35 1.2946246862411501 0.91125284794847328 0.53650715206675215 2.7826617956161503 1.0000000180025095e-35 0.032916404154818342 0.31840721315068282 3.2580566406250004 3.9981586158983733 0.44825707527455844 0.59283710889839769 0.84667968750000011 254.50000000000003 0.89587973461402759 0.72685412594425991 5.0530509062989237 0.93750000000000011 -8.7471351623535138\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 4 13 6 15 8 -7 16 -4 25 -8 23 -15 24 18 -18 -1 -20 -21 22 -10 -2 -6 -11 -26 -17 -5 -30\nright_child=3 -3 10 28 5 7 12 -9 21 11 -12 -13 -14 14 -16 27 17 -19 19 20 -22 -23 -24 -25 26 -27 -28 -29 29 -31\nleaf_value=-0.053711446152768891 -0.0024911102671597374 0.048633217339124561 -0.049793157113681229 0.053469804042036877 -0.047398224736512215 -0.023328666949563578 0.036333174097415051 0.045145138492023909 0.021478487859360962 0.010048323194203146 0.036614694229264923 0.030003889582870997 -0.019269204814527111 -0.054894946332135548 0.036155382461948327 -0.046485032868974797 0.01583448916435368 -0.043160638157715731 -0.051386048123341715 0.023607822136908391 -0.038168847511213959 0.035589621341264803 -0.039495141461470193 0.046021729853963418 -0.0047739719441519737 -0.03912972110980522 -0.053998498438505231 0.042615650632816325 -0.052927471809266734 0.043055881864142588\nleaf_weight=1794.2686205309874 13.919612122583201 196.54301439016126 10.526169274933634 870.07418139185756 131.67443266275222 82.425797061179765 61.744183568283916 194.28834504564293 63.780034181429073 16.424686491722241 66.402040008455515 19.876644843956456 22.083591365488246 12.299762164708225 34.032027375884354 4.2964722923934451 21.050748820882291 113.40760346021852 113.52259853796568 26.836308087804355 34.035602153628133 145.93163814430591 16.16957010619808 947.6342342418211 43.414101274567656 54.841474985529203 16.76685462705791 11.13535188930109 7.3748674131929866 43.831130084581673\nleaf_count=193305 400 421 1096 415 7956 3782 358 628 730 522 400 219 329 269 87 274 308 9305 12015 557 2898 640 512 1747 1548 4081 1203 100 84 58\ninternal_value=0.000426867 -0.0394788 -0.0471218 0.0365945 0.0286341 0.00938519 -0.0183288 0.0254423 0.0129201 -0.0496358 0.0243042 -0.0150157 0.0216308 0.0437682 0.011335 -0.0345148 -0.0511283 -0.0338757 -0.0522889 -0.0374298 -0.0109923 0.0261921 0.00881119 0.0453176 -0.0386816 -0.0278135 -0.0191467 0.0164893 0.0521134 0.028483\ninternal_weight=5190.61 2467.74 2271.19 2722.88 1801.6 793.71 291.115 502.595 308.307 2194.26 76.9282 91.1428 83.8278 1007.89 46.3318 207.287 2103.12 134.458 1968.66 174.395 60.8719 225.881 79.9496 961.554 191.855 71.2662 60.181 15.4318 921.28 51.206\ninternal_count=246247 225127 224706 21120 20563 18060 11768 6292 5664 223210 1496 4822 687 2503 356 11081 218388 9613 208775 15470 3455 1882 1242 2147 10707 4603 2751 374 557 142\nis_linear=0\nshrinkage=0.03\n\n\nTree=3\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 4 1 11 18 5 0 4 13 4 18 1 18 0 4 19 2 18 2 0 18 4 1 4 21 9 19 10 2\nsplit_gain=7890.5 1554.52 461.897 336.548 343.2 211.328 121.146 115.766 172.422 91.8434 133.326 78.2824 68.5623 66.0518 43.8961 78.2494 42.2346 37.5653 36.6263 36.3388 33.914 36.1257 35.3786 29.7876 72.3431 40.1907 29.7149 41.5859 27.857 33.8516\nthreshold=1.3779212206518803 1.0000000180025095e-35 11.379394531250002 2.1468275719512682 3.8857805861880484e-16 2.1383330595080281 3.4492187500000004 0.82186659671441242 4.4806598424911508 1.0000000180025095e-35 0.84667968750000011 1.2424533248940002 4.0016611463566001 6.3465127852423047 0.44654938290861101 6.0415039062500009 263.50000000000006 -43.562467575073235 1.4978661367769956 -10.615592002868651 0.53713645201195115 4.2121001622835372 1.6286621093750002 1.0240454605967615 3.8417438268661503 2.5000000000000004 1.0000000180025095e-35 213.50000000000003 1.2424533248940002 53.951957702636726\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 5 -2 20 23 18 10 -5 -12 14 -4 15 16 -13 -6 -9 -8 -7 22 -22 28 -25 -26 27 -16 -1 -30\nright_child=3 -3 13 9 17 6 19 8 -10 -11 11 12 -14 -15 26 -17 -18 -19 -20 -21 21 -23 -24 24 25 -27 -28 -29 29 -31\nleaf_value=-0.051269686863635838 0.028733105866053571 0.045260213427418351 0.033917887949891971 -0.032497497870818605 -0.042883858454656296 -0.04837897200826153 -0.047153578601436305 -0.011764610489937568 0.025496894947819936 0.050841352095422759 0.047585348076105122 -0.044373367403586127 0.043984835975669469 -0.047904653108614366 -0.049033682647899292 0.039715849353435445 0.027750367175221457 0.046494638781636115 -0.044693346266755571 0.033307267487897911 -0.052500909104562364 0.029262884135482236 -0.011095902732805089 -0.049741762767026673 0.022529452563328074 -0.048414962178889492 0.050874426649324261 0.02735232328638048 -0.037267448766903383 0.043525774631101993\nleaf_weight=1819.002209381084 111.28471561381593 185.72843184275553 73.924981597810984 23.394406601553783 3.5803442727774373 76.971272708819015 4.9371033697389057 37.311186488950625 43.525202299468219 718.71927990438417 536.93753868620843 27.375198098132387 298.25186633970588 9.1864728804212046 5.6468770569190374 27.650096390862018 9.0754767907783371 207.95698774443008 186.72991788655054 36.788625095970929 30.555977941257879 10.570041798986493 46.883876481791958 79.719230356859043 24.511799612082541 9.0077892718836647 49.922730459831655 372.99488224775996 16.690554037224505 5.4136657333001486\nleaf_count=191975 908 430 476 1367 293 6209 314 1751 865 391 450 1747 574 940 192 95 126 422 17652 223 2760 200 2035 8052 585 754 38 2786 1349 15\ninternal_value=9.76704e-05 -0.0380523 -0.044782 0.0366523 0.0152721 -0.00444106 -0.0223534 -0.0473801 -0.0286921 0.0421117 0.0374343 0.0386899 0.0325985 0.0244119 0.0256302 0.00243251 -0.0261545 0.0447872 -0.0392861 0.0229061 -0.0338691 -0.0207781 -0.0278944 -0.0499226 -0.0343568 0.00261417 0.0290861 0.026087 -0.0508608 -0.0164525\ninternal_weight=5090.25 2490.75 2305.02 2599.5 529.529 317.992 206.707 2221.91 267.566 2069.97 1351.25 1327.85 790.917 83.1115 492.665 64.1008 36.4507 211.537 224.041 41.7257 164.981 88.0099 77.4399 1954.35 113.239 33.5196 428.564 378.642 1841.11 22.1042\ninternal_count=245974 224844 224414 21130 13364 12649 11741 222998 20268 7766 7375 6008 5558 1416 4984 1968 1873 715 19403 537 11204 4995 4795 202730 9391 1339 3016 2978 193339 1364\nis_linear=0\nshrinkage=0.03\n\n\nTree=4\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 16 1 11 2 18 18 14 7 1 17 2 4 2 1 2 14 12 4 7 12 18 7 3 1 14 21 17 22\nsplit_gain=7573.55 1564.25 348.07 255.484 277.24 211.887 158.755 109.209 98.7055 88.3125 59.3704 56.0678 52.5845 48.4086 43.9966 41.87 34.6907 33.885 32.4978 31.8629 31.3224 31.2083 51.665 47.0176 67.1414 40.4151 28.9044 26.9219 25.8551 24.2551\nthreshold=1.4475910192202701 0.49644476220754463 1.0000000180025095e-35 2.3713805701640611 3.8857805861880484e-16 -14.08206272125244 2.9174053685313028 2.4414009612931857 0.85380319876708544 0.89599609375000011 0.95935769349451683 3.1006363674515387 -19.19599628448486 4.9616699218750009 -18.054428100585934 3.3033179733375269 -43.562467575073235 0.62855356909667792 0.063354284020036747 2.7826617956161503 3.2763671875000004 0.24517227696643781 2.524928003624769 2.9707031250000004 0.29586597258655462 2.9477339485081675 0.85380319876708544 7.5000000000000009 3.9281938047451082 0.55000000000000016\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 6 -4 26 10 13 19 20 21 -11 -7 -6 -13 -3 -12 -15 -8 -1 22 -5 24 -24 -26 28 -27 -2 -23\nright_child=3 16 5 11 14 8 9 -9 -10 12 17 15 -14 18 -16 -17 -18 -19 -20 -21 -22 29 23 -25 25 27 -28 -29 -30 -31\nleaf_value=-0.038101999538742999 0.028158670704020294 -0.042892892602873156 -0.045727503059416807 0.043372294931750217 -0.020961526485535242 -0.026087994395424221 -0.043586207097904568 -0.049160383977050873 -0.039904936931658967 -0.034715386470514371 0.028192241704461066 -0.04934123173805402 0.015493245508153345 0.036928171614245421 0.046023209494772205 0.023430391138812755 0.043856972469701008 -0.045795754836422786 -0.044844389899702401 0.015662451289276035 0.034132432816814211 0.049191288011669942 -0.050841539073404753 0.048135618764473892 0.0019140810607145109 0.034688232453818711 -0.018005667461544823 -0.043396942804364791 -0.04466484645860188 0.023522660669589839\nleaf_weight=70.70216596510727 170.22103771939874 3.4803529307246199 81.930147571838461 867.49986842740327 8.5981860214378667 13.211244292557238 78.986801534309052 1979.6917017343221 23.551170402206481 23.759206537040882 28.163800203008577 8.7400695802643877 76.730631788726896 97.323932857485488 254.02898149704561 25.449683440383524 203.36229653144255 5.9424679542426011 3.7597863236442199 8.5965583669021708 5.0828342037275425 681.53402107907459 10.212894601514561 79.628161011263728 51.657097655814141 146.30161587661132 13.593551974976434 3.2683126223273566 3.640798491658642 37.68067000200972\nleaf_count=5826 1519 244 7879 1277 415 632 6627 208526 1357 1726 505 584 1762 844 315 261 735 469 152 247 27 374 562 57 1456 963 544 132 185 151\ninternal_value=0.000232584 -0.0366861 -0.0437667 0.0366602 0.0201768 -0.00781411 0.00361212 -0.0475426 0.0148871 -0.0158205 -0.0182171 0.0421484 0.00339713 0.0264255 0.043759 0.00395397 0.042198 0.0143637 0.0334772 -0.0376596 -0.0328759 0.0428414 0.0396743 0.028542 0.0209712 0.0248493 0.0230893 0.0327051 0.026387 0.047881\ninternal_weight=5066.33 2516.2 2309.36 2550.13 638.156 219.776 375.529 2089.58 137.846 188.073 109.891 1911.97 100.49 114.295 262.627 34.1898 206.843 34.1063 101.084 87.5834 75.785 1877.78 1158.57 291.068 211.44 201.227 187.455 149.57 173.862 719.215\ninternal_count=246353 227196 226217 19157 13340 10864 12610 215353 2985 10362 6827 5817 3488 1628 730 845 979 974 996 6874 5853 4972 4447 3170 3113 2551 2248 1095 1704 525\nis_linear=0\nshrinkage=0.03\n\n\nTree=5\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 0 13 16 18 2 4 3 11 3 4 13 0 9 2 4 13 18 2 3 4 4 3 13 4 9 17 18 17\nsplit_gain=4719.84 1739.66 650.274 656.739 455.528 243.967 230.931 202.381 144.151 88.1298 81.0277 75.3196 74.2513 65.2291 62.2663 93.187 59.6653 54.7945 63.6829 50.6834 49.1804 45.9208 44.355 34.3524 34.045 31.4457 29.1455 22.4408 21.0366 20.6852\nthreshold=0.26534039634031065 1.0000000180025095e-35 0.75738199555027408 -0.05277688766252419 1.0000000180025095e-35 2.524928003624769 -18.836229324340817 9.6361774206161517 0.40911314460322756 0.21084670836356409 0.17443601580070725 0.84667968750000011 0.14796868535883259 0.29549219484135919 1.0000000180025095e-35 18.240050315856937 1.8393554687500002 1.0000000180025095e-35 1.8688348091416842 132.99933624267581 0.36072066459518604 6.2602539062500009 0.84667968750000011 0.45898141303402712 -0.4958236943477588 0.84667968750000011 1.0000000180025095e-35 0.24468021674538498 3.9415346756528771 0.26690563994538458\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 7 5 10 -6 8 -2 16 26 12 23 -10 15 -8 -5 18 -13 -17 21 -20 -15 -4 -24 -12 -1 -27 -9 -26\nright_child=2 -3 11 9 6 -7 14 28 13 -11 25 17 -14 22 -16 19 -18 -19 20 -21 -22 -23 24 -25 29 27 -28 -29 -30 -31\nleaf_value=-0.036479406826896925 -0.042646914773519604 0.044764379538302076 -0.048592698202576091 -0.036712578695303505 -0.045019075906120289 -0.047461122231247474 -0.046204264615827648 -0.039346672261890196 -0.040893877732364227 0.044289965902680903 -0.043831561255998706 0.040102271068473368 0.04578722455542697 -0.04728909058892028 0.038359653646634004 0.03874348394037036 0.023763401382938941 0.046335704861267993 -0.014102924492743155 -0.019351401444904427 0.029743307220017164 0.042547930347165433 -0.014712833938697068 0.0075945389214083503 -0.02407899646027643 -0.042573488004380086 0.035739950057454659 0.025451428482760038 0.04244795105381282 0.025007947820305288\nleaf_weight=59.087174272746779 220.87257341656368 245.05657894979231 18.999716694001108 19.825196651509032 71.460808105766773 1751.476652630372 25.003149352036417 2.2511524490546426 36.047817931277677 262.10938755562529 6.4882279394660136 763.43164661806077 23.180926125496626 12.571221928577868 78.385084949433804 43.523398618446663 48.889711301773787 572.02635787148029 46.92885830020532 18.679706882219762 365.69406463357154 16.609711611643434 34.797377299517393 18.731172626139596 7.7935105724027371 3.7848161491565397 4.7011645887978366 64.637143901316449 45.874537558760494 76.174260070780292\nleaf_count=4599 21931 421 1742 1636 7031 184665 1809 29 3153 666 626 1580 10 1266 530 407 1077 364 2454 317 3495 55 2145 629 471 256 22 1076 253 1449\ninternal_value=-0.000222108 -0.0307943 0.027751 0.00456711 -0.0395157 -0.0444295 -0.000311818 -0.0195838 -0.0267881 0.0363871 -0.00608133 0.0374861 0.00460633 -0.00573242 0.019136 0.00161786 0.00603892 0.0386178 0.0348804 0.0212673 0.0255211 0.00113606 0.00415441 -0.0212231 0.00994906 0.0149997 -0.0306874 0.0211295 0.0378988 0.0202221\ninternal_weight=4965.09 2372.28 2592.81 767.207 2127.23 1890.18 237.052 436.382 388.257 330.824 138.699 1825.6 60.9118 167.384 165.591 87.2063 68.7149 1764.69 1192.66 62.2031 429.233 63.5386 131.336 37.7309 118.765 74.9102 63.7883 68.422 48.1257 83.9678\ninternal_count=246164 201759 44405 34076 201338 191244 10094 30697 30415 3379 6579 10329 2381 8484 3063 2533 2713 7948 7584 724 6004 2509 5331 2371 4065 1958 4621 1332 282 1920\nis_linear=0\nshrinkage=0.03\n\n\nTree=6\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 0 13 16 2 6 14 18 14 3 3 7 13 7 13 6 13 3 14 3 7 0 7 0 18 7 2 14 0\nsplit_gain=4584.34 1634.88 536.211 573.519 496.146 228.747 163.202 156.007 129.633 111.014 102.923 90.3977 88.5594 73.207 70.5997 58.9031 42.56 40.9979 45.6876 70.3143 38.8531 37.0686 52.1498 33.5795 33.2204 30.2621 29.4374 59.9331 28.5054 27.268\nthreshold=0.26802264518526847 1.0000000180025095e-35 0.76735450420087303 -0.05277688766252419 1.0000000180025095e-35 -25.182852745056149 3.0683593750000004 0.52722272431259987 2.524928003624769 0.77909152491212919 0.37248792739977959 0.18631447338168272 1.0000000180025095e-35 1.0000000180025095e-35 0.89599609375000011 0.14796868535883259 1.4980468750000002 0.14796868535883259 0.42638606281916264 0.39695441677226773 0.53650715206675215 3.1132812500000004 1.5447487307370646 1.0000000180025095e-35 1.8987378677022575 6.0438190738591979 3.1982421875000004 -1.7340650558471677 0.28824823456061044 0.38644940889434637\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 6 8 -6 7 10 11 -7 -2 -1 15 14 -5 24 -11 18 19 -14 -9 22 -21 -13 -4 -16 -10 -28 -3 -22\nright_child=2 28 12 13 5 9 -8 20 26 16 -12 23 17 -15 25 -17 -18 -19 -20 21 29 -23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.040076459480503854 -0.031573536882964062 -0.015799720242938353 -0.018177496993676594 -0.028969602009215025 -0.04686052269465886 0.032656419641764314 0.038666910552806855 -0.041767872586470661 -0.046371672471019039 -0.046908059467898756 0.02164099329219445 -0.043911309292385077 0.03559789807553617 0.044491340890283705 0.033112219181315149 0.044050644298413349 0.011348935021344653 0.045542147515851721 0.038231865693761947 -0.022881269033542231 -0.045824390230603342 0.036963937238918151 0.023358208703673371 0.019568162479456152 0.029447202244743065 -0.041375926202745571 -0.040291430453702821 0.032545629799069851 0.044894722377324642 0.0064418029863533097\nleaf_weight=58.544946821173653 49.595238983165473 6.7077972656115881 52.788989421213046 25.0434624934569 62.769977936055511 135.44113867869601 42.9465959565714 208.97785114985891 1699.4228329295292 24.853502651210874 91.564847593195736 7.6876445715315631 199.07239045039751 242.63459604652599 71.359017266426235 28.354010917246342 19.459492130205035 472.97780728340149 983.07094534230419 50.933894534711726 11.899340532254426 25.687880076467991 37.039765671826899 55.563378134509549 16.494337726850063 4.429996160324662 41.99416340701282 12.489401292055843 239.46072394261137 28.948885577730834\nleaf_count=4905 4527 162 3088 1937 6424 1543 146 21011 180851 1669 2346 721 785 211 801 14 486 260 2749 2694 1133 107 198 1021 84 392 4456 222 245 1020\ninternal_value=0.000478297 -0.0298662 0.0276167 0.00667132 -0.038372 0.00199915 -0.0162405 -0.0223548 -0.0435901 0.0192258 0.002854 -0.0135265 0.0363326 0.0355809 0.0139133 0.00841946 -0.0216394 0.0378678 0.0350336 0.0248901 -0.0365532 0.00598907 -0.00336923 0.0112894 -0.00658179 0.0282761 -0.0456689 -0.0233092 0.0431836 -0.00951485\ninternal_weight=5008.21 2364.39 2643.82 777.4 2118.23 242.524 433.933 390.986 1875.7 179.754 141.16 121.796 1866.42 343.467 100.832 97.6373 44.313 1768.78 1295.8 312.734 249.826 113.662 87.9737 63.251 69.2833 75.789 1753.91 54.4836 246.169 40.8482\ninternal_count=246208 202705 43503 33524 202298 10122 30183 30037 192176 3698 6873 6647 9979 3341 3130 3186 2155 6793 6533 3784 23164 2999 2892 1742 3172 1193 185529 4678 407 2153\nis_linear=0\nshrinkage=0.03\n\n\nTree=7\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 4 1 11 18 18 4 4 11 13 10 7 13 4 2 7 18 14 2 1 18 4 1 18 11 2 18 1 4\nsplit_gain=6030.16 1096.52 329.513 240.599 217.555 189.324 132.531 83.2968 60.4834 57.0455 50.8369 42.0397 39.2825 36.1262 49.1162 35.7145 45.0164 33.9642 33.053 32.3245 31.6592 30.7805 28.1266 26.7836 25.9471 24.8334 23.282 22.6632 22.3767 67.5607\nthreshold=1.3779212206518803 0.49644476220754463 11.379394531250002 2.5144622974930591 3.8857805861880484e-16 2.9174053685313028 2.4414009612931857 6.2768554687500009 4.9616699218750009 0.027600239276592001 -0.041888724317900632 1.7005986908310777 0.89599609375000011 1.0000000180025095e-35 0.84667968750000011 -8.3227787017822248 0.89599609375000011 6.3465127852423047 0.66452579377975118 -43.562467575073235 3.3033179733375269 2.1383330595080281 2.3505859375000004 0.95935769349451683 1.4978661367769956 5.5511151231257837e-16 -14.08206272125244 5.0271592211193568 1.0240454605967615 3.8417438268661503\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 4 5 12 8 15 18 17 -3 28 21 14 -5 -7 27 -4 23 -6 25 -2 -18 24 -1 -16 -13 -17 -8 -30\nright_child=3 10 9 13 19 7 11 -9 -10 -11 -12 26 -14 -15 20 16 22 -19 -20 -21 -22 -23 -24 -25 -26 -27 -28 -29 29 -31\nleaf_value=0.0011255307245533358 0.016485308164248934 -0.043476863216639607 0.036625766163814015 -0.020414951861974253 -0.039261838322571817 -0.038603143740238322 -0.044932347117981566 0.028000742442033481 0.02910793335423317 -0.045843376555217447 0.040019958875963275 -0.040039409104315471 0.032469437060434946 0.04406194472514971 0.027975193789728237 -0.038725669396465053 0.033209221054559576 -0.039858407948520386 -0.046254242829594866 0.040819125880033294 0.038788684948692378 -0.026233902189189189 -0.011505734927473416 0.017814125482610917 -0.041817157152456416 -0.044502224807364092 0.029685345282804157 0.011631540563899407 -0.0446874228463673 0.0053471373966067362\nleaf_weight=23.362794562242925 67.977856931276619 6.0283078143838784 63.02665672916919 13.58599463989958 3.8680840013548723 77.109760721446946 1884.1122342075687 32.978343773400411 24.757500726729631 9.1584665908012521 174.78860683739185 5.8848521467298296 154.5412610899657 670.23416012059897 281.51027391455136 45.223999188863672 22.44846112607047 4.8777506323531261 33.724044624017552 259.62079695146531 820.63104092050344 18.508810762548819 26.714524974115193 20.701280153356493 26.520181415136904 3.4618186522275201 10.545926405582575 9.2575331674888712 87.874208384426311 33.14977133157663\nleaf_count=861 844 550 230 761 342 6618 204225 453 188 673 420 593 713 369 2150 3848 289 481 2833 518 1536 1073 1193 480 2499 94 154 351 9255 1390\ninternal_value=1.17617e-05 -0.0338836 -0.0396521 0.0325674 0.0178632 0.00526787 -0.0418369 -0.0152505 -0.0122491 0.0210734 0.0370124 -0.0437189 0.0234505 0.0384562 0.0350628 -0.0232523 -0.0115717 0.030633 -0.0222484 0.0394995 0.0357623 0.00714771 0.0091667 -0.0102928 -0.022078 0.0269422 0.00384833 -0.0301135 -0.0441104 -0.0310509\ninternal_weight=4916.19 2408.51 2227.7 2507.67 718.249 454.761 2150.63 213.733 129.066 77.0629 180.817 2021.57 241.028 1789.42 1119.19 180.754 103.645 67.9044 104.308 263.489 1105.6 86.4867 49.163 70.5843 49.883 284.972 16.4308 54.4815 2005.14 121.024\ninternal_count=245984 224832 223862 21152 16242 15382 222478 12752 6861 1384 970 215617 2630 4910 4541 12299 5681 711 6673 860 3780 1917 1482 3840 3360 2244 747 4199 214870 10645\nis_linear=0\nshrinkage=0.03\n\n\nTree=8\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 9 1 11 18 0 7 7 0 0 5 11 16 19 18 12 7 18 14 3 18 7 5 14 5 1 3 14 14\nsplit_gain=5515.77 1192.24 299.648 222.008 229.021 213.064 112.875 61.0426 60.6643 60.2928 52.0854 68.4266 49.038 48.0537 43.7504 34.2774 31.4211 35.7447 39.0043 27.5863 27.1213 23.6456 26.8763 55.2302 21.6737 19.5457 19.5442 19.1188 16.5496 16.1751\nthreshold=1.4130033503388202 0.49644476220754463 1.0000000180025095e-35 2.5144622974930591 3.8857805861880484e-16 2.249904835165133 0.65747518837589958 0.89599609375000011 4.2421875000000009 0.32384722065649935 0.82186659671441242 3.1562500000000004 0.28995392115848545 1.0000000180025095e-35 250.50000000000003 6.8062759569045816 0.24517227696643781 1.0000000180025095e-35 3.0677824455408698 0.87268561280719448 0.29938127432470335 1.2424533248940002 0.89599609375000011 0.89599609375000011 0.30018599471723478 1.4980468750000002 -0.79535375801325292 0.45898141303402712 0.6618576200563534 0.57916335608139347\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 8 4 5 -2 13 29 10 -9 -1 12 -12 -4 -3 19 17 18 27 26 -11 -19 23 -23 -25 -26 -15 -5 -22 -7\nright_child=3 14 6 16 -6 7 -8 9 -10 20 11 -13 -14 15 -16 -17 -18 21 -20 -21 28 22 -24 24 25 -27 -28 -29 -30 -31\nleaf_value=-0.04371225153185674 0.026459524530062801 0.039053198366697579 -0.043583515233931593 -0.017795129159371093 0.03845250112184586 -0.022151431399576663 0.037956575268150147 -0.040483619977473233 0.027949463629000654 -0.022896221578201294 -0.03724195050152445 0.014970326585342272 0.034976300461745824 -0.041256020172043641 -0.044748975465429587 -0.041771349669767477 0.042211248595202228 0.039752595495115811 -0.044867755591421053 -0.040279458841650989 0.027224263647383153 0.031083604349314231 0.035228826758207391 0.014483950829095031 -0.051543820799274935 -0.0010188123059354719 0.031573157186176336 0.036254033608872589 -0.0063848424966439273 -0.045988805960916687\nleaf_weight=1813.1033081959467 181.93756575428415 199.98985103168525 35.457553149200976 7.2707866039127138 262.6359353242442 60.720021760324016 52.696303960401565 26.294863078743219 10.59508882742375 16.545355948852375 208.86087902949657 27.999190012458712 8.0633653025142831 2.8249631787184617 4.9710959789808831 11.822811595164238 679.78902100771666 496.70584173267707 9.433030952815896 6.1048280633985987 51.879886072594672 141.82551852008328 384.63890839880332 26.495751969981939 11.210774988867344 15.912005918100474 32.034172353567556 21.645443975459784 17.124775961739942 48.691586945205927\nleaf_count=198158 1300 648 3854 258 835 4257 326 2006 129 945 19463 691 55 109 349 1307 509 354 702 373 652 592 1365 284 384 288 261 149 687 4597\ninternal_value=-6.0937e-05 -0.0322681 -0.0386899 0.0315414 0.0167099 0.00248686 0.00361112 -0.017256 -0.0415723 -0.00162015 -0.0419436 -0.0287264 -0.0344093 -0.0170942 0.0368133 0.00117846 0.0370307 0.0338377 0.0052276 0.0142744 0.0106548 0.0348509 0.0305925 0.0213351 -0.00454741 -0.0229928 0.0246852 0.0225232 0.0189115 -0.0330802\ninternal_weight=4875.28 2414.52 2209.56 2460.76 665.83 403.194 140.941 221.256 2068.62 111.845 2058.03 244.923 216.924 88.2443 204.961 52.7868 1794.93 1115.14 38.3493 40.964 85.55 1076.79 580.083 195.444 53.6185 27.1228 34.8591 28.9162 69.0047 109.412\ninternal_count=245887 225723 224726 20164 15279 14444 6230 13144 218496 4290 218367 20209 19518 5904 997 2050 4885 4376 1109 743 2284 3267 2913 1548 956 672 370 407 1339 8854\nis_linear=0\nshrinkage=0.03\n\n\nTree=9\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 4 1 12 18 4 0 18 2 4 3 4 14 17 1 3 4 18 19 16 14 14 18 13 18 7 0 17 0\nsplit_gain=5294.05 990.617 398.299 159.732 161.459 146.288 94.945 88.0821 86.393 50.5056 48.3669 61.0501 38.4549 37.5735 33.466 30.34 29.0219 40.2498 47.4449 33.7771 27.7358 45.8995 27.5572 24.8852 24.8113 24.4446 32.442 25.6781 21.9257 18.231\nthreshold=1.4475910192202701 1.0000000180025095e-35 9.6361774206161517 2.5144622974930591 2.2204460492503136e-16 2.524928003624769 1.6286621093750002 1.4460303622695712 6.3465127852423047 -26.006598472595211 7.3286132812500009 0.37963534769016083 4.1440429687500009 0.11320561501600769 3.1006363674515387 1.7622051334857016 0.31840721315068282 3.2580566406250004 3.9981586158983733 245.50000000000003 1.0000000180025095e-35 0.075876460218247857 0.96902457618464355 2.8617925509761908 0.14796868535883259 1.7005986908310777 2.5244140625000004 0.29549219484135919 3.325091732892195 -0.064617101914726832\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 5 28 -7 16 22 -6 11 13 -9 -8 24 -13 20 -18 19 -19 -1 29 -4 -16 25 -5 27 -27 -2 -22\nright_child=3 -3 8 14 9 6 10 12 -10 -11 -12 15 -14 -15 23 -17 17 18 -20 -21 21 -23 -24 -25 -26 26 -28 -29 -30 -31\nleaf_value=-0.042913565054216445 0.02689522675699858 0.036790665422152188 0.032380349215984115 0.037791068663835581 -0.043577127714642318 -0.039612671779285923 0.011088481583400882 -0.02179297994187707 -0.042250935639541541 0.039204175853447924 0.036845663381527213 -0.024009302037380515 0.030319522490455966 -0.041678302853826586 0.025203273309058163 0.023267698988226251 -0.039615635684548303 -0.02640194545822911 -0.033437089140534837 0.038044796849807841 -0.02135046626391535 -0.036960842577106939 -0.039107294881856818 -0.030621375071526433 0.041887022879499428 -0.016737441289241428 0.042272646521078823 0.025705377924955631 -0.041134596671224245 0.031578721414843497\nleaf_weight=1780.33994260896 208.86691949842498 175.88487453502603 91.73028192226775 655.69827558100224 6.0147701473906627 64.114274451043457 16.334956534672529 30.941885537002236 17.389205121435225 264.88357603177428 30.2200605974067 14.517378772841765 20.205171031877398 42.61792105215136 17.346286445390437 58.300010478706099 119.62852420646232 10.196062981849538 35.859485220396891 20.682947812136263 8.7953068115748483 79.972266377881169 4.3003948987461618 10.824930482544003 552.246014341712 12.651545963482929 121.3298657676205 359.8831868886482 3.4990328107960513 13.823804196203126\nleaf_count=196440 1430 445 612 665 416 5825 516 1965 1885 363 169 867 169 3490 122 1087 13044 566 3584 149 513 7465 168 546 277 519 102 2604 197 93\ninternal_value=-7.12897e-05 -0.0316028 -0.0370019 0.0310778 0.0190463 0.00776352 -0.00901574 -0.0399315 0.0176402 0.0372127 0.00326298 -0.00465834 -0.00101977 -0.0271163 0.0359974 0.0136447 -0.0408924 -0.0292577 -0.0103803 0.0166042 -0.0420278 -0.0263904 0.028809 0.00343801 0.0365341 0.0339283 0.0287338 0.024223 0.0255868 0.010965\ninternal_weight=4849.1 2409.75 2233.87 2439.35 709.369 438.471 226.105 2120.45 113.42 270.898 161.99 131.77 51.1471 58.9529 1729.98 72.8174 2069.3 186.367 66.7385 30.879 1882.93 102.591 96.0307 28.1712 1701.81 1149.56 493.865 372.535 212.366 22.6191\ninternal_count=246293 227098 226653 19195 14360 13581 11954 223988 2665 779 6129 5960 2134 4006 4835 1954 221854 17343 4299 715 204511 8071 780 668 4167 3890 3225 3123 1627 606\nis_linear=0\nshrinkage=0.03\n\n\nTree=10\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 16 1 11 2 7 13 0 14 0 2 0 11 13 3 7 0 9 2 14 4 1 14 0 1 2 3 14 4\nsplit_gain=5058.4 877.095 277.175 180.48 182.231 177.236 118.402 85.588 81.2844 67.4821 60.712 45.2974 43.9096 38.0561 30.7966 29.8531 29.5483 28.4067 26.853 44.0319 26.2937 24.0106 20.923 20.7227 19.3751 19.226 19.3952 18.9734 17.7033 16.7025\nthreshold=1.4475910192202701 0.60365083802632757 1.0000000180025095e-35 2.5144622974930591 3.8857805861880484e-16 -14.08206272125244 0.89599609375000011 0.14796868535883259 1.0212340148387515 0.85380319876708544 0.38644940889434637 -26.006598472595211 -0.53258533561495902 0.11347059467310934 1.0000000180025095e-35 0.34948566548243237 1.0000000180025095e-35 1.8987378677022575 1.0000000180025095e-35 23.600493431091312 0.84677952177506588 5.1123046875000009 1.7090960825073716 0.32729182297007969 1.6765212893675572 3.3033179733375269 -28.784247398376461 0.26534039634031065 0.26756436277776396 3.2895507812500004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 6 -4 8 17 29 13 12 -6 -8 18 16 21 24 -1 19 -7 28 23 -17 27 -5 26 -18 -12 -10 -2\nright_child=3 -3 5 14 11 9 10 -9 20 -11 15 -13 -14 -15 -16 22 25 -19 -20 -21 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.040794278991921805 -0.038215842332251497 0.036741303540165249 -0.040789665060681456 -0.0088400845840511284 -0.041862807098319903 -0.032257934016102545 0.02083543534436828 0.034540416983468089 -0.024071997659590047 -0.033300441781358554 -0.041787269277384631 0.038052248881358532 -0.04488122036975839 -0.046687131608433505 0.040592506119710263 -0.0060223156417527858 0.0032093905486660808 0.0041588169553242391 0.039597348581436341 0.028947234574404072 -0.04365367545580158 0.030121366065287914 0.032989157759956318 -0.045327807294119643 0.033391273280923679 0.035586962963379896 0.028879969388574876 0.02633997576987112 0.018390622619927861 0.018739570389401581\nleaf_weight=1959.4283373358194 82.17617851763498 157.08768295333721 77.232742185471579 25.860555490711704 5.7583368388004592 14.416136513929812 14.824291498400269 13.060483190696685 9.3990310612134618 20.959025762276724 4.2291365708224502 259.68736921111122 21.110104419291019 5.6464188869576892 637.91209312342107 13.180798868183045 29.807985036866739 12.556193426717071 68.644018003484234 32.982816350180656 7.3819084321148685 15.461255969014017 148.59065599530004 11.010924154659731 14.434471721760927 777.81471176631749 257.75127485115081 11.514592255000023 61.838839226053096 4.3325184672139576\nleaf_count=215077 7420 487 7939 1065 444 1071 338 34 296 1408 255 363 1820 182 364 369 642 443 176 258 440 109 741 711 62 1384 1354 90 854 117\ninternal_value=-0.000369527 -0.0315751 -0.0364567 0.0301635 0.0168979 -0.00447526 0.00421149 -0.0399952 -0.0152852 0.0153332 0.017638 0.036168 -0.0179405 0.0239127 0.0352521 0.0239624 0.0321371 -0.0405061 0.0277175 0.0100501 0.00677493 0.00144691 0.0297931 -0.015612 0.00673028 0.0330909 0.0262393 0.00663668 0.0125325 -0.0351819\ninternal_weight=4776.09 2362.01 2204.93 2414.08 670.496 219.881 405.05 1985.05 165.128 142.648 239.922 265.446 35.9344 121.689 1743.58 203.987 1105.67 1971.98 116.043 47.399 78.6198 42.2159 161.771 26.7547 40.295 1065.37 287.559 15.7437 71.2379 86.5087\ninternal_count=246313 227075 226588 19238 14367 11034 13560 215554 9127 3095 4433 807 2158 1687 4871 2275 4507 215520 1505 1329 1590 1165 1110 1056 1127 3380 1996 345 1150 7537\nis_linear=0\nshrinkage=0.03\n\n\nTree=11\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 0 13 4 6 14 18 2 3 7 11 13 4 10 7 3 0 3 13 18 7 4 3 18 18 0 2 3 19\nsplit_gain=3456.09 1209.56 427.987 467.86 358.733 140.083 114.232 105.814 105.513 97.2219 83.3643 66.1609 51.4479 50.0937 45.0244 40.12 38.118 37.0451 36.7927 35.3041 39.8004 37.1652 63.145 41.584 29.3699 28.9784 20.4083 20.0421 18.7755 21.633\nthreshold=0.26534039634031065 1.0000000180025095e-35 0.75738199555027408 -0.05277688766252419 8.8319251537323016 3.0683593750000004 0.52722272431259987 2.524928003624769 -18.836229324340817 0.38657275885365588 1.0000000180025095e-35 0.21084670836356409 -0.032724337246802453 4.8474121093750009 1.2424533248940002 2.9707031250000004 0.48555671035752257 1.8987378677022575 0.22136347059745312 0.14796868535883259 1.2424533248940002 0.89599609375000011 1.4160156250000002 0.42638606281916264 5.3057863027301417 2.602003343538398 1.8987378677022575 -45.689584732055657 0.50207005315878861 259.50000000000006\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 5 7 6 9 13 -6 -2 12 15 17 18 -9 16 -5 -4 26 20 -12 22 -22 -24 -16 -25 -1 -13 -8 -30\nright_child=2 -3 10 11 8 -7 28 14 -10 -11 19 27 -14 -15 24 -17 -18 -19 -20 -21 21 -23 23 25 -26 -27 -28 -29 29 -31\nleaf_value=-0.037176294909977231 -0.033225657961239052 0.038035692524759177 -0.032521390263140743 -0.034862424970155327 -0.03391471352619084 0.035170317983678055 -0.039622334183849882 -0.040402479136289027 0.031436948010964728 0.019393292335489958 0.036293850068646924 -0.012717473470569571 0.02896433224874137 0.02837245570997847 0.031053866250095225 0.036722082718283193 0.018284664381188462 0.030118730988216196 0.0082509475848680045 0.039979763282578185 0.02566132026624824 0.029834775269789511 -0.045989029792564237 0.017599147081837949 -0.039055880815281739 -0.04455626893416554 0.004639286869914592 0.038894838016298831 -0.032385578872008915 0.010079571718505735\nleaf_weight=66.825814579613507 49.875516958301887 228.665285136085 36.39074258110486 25.939001115970314 28.457372528966516 40.825044247321784 187.94693301618099 1718.2840310933534 90.643100323854014 82.423197046853602 565.50824130838737 6.5099161779507986 35.150839221198112 22.247338690329343 12.844557271339001 22.586927476339042 21.257409833837301 10.139715449884532 29.409748659934849 488.99063374847174 137.2161882603541 499.03707584843505 19.330775129841641 35.062359274365008 7.8013024902902535 7.3198476023972026 12.184249725658445 260.06970434216782 29.791756940539926 16.030459286412224\nleaf_count=5381 5007 410 2899 2144 2710 145 20624 190279 682 1978 668 300 342 174 152 95 468 59 870 272 1049 2800 1478 335 762 468 409 363 2438 528\ninternal_value=0.00026709 -0.0271865 0.0238871 0.00467465 -0.0347013 -0.0169495 -0.0228421 -0.0379168 0.0156697 -0.000544057 0.0316658 0.0308058 0.00201497 -0.011771 -0.0398735 0.00471822 -0.0110421 -0.0185246 -0.0201923 0.0330459 0.0303328 0.025458 0.014384 -0.0107 0.00398157 0.00608465 -0.0307505 0.0375916 -0.0354053 -0.0176316\ninternal_weight=4794.77 2217.36 2577.4 743.256 1988.7 406.893 366.068 1869.6 119.1 132.299 1834.15 336.363 81.6813 130.667 1738.93 69.7833 47.1964 46.5305 108.42 1752.47 1263.47 697.966 198.929 61.713 20.6459 42.3822 79.0101 266.58 233.769 45.8222\ninternal_count=246289 201829 44460 34090 201419 30720 30575 198027 3392 6985 10370 3370 3300 6834 191193 2707 2612 2958 6660 7070 6798 6130 3330 2281 914 803 5790 663 23590 2966\nis_linear=0\nshrinkage=0.03\n\n\nTree=12\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 8 1 11 18 2 7 14 0 2 13 18 7 3 2 6 3 19 11 6 2 7 0 0 2 18 11 2 14\nsplit_gain=4434.94 826.372 228.28 158.653 166.185 167.094 121.197 79.5378 66.9548 40.5544 34.8215 33.4676 32.0721 37.0011 35.1618 26.5093 25.2751 27.1708 25.0507 24.3868 24.1785 24.1299 22.9652 22.6211 21.5369 19.3846 17.1984 15.754 39.0724 15.2865\nthreshold=1.4130033503388202 1.0000000180025095e-35 1.0000000180025095e-35 2.5144622974930591 3.8857805861880484e-16 2.249904835165133 -25.182852745056149 0.89599609375000011 0.87268561280719448 1.8987378677022575 -30.0182991027832 0.14796868535883259 2.6735537653587351 2.5244140625000004 0.29586597258655462 -42.103004455566399 1.4980468750000002 0.33504387972412514 237.50000000000003 0.75683518236239655 9.1289062500000018 -43.562467575073235 2.6738281250000004 -0.7405159439372101 0.38644940889434637 11.076014518737795 1.4978661367769956 0.71017588656534381 -5.9238033294677725 0.47889823060390885\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 4 5 -2 -4 -7 23 19 -9 12 -5 14 -14 -16 17 22 -17 26 -10 -6 -12 -8 -19 -11 29 28 -3 -1\nright_child=3 27 6 11 21 7 8 10 20 25 16 -13 13 -15 15 18 -18 24 -20 -21 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=0.0050006764356173902 0.024670553223429634 -0.047462103120340854 -0.040607994499680798 0.033220174006198352 -0.03523100894900149 -0.033149863285753811 -0.023855877616918982 -0.03356733728379057 -0.041089880281426704 0.031080307861308713 -0.04274306390689845 0.039416135557897192 -0.031948603844730229 0.038251084629269898 -0.015517478393492426 -0.042637475210197937 0.033947016105836128 -0.039265454430169378 0.023317493062245669 0.022304355323709349 0.030842821083229815 0.035374983801943885 0.02832862965893709 0.023464884481747078 0.01854039764152475 -0.037744070052961556 -0.039302020514689222 0.038966136369686599 0.029769389162968542 -0.040808093056121458\nleaf_weight=15.56557186599821 189.9300928900484 6.2124890715349457 60.662377584725618 840.2660554475151 3.6712818297091863 100.61109292157926 9.1193628213368338 18.953255121828988 25.441974417772148 10.725072669796644 16.948351218132306 549.59488532505929 12.810455461731179 83.458019061014056 18.731658390723169 4.4562474926933637 27.20889137336053 5.6204017298296085 169.97040910203941 5.2671233010478309 4.2431279905140391 267.11555249709636 4.5368715766817322 108.36918325116858 48.743558794027194 4.4116467237472525 1885.7175177318859 128.44762083049864 32.679286147933453 10.09493422473315\nleaf_count=617 1310 265 6727 1424 331 8781 610 1563 1984 101 1358 281 703 111 585 206 178 480 1513 38 9 507 46 1592 684 333 212477 96 78 871\ninternal_value=-0.000585148 -0.0301658 -0.0351887 0.028299 0.0160765 0.00406668 -0.00528568 -0.0135432 0.00944935 -0.0384015 0.00276256 0.0332601 0.0302329 0.0214758 0.0145266 0.0177575 0.00970232 0.000676608 0.021399 -0.0387882 -0.0301379 0.0342916 -0.027143 0.0196216 0.0119629 0.00992527 -0.0389681 0.0338861 0.0165521 -0.0138754\ninternal_weight=4669.58 2306.96 2139.62 2362.63 683.339 412.553 207.836 222.622 147.174 1931.78 122.011 1679.29 1129.69 289.427 205.969 193.158 103.058 75.8492 174.427 1916.65 29.6851 270.787 21.4852 117.489 54.364 15.1367 1911.38 167.339 38.8918 25.6605\ninternal_count=245859 225798 225359 20061 15238 14400 10922 13090 4195 214437 4309 4823 4542 3118 3007 2304 2746 2568 1719 214003 1993 838 1404 2202 1164 434 213965 439 343 1488\nis_linear=0\nshrinkage=0.03\n\n\nTree=13\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 4 1 11 8 0 7 13 1 0 14 11 13 14 0 7 0 7 0 7 3 0 7 1 14 14 4 22 22\nsplit_gain=4138 849.526 293.633 159.774 157.618 80.9317 85.2724 59.8823 48.8517 48.5847 54.4558 83.1605 42.1189 40.2589 37.8426 32.4369 31.9825 30.7714 28.7679 25.4185 25.2634 24.8681 22.1614 24.9557 20.7977 18.4275 18.26 18.2135 29.6035 17.5155\nthreshold=1.4130033503388202 0.49644476220754463 11.379394531250002 2.1468275719512682 3.8857805861880484e-16 1.0000000180025095e-35 0.13844441961022361 0.89599609375000011 0.14796868535883259 4.0016611463566001 0.76735450420087303 0.56086626140189244 0.027600239276592001 -0.041888724317900632 0.8755209476635436 1.0212340148387515 0.89599609375000011 0.38644940889434637 2.6738281250000004 -0.7405159439372101 1.0000000180025095e-35 0.48555671035752257 0.88405423821904405 3.2763671875000004 0.95935769349451683 1.0000000180025095e-35 0.29138017841767527 6.5971679687500009 0.93750000000000011 1.0000000180025095e-35\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 4 7 22 25 15 9 10 11 16 19 -3 -8 -2 21 18 -9 -4 -12 -5 -1 24 -24 -7 -26 28 -19 -6\nright_child=3 13 12 8 29 6 14 17 -10 -11 20 -13 -14 -15 -16 -17 -18 27 -20 -21 -22 -23 23 -25 26 -27 -28 -29 -30 -31\nleaf_value=-0.039378764353218673 -0.030889680332031658 -0.039799057202966497 -0.034431089929755118 -0.032677895683314147 0.034919866663166696 0.010576231153583375 0.024260019924917817 -0.041035562863209418 0.038679265747691476 0.034887778446507971 -0.0031530154751385321 -0.038988342223527934 -0.040469939678833868 0.034439265767752081 -0.037318208954559916 0.00065929843667910282 0.02889834417912774 -0.011698559772010748 0.019648379065127061 0.03288640402813995 0.028179797039141984 0.017305471138898645 -0.035387055473165945 0.027758122598683477 0.017334067185179082 -0.038356369854525361 -0.025649904820734745 0.035938288050838846 0.023103756187301507 -0.00090335146119949379\nleaf_weight=1711.2613615326118 73.624603296164423 6.0157142954412839 4.6417647940106681 15.711237344658 179.07526885578409 7.1989813670516005 48.022004898637533 17.366493852809075 592.37226960249245 557.0872190205846 23.927356881322339 28.35593585530296 8.1079296539537591 182.73393524484709 10.063951849006115 48.254500435199589 73.816552030853927 35.716388273052871 10.619615023024378 69.184690839145333 577.30590541055426 18.375479642767459 136.07001750764903 6.2419405500404528 13.520918062888084 75.355597165413201 22.267790882149711 30.694849936757237 54.315339188091457 13.129817884182556\nleaf_count=198002 6638 563 485 1022 432 116 751 1647 315 808 749 1942 663 431 799 1243 674 1166 228 251 1893 272 13731 66 384 7901 1704 91 654 248\ninternal_value=-0.000383103 -0.0289758 -0.0344268 0.027613 0.011783 -0.036675 -0.0145835 -0.00298429 0.0314872 0.0281689 0.0230533 0.00587756 0.021044 0.0318759 0.01314 -0.0184951 0.0179176 0.00976637 -0.018125 0.0282503 0.0269297 -0.00600758 -0.0383108 -0.0278871 -0.030057 -0.0340138 -0.00938087 0.0163278 0.00932052 0.0324807\ninternal_weight=4650.44 2300.69 2111.94 2349.75 462.797 2030 140.641 270.592 1886.95 1294.58 737.492 136.259 81.9344 188.75 58.086 121.879 107.903 148.713 27.9861 73.8265 601.233 34.0867 1889.36 178.101 171.859 82.5546 35.7887 120.727 90.0317 192.205\ninternal_count=245869 225847 224853 20022 12347 223454 9567 11667 7675 7360 6552 3910 1399 994 1550 7881 1968 3786 1875 736 2642 1294 213887 15885 15819 8017 2088 1911 1820 680\nis_linear=0\nshrinkage=0.03\n\n\nTree=14\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 4 1 13 18 5 2 18 10 13 1 18 4 5 18 13 2 4 14 4 18 14 14 14 2 2 2 18 18\nsplit_gain=3918.87 735.503 299.071 150.394 141.461 104.004 118.002 86.5092 58.8883 45.1851 35.3652 38.3883 32.4217 50.8124 28.2745 68.01 25.9022 24.1669 30.3434 27.1779 22.8201 33.7246 22.809 22.0574 21.89 20.9576 19.8163 17.9433 16.8066 16.6334\nthreshold=1.3779212206518803 1.0000000180025095e-35 9.6361774206161517 2.5144622974930591 1.0000000180025095e-35 2.9174053685313028 3.4169921875000004 -13.633526802062987 1.4978661367769956 1.7005986908310777 1.0000000180025095e-35 4.0016611463566001 3.0201273556387074 5.9252929687500009 3.9609375000000004 5.2284276452838947 -0.41325737289625047 -18.054428100585934 1.6286621093750002 0.60840604842079971 1.6286621093750002 1.2424533248940002 0.66452579377975118 0.28824823456061044 0.86227501112960836 2.5338792800903325 -17.667204856872555 -7.9326825141906729 3.1984648276080736 4.166635176627655\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 8 4 5 20 17 -4 22 14 11 12 -5 16 -10 24 -14 -7 -19 -20 21 -2 -1 29 -16 -23 -8 -11 -21 -18\nright_child=3 -3 7 10 -6 6 26 -9 9 27 -12 -13 13 -15 15 -17 23 18 19 28 -22 25 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=0.0099866115411948037 0.026273665374153803 0.033158281064738629 -0.039093124283972443 0.026049550030043368 0.035954298621008927 -0.036214882779846982 -0.034950606278238407 0.028225259117466425 -0.038062249265780702 -0.033010538765841084 0.037283701840348168 0.033469771418017644 0.012236698004506715 0.036374441056993466 0.021758874895574758 -0.038313967207622888 0.015069034912045963 -0.032655941005920473 0.011315333395651066 0.015960402231428802 0.024348073295301056 0.0027128170735199785 -0.039084134024240449 -0.045130425672144196 -0.036578580993703161 -0.045152686576063555 0.030886606869981122 0.032578520246985772 -0.0410825714578286 -0.041051131083342872\nleaf_weight=34.450551594840363 29.446288839913905 160.61916360538453 20.404172081500292 382.15509939100593 203.69864041171968 58.927806311286986 3.5978034297004333 87.387135334312916 1811.4374633276602 4.4806312932632872 617.89784461632371 567.81213314505294 34.796798371477053 35.930597402621061 30.37621657573618 75.200086741242558 15.45043698558584 41.837035723379813 50.937213682336733 5.5972037659958032 198.28792110388167 19.482050185091794 10.226887965807689 19.142942875158042 6.152765571139752 13.125090332236139 60.172772179357708 11.302599121350791 17.556776040932164 5.6855356486048541\nleaf_count=1002 173 430 2060 1253 193 5568 290 560 208848 478 374 829 623 62 833 9146 209 3884 1563 101 1230 519 794 1147 516 814 413 105 1470 352\ninternal_value=-0.000188199 -0.0285534 -0.0333063 0.0266353 0.0149502 0.00632075 -0.00796997 0.0152401 -0.0359507 -0.036735 0.0315141 0.028122 0.0219094 0.0075509 -0.0371506 -0.0221634 -0.00654061 -0.0206905 -0.012557 -0.000923932 0.0193999 0.00345285 -0.00185567 -0.0226657 0.0112199 -0.0173243 0.0266899 0.0131013 -0.0272032 -0.0011997\ninternal_weight=4633.58 2252.04 2091.42 2381.54 702.667 498.968 238.627 107.791 1983.63 1938.95 1678.87 1060.97 493.161 111.006 1923.17 111.729 75.0757 174.856 115.928 74.0912 260.341 62.0534 44.6774 40.2789 36.529 32.6071 63.7706 15.7832 23.154 21.136\ninternal_count=245839 224772 224342 21067 16218 16025 13289 2620 221722 219926 4849 4475 3646 2393 219343 10495 2331 12586 7018 3134 2736 1506 1796 1708 1349 1333 703 583 1571 561\nis_linear=0\nshrinkage=0.03\n\n\nTree=15\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 4 1 11 18 5 18 1 11 13 13 1 17 4 7 1 4 2 2 5 14 13 14 7 17 9 17 0 4\nsplit_gain=3835.18 765.432 246.598 164.586 168.622 139.1 76.0331 54.0018 51.0384 39.06 35.1502 33.3197 30.0352 35.535 29.8462 24.5411 24.8756 24.1283 23.5458 23.5268 23.0207 18.4701 23.0911 18.2868 18.2679 16.5396 16.1776 27.4833 18.5808 13.2362\nthreshold=1.3399809614428995 0.49644476220754463 11.379394531250002 2.5144622974930591 3.8857805861880484e-16 2.249904835165133 4.0507812500000009 2.4414009612931857 0.95935769349451683 0.027600239276592001 -0.041888724317900632 0.14796868535883259 3.3033179733375269 3.1006363674515387 1.6286621093750002 0.89599609375000011 2.1468275719512682 0.84667968750000011 -25.182852745056149 -43.562467575073235 1.8691406250000002 0.79531390260019708 -0.51158128144345905 0.29138017841767527 2.6738281250000004 1.2596354777617942 1.0000000180025095e-35 0.32406367513646889 -0.25794343014581789 2.7826617956161503\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 5 14 15 8 -1 18 -3 12 13 17 -2 29 20 -5 -4 -6 25 -19 -23 -10 -22 -17 -9 28 -28 -7\nright_child=3 10 9 11 19 6 -8 26 23 -11 -12 -13 -14 -15 -16 16 -18 21 -20 -21 24 22 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.028274133354606941 -0.0024431738579114783 -0.038058913087433602 -0.032987449188487492 -0.02470940067299051 -0.034197666765949743 -0.031259682897883445 0.029872880928139663 -0.037345917675183457 0.032479191236989675 -0.039413744584891421 0.033258622092581437 0.037555613051450699 0.031412457862389706 -0.042675688856752381 0.027670629755987299 0.014457037422217045 0.0126126579934736 0.026589263046637206 0.031848729745762007 0.033209968377822266 -0.043036661946473584 0.019946200421713006 -0.024824787733153133 -0.013122557355118695 0.011212225431045834 -0.038196646829898449 -0.033610525755488896 -0.03493193102337025 0.026688634277382488 0.0063786798889177779\nleaf_weight=75.228398517472669 36.583445466589183 5.6481361147016278 4.6745521910488597 8.8890261785127205 3.9730665718670926 105.38998932612594 32.314424769487232 1784.0824002581649 20.900915438774973 8.1450408701784891 174.60663598123938 549.10252792201936 847.42750546149909 6.9490742639172813 151.27261434169486 21.70294806431048 34.187624140409753 254.18107644328848 62.826458941213787 280.50981055875309 30.907896010670811 31.441792524885386 14.272466652560977 11.696402468718587 6.3349798823473966 6.0348932542838147 5.72583467839286 39.152659860206768 13.721916515380142 8.7967423265799862\nleaf_count=6074 802 546 440 476 365 9678 257 209357 137 687 405 285 1739 419 629 678 741 1308 274 568 2545 237 377 650 171 495 639 4276 317 350\ninternal_value=0.000341899 -0.0282862 -0.0335577 0.0263339 0.0142486 0.0024532 -0.0123877 -0.0356196 -0.0147996 0.0193769 0.0308263 0.0313904 0.0284502 0.0204255 0.0218243 -0.018898 -0.00786327 0.0219788 0.0269427 0.0321521 -0.0186639 0.0234401 0.00573604 0.016213 -0.0337571 0.00207072 -0.0368292 -0.0207341 0.00815977 -0.0283299\ninternal_weight=4636.68 2206.41 2026.15 2430.27 718.008 433.526 245.669 1950.51 107.826 75.6461 180.255 1712.26 1163.16 315.733 187.856 213.355 99.1683 308.784 67.501 284.483 64.9807 299.895 45.7143 32.5973 37.2429 27.7378 1842.68 58.6004 19.4478 114.187\ninternal_count=245922 223802 222851 22120 17279 16346 14915 221450 6861 1401 951 4841 4556 2817 1431 14658 4630 2398 714 933 3889 1922 614 787 2716 1173 214589 5232 956 10028\nis_linear=0\nshrinkage=0.03\n\n\nTree=16\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 4 1 11 0 0 4 4 0 4 13 1 0 17 1 3 14 17 19 0 14 2 21 7 9 0 17 4 4\nsplit_gain=3557.98 818.447 256.068 139.975 132.837 108.535 55.3764 47.9118 45.54 40.5143 58.448 35.423 33.7141 32.0513 29.3485 25.9486 24.0323 23.0746 22.5701 22.3322 21.8521 20.8412 20.6524 19.3713 19.2994 16.9986 16.6403 16.1566 15.3037 14.8199\nthreshold=1.3779212206518803 1.0000000180025095e-35 9.6361774206161517 2.5144622974930591 3.8857805861880484e-16 0.70465417701170696 -0.7405159439372101 1.6286621093750002 7.0266062021255502 0.82186659671441242 3.5791015625000004 0.14796868535883259 3.3033179733375269 1.3771184689138856 2.6250469244966204 0.60367420583093068 0.37963534769016083 0.51224307552198323 3.5426598534814651 205.50000000000003 1.8987378677022575 0.82879072691359035 -43.562467575073235 2.5000000000000004 4.5273437500000009 1.0000000180025095e-35 0.91093860112668257 0.6411330003973541 7.6279296875000009 2.1186523437500004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 4 5 8 -4 13 16 -1 20 12 14 -7 21 -12 -2 27 -9 -8 -11 26 -6 -23 -21 -19 28 -18 29 -5\nright_child=3 -3 6 11 22 7 19 18 -10 10 15 -13 -14 -15 -16 -17 17 25 -20 24 -22 23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.036796282543968446 -0.035321383005932988 0.032166218174842304 -0.037084055036945167 0.0094243726144814365 -0.033126475746736772 -0.025360640249992513 -0.029926069070495897 0.023073783882343723 0.026268674235052782 -0.033043312226408834 -0.028850538563265089 0.03676292482872353 0.030559863555608639 0.015700977099971536 -0.040441159325512462 0.022406013290878459 -0.03693771018515439 -0.034015150768145846 -0.038734100488476982 0.030472262032220716 0.0080950195402835004 -0.042658409828372046 0.031883913671810482 0.010611506866670454 -0.038727677741134621 0.031792810530570488 0.026554612000916949 0.021513438449323824 0.036589821287894467 -0.029893194413025466\nleaf_weight=1690.6701419919264 68.707796690519899 190.64653541427106 15.350246876943855 33.401626353617758 3.7037301631644359 37.598312624613754 5.7997225914150468 213.21772759361193 18.213127342052758 174.76819751586299 11.325898444280027 551.66254899278283 813.28073339723051 29.828221543924883 6.4670434119179836 31.221423890325241 4.2495843477081534 27.451526313554499 4.5805462296120814 76.609723281580955 12.06680651148781 7.3750044610351351 267.25014308234677 22.126063286326826 2.9762864322401574 3.2363973110914221 214.79322803812101 22.520783635787666 12.008348878473042 10.516672384226693\nleaf_count=201662 6540 457 1805 748 350 2461 124 1943 203 18423 930 297 1737 435 428 530 335 2651 279 548 403 318 524 266 132 27 661 473 6 403\ninternal_value=0.000241511 -0.0271273 -0.0327343 0.0257563 0.0144684 0.00407183 0.0137722 0.0147849 -0.0170483 -0.0351802 -0.0231608 0.0304754 0.0273484 -0.00723177 0.0187605 0.00841257 -0.0234588 -0.00891218 0.0216028 0.0232384 -0.0303574 0.0201658 0.0308767 -0.00366748 0.0274359 -0.0262906 0.0227669 0.0112388 0.00800885 -0.000446508\ninternal_weight=4583.62 2211.43 2020.79 2372.19 700.558 429.604 100.736 285.225 144.379 1920.05 229.382 1671.63 1119.97 67.4265 306.688 42.5473 126.166 57.4583 217.798 85.3857 186.835 300.221 270.954 29.5011 79.586 30.6879 270.72 26.7704 55.9266 43.9183\ninternal_count=246099 225014 224557 21085 16221 15347 2609 5118 10229 221948 20286 4864 4567 2896 2830 1460 10026 3486 2222 804 18826 2402 874 584 680 2678 1818 808 1157 1151\nis_linear=0\nshrinkage=0.03\n\n\nTree=17\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 4 1 11 18 7 18 18 2 13 18 7 3 11 16 18 7 14 3 14 1 4 18 2 1 14 7 4 19\nsplit_gain=3318.64 639.542 276.508 135.754 131.108 114.06 61.5052 58.8469 57.2165 33.5143 33.0728 35.9419 44.8685 41.5936 31.6326 30.6906 40.6427 29.6829 28.9887 25.8692 20.4288 20.1503 19.8773 19.758 19.741 18.6519 17.1638 15.9704 20.9974 16.3318\nthreshold=1.3779212206518803 1.0000000180025095e-35 9.6361774206161517 2.5144622974930591 3.8857805861880484e-16 2.9174053685313028 2.7412109375000004 6.3465127852423047 1.4978661367769956 -30.420168876647946 0.14796868535883259 2.249904835165133 2.5244140625000004 0.37248792739977959 0.75683518236239655 1.0000000180025095e-35 5.2284276452838947 1.0000000180025095e-35 0.8755209476635436 0.37963534769016083 0.66452579377975118 4.0016611463566001 1.6286621093750002 3.1567740231385479 -43.562467575073235 1.7622051334857016 0.96902457618464355 0.89599609375000011 1.2946246862411501 247.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 8 4 5 17 19 26 20 -3 11 -5 13 23 15 -10 18 -2 -17 -7 -1 27 -21 -13 -6 -24 -4 28 29 -15\nright_child=3 9 7 10 24 6 -8 -9 14 -11 -12 12 -14 21 -16 16 -18 -19 -20 22 -22 -23 25 -25 -26 -27 -28 -29 -30 -31\nleaf_value=0.010329357806974479 -0.014774187245340293 -0.042846187078702143 0.027933221046883925 0.030448501963085992 -0.032524465431693197 -0.032182763949107143 0.025841882406810731 -0.036630380997605468 -0.036293588439106045 0.032287929697777419 0.036185807316994702 0.0065805791375873673 0.035986018076386794 -0.03893326531669157 0.025339478849464664 0.016620549025916899 -0.035476818083460239 0.020797611331905188 -0.035272938455296964 -0.028072306834291644 -0.036908304344552181 0.027512615818436163 -0.027430945949172916 -0.036417206558266212 0.031001736650673489 0.01369276450657316 -0.033087298504156433 0.018128508690974376 -0.033854748027672507 0.015611788609267447\nleaf_weight=34.651654105633497 22.698006701888517 4.696268619503825 85.399804689688608 744.26966833975166 3.7056468992959699 85.313164429273456 29.143505130894482 15.49013517051935 1737.1721613573609 158.983587609604 494.72049470990896 17.163038302212954 96.463681955356151 4.7960508530959478 7.1029357248917213 33.243842385476455 59.305109086912125 214.38958586729132 12.513484163675455 27.738637946429662 9.7166338074021024 92.528164125047624 12.368815419264136 20.143803284969181 269.32325354614295 37.634984336211346 3.5437914216890922 70.473863234743476 13.938524211756883 33.441878923913464\nleaf_count=1005 984 202 588 1071 354 7846 329 1859 211105 258 275 242 115 266 46 900 6943 1732 1052 2588 802 290 972 1088 532 978 154 493 568 412\ninternal_value=-0.000336126 -0.0269893 -0.0316608 0.0248231 0.0138112 0.00345393 -0.0136817 0.0157354 -0.0342815 0.0298851 0.0296817 0.026708 0.0186658 0.0119326 -0.0350888 -0.0353354 -0.0192994 0.0173443 0.00197811 -0.0208603 -0.000601171 0.0170009 -0.0081844 -0.0169806 0.030024 0.00318213 0.0251611 0.00891235 -0.00364259 0.00795251\ninternal_weight=4452.07 2161.82 1998.14 2290.25 702.316 429.287 192.199 104.434 1893.71 163.68 1587.94 1093.22 348.949 252.485 1849.34 1842.23 105.062 237.088 45.7573 163.056 44.3683 215.178 77.7424 37.3068 273.029 50.0038 88.9436 122.65 52.1765 38.2379\ninternal_count=246049 224914 224454 21135 16315 15429 12713 2601 221853 460 4820 4545 3474 3359 220046 220000 8895 2716 1952 12384 1807 2029 4538 1330 886 1950 742 1739 1246 678\nis_linear=0\nshrinkage=0.03\n\n\nTree=18\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 16 13 18 7 1 2 4 14 18 18 3 11 14 5 1 3 19 2 4 14 5 2 2 7 4 19 14 19\nsplit_gain=3262.43 654.977 195.802 145.804 176.636 111.859 108.417 107.274 45.299 37.5091 35.1147 32.5058 32.4872 31.5863 27.7488 26.2824 25.0157 29.1108 36.161 24.7826 41.8289 24.8377 19.7628 19.6555 19.0326 16.0559 15.7715 17.8721 15.6212 16.5681\nthreshold=1.3399809614428995 1.0000000180025095e-35 1.0000000180025095e-35 0.14796868535883259 3.3842466058243152 2.5244140625000004 2.9477339485081675 -14.08206272125244 0.84667968750000011 0.80526343422453606 1.4978661367769956 1.7005986908310777 0.33061855659870582 0.75683518236239655 0.80526343422453606 7.1621093750000009 2.9477339485081675 0.42638606281916264 247.50000000000003 -14.450672149658201 1.6286621093750002 0.60840604842079971 4.4472656250000009 -10.988907814025877 -22.378742218017575 0.89599609375000011 3.5791015625000004 241.50000000000003 0.34961562014038972 254.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 10 4 8 6 19 -4 23 -9 25 -10 -8 -12 -14 -11 17 18 -13 -6 -21 -22 -23 -2 -7 -1 27 -3 -20 -30\nright_child=3 26 7 -5 5 24 12 9 11 15 13 16 14 -15 -16 -17 -18 -19 28 20 21 22 -24 -25 -26 -27 -28 -29 29 -31\nleaf_value=-0.022929173870602441 0.013973891757091461 0.025204561816281144 -0.032623033402161911 0.035850294634151278 -0.035897211281999845 -0.033832490276416903 -0.041230803092579434 0.021241285583641859 0.029559539471340737 -0.037390268460746792 -0.035987021196845076 -0.025378221320866829 0.02549876838728338 0.025234177462420224 -0.039980533980707764 0.03154912052043532 0.025877438662054823 0.020518024260985882 0.027956677329553647 -0.039122495395661518 0.012003383196331546 -0.039492513046881429 0.031444611745087836 -0.03274275237838857 0.031124694671674571 0.016686592156205077 0.033654510823255976 -0.044061037058276409 -0.041464394120071434 0.012474626267687727\nleaf_weight=19.6004944446031 14.545096128946172 7.6480681356042606 71.155549033544958 699.33664370980114 71.806222876301035 3.4456176515668622 7.1913986830040804 100.97693953337148 765.4748052326031 18.086991163436327 1699.5709201653954 33.909019190818071 87.722854959778488 6.9620694206096223 5.3546663187444201 5.8799340650439254 255.94994206307456 137.18582754395902 29.415093356743455 35.274775737663731 49.750824584392831 17.147912291693505 3.5943870376795521 16.167905607027933 96.155217720894143 15.706495420774443 148.50224802433513 4.4768332480452955 6.6071615852415588 14.795094141736625\nleaf_count=1368 389 23 7889 397 6763 265 415 1491 1189 1482 210496 1283 610 39 240 10 702 1044 190 3589 1407 1440 6 1018 438 383 222 184 335 310\ninternal_value=9.66909e-05 -0.0270853 -0.0319121 0.0243631 0.01948 0.00144879 -0.00833886 -0.00347248 0.0248165 0.013256 -0.03511 0.0257062 0.0164959 -0.0357226 0.0213469 -0.0198908 0.0194847 0.0120284 -0.00179656 -0.0223759 -0.012946 0.000516744 -0.0263191 -0.0108756 0.0285636 -0.0053317 0.0309854 -0.00177862 0.0141249 -0.00528467\ninternal_weight=4449.4 2098.57 1937.94 2350.83 1651.49 377.444 277.843 196.099 1274.05 124.944 1741.84 1243.34 100.269 1706.53 93.0775 23.9669 477.862 221.912 84.7264 177.574 105.768 70.4931 20.7423 30.713 99.6008 35.307 160.627 12.1249 50.8173 21.4023\ninternal_count=245617 223587 223158 22030 21633 15173 14470 10872 6460 2983 212286 5053 1265 210535 850 1492 3864 3162 2118 13205 6442 2853 1446 1407 703 1751 429 207 835 645\nis_linear=0\nshrinkage=0.03\n\n\nTree=19\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 12 0 8 11 2 7 18 5 14 11 14 2 3 3 0 11 18 7 4 22 2 14 14 3 18 18 3 3 14\nsplit_gain=2205.75 825.009 262.791 257.936 257.224 117.759 112.928 90.623 81.4208 59.9027 57.0667 50.4491 49.702 48.5114 41.0462 37.5492 36.1463 69.0054 50.997 75.9942 48.6559 57.1622 28.6617 27.8856 26.0334 21.8681 19.9958 17.5912 16.837 17.0717\nthreshold=0.26534039634031065 0.27402411187416448 0.52258841453381233 1.0000000180025095e-35 0.66750229771629133 -25.182852745056149 1.0000000180025095e-35 2.524928003624769 3.7011718750000004 0.87268561280719448 0.47410876524022461 0.52010779807184393 -43.562467575073235 0.41780940659066851 0.20619759271271135 1.3771184689138856 3.8857805861880484e-16 2.249904835165133 0.89599609375000011 1.4160156250000002 1.0000000180025095e-35 -8.7471351623535138 0.6254473112327732 0.69550561433971103 0.40911314460322756 6.8062759569045816 0.89587973461402759 0.36709515926886871 0.53650715206675215 0.90649446645044673\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 4 7 8 -5 10 14 11 25 15 13 -3 -2 26 -4 17 28 19 24 -18 -22 -20 -10 -19 -7 -1 -24 29 -8\nright_child=2 12 6 5 -6 9 16 -9 23 -11 -12 -13 -14 -15 -16 -17 20 18 22 -21 21 -23 27 -25 -26 -27 -28 -29 -30 -31\nleaf_value=0.0094646983442538482 -0.028668330597358842 -0.034999311172237195 -0.027283704912479178 -0.036627434241787771 0.034940092581794834 0.025922227220796179 0.024511636983331555 -0.035676885207924461 0.024942005600697176 -0.026033611243403938 0.033225633871716016 -0.035104344732523864 0.032201458498829938 0.01382169529596637 0.012370067867151594 0.01472964150525527 0.034828531074051775 -0.020470082463225417 0.026311062773410165 -0.033928029572902411 -0.052756364823262729 0.02017568944291406 -0.035174633370821447 -0.0153057923137739 0.024697649223217943 -0.03514308965911072 -0.031308677440040796 0.0091940440262580653 0.032452051836531576 -0.020129280995201271\nleaf_weight=13.000166060868649 46.189343568868935 9.553284398512913 65.02210121951066 52.070407864637673 125.48557967529632 124.19458479713649 421.84935591858812 1469.9958657053066 48.626553111476824 25.639763579238206 27.760291032958776 145.57308454927988 232.03454351099208 48.714490125188604 49.772271696012467 26.22207201924175 587.0437139372807 13.629094159696249 176.09184492123313 45.585796674480662 10.867084073135628 51.542132888920605 8.258241893257944 21.726101359119639 55.064718074630946 4.6700070397928348 53.720882069435902 60.584637216525152 406.22243661992252 7.1456973883323363\nleaf_count=383 4648 1290 6063 6347 182 1318 1318 183241 695 1738 22 17187 821 1106 1205 473 589 1040 1477 4008 198 123 897 1740 488 443 4977 1400 503 252\ninternal_value=0.00042821 -0.0225431 0.0199103 -0.0295555 -0.00114721 0.00196892 0.0245893 -0.033659 -0.0157814 0.0151284 -0.00370634 -0.0240646 0.0294108 -0.00694134 -0.00810833 -0.015181 0.0264165 0.0232725 0.0122279 -0.004297 0.0321541 0.00684458 0.0199241 0.0124725 0.0155711 0.0234557 -0.0233418 0.00344425 0.0280022 0.023723\ninternal_weight=4433.86 2034.65 2399.2 1793.06 436.315 206.575 1962.89 1586.49 310.83 154.504 119.004 240.477 241.588 94.9038 116.493 91.2442 1843.88 1194.43 359.214 114.28 649.453 62.4092 244.935 70.3527 68.6938 128.865 66.721 68.8429 835.217 428.995\ninternal_count=246172 201763 44409 199652 25558 9846 18851 189806 25376 3499 6558 22941 2111 5754 6565 6536 12293 11383 9310 5536 910 321 3774 2435 1528 1761 5360 2297 2073 1570\nis_linear=0\nshrinkage=0.03\n\n\nTree=20\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 7 13 1 2 4 0 17 2 7 18 14 19 2 10 3 4 18 4 0 3 14 4 19 21 13 17 3 7\nsplit_gain=3033.13 612.819 213.968 114.575 126.282 95.0938 84.349 74.2747 72.0907 50.4157 38.0703 61.9985 43.8235 28.474 26.177 25.6511 24.7467 24.4257 18.5937 21.5966 18.5239 18.2289 26.1825 32.763 17.4517 18.901 15.807 26.4 23.6912 14.9984\nthreshold=1.4475910192202701 1.0000000180025095e-35 3.3496093750000004 0.14796868535883259 2.9477339485081675 -14.08206272125244 0.84667968750000011 1.4460303622695712 3.9281938047451082 -43.067142486572259 2.1386718750000004 3.1984648276080736 0.60840604842079971 246.50000000000003 -14.450672149658201 1.8688348091416842 0.53650715206675215 3.6123046875000004 0.89587973461402759 5.0736774206161508 1.6765212893675572 0.31840721315068282 0.46941259862495821 3.2580566406250004 233.50000000000003 2.5000000000000004 0.14796868535883259 0.41124616060114544 0.65908129556435835 4.5273437500000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 6 -4 -2 15 9 16 11 13 14 20 -13 21 -8 -9 -6 24 -11 -1 23 -23 25 -20 27 -3 -29 -7\nright_child=3 26 5 -5 18 29 8 17 -10 10 -12 12 -14 -15 -16 -17 -18 -19 19 -21 -22 22 -24 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.0356610456549558 -0.034789712726462314 0.030218068223832033 -0.031957385185437337 0.034951762016963581 0.033785901659215127 0.026293496213271185 -0.039928086713561732 -0.013807698328741539 -0.040208506811871408 -0.0104451367751745 0.029112980202100111 -0.028903750675026686 -0.03445033501605315 0.023365248596710294 0.013842832077865248 0.005390614446443248 0.020189633529662476 0.028191361504373551 -0.047986207951188027 0.035116771701841805 0.01881052319602353 -0.028919101022812876 -0.030843288621172207 0.029186964107283095 0.022519483945494589 0.013022142454078723 0.034590380152203989 -0.049594485295452143 0.029308278688939756 -0.036280376912905053\nleaf_weight=1576.6155051084934 34.739801114657894 23.635757699608803 38.425195920513943 645.4055612180382 276.11066363751888 85.844031716929749 21.18343022372574 31.193807052331977 21.538758475799114 43.237492705695331 155.1490806483198 15.145242783473803 35.341552852187306 248.97275142697617 63.390976108377799 14.252825374947859 7.8210965075995764 19.343651610193774 5.6417138339020338 133.79275342263281 33.825197217985988 17.030664720572531 125.30788788525388 15.937814724398775 495.17851914884523 13.7563478667289 127.50327081372961 8.3636703351512534 4.5757506899535647 2.7476174547336987\nleaf_count=201105 3228 72 4479 384 201 1012 1860 1929 1836 930 705 1135 3122 1265 1713 594 153 196 82 164 183 1914 14759 362 2085 138 125 264 4 115\ninternal_value=0.000100445 -0.0259086 -0.0306497 0.0242672 0.0199512 0.00693301 0.0101198 -0.0333046 0.0125865 0.01447 0.0163613 0.0118082 -0.00720142 0.0184496 0.00533965 -0.03434 -0.0235847 0.0025029 0.0271676 0.0243015 0.00247255 -0.0346693 -0.024629 -0.000796834 0.0214268 -0.00620409 0.0296129 0.0118023 -0.0215913 0.0239695\ninternal_weight=4341.01 2090.78 1926.7 2250.23 1604.83 127.017 680.345 1799.68 645.606 624.067 595.062 439.913 113.878 326.035 78.5362 1749.14 29.0045 50.5375 924.48 648.369 77.0627 1734.89 158.276 32.9685 514.577 19.3981 164.078 36.5752 12.9394 88.5916\ninternal_count=246114 226930 226465 19184 18800 5606 16130 220859 12902 11066 9053 8348 5970 2378 2848 218734 2013 2125 2670 2469 1113 218140 17035 2276 2305 220 465 340 268 1127\nis_linear=0\nshrinkage=0.03\n\n\nTree=21\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 7 13 1 2 0 7 0 3 10 14 4 4 7 2 2 1 4 0 2 13 22 1 7 19 3 0 19 6\nsplit_gain=2823.27 606.211 217.867 117.527 134.589 93.6738 74.1887 79.2193 62.5699 38.214 27.4155 21.4262 20.9011 19.6717 19.0491 18.2732 18.1104 17.4888 41.7354 29.5763 17.2007 15.6611 27.5045 15.4582 14.7219 16.0523 13.2473 13.1889 20.8778 31.5853\nthreshold=1.4475910192202701 1.0000000180025095e-35 3.3496093750000004 0.14796868535883259 2.9477339485081675 -13.633526802062987 0.94290057586474996 2.1386718750000004 1.5447487307370646 0.47260169401963864 1.7005986908310777 0.45315309081690569 1.9753417968750002 1.8393554687500002 1.0000000180025095e-35 -22.80348968505859 23.600493431091312 0.95935769349451683 3.2895507812500004 0.21782845461571179 -46.825880050659173 0.14796868535883259 1.0000000180025095e-35 -0.79535375801325292 0.89599609375000011 240.50000000000003 0.33966381556979741 1.5447487307370646 246.50000000000003 1.2426757812500002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 8 4 6 -4 7 9 10 11 17 -2 -11 -10 26 -9 -12 -1 -19 -20 -6 22 -3 -7 25 -22 -8 28 29 -16\nright_child=3 21 5 -5 20 23 14 15 13 12 16 -13 -14 -15 27 -17 -18 18 19 -21 24 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.035056863108898287 -0.012010973392187917 0.021532144309880211 -0.031761709447132205 0.034492581345803609 -0.0069829883003961956 -0.03531191868985166 -0.027737697165668047 -0.033071948740475518 -0.024903003555596891 -0.015554197336699823 -0.022933513507511043 -0.037863722800575948 0.017300081819306361 0.018048615287429369 -0.023878386128383857 0.025333687069225258 0.033794048426488366 -0.034105640551241871 -0.035118476808112295 0.016177494374143949 -0.0023616027228533839 0.033533122399076153 -0.045654584112379194 0.025931021927676687 0.030165034347467153 0.02373947121387146 0.010060421746811873 0.025756464336111354 0.01929234157994764 0.024367731097550142\nleaf_weight=1566.0266579948366 61.907933356240392 38.668157184496522 38.107996678678319 633.76129184290767 13.079523193184285 3.0277826618403187 12.488871841691433 4.3098620604723683 13.468744545243679 29.08845180994831 10.361159320920708 55.398531025042757 40.478764005703852 27.564111968269572 27.542693696450442 73.78513221279718 8.1456683836877328 108.4602476186119 13.704147962154819 31.422737984918058 22.653476326493546 129.81194816017523 5.4087918028235427 89.287803381448612 562.45870025409386 295.64175333781168 20.976374871563166 153.83015924645588 154.69470779644325 20.359344342257828\nleaf_count=202991 3710 165 4461 386 280 218 622 368 983 1825 804 5522 792 573 776 545 23 12989 1609 996 303 126 174 889 954 1130 522 461 930 51\ninternal_value=-0.000238387 -0.025211 -0.0300953 0.0236021 0.0191261 0.00728096 0.00877469 -0.00346839 -0.0328378 -0.0140598 -0.0337019 -0.0244369 0.00353683 0.003693 0.0170826 0.0217331 0.00248722 -0.0340926 -0.024097 0.000176283 0.0266918 0.0282927 0.0124576 0.0235655 0.0271992 0.0218802 -0.00444096 0.0191103 0.0139621 -0.00329382\ninternal_weight=4265.92 2083.47 1909.58 2182.46 1548.69 130.424 654.861 264.969 1779.15 186.874 1738.12 117.306 69.5672 41.0329 389.892 78.095 18.5068 1719.61 153.587 45.1269 893.833 173.889 44.0769 92.3156 880.754 318.295 33.4652 356.427 202.597 47.902\ninternal_count=246178 227001 226536 19177 18791 5568 16124 12762 220968 11849 219412 9232 2617 1556 3362 913 827 218585 15594 2605 2667 465 339 1107 2387 1433 1144 2218 1757 827\nis_linear=0\nshrinkage=0.03\n\n\nTree=22\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 9 1 11 18 2 5 8 1 4 17 0 2 7 19 11 22 7 19 11 11 4 18 17 2 2 1 18 2\nsplit_gain=2857.8 467.567 158.963 146.969 140.174 83.9243 69.61 50.0247 36.1102 31.091 32.5263 30.9022 31.4069 29.4642 29.216 23.6901 21.7666 37.9172 29.5027 28.6024 21.5684 20.8255 20.464 19.4415 22.491 18.5492 16.5281 16.4383 15.7757 15.0715\nthreshold=1.3399809614428995 0.49644476220754463 1.0000000180025095e-35 2.5144622974930591 3.8857805861880484e-16 2.249904835165133 -25.182852745056149 3.1982421875000004 1.0000000180025095e-35 0.95935769349451683 3.5791015625000004 3.1006363674515387 1.2661233611507157 9.8937015533447283 4.0683593750000009 251.50000000000003 0.24602455912239149 0.93750000000000011 1.0000000180025095e-35 253.50000000000003 0.12108388113305064 0.71017588656534381 0.84667968750000011 0.89587973461402759 0.068176441062887241 -10.988907814025877 -43.562467575073235 1.0240454605967615 2.1383330595080281 -30.0182991027832\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 4 5 22 -4 15 -8 14 23 16 -13 -12 -1 -7 18 -18 28 -19 -10 27 -2 -11 -25 -17 -6 -3 -5 -20\nright_child=3 21 6 11 26 7 8 -9 20 10 13 12 -14 -15 -16 25 17 19 29 -21 -22 -23 -24 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.034560805231617447 -0.03656980483427235 -0.015936084828571773 -0.035149653361890527 0.0176613550121596 -0.030241118465044528 -0.027723971205708818 -0.026528473258432241 0.015639856199396646 0.029118246034165039 0.02542216187782358 -0.027409513218657835 -0.042349061125115224 0.023325917445613641 0.025237156024358876 0.013531187544417591 -0.020330611253665844 0.034668526540007009 0.016483608386669499 0.014287230805987627 -0.051484708769339499 -0.035282227047231891 0.03153787330768712 0.017659584186172434 0.011698611141937664 -0.036180311670633118 0.0087498121974602088 0.028701497994762501 0.02486764636264405 -0.021941036908759329 0.027693107422975701\nleaf_weight=1552.6942710444564 5.5948019472416481 16.918876003241166 30.734839116921648 17.500355216208845 3.5807594931684426 108.96634013182484 13.782775097060947 46.886798458872363 69.232908517820761 5.0585612994618705 14.870087245246394 10.571190824965015 14.145992453210056 23.13530285586603 10.966169807594268 38.773419080302119 551.00283805746585 19.124941023066643 86.220149689121172 6.7082599480636409 4.1859514303505412 110.65910931117833 167.28592368820682 9.3926883290987444 83.264707848895341 38.357110769953579 265.5411589872092 16.67253669956699 16.735143037512898 882.59722303319722\nleaf_count=203392 358 643 4012 155 367 9254 1400 918 516 51 1526 605 64 414 269 3204 347 45 743 75 238 200 1072 456 10600 1450 573 90 792 2049\ninternal_value=-8.79426e-06 -0.026549 -0.0306765 0.0228304 0.0110628 -8.19014e-05 0.0029802 -0.0118128 0.0166655 -0.0330126 -0.0190095 0.0277706 -0.00531687 0.00446387 -0.0342151 -0.0187761 0.0282912 0.0330059 0.0255498 -0.00242706 0.0250217 0.0253039 0.0157044 -0.0280919 -0.0312537 -0.00594199 0.0278082 0.00444959 -0.00177055 0.026516\ninternal_weight=4241.16 1961.57 1817.32 2279.59 674.986 405.864 117.936 232.984 87.2016 1699.38 135.721 1604.61 24.7172 38.0054 1563.66 186.097 1579.89 576.836 1003.05 25.8332 73.4189 144.251 172.881 97.716 92.6574 77.1305 269.122 33.5914 34.2355 968.817\ninternal_count=245878 223807 222874 22071 17196 16256 6166 14826 2154 216708 13047 4875 669 1940 203661 13908 4206 467 3739 120 754 933 1430 11107 11056 4654 940 733 947 2792\nis_linear=0\nshrinkage=0.03\n\n\nTree=23\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 9 1 11 7 2 13 18 4 18 13 1 11 18 18 7 2 8 14 13 2 7 2 18 3 4 4 19 19\nsplit_gain=2721.67 476.799 161.543 123.586 117.486 72.4494 69.047 57.6019 48.3868 37.9745 32.7323 32.137 31.6509 30.3904 28.1503 27.9422 27.2486 25.1403 25.1395 22.3658 19.3826 24.994 17.3869 16.3599 14.6979 14.3347 29.5317 18.9535 13.8997 13.5958\nthreshold=1.4130033503388202 0.60365083802632757 1.0000000180025095e-35 2.5144622974930591 3.8857805861880484e-16 0.89599609375000011 -25.182852745056149 0.14796868535883259 2.1383330595080281 9.6361774206161517 4.2696729980286863 1.0000000180025095e-35 3.3033179733375269 0.12108388113305064 1.2424533248940002 1.4978661367769956 0.89599609375000011 28.11904335021973 1.0000000180025095e-35 0.49906193250331299 0.14796868535883259 -15.657813072204588 0.89599609375000011 -19.19599628448486 3.4811217321331038 0.31840721315068282 3.8417438268661503 3.8664550781250004 262.50000000000006 257.50000000000006\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 5 8 -4 9 28 15 -7 12 14 18 -5 22 29 -11 24 23 21 -6 -1 -12 -8 -17 -27 -28 -2 -16\nright_child=3 -3 6 11 20 10 13 -9 -10 17 19 -13 -14 -15 16 25 -18 -19 -20 -21 -22 -23 -24 -25 -26 26 27 -29 -30 -31\nleaf_value=-0.024671769792688578 0.010741092697050581 0.027622688721419139 -0.034900126592811317 0.027560934250165991 -0.037772247632632476 0.017705940133044541 0.019927259794648364 0.027570599335766971 -0.02715941727799848 -0.026860599995793006 -0.033172337737360856 0.03343064804720329 0.027563560946662357 -0.040237877586184426 -0.016317835769588552 -0.034570768091869208 0.017542433465863295 0.02986797038391804 0.029653340862383928 -0.036997927199256861 0.034082374778574731 0.020837032657611866 0.015304701354057716 0.016853803797352688 -0.03334592818326651 -0.032935169053056337 0.0383534105805085 -0.012328017522559234 -0.043109450862453359 0.011346052340733746\nleaf_weight=21.511217333143577 56.993895374238491 140.20452963281423 30.368916973238811 109.37919083889574 6.1498573205899438 187.33447657921351 7.3791783736087408 13.76870820391923 97.101868067285977 12.330082913395019 6.7064648857340243 541.74022233486176 740.47122660558671 6.8903086544014505 44.330269321682863 1485.7093764008023 106.69905484654009 13.892065907595677 68.109829102177173 16.618523363722488 159.47302079573274 90.055164200253785 16.434528789483011 24.06244882917963 9.6924857331905496 101.91930549533572 7.1458214069716623 35.053475953638554 3.6790353381074956 23.83682444691658\nleaf_count=1451 724 475 4029 155 463 1968 129 33 8966 1049 674 361 1711 362 1481 200027 851 25 518 1577 105 269 353 477 1192 13545 24 3046 94 294\ninternal_value=9.39155e-05 -0.0255478 -0.0296338 0.0229091 0.0118657 0.00154096 0.00362516 -0.0320191 -0.0141281 -0.0325194 0.0120886 0.0274683 0.0242866 0.0165708 0.0156832 -0.0330824 0.00813921 0.00318644 0.0215796 -0.0100252 0.0276477 0.0167404 -0.00740932 0.00524903 -0.0104903 -0.0336752 -0.0242678 -0.00305671 0.00682969 -0.00662008\ninternal_weight=4185.04 1970.41 1830.21 2214.63 648.175 392.497 122.441 1707.76 157.775 1694 234.722 1566.46 1024.72 92.0718 284.245 1667.77 174.866 26.2221 85.1815 47.3874 255.678 96.205 37.9457 30.7689 17.0717 1629.83 144.119 42.1993 60.6729 68.1671\ninternal_count=246428 226258 225783 20170 15317 14480 6230 219553 9784 219520 4696 4853 4492 2201 2781 218446 2626 1074 1839 2728 837 732 1804 1151 1321 216642 16615 3070 818 1775\nis_linear=0\nshrinkage=0.03\n\n\nTree=24\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 7 13 1 18 5 2 18 14 7 17 2 17 10 7 13 18 2 18 8 7 3 14 18 5 18 5 14 19\nsplit_gain=2532.74 520.025 180.919 145.447 142.043 96.8171 103.447 77.2918 47.9833 37.6566 33.5489 24.8444 26.6291 24.0267 23.6449 22.5986 20.264 19.305 17.3112 15.1974 22.0983 17.7946 14.7734 30.0516 15.6061 13.0624 12.2965 16.4528 17.0022 19.9618\nthreshold=1.3399809614428995 1.0000000180025095e-35 3.3496093750000004 0.14796868535883259 3.3033179733375269 3.0201273556387074 3.1982421875000004 -13.633526802062987 1.4978661367769956 0.51224307552198323 1.0000000180025095e-35 2.1810207941665634 -18.054428100585934 3.9281938047451082 1.8688348091416842 0.89599609375000011 -0.41325737289625047 1.8688348091416842 -22.378742218017575 3.0201273556387074 1.0000000180025095e-35 2.3505859375000004 0.53650715206675215 0.90427466785775545 0.89587973461402759 3.5976562500000004 0.89587973461402759 3.7255859375000004 0.90851014093827787 258.50000000000006\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 8 4 5 10 9 -4 15 11 17 12 16 22 19 -1 -7 -2 -8 20 21 -10 23 24 -12 -16 -6 28 -28 -30\nright_child=3 -3 7 -5 26 6 18 -9 14 -11 13 -13 -14 -15 25 -17 -18 -19 -20 -21 -22 -23 -24 -25 -26 -27 27 -29 29 -31\nleaf_value=-0.021618607578101579 0.0059199864532761774 0.027414663132134909 -0.030710531535276274 0.033812143635801817 0.032079527586954565 0.015424292728800276 -0.032158653681453757 0.020856140570405628 -0.030523555865173357 -0.031040389454640636 0.028950757514827164 -0.037047836293579932 0.0099954285043854386 -0.0397148884576978 0.025318564963521247 0.021135184518879812 -0.037170561675744257 -0.038642771246252892 0.023551961602793142 -0.034140984562239879 0.013559908516723731 0.01575781167124839 0.025925682427388505 -0.0319907354644514 0.012255353811362675 -0.0317046682399795 0.022075761791115586 0.034680018347309133 -0.04339456655491207 0.024268185855388882\nleaf_weight=21.675949755008332 22.000007508788258 156.53504537907429 36.278403754113242 664.57043343130499 236.41958739049733 8.5128828482702392 4.45760056329891 86.839319585589692 97.239643660723232 91.060076967114583 69.851454242132604 18.107358977664262 73.451863635564223 5.3731683637015513 9.3108784032519889 20.847964396234602 22.52606457832735 13.289234528085215 92.419473069021478 1472.5777208671207 12.967680568806825 7.4920225075911722 186.17047580878716 10.934604650363324 186.36766684427857 4.5894173870328805 419.78989382227883 100.15492420922965 7.0981739759445173 6.7087647374719381\nleaf_count=1418 411 455 4500 404 161 215 385 1060 11772 8405 113 1738 2264 250 161 365 2095 957 679 203560 246 215 504 357 1491 438 1315 78 189 11\ninternal_value=0.000349174 -0.0248649 -0.0295018 0.0220413 0.0170554 0.00822414 -0.00489297 0.00553627 -0.0321232 -0.0165334 0.0164632 -0.00559438 0.000155755 0.018632 -0.0329565 -0.000653932 -0.0227145 -0.0114639 0.0206862 -0.033297 -0.0225394 -0.0270951 0.0194095 0.0147888 0.0169018 0.00562714 0.0262578 0.023624 0.0209963 -0.0110988\ninternal_weight=4165.62 1926.35 1769.82 2239.26 1574.69 804.522 310.535 123.118 1646.7 213.658 493.987 122.598 104.491 458.697 1604.18 42.5239 31.0389 35.2892 96.8771 1590.28 117.699 104.732 453.324 267.154 256.219 13.9003 770.171 533.752 433.597 13.8069\ninternal_count=246212 224190 223735 22022 21618 19864 15781 5560 218175 14717 4083 6312 4574 2715 216392 1783 2310 1368 1064 215793 12233 11987 2465 1961 1604 599 1754 1593 1515 200\nis_linear=0\nshrinkage=0.03\n\n\nTree=25\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 16 13 1 2 0 5 14 17 14 0 13 0 5 4 2 9 13 2 8 13 17 22 13 14 0 5 19 12\nsplit_gain=2531.26 497.378 157.35 102.066 103.871 76.4527 64.8581 67.6196 43.7689 31.1885 27.966 26.4532 26.0661 24.8354 30.7706 21.1709 25.6106 18.8779 17.2896 16.6433 16.0769 22.6011 14.5615 14.5165 12.8598 20.0715 23.2125 11.9675 30.0525 19.9633\nthreshold=1.4475910192202701 0.49644476220754463 1.0000000180025095e-35 0.14796868535883259 2.9477339485081675 -25.182852745056149 0.70465417701170696 4.1191406250000009 0.87268561280719448 3.9281938047451082 0.52722272431259987 0.31303196667935901 -0.041888724317900632 0.88405423821904405 2.8623046875000004 0.84667968750000011 -43.067142486572259 1.0000000180025095e-35 0.14796868535883259 28.11904335021973 1.0000000180025095e-35 -0.28983840832261526 0.61464819824310435 0.93750000000000011 0.14796868535883259 0.70873219185162306 1.4460303622695712 3.4169921875000004 247.50000000000003 0.45050241529338036\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 13 4 6 -4 7 10 17 15 20 -6 -3 -1 18 -8 -17 19 -15 -7 21 22 -2 -5 25 -14 -27 28 -18 -29\nright_child=3 12 5 23 11 8 9 -9 -10 -11 -12 -13 24 14 -16 16 27 -19 -20 -21 -22 -23 -24 -25 -26 26 -28 29 -30 -31\nleaf_value=-0.034005935514985257 -0.034138703598461093 -0.034925967111351257 -0.034616148966933984 0.034352130877064922 -0.0098514480470905542 -0.017427761305403276 -0.012334609091033448 0.027440628015835071 -0.029044174407245493 -0.038312121146785748 -0.030494434680853451 0.026038317427210616 -0.044281857974268882 -0.028563164271084671 0.010321678778514014 -0.04034132639536072 0.0020265914789799576 0.026573616586866074 0.027920902985687248 0.018308594111957448 0.027025247425930328 -0.02291744020056171 0.01765330217917338 0.013904650157965156 0.032464560662923476 0.028303575635224721 -0.027926263836662038 0.034184647831867976 0.021466978640925654 -0.0025818765508989973\nleaf_weight=1409.6095245805336 4.8930253311991683 5.5943323697429141 49.626626345911063 550.75465848855674 18.408885624725372 21.320299980579875 21.252538155065849 43.44453464797698 22.056535026757047 8.6184641025029105 59.344339762814343 854.90504919900559 3.9404638926498583 136.22513500670902 23.097448810352944 5.8094908113125703 99.389434663462453 65.428009308874607 4.2950513251125804 23.724615374114364 11.934971611481158 44.368114224867895 28.697602144442499 35.087970376014709 110.11031139735132 40.263508384581655 7.0353393526747814 79.771275309845805 256.7281950104516 15.748346667736767\nleaf_count=199270 465 557 6723 298 462 1491 1209 208 1961 582 5686 2187 131 15346 778 300 1759 407 3 241 99 3167 745 79 105 159 46 147 1718 55\ninternal_value=2.84871e-05 -0.0249483 -0.0296549 0.0224748 0.0184286 -0.00323901 0.00961623 -0.00509932 0.00868164 0.0154332 -0.0146916 0.0252714 0.0244222 -0.0327095 -0.0214012 0.0164787 0.0178413 0.0164068 -0.0266516 0.001383 -0.00403845 -0.00907614 0.00934012 0.0331532 0.0266824 0.0138734 0.0195348 0.0186772 0.0160622 0.028159\ninternal_weight=4061.48 1922.33 1755.38 2139.16 1553.31 182.156 680 192.683 132.529 487.318 149.238 873.314 166.944 1573.23 163.618 478.699 457.447 110.473 140.52 45.0449 89.8937 77.9587 33.5906 585.843 161.35 51.2393 47.2988 451.637 356.118 95.5196\ninternal_count=246384 227218 226220 19166 18789 10823 16140 10370 4100 5770 10162 2649 998 215397 16127 5188 3979 2139 15349 1732 4476 4377 1210 377 441 336 205 3679 3477 202\nis_linear=0\nshrinkage=0.03\n\n\nTree=26\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 16 13 1 2 2 4 7 14 1 7 4 13 21 3 14 17 13 4 4 13 2 21 1 3 12 1 19 4\nsplit_gain=2441.71 413.565 129.933 118.37 130.956 71.9549 64.5702 45.1231 39.4617 36.616 28.3988 27.1757 38.4459 26.3743 32.4954 24.8245 34.3663 25.5663 22.4904 20.0169 19.6105 17.5943 17.567 15.9046 15.3087 15.129 13.7376 13.1001 13.0706 12.7914\nthreshold=1.3779212206518803 0.49644476220754463 1.0000000180025095e-35 0.14796868535883259 2.1468275719512682 -13.633526802062987 -18.054428100585934 10.818688511848451 1.0000000180025095e-35 0.84146630540292722 4.0016611463566001 0.89599609375000011 1.4160156250000002 -0.49896153351186062 2.5000000000000004 0.56033297298897644 0.58611746946415877 1.849927262362129 -0.041888724317900632 6.5971679687500009 0.84667968750000011 0.14796868535883259 8.3473243713378924 7.5000000000000009 2.9477339485081675 0.45898141303402712 0.063354284020036747 2.6981674883069502 260.50000000000006 1.6286621093750002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 21 4 6 -4 23 15 25 26 11 12 -10 -14 24 16 17 20 -3 -19 -8 -1 -17 -2 -15 28 -7 -27 -6 -13\nright_child=3 18 5 -5 8 9 7 -9 10 -11 -12 29 13 14 -16 22 -18 19 -20 -21 -22 -23 -24 -25 -26 27 -28 -29 -30 -31\nleaf_value=-0.032793489750828519 -0.033202577562349879 -0.033358228213056951 -0.029634150493717634 0.032936734973638269 -0.027091942467066078 0.019832427966894061 -0.034377215885344876 0.037265870902286363 0.01953860385619674 -0.033492998047580783 0.028654945368570373 -0.03719422612795005 0.017005436188977055 -0.027788908721082438 -0.048164552505701765 -0.021492889956531654 -0.018972198806284115 -0.040628843538062932 0.026261354559811646 0.019540259196490661 0.015753651178257277 0.022308874300568507 0.025831380140505469 0.0072339739879280782 0.01730602675518977 -0.032929171025795838 -0.034833956271442999 0.018765826990829577 0.024337662614215768 0.023406173488475762\nleaf_weight=1513.1071295419242 51.25414486345835 5.1186689247842869 64.147364680771716 584.78930484037846 22.727685384335931 87.594726548413746 6.5958630383247501 37.680891926400363 128.67106850037817 13.8170289772097 483.72720632655546 2.3024226389825335 24.846661010524258 9.5915474221110362 20.453702329425141 7.3910810190718612 71.838257207535207 15.130815417156557 134.02076018997468 6.4799736887216559 104.23567944485694 4.5858356654644004 51.152608074713498 10.132806670153512 17.808124504983425 4.2121497422922394 3.4503996646963051 29.961315630702302 4.6498344154097131 457.89309601928107\nleaf_count=213519 4849 535 7918 390 1242 1369 641 37 853 1307 664 67 312 442 721 393 3794 1157 418 32 1636 18 385 462 148 342 156 333 58 2262\ninternal_value=-0.00021981 -0.0257337 -0.0298353 0.0214137 0.0171 -0.00485167 0.00126613 0.00701118 0.021843 0.0104453 0.0230996 0.0190005 0.00973945 -0.00763375 -0.0205133 0.00254878 -0.00240839 0.00665086 0.0238457 -0.0225209 0.0124895 -0.0326124 0.0195961 -0.026519 0.00109742 -0.00153383 0.0174048 0.011639 -0.0178072 0.023023\ninternal_weight=3979.37 1825.84 1686.7 2153.53 1568.74 169.01 361.892 300.505 1206.84 104.862 1145.29 661.567 201.371 72.7 47.8534 262.824 204.281 132.442 139.139 21.6108 110.832 1517.69 58.5437 61.387 27.3997 61.551 91.0451 34.1735 27.3775 460.196\ninternal_count=246460 225240 224287 21220 20830 10750 13386 8075 7444 2832 5469 4805 2476 1623 1311 8038 7260 3466 953 1189 2277 213537 778 5311 590 1975 1525 675 1300 2329\nis_linear=0\nshrinkage=0.03\n\n\nTree=27\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 4 13 1 18 7 2 2 0 4 12 6 1 3 18 18 10 14 4 7 1 0 12 3 14 1 1 7 7\nsplit_gain=2230.62 461.674 185.814 126.584 132.115 85.0619 79.6168 49.5715 49.5442 33.9677 36.9341 32.0773 22.5337 29.2324 26.7464 21.7167 20.992 17.9367 19.8166 16.1185 20.3 16.6486 21.1567 15.9646 15.8026 21.1823 15.7666 14.0327 13.5748 12.7778\nthreshold=1.3399809614428995 1.0000000180025095e-35 8.8319251537323016 0.14796868535883259 2.9477339485081675 2.0126758453675748 0.89599609375000011 -33.804542541503899 -14.08206272125244 0.82186659671441242 4.9616699218750009 0.027767680665956421 4.1835937500000009 1.6109634896136267 0.62297257604742995 0.89587973461402759 1.4978661367769956 1.2424533248940002 0.0095533965833058416 4.6818847656250009 1.4980468750000002 4.0016611463566001 0.34392793676569194 0.49742347178520357 0.36709515926886871 0.35710832892811234 2.5144622974930591 0.60367420583093068 0.89599609375000011 0.89599609375000011\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 4 5 -2 11 -8 -4 17 16 26 13 14 -9 -6 28 -1 -19 20 21 22 -17 -3 25 -15 -7 -12 -11 -24\nright_child=3 23 8 -5 15 6 7 12 -10 10 27 -13 -14 24 -16 19 -18 18 -20 -21 -22 -23 29 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.0333025648834007 0.019163614211903414 -0.0097037287992486013 -0.026136207271757941 0.032667859204518222 0.031414702879682961 -0.029629314025262813 -0.031895013912233962 -0.034812453865230533 0.020602359392304058 -0.018911309958907947 -0.031913109083753596 0.02033292777636268 0.034357052863872722 0.013481378533241157 0.028084840985585825 -0.037607889819988055 -0.029644793544406797 0.033590919914323683 -0.031604586104454296 0.03090045358086169 -0.021085973471351757 0.024212264882437731 0.0053951451112288478 0.029649276672505012 0.019167461809762823 -0.029279026203588191 -0.0054863617700450082 0.02397860552343541 0.0182435369392752 0.020870951963270625\nleaf_weight=1355.5508708028356 278.18979253154248 9.495423518586902 26.083349779713899 597.52802852261811 260.85777164623141 104.76977998786606 24.164853535708971 22.135677461046729 82.506716141477227 19.958430512808263 4.0409721287433085 15.657886121654881 34.915201054885983 27.834896637592465 7.4133529430255285 6.2985956366173914 136.10230559774209 7.1227604858577234 7.6797526924638078 119.48942868923768 10.630198780447243 277.11249997233972 81.77445087605156 135.52308825356886 115.87337425886653 15.337035035248844 32.301467700395733 21.087539395317435 14.341483173193408 118.51161770382896\nleaf_count=198701 1432 274 3023 399 200 10279 2079 1450 747 1421 352 205 32 520 49 314 17588 16 952 169 124 636 887 179 1054 915 1070 273 304 347\ninternal_value=0.000254434 -0.0241867 -0.0286352 0.0209323 0.0163996 0.00645261 -0.0024044 0.00809393 0.0092397 -0.031266 -0.0193969 -0.019371 0.0125402 0.00834121 -0.0186298 0.0241006 -0.024392 -0.0329442 -9.29595e-05 0.0209462 0.0184826 0.0193939 0.012845 0.0270237 0.0134372 -0.0020284 -0.0240209 0.0141131 -0.0032994 0.0146106\ninternal_weight=3970.29 1819.49 1674.47 2150.8 1553.27 678.593 400.404 247.674 108.59 1565.88 195.531 152.729 223.51 188.594 29.549 874.675 170.402 1370.35 14.8025 613.817 494.327 483.697 206.585 145.019 159.045 43.1719 137.071 25.1285 34.2999 200.286\ninternal_count=245991 223830 223377 22161 21762 19085 17653 6099 3770 219607 19938 11554 4020 3988 1499 2677 19313 199669 968 2477 2308 2184 1548 453 2489 1435 11349 625 1725 1234\nis_linear=0\nshrinkage=0.03\n\n\nTree=28\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 12 4 1 12 18 5 18 12 18 7 1 4 12 12 3 3 19 7 0 10 2 2 3 0 1 4 12 5 10\nsplit_gain=2231.47 385.378 147.052 118.883 78.5633 56.1193 50.7508 52.533 38.8086 55.9081 55.2139 42.9006 29.6321 27.5515 27.3713 26.4962 21.3726 21.3265 19.689 27.7359 16.3522 16.2321 15.4244 14.1414 13.4829 12.4902 13.3843 11.4479 11.3732 11.2088\nthreshold=1.3399809614428995 0.49742347178520357 11.379394531250002 2.1468275719512682 2.2204460492503136e-16 2.4414009612931857 3.8603515625000004 2.1383330595080281 0.24517227696643781 2.249904835165133 0.89599609375000011 4.0016611463566001 4.9616699218750009 0.86223072954082747 0.026890617030159995 0.26802264518526847 0.48555671035752257 258.50000000000006 2.9707031250000004 0.29549219484135919 1.8688348091416842 34.417465209960945 -7.9326825141906729 0.19435047505660144 -0.7405159439372101 1.7622051334857016 1.6286621093750002 0.29026868663126176 2.7412109375000004 1.4978661367769956\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 4 6 12 7 23 9 -5 11 17 27 15 24 28 -9 -11 19 29 -7 -17 -8 -2 -4 -18 -27 -1 -3 -12\nright_child=3 13 14 8 -6 20 22 16 -10 10 18 -13 -14 -15 -16 21 25 -19 -20 -21 -22 -23 -24 -25 -26 26 -28 -29 -30 -31\nleaf_value=-0.020389732390200335 -0.036878777708499851 -0.034851698989261028 -0.028588444955816552 0.026005973464040651 0.02380312969494745 -0.032534432788130285 -0.032311082814880626 -0.024598682228715998 0.030786909690730668 -0.029836081406390578 -0.03962796562270246 0.017830207541785693 0.021210123246033552 0.032084752035019315 -0.036463310575077554 -0.046854204823018164 -0.02075437318502471 0.0039144823513609381 0.032895114610564449 0.017520835599015312 0.004337973998260948 0.014695061370957871 0.030248157509422596 0.013648330956598977 0.024814594915322907 -0.030993306318042542 0.021724268873571897 0.026156533330879448 0.02517582669550961 0.025853142377191558\nleaf_weight=73.844536329270341 4.3442890695296219 2.2719965823926058 3.7744182830210766 710.40254625631496 165.30849944218062 1450.3043111908482 3.0359279452823094 114.74009517231025 590.16450569569133 50.632026510545984 10.495159535203127 44.979631322436035 21.955217867856845 93.733583134133369 7.5563454148359588 10.532105681719257 15.08293433790095 24.698503356892616 71.443040343001485 154.75408192188479 10.683032117434776 5.2080312105827025 45.437673009233549 84.651701448485255 56.41236984799616 4.3838659715838757 22.71614720672369 4.2388549593742928 28.588895146036521 2.2161598959937683\nleaf_count=6607 151 136 478 1354 731 214355 173 10337 649 2924 644 283 199 75 644 515 1088 601 82 1148 558 29 180 784 255 495 346 16 141 3\ninternal_value=8.26117e-05 -0.0247846 -0.0288953 0.0208408 0.00732 -0.0307734 -0.00198492 -0.00753519 0.0245789 0.0211281 0.0114361 -0.00511148 -0.00892873 0.022792 0.0142004 0.00381904 -0.0179359 -0.018864 0.0197558 0.0139884 -0.032262 -0.0268278 0.0257561 0.0107995 0.0210531 0.000337192 0.0123969 -0.0175641 0.0198014 -0.0271587\ninternal_weight=3888.59 1769.1 1628.77 2119.49 459.701 1561.03 294.393 245.919 1659.79 1069.62 359.219 120.31 100.039 140.335 67.7431 46.601 156.923 75.3305 238.908 167.465 1460.99 15.7401 48.4736 88.996 60.1868 42.1829 27.1 78.0834 30.8609 12.7113\ninternal_count=245981 224008 223112 21973 14285 221735 13554 13201 7688 7039 5685 3808 6822 896 1377 821 12266 3525 1877 1795 214913 544 353 935 733 1929 841 6623 277 647\nis_linear=0\nshrinkage=0.03\n\n\nTree=29\nnum_leaves=31\nnum_cat=0\nsplit_feature=18 13 4 7 13 18 2 12 12 0 4 7 4 0 18 19 14 8 18 12 13 18 0 14 18 9 14 21 13 14\nsplit_gain=1317.46 737.291 232.031 120.261 69.4819 73.3198 69.364 62.4785 53.0428 52.1651 83.6916 50.007 46.1999 34.8462 32.0266 31.6338 26.3608 25.485 49.9154 24.9338 21.3399 25.4304 27.4809 20.0542 17.6139 17.3624 16.1936 14.8567 14.172 12.6029\nthreshold=3.3497501700808394 1.0000000180025095e-35 8.3627929687500018 1.0000000180025095e-35 1.0000000180025095e-35 1.7005986908310777 -25.182852745056149 0.30400123877355156 0.76965903234489019 0.52258841453381233 3.8073730468750004 0.89599609375000011 1.4160156250000002 1.8987378677022575 4.1820210059610323 238.50000000000003 0.84146630540292722 1.0000000180025095e-35 5.2284276452838947 0.43172124950159557 0.14796868535883259 3.9219243190762363 0.24908287076908558 0.56086626140189244 2.3502401828962087 1.0000000180025095e-35 0.88066063750466206 6.5000000000000009 -0.59662603815460169 0.87121128816919835\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 2 8 7 5 19 -4 13 9 17 14 12 27 -1 -11 -12 -8 -2 26 29 21 -3 -23 -16 28 -13 -19 -7 -14 -5\nright_child=1 20 6 4 -6 11 16 -9 -10 10 15 25 24 -15 23 -17 -18 18 -20 -21 -22 22 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.022409861456167066 -0.031569926554469115 -0.049647972499057294 -0.033134293518372264 0.025837863259522208 0.030890968036927719 0.013379871051981343 0.027232512276975544 0.030433881507604656 0.031800562481458398 -0.027738421783727308 -0.035540687853213418 0.012042978985153925 0.020650151446522377 0.018125680249038555 0.012581608245314806 0.025576642174133784 -0.023269605257387325 0.023735517238244897 -0.034367277131238237 -0.027213569170723054 0.032050975783823321 -0.041176487814368327 0.028022977460136048 -0.026260862076036923 -0.035912755654045746 0.028787652152178461 -0.028774200293524614 -0.021676301094620449 -0.019090671184883289 0.0047343481176089419\nleaf_weight=106.14883341430686 1148.0697841272922 4.9005624651908866 23.026533310418017 585.93467111885548 481.27392344875261 87.191366752143949 105.89203632518183 33.035741801373661 12.913862091954796 260.04838246654253 8.1376901546027494 280.59148610418197 10.952437228756024 22.431582637364045 32.140666325576603 47.570197964087129 9.4346596782561374 27.297623095568269 42.836808992899023 7.5931892856024197 238.50545603223145 5.2211245466023675 31.695608899928629 17.723819257342257 38.375013643875718 69.807974274037406 5.6076611946336916 11.498658805037847 24.148096691118553 27.14793904684484\nleaf_count=9906 166927 28 3442 1932 418 2439 755 47 43 32742 810 5331 193 279 1651 987 677 585 6020 74 185 264 65 2489 4621 320 641 487 1032 501\ninternal_value=9.76199e-05 -0.0164891 -0.0237612 0.0188644 0.0213186 0.0172655 0.0134337 -0.0057844 -0.0269791 -0.0274764 -0.0174521 0.00896034 -0.00426341 -0.0152529 -0.0235301 0.0161795 0.0229365 -0.0304545 -0.0133567 0.0242322 0.0285917 0.00857395 0.0174083 -0.00146941 -0.0221192 0.0154513 0.0142022 0.00911891 -0.0064618 0.0249126\ninternal_weight=3807.15 2021.02 1740.7 1786.13 1624.51 1143.24 138.353 161.616 1602.35 1589.43 365.621 522.565 172.166 128.58 309.913 55.7079 115.327 1223.81 75.7421 620.676 280.323 41.8173 36.9167 49.8645 73.4755 350.399 32.9053 98.69 35.1005 613.083\ninternal_count=245891 218311 217769 27580 17348 16930 4874 10232 212895 212852 38679 14423 8772 10185 36882 1797 1432 174173 7246 2507 542 357 329 4140 5846 5651 1226 2926 1225 2433\nis_linear=0\nshrinkage=0.03\n\n\nTree=30\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 4 13 1 18 5 8 18 7 1 0 3 4 0 14 17 18 13 1 19 3 14 17 3 4 0 14 18 14\nsplit_gain=1991.94 421.036 167.766 122.965 121.032 64.0786 93.8268 39.6671 44.0759 35.2898 33.1131 26.5362 26.0547 26.9179 24.2053 23.217 20.2386 19.9482 17.5593 21.8795 16.5486 15.8194 16.3218 15.4637 14.6823 14.3435 13.6595 12.9851 12.9213 12.7556\nthreshold=1.4130033503388202 1.0000000180025095e-35 11.379394531250002 0.14796868535883259 3.3033179733375269 2.3502401828962087 3.8603515625000004 1.0000000180025095e-35 5.1619206365690227 0.89599609375000011 1.7622051334857016 0.34392793676569194 0.31840721315068282 2.9518512487411503 0.29896707506516013 0.8755209476635436 3.325091732892195 6.3465127852423047 0.14796868535883259 0.18336881478606726 245.50000000000003 0.31840721315068282 0.2247238965590572 0.70050815440916869 0.0054559557070609941 2.7826617956161503 0.29896707506516013 0.11320561501600769 4.8242976514536702 0.90851014093827787\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 5 16 9 12 15 25 27 -12 -1 -14 -15 23 29 -4 19 -3 28 -13 -23 -9 -25 -7 -21 -11 -16 -2\nright_child=3 18 17 -5 -6 6 -8 8 -10 10 11 21 13 14 20 -17 -18 -19 -20 26 -22 22 -24 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.033001978433115753 0.018209794069885432 -0.045290183353151575 0.020426072467688416 0.032080600662560063 0.024563900715940213 -0.024816226029730128 0.025279056598138464 0.028100330725530479 -0.03177213089857657 0.0080890798508021621 -0.027789829716463516 -0.018446688402585211 -0.029379417099581091 -0.030197730332060913 -0.027164001897502062 -0.031453169740605906 -0.038547267775408248 -0.030958703431241689 0.031147309340053213 -0.037163323369114064 0.028483688282526964 -0.040638308052662199 0.01880159396640152 0.021938353763379659 -0.033077574272913378 0.010955963494069881 0.023825435148370645 -0.038081329028186832 0.034251062750403981 -0.0034575949163382193\nleaf_weight=1274.5841990616173 342.96497072698548 5.1830204764264627 68.264004684286192 562.40843694284558 687.75503475335427 117.10597367759328 86.873472456354648 29.200440430082381 54.399977708584629 6.6445426545105901 16.604925772175193 12.683683520881457 85.000195052241907 17.343441039207391 11.22121404937934 10.956431583501397 5.1550913122482589 6.6504874500678843 109.59509688429534 2.8596040848642614 21.049902230035514 3.4363526492379597 90.611957961227745 7.1309515507309689 8.5263009546324593 10.472953405464066 26.078965619206429 23.738439063541591 3.1903034236747763 26.01021280977875\nleaf_count=197764 1559 165 485 387 1739 10039 440 375 7648 217 1301 571 12346 2597 1062 1083 216 937 121 102 447 72 950 58 478 361 64 2041 58 346\ninternal_value=-0.000282606 -0.0237154 -0.0281412 0.0201971 0.0155045 0.00708899 -0.00179758 -0.030298 -0.0128071 -0.0102196 -0.000515298 0.00643822 -0.0316528 -0.0190264 -0.00212476 0.00591134 0.0158155 0.0155167 0.0253353 0.00631242 0.0120568 0.0119718 0.0162251 0.0155244 -0.00820827 -0.0218173 0.0167587 -0.0280482 -0.0122282 0.0166772\ninternal_weight=3733.7 1741.23 1597.52 1992.47 1430.06 742.303 368.172 1522.6 110.214 281.299 153.72 123.337 1412.39 137.805 52.8049 55.8141 374.13 74.9145 143.717 34.1216 35.4614 106.732 94.0483 44.8577 15.6573 127.579 28.9386 30.383 14.4115 368.975\ninternal_count=246029 225790 225338 20239 19852 18113 15992 223916 9642 15552 5152 2894 214274 16510 4164 1994 2121 1422 452 331 1567 1593 1022 911 536 10400 166 2258 1120 1905\nis_linear=0\nshrinkage=0.03\n\n\nTree=31\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 4 0 13 0 2 7 3 13 0 3 7 8 2 17 13 0 0 13 17 4 7 13 11 2 17 19 17 12\nsplit_gain=1370.93 642.158 204.858 148.253 200.912 91.0794 55.828 51.2154 49.3096 40.9439 36.3468 41.3654 28.0462 24.806 24.8821 24.4472 21.9499 19.9593 27.7773 19.8754 14.2214 13.9208 13.285 14.5975 16.2219 13.1956 12.4385 11.8566 11.7927 11.1066\nthreshold=0.28904431203898151 1.0000000180025095e-35 8.8319251537323016 0.75738199555027408 1.0000000180025095e-35 1.1495945197163422 -14.450672149658201 2.5244140625000004 0.47260169401963864 0.14796868535883259 1.5447487307370646 0.44825707527455844 3.1132812500000004 1.0000000180025095e-35 2.5338792800903325 2.4286772172182522 -0.44620633588241487 0.56276549863052994 0.57379665772197053 -0.64803833042221404 0.48975197669932524 0.84667968750000011 1.0000000180025095e-35 -0.46491054376387103 0.3376542910797411 -21.99824237823486 0.12494407617242172 250.50000000000003 0.68882141110442896 0.29026868663126176\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 4 7 13 -4 8 17 10 11 12 22 -1 -15 21 28 26 -19 -9 -16 -7 -5 -24 -25 -21 -2 -18 -10 -3\nright_child=3 29 6 9 -6 15 -8 19 16 -11 -12 -13 -14 14 20 -17 27 18 -20 25 -22 -23 23 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.03231273686705706 -0.00078732492084768429 -0.00071333573925032518 -0.026333774435881991 -0.026435928459266313 0.030084542997201931 -0.031999105151450415 0.023695486704982362 -0.0206659967350563 -0.00064375724493693522 0.03151066527432031 0.025920249964383673 0.019656031501623643 0.026933616248799332 -0.026284829085245282 0.025177693111251578 -0.030935412924247285 -0.0030043434874000192 0.026603276314701253 -0.02254361803235953 -0.028868093450550827 -0.019563368854220831 0.0054944762458255855 -0.023465475494888567 0.0072183664136891328 -0.045456599459228926 0.022100313681188824 -0.035157791200557606 0.021652816389779662 -0.035593839750241454 0.031024117749988533\nleaf_weight=1229.0209365124465 10.433943381533025 10.382419839559587 24.891287277685478 18.279187203734182 173.44071651436388 8.8463966897688788 89.633226056816056 13.201005141949279 16.957065042573959 341.90392482094467 461.40362308849581 441.79219358949922 33.252777011599392 52.979966577258892 16.696573106572036 24.343688747962005 31.733881333144382 13.244503832189364 37.341671182890423 4.0825456827878943 9.1122750776121375 93.257676599081606 18.704576882650144 103.22945822984912 4.5499389579053959 61.849922199267894 107.51124788168818 37.683124200906605 16.900595472659916 203.82664137519896\nleaf_count=191876 737 192 2959 1792 202 985 564 1300 927 254 1050 2716 107 6766 211 2223 1306 316 3804 503 529 2867 1283 1779 87 699 15325 623 2037 272\ninternal_value=6.39455e-05 -0.0190961 -0.0258307 0.0174097 0.00376468 -0.028913 0.0126881 -0.0092916 -0.0154648 0.0224314 0.0195338 0.0147455 0.00251707 -0.0312623 -0.0146778 -0.00456027 0.000868321 -0.0254196 -0.00940968 0.0118403 0.00924056 0.00196715 -0.00323782 0.000295785 0.00459724 0.0185602 -0.0321494 0.0104982 -0.0186103 0.0294897\ninternal_weight=3710.49 1762.99 1548.78 1947.5 524.38 1434.26 114.525 350.94 271.806 1423.12 1081.21 619.808 178.016 1307.81 78.7888 126.448 103.275 168.531 50.5862 79.1335 25.8088 102.104 144.763 126.484 107.779 65.9325 117.945 69.417 33.8577 214.209\ninternal_count=246291 209444 208980 36847 27779 205457 3523 27577 25075 9068 8814 7764 5048 199382 7506 6075 4893 20182 4120 2502 740 3852 4941 3149 1866 1202 16062 1929 2964 464\nis_linear=0\nshrinkage=0.03\n\n\nTree=32\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 8 13 1 18 18 5 18 14 4 18 1 3 13 3 22 18 5 3 14 3 3 17 4 9 14 11 4 11\nsplit_gain=1889.29 415.545 120.607 111.178 111.682 57.7316 55.0197 48.5773 31.6467 30.0973 26.2875 25.3463 21.9618 20.9849 20.5396 17.2697 16.597 15.7005 20.8505 20.3011 15.5369 15.3878 15.3284 15.2707 13.9644 13.3626 14.4002 23.4611 12.7844 11.5289\nthreshold=1.4130033503388202 0.49644476220754463 1.0000000180025095e-35 0.14796868535883259 2.9477339485081675 5.6716020950760067 2.0126758453675748 4.2558593750000009 2.4414009612931857 0.87268561280719448 0.84667968750000011 4.297262267176281 0.95935769349451683 0.10824180149310623 -0.041888724317900632 0.44825707527455844 0.93750000000000011 2.249904835165133 3.7255859375000004 0.30295287721642777 0.83758563423804033 0.0037166969630965979 0.19914108919933238 1.1019619337295856 1.9511718750000002 1.0000000180025095e-35 0.57401229820475252 0.81454645780468748 4.9616699218750009 0.12799080675092075\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 8 4 6 9 -2 10 12 28 -8 22 -1 23 -3 -15 -5 -6 19 -19 -21 -7 -12 -13 -22 26 -24 -28 -4 -30\nright_child=3 14 5 16 17 21 7 -9 -10 -11 11 13 -14 15 -16 -17 -18 18 -20 20 24 -23 25 -25 -26 -27 27 -29 29 -31\nleaf_value=-0.023190681072209562 0.016382294891861049 -0.031836929167966592 -0.01179381572308528 0.032940984917117389 0.026032888352385854 0.0089084650704746998 -0.034249006855328855 0.023396370952198382 -0.031998735110937461 -0.026242127159132655 -0.033318890570046028 0.03418112277522075 0.0091380716956226214 -0.035158025633013512 0.025802353530001536 -0.0043791446285326484 0.01097086179608631 -0.026608769407198841 0.031747342599171773 0.017533770543832577 0.0069427328826312702 -0.032809762433150272 0.009618959538777944 -0.031565789429726591 -0.042018466013246672 0.032629552519182432 -0.016516336025237863 0.027457383790657441 0.023620709787894385 -0.031815175610586839\nleaf_weight=58.1535421902081 269.00664306664839 4.9608603632077566 11.517869824194348 515.85138026066124 541.99449998931959 8.9266699138097447 26.821511439513415 57.62848401023075 1274.2747736866586 16.508882680209354 9.1637801756151003 10.053610751405357 27.141730918549001 43.745390644180588 148.84477266459726 26.406203637598082 34.262649485841393 10.681055359309537 63.644044758751988 163.15808066749014 15.870109755545856 51.920517152757384 96.119698708644137 3.5727307694032779 6.6709174616262308 13.072799133602528 56.38782754377462 12.538353584706782 73.737219893082511 2.6959905810654154\nleaf_count=5930 1302 549 721 324 774 416 3217 262 207229 1519 764 43 931 4793 435 1405 73 431 69 1038 195 7317 2081 315 158 37 2717 22 698 125\ninternal_value=0.000265664 -0.0231308 -0.0278719 0.020137 0.0156994 -0.00365122 0.00617265 -0.00157704 -0.030812 0.00986776 -0.00647054 -0.00360607 -0.0129167 -0.0172178 0.0237496 -0.0237967 0.031595 0.0231007 0.0169125 0.0119683 0.0143178 -0.0266444 0.00254206 0.0159632 -0.00871651 0.00456273 0.00215652 -0.00823882 0.0168646 0.0212797\ninternal_weight=3655.33 1678.68 1524.88 1976.65 1426.54 165.307 624.517 355.51 1359.57 104.46 297.882 271.06 85.2953 83.7779 153.806 70.1516 550.114 802.019 260.024 196.38 185.699 60.8472 187.282 13.6263 22.541 178.119 165.046 68.9262 87.9511 76.4332\ninternal_count=245890 225870 224886 20020 19623 10796 16958 15656 214090 3063 15394 12177 6861 6556 984 6198 397 2665 1891 1822 1391 7733 5621 358 353 4857 4820 2739 1544 823\nis_linear=0\nshrinkage=0.03\n\n\nTree=33\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 8 13 1 2 18 8 14 7 14 22 2 7 0 2 5 13 14 18 18 17 3 13 1 17 9 2 7 3\nsplit_gain=1743.04 413.419 151.115 108.777 107.788 69.3583 48.642 54.2873 35.2883 28.2372 26.153 23.6208 21.5191 20.4136 20.385 20.2285 19.4358 19.3242 17.0562 18.5947 16.8518 16.6476 15.5345 13.5025 13.2857 12.6823 12.5215 27.3631 11.7572 11.6129\nthreshold=1.4475910192202701 0.49644476220754463 1.0000000180025095e-35 0.14796868535883259 2.9477339485081675 -13.633526802062987 2.1383330595080281 1.0000000180025095e-35 0.67646516896599407 0.89599609375000011 0.85380319876708544 0.93750000000000011 -33.804542541503899 1.0000000180025095e-35 0.83245446090221253 -1.7340650558471677 2.8623046875000004 -0.041888724317900632 0.90427466785775545 1.7005986908310777 2.3502401828962087 3.0352616308921099 0.45898141303402712 0.14796868535883259 0.60367420583093068 1.1891760317644442 1.0000000180025095e-35 23.600493431091312 4.0156250000000009 0.47260169401963864\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 14 4 6 -4 18 9 -9 20 26 -5 -11 22 -1 -13 23 -3 29 -20 25 -14 -6 -16 -18 -8 27 -7 -28 -2\nright_child=3 17 5 11 13 10 7 8 -10 12 -12 15 21 -15 16 -17 24 -19 19 -21 -22 -23 -24 -25 -26 -27 28 -29 -30 -31\nleaf_value=-0.032509283913187623 0.010194126376766031 -0.031346807448480805 -0.028196071324915157 0.032730846782210658 -0.037003294569997715 -0.028897770845013476 -0.034090587127945521 0.030833718834249957 -0.036088534708175128 -0.027881871957445025 -0.025285678915630952 -0.034054707315101414 0.0078264835849129466 0.023385763023087602 -0.026401094643707705 0.020326015099367704 -0.031882976703247294 0.026017731200741127 -0.04546565207831868 0.013040024794775873 -0.024300495937591426 -0.029141546708509464 0.010993233347214859 0.0259996602856656 0.019759907991654526 0.025097515503714841 0.031264513248442392 0.020684264400971084 -0.0092987955196432784 0.023258027512679181\nleaf_weight=1160.7964487639838 123.54296860657632 4.6679363476578137 56.646575971797574 484.70598324388266 7.4569919910281923 13.663460409035904 3.5006722260732195 54.944277135655284 7.2679348997771731 20.40268459264189 13.962300056591628 7.9023072347044971 120.88493324676529 733.15398435737006 141.55817526835017 19.004714019596577 5.1796468396205482 147.86409717192873 7.6526504261419204 10.481627274304627 96.701349760987796 11.06487252935767 20.530032877111807 3.8059809729456893 15.503060094895771 11.137551601743324 50.990629027364776 30.721095342654735 6.884674593107774 126.07815325586125\nleaf_count=195899 860 538 7949 311 349 1077 170 219 399 1745 1313 47 2321 2085 18260 26 573 427 147 109 9007 697 225 3 311 103 175 280 93 345\ninternal_value=-0.000431402 -0.0228657 -0.0276522 0.0194302 0.0149603 -0.00126077 0.0053656 -0.00241909 0.0225683 -0.0083773 0.011996 0.0312146 3.64811e-05 0.022427 -0.0310871 0.00370698 -0.0210101 0.0240695 0.0148146 -0.0127351 -0.0197882 0.00453059 -0.00263055 -0.0248609 0.00595283 0.00966809 0.0172819 0.00512038 0.0263632 0.0168589\ninternal_weight=3518.66 1652.24 1499.71 1866.41 1354.8 172.869 593.66 325.904 62.2122 263.692 116.222 511.613 152.352 761.141 1326.84 26.907 166.047 152.532 267.755 18.1343 111.34 131.95 27.987 145.364 20.6827 14.6382 102.26 44.3846 57.8753 249.621\ninternal_count=246063 226898 225933 19165 18781 10887 16122 14661 618 14043 2938 384 4763 2659 215046 73 19147 965 1461 256 9280 3018 574 18263 884 273 1625 1357 268 1205\nis_linear=0\nshrinkage=0.03\n\n\nTree=34\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 4 13 1 5 7 18 18 0 4 2 14 1 18 4 9 18 17 10 14 16 0 5 0 19 14 4 7 19\nsplit_gain=1693.06 344.462 150.124 114.789 112.129 84.8427 41.6904 39.9375 38.8479 34.2172 42.0747 29.855 25.8033 24.0649 23.1592 20.1335 18.9577 23.7378 18.1356 14.6326 16.5917 13.6073 13.5178 13.2486 24.1914 15.1426 18.4084 14.9998 11.3834 17.6619\nthreshold=1.3399809614428995 1.0000000180025095e-35 8.9774329662323016 1.0000000180025095e-35 2.1468275719512682 4.0332031250000009 1.0000000180025095e-35 2.1383330595080281 6.3465127852423047 0.82186659671441242 4.4806598424911508 -34.252307891845696 0.11320561501600769 4.0016611463566001 2.9174053685313028 1.7380371093750002 1.0000000180025095e-35 1.4978661367769956 1.5185086528259144 1.2424533248940002 0.016582161273737187 1.0000000180025095e-35 0.34392793676569194 3.1982421875000004 0.40053340798327824 258.50000000000006 0.82532919407277039 2.4204101562500004 4.2558593750000009 205.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 4 5 7 14 15 28 19 21 22 18 16 -6 -2 17 -13 -9 -1 -21 -11 -8 24 -19 26 -26 -27 29 -4\nright_child=3 -3 8 -5 6 -7 11 12 -10 10 -12 13 -14 -15 -16 -17 -18 23 -20 20 -22 -23 -24 -25 25 27 -28 -29 -30 -31\nleaf_value=-0.031905158516540369 -0.014520834652369868 0.024559316822572331 -0.034012592726983898 0.029923367423005481 0.0066754775505883501 0.02581179661327946 -0.033426513039806316 0.021163383081996313 -0.032227021345329794 -0.025861097043898641 0.014680728633195672 0.023014608735364178 -0.025228381357522166 0.027329088550934747 -0.03527083024338877 0.015797370746345296 0.035909131537778687 -0.035113378669190787 -0.032114474940970028 0.031648236044697839 -0.030052692737684329 0.014211422116631086 0.0062406492495098486 0.022655821776447909 0.0058803374124928188 0.023003240319275954 -0.03133502650642539 -0.025827629752641967 -0.0084858869194447094 0.025610543903043764\nleaf_weight=1160.3370783075225 28.528364623198286 125.68422625405947 3.9454277954064301 591.81091726198792 35.15352767938748 76.1378963959869 7.7972305463626972 20.09855625755154 16.453188066836447 143.16563942551147 30.054920702474192 208.0027071791701 125.66608280286891 393.01246805337723 16.684096640441567 58.636772020719945 44.746840809471905 11.257239666301755 6.9345098561607292 6.4931667479686421 7.2314911707653664 7.4440849130041888 62.939975833985955 52.403367589693516 107.9985585808754 67.237569289514795 12.439923284575341 5.3537582913413635 12.730448323301969 65.511440300848335\nleaf_count=198851 592 437 112 562 575 377 603 418 2597 18979 796 348 11771 536 1377 344 59 877 680 11 1012 243 721 199 1228 323 453 51 247 566\ninternal_value=9.6779e-05 -0.0229409 -0.0270627 0.0189197 0.014045 -0.00157564 0.0188575 -0.01034 0.00859297 -0.0296632 -0.0173527 0.0202535 -0.0195439 0.021723 -0.00723709 0.00582234 0.0173531 0.0154931 0.0068385 -0.0315412 -0.000694954 -0.0237998 0.0014632 0.0093325 0.00582111 0.00837702 0.00181004 0.0190998 0.0170539 0.021789\ninternal_weight=3511.89 1579.05 1453.37 1932.84 1341.03 316.002 1025.03 239.864 98.6405 1354.73 180.665 973.19 152.699 902.452 51.8376 87.1651 509.44 464.693 27.0331 1174.06 13.7247 150.61 70.7372 256.69 204.287 193.03 120.438 72.5913 82.1873 69.4569\ninternal_count=245945 223851 223414 22094 21532 14182 7350 13805 3522 219892 20018 5398 12869 4074 1952 936 3538 3479 1098 199874 1023 19222 1324 3131 2932 2055 1681 374 925 678\nis_linear=0\nshrinkage=0.03\n\n\nTree=35\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 13 2 18 5 11 14 11 0 11 0 18 14 2 18 17 9 19 22 17 18 5 5 13 0 17 18\nsplit_gain=1644.35 347.502 112.442 118.453 112.2 85.6089 52.91 59.1904 26.3537 32.8619 24.0434 23.8651 19.2182 19.2541 18.6791 17.9351 16.9479 16.9289 16.9078 16.6389 19.4486 15.2016 15.0741 15.0556 22.1575 14.9277 15.8963 14.2397 12.1333 11.9659\nthreshold=1.3779212206518803 1.0000000180025095e-35 0.14796868535883259 2.9477339485081675 0.14796868535883259 -30.420168876647946 2.9174053685313028 3.1982421875000004 0.60365083802632757 0.87268561280719448 0.75683518236239655 1.8987378677022575 0.11347059467310934 -0.7405159439372101 1.2424533248940002 0.84977918009475439 -10.615592002868651 4.9593358439305844 1.2428969707451916 1.0000000180025095e-35 217.50000000000003 0.93750000000000011 0.28904399529622732 1.7005986908310777 3.7255859375000004 4.2695312500000009 1.0000000180025095e-35 0.29549219484135919 2.4945511211662157 4.2555875595453383\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 6 10 -3 14 16 9 12 11 -1 13 -7 15 -2 -8 29 -19 20 -16 -4 -17 -5 27 26 28 -25 -22 -18\nright_child=2 5 21 23 -6 8 7 -9 -10 -11 -12 -13 -14 -15 19 22 17 18 -20 -21 25 -23 -24 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.031532287520470471 0.023803110725802403 -0.032444173804930129 0.032414687539479042 0.026017088224220629 0.027903941927847078 -0.031615844749387972 -0.026748613882158156 0.015383462451833408 0.0285683858770482 -0.027349147374428184 0.0097225779315859139 0.0081463962460763002 -0.037991790031995661 0.02018211023337747 -0.03930730389290446 0.027369311045085984 -0.0078583794277804521 0.02470524086907443 -0.03190416202528594 0.031139366168053031 0.0017550170354321897 0.012466766957710413 -0.04462184019149127 -0.016917978976344842 0.031575192883103176 0.030198753981561636 0.029602036613804145 0.014872175365720388 -0.037764238609218015 -0.034843347662110329\nleaf_weight=1236.3816390577704 123.73370801378042 37.288033219403587 461.62431099172682 417.03995553497225 29.384360554628074 6.0376617605797938 71.501008123042993 80.005620502633974 89.907985120080411 19.741324746632017 12.697694638743995 13.422422935953362 5.1599677968770257 96.410631476319395 7.9973746039904645 3.3946130797266942 60.57301548589021 20.988658820744607 5.1368773260619482 21.392250095494092 152.56155684171245 38.951550219208002 5.8361823353916407 12.51828743657097 77.448193991556764 19.280597049742937 17.134948222432286 261.92667049641022 6.3256049719639114 18.971596599090844\nleaf_count=212954 302 6305 311 506 53 677 7273 1001 172 2101 50 432 243 1462 277 11 3365 439 516 42 2236 83 92 422 82 29 22 1646 339 2148\ninternal_value=-0.000101598 -0.02302 0.0187092 0.0142835 -0.0293289 0.00898875 0.0044567 -0.00590856 0.016216 0.00734061 -0.030682 -0.0311 0.0138979 0.0168534 0.0119056 0.0207838 -0.0155558 -0.00780254 0.0128087 0.0065831 0.0038594 0.0308904 -0.0180597 0.022125 0.0174518 0.00581825 0.00299222 0.013366 -4.46737e-05 -0.0146471\ninternal_weight=3430.77 1546.43 1884.34 1383.77 1291.89 254.546 614.834 257.177 217.258 127.35 1262.5 1249.8 107.608 102.448 357.657 132.965 177.171 105.67 26.1255 224.692 203.3 500.576 9.2308 768.933 351.893 195.303 176.022 274.445 158.887 79.5446\ninternal_count=245591 224449 21142 20748 213489 10960 18092 14742 4655 4483 213436 213386 2382 2139 3350 405 13741 6468 955 2945 2903 394 103 2656 2150 2626 2597 2068 2575 5513\nis_linear=0\nshrinkage=0.03\n\n\nTree=36\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 7 1 11 7 18 18 18 13 2 4 17 7 10 22 4 14 18 4 7 19 17 21 21 11 2 21 0 2\nsplit_gain=1632.91 292.443 127.849 102.735 78.575 64.3437 60.0103 42.2172 32.663 32.0944 31.5348 30.2832 27.5041 22.7563 21.9518 20.7511 37.1081 18.9428 22.6092 17.6855 16.6964 16.5514 15.2925 15.66 13.829 17.1623 13.3375 13.0107 12.4087 11.7276\nthreshold=1.3399809614428995 0.71017588656534381 3.5976562500000004 2.5144622974930591 3.8857805861880484e-16 0.89599609375000011 2.4414009612931857 1.2424533248940002 6.3465127852423047 0.14796868535883259 -33.804542541503899 4.9616699218750009 4.2643260401066021 1.0000000180025095e-35 1.7005986908310777 1.0000000180025095e-35 5.8393554687500009 0.5668027336126189 4.4829865921051741 2.7826617956161503 3.8603515625000004 242.50000000000003 3.9281938047451082 2.5000000000000004 2.5000000000000004 0.74021522701736098 -33.804542541503899 5.5000000000000009 0.48517496193768689 23.600493431091312\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 4 5 7 11 -2 20 12 -7 -1 13 -5 -8 16 21 -15 24 -9 -4 -12 23 -22 25 -19 -6 -26 -17 -16\nright_child=3 -3 8 9 26 10 14 19 -10 -11 15 -13 -14 17 29 28 -18 18 -20 -21 22 -23 -24 -25 27 -27 -28 -29 -30 -31\nleaf_value=-0.016463485845539482 0.014968028972915279 0.027184987010211128 0.030191385735621041 -0.00081288076809046112 -0.011861559771841459 -0.028586388724478648 -0.031495187823730619 -0.02608602518517366 -0.031897249227848437 0.030894746865435616 -0.035080576939667006 0.020935895700234679 -0.025367989824115873 0.024812550272221155 -0.018800036235096792 -0.029125732541765797 0.022685322322824785 -0.011226277243404912 -0.03420597113175506 0.011278741681296483 -0.0410989402955515 -0.0012675566540264356 0.026257194269843132 0.0079614481342474477 0.024933094017982719 0.02904942914713091 0.025216904656528583 0.0093596664853591912 0.025176249170063836 0.030884424490939692\nleaf_weight=71.729159202193841 34.026337046641856 95.367577804950997 37.29501600866206 41.615811377065256 8.5933654796099272 16.488379383110441 1201.0136705162004 112.26838341832627 14.193763191113247 361.40334177389741 17.057689978275448 25.672664122190326 10.935419917455873 609.75957747519715 8.2453431424219144 3.1336955263977861 54.801205317024142 30.32614899543114 7.212940325960516 12.118512482498771 9.2995619187131542 47.073381606023759 13.012515214039011 12.962857497623189 143.97720427345484 12.68186270073056 189.20367036061361 75.015096243470907 98.404600069159642 6.9810269353911272\nleaf_count=6664 216 211 286 1027 455 1591 213156 10382 2440 279 971 191 497 1556 832 303 402 429 414 393 193 1537 105 237 367 12 456 336 533 21\ninternal_value=-3.50343e-05 -0.023435 -0.0268991 0.0185249 0.00816911 0.00043212 -0.0292277 -0.014318 0.0082874 0.023266 0.0103117 -0.00649337 0.0202763 0.0208454 -0.0310415 0.0133367 0.00476484 0.0218706 0.0151401 -0.0223848 0.0164737 -0.0106617 0.00153805 -0.0134181 0.0166235 0.00104013 0.0235545 0.0196653 0.0232327 0.00447892\ninternal_weight=3381.87 1495.77 1400.41 1886.1 593.169 395.372 1313.64 158.413 86.7637 1292.93 236.959 97.4018 931.524 920.589 1216.24 220.471 118.932 878.973 269.213 124.387 72.57 64.1311 35.2749 22.2624 262 43.008 197.797 218.992 101.538 15.2264\ninternal_count=246492 224336 224125 22156 17239 16328 220864 10991 3261 4917 5337 6855 4638 4141 214009 3746 2910 3114 1558 10775 821 2508 535 430 1144 441 911 703 836 853\nis_linear=0\nshrinkage=0.03\n\n\nTree=37\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 8 13 1 4 2 18 5 18 14 3 14 17 13 1 2 3 4 1 14 3 19 18 5 3 3 19 2 13\nsplit_gain=1485.71 338.993 134.36 100.658 85.9619 68.2575 59.2237 48.4244 47.5785 34.552 32.104 23.176 44.4249 18.1509 17.7289 16.2754 16.0421 15.9689 24.3607 15.6769 19.0805 13.6817 13.6301 13.6217 20.4334 16.9692 12.5316 11.1962 10.7905 10.7711\nthreshold=1.4475910192202701 0.49644476220754463 1.0000000180025095e-35 0.14796868535883259 3.3033179733375269 0.84667968750000011 -25.182852745056149 2.1383330595080281 3.3496093750000004 2.4414009612931857 0.87268561280719448 0.58795853567030554 0.53481096685124585 2.5789308873416652 -0.041888724317900632 1.9634439907166827 -10.615592002868651 0.31840721315068282 3.8417438268661503 0.95935769349451683 0.29138017841767527 0.3261289997719386 241.50000000000003 1.2424533248940002 3.7255859375000004 0.2352687277455576 0.40911314460322756 29.000000000000004 -46.825880050659173 -0.45698050410493757\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 4 5 -2 -4 -7 11 19 22 12 13 16 -3 -13 26 -11 -19 29 -21 -20 27 -6 25 -25 -9 -8 -27 -1\nright_child=3 14 6 -5 23 7 10 8 -10 17 -12 15 -14 -15 -16 -17 -18 18 21 20 -22 -23 -24 24 -26 28 -28 -29 -30 -31\nleaf_value=-0.00022269223441592957 -0.035136672831625533 -0.030899872398868371 -0.032320448580906463 0.030401903974577246 0.027237871971322349 0.017571434407449397 -0.021441915625716606 -0.027960539677995073 0.018790843243726162 -0.032156409186393904 -0.026637738558444413 -0.035003532966043992 -0.029916766827049433 -0.033925962058682868 0.024124310270272696 0.020204793032996049 0.011092337414560314 -0.028620270436257936 0.031389110758942101 0.026101229934880857 -0.023003525945020052 -0.010364452796656962 -0.030793612856779437 -0.038724721007383958 0.031944718342227139 -0.015613379971529402 0.0061393826919153077 0.020609440508939324 0.016498558738649964 -0.028618500098592986\nleaf_weight=17.322255664505064 32.198066451295745 4.6504430621862403 39.091598853934556 477.43982827477157 277.77807757072151 307.82195959193632 5.2641453295946112 20.124246507068165 96.67385343555361 1028.5822678292752 17.849527619080618 4.5498587335459879 67.519740379764698 11.859474273864178 138.73739343159832 38.494176617590711 80.634260657476261 76.578945113986265 9.297661291901024 17.375875660916794 10.684828726109115 21.531342617701739 4.4854364413768044 4.3820413414505301 76.832439405843616 9.0014901505783183 17.380043831421062 96.143841637996957 271.58127788826823 38.412382345530204\nleaf_count=1380 3443 548 6776 394 256 1263 67 1692 803 192435 1972 319 5801 1186 422 399 1677 13049 230 173 827 2392 413 150 75 160 419 1705 1120 4480\ninternal_value=0.000101271 -0.0216515 -0.0262055 0.0186035 0.014306 0.00684198 -0.000604504 0.00898848 0.00112535 -0.0296198 0.00959365 -0.00602223 -0.0103691 -6.53329e-05 0.0221381 0.0137069 0.00359003 -0.0309926 -0.0196537 -0.0108634 0.00725518 0.00281848 0.0158929 0.0221867 0.0182442 0.0144515 -0.0124091 0.0182385 0.0154166 -0.0199526\ninternal_weight=3320.28 1526.01 1382.62 1794.27 1316.83 677.256 162.835 645.058 337.236 1219.79 123.743 240.562 197.518 129.998 143.388 43.044 118.139 1135.99 107.408 83.7953 28.0607 30.829 105.893 639.575 361.797 284.965 37.5043 101.408 280.583 55.7346\ninternal_count=246026 226869 225899 19157 18763 17002 10933 13559 12296 214966 4157 11493 10775 4974 970 718 3788 208106 15671 6860 1000 2622 2185 1761 1505 1430 2111 1772 1280 5860\nis_linear=0\nshrinkage=0.03\n\n\nTree=38\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 13 18 5 6 22 3 1 3 14 2 11 10 14 2 17 12 12 1 14 17 2 5 0 2 18 5 7 6\nsplit_gain=1453.16 237.323 153.1 159.081 102.361 99.1588 51.5875 50.8423 42.2982 24.7762 24.1332 23.9422 23.7939 21.4629 20.6159 19.799 16.9839 16.9784 24.2258 16.7364 16.2529 20.0847 20.3781 13.9436 12.937 11.9724 11.5449 11.8314 11.9404 11.3159\nthreshold=0.95935769349451683 0.62517033806334033 1.0000000180025095e-35 3.0677824455408698 3.4169921875000004 3.2382812500000004 0.67500000000000016 0.47260169401963864 2.9477339485081675 0.47260169401963864 0.79704415596844569 -8.7471351623535138 0.027600239276592001 0.89587973461402759 0.52722272431259987 -22.378742218017575 2.8667313866239854 0.86223072954082747 0.67581149576909094 2.6981674883069502 0.48434979059990024 1.3227116620644523 -15.262049674987791 4.9648437500000009 1.4460303622695712 50.658309936523445 2.9174053685313028 2.5244140625000004 1.0000000180025095e-35 2.9707031250000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 3 8 7 13 -4 20 9 10 16 -8 23 24 -9 -6 26 18 -3 -16 21 22 -5 -7 -1 -15 27 28 -2 -17\nright_child=2 17 6 4 15 12 11 14 -10 -11 -12 -13 -14 25 19 29 -18 -19 -20 -21 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.031333571966494465 -0.019051200031203457 0.02083448673235969 0.031411439974360758 -0.025639555640031907 -0.030561380064558389 -0.031075215435425352 -0.048752181264086876 0.014116915488637832 0.022881837670818223 0.017811905497597928 -0.02403989866223704 0.0076765647947204096 -0.035388009284384542 -0.015286684187978218 -0.029385847687371777 0.013922856472419804 -0.035708094937202381 0.030606535386631387 -0.035590505760030512 0.005022997880234299 -0.031342845438682418 -0.032428949647332499 0.0056685683351076141 0.025282929679813507 -0.00913329086977906 0.027976464320224027 -0.035420435489685133 0.024793567552538604 0.0089722798933366908 0.033036109690514515\nleaf_weight=1093.2385391996941 14.711963616544379 20.416946383193139 542.98949044221081 28.210149923688732 6.1430385756539172 3.5141925556235938 7.6643894030712536 64.54025812051259 566.85865184903378 182.48218561708927 25.785365294665098 30.049082656390965 7.2109841839410356 20.430897943326276 23.239940196159296 76.132648030528799 7.175524257705546 61.385906063020229 9.0727276152465475 26.255888324230909 90.997654921724461 36.166925678553525 51.302470024267677 41.151256674900651 25.163739740790334 6.8851038007996967 4.2274345589103168 36.560702338581905 107.33474028133787 45.758139309473336\nleaf_count=205062 841 131 460 3113 686 403 108 1010 1125 1211 1368 60 665 2181 2602 1091 508 67 117 426 12782 4104 2252 417 1729 62 297 154 1253 55\ninternal_value=-4.3893e-05 -0.0248167 0.0161254 0.0107204 -0.0039824 -0.0283645 0.0290585 -0.0130283 0.017698 0.00989011 0.00245399 -0.00475435 0.0120064 -0.0302013 0.00304097 0.0186022 0.00658937 0.0217374 0.00287696 -0.0113951 -0.0218667 -0.0142076 -0.00561968 0.0202654 -0.0308427 -0.00377914 0.00865984 0.0100493 0.00546685 0.0213085\ninternal_weight=3263.06 1288.47 1974.59 1393.88 448.747 1197.59 580.703 320.713 945.137 378.278 195.796 37.7135 51.8764 1145.72 114.036 128.034 170.01 90.8756 29.4897 49.4958 206.677 115.68 79.5126 44.6654 1118.4 27.316 162.835 158.607 122.047 121.891\ninternal_count=246340 210834 35506 34878 28121 210519 628 26289 6757 5632 4421 168 1485 209034 4038 1832 3053 315 248 3028 22251 9469 5365 820 206791 2243 2545 2248 2094 1146\nis_linear=0\nshrinkage=0.03\n\n\nTree=39\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 16 1 11 2 18 7 14 18 17 5 2 12 22 7 5 19 6 0 2 0 14 9 2 14 14 21 2 17\nsplit_gain=1460.69 265.742 106.679 82.9422 81.2081 48.4043 46.581 31.0514 25.5156 24.4632 23.3368 20.8371 19.069 18.8804 37.7837 32.6767 22.6013 18.3583 31.1169 17.8631 17.1885 13.5858 13.4621 12.8934 19.2005 12.839 12.0744 20.541 11.1904 11.1824\nthreshold=1.3779212206518803 0.49644476220754463 1.0000000180025095e-35 2.5144622974930591 3.8857805861880484e-16 -25.182852745056149 2.1383330595080281 0.89599609375000011 0.87268561280719448 2.4414009612931857 3.1006363674515387 2.4414062500000004 -33.804542541503899 0.24517227696643781 0.93750000000000011 1.0000000180025095e-35 2.8037109375000004 251.50000000000003 1.8691406250000002 1.8987378677022575 -30.838275909423825 1.8987378677022575 0.66452579377975118 1.0000000180025095e-35 23.600493431091312 0.88839509306835562 0.57401229820475252 2.5000000000000004 -43.562467575073235 1.0329219259280322\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 4 6 -4 25 -8 23 22 13 -3 -9 15 -15 21 19 18 -14 -16 -17 -5 -1 24 -7 -2 -22 29 -6 -28\nright_child=3 11 5 10 28 8 7 12 -10 -11 -12 -13 17 14 16 20 -18 -19 -20 -21 26 -23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.0045190557194085164 0.014663466069884598 -0.027729328232604791 -0.032028305874531093 -0.021480935972554148 -0.026754731328406514 -0.02251195367168279 -0.022320413604522287 -0.032624061803351652 -0.025948127261145405 -0.030947361239511011 -0.0052792056871828683 0.023214319434008559 -0.018684769684717186 0.031237145268367232 -0.051452490485283282 0.0075221771669490836 0.035099609209283712 0.02039378680244109 0.025413639819414354 0.011421532214477661 0.025677918857367758 0.023516188211433135 -0.032618484911132103 0.023076498801059529 0.017898016835152533 -0.023562359501632189 -0.026628283657812349 0.020433900853976443 0.024688899877722489 0.012439173161422516\nleaf_weight=52.714697852847166 146.75542246131226 6.8951440032105884 36.690937102132011 22.294719479046766 3.0993589891586444 17.383903304755222 89.008148654946126 12.488990198704412 16.638407231075689 1066.0315175834112 25.889212445064913 112.0351019250229 44.620206602965482 358.01073064282537 9.6116130370646733 69.920485025504604 6.4549062252044669 42.0909941829741 20.069188821129501 6.0940954312682143 508.67597864940763 7.2403900772333136 20.886380604293663 52.71906711673364 23.938014826271683 7.476999658625572 13.186437502969058 160.70443334244192 202.46134109422565 11.487149036023764\nleaf_count=3914 994 374 6688 904 352 1454 9584 1477 1936 206550 671 570 2376 329 101 734 4 640 137 10 1080 19 2770 423 262 214 261 539 525 127\ninternal_value=9.2113e-05 -0.0227204 -0.0266856 0.0182391 0.00877426 -0.00264695 0.000219015 -0.00901795 0.00729273 -0.0297891 0.0227103 0.0200542 0.00101243 0.0233317 0.0288877 0.0206368 -0.00877801 0.00521273 -0.00482511 -0.0278329 0.0218343 -0.0100412 -0.0128234 0.0133851 0.000767532 0.0126705 0.0232633 0.0165353 0.0237875 -0.00866379\ninternal_weight=3173.57 1405.93 1287 1767.64 568.071 147.37 362.51 208.278 110.679 1139.63 1199.57 118.93 119.269 1173.68 380.171 793.51 22.1606 106.78 64.6894 15.7057 763.974 29.5351 73.6011 94.041 41.3219 154.232 694.054 185.378 205.561 24.6736\ninternal_count=246019 224941 223997 21078 16299 10763 15422 14214 4075 213234 4779 944 4630 4108 444 3664 115 3153 2513 111 2741 923 6684 2139 1716 1208 2007 927 877 388\nis_linear=0\nshrinkage=0.03\n\n\nTree=40\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 1 12 12 2 0 12 7 4 0 0 7 2 22 17 22 21 7 2 12 19 0 2 9 12 1 3 18 6\nsplit_gain=1372.06 238.202 98.9103 68.2521 67.7224 65.6509 41.2877 38.8156 43.9007 34.8168 31.8587 29.9141 32.0242 28.4004 27.3797 25.4577 24.1731 22.5779 22.3929 22.387 20.8805 18.5087 17.262 16.9129 30.4634 26.2266 13.8912 15.2825 13.1676 13.1506\nthreshold=1.211689956917019 1.0000000180025095e-35 2.1468275719512682 2.2204460492503136e-16 0.53482553183223214 -25.182852745056149 1.5447487307370646 0.24517227696643781 1.0000000180025095e-35 2.7826617956161503 1.3771184689138856 0.83245446090221253 3.1132812500000004 -14.450672149658201 0.55000000000000016 2.7806893437657094 1.0000000180025095e-35 2.5000000000000004 2.9707031250000004 -1.7340650558471677 0.86223072954082747 265.50000000000006 0.45911736242885776 20.814876556396488 1.0000000180025095e-35 0.026890617030159995 0.95935769349451683 0.12808136320582797 1.8688348091416842 1.8691406250000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 6 11 -3 9 8 -4 28 15 -1 26 -11 -9 18 -5 -6 22 -16 23 -18 -10 24 -7 -26 -13 -28 -2 -17\nright_child=2 5 7 16 17 20 -8 14 10 13 -12 12 -14 -15 19 29 21 -19 -20 -21 -22 -23 -24 -25 25 -27 27 -29 -30 -31\nleaf_value=-0.031597697662736707 0.00021436005848220365 -0.029290413625808868 -0.010693097419216506 0.024828000719425793 0.024123691869731442 -0.028590676485527346 0.018046431406006147 0.030329918060874737 -0.0073425551539885317 -0.033226890812630555 0.024560124851003642 -0.02538545960079145 0.027362462377171302 0.011487624295599673 -0.035252961645367509 -0.038150616815320657 -0.045559511358511745 -0.039982391039364365 0.031115510555801779 0.018309504466377448 0.030428855670655463 0.022889004261105503 0.013929528994100859 0.020614609174849769 0.021915735003334408 -0.030648900204386802 0.028490705431036548 -0.020762184633736962 -0.023158003481697845 0.019948387374400137\nleaf_weight=940.56957764842082 26.787951905047521 37.483689300657716 46.288000477361493 139.41842796024866 28.141521246405318 24.334254042594694 62.684626039117575 420.83240430615842 37.864223578711972 14.115916791139169 462.90895124292001 109.31347576779081 12.661125678801907 82.328433577087708 9.5357387356925774 11.901914401969405 7.4019645422231388 5.0179007183760396 62.281354104168713 20.063913505524397 48.900951832067221 5.3444610079750419 317.02023042214569 50.848125191696454 41.467371752951294 9.7676033512689155 9.2143362008500862 11.746738557470961 112.6846805640962 4.0976856658235183\nleaf_count=191006 423 6753 1928 545 138 2634 544 453 1398 1673 764 16596 118 1916 149 741 319 325 74 34 50 35 2183 632 456 345 145 1248 12744 26\ninternal_value=0.000407325 -0.0228137 0.0171562 0.00492789 -0.0283531 0.00625073 -0.00341281 0.0211074 0.0176083 -0.00916677 0.0190806 -0.0296464 -0.0166755 0.00466924 0.0283728 0.0131816 0.0212107 0.0135363 0.0146199 0.000465817 0.0139726 -0.0172824 0.0116457 0.00741809 -0.00161468 0.0114997 -0.0211312 0.00120047 -0.0186994 -0.0229726\ninternal_weight=3173.03 1329.47 1843.56 450.766 1116.66 212.802 298.602 1392.79 942.362 235.917 896.074 1083.51 142.936 96.4444 450.432 433.165 152.165 33.1594 417.166 29.5997 175.318 12.7464 354.884 126.417 75.5692 51.235 130.275 20.9611 139.473 15.9996\ninternal_count=246395 220446 25949 18199 209576 10870 17300 7750 7114 16756 5186 209113 18107 3589 636 4422 899 463 3655 183 4117 354 3581 4067 3435 801 17989 1393 13167 767\nis_linear=0\nshrinkage=0.03\n\n\nTree=41\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 13 2 7 18 22 18 12 6 7 11 5 22 2 18 5 1 2 3 4 1 4 2 18 7 4 5\nsplit_gain=1259.7 268.002 114.694 98.786 94.8787 67.9401 47.7161 38.1848 37.1296 26.8234 25.0594 20.8946 20.6985 20.3354 19.5786 26.9261 19.0604 18.7146 17.6526 15.1045 14.2613 16.8585 13.9463 12.667 31.8861 11.7931 11.6498 10.9752 10.7008 14.7719\nthreshold=1.3399809614428995 1.0000000180025095e-35 1.0000000180025095e-35 2.9477339485081675 0.14796868535883259 -30.420168876647946 0.89599609375000011 1.8688348091416842 0.93750000000000011 4.2696729980286863 0.76965903234489019 4.3886718750000009 1.0000000180025095e-35 0.75683518236239655 3.1982421875000004 0.93750000000000011 -8.7471351623535138 1.4978661367769956 2.2500000000000004 -0.90051303354301027 -29.596046447753903 0.50207005315878861 4.9616699218750009 1.0240454605967615 3.8417438268661503 -1.7340650558471677 6.8062759569045816 0.89599609375000011 1.0693359375000002 2.5244140625000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 6 13 -3 7 28 -4 14 19 -11 -5 17 15 -8 -10 -1 -17 -7 21 27 -21 -19 -25 -26 -24 -14 -2 -30\nright_child=2 5 8 12 -6 10 9 -9 16 11 -12 -13 20 -15 -16 18 -18 23 -20 22 -22 -23 26 24 25 -27 -28 -29 29 -31\nleaf_value=-0.006214157437434354 0.015063412159771382 -0.031667953179521438 0.031169377356051284 -0.0081181764867727206 0.024744036401320946 -0.034418386492425936 -0.0085140873540111069 -0.018314125640043991 -0.045351734495365642 -0.014771843809625724 0.030706508135213108 0.026878124428819077 -0.042695881088163572 0.016106371097859118 0.024968197187350025 0.01966882983122354 0.0097966312604212263 -0.031475553441048743 -0.02605078149366763 -0.015772707393571374 0.021861292644100187 0.018552805205095012 0.015604065351895069 -0.030743223332956232 -0.026864811720739325 0.02531523217688619 -0.028660465666222087 0.00080069895739226244 -0.015998355303684444 0.031436262290884913\nleaf_weight=29.7225418536691 54.585144718643278 32.550030245794915 461.48370806267485 23.139912510174327 29.030127281555906 6.4371003950946024 58.809030651114881 135.04667013010476 5.7197521142661563 56.549984941724688 59.076394579838961 12.398710806854067 6.9150063213892308 8.1266125030815584 92.616393871605396 103.79976949782576 34.127191149629653 976.24520362721523 7.3441342092119148 16.062043064041063 590.02472154580755 36.556308035738766 105.53492757334607 47.877830274694134 4.1822768556885448 15.855285744415594 4.7359070895472533 15.233159075025467 27.229683217592537 6.499467395246028\nleaf_count=1762 432 6356 407 566 55 608 1408 11744 91 3761 69 23 175 43 376 782 56 200818 192 1438 1490 166 1844 8822 691 363 683 199 351 11\ninternal_value=-0.000214413 -0.0220887 0.0166928 0.0117357 -0.0281243 0.00778705 0.00236022 -0.00833544 0.0287753 0.00956984 0.0146108 -0.00700495 0.0194636 -0.0295643 0.0139323 0.0078074 0.000963982 -0.0299246 0.0164424 0.00727291 0.0204584 0.00622792 0.00965084 -0.0305929 -0.017633 0.0136648 0.0134625 -0.0140374 0.00695777 -0.00621591\ninternal_weight=3063.52 1335.44 1728.08 1226.75 1111.04 224.396 554.879 223.361 501.331 331.518 191.846 68.9487 671.869 1082.01 262.569 169.953 39.8469 1073.88 111.144 132.77 648.729 58.7045 126.333 1044.16 67.9154 20.0376 110.271 22.1482 88.3143 33.7292\ninternal_count=245782 223552 22230 21676 212554 10998 19080 12538 554 6542 4642 3784 2596 212499 2758 2382 147 212456 974 4573 2030 540 3965 210694 9876 1054 2527 374 794 362\nis_linear=0\nshrinkage=0.03\n\n\nTree=42\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 1 13 7 13 13 18 7 5 1 18 7 0 7 17 1 4 17 17 1 22 1 3 0 6 14 13 4 5\nsplit_gain=1227.85 203.386 138.308 85.4412 63.2775 57.1742 50.5915 46.9524 42.5283 38.7228 31.3033 29.1177 31.3752 28.3091 25.5762 19.5342 15.4614 14.589 16.4991 17.7776 14.5721 14.1461 12.9376 13.9416 12.6156 12.5263 12.4782 13.118 11.8023 27.7213\nthreshold=0.95935769349451683 1.0000000180025095e-35 2.1468275719512682 0.14796868535883259 2.1386718750000004 -0.032724337246802453 0.14796868535883259 2.1383330595080281 1.0000000180025095e-35 4.4121093750000009 0.44640776988282155 1.2424533248940002 2.5244140625000004 0.34392793676569194 3.7011718750000004 3.9281938047451082 0.26032702032866095 1.7380371093750002 0.92169800002628566 1.4144231987897788 4.0016611463566001 0.93750000000000011 4.0016611463566001 0.53650715206675215 1.8987378677022575 1.0000000180025095e-35 0.43714636302263388 -0.43403209982848934 2.7826617956161503 2.0131835937500004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 9 3 4 7 14 8 17 24 -1 -11 -10 13 20 -3 -6 -16 18 -2 -20 -13 -8 23 26 -4 -19 -15 -28 -9 -30\nright_child=2 5 6 -5 15 -7 21 28 11 10 -12 12 -14 22 16 -17 -18 25 19 -21 -22 -23 -24 -25 -26 -27 27 -29 29 -31\nleaf_value=-0.031285816550351148 -0.041236217391196726 -0.026508581921613754 -0.018897428941574413 0.027343254540472765 0.016622118035954498 0.020009998150425194 0.031612689835987078 -0.023843572599124039 0.024534675357284891 -0.026259642749999937 0.023289460649170247 -0.035990050846081312 0.028294060405301916 0.0097303735000036067 -0.020871103945772747 -0.033706344806532534 0.021058565856794925 0.016917107188215562 0.015618290292814549 -0.036671914593209204 0.0080875079683475522 0.0090946895356798438 0.019356795421077242 0.015639923439719476 0.01976883858153565 -0.036585253884834364 0.0036937679713557473 -0.022398776622902631 0.022922106359125401 -0.023319522532276149\nleaf_weight=929.49955420405604 9.68022610072512 51.546599861583672 37.535047362558544 95.730453982949257 120.8650108986767 97.890601853432599 322.24249679595232 151.59040632715914 362.13065818697214 22.799925214610994 21.168124030926265 16.500621083192527 105.12095970986411 89.139494510367513 11.064827357186003 6.4443306560860938 22.23618486453779 61.613131118472666 21.934852077160034 6.8594833358656606 10.482560824137183 28.291755199432373 143.36115866369801 127.34872145950794 8.5913151851855201 3.2764020985923699 41.530611749039963 28.487289423355833 17.333435233449563 31.247424325498287\nleaf_count=195251 437 7871 1931 92 1876 1368 260 20050 430 4358 535 1320 217 910 952 602 197 423 193 343 109 66 354 727 44 82 566 834 465 3292\ninternal_value=-0.000305568 -0.0245391 0.0148667 0.00186126 -0.00385086 0.00448126 0.0200465 -0.0112884 0.0164861 -0.0299857 -0.00241714 0.0178903 0.0135727 0.010131 -0.013459 0.0138317 0.00692442 0.00510088 -0.00938405 0.0023488 -0.0191895 0.0298263 0.0120117 0.00828182 -0.0114289 0.0137014 0.00233339 -0.00708809 -0.0197347 -0.00669116\ninternal_weight=3003.54 1156.21 1847.34 526.575 430.845 182.738 1320.76 303.535 970.228 973.468 43.968 924.102 561.971 456.85 84.8476 127.309 33.301 103.364 38.4746 28.7943 26.9832 350.534 429.867 286.506 46.1264 64.8895 159.157 70.0179 200.171 48.5809\ninternal_count=246155 210532 35623 27855 27763 10388 7768 25285 7442 200144 4893 5467 5037 4820 9020 2478 1149 1478 973 536 1429 326 3391 3037 1975 505 2310 1400 23807 3757\nis_linear=0\nshrinkage=0.03\n\n\nTree=43\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 8 2 11 1 2 7 18 7 22 18 2 9 0 0 12 22 12 4 2 18 10 1 5 0 7 1 19 0\nsplit_gain=1176.21 270.275 95.7885 63.7239 61.8108 69.9276 37.4556 31.9544 31.1392 32.4018 29.4733 28.3561 26.6854 24.3605 24.0923 21.473 20.6581 25.895 17.8128 17.2069 17.095 16.0788 14.8218 12.8803 12.2604 28.6741 11.6872 14.2036 12.8477 11.6705\nthreshold=1.5661550609387962 3.8857805861880484e-16 1.0000000180025095e-35 -43.562467575073235 0.27595666403761582 2.3713805701640611 -25.182852745056149 0.89599609375000011 2.249904835165133 2.9707031250000004 1.0000000180025095e-35 1.4978661367769956 -8.7471351623535138 1.0000000180025095e-35 0.29549219484135919 0.38644940889434637 0.86223072954082747 1.0000000180025095e-35 1.0000000180025095e-35 2.7826617956161503 23.600493431091312 2.1383330595080281 2.524928003624769 0.34439332195363109 3.9609375000000004 0.31303196667935901 0.89599609375000011 4.0016611463566001 259.50000000000006 -1.4843156180481063\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 11 -3 5 7 -4 19 -7 14 -6 23 -12 20 -10 29 17 22 -19 21 -8 -2 -5 -1 -13 -26 27 28 -16 -9\nright_child=4 3 6 16 10 8 13 15 9 -11 12 24 -14 -15 26 -17 -18 18 -20 -21 -22 -23 -24 -25 25 -27 -28 -29 -30 -31\nleaf_value=-0.031599843904095364 -0.0033602869143385355 -0.0316971270882569 -0.031315693563829135 0.01598394290077686 0.029165342048398522 0.022117978679175879 -0.02365460316909699 0.017387847873599711 -0.030203550472478056 0.03101024162612237 -0.042981131837361859 -0.030821057448090054 0.014429276514051141 0.023521743085334189 -0.020309731259688754 0.012851904521541495 0.029577756097820637 0.029757996472201915 -0.044723257768514879 0.018865530601111694 0.010406355437044001 -0.028971098479217937 -0.035319529416439732 0.0075211125200149278 -0.030160266628472449 0.016552353434944232 0.013681450433529686 0.013206762182637869 0.010459343762686882 -0.030558570268657406\nleaf_weight=9.3599586762720701 40.309041491243988 27.899460824264679 27.636399569397327 110.04881186556304 429.70225891191512 554.15515957958996 25.104101117583923 5.4384367477614459 15.427064355346372 51.561863423325121 7.9344766661524764 942.78079127788078 38.648754253983498 45.206819620740134 36.473935515503399 140.26359331654385 66.187627295497805 3.0185637883841983 11.191820765961891 12.61345465457998 26.10084663447924 48.535693457000889 4.388503885944373 27.253098356653936 36.364243430376519 16.53683316858951 119.44540527416393 38.57491190667497 17.365562407299876 16.560471055214293\nleaf_count=1023 579 5998 5573 2483 617 1142 2677 121 931 66 262 200100 76 406 1653 1772 80 17 804 242 579 5948 212 829 7884 647 859 241 315 1546\ninternal_value=-3.97269e-05 -0.0202561 -0.0260815 0.00996821 0.0176858 0.0137448 -0.00116247 0.000278578 0.0180026 0.00977432 0.0267129 -0.0290719 0.00395284 0.00771369 0.00484204 0.008519 0.0160659 0.00893965 -0.0279857 -0.0128549 -0.00642585 -0.0175182 0.0137276 -0.00305421 -0.0300246 -0.0155218 0.00752387 -0.000496041 -0.0103756 -0.0184649\ninternal_weight=2952.09 1379.08 1156.34 222.735 1573.01 1096.72 124.048 263.721 833.004 278.849 476.285 1032.29 46.5832 96.4118 227.287 162.263 194.835 128.648 14.2104 101.458 51.2049 88.8447 114.437 36.6131 995.682 52.9011 211.86 92.4144 53.8395 21.9989\ninternal_count=245682 229312 219718 9594 16370 15415 9235 10208 5207 4065 955 210483 338 3662 3999 3439 3596 3516 821 6769 3256 6527 2695 1852 208631 8531 3068 2209 1968 1667\nis_linear=0\nshrinkage=0.03\n\n\nTree=44\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 4 13 18 18 5 2 3 14 3 14 18 7 4 14 5 6 19 17 14 14 14 22 18 4 3 18 14 7\nsplit_gain=816.987 369.126 153.24 113.21 117.84 67.8513 60.2093 55.3607 48.3971 31.7917 27.9148 36.4713 26.4343 22.0919 26.0896 18.7822 15.9409 15.4551 21.6894 15.2484 13.9745 13.2252 12.8825 12.5133 12.485 12.1861 16.6538 11.9577 11.5181 12.9844\nthreshold=0.28287969384316608 1.0000000180025095e-35 8.0136718750000018 0.14796868535883259 2.9174053685313028 2.524928003624769 3.7011718750000004 -20.765821456909176 0.37248792739977959 0.52722272431259987 0.53650715206675215 0.91125284794847328 3.3497501700808394 0.89599609375000011 1.0693359375000002 0.61561876504725965 1.2426757812500002 4.3378906250000009 237.50000000000003 2.6250469244966204 0.6618576200563534 0.8507468585983845 0.064908752851949539 0.93750000000000011 0.89587973461402759 4.1025390625000009 0.22136347059745312 0.89587973461402759 0.4896816566336385 3.3134765625000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 4 10 19 8 -4 -6 22 11 13 21 14 27 -9 -16 18 20 24 -8 -11 -10 -5 -1 26 -26 -2 -15 -30\nright_child=3 -3 7 23 6 -7 17 15 9 12 -12 -13 -14 28 16 -17 -18 -19 -20 -21 -22 -23 -24 -25 25 -27 -28 -29 29 -31\nleaf_value=0.01859219367838735 0.025355310620119687 0.026667774198079714 -0.031253725477539414 0.031302471012055508 -0.025980214733115993 -0.030046204551874379 0.011534515072302963 0.029114146864621812 -0.034284094220547422 -0.017710134750148172 0.022820724135141222 -0.037678792696329455 -0.021885764830844444 0.023319699095024387 -0.037973290665274932 -0.00098780592199576101 -0.0020249717606877589 0.032792839618112732 0.018521316432920279 -0.032102326430020561 -0.032316077736277864 0.020442638285654888 0.011522919066114367 0.011185046618374011 -0.026613333680386174 0.018569667095846863 0.0031039588901370014 0.0041721347385802075 0.0055348013306476809 0.028647520348537008\nleaf_weight=21.303078432567418 39.076765200356022 147.71721562399762 22.708301241800655 311.81202078727074 100.00454261753475 904.74684744764818 10.609508280875163 62.774791053787339 4.8424266990041351 10.976826846366746 381.05632893624716 12.127264955313874 80.240364566561766 162.32173057016917 15.507900897646321 26.277728553628549 34.275940497056581 34.226761650294065 61.961872190935537 15.887794643989762 14.510944232577456 25.28638237901032 117.43306467903312 32.140153111889958 33.46114873274928 13.259722419432363 33.073763269814663 60.951039870386012 94.371947142761201 27.764873251318932\nleaf_count=330 107 452 4560 274 16452 193978 202 294 321 539 1141 682 9554 357 576 856 542 69 1306 1952 1188 441 2862 81 3651 183 1013 1198 747 80\ninternal_value=0.00011377 -0.0178079 -0.0236224 0.0141899 0.0100909 -0.0272684 -0.00207877 0.00965611 -0.00857158 -0.00120262 0.0168519 0.0117172 -0.0123635 0.01318 0.00382155 0.0203193 -0.0137503 0.0159758 0.00906872 -0.00571289 -0.0140634 0.0087285 0.00944545 0.0294605 -0.00129191 -0.00676486 -0.012014 0.0126166 0.018078 0.0109789\ninternal_weight=2912.71 1281.21 1133.49 1631.5 1287.55 1021.73 460.093 111.761 338.784 238.779 827.454 446.397 116.504 434.27 149.812 89.0525 49.7838 121.309 87.0823 116.986 25.1205 36.2632 122.275 343.952 101.098 79.7946 66.5349 100.028 284.459 122.137\ninternal_count=245988 207269 206817 38719 38364 201107 32934 5710 30169 13717 5430 4289 10534 3607 2423 1150 1118 2765 2696 7129 1390 980 3183 355 5177 4847 4664 1305 1184 827\nis_linear=0\nshrinkage=0.03\n\n\nTree=45\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 13 8 1 0 7 0 14 17 4 2 4 0 10 13 9 1 19 21 7 22 13 19 3 7 0 3 6 19\nsplit_gain=1065.64 269.624 97.864 96.7686 69.1462 40.8232 38.7386 31.4473 22.2056 21.8176 20.1814 18.7135 17.2495 16.8448 16.2686 15.6546 14.8794 17.5507 22.1651 14.6644 14.167 14.0684 13.599 13.2825 13.1515 12.7903 12.2211 12.1248 11.9351 19.7099\nthreshold=1.3779212206518803 0.49644476220754463 0.14796868535883259 1.0000000180025095e-35 2.9477339485081675 -0.7405159439372101 0.89599609375000011 1.0212340148387515 0.87268561280719448 3.2718994203396332 1.0693359375000002 -34.252307891845696 2.7826617956161503 0.84505259813102007 3.8815106545092593 -0.041888724317900632 1.0000000180025095e-35 4.0016611463566001 253.50000000000003 5.5000000000000009 1.0000000180025095e-35 0.93750000000000011 -0.20399465096770289 263.50000000000006 0.28287969384316608 3.1132812500000004 1.6765212893675572 0.58795853567030554 4.0507812500000009 250.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 4 13 6 -5 7 12 -7 11 20 23 -2 -1 15 -3 17 18 19 24 -9 -4 -21 -8 -6 -15 27 28 29 -13\nright_child=2 14 21 5 16 8 9 10 -10 -11 -12 26 -14 25 -16 -17 -18 -19 -20 22 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.030486020521375497 -0.022974015160183072 -0.029090928862283968 0.031290611225307546 -0.029209985462339452 -0.03648443337371466 0.013401196737770998 -0.03346252573087162 -0.012733520028125059 -0.025152876224268025 -0.022109527040366127 -0.010844419485264233 -0.0042441935973249363 0.016700135133419843 -0.020907305491867399 -0.040145097334916544 0.024909416960396884 0.034768317124893686 0.021532882836087199 0.021108082433911612 -0.00075157176173660557 0.01955885593340535 0.010244976848867868 -0.045752144174224157 0.011589261177783567 0.0099728703721796205 0.017405929662310578 0.025376725703291973 0.022365488518834034 0.02812735720312274 0.016555047821936845\nleaf_weight=845.89896642969688 84.243640823522583 4.2315592576051122 340.97453271737322 35.798250464256853 4.7740092622116199 87.369348880718462 12.449453252484092 14.310838114353826 14.911311193369327 16.70800422236789 48.309722119243816 99.034177750349045 10.428565347567199 118.80329872533912 2.9299411233514538 116.34340460947715 46.978733124211431 362.16011393599911 116.46841345727444 17.240260527469221 64.278362241457216 32.540632592514157 8.3348222135100496 9.7751953110564482 121.00301381712779 7.6558174181263885 66.764562726952136 61.482510620495304 20.452949502505362 68.61119448998943\nleaf_count=194974 9410 539 305 6826 238 2044 1290 703 1877 1242 1047 2272 296 18036 38 395 51 831 432 132 367 75 152 235 846 126 138 266 31 732\ninternal_value=8.30556e-05 -0.0209326 0.0160215 -0.0256097 0.0119891 -0.00209116 0.00434579 -0.00548344 0.00759135 0.010474 0.00427739 0.0121424 -0.0184749 -0.0289438 0.0209871 0.0227906 0.0184861 0.0172187 0.0113265 0.00371023 0.0135924 0.0294894 -0.0165869 -0.0140013 0.00793566 -0.0184695 0.0140174 0.0108896 0.00704008 0.00431587\ninternal_weight=2861.27 1233.94 1627.32 1110.44 1253.81 138.079 576.849 221.571 102.281 355.278 126.899 338.57 94.6722 972.358 123.505 120.575 676.959 629.981 267.821 151.352 78.5892 373.515 25.5751 22.2246 125.777 126.459 316.345 249.581 188.098 167.645\ninternal_count=245946 224855 21091 223883 20711 10747 18029 11823 3921 6206 2117 4964 9706 213136 972 934 2682 2631 1800 1368 1070 380 284 1525 1084 18162 3439 3301 3035 3004\nis_linear=0\nshrinkage=0.03\n\n\nTree=46\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 13 8 18 5 1 2 18 22 14 4 11 21 1 19 13 21 18 4 14 4 3 6 4 19 17 5 3 14\nsplit_gain=1085.6 236.843 91.1113 81.5439 72.3913 53.9841 37.7161 36.0739 22.68 19.3953 18.19 16.7243 15.2571 17.5316 14.0804 13.5485 21.2307 13.0748 12.4212 12.8852 12.1803 12.1524 11.9834 11.5178 13.5077 11.1817 13.036 11.1292 18.0441 10.5943\nthreshold=1.3779212206518803 0.49644476220754463 0.14796868535883259 1.0000000180025095e-35 1.8688348091416842 3.7255859375000004 2.6981674883069502 -25.182852745056149 2.9174053685313028 0.93750000000000011 0.87268561280719448 0.84667968750000011 0.71017588656534381 2.5000000000000004 1.9634439907166827 251.50000000000003 -0.026449291562135987 5.5000000000000009 3.1132683346437333 2.3505859375000004 0.69039108051708775 4.9616699218750009 0.37963534769016083 3.3837890625000004 4.2863769531250009 242.50000000000003 1.5185086528259144 2.6738281250000004 0.34948566548243237 0.6618576200563534\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 4 8 14 6 11 -5 -1 -4 21 -6 13 -3 23 16 -13 -11 -17 22 -8 -9 -20 24 27 -18 -27 28 -2 -7\nright_child=2 12 9 7 5 29 20 10 -10 17 -12 15 -14 -15 -16 18 25 -19 19 -21 -22 -23 -24 -25 -26 26 -28 -29 -30 -31\nleaf_value=-0.014912368922588277 -0.038506055797777043 0.01455661556934934 0.031265995419814667 -0.031268189686105209 -0.032052115895195903 0.027277515929282296 0.012255474616957708 -0.020611485589209964 -0.030478123062915443 -0.011005569172622125 -0.024879676301772121 -0.019850848626712825 0.026276899408373355 -0.038179174822886908 0.021411026095487614 0.013044862148161069 -0.014499673806832442 0.027347708043670135 -0.0300395312921856 -0.035115276384029503 -0.0033150767256621382 0.014400260306702608 0.0095267255684113768 0.033941844044898764 -0.046500122449863207 0.024651242514272104 -0.029595300755119923 0.03340763299592403 0.0100199043593207 0.0080871173578589018\nleaf_weight=96.759562548715621 8.6247922482434642 19.953574413666505 341.79316380154341 31.276386506156996 21.345960058039054 99.03532130876556 208.23393480584491 9.1938911955803615 850.33024806430331 17.197557304054502 14.474474264308808 85.826088581525255 84.153517116326839 6.7802073113853103 533.30357354693115 54.186503282398917 15.200323638040571 13.353034470230339 7.8393484175903714 10.734330338833386 57.211841193959117 76.616707819746807 30.236747017712332 9.2087053116410953 4.2827953519299617 24.648003934533339 3.8478704504668704 8.2762639895081502 23.404185297898948 36.398110205773264\nleaf_count=12191 271 180 312 6781 3246 280 1798 718 201225 51 1877 6604 224 561 1189 677 778 21 1101 1129 1030 1403 1333 11 37 308 414 8 206 394\ninternal_value=0.000194393 -0.0215414 0.0162155 -0.0258337 0.0122952 0.00542348 0.00100951 -0.00367101 -0.0289053 0.0292224 0.00515959 -0.00724314 0.0200697 0.00028742 0.0199365 -0.0048381 -0.0112601 0.00609333 0.00326837 -0.00762893 0.00889969 0.0104548 0.000820298 0.00519492 -0.00135645 0.00568016 0.0165712 0.00445011 -0.00381741 0.0222171\ninternal_weight=2803.73 1189.54 1614.19 1078.65 1241.84 654.744 519.311 131.561 947.09 372.344 100.285 253.865 110.887 26.7338 587.1 232.519 129.522 30.5506 102.997 48.8104 265.446 85.8106 38.0761 53.7967 44.588 43.6962 28.4959 40.3052 32.029 135.433\ninternal_count=246358 225160 21198 224195 20814 19092 18418 10779 213416 384 3998 15590 965 741 1722 12344 8104 72 4240 3563 2828 2121 2434 533 522 1500 722 485 477 674\nis_linear=0\nshrinkage=0.03\n\n\nTree=47\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 8 13 18 18 5 14 18 14 18 3 7 7 0 11 19 22 21 5 12 3 14 3 14 22 18 14 19 4\nsplit_gain=673.935 342.136 133.609 106.573 103.924 85.9606 46.0019 48.6674 40.2222 29.5043 29.2772 26.0464 25.2312 24.3732 21.3934 20.1626 22.2268 19.0662 18.3078 17.8522 16.6063 16.0183 14.5653 14.5023 15.1723 16.2371 14.4318 15.8492 12.9363 12.345\nthreshold=0.28904431203898151 1.0000000180025095e-35 1.0000000180025095e-35 0.14796868535883259 2.9700856263602158 2.524928003624769 3.4169921875000004 0.52722272431259987 6.8062759569045816 0.77909152491212919 1.2424533248940002 0.53650715206675215 0.89599609375000011 1.0000000180025095e-35 0.29549219484135919 0.33135327358542549 262.50000000000006 0.93750000000000011 3.5000000000000004 6.0488281250000009 0.29026868663126176 0.17443601580070725 0.075876460218247857 0.40160889672495398 0.33900667957710101 0.93750000000000011 5.2067464199591091 0.95767813775662891 213.50000000000003 5.6752929687500009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 4 10 13 7 14 9 -4 15 -9 23 -1 -6 -2 -17 -5 -13 -11 -3 22 -15 -12 -25 -26 27 28 -8 -14\nright_child=3 20 8 17 6 -7 26 11 -10 19 12 18 29 21 -16 16 -18 -19 -20 -21 -22 -23 -24 24 25 -27 -28 -29 -30 -31\nleaf_value=-0.031847468125787044 0.023552988968376376 -0.015186818777416828 0.018559863536111935 0.031222179840423192 -0.028581534350385095 -0.030225808297477751 -0.019225569788528421 -0.030319648484738698 -0.028163438036638698 -0.027892411259692336 -0.018229794046965734 -0.029565444670477645 0.011644001064458492 0.013879201484976903 0.0065720279559722278 -0.04173556697107203 0.033094794950952994 0.0052366177407319739 0.010974443800943867 0.019206413451620757 0.028075839863019213 0.012322202695775761 -0.031599780271975311 0.019569027101433083 -0.02861246486512491 0.0047691761833505838 -0.0077615615343236558 -0.035595145454782907 0.024174104133355946 0.026057387138017783\nleaf_weight=19.204697472450789 338.85844298545271 7.8511280524544409 96.15337457699934 286.80636450601742 16.60235498379916 809.59687572356779 5.9507125988602629 98.191731186932884 29.29841734765796 23.249737546429969 30.081237586855423 13.941133704269303 204.81248105701525 9.2661530246259662 135.05322848306969 8.2508212719112617 4.9008602332323781 28.364549175836146 29.506241233320907 9.5044471863657218 142.42088862787932 65.828765698126517 16.32178361521801 39.473274912452325 16.458835577592254 51.878387235105038 23.341089642839506 3.7931360499933353 82.081385172670707 73.38462272984907\nleaf_count=2823 731 191 1639 277 2067 191415 369 18058 6129 2413 1633 1264 1246 386 5126 69 5 80 1081 117 273 1434 2391 223 665 705 1920 230 817 221\ninternal_value=-0.000110261 -0.0165556 -0.0224662 0.0134373 0.0092722 -0.0268403 -0.00295051 -0.00925388 0.00298491 0.0102267 0.0157674 -0.0218342 0.0103403 -0.00201457 0.00256161 0.022135 -0.0134825 0.0289075 -0.00240627 -0.0140614 0.0257301 0.00452947 -0.0152279 -0.000157935 0.00500566 -0.00356741 0.0130438 0.018359 0.0210574 0.0155258\ninternal_weight=2720.43 1228.7 1078.42 1491.73 1176.56 920.218 408.461 293.295 158.206 128.908 768.099 141.639 416.089 110.621 151.656 352.01 13.1517 315.171 43.4474 32.7542 150.272 91.4167 25.5879 137.892 107.81 68.3372 115.166 91.8252 88.0321 278.197\ninternal_count=245998 209211 208747 36787 36430 198449 30932 27596 10298 4169 5498 20403 4693 7034 7193 805 74 357 2345 2530 464 4211 2777 3226 1593 1370 3336 1416 1186 1467\nis_linear=0\nshrinkage=0.03\n\n\nTree=48\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 7 13 1 5 0 18 18 7 22 7 22 7 4 1 4 21 14 1 1 1 2 2 0 18 14 14 21 14\nsplit_gain=982.057 227.442 122.311 101.648 70.269 43.0205 33.2284 32.7732 26.8277 24.7619 18.7215 17.3544 16.5674 16.4015 21.8964 16.1152 15.4703 29.2332 14.9797 14.757 14.5284 14.3987 13.4239 11.9357 23.3412 11.7219 10.6697 26.648 20.3913 10.4569\nthreshold=1.3779212206518803 1.0000000180025095e-35 3.5693359375000004 1.0000000180025095e-35 2.9477339485081675 3.7500000000000004 0.82186659671441242 2.3502401828962087 6.3465127852423047 0.89599609375000011 0.93750000000000011 1.0000000180025095e-35 0.93750000000000011 0.89599609375000011 2.7826617956161503 2.1468275719512682 3.9514160156250004 3.5000000000000004 0.27091919595055813 4.0016611463566001 0.60367420583093068 -0.51493959326534589 24.399772644042972 -39.733196258544915 1.0410863859294157 1.4978661367769956 0.59843054868820844 0.61113007853467505 1.5000000000000002 0.67646516896599407\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 4 5 7 -1 12 -4 -8 -5 19 18 14 -9 29 17 -12 -2 -6 25 -3 -19 24 -13 -11 -25 -28 -29 -15\nright_child=3 21 8 10 11 -7 9 13 -10 20 16 23 -14 15 -16 -17 -18 22 -20 -21 -22 -23 -24 26 -26 -27 27 28 -30 -31\nleaf_value=-0.030508877498942051 0.01223616161379025 -0.025848627678743009 0.018305618624982891 0.030026244500743829 -0.03572340441850503 0.020994032755191624 -0.024726734267843591 -0.023702116308379761 -0.030149294431344828 0.01130908284929965 0.026249938524082531 -0.039140776049005371 0.013194712048180565 -0.0050587247431696887 0.017071854497018563 0.0088275014856470796 0.038284367513731786 -0.051634936620046111 -0.013992067460923843 0.010556223360999328 0.013053670901920198 0.024689495012113694 0.013985991535559437 0.021985711513547848 0.018730159918958485 -0.031389060934795709 -0.029963455192836289 -0.022811593579245129 0.019449385669226979 -0.033937421919479349\nleaf_weight=812.17730348068289 31.128965036012232 4.5695292314048848 66.057251002639532 386.07857647677884 8.8402704067993909 95.548552293796092 87.781096624617931 86.751271428132895 11.217199486796742 8.1851194805931318 12.634705554228278 8.5703287655487674 139.87910463591106 40.131794137938414 12.97162343515083 61.04971332475543 9.1392573006451112 9.8424399909563345 49.385942302993499 15.915915930061601 33.71878655772889 94.832604468334466 3.2956443838775149 430.12042955146171 17.207042677327991 15.797050646506248 11.320364810526369 10.324229608289896 129.71785016963258 14.953599334112367\nleaf_count=200877 213 93 906 420 421 589 15766 10201 2607 265 29 296 773 2233 365 1253 8 99 923 179 1271 354 5 993 34 2946 40 180 471 1566\ninternal_value=0.000254848 -0.0210451 -0.0252095 0.0155041 0.0109182 0.00287314 -0.0281378 -0.00113466 0.0109766 -0.0147829 0.0281258 0.0176723 0.0069875 -0.00942148 -0.0182815 -0.00172241 0.00685885 -0.00529865 -0.00382591 -0.00671527 0.000508326 0.0221319 -0.0353492 0.0186759 -0.00125364 -0.0169468 0.01956 0.012568 0.0161942 -0.0133635\ninternal_weight=2719.14 1134.34 1034.93 1584.81 1163.82 531.801 957.659 436.252 77.2745 145.482 420.991 632.016 220.394 215.858 99.7229 116.135 34.912 25.7728 80.5149 24.7562 57.701 99.4021 13.1381 607.26 25.7774 23.9822 581.483 151.362 140.042 55.0854\ninternal_count=246376 225085 224638 21291 20730 18116 221125 17527 3513 20248 561 2614 1909 15618 10566 5052 141 133 1136 600 4482 447 104 2014 330 3211 1684 691 651 3799\nis_linear=0\nshrinkage=0.03\n\n\nTree=49\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 7 13 1 2 17 7 0 0 4 1 0 0 8 4 22 3 14 0 7 5 13 17 2 14 12 17 1 19\nsplit_gain=933.372 249.24 117.525 86.8276 65.5934 52.9429 37.5594 35.3973 20.509 18.1918 16.2376 25.0501 23.7197 16.688 15.9658 15.5018 14.7833 12.8836 13.8283 12.4775 17.7571 18.9315 18.2326 11.999 11.4245 12.8648 11.1643 13.6732 9.77697 9.59645\nthreshold=1.4475910192202701 1.0000000180025095e-35 3.1132812500000004 0.14796868535883259 3.3033179733375269 -13.633526802062987 3.9281938047451082 0.89599609375000011 1.0212340148387515 0.82186659671441242 10.818688511848451 1.7622051334857016 0.34392793676569194 -0.53258533561495902 1.0000000180025095e-35 2.7826617956161503 0.93750000000000011 0.33504387972412514 0.39695441677226773 1.3771184689138856 2.5244140625000004 1.4980468750000002 -0.51849643425434 0.17341418658717714 2.5338792800903325 0.77909152491212919 0.026890617030159995 0.70050815440916869 2.6981674883069502 264.50000000000006\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 4 6 -4 7 8 15 -1 11 13 -13 -9 -11 28 -5 18 -14 20 21 -6 -23 -24 -10 -26 -7 -28 29 -2\nright_child=3 -3 5 16 19 26 -8 10 24 14 -12 12 17 -15 -16 -17 -18 -19 -20 -21 -22 22 23 -25 25 -27 27 -29 -30 -31\nleaf_value=-0.030114927812458354 -0.031010071412739249 0.023787443276935025 -0.027500310085512174 0.030900961963619408 0.016862562525524493 0.021121190684945296 -0.03413394472010893 0.023755476564446348 0.012314404741377207 -0.0215443890109424 0.030044139446179771 -0.021079731040601495 0.010566778293668909 -0.018210933322708074 0.010921660204846338 0.019244709504868521 0.0070091672410345432 0.017929997872986356 -0.027284151184946002 0.023720845126294725 0.030515843632064155 -0.045581393127431175 -0.042291252247359766 0.0068091282552981795 0.0012806804603742122 -0.034287860864836851 0.016138871667600158 -0.035390315463601434 -0.0019388406275852514 0.0017834096011983937\nleaf_weight=775.38517617376056 45.709315151034389 104.60586457478348 34.967188424488995 313.35440686484799 148.8385539513547 68.75772196642356 20.137086112517864 9.8489355631172639 73.204581310274079 121.67232542176498 39.657754334621131 16.272138832951896 28.97879656823352 40.011730643978808 14.723563752835615 10.219673328334464 25.910032362677157 238.60581616126001 11.231095570139585 288.28178517566994 50.552341546863317 6.700046752113848 3.8392398594878605 60.025200761388987 49.778961342060938 10.195675449445842 10.145312124164777 6.9667899361811569 22.569865104975179 9.5717765355948341\nleaf_count=196509 6271 446 7705 317 735 1296 2340 135 673 19947 64 1106 275 1724 435 250 75 1293 403 420 62 90 54 420 716 522 121 369 760 602\ninternal_value=0.000386244 -0.0201697 -0.0246414 0.0157336 0.0118823 0.00313684 0.0052019 0.00656341 -0.00298962 -0.0283222 0.0120524 0.0099031 0.0132228 -0.00961726 -0.0179858 -0.0141392 0.0291023 0.0152929 -0.000410964 0.0193509 0.0146139 0.0108281 -0.00195779 0.00325095 0.00443393 -0.00522913 0.0157856 -0.0056357 -0.01871 -0.0254012\ninternal_weight=2660.72 1137.22 1032.62 1523.49 1184.23 120.837 625.993 605.856 221.25 911.781 384.606 344.949 295.088 49.8607 136.396 88.0706 339.264 278.816 40.2099 558.237 269.955 219.403 70.5645 63.8644 133.179 59.9746 85.8698 17.1121 77.851 55.2811\ninternal_count=246135 226828 226382 19307 18915 9491 17134 14794 9794 216891 5000 4936 3077 1859 20382 7883 392 1971 678 1781 1361 1299 564 474 1911 1238 1786 490 7633 6873\nis_linear=0\nshrinkage=0.03\n\n\nTree=50\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 0 11 0 7 14 7 7 0 22 3 14 3 11 11 4 4 3 14 17 4 3 14 4 13 0 17\nsplit_gain=880.247 173.539 112.601 88.1557 49.5943 40.8883 39.0868 32.4627 22.2227 21.9863 21.8852 21.477 20.8099 19.952 17.5821 16.8024 17.8738 16.0918 15.9439 19.8433 15.1993 14.681 13.1482 12.5255 12.8426 18.1953 17.2223 11.8143 11.67 12.1031\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 2.9477339485081675 0.75738199555027408 0.66750229771629133 -0.7405159439372101 2.5244140625000004 0.52722272431259987 1.0000000180025095e-35 0.89599609375000011 0.97942481616843191 0.93750000000000011 0.47260169401963864 0.84677952177506588 0.47260169401963864 0.042545688445982228 0.71017588656534381 1.0693359375000002 1.4716796875000002 1.0000000180025095e-35 0.87268561280719448 2.5789308873416652 5.4291992187500009 0.53650715206675215 0.57401229820475252 2.1186523437500004 -1.0000000180025095e-35 1.1851094193663554 1.0329219259280322\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 3 4 7 28 20 8 15 -5 11 -6 -4 22 18 16 -2 21 -13 -20 -3 -8 -9 24 25 -11 -27 -28 -1 -30\nright_child=2 6 12 9 10 -7 17 13 -10 23 -12 14 -14 -15 -16 -17 -18 -19 19 -21 -22 -23 -24 -25 -26 26 27 -29 29 -31\nleaf_value=-0.030509744347172733 -0.019779201961339997 0.017179984891536149 0.030877900649624635 -0.011509803420080272 -0.029225263986806378 0.017921904404367986 0.01024664857535079 0.02176941010633348 -0.028513056429034012 0.016215347836177278 0.013121155485536792 0.014836944206658157 0.0047912989008231134 -0.040209348373753687 -0.02762264804212235 0.013290722595642948 0.01818851631552126 0.027468447950146056 -0.040542117274660443 0.0042056035494912325 -0.031806344035471515 -0.023989928611798546 -0.029908733535939952 0.030761714735858568 0.021657034221048165 0.010995074995079242 -0.035812021376462985 0.0099308584772825042 0.0012847753457167566 -0.031733919042235929\nleaf_weight=732.77666904419311 67.769466769997962 6.3880691028898573 317.65074772131629 22.037771384057123 20.027221627708059 15.989606556249781 74.063814485445619 44.68556937307585 70.882997203676496 207.63188589643687 262.06331525021233 80.411938092554919 30.566332597285509 5.2180777008179566 13.863320675329303 26.955739645520225 12.504688177141359 43.332577012479305 10.959394228179006 34.480846069636755 28.483714189147577 12.295379555784164 4.0368227898143223 67.979445893783122 263.27087347907946 31.748523148242384 16.137413451448086 6.7034825589507809 22.547664828598499 16.835710708954139\nleaf_count=195564 8922 251 332 585 2617 101 1844 657 12651 839 2475 1262 86 212 995 743 309 94 567 667 6469 1712 439 119 589 251 235 60 1860 2641\ninternal_value=4.47681e-05 -0.0228176 0.0135159 0.00935641 0.00167993 -0.028662 0.00517688 -0.00943453 -0.0155914 0.0174952 0.00780232 -0.000950754 0.0286083 0.0108095 0.00328108 -0.00686523 -0.0137138 0.0127624 0.00687622 -0.00722737 -0.0226039 0.00515341 0.0169723 0.0185898 0.0169598 0.0121794 -0.00318677 -0.0225337 -0.0296486 -0.0132664\ninternal_weight=2570.3 952.713 1617.59 1269.37 653.859 788.15 164.564 232.053 178.113 615.509 421.806 159.743 348.217 53.9405 139.715 107.23 80.2742 129.692 125.852 45.4402 34.8718 86.3592 48.7224 593.472 525.492 262.221 54.5894 22.8409 772.16 39.3834\ninternal_count=246148 210536 35612 35194 32516 200166 10370 23933 22625 2678 8583 6108 418 1308 3491 9974 9231 3650 2496 1234 6720 3556 1096 2093 1974 1385 546 295 200065 4501\nis_linear=0\nshrinkage=0.03\n\n\nTree=51\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 4 13 18 5 18 3 2 14 18 7 22 14 3 11 14 18 19 17 17 19 14 18 4 3 2 8 0 18\nsplit_gain=605.31 266.983 111.277 106.606 128.102 58.8862 49.2559 37.6937 35.1512 30.4032 25.3791 20.9045 20.4332 20.3782 19.9773 19.7344 16.4103 15.7088 15.6446 14.6671 12.6258 12.4975 11.8375 11.7093 10.8089 10.5933 10.3196 9.73776 18.0019 13.7276\nthreshold=0.26802264518526847 1.0000000180025095e-35 8.0136718750000018 0.14796868535883259 3.0677824455408698 3.4169921875000004 2.9174053685313028 0.37248792739977959 -25.182852745056149 0.51224307552198323 1.2424533248940002 0.89599609375000011 0.93750000000000011 0.27639659288032453 0.42638606281916264 0.43728734921738482 0.58665773875949134 6.0438190738591979 262.50000000000006 2.6250469244966204 0.2328254379290535 258.50000000000006 0.088129815934729419 3.2769667020129059 1.0693359375000002 0.53650715206675215 -58.62553405761718 1.0000000180025095e-35 0.13844441961022361 4.8559213341406453\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 4 10 7 13 -6 -4 22 15 14 -5 19 -12 -2 -10 23 -17 -1 -14 -11 -9 -7 -16 -23 -24 -8 -29 -30\nright_child=3 -3 8 12 5 17 27 9 16 21 11 -13 20 -15 24 18 -18 -19 -20 -21 -22 25 26 -25 -26 -27 -28 28 29 -31\nleaf_value=0.011596825359795964 0.022545889664089481 0.025933079351146682 -0.030123740516437035 0.030801642912200999 -0.028151273881885094 -0.036880218013447701 -0.030349821681625835 -0.033262258822232278 0.026125355339760344 -0.02713806626337632 -0.016044814508907951 0.015279341197709661 -0.032907349839547101 -0.019542491092024859 0.014497135752198232 -0.046896888833877999 -0.003062088420365385 -0.020109635397093401 0.02717519900376603 -0.030223222969948699 0.014072233043520189 -0.022002150102668504 -0.035382996928047333 0.017683509132561431 -0.0052503800138494795 0.010253655774328354 0.010687980280683313 -0.03154992633810734 -0.021428662704544476 0.027506370276593296\nleaf_weight=46.18251808732748 340.17497137770988 112.45656889281236 16.865983571682591 271.67618471337482 95.031345629366115 2.7545683735515913 660.75145595776848 5.5024600562173864 51.715265589649789 52.095893093908671 41.351313653140096 284.75332459429046 5.7199022006243494 67.758954158227425 75.811914499034174 5.8962311176583153 25.544267356221098 11.255522704916073 3.2655236199498177 8.0048296218155865 21.100880382582545 15.32568676152732 3.6275059909094116 103.36792813299689 36.312000039033592 19.963959578773938 92.96762936207233 24.867687712598126 8.147583152342122 10.924766141688451\nleaf_count=1608 786 413 4225 297 19439 75 179259 377 356 7584 2577 1763 24 8408 775 61 1011 1552 5 1112 60 2057 239 2189 748 516 2332 5287 930 145\ninternal_value=9.40187e-05 -0.0175368 -0.0228683 0.0123402 0.00830374 -0.00546944 -0.0263846 -0.0128339 0.00796562 -0.00504433 0.0153302 0.0104826 0.0283434 -0.00869248 0.00154155 0.021364 0.0165593 0.012339 -0.0204192 0.00498325 0.00323048 -0.0179346 0.00605545 0.0159362 0.00811191 -0.00397477 0.00861302 -0.0294295 -0.0152233 0.00657554\ninternal_weight=2521.17 1033.22 920.763 1487.95 1189.46 401.892 826.638 284.514 94.1255 189.483 787.565 438.229 298.497 121.946 153.475 349.337 77.2595 117.378 9.16175 54.1873 26.8208 87.3855 102.098 106.122 112.124 35.2896 96.5951 704.691 43.94 19.0723\ninternal_count=246210 202754 202341 43456 43075 36360 196749 32544 5592 13105 6715 5863 381 11128 4100 852 1367 3816 66 2720 84 10157 2948 2264 1523 2573 2571 185621 6362 1075\nis_linear=0\nshrinkage=0.03\n\n\nTree=52\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 16 11 1 7 2 22 7 2 17 18 7 18 7 18 2 18 18 12 9 2 7 12 17 4 12 12 4 4\nsplit_gain=824.791 201.611 83.0108 78.1164 94.3135 32.108 31.7605 29.3969 28.9719 23.1194 22.9021 20.0686 27.1723 19.0201 22.9234 18.3747 17.3191 16.4816 15.7406 15.3222 14.0845 21.8433 12.6783 11.4115 11.3162 11.1141 13.8553 10.9144 10.8872 16.1282\nthreshold=1.211689956917019 0.60365083802632757 1.0000000180025095e-35 0.042545688445982228 2.3713805701640611 0.89599609375000011 -25.182852745056149 1.0000000180025095e-35 1.0000000180025095e-35 -8.7471351623535138 4.4301796584464421 3.2769667020129059 2.6015625000000004 1.7005986908310777 2.9707031250000004 4.6200466650627661 -47.402034759521477 4.2696729980286863 2.3502401828962087 0.063354284020036747 1.0000000180025095e-35 23.600493431091312 2.5244140625000004 0.86223072954082747 0.54218531936492143 6.9193702936172494 0.61117045150363192 0.29026868663126176 6.3295898437500009 2.3505859375000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 11 4 5 18 -4 -5 -6 -9 13 12 27 -10 15 16 -15 28 24 20 21 -8 -19 25 -2 -3 -27 -1 29 -7\nright_child=3 23 6 7 8 17 19 9 10 -11 -12 -13 -14 14 -16 -17 -18 22 -20 -21 -22 -23 -24 -25 -26 26 -28 -29 -30 -31\nleaf_value=-0.024343745045426402 -0.025077341166462486 0.019406187078571512 -0.030811371323058178 0.0274359469607071 -0.010151277159050674 0.017105876774263198 -0.026353169677015325 -0.039785317226854522 0.022414143186466364 0.012555857577202576 -0.034972588157770108 -0.03071007224568862 0.022571380139006249 -0.036781397600852583 0.030042104508880969 -0.019006193755905874 0.012165516184498034 -0.027674977486246699 -0.023883256889093023 -0.034978539042737788 0.021210656447042838 0.014961069204784218 0.0055643172206707118 0.0297475028862936 0.0044905952001754942 0.019771674457509769 -0.044082265008439939 0.020969313807951376 0.0244483761106596 -0.0055780758461487735\nleaf_weight=92.339042189792963 14.694509415188802 21.268322228454057 26.099166277912445 403.60695010027848 36.042408481938764 50.430650175258052 18.858009896357544 8.6915297398809326 369.09504005964845 34.122978495201096 6.603439280181191 667.38694195114658 13.170743041759122 5.7358525192830703 54.002799661830068 19.065607878030278 243.74645916000009 21.320596467063297 74.743313547573052 6.6396811229642472 44.015809059143066 26.441932176239789 18.69628193648532 57.681126612704247 44.367309664143249 4.6479597743600589 5.9835307849571109 4.3055359807331106 31.595180663047358 61.685256607132033\nleaf_count=17688 458 234 6653 1022 1388 761 2503 448 563 99 411 191386 241 196 74 833 1737 3160 11586 423 506 487 601 77 815 20 119 59 152 1582\ninternal_value=0.000395689 -0.0208578 -0.0251344 0.0144275 0.00994866 -0.00370439 -0.00211397 0.0249344 0.0158497 0.00126484 0.0172044 -0.0287423 -0.0166143 0.0177524 0.0123651 0.00871215 0.0108978 0.00443288 -0.0148277 0.0059288 0.00932345 -0.00243544 -0.0123882 0.0218763 -0.00316092 0.00717217 -0.0168663 -0.022131 0.00917001 0.00468637\ninternal_weight=2487.08 988.838 899.257 1498.25 1051.82 317.533 122.055 446.421 734.292 42.8145 698.249 777.202 109.815 691.646 322.551 268.548 249.482 183.728 133.805 95.9554 89.3158 45.2999 40.0169 89.5809 59.0618 31.8998 10.6315 96.6446 143.711 112.116\ninternal_count=246282 220396 219946 25886 24317 19115 10572 1569 5202 547 3814 209374 17988 3403 2840 2766 1933 6256 12859 3919 3496 2990 3761 450 1273 373 139 17747 2495 2343\nis_linear=0\nshrinkage=0.03\n\n\nTree=53\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 12 8 7 0 11 0 6 2 14 14 0 12 2 7 2 7 21 14 14 0 2 19 10 4 7 3 12 17 3\nsplit_gain=550.862 262.689 96.6487 67.1286 62.0894 48.9386 75.5198 40.6015 35.5049 27.2423 22.9954 24.064 22.2695 19.9997 17.3603 19.4281 17.1099 16.7734 16.6911 16.5838 16.2714 13.0103 12.4952 12.2763 12.2521 23.5716 14.1381 15.1378 12.1364 11.2916\nthreshold=0.28904431203898151 0.27402411187416448 1.0000000180025095e-35 1.0000000180025095e-35 1.1851094193663554 0.74021522701736098 0.52258841453381233 4.6093750000000009 -25.182852745056149 0.77909152491212919 0.66452579377975118 0.7384456241152747 0.47784065692827282 -43.562467575073235 3.1982421875000004 -1.7340650558471677 1.0000000180025095e-35 2.5000000000000004 0.85700205199046653 0.51224307552198323 0.94290057586474996 -28.340197563171383 234.50000000000003 2.9174053685313028 3.8073730468750004 1.4980468750000002 0.53650715206675215 0.28697820620240239 2.6250469244966204 0.36709515926886871\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 12 14 6 7 19 -4 -10 24 -12 20 -3 -1 -16 -6 18 -13 21 -2 -5 -19 -15 25 26 27 29 -18 -8\nright_child=3 13 8 5 16 -7 10 -9 9 -11 11 17 -14 23 15 -17 28 22 -20 -21 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.030414442879081082 -0.030210144102594087 -0.028889185572659629 -0.03055311978931665 -0.026251288839859284 -0.031968018832238612 0.026463251465355428 -0.0027095725293114449 0.03056553219740054 0.015688108609455683 -0.022876378179378778 -0.028333473785589931 0.006047956802246221 0.023411709902754253 0.026212037407903534 -0.02424324797132324 0.026996274046537073 0.0086292415921346566 -0.018847675049797237 -0.029672061154746302 -0.023762168722086391 -0.0030976834020424006 0.0063592583834049228 0.01600358865111496 -0.024083236863148241 0.026184503830796752 -0.023242485072024002 0.022833024385011558 0.026768792350008837 -0.030342915372458448 0.012637207917225404\nleaf_weight=663.69955772886169 43.470511625753716 5.8000822213944039 24.692202351463493 13.837408677587517 13.122979838924946 237.55403410946019 54.627196872519562 19.897253143135458 88.649034303613007 19.275541626731865 15.616774827823972 35.684645699569955 12.759250209201126 126.52375277632382 17.029512404842539 9.5886039562756178 61.963825403567171 9.0354333461727929 16.38904801174067 73.655890809139237 36.546390973002417 41.752127693034708 152.02100630779751 3.7773082260973743 123.03501463425346 12.902107127360066 198.62743757013232 54.14276680490002 7.1336856882553539 198.88079393084627\nleaf_count=186986 8464 1425 6473 1407 1981 413 1741 60 1576 2075 2191 635 35 894 4423 242 2075 361 836 12542 1597 1927 1488 58 482 225 583 117 961 1668\ninternal_value=-0.000243482 -0.0166311 -0.0224741 0.0123952 -0.0265309 0.0142127 0.0113368 -0.00834459 0.00117961 0.00865363 0.0147127 0.00650641 -0.0121314 0.0221089 -0.0294622 -0.00547615 -0.00186196 0.00917965 -0.00554138 -0.0145235 -0.0180181 -0.00207864 0.0139442 0.0245817 0.017621 0.0155452 0.0165775 0.0124779 0.00423025 0.00933255\ninternal_weight=2391.69 1041.26 905.155 1350.44 772.538 1257.66 1020.1 149.143 132.617 107.925 870.962 228.747 92.7762 136.101 690.318 26.6181 82.2205 213.13 52.0737 129.245 80.0169 55.5895 161.056 130.301 642.215 519.18 506.278 307.651 69.0975 253.508\ninternal_count=245941 209169 206792 36772 196668 26676 26263 15936 10124 3651 10327 5511 10096 2377 191651 4665 5017 3320 1471 15876 10061 3334 1849 952 4816 4334 4109 3526 3036 3409\nis_linear=0\nshrinkage=0.03\n\n\nTree=54\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 13 18 2 8 7 14 7 2 18 7 11 14 2 4 0 21 17 19 7 2 18 9 13 6 6 5\nsplit_gain=746.818 185.475 102.022 66.9069 51.2606 45.1432 42.1891 26.238 24.3662 19.0105 16.5407 16.3429 14.7855 27.0791 13.7859 21.068 13.2086 12.4432 18.7961 12.9806 12.0945 11.3441 10.2816 9.57062 9.48619 11.9677 10.7342 8.44803 9.27451 20.7296\nthreshold=1.211689956917019 1.0000000180025095e-35 0.14796868535883259 3.3033179733375269 0.14796868535883259 3.0201273556387074 -30.420168876647946 1.0000000180025095e-35 0.89599609375000011 0.67646516896599407 1.0000000180025095e-35 -24.354135513305661 3.5972184255501678 2.6015625000000004 0.86534941175790137 0.84146630540292722 -43.067142486572259 5.6752929687500009 1.3771184689138856 5.5000000000000009 3.9281938047451082 238.50000000000003 2.4414062500000004 24.399772644042972 1.2424533248940002 1.0000000180025095e-35 1.0000000180025095e-35 6.8105468750000009 6.0937500000000009 5.4785156250000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 5 12 10 -3 8 -7 -9 -2 -10 13 -1 15 27 -12 18 19 -5 22 -13 24 -15 -18 26 -26 28 29 -8\nright_child=2 6 -4 17 -6 7 14 9 11 -11 16 21 -14 23 -16 -17 20 -19 -20 -21 -22 -23 -24 -25 25 -27 -28 -29 -30 -31\nleaf_value=-0.024405753422712614 -0.010389261359373909 -0.030462727611356034 0.029110734307766846 0.012561046179389789 0.021737316956233231 -0.02372890153041007 0.013739395822319778 0.025003087087600497 -0.023548218790113606 -0.0309884651027869 -0.036562835570999153 -0.016462285443296431 -0.030571207978700943 0.02451714504683607 0.028165363245323084 -0.02543156572002549 0.017287585997502422 0.031035076924196391 0.021747798935183325 -0.0079244053049385196 -0.031817966322489288 0.0082757791755058811 0.024630946352906025 -0.031434252984472402 0.0015575241657576849 0.033104232398702929 0.026978869055213479 -0.03187374285366544 0.034419532968132653 -0.041345942296567038\nleaf_weight=117.27338234390481 34.087007864232874 22.76575986694661 302.70302086137235 159.07062183140079 18.022109061246738 82.990955626883078 77.884747625968885 31.411019719089381 25.610411710862536 5.6756459567695847 4.1855480817612252 20.283545536221936 603.24642420312739 14.698406145791525 37.211527304258198 14.492600155528633 110.17446084599942 57.264893997460604 234.13410582067445 32.98233520728536 4.6806562655838198 76.49518348131096 53.563138612546027 2.4403381408192208 157.41130810126197 12.753103899303822 15.536671191453932 2.8708433087449512 15.465590446954591 5.7029574878979465\nleaf_count=25621 1624 6299 406 977 53 11982 2036 290 3067 502 138 1031 183479 254 55 2189 312 81 395 304 294 2243 153 150 2075 22 25 80 20 142\ninternal_value=-9.66924e-07 -0.0208564 0.0136836 0.00948553 -0.0273152 0.00307299 0.0068115 -0.00708588 -0.0112835 0.0158456 0.00935241 -0.00272042 -0.0285414 -0.0192731 0.0124963 0.00729067 0.0112542 0.0178826 0.0160469 0.00902014 0.0119208 0.00297567 0.0125957 0.0152566 0.0103414 0.0061497 0.0039832 0.0121453 0.013727 0.00953671\ninternal_weight=2353.09 932.075 1421.01 1118.31 755.681 634.859 176.394 242.467 205.38 37.0867 392.392 122.389 737.659 134.412 153.628 116.417 358.305 483.452 426.187 192.053 354.119 96.7787 349.439 17.1387 295.876 185.701 172.948 101.924 99.0533 83.5877\ninternal_count=246299 220378 25921 25515 209557 23758 10821 19115 18323 792 4643 6341 209504 26025 4522 4467 3019 1757 1676 1281 2881 3274 2587 404 2434 2122 2100 2278 2198 2178\nis_linear=0\nshrinkage=0.03\n\n\nTree=55\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 0 19 4 18 3 4 7 3 9 18 12 14 18 17 3 4 14 12 6 5 17 4 14 3 7 4\nsplit_gain=724.689 159.141 117.506 89.8514 40.348 33.9511 30.53 26.4498 21.5802 23.0755 21.434 27.2834 16.697 16.0425 14.7888 19.6302 14.31 10.985 10.9528 20.8815 12.8465 11.108 10.2847 9.90884 10.4551 9.41745 9.05086 9.32028 8.97337 12.0112\nthreshold=0.95935769349451683 1.0000000180025095e-35 1.0000000180025095e-35 2.5144622974930591 1.5447487307370646 238.50000000000003 1.6286621093750002 0.89587973461402759 0.38657275885365588 5.9760742187500009 2.9707031250000004 0.40160889672495398 1.0000000180025095e-35 3.0201273556387074 0.88192242588002812 0.8755209476635436 5.2755571088971331 2.5380284121401901 0.47260169401963864 2.7826617956161503 0.85380319876708544 0.61950675842039238 0.89599609375000011 3.0205078125000004 0.12494407617242172 0.84667968750000011 0.54439270847295596 0.62297257604742995 1.0000000180025095e-35 1.1459960937500002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 3 4 6 14 28 -5 -1 -10 11 13 16 -9 15 22 18 25 19 -8 -21 -20 -3 24 -6 -13 -27 -28 -2 -30\nright_child=2 5 -4 7 23 -7 12 10 9 -11 -12 17 -14 -15 -16 -17 -18 -19 21 20 -22 -23 -24 -25 -26 26 27 -29 29 -31\nleaf_value=-0.029924840755216991 -0.032319007409354797 -0.0077163643650300451 0.027287282051022065 0.024871483370320861 -0.034767612340782676 -0.026799853993675286 -0.020358781976403367 0.0023917847448816716 -0.025942503533717021 0.022811994507077521 0.028336635441649681 -0.015287147380970249 0.018906715373654424 -0.025537866996684592 0.028432062114714322 -0.02573752653744792 -0.024503676283125553 -0.022720776516108945 0.0098292633983346166 0.0089179868910699255 -0.030941484850452301 -0.040095070193634827 0.016839927079334349 0.029662793858617639 0.01166453259764862 0.018155871991278084 0.0016197753804913274 0.017583788501796843 -0.0011407785383755009 -0.027670529645360802\nleaf_weight=641.04241537561757 39.064321706886403 19.571378339431249 367.56004381668754 170.63568351231515 3.7681636456400147 24.222265429096296 56.902884646173334 58.318468447367195 19.345916202291846 14.345694167888722 56.877242214977741 9.3201589901000244 30.185711912345141 26.014330638106912 32.524070544168353 14.588896601868326 27.209825405239826 6.9287629823666057 73.132670611375943 59.680832377169281 7.2829233986558384 3.263432875275611 62.659221279784106 28.584019728004932 55.143865172984079 200.73953184881248 82.098539255443029 54.197938818950206 32.84341229259735 27.999975630169502\nleaf_count=195312 7932 1277 647 232 134 5785 6302 487 4372 447 92 366 396 1358 45 2327 4009 184 1600 2003 816 67 941 91 587 669 895 204 2691 3721\ninternal_value=1.70336e-05 -0.0224304 0.0126058 0.00772486 -0.00270612 0.00510634 -0.00717792 0.0146982 -0.0286959 -0.00512395 0.0111445 0.00885058 -0.00169283 -0.00642205 0.0112412 0.00523475 -0.00449719 0.0125042 -0.00167065 -0.00715938 0.00419142 0.00720614 0.0109729 0.0154432 0.00816387 0.0132722 0.0141048 0.00804966 -0.0211668 -0.0135997\ninternal_weight=2306.05 828.3 1477.75 1110.19 445.062 153.566 357.566 665.131 674.734 33.6916 494.495 437.618 257.658 84.3328 129.344 96.8195 227.473 353.285 200.263 123.867 66.9638 76.3961 82.2306 87.496 58.912 346.356 337.036 136.296 99.9077 60.8434\ninternal_count=245989 210506 35483 34836 30349 10375 29537 4487 200131 4819 4255 4163 15193 1845 4590 4545 14797 2318 10788 9121 2819 1667 2218 812 721 2134 1768 1099 14344 6412\nis_linear=0\nshrinkage=0.03\n\n\nTree=56\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 0 13 18 16 6 3 5 14 2 11 3 14 14 2 14 13 19 14 2 6 7 3 18 17 17 5 5 6\nsplit_gain=494.997 229.098 91.0005 84.8828 74.8511 42.1824 41.8193 38.1871 30.3787 25.0093 19.7637 14.9728 13.8442 19.9114 12.8144 12.55 12.2043 11.9595 11.3108 11.444 9.61983 9.41555 9.19138 13.7348 10.9101 9.91584 8.22026 7.6295 16.8764 15.7753\nthreshold=0.28287969384316608 1.0000000180025095e-35 0.94290057586474996 0.14796868535883259 2.249904835165133 1.0000000180025095e-35 4.3378906250000009 0.44825707527455844 2.5244140625000004 0.4896816566336385 -25.182852745056149 0.75683518236239655 0.53650715206675215 0.91125284794847328 0.70873219185162306 -39.733196258544915 0.87268561280719448 -0.56995612852957833 254.50000000000003 0.075876460218247857 -25.182852745056149 3.0683593750000004 0.89599609375000011 0.37248792739977959 4.2835627800822236 0.70050815440916869 2.6250469244966204 4.4589843750000009 3.1982421875000004 1.8691406250000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 4 12 21 7 9 14 15 -7 -11 13 -2 18 -6 -12 -9 19 26 -3 -1 23 -17 -24 -25 -4 28 -19 -29\nright_child=3 20 8 -5 6 10 -8 17 -10 11 16 -13 -14 -15 -16 22 -18 27 -20 -21 -22 -23 24 25 -26 -27 -28 29 -30 -31\nleaf_value=-0.030263932553729739 0.012257633116187014 -0.016999311375736594 0.012938545075550291 0.027483902112791427 -0.032006598893429175 -0.030361979189306363 0.031603081345311403 -0.0094767140666794959 0.017051235835051855 -0.024655780003461327 0.0075547896568941789 0.018027005513565605 0.020551394679608936 -0.035301399185705677 -0.031200914467653395 -0.032369185925364981 -0.028953519610874955 0.0048912816402615755 0.015254022467572894 -0.02086840677714704 0.026036369712653922 0.0071990529222230445 0.015590281939002925 0.019470666573549584 -0.019341782407647345 -0.021255378853834314 -0.028019324262204645 -0.04614415456394412 0.026606766982856363 0.019024781602632348\nleaf_weight=584.59648909370298 293.42545538733248 4.2396429767832151 20.459065128408842 245.18894061422907 9.6849328828393464 22.580956161546055 32.972603569040075 41.841636749217287 51.446646447206149 105.89762367139338 52.341097702970728 7.2455054409801951 258.01260110596195 7.1982713171746573 21.14588553190697 13.316641935787628 8.7505041811964457 159.67432191141415 17.739432241825853 40.392717675043968 102.73526010720525 5.7501773173571555 46.977283197804354 14.281328919925729 8.7812563809566182 7.3435077667527358 4.5708026399952351 5.831954173743723 39.572129325009882 5.7342830244451761\nleaf_count=184413 1861 208 827 365 1360 6472 83 3746 1027 20545 1593 30 694 323 4222 2231 1518 4695 898 4650 234 556 519 318 831 470 754 105 285 41\ninternal_value=-0.00034323 -0.0169666 -0.0222952 0.0116116 0.00791005 -0.0270646 -0.000534879 -0.00287074 -0.00161451 -0.0121822 -0.00703056 -0.0217885 0.0154433 0.0110056 -0.0108953 -0.00124694 0.00194341 0.00501573 -0.00542275 -0.0111745 0.0241891 -0.0298875 0.00235079 -0.00954955 0.00984508 0.00531245 0.00466998 0.00792332 0.0093155 -0.014892\ninternal_weight=2239.73 936.749 829.774 1302.98 1057.79 674.019 499.155 466.182 155.755 213.528 83.6726 113.143 558.636 300.624 104.308 100.385 61.0916 252.654 83.162 65.4226 106.975 590.347 90.7 34.9415 55.7585 21.6248 25.0299 210.813 199.246 11.5662\ninternal_count=245874 207372 206930 38502 38137 194552 35259 35176 12378 26304 9583 20575 2878 2184 11351 5729 3111 8872 7129 6231 442 184969 4369 3019 1350 788 1581 5126 4980 146\nis_linear=0\nshrinkage=0.03\n\n\nTree=57\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 13 18 2 9 4 18 0 5 12 14 5 2 7 2 19 2 5 18 1 18 14 1 17 19 19 17\nsplit_gain=700.946 153.161 66.4941 55.9073 46.4809 36.8267 35 25.2002 19.8779 18.905 17.7979 20.1842 15.7093 17.0554 15.0518 13.6109 13.98 16.0623 15.0585 12.3063 11.5441 14.8921 10.6742 10.1582 12.1247 10.126 16.4006 11.7833 10.0914 11.7176\nthreshold=1.3779212206518803 1.0000000180025095e-35 0.14796868535883259 3.3033179733375269 0.14796868535883259 1.8688348091416842 -30.420168876647946 1.0000000180025095e-35 0.84667968750000011 5.2755571088971331 0.83245446090221253 4.1015625000000009 0.86223072954082747 0.87268561280719448 3.1982421875000004 32.956295013427741 1.4980468750000002 0.82504272460937511 263.50000000000006 -30.420168876647946 3.7255859375000004 1.7005986908310777 2.3713805701640611 3.1984648276080736 0.22768215226658742 4.0016611463566001 0.93623644617438528 253.50000000000003 237.50000000000003 3.5112638802509566\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 5 10 -2 -3 8 -7 14 -1 -12 13 -8 15 16 17 18 19 -10 21 -5 -20 -21 -25 26 -23 -28 29 -16\nright_child=2 6 -4 20 -6 7 12 -9 9 -11 11 -13 -14 -15 28 -17 -18 -19 22 23 -22 25 -24 24 -26 -27 27 -29 -30 -31\nleaf_value=-0.029743062278361921 0.013715051141397143 -0.030169022518488416 0.027425023545686374 0.021306310585180486 0.020420147554470122 -0.031819039916502492 0.0096712192173471767 0.02139154215042351 -0.031676862124447175 -0.029439977300058758 -0.021282282348165243 0.0161929950018154 0.028680412085191485 -0.025096637855896137 -0.030046693213038028 -0.029232555698372294 -0.029883633602812199 0.01028939483697692 -0.019646480834752206 0.0058724836370746434 0.031018912442623905 -0.018033236810380333 0.021051797393978187 0.012938557881513338 -0.030674740901962819 0.015528763533523341 -0.0042989767949807647 0.026884774203550871 0.022354117491236933 0.018993040081667491\nleaf_weight=592.75904590782011 232.70846309023909 20.638028697983827 264.71009518578649 241.05094141559675 17.875902316300198 21.07177308970131 98.779248754202854 37.348101121024229 22.998844952497166 20.452111095626606 94.656399861996761 14.208422372583298 32.804079673718661 13.580182171426712 7.8302754652686399 16.229848353425041 13.521315169928128 85.743010636651888 6.8363388029392835 30.466788566322066 57.129888828843832 25.427676558494568 22.449236185057089 7.2577762304572415 19.907551060779952 102.44662326527759 20.541986213997006 21.686218719463795 37.256419505923986 7.7662271754816175\nleaf_count=195215 1289 6296 390 391 60 3590 2471 235 2502 2264 18831 275 52 2179 176 1170 774 2702 615 579 85 337 150 282 2393 537 305 69 178 75\ninternal_value=0.000171531 -0.0204767 0.0139952 0.0106128 -0.0264598 0.00445609 0.0054903 -0.00159847 -0.0043418 -0.00230382 -0.0276816 -0.0162883 0.0107317 0.00528473 -0.000211061 -0.00360101 -0.0014827 0.000621965 -0.00694412 -0.0135967 0.0183417 0.0165092 0.0112163 -0.00596434 -0.0189769 0.00962655 0.000593231 0.0119668 0.014045 -0.00595446\ninternal_weight=2208.14 885.301 1322.84 1058.13 719.5 589.844 165.802 357.136 319.788 298.716 701.624 108.865 145.164 112.359 278.264 225.411 209.181 195.66 109.917 80.631 468.283 411.153 29.2856 57.6321 27.1653 170.103 67.6559 42.2282 52.8529 15.5965\ninternal_count=246467 225379 21088 20698 214381 18974 10998 17685 17450 13860 214321 19106 4702 4650 11596 11167 9997 9223 6521 5756 1724 1639 765 3254 2675 1248 711 374 429 251\nis_linear=0\nshrinkage=0.03\n\n\nTree=58\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 13 1 4 4 18 7 19 10 17 0 11 18 17 14 3 17 18 5 18 7 14 11 12 0 3 5 14 18\nsplit_gain=624.604 130.149 94.61 81.8229 63.9313 34.2821 27.4914 21.7796 23.5544 20.5561 19.4475 17.4053 16.6616 15.0957 14.9143 14.5337 18.2347 13.4269 10.9782 10.8337 9.30692 8.95677 8.32354 12.7962 11.8209 10.2127 9.51345 9.51741 7.98438 7.98039\nthreshold=0.95935769349451683 0.62517033806334033 0.14796868535883259 2.1468275719512682 11.379394531250002 3.4438476562500004 1.2424533248940002 2.5244140625000004 244.50000000000003 0.89587973461402759 1.490403556610195 1.3170161875856263 0.027600239276592001 1.4978661367769956 3.325091732892195 0.11320561501600769 0.47260169401963864 3.9281938047451082 3.1984648276080736 2.7412109375000004 5.2390944889447963 1.0000000180025095e-35 0.71043588483438491 0.91125284794847328 2.2204460492503136e-16 0.45494749303197096 0.53650715206675215 4.0332031250000009 0.032916404154818342 2.249904835165133\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 5 9 10 -5 8 18 13 11 15 -6 -1 20 -2 -17 22 19 -8 21 -10 25 24 26 -7 -24 -28 -11 -3\nright_child=2 29 -4 6 12 17 7 -9 14 28 -12 -13 -14 -15 -16 16 -18 -19 -20 -21 -22 -23 23 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.0046598181393282569 0.012852120980674875 -0.0021320740823864972 0.02723782266643749 0.020605265413420087 0.017885535367327662 -0.0045619247944050957 0.0031333888028773379 0.024118223249490485 -0.0080124806397471049 0.026185585217084688 -0.029523281452805191 0.0084911317784405593 -0.032400810517843083 -0.02999078699336126 -0.032658443720503659 -0.027357847800013678 -0.001404733457844758 -0.030968630638367484 -0.023794656479233171 -0.041843971264066547 -0.029018771644192896 0.01261542348223778 -0.014935650623787856 0.026950024268424317 -0.041345481072671039 0.021042268955123511 -0.028912620195475004 0.033032473605758776 -0.011672091775395507 0.024986145874180164\nleaf_weight=22.34340658452129 15.822825204464605 11.973084758967159 277.42396433814429 278.07681286241859 35.596706622454803 17.598887304076925 55.822997880575713 69.917247312841937 19.571641922579147 6.7195979445241418 44.579505630303174 44.887559102498926 6.1308716683415687 615.9286351891933 6.3575204547378226 62.704706199729117 39.63773475389462 6.7907211908604941 25.7979262394947 4.266805187566205 4.3045640499331048 297.36111361882649 15.39975455531385 10.304879464209078 7.17332466575317 62.369939613854513 2.0364231388084582 9.0181720438413304 13.320139045390532 48.635477971984074\nleaf_count=1581 740 73 423 524 709 1232 634 214 986 62 7972 749 686 205325 540 11794 2074 949 1525 66 387 2533 1297 12 238 525 106 53 2073 243\ninternal_value=-0.000438517 -0.0222411 0.0116106 0.00764744 -0.0258936 -0.00461935 0.0130965 0.00874344 0.00609174 -0.0281748 -0.0121732 -0.00727852 0.00991176 -0.0291127 0.00975336 -0.0132899 -0.0174169 0.00738285 -0.007826 -0.000693004 0.0106915 0.0113204 0.00971591 -0.000780508 -0.00981114 0.0154208 -6.58165e-05 0.0201694 0.00166334 0.0196814\ninternal_weight=2137.87 760.648 1377.23 1099.8 700.039 338.324 761.477 483.4 413.483 658.312 207.632 163.053 41.7276 638.272 327.595 118.165 102.342 130.692 85.8877 60.0898 321.237 316.933 123.901 43.9326 33.6277 79.9688 26.4543 11.0546 20.0397 60.6086\ninternal_count=246325 210752 35573 35150 210436 27741 7409 6885 6671 209041 23329 15357 1395 206906 4446 14608 13868 4412 2225 700 3906 3519 3463 1706 1694 1757 1456 159 2135 316\nis_linear=0\nshrinkage=0.03\n\n\nTree=59\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 7 13 0 0 7 3 22 2 4 14 6 16 0 17 14 22 11 11 4 3 4 2 2 12 5 14 0 14\nsplit_gain=427.593 214.119 79.1854 65.7524 64.4634 50.3039 44.2432 30.7218 30.6902 25.3936 23.5677 19.2914 17.9611 16.5636 17.9417 15.8284 15.285 13.4771 41.248 19.4193 18.3128 12.541 12.3561 11.7303 11.2176 10.9503 10.4153 13.9617 9.85276 9.55647\nthreshold=0.28904431203898151 1.0000000180025095e-35 3.7958984375000004 0.14796868535883259 1.3771184689138856 1.1495945197163422 0.89599609375000011 0.40911314460322756 0.93750000000000011 -34.692844390869134 1.2946246862411501 0.48434979059990024 4.3378906250000009 1.0000000180025095e-35 0.17393699858242903 1.849927262362129 0.72273155396328503 1.0000000180025095e-35 0.74021522701736098 1.0000000180025095e-35 5.5515136718750009 0.53650715206675215 7.0112304687500009 50.658309936523445 -13.633526802062987 0.29026868663126176 1.7006835937500002 0.57916335608139347 0.25265222477800581 0.31851518359289588\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 4 6 13 7 -2 -5 -8 16 -11 17 -1 23 29 -9 18 20 -19 -13 -6 -22 -15 -4 -3 -12 -28 -16 -7\nright_child=3 25 24 8 21 15 9 10 -10 11 26 12 -14 14 28 -17 -18 19 -20 -21 22 -23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.028873671595417795 -0.024896211636734746 -0.0097338989654753103 -0.012580658967165318 0.0301556576296713 0.012133936279450658 0.013402631035208716 -0.017141696820580837 0.013461499312891942 -0.0041463498892936644 0.018701330126994001 -0.026094347522372909 -0.03307717399174994 0.029240921933823008 -0.03245956112411777 0.025844190096121641 -0.025960318481075894 -0.014300023216368602 0.014562101198804481 0.02266634564129575 -0.042372152486118045 0.010475049579593236 0.022982827923914612 -0.025224778836671756 0.010975380279107662 0.021763575487155361 0.026230400182613075 0.016339276910157473 -0.033229795686889212 -0.011129823181100345 -0.0079094484298700297\nleaf_weight=568.57433477224549 70.48361349379411 7.7507257675169958 10.129358860198407 193.20487058861181 203.94916922028642 37.076974347932264 31.973731405771105 80.167203058546875 26.446570008993149 146.31912325439043 35.96079860185273 36.16359733545687 22.213390541728586 30.805023583729053 12.157389335217887 21.338295416266192 21.996837597514968 73.076551288075279 30.720653098542243 4.8542569206329054 22.54008905228693 188.49405592691619 12.880282527999951 6.2726644901558748 39.668440646433737 98.614775174530223 14.727265458786862 6.5782173273619255 11.960867548245004 36.738250667083776\nleaf_count=191669 13738 197 1555 280 1139 983 3393 1920 82 1514 2713 4918 51 7568 183 2638 1998 1132 93 147 1102 528 796 144 326 266 299 947 1026 2575\ninternal_value=-0.000286455 -0.0162089 -0.0216901 0.0111908 0.00791973 -0.0242044 0.00181514 -0.00855389 0.0260255 0.00808257 -0.00121886 0.0104374 0.00440216 -0.0272722 -0.0121675 -0.0038279 0.00740352 0.00119642 -0.00594183 0.0105238 -0.0183623 0.0173899 -0.00284301 -0.0250161 0.0146675 0.0235435 -0.0164644 0.000230864 0.00779473 0.00283155\ninternal_weight=2103.84 881.087 774.722 1222.75 1003.1 724.924 610.656 229.914 219.651 380.742 159.43 348.768 202.449 629.77 61.1959 95.1535 102.164 180.235 102.305 77.9308 71.584 392.443 35.4204 37.0777 49.7978 106.366 57.2663 21.3055 24.1183 73.8152\ninternal_count=245920 209130 208667 36790 36428 206786 34761 21615 362 13146 7877 9753 8239 200590 8921 6196 3918 8188 6909 1279 6816 1667 1898 7712 1881 463 3959 1246 1209 3558\nis_linear=0\nshrinkage=0.03\n\n\nTree=60\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 7 22 11 18 18 7 18 18 7 5 6 3 12 11 7 5 18 18 4 6 5 4 7 0 17 4\nsplit_gain=584.554 131.357 80.8694 83.324 44.7609 39.3652 31.3491 25.6318 22.3461 20.9535 15.5924 17.2213 15.5051 19.7559 14.7215 14.6886 13.3639 11.9222 11.173 12.1242 10.6701 10.594 15.0268 10.0349 16.1768 13.2128 10.3136 9.87669 9.83219 9.656\nthreshold=0.95935769349451683 1.0000000180025095e-35 1.0000000180025095e-35 2.9477339485081675 0.89599609375000011 0.93750000000000011 0.66750229771629133 6.9032449973715968 4.2696729980286863 0.89599609375000011 2.4414009612931857 5.0271592211193568 2.2500000000000004 1.4980468750000002 4.3886718750000009 0.42638606281916264 0.86223072954082747 0.93735200266714214 2.1386718750000004 2.1386718750000004 4.4942230200312032 3.6242520361853052 5.9760742187500009 3.2382812500000004 3.9980468750000004 5.7263183593750009 4.5273437500000009 0.96046990436194013 2.0954123253646428 1.0693359375000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 4 10 -4 21 16 18 15 29 -12 13 -7 17 -5 23 -10 19 -6 -11 22 -1 24 25 27 -25 -3 -21 -2\nright_child=2 7 5 9 8 12 -8 -9 14 20 11 -13 -14 -15 -16 -17 -18 -19 -20 28 -22 -23 -24 26 -26 -27 -28 -29 -30 -31\nleaf_value=-0.022963566081640553 0.0056294790823609719 -0.031457037059749865 0.028687648856513728 -0.0071717409298733227 0.010528186296457093 0.0026179525670447417 0.01691589402806571 -0.024759181741094736 -0.018079617278515817 0.022233870805012393 -0.025420810788377837 0.0042838484039041513 0.033857093492981077 -0.051310325908625963 0.024661538864287201 0.012835318463523026 0.027972362906435917 0.031212969292579891 0.018786631552631935 -0.0174595322322811 -0.0017853510599196323 -0.030815248388184748 0.019726113475660473 0.020006131087920591 -0.037128108789260948 0.018574210070770127 -0.031344586633080353 0.014814581531294837 0.011685375781326834 -0.013812842772187518\nleaf_weight=95.970635483710794 53.204677015193738 10.883498653536664 282.14145502215251 42.223072522669099 121.71690132154617 16.62216808274389 13.175457124598323 21.44973691413179 55.509791673743166 309.16979212523438 86.024073502136162 21.679036941175582 5.8171851634979239 8.5882032543886435 9.2177651948295516 142.07210756349377 27.897063665557653 3.8892298936843863 89.872939488384873 29.564520300016738 17.438314496306702 467.87150763755199 7.2861943819443686 36.711814211681485 9.8119105881778506 22.997986997943372 2.9905092772096387 5.6424343185499302 14.95902407215908 38.978642964968458\nleaf_count=24834 1056 1939 460 700 2214 85 97 6180 7141 736 16280 2513 6 87 32 858 49 3 731 817 327 175209 218 278 736 865 130 174 353 1054\ninternal_value=0.000168413 -0.0215855 0.0118325 0.00777848 -0.000622503 0.02517 -0.0278008 0.00466888 0.00624734 0.0163893 -0.0117586 -0.0194573 -0.0067139 -0.0170101 -0.00886399 0.00823762 0.0102311 -0.0143888 0.0103146 0.00564954 0.0209499 -0.0288602 -0.0197909 0.00442357 -0.0045911 0.00420317 0.0154632 -0.0157147 -0.00762583 -0.00265149\ninternal_weight=2071.38 722.689 1348.69 1035.52 524.617 313.169 584.304 138.385 324.73 510.903 199.886 107.703 31.0276 25.2104 68.6168 184.295 116.935 59.399 256.113 166.24 326.608 571.128 103.257 89.0382 49.3358 39.5239 39.7023 16.5259 44.5235 92.1833\ninternal_count=246162 210709 35453 34815 32194 638 200358 10351 11291 2621 20903 18793 178 172 7176 1558 4171 7144 4115 3384 1063 200261 25052 4122 3714 2978 408 2113 1170 2110\nis_linear=0\nshrinkage=0.03\n\n\nTree=61\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 13 1 18 7 22 0 17 5 0 11 18 4 1 4 10 10 1 18 6 0 14 12 7 21 17 0 18 19\nsplit_gain=576.883 128.959 83.8812 77.2123 47.8638 53.1213 28.9451 26.8762 19.9688 17.169 16.9174 13.4154 13.3471 19.4937 13.1439 11.1198 10.5081 13.1623 15.4653 12.5224 11.3078 10.1625 9.60663 9.43614 8.64387 17.5838 8.38914 8.17212 8.95356 8.07473\nthreshold=0.95935769349451683 1.0000000180025095e-35 1.0000000180025095e-35 4.0016611463566001 1.2424533248940002 0.89599609375000011 0.67500000000000016 -0.7405159439372101 3.9281938047451082 4.8398437500000009 0.31303196667935901 0.93735200266714214 3.6242520361853052 5.9760742187500009 2.5144622974930591 2.7826617956161503 2.524928003624769 1.8688348091416842 0.26032702032866095 2.9700856263602158 1.2426757812500002 0.52258841453381233 0.97076046726095988 0.21199991874952331 2.2500000000000004 3.5000000000000004 0.068176441062887241 0.53713645201195115 2.3502401828962087 265.50000000000006\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 12 3 4 -2 14 -4 11 9 10 22 -3 13 -1 15 26 17 18 19 -9 -20 -16 -7 -12 25 -8 -6 -23 -29 -27\nright_child=2 7 6 -5 5 8 24 16 -10 -11 23 -13 -14 -15 21 -17 -18 -19 20 -21 -22 27 -24 -25 -26 29 -28 28 -30 -31\nleaf_value=-0.022615071114447074 0.016196840300599452 -0.031030562557367216 0.028609316729920512 0.021209552189595075 0.0060904705886953639 -0.021648976273157892 0.013645214532264256 0.012183765799789244 -0.026906898414375942 0.023827339772605455 0.0078270687593621907 0.025077282842068439 -0.030720112382943716 0.020689851368399372 -0.032521433609389153 0.0082518004138146563 -0.025532717762789318 0.031049526510286606 -0.012694215284730444 -0.028813820783055357 0.023348838897777693 0.030972038700482804 0.021329154816502598 -0.022479910183199735 0.031030613981244324 -0.049217220593948598 -0.02464405249736543 0.005413896526716741 -0.015913752752066523 0.0063899426652854014\nleaf_weight=94.465188028465491 176.69982709665783 22.476741926569961 270.63808676064946 272.1543645869242 8.2512745942221937 29.605407974304399 12.501749694347387 10.803895225748422 15.544038638880009 45.527625053771771 260.06150694441749 3.803396224975585 454.02252058257 9.5936025495175254 9.1932760546915215 11.21948172355769 5.7485211826860896 26.707522706361488 9.0270498208701593 14.958532273012677 39.574710305547342 5.7382432641461483 4.6990526597946873 8.6389304273761791 4.5054021142423144 8.5713923859875631 106.76264192987583 41.301044499035925 29.56298554019304 2.835024499334394\nleaf_count=24869 695 6702 451 784 820 4015 49 156 2102 230 4920 4 174909 218 795 688 246 133 940 2000 245 34 170 210 6 110 17971 357 1082 11\ninternal_value=4.35909e-05 -0.0221533 0.0116391 0.00754619 0.00258503 -0.00161074 0.0255952 0.0043552 0.00533895 0.00684993 0.00422902 -0.0222888 -0.0284719 -0.0184637 -0.0135131 -0.0196374 0.0110561 0.0133669 0.00668854 -0.011808 0.0165334 -0.00437996 -0.0153244 0.00677294 -0.00311086 -0.0105322 -0.0224056 -0.000627059 -0.00358118 -0.0359952\ninternal_weight=2015.19 691.182 1324.01 1024.96 752.805 576.106 299.052 133.1 364.077 348.533 303.005 26.2801 558.081 104.059 212.029 126.233 106.82 101.072 74.3642 25.7624 48.6018 85.7955 34.3045 268.7 28.4136 23.9082 115.014 76.6023 70.864 11.4064\ninternal_count=245922 210422 35500 34873 34089 33394 627 10426 11647 9545 9315 6706 199996 25087 21747 19479 3720 3474 3341 2156 1185 2268 4185 5130 176 170 18791 1473 1439 121\nis_linear=0\nshrinkage=0.03\n\n\nTree=62\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 16 13 7 18 0 18 0 5 5 3 9 7 4 7 17 22 7 2 2 6 6 19 4 3 17 12 3 0\nsplit_gain=404.958 177.004 87.2242 63.834 56.7876 55.131 43.7498 27.1081 25.4196 24.9272 14.7956 16.5073 14.7338 14.5614 43.4314 13.7415 13.556 11.3585 11.348 10.7132 14.1057 11.6533 10.4496 10.1297 12.0171 9.77998 9.44748 9.34204 9.47317 12.0215\nthreshold=0.29586597258655462 0.14796868535883259 1.0000000180025095e-35 0.14796868535883259 1.0000000180025095e-35 2.4414009612931857 1.3771184689138856 5.2755571088971331 0.29549219484135919 3.1982421875000004 1.4980468750000002 0.53650715206675215 1.0000000180025095e-35 0.89599609375000011 1.4160156250000002 3.4804687500000004 2.4286772172182522 0.93750000000000011 1.0000000180025095e-35 18.240050315856937 1.0000000180025095e-35 6.0937500000000009 1.0000000180025095e-35 211.50000000000003 4.8474121093750009 0.45898141303402712 0.17669220786083642 0.74643410817732136 0.42638606281916264 1.4460303622695712\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 4 25 16 8 19 12 10 13 22 -6 14 -10 -9 18 -5 -1 20 21 26 -12 24 -21 -2 -4 28 29 -8\nright_child=3 -3 7 17 6 -7 27 15 9 -11 11 -13 -14 -15 -16 -17 -18 -19 -20 23 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.026741476023673435 -0.029843818857613592 0.02694905260017761 0.01850440246454374 0.029657666427593026 -0.021562014909758783 -0.029317635812996731 -0.028757858534437748 -0.032503794869134524 0.013007540453521958 0.019434897929383906 -0.01819611234181865 0.0068171909720988457 0.01287506818212235 0.018379215418160427 -0.024798348719955472 0.0030660664839875602 -0.031879046959800178 0.0049275996838757455 0.0066375223545015079 -0.032253782576237171 -0.037960505289832622 0.027428804217824773 0.011477688551830676 0.0280826688431196 0.0180079329530075 -0.0071258750281979581 -0.024367279739971 -0.01370522781769035 0.021405774602191775 0.012800021579144814\nleaf_weight=9.7110670333204308 32.110118191369111 75.738688555313274 6.5011104152072283 168.58592251455411 46.615120468719397 524.81361970806029 5.9766894443891934 24.403211219439982 92.670682808326092 108.86075386253651 66.239855424326379 64.385812988737598 13.8758747120155 77.166048907034565 37.630502923071617 15.628419694083275 11.561297871172426 19.034205375239253 62.727069072541781 5.7175421034917262 6.7375591644085935 13.287058667745439 11.990039636555592 35.350189039483666 10.804626175668089 36.899418184242677 11.332764338701962 7.4126661401242009 278.50879784184508 65.076390439528041\nleaf_count=1760 6997 158 202 271 8773 194189 107 6658 2190 1294 4637 716 447 1263 3953 1310 1949 70 2734 427 361 53 375 167 130 2528 1177 49 702 353\ninternal_value=-0.000349841 -0.0164759 -0.0209594 0.0112442 0.00805474 -0.025712 0.0101222 0.00136467 0.00455082 0.0069651 0.00304322 -0.00437738 -0.0135817 0.00814077 0.00198363 -0.018881 -0.00310452 0.0271882 0.00185953 0.0105054 -0.0016356 0.00716556 -0.0135606 0.0192297 -0.000233374 -0.0179718 -0.00858625 0.0181939 0.0189075 0.00895374\ninternal_weight=1947.35 814.314 738.576 1133.04 945.419 608.813 876.409 129.762 519.435 458.944 350.083 142.616 60.491 207.467 130.301 40.0316 83.9994 187.62 72.4381 89.7308 37.8585 31.1209 78.2299 51.8724 16.5222 69.0095 17.8339 356.975 349.562 71.0531\ninternal_count=246000 211275 211117 34725 34384 200632 24859 10485 23648 14428 13134 5728 9220 7406 6143 7968 6443 341 4494 2517 1793 1432 5012 724 557 9525 1379 1211 1162 460\nis_linear=0\nshrinkage=0.03\n\n\nTree=63\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 13 6 7 13 18 11 17 9 10 0 7 19 11 0 11 14 11 17 19 18 7 2 2 0 14 7 0 7 18\nsplit_gain=338.323 188.636 86.7975 63.1092 37.0912 35.5466 23.9655 23.6947 21.1867 20.6809 21.8936 20.395 20.7182 18.941 17.194 16.6023 15.716 15.2298 13.7574 13.1965 12.4257 17.0015 11.6518 11.5007 11.2871 11.1507 14.4543 16.2115 9.98326 9.11729\nthreshold=0.72961209499321422 1.0000000180025095e-35 3.2382812500000004 1.0000000180025095e-35 0.14796868535883259 1.2424533248940002 0.77813183983917178 4.4301796584464421 1.0000000180025095e-35 1.2424533248940002 0.52258841453381233 3.0205078125000004 235.50000000000003 0.30499968189655369 1.8987378677022575 0.43728734921738482 0.56086626140189244 0.29887140897736025 2.3399293066241302 262.50000000000006 4.2121001622835372 1.2426757812500002 -18.836229324340817 -8.7471351623535138 1.4460303622695712 0.34961562014038972 0.89599609375000011 0.97942481616843191 1.2426757812500002 1.4978661367769956\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 13 5 15 9 8 11 10 -1 12 29 14 -2 -5 20 -10 -3 -17 21 -12 -4 -19 25 -14 -27 -28 -26 -7\nright_child=3 18 22 4 -6 7 -8 -9 17 -11 16 -13 24 -15 -16 19 -18 23 -20 -21 -22 -23 -24 -25 28 26 27 -29 -30 -31\nleaf_value=-0.027246933951180472 -0.024552108858234677 0.02633115074021514 -0.026121591956349752 0.019717697588735792 0.026008801995208358 0.0097249170746865655 0.025815364740760106 -0.031481600991716867 0.029666176349455207 0.0077371004416450567 -0.020625846004106191 0.022234718758557485 0.0089279363513698421 0.020913002438454856 0.010863815773286087 -0.044889883722967525 -0.026840307508693182 0.020986565015948967 -0.020072496109991317 0.026137843356620404 0.016482740051396133 0.010964131535413056 0.022429117969331665 -0.039957516865503147 0.017810003086988543 -0.015611736809628836 0.013482174458564483 -0.0088161352929995006 -0.012129186415131584 -0.028079642440072723\nleaf_weight=516.09485533952829 66.163695972383721 94.670143256313168 4.0855745000299057 291.25744545063935 161.97183646541089 6.7193846860900512 8.2037488548085076 14.164604705380041 50.356796939042397 18.115880323661258 51.184058560087578 49.638412512722425 103.47228678385727 12.102363439509643 14.503920464776455 5.6811698982492072 50.982073520950507 4.5175067679956555 5.4097708925837642 2.8275539558380842 16.944014892389532 20.972413327399408 45.410786415450275 4.7559430380351824 71.777270434424281 57.58671634120401 59.385176770505495 56.13773331939592 10.906806683167813 26.406950346543454\nleaf_count=176427 12861 364 1132 1250 308 89 42 3236 263 1573 11792 293 3563 46 266 71 16277 26 166 5 1258 1435 287 100 521 6215 2270 1269 117 2750\ninternal_value=-0.000473666 -0.0148127 -0.0200964 0.0106769 0.0129221 0.0102945 -0.0228803 0.00550204 0.00660684 -0.0235101 -0.0243833 0.00431491 0.00199726 -0.0128673 -0.0181293 0.0184953 -0.0137145 0.0232452 0.0236491 -0.02102 -0.00598959 -0.0114198 0.0179837 -0.011116 0.00410867 0.00118333 -0.00347052 0.00266376 0.013763 -0.0203511\ninternal_weight=1902.41 832.073 731.993 1070.33 977.564 815.592 682.497 515.826 501.661 674.293 656.177 442.031 392.392 92.77 80.6676 299.766 140.083 59.6302 100.08 8.50872 89.1005 72.1565 49.4964 9.27345 359.266 276.582 173.11 115.523 82.6841 33.1263\ninternal_count=246272 210753 210223 35519 22346 22038 208804 20712 17476 208762 207189 17087 16794 13173 13127 1326 30762 389 530 76 14485 13227 1419 126 13955 13317 9754 3539 638 2839\nis_linear=0\nshrinkage=0.03\n\n\nTree=64\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 8 18 13 18 5 2 13 4 14 3 3 11 17 13 3 22 3 6 4 6 13 4 0 0 18 2 14 3 0\nsplit_gain=359.757 155.029 65.5954 65.3795 69.4633 36.0276 33.2198 29.8388 29.1816 29.0878 21.7204 21.1495 16.8899 14.8407 14.5425 14.453 14.0694 12.9214 11.8858 11.651 10.6718 11.975 9.72597 9.11544 9.04376 8.06335 7.93464 7.74955 8.76742 9.07967\nthreshold=0.26534039634031065 1.0000000180025095e-35 2.602003343538398 0.14796868535883259 3.0677824455408698 3.7011718750000004 -30.420168876647946 0.14796868535883259 0.84667968750000011 0.52722272431259987 0.47260169401963864 0.31033658837561479 0.3376542910797411 2.8667313866239854 -0.10757639301670423 0.11499522373616446 1.0000000180025095e-35 0.44825707527455844 1.2426757812500002 6.9193702936172494 4.3378906250000009 -0.36167213950920335 5.9891631007194528 0.13844441961022361 0.82186659671441242 3.3497501700808394 -42.103004455566399 0.53481096685124585 0.36709515926886871 1.3771184689138856\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 15 4 8 9 -3 -4 -2 11 12 -6 13 18 19 -1 -5 -11 27 -8 21 22 -7 -21 26 -23 -13 -10 -29 -30\nright_child=3 6 7 16 5 20 14 -9 10 17 -12 24 -14 -15 -16 -17 -18 -19 -20 23 -22 25 -24 -25 -26 -27 -28 28 29 -31\nleaf_value=-0.025197447976629466 -0.021955217672985208 -0.029512028821967826 -0.029312206434138861 0.029340952863974052 -0.032554442617804061 0.010920842684563437 -0.029716672149467861 0.021807601555764753 0.01076515891557767 -0.030321859273504798 0.01812023307737046 -0.015932613586306144 -0.039617092366934496 -0.033124242253444451 0.020370481844698723 0.0058450101244246653 0.0036298746229721478 -0.0097080725873898484 0.028347208559168977 -0.016230056923853157 0.028916210408296183 -0.034915683588056849 -0.031315201862981347 0.018610612543662815 -0.027748419292085873 0.017210894287805079 0.0089216003686220183 -0.022056505865376788 -0.012347941346777849 0.014333747115833509\nleaf_weight=15.566735931963192 20.065616759820841 16.315336906642187 459.34351754753152 171.30443079350516 15.055244265153304 7.7596148806042065 8.758108059933873 9.8289911365136486 141.35045854811324 59.466966342646629 422.04892548616044 12.108110409637446 5.8883997051743782 6.650254158244933 103.31640011642594 71.885976591875078 21.893610273487866 52.479866663808934 25.609330599196255 8.9218109103967453 24.885959462029859 1.9279084964655329 10.705035394930745 20.699132969370112 6.5639530458720392 42.63620757573517 101.44194824967417 13.918095798057037 17.544856015825644 29.299344166414812\nleaf_count=3095 1596 6056 184079 298 3384 678 1682 32 1092 19652 1716 1228 130 358 1255 4172 77 5326 72 901 81 24 1424 324 347 810 4221 1093 648 196\ninternal_value=0.000541347 -0.0163242 -0.0237596 0.0105025 0.00744914 -0.00373087 0.00985335 -0.0281972 0.0129357 -0.00959275 0.0140243 -0.000201643 0.0067839 0.00811305 0.0145611 9.69988e-05 0.026461 -0.0208298 0.00945688 -0.00119799 0.0126686 0.00590211 -0.0139167 0.0079293 0.00408881 0.014239 0.00615538 0.00693554 -0.00200331 0.00429127\ninternal_weight=1925.24 714.636 556.625 1210.6 1017.41 335.031 158.011 469.173 682.375 247.116 662.31 135.169 240.261 234.372 141.695 87.4527 193.198 111.947 227.722 38.3791 87.9147 63.0288 18.4647 29.6209 120.114 44.5641 113.55 202.113 60.7623 46.8442\ninternal_count=246047 201596 191378 44451 44076 37175 10218 184111 6901 34158 5305 9180 3589 3459 4162 7267 375 24978 3101 2907 3017 2936 2102 1225 5796 834 5449 3029 1937 844\nis_linear=0\nshrinkage=0.03\n\n\nTree=65\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 8 13 1 5 18 2 0 22 5 4 18 7 7 5 3 14 13 14 11 9 14 2 3 3 4 0 1 18\nsplit_gain=475.595 114.084 58.2537 51.0745 51.3897 25.401 27.2197 24.5883 23.2762 18.7341 14.1021 13.1791 18.5472 11.9319 10.6252 11.4286 10.1914 19.9025 16.4241 10.1902 9.94848 10.6373 13.5039 10.8756 9.43555 9.2859 14.8194 10.7904 8.66613 12.3359\nthreshold=1.3779212206518803 0.71017588656534381 1.0000000180025095e-35 0.14796868535883259 2.9477339485081675 3.1982421875000004 3.1984648276080736 -14.08206272125244 0.84505259813102007 0.93750000000000011 4.1015625000000009 4.9233398437500009 2.249904835165133 1.0000000180025095e-35 2.2500000000000004 1.4980468750000002 0.58795853567030554 0.55141373229880208 -0.16259509587407081 0.88396997525697196 0.11347059467310934 1.0000000180025095e-35 0.69452714305686503 23.600493431091312 0.31033658837561479 0.31840721315068282 3.2580566406250004 0.29896707506516013 1.9634439907166827 2.3502401828962087\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 8 4 5 6 13 -4 25 -5 -10 12 16 -2 15 -11 17 -6 -19 24 21 22 23 -9 -14 -1 -27 -28 29 -15\nright_child=3 -3 7 9 11 -7 -8 20 10 14 -12 -13 19 28 -16 -17 -18 18 -20 -21 -22 -23 -24 -25 -26 26 27 -29 -30 -31\nleaf_value=-0.030343751528651285 -0.016777577923586556 0.022726021898803527 -0.026375840070503226 0.029275196725617215 0.019133513034856539 0.014276289984865766 -0.015121994936575144 -0.015130111843125935 -0.016941749631303928 0.0041734117019850752 0.015942174497546611 0.02847882420567727 -0.024967348798954358 0.0019871586584787656 0.034088750733227001 -0.046698317192751485 0.025022711087070999 0.0040420751429926871 -0.044511429709516791 -0.032507668112323529 -0.020546356115139729 0.022119459662767158 -0.031816073002947989 0.022398986132900457 0.0084551842649953253 -0.028657668488442433 -0.0225767118230417 0.020831027106943529 0.0096870695006295807 -0.034943802039499969\nleaf_weight=414.41566639574012 22.633586891344748 54.291868592961691 26.686868498160038 177.62772143771872 137.53990770503879 109.96351051609963 101.92564123519696 9.949318802449854 83.665484423749149 13.117098227143293 12.817783929640425 65.263172920793295 7.1491450193570918 52.058956584427506 5.3354022502899161 4.6160550415515891 117.30440955003724 38.004686667118222 6.4800460222177199 5.3685798898804924 9.8832761724479479 34.116943305591121 8.5825942460214701 18.465973407495767 120.91524846095126 31.827676888351562 7.4679437594022628 12.224977428268174 145.48231946746819 8.6294375171419215\nleaf_count=181796 1462 230 7970 318 313 1196 12874 760 17942 43 241 169 351 590 5 38 201 135 78 260 336 274 1469 176 1160 10658 2244 909 1206 722\ninternal_value=0.000200238 -0.018791 -0.0221835 0.01228 0.0093244 0.00185451 -0.00230616 -0.00194723 -0.0260478 0.0259895 -0.0124544 0.0159174 0.0139618 0.00344388 0.000524422 -0.0108548 0.0180994 0.0135253 -0.00385377 0.00461439 0.00632489 0.0102925 -0.000908652 0.00919092 0.00641039 -0.0288299 -0.016307 0.00407441 0.00573407 -0.00374548\ninternal_weight=1863.81 724.396 670.105 1139.41 938.719 440.693 330.73 107.685 562.42 200.696 96.4833 498.025 432.762 228.804 23.0686 17.7332 299.329 182.025 44.4847 133.433 80.9981 71.1148 36.9979 28.4153 128.064 465.936 51.5206 19.6929 206.171 60.6884\ninternal_count=246126 225005 224775 21121 20717 18050 16854 10985 213790 404 18183 2667 2498 3980 86 81 727 526 213 1771 3015 2679 2405 936 1511 195607 13811 3153 2518 1312\nis_linear=0\nshrinkage=0.03\n\n\nTree=66\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 8 13 18 7 18 2 22 3 7 2 3 13 18 7 17 17 2 6 6 0 17 13 11 19 12 13 19 19\nsplit_gain=309.173 142.289 67.0713 61.0968 64.559 32.8686 32.5026 20.87 20.2231 18.5538 16.7114 16.093 13.4202 13.249 10.5339 9.99821 9.94855 9.44655 8.30724 10.7406 8.23453 12.4029 8.9822 8.45463 7.82937 7.4823 7.07628 6.88763 6.89063 6.8595\nthreshold=0.26534039634031065 1.0000000180025095e-35 1.0000000180025095e-35 0.14796868535883259 3.0677824455408698 0.89599609375000011 2.524928003624769 -25.182852745056149 1.0000000180025095e-35 0.53650715206675215 0.89599609375000011 -34.252307891845696 0.41780940659066851 -0.64803833042221404 0.89587973461402759 1.0000000180025095e-35 1.4392430184994598 2.4286772172182522 18.240050315856937 1.2426757812500002 2.9707031250000004 0.29549219484135919 0.077783200051907503 -0.015587343939122865 5.5511151231257837e-16 263.50000000000006 0.29026868663126176 1.0000000180025095e-35 256.50000000000006 211.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 4 9 12 15 -4 -5 10 14 25 -6 -13 -2 -1 27 -17 19 -9 21 23 -23 -15 -21 -7 -3 28 -14 -20\nright_child=3 26 7 8 5 11 -8 18 -10 -11 -12 13 16 20 -16 17 -18 -19 29 24 -22 22 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.030502658124925541 0.01225729259637077 -0.0070299151733778991 -0.029549536167498246 0.028899245858675367 -0.028486946644952001 -0.03147685793618113 -0.029017952471527272 -0.024538063958254665 -0.0014093179516177072 0.017353435249092046 0.011994981814924501 -0.01381770687486426 -0.017119428421333138 -0.027946634814199248 -0.006886580046877735 0.0039529392525810289 -0.031986010150206849 -0.030821726707466943 -0.0026414443275963855 0.015261035303285922 0.026700288062237055 -0.024210575211519262 0.012726429187847442 0.0088496027237069882 -0.015923370329029463 0.0020972892166872701 0.025972970577989907 0.025010095950786373 0.0056380483066007661 0.023824241561936195\nleaf_weight=11.719180917571064 34.623551975819282 5.9989091292954972 16.431524927495047 156.70284003647976 53.565491676796228 15.806630956241863 429.28642682582722 12.914024758851154 22.556860746815801 278.58075507869944 215.46092336723814 25.54351730691269 24.299775855150074 13.354843023727883 93.568467573437374 48.50333616649732 11.628488725284113 7.195447987469378 12.259241642197592 24.801132908673026 19.961886313511059 5.3910037119640029 98.936002131173154 8.6038808437297103 9.1608082775492203 9.084486593201289 65.47823129966855 6.246155744418501 22.016543748031836 28.761940743366722\nleaf_count=2626 200 182 6367 292 16705 3440 184649 1988 80 1288 1678 3928 2811 2575 3614 2719 2005 1221 515 498 78 159 2904 425 373 963 235 12 1261 288\ninternal_value=-0.000581566 -0.0166352 -0.0213966 0.00909932 0.00601386 -0.00505485 -0.0259811 0.000469702 0.0250988 0.0116064 0.00691084 0.00245833 -0.0175187 0.00575167 -0.00166113 -0.00650243 -0.00806948 -0.0010038 0.00635078 -0.00228644 0.00924027 0.0062923 0.010606 -0.0137716 0.00663458 -0.0196156 0.0231443 -0.00228 -0.00641109 0.0160403\ninternal_weight=1788.44 672.51 601.033 1115.93 936.672 314.439 496.704 104.329 179.26 622.234 343.653 196.682 117.756 171.791 128.192 67.418 64.191 55.6988 87.8971 46.876 146.248 126.286 104.327 21.9587 33.9619 24.8911 71.4771 52.5625 46.3163 41.0212\ninternal_count=246079 201661 201244 44418 44046 37266 191215 10029 372 6780 5492 14472 22794 10069 3814 6566 6089 3940 3662 2859 6141 6063 3063 3000 871 4403 417 4084 4072 803\nis_linear=0\nshrinkage=0.03\n\n\nTree=67\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 1 11 7 2 11 2 13 22 7 5 4 4 18 12 17 17 2 9 18 4 3 1 6 2 18 13 21 21\nsplit_gain=444.042 111.232 65.9608 46.2905 31.9244 23.0905 19.5818 17.9834 17.6479 23.4434 17.6562 18.6482 18.3167 16.05 13.3862 11.9464 10.6766 10.499 9.41578 9.12687 9.09929 9.00415 9.95646 8.95786 8.99789 16.2419 8.85117 8.52581 8.32175 8.24516\nthreshold=0.95935769349451683 1.0000000180025095e-35 2.5144622974930591 3.8857805861880484e-16 0.89599609375000011 -30.420168876647946 0.66750229771629133 -8.3227787017822248 1.0000000180025095e-35 0.93750000000000011 0.89599609375000011 0.89599609375000011 0.84667968750000011 3.9514160156250004 2.249904835165133 0.88192242588002812 1.2133651393498066 1.0329219259280322 -33.804542541503899 1.0000000180025095e-35 2.1383330595080281 7.8217773437500009 0.58795853567030554 -0.90051303354301027 6.0937500000000009 50.658309936523445 1.2424533248940002 -0.59662603815460169 4.5000000000000009 1.5000000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 4 16 -3 14 19 10 -10 11 12 -4 -11 17 23 -2 -1 -5 21 -9 22 26 -7 25 29 -6 -13 -29 -25\nright_child=2 5 8 18 7 15 -8 20 9 13 -12 27 -14 -15 -16 -17 -18 -19 -20 -21 -22 -23 -24 24 -26 -27 -28 28 -30 -31\nleaf_value=0.0036386554144293673 -0.0096946170603947092 -0.029275817190493627 -0.018588626068548456 -0.016283739372288573 0.013523307200593599 -0.031842453481963316 0.014433970968531046 0.017627349273389978 0.028818193721827645 -0.032724200848212343 0.018145283664567986 0.017731878761449077 0.016890253324352178 0.032452094566762284 -0.029985568168106613 0.028620194945589546 -0.028331800603568241 -0.030792446267580768 0.018417260011134185 0.017909891624022851 0.0023627285265546455 0.01970191453809313 0.011462182656428729 -0.034297100230979302 0.024257318065954655 0.02204471078552734 -0.031661078831385525 -0.0046100924051339963 -0.034160223069676483 -0.0024455270839315217\nleaf_weight=20.408556721115019 95.984185465640621 14.645868140942182 13.725265009154098 6.6915284626884377 3.8685012217611066 4.7727022730978197 9.1623418657109124 63.50207169726491 145.0978711603675 9.3222460309043509 342.81562363426201 11.36009068507701 130.06729818251915 4.139751434326171 426.34323143205256 22.601765771862119 39.017628067696933 11.890059116354676 130.00029059511144 7.6929338724585241 78.021939363912679 4.9588501116959369 6.8980596201727158 8.9142949359375034 19.029648091876879 20.084595054562669 31.647294880618574 52.232689153170213 9.3717972988961247 28.567722561187111\nleaf_count=2085 11656 6103 725 766 17 627 95 365 276 74 1669 139 672 4 195135 43 7699 2494 818 171 3532 139 103 1076 115 281 5294 961 296 2046\ninternal_value=0.000200605 -0.0211339 0.0107547 0.00198801 -0.0040725 0.00478156 -0.0277013 0.00364147 0.0164607 0.0253152 0.013911 0.00716162 0.0133987 -0.0118235 -0.0285726 0.00981225 -0.0152498 -0.00958052 0.0166126 -0.0107765 0.00928761 -0.0157089 -0.0202074 0.00430395 0.00693757 0.000895793 -0.0265039 -0.00514493 -0.00957935 -0.0107151\ninternal_weight=1772.84 586.421 1186.42 468.283 331.591 118.617 467.804 196.59 718.133 158.56 559.573 216.757 143.793 13.462 458.642 103.971 135.002 32.2986 136.692 55.0656 141.524 47.3727 42.4139 81.369 76.5963 57.5666 35.5158 72.9646 61.6045 37.482\ninternal_count=245476 210100 35376 30560 28976 10291 199809 9621 4816 354 4462 2793 1397 78 199714 4188 19355 4579 1584 5724 3897 5553 5414 4145 3518 3403 5311 1396 1257 3122\nis_linear=0\nshrinkage=0.03\n\n\nTree=68\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 1 13 7 22 12 2 18 0 13 17 14 0 7 7 22 18 13 14 10 1 6 7 13 12 1 19 14 5\nsplit_gain=424.316 99.8396 64.0424 47.7729 30.6211 29.053 23.6868 21.7318 19.8628 19.6854 16.7655 12.9993 12.1885 11.6826 14.4182 11.6219 14.8464 11.1333 11.2753 10.7758 10.5594 14.2857 9.30755 9.24993 16.9146 8.7139 8.49659 12.9764 8.43734 8.33677\nthreshold=0.95935769349451683 1.0000000180025095e-35 3.3033179733375269 1.0000000180025095e-35 0.89599609375000011 0.67500000000000016 0.76965903234489019 -34.252307891845696 1.2424533248940002 -0.7405159439372101 0.14796868535883259 3.9281938047451082 0.77173600245215745 0.72961209499321422 2.6015625000000004 2.5244140625000004 0.93750000000000011 0.89587973461402759 -0.53446807821110365 0.42891595223756424 1.8688348091416842 0.26032702032866095 1.2426757812500002 0.89599609375000011 1.0000000180025095e-35 5.5511151231257837e-16 4.0016611463566001 251.50000000000003 0.8176537162135239 1.7006835937500002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 10 3 4 8 -5 9 -6 -2 -3 13 12 15 -1 -15 16 19 -4 -19 -9 21 -11 -23 24 26 -22 27 -20 -14 -10\nright_child=2 6 17 5 7 -7 -8 11 29 20 -12 -13 28 14 -16 -17 -18 18 23 -21 25 22 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.030873231493669169 0.0078266892272261419 -0.030724031283781265 0.026418282404478716 0.025812630642837724 -0.019455377354993225 -0.018301381941044335 0.026615012020674414 0.0047659792683225908 -0.016865913903521976 -0.024764810187839985 0.013401837062046616 -0.023558369514455772 -0.034969617737517139 -0.022761168513019181 0.013515061788146954 0.022807786014001075 0.015996039466402791 -0.0027546985252257824 -0.027199949746315035 -0.021680567348164922 0.029338919247072222 -0.023089276861539373 0.018046621028652659 0.021505516387423079 0.027252056405509536 -0.013299695024857699 0.0097273643566594733 0.0136483386288803 0.0016093469641960906 -0.00010387211525799916\nleaf_weight=359.65144197217887 51.317891123471782 18.707825760444393 125.36866605561227 114.09947533975355 26.549349705426721 14.493815513094885 28.900127943139523 51.248370292480104 129.6790732377267 18.322748846898321 8.2284075247589481 10.152702254126778 5.2395191412651902 74.836488697881578 10.662085896008646 71.103604941861704 93.003666349104606 26.784015187062323 15.581140967551621 17.889139916107524 17.419628256466243 5.4753418988548246 20.871944457641803 196.32873850222677 38.855689948424697 4.9865737583022556 62.441785073257051 11.291907852981238 62.020054027438164 33.57901490732911\nleaf_count=176418 390 6685 195 254 3840 141 73 1209 17760 2117 40 1280 334 23599 364 485 936 201 347 1231 163 903 204 638 79 197 400 91 2262 3377\ninternal_value=-0.000158991 -0.0213712 0.010265 0.00434639 0.000495791 0.0207377 0.00361121 0.00612703 -0.00834109 -0.00435601 -0.0276847 0.00836877 0.009522 -0.0284725 -0.0181334 0.0127546 0.0082332 0.0186864 0.0158598 -0.00228854 0.00334143 -0.00510883 0.00896629 0.0174 0.0109934 0.0196868 0.00369684 -0.0101619 -0.00171071 -0.0134399\ninternal_weight=1725.09 568.063 1157.03 680.376 551.782 128.593 114.684 337.206 214.576 85.7841 453.378 310.657 300.504 445.15 85.4986 233.245 162.141 476.652 351.283 69.1375 67.0762 44.67 26.3473 324.499 128.171 22.4062 89.3148 26.873 67.2596 163.258\ninternal_count=246213 210763 35450 33499 33104 395 10342 11577 21527 10269 200421 7737 6457 200381 23963 3861 3376 1951 1756 2440 3584 3224 1107 1555 917 360 838 438 2596 21137\nis_linear=0\nshrinkage=0.03\n\n\nTree=69\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 1 11 11 18 9 2 7 3 0 3 4 4 7 12 21 4 19 14 5 21 2 4 2 14 5 4 16 4\nsplit_gain=413.557 104.009 40.7277 36.3458 28.5426 32.5288 26.0465 22.3858 17.0355 15.6533 12.7306 14.0475 14.2641 12.6168 16.8118 12.022 11.2667 16.319 12.1846 10.4858 9.79415 9.63016 11.4648 11.4528 10.6788 9.28601 8.80568 8.51968 8.48984 7.91732\nthreshold=1.3779212206518803 1.0000000180025095e-35 4.0016611463566001 0.49644476220754463 0.042545688445982228 1.2424533248940002 1.0000000180025095e-35 -30.420168876647946 0.89599609375000011 0.51817819236196982 1.8987378677022575 0.31840721315068282 3.5791015625000004 2.3220214843750004 2.9707031250000004 0.76965903234489019 3.5000000000000004 4.2182617187500009 261.50000000000006 0.87268561280719448 2.8037109375000004 2.5000000000000004 -31.721868515014645 4.4182128906250009 32.956295013427741 0.48434979059990024 4.2558593750000009 1.6286621093750002 1.0000000180025095e-35 1.6286621093750002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 4 10 5 29 8 -3 9 -7 11 -1 -13 27 21 19 -6 18 -18 -9 -5 24 -23 -24 28 -14 -25 -10 -15 -2\nright_child=2 7 -4 20 16 6 -8 15 13 -11 -12 12 25 14 -16 -17 17 -19 -20 -21 -22 22 23 26 -26 -27 -28 -29 -30 -31\nleaf_value=-0.029312432627202997 0.0063832957689506547 -0.029253010878789131 0.021053608661025784 -0.033719696002131344 0.022482406111341429 -0.016664696556236695 0.02694572715191039 0.007520151597823454 -0.032707349998630009 0.0026993591386817463 0.0069483739039217767 -0.024741413353899799 0.026441484078823296 -0.039108885488470857 0.018321047463490114 0.026061063141505872 -0.036370733015967098 0.019959192465231739 0.008748105029410572 -0.023608096993926814 0.017518710938682003 -0.028308491294198049 0.0082110123688969808 -0.039229279661182496 0.001211381170967672 -0.010398933758595489 0.012381222800795602 0.015280556636466432 0.011856043634096213 0.021201379833128454\nleaf_weight=410.03680400589656 48.507192214485258 14.325199027982306 309.62010347016621 2.9897586922161308 140.848330817651 96.77460255502956 29.938900518231094 81.049374046968296 2.5651476704515508 60.617314609873574 9.6096125565236417 38.399888164480217 9.3016384835355002 18.813926427799746 30.01628256496042 28.121136127272621 10.124119378917387 28.163199318805709 9.8367897844873351 10.070882105443159 21.831587626773398 10.957918595959198 65.400831946695689 7.9853754505747911 14.103729404509066 14.398838640132452 3.8635008251294494 75.057279960135929 2.9845464860554776 104.49574469961226\nleaf_count=197149 259 6317 967 374 696 10280 191 2457 163 2042 431 12721 269 1589 203 71 237 109 139 2229 338 969 1152 372 214 2746 27 1107 98 261\ninternal_value=0.000489341 -0.0185696 0.011891 -0.0247688 0.00814059 0.004785 0.000451169 0.00495131 -0.00165106 -0.00923694 -0.0265922 -0.0272882 -0.0136695 0.00351844 -0.00146125 0.0092696 0.0182204 0.00554276 -0.0147795 0.00386084 0.0102801 -0.00635385 -0.000959181 0.00323253 -0.0192228 0.00454482 -0.022747 0.0132989 -0.031917 0.0165755\ninternal_weight=1710.81 640.135 1070.67 506.568 761.055 572.082 419.079 133.567 389.14 157.392 481.747 472.137 62.1004 231.749 154.126 119.241 188.972 48.1241 19.9609 91.1203 24.8213 124.11 88.2076 77.2497 35.9022 23.7005 11.8489 77.6224 21.7985 153.003\ninternal_count=246177 225102 21075 214028 20108 18927 18407 11074 18216 12322 213316 212885 15736 5894 4624 4757 1181 485 376 4686 712 4421 2520 1551 1901 3015 399 1270 1687 520\nis_linear=0\nshrinkage=0.03\n\n\nTree=70\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 18 11 5 2 14 13 0 22 14 7 5 3 4 11 11 6 2 2 2 4 14 4 11 17 3 3\nsplit_gain=379.145 108.697 60.8289 47.601 26.1109 24.2606 23.6825 21.0014 13.2558 17.6673 13.2228 12.6269 12.1361 12.0262 17.1186 14.4909 10.447 9.8216 10.715 11.6454 9.73398 9.46598 13.444 9.5412 9.28209 9.58626 8.83509 8.42796 8.39416 8.17253\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 2.5144622974930591 2.9174053685313028 0.66750229771629133 3.4169921875000004 -30.420168876647946 0.57916335608139347 -0.37079164458764446 0.58526177796276302 0.93750000000000011 0.95830763268103369 0.89599609375000011 0.89599609375000011 0.53650715206675215 4.4806598424911508 0.71017588656534381 0.027600239276592001 1.4980468750000002 32.956295013427741 7.8238990306854257 18.821963310241703 1.6286621093750002 0.0031798781733466956 0.84667968750000011 0.61174042517169969 2.5789308873416652 0.56033297298897644 0.47260169401963864\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 3 4 12 29 21 -3 13 10 -10 -4 16 14 24 -16 27 18 19 20 -9 -6 23 -23 -5 -26 -13 -2 -28 -1\nright_child=2 7 11 8 6 -7 -8 17 9 -11 -12 26 -14 -15 15 -17 -18 -19 -20 -21 -22 22 -24 -25 25 -27 28 -29 -30 -31\nleaf_value=-0.029779714878049889 0.004425387507833869 -0.029022376179511998 0.028704475383752176 -0.026577969236573681 -0.022881646137022756 0.015899177996086786 0.006790454399736693 -0.02505142069677093 -0.030993704663955641 -0.010330292104244739 0.013320079937863046 0.032368621725297191 -0.033772823965363491 0.019531067613201804 -0.019991224868776733 0.012386973455427596 0.018471226265984284 0.024591340055415807 -0.02501075446736457 0.018162340956186816 0.011684710468814837 -0.026794782701422971 -0.025659414559240267 0.016591295605359612 -0.0096093551026317624 0.020860950915729242 -0.029840630842909201 -0.030794189684546107 0.010222691058868374 -0.00081420603129912881\nleaf_weight=401.00682963864529 149.144161017437 13.359105132753028 156.50868281908333 3.6200356017798176 81.859140429864055 10.456758986227213 53.432544929033611 12.762184419785628 5.4513058241282115 46.556847867614124 119.34769592899829 6.597266802564266 5.8155498732812694 237.54008020029869 27.504177059163339 21.1785769732669 51.995252762222663 27.83846241934225 9.0798637041589227 39.391523071855772 11.447966457810251 4.7771222903393236 16.650797003123444 21.447445127123501 9.8495393882039917 84.101027445634827 6.8620206173509386 5.3825050288287448 10.863617179915307 9.0204489321913552\nleaf_count=198562 3204 6185 335 48 18525 98 1711 2462 389 1070 686 18 369 891 611 120 266 98 585 692 377 1378 3504 992 392 350 32 590 31 1696\ninternal_value=-0.000174712 -0.0209634 0.00969652 0.00663272 -0.00139206 -0.0279908 -0.0098322 0.00500204 0.0122699 0.0052903 0.0111534 0.0255939 0.00570274 0.0153643 0.00851004 -0.00593985 0.00697174 0.00976522 0.00382984 0.00827058 -0.007906 -0.0169502 -0.00536232 0.00799408 0.0157069 0.017599 0.00523571 0.0030085 -0.00605211 -0.029146\ninternal_weight=1660.85 534.363 1126.49 945.654 390.505 420.484 178.167 113.879 555.149 171.356 124.799 180.832 212.337 383.793 146.253 48.6828 206.522 100.52 72.6815 63.6017 24.2102 124.735 42.8754 26.2246 97.5706 93.9506 24.3229 154.527 17.7256 410.027\ninternal_count=246267 210755 35512 35096 30539 200356 26110 10399 4557 2145 1075 416 4429 2412 1521 731 4060 4214 4116 3531 2839 24399 5874 2370 790 742 81 3794 63 200258\nis_linear=0\nshrinkage=0.03\n\n\nTree=71\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 7 18 2 3 2 4 3 14 22 17 19 4 9 2 13 7 3 0 0 2 18 13 2 4 13 0\nsplit_gain=365.203 105.782 54.1982 50.2397 33.0309 20.9153 19.7996 17.5495 16.3121 13.7237 16.8088 17.8049 12.9993 12.6495 11.9981 11.0183 10.2408 16.0327 11.3003 10.0258 10.9785 9.24326 8.9155 8.83798 8.7477 8.44474 8.42387 7.82855 7.82017 9.07447\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 2.9477339485081675 0.89599609375000011 2.4414009612931857 -30.420168876647946 0.38657275885365588 -34.252307891845696 4.1440429687500009 0.42638606281916264 0.41892379644680228 0.93750000000000011 3.9281938047451082 240.50000000000003 6.3784179687500009 1.0000000180025095e-35 23.600493431091312 0.14796868535883259 4.1191406250000009 0.41780940659066851 1.3771184689138856 1.4460303622695712 15.156654357910158 1.2424533248940002 -0.26766225074600319 30.514518737792972 1.0693359375000002 -0.076599057954334346 1.8987378677022575\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 7 3 4 5 23 -3 22 24 10 11 14 -4 26 -5 -9 17 18 -8 -18 -21 -12 -1 25 -6 -2 -10 -27 -23 -30\nright_child=2 6 12 9 8 -7 16 15 13 -11 21 -13 -14 -15 -16 -17 19 -19 -20 20 -22 28 -24 -25 -26 27 -28 -29 29 -31\nleaf_value=-0.030113867439399689 -0.0036672608556407274 -0.028950453994689963 0.028499533254684059 -0.032649993469724928 0.025436574969414361 -0.020300002178891455 -0.032590764555300474 -0.017205190278664385 0.0064070386685943772 0.026906941068843376 0.0095999895153095881 -0.029454650959490219 0.0030749723003334951 -0.023471046534915576 0.010610173583562161 0.022940787177585006 0.022692774019144261 0.01470925473370387 0.023688790501106862 -0.03182517537563375 0.022342365207491811 0.02306014281083792 -0.0054231088950847288 -0.012085585687935479 -0.023621076242478188 0.024727473794921274 0.018291066653684376 -0.016683658203697469 -0.010044217963003792 0.023524492319040802\nleaf_weight=368.5238586353662 32.470511394436471 12.794103245425502 149.34711824299302 5.4144030008465043 2.9892995953559902 86.329162596884998 15.446272336470431 16.397076665423814 208.53574790852144 65.862281970912591 144.92392966488842 14.294475450180469 20.869972535409033 9.9302145281981193 58.731747385812923 8.5334289236925525 50.078027232142631 22.053508091950789 3.2205005884170523 6.403259003418496 5.2391744488850236 135.64195480942726 14.155624778941272 33.713354039588012 21.321541306781 24.581322305602956 72.332659576903097 4.1671799678588277 15.134199613705277 12.237005116418004\nleaf_count=193879 712 6128 335 132 3 18947 2704 4520 5697 226 1016 495 78 1281 468 279 707 481 5 270 25 271 1632 1076 3773 216 667 145 71 28\ninternal_value=0.000203932 -0.0204717 0.00987559 0.00706278 0.000468068 -0.00970421 0.00487484 -0.0276317 0.00633169 0.014284 0.0120684 -0.000329296 0.0254184 0.00831838 0.00651357 -0.00310891 0.00933841 -0.00232305 -0.0221703 0.0169465 -0.00761025 0.0152173 -0.0292171 4.2506e-05 -0.0168222 0.00680835 0.00952159 0.0183658 0.0201223 0.00526373\ninternal_weight=1641.67 522.845 1118.83 948.611 496.371 181.262 115.235 407.61 315.109 452.24 386.378 78.4406 170.217 290.799 64.1462 24.9305 102.441 40.7203 18.6668 61.7205 11.6424 307.937 382.679 94.9324 24.3108 61.219 280.868 28.7485 163.013 27.3712\ninternal_count=246267 210630 35637 35224 32517 21096 10320 200310 11421 2707 2481 1095 413 7645 600 4799 4192 3190 2709 1002 295 1386 195511 2149 3776 1073 6364 361 370 99\nis_linear=0\nshrinkage=0.03\n\n\nTree=72\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 1 13 7 12 7 0 2 22 5 18 17 4 5 22 14 2 1 1 7 5 3 17 10 4 0 18 2 4\nsplit_gain=349.086 78.4908 49.4914 40.5729 32.7102 29.1402 20.4778 17.3198 17.0838 15.5067 15.9379 14.9542 12.7565 12.5743 11.8707 15.1375 15.9879 15.3116 10.5404 10.5258 10.2186 9.60376 9.50079 14.3582 9.4636 10.6713 9.15815 9.11839 8.18571 8.14307\nthreshold=0.95935769349451683 1.0000000180025095e-35 2.9477339485081675 1.0000000180025095e-35 0.89599609375000011 0.88192242588002812 3.9980468750000004 0.84505259813102007 -33.804542541503899 0.67500000000000016 1.4980468750000002 2.4414009612931857 3.9281938047451082 3.9514160156250004 4.2558593750000009 0.93750000000000011 0.39236630927281513 18.240050315856937 2.1468275719512682 -0.90051303354301027 4.1191406250000009 2.5244140625000004 0.47260169401963864 1.5700365906568221 2.6735537653587351 5.7263183593750009 1.6765212893675572 3.5694334999727624 -1.7340650558471677 3.5791015625000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 3 4 11 6 7 -1 26 -5 -11 18 13 22 15 16 29 -18 -2 -3 24 -19 23 -4 25 -21 -6 -9 -17 -10\nright_child=2 19 12 9 8 -7 -8 27 14 10 -12 -13 -14 -15 -16 28 17 21 -20 20 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.030245200203591019 -0.014191151902168485 -0.029492775980633417 0.011255239507831619 0.025086957995399915 -0.027618970868792427 0.025313671626400792 0.0088763427688770893 -0.016385636805405997 -0.0061528673731386973 0.016035824901062118 -0.045393269566041228 -0.01984379628154426 -0.019004056902068297 0.027458796505369063 0.018627335889598876 0.0021860203050411155 -0.033035046619188158 -0.03181326726702785 0.0072685634344646661 -0.01542219812500256 -0.022183753980207631 0.011092560493968978 0.017814691014408711 -0.021654579212799261 -0.032022699151013662 0.023403347169798271 0.02328504893167967 0.02303037800398931 0.02014576204364148 0.016770531138509424\nleaf_weight=345.41381568327779 34.864429576671682 5.00944726212765 142.2532495346386 87.362412108224817 18.174311881084581 9.3093136756215227 13.929775726544902 60.612849693439784 23.703951471485198 8.8435778750572336 5.2382544728461644 84.104115184425609 8.3642620702739787 83.184274464612827 66.727013156167232 39.420289662724826 23.740050890075509 5.1973503249464583 47.438128567533568 6.5749060406233175 6.1678861470136317 18.374848603270948 297.72467972314917 12.095405786763875 2.6181283128971691 50.272392978338758 3.0201430101878932 4.9085835859295903 54.199751301202923 31.515456210065167\nleaf_count=188822 1605 1491 1014 233 3737 22 941 15780 1063 45 76 18718 324 245 606 913 3067 536 504 1150 274 369 1112 220 259 1760 6 351 381 554\ninternal_value=0.000201885 -0.0204152 0.00971198 0.00337393 -0.000500333 -0.0251792 -0.0263424 -0.0275549 0.00568198 0.0204152 -0.00830966 -0.0110251 0.0161288 0.0167122 0.00779049 0.00402823 -0.00392371 -0.0165492 -0.00188328 0.00880781 0.0121298 0.000722716 0.0146786 0.0085535 0.0159969 0.0187238 -0.019643 -0.0131309 0.0126865 0.00699583\ninternal_weight=1600.36 504.817 1095.55 551.924 450.48 434.174 424.865 410.935 284.073 101.444 14.0818 166.407 543.622 535.258 262.879 196.152 102.532 47.3122 82.3026 70.6428 65.6333 23.5722 452.073 154.349 59.4654 56.8473 21.1945 65.5214 93.62 55.2194\ninternal_count=246178 210850 35328 32413 32059 205916 205894 204953 11232 354 121 20827 2915 2591 7489 6883 5589 3972 2109 4934 3443 905 2346 1234 3169 2910 3743 16131 1294 1617\nis_linear=0\nshrinkage=0.03\n\n\nTree=73\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 7 2 3 2 0 3 4 4 0 5 0 0 4 1 8 13 14 4 10 0 7 7 5 3 2 2\nsplit_gain=330.23 79.4761 50.6345 50.7032 29.8654 22.8445 20.1183 17.2075 16.262 13.3635 12.4643 12.3197 12.1329 10.8762 13.6534 9.49834 9.42882 9.32965 9.26957 9.10302 8.94763 8.46283 9.10308 8.19493 7.43394 7.38976 8.03757 7.55416 6.87908 6.81666\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 2.9477339485081675 0.89599609375000011 -34.252307891845696 0.38657275885365588 -30.420168876647946 0.97942481616843191 0.53650715206675215 5.9760742187500009 4.1440429687500009 1.1851094193663554 1.4980468750000002 1.3170161875856263 1.6765212893675572 1.0693359375000002 -0.90051303354301027 1.0000000180025095e-35 -0.45698050410493757 0.58611746946415877 5.7263183593750009 3.6242520361853052 1.2661233611507157 1.0000000180025095e-35 0.89599609375000011 0.89599609375000011 0.45898141303402712 -10.988907814025877 -3.4835662841796871\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 4 8 -6 12 -3 18 11 -8 13 -1 25 -15 -7 24 -9 -2 -14 -16 -19 -23 -11 28 26 -5 -28 -10 -22\nright_child=2 7 -4 9 5 15 10 17 16 23 -12 -13 19 14 20 -17 -18 21 -20 -21 29 22 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.029762595877549758 -0.023461256786233151 -0.028772121241174439 0.025187768968427268 0.010795635372750268 -0.024718610463273678 0.0054683339037486954 -0.01852448244732486 -0.03180589584789642 0.012889923729303591 0.01170882509046467 0.02257085920682677 0.023548062517675734 0.015140031986380361 -0.021477972808014233 0.015108336002821108 0.020185700469188446 -0.012781968726732039 -0.012485814553658112 0.015259406865683759 -0.020727515593600244 0.010662198760202845 0.016259921438043931 -0.03357094954435242 0.024349760704467632 0.011392106320250535 0.02153487083306143 0.00066251420537582356 -0.040086420354682038 -0.027360048244490388 -0.036967524166959621\nleaf_weight=335.6293809824565 67.824080161284655 12.100491061471983 157.25290409277659 62.605315634631552 19.682639826030936 235.97749731095973 14.018591617583299 4.0546800174051887 5.3707056757993987 76.281619922956452 10.910067097400313 42.151009164750576 9.0575410648161796 24.456595896277577 28.474584378302097 46.773705940926448 41.847200862190221 11.81916633795481 5.3809515322791404 16.725341481011128 4.0862405430525541 70.55026629351778 2.5457498619798562 129.84679991519079 46.171996008604765 38.779040455818176 14.084855344030077 4.754345867317169 9.1471836558775959 5.3530565868131799\nleaf_count=191076 17187 6157 419 609 3763 7405 4310 600 241 446 473 175 860 573 75 203 1798 1641 305 3476 12 1806 116 274 585 136 251 95 913 39\ninternal_value=0.000168937 -0.0202634 0.00951888 0.00678368 5.39449e-05 0.00576393 -0.0264457 0.00339582 -0.00974936 0.0142348 -0.000362193 0.00911059 -0.0282388 0.00566766 -0.00445113 0.00796549 -0.00198762 0.00804204 -0.0204478 -0.0080324 0.00680183 0.0102928 0.0140948 0.0197509 0.0055419 0.0109106 0.00565208 -0.0111233 -0.012599 -0.0173025\ninternal_weight=1553.71 487.411 1066.3 909.049 478.176 302.434 386.341 101.07 175.742 430.873 24.9287 224.745 361.412 182.594 62.3705 282.751 102.537 88.9699 73.205 25.7829 37.9139 84.9152 73.096 206.128 60.6899 120.224 81.4445 18.8392 14.5179 9.4393\ninternal_count=246019 210515 35504 35085 32400 11371 200195 10320 21029 2685 4783 1965 195412 1790 699 7608 3537 4163 17492 4336 126 3563 1922 720 1739 1091 955 346 1154 51\nis_linear=0\nshrinkage=0.03\n\n\nTree=74\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 4 2 11 18 22 2 1 18 14 21 4 1 12 7 6 2 18 4 9 18 14 2 1 4 13 13\nsplit_gain=294.686 94.9268 50.5981 43.8816 25.6664 21.5617 20.2207 18.8186 18.5339 18.4146 13.3895 14.4085 12.4887 11.461 10.1316 8.67643 8.49744 9.35103 10.0141 15.3073 8.29737 10.6054 8.19066 8.90999 7.72079 13.3364 7.59486 7.40599 6.73469 8.18195\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 2.1468275719512682 2.7826617956161503 -14.450672149658201 0.66750229771629133 2.4414009612931857 0.93750000000000011 -30.420168876647946 4.0016611463566001 3.3497501700808394 0.69039108051708775 4.5000000000000009 6.9750976562500009 1.8223781763416411 0.86223072954082747 4.2558593750000009 6.0937500000000009 50.658309936523445 3.5972184255501678 6.3784179687500009 1.0000000180025095e-35 1.2424533248940002 0.67306332231011912 -16.491968154907223 0.63323550451436506 2.3505859375000004 1.0000000180025095e-35 -0.17886242654860576\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 4 7 -6 20 28 -4 -3 11 22 14 -7 27 -15 17 18 19 26 21 -1 23 -5 -12 -26 -11 -13 29 -2\nright_child=2 9 8 10 5 13 -8 -9 -10 16 24 12 -14 15 -16 -17 -18 -19 -20 -21 -22 -23 -24 -25 25 -27 -28 -29 -30 -31\nleaf_value=-0.021794010834861266 0.001169539856013303 -0.02867273544753465 0.028816809086626771 0.015278687341436822 -0.027465638634158261 0.014150216900392481 0.013942534561056196 -0.022622700037644636 -0.001097635348211886 -0.018080448718174123 0.018549315239175385 0.0019290043528951476 -0.031345103031940234 -0.025448244931272845 0.027530419834633404 0.011226573615266709 0.027169485905123466 -0.030821771046861894 0.028976918159807939 0.023952307585424458 -0.030262524346192463 0.018217460091495236 0.029239001429603069 0.0045080317980722463 0.023472227382395253 -0.014314902832997798 0.010716005700663691 -0.023210530498071719 0.024498919312523904 -0.024707285557063374\nleaf_weight=65.824436206035898 50.591497076762607 11.638540676474802 131.64778988587204 112.84980709571391 14.760748111744759 99.341389801411424 10.103001998737453 76.403276635202928 21.604079972952604 25.246250738797244 193.95696446567308 42.941837176273111 11.95342853994225 12.196320399525577 9.7791602052748186 9.5743014842737448 19.260693570366129 5.0323328566737464 15.058331297943367 17.899891746172216 289.76350475366053 5.7971041592536485 17.862699557095766 179.24535788118374 14.152472662273793 17.902721143327653 11.228026103577575 12.924659609328954 7.1602090448141089 13.022806989552917\nleaf_count=24100 1323 6155 329 335 2986 2444 97 19464 89 3112 582 1844 1531 793 27 136 48 241 44 313 175996 146 40 2068 104 157 448 681 17 534\ninternal_value=-0.000139301 -0.019672 0.00874274 0.00600457 -0.00375785 0.00562466 -0.0267847 -0.0123992 0.02462 0.00542169 0.0105069 0.00709252 -0.0044238 0.0098495 0.000875077 -0.00953465 0.00991342 0.00514917 0.00815687 0.00197182 -0.0279586 -0.0183492 0.00998063 0.00870716 0.0163048 0.00257362 -0.00916651 -0.00419289 -0.00120114 -0.00442823\ninternal_weight=1526.72 476.852 1049.87 896.62 283.051 135.873 371.488 147.178 153.252 105.364 613.569 387.557 77.5991 121.112 65.6457 21.7706 93.7255 74.4648 69.4325 54.3742 361.385 71.6215 309.958 292.095 226.012 32.0552 36.4743 55.8665 70.7745 63.6143\ninternal_count=246184 210700 35484 35066 27697 6359 200339 21338 418 10361 7369 6526 4083 3373 2552 929 4206 4158 3917 3873 200242 24246 2443 2403 843 261 3560 2525 1874 1857\nis_linear=0\nshrinkage=0.03\n\n\nTree=75\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 7 4 7 2 3 2 4 4 0 19 12 2 7 3 4 12 0 13 2 13 2 2 6 5 0 3\nsplit_gain=294.636 75.5187 53.4087 42.1578 19.0717 19.1747 20.3965 18.7656 17.0006 15.7598 14.9768 13.8005 13.6626 13.2456 11.2314 11.9145 10.3092 15.0338 12.1694 9.98504 9.7787 10.2566 8.97243 8.35881 8.23302 8.06259 8.31355 10.064 12.3551 12.389\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 2.1468275719512682 0.89599609375000011 1.0693359375000002 1.0000000180025095e-35 -30.420168876647946 0.38657275885365588 -30.420168876647946 1.7380371093750002 5.9760742187500009 1.1851094193663554 261.50000000000006 0.28848765710462548 39.736396789550788 2.9707031250000004 0.53650715206675215 1.7380371093750002 0.86223072954082747 0.60985949876859957 -0.48217012181339464 -21.560756683349606 -0.45698050410493757 -35.542255401611321 18.240050315856937 4.3378906250000009 2.0131835937500004 -0.53258533561495902 0.65908129556435835\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 3 7 5 6 -5 -2 12 -3 -9 -10 -1 -7 16 22 17 18 -6 -11 -20 -22 -16 -14 -8 26 27 -12 -29 -30\nright_child=2 9 -4 4 14 13 24 10 11 19 25 -13 23 -15 15 -17 -18 -19 20 -21 21 -23 -24 -25 -26 -27 -28 28 29 -31\nleaf_value=-0.030456489531276651 -0.026880785990460232 -0.028674221704267255 0.024744800562765928 -0.019008664822492052 0.023343596963932633 -0.014944671891777223 -0.014186340638459808 -0.013284435371358885 -0.022750110673562838 0.0017402116722481803 0.0058201630012927399 0.021486885085570912 0.01535512411420915 0.014850350377550853 0.007493285417768886 0.0231244964596071 0.028597449586701216 0.020315699458685332 -0.016937687039668924 0.026028383091814607 -0.01530896831969346 0.0099158005212545402 -0.045572183477866894 -0.019603661852308191 0.01536263431513884 0.01023407267109106 0.021608820761161242 0.015506596481912693 -0.027294700980882172 0.022113750282649305\nleaf_weight=309.32073640347517 28.474203922422021 11.538305740192298 151.12334526563063 19.089324449189007 31.46971178217791 62.881386112014297 8.3264569470193219 69.052103032270679 12.651704793097453 66.646838552260306 38.393779642152367 10.870512278866952 8.8573981719091552 16.139384797308594 4.104272992350162 5.7654076633043578 45.598471521399915 142.99151123221964 18.368923184461892 18.811225317418575 16.164308239240199 95.239291889593005 6.9191430746577689 15.65784367037122 126.76381666981615 78.100638998905197 10.360505043063311 9.3293673499720153 36.176455689506838 4.4485469926148644\nleaf_count=191225 8044 6254 415 1410 221 2083 206 11049 4372 4137 2511 468 862 295 80 14 109 503 984 51 141 523 84 3428 816 2119 29 348 3434 20\ninternal_value=-0.00057745 -0.0206611 0.00833413 0.00547203 0.00992234 0.00319576 0.00933505 -0.00425904 -0.0270176 0.00279447 -0.00154084 -0.00226397 -0.0287498 -0.00874973 0.0141806 -0.00934284 0.015333 0.0132577 0.00691199 0.00732579 0.00277931 0.00615213 -0.0268347 -0.00686622 0.0134505 0.00309448 -0.00260051 -0.0056496 -0.0144062 -0.0214831\ninternal_weight=1479.63 454.355 1025.28 874.157 599.821 233.2 154.18 274.336 357.358 96.9964 245.861 23.5222 333.836 79.0208 366.621 16.7888 349.832 304.234 161.242 85.4581 129.773 111.404 11.0234 24.5152 135.09 176.809 98.7087 88.3481 49.9544 40.625\ninternal_count=246235 210797 35438 35023 7469 4810 2432 27554 200355 10442 19510 4840 195515 2378 2659 178 2481 2372 1869 4188 1648 664 164 4290 1022 8461 6342 6313 3802 3454\nis_linear=0\nshrinkage=0.03\n\n\nTree=76\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 7 0 13 0 7 3 0 5 14 9 8 18 7 11 2 9 18 18 3 10 11 0 7 11 18 19 2 14\nsplit_gain=218.188 89.5871 58.4583 53.546 29.3333 36.2014 18.0376 20.2847 19.0389 14.5574 13.9729 13.47 13.0314 16.0983 12.3732 12.2521 12.0454 11.7519 10.6508 9.74246 9.61974 9.56528 8.45779 8.44773 7.96034 8.85521 14.0169 7.75803 7.49836 6.59845\nthreshold=0.31840721315068282 1.0000000180025095e-35 3.4169921875000004 1.1495945197163422 0.14796868535883259 1.3771184689138856 0.89599609375000011 0.40911314460322756 0.29549219484135919 0.89599609375000011 0.4896816566336385 1.0000000180025095e-35 1.0000000180025095e-35 5.2755571088971331 1.0000000180025095e-35 0.28995392115848545 -25.182852745056149 1.0000000180025095e-35 5.0530509062989237 0.89587973461402759 0.53650715206675215 1.2424533248940002 8.8817841970012543e-16 1.2225960941547209 0.89599609375000011 0.45066394906892915 1.4978661367769956 261.50000000000006 -6.7439365386962882 0.050734622491290134\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 12 5 6 7 -2 11 14 -10 29 -1 18 -9 -3 -4 19 21 -5 23 -14 -18 -12 -7 -26 -27 -11 -13 -8\nright_child=4 15 16 17 -6 24 8 9 10 27 20 28 13 -15 -16 -17 22 -19 -20 -21 -22 -23 -24 -25 25 26 -28 -29 -30 -31\nleaf_value=-0.02827641385343491 -0.026125591097955002 -0.022102804374909302 -0.026680351311125481 0.012833512065835617 0.023974139375730636 0.0086511610985522264 0.012289658834231559 -0.014938142267698377 0.017518598060379299 -0.0197661399040618 -0.0016296813060721217 0.02383414656268067 -0.030657516333126522 -0.031645883525565778 0.013279527501916079 0.022571098013241076 0.02189984180758207 0.029027522106905873 0.026870090282093806 -0.010638620126395569 0.010479644239514011 0.017067110430588259 -0.0078149697687503253 -0.038337814930480045 0.021054755818108405 -0.04327059756358214 0.015525600434229681 0.0047404998172776922 -0.028669789995344315 -0.034472144595146255\nleaf_weight=342.90044313629915 33.42290790885454 5.2759746345691374 5.9356390305911182 20.669073412427679 110.9657887397334 99.961375038721599 2.6421301843947758 16.762568672769703 110.32118331687525 37.425105219648685 60.39142993153655 8.8121891372720693 8.237748443149032 17.055705889797537 66.336693153483793 60.925114736659452 39.542431202076841 9.6441544509725627 10.606828287127426 59.451662236577249 76.183034266578034 5.5566384388366705 10.39362220224575 5.2363603700650847 179.2525084593799 4.2634967621415845 10.918503796216099 16.192319446417969 2.3392461409093803 18.399042950361036\nleaf_count=197085 9047 178 3251 553 327 835 76 2498 953 3571 3443 222 1292 6609 1033 318 793 53 260 5905 781 194 419 153 551 46 51 859 147 4691\ninternal_value=-2.92293e-05 -0.0139625 -0.0180743 -0.0214916 0.00963699 0.00748338 0.00216393 -0.00554211 0.00677848 -0.000356937 0.00955985 -0.0146369 -0.0263524 -0.0101471 0.00747759 0.0187348 0.0107925 -0.000617623 0.0055341 -0.00449986 0.00328906 -0.011579 0.015683 -0.00509002 0.01566 0.0192006 -0.00263952 -0.0124139 0.0113674 -0.0283091\ninternal_weight=1456.02 596.195 529.994 474.122 859.826 748.86 454.464 170.14 284.325 136.717 252.132 32.1926 384.357 41.4569 83.0993 66.2011 55.8717 89.7649 24.4012 80.1207 141.811 13.7944 49.9361 65.6278 294.396 194.435 15.182 53.6174 11.1514 21.0412\ninternal_count=246194 216910 216414 211951 29284 28957 27474 17008 10466 7961 5330 5136 205440 8355 3531 496 4463 6511 1746 6458 4377 1486 1212 3596 1483 648 97 4430 369 4767\nis_linear=0\nshrinkage=0.03\n\n\nTree=77\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 2 22 3 7 1 3 4 18 4 4 11 19 18 18 4 3 2 3 14 2 2 14 2 18 2 4\nsplit_gain=271.864 76.6112 39.3305 32.9215 25.5413 25.2467 23.1502 16.8753 14.8854 16.6445 12.6945 12.6405 13.6361 12.3768 11.3944 10.1977 8.3534 8.50167 9.34554 12.0013 8.16596 9.48913 10.8913 8.3182 8.22274 8.12672 8.1126 9.84674 8.34101 9.99601\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 2.5144622974930591 -34.692844390869134 0.93750000000000011 0.38657275885365588 0.89599609375000011 -0.32709731491006455 0.0037166969630965979 5.5791015625000009 1.7005986908310777 2.2978515625000004 5.9760742187500009 0.042545688445982228 243.50000000000003 2.0126758453675748 2.6735537653587351 3.4626464843750004 0.44825707527455844 20.132114410400394 0.56033297298897644 0.67646516896599407 -6.7439365386962882 -7.9326825141906729 0.67306332231011912 11.680273056030275 1.2424533248940002 9.8937015533447283 6.9193702936172494\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 4 -2 -4 -1 14 9 -3 11 -5 15 -8 -6 -13 -14 -18 -19 -20 21 22 26 24 -24 -12 28 -28 29 -9\nright_child=2 8 5 10 7 -7 13 20 -10 -11 25 12 16 -15 -16 -17 17 18 19 -21 -22 -23 23 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.029179765052140382 -0.028864012819100535 0.014163319993678446 0.028410953046477169 0.014326747217194806 -0.010791558438021112 -0.0079604210680786509 -0.018279315030258033 -0.0066092652223943378 0.01084927761804623 -0.033538435646627042 0.029618067130609337 -0.0095907931802684341 -0.030476702012272927 0.022329412165349203 0.02214663629534715 0.012116360538127761 0.011129830767995048 -0.025248948103646959 -0.033957726525707235 0.021050341668380351 0.013735283109809751 0.015648122930643067 -0.018583932358559426 -0.036750642034533293 0.027337909497655022 -0.0068933191443268848 0.017955602633278708 -0.0419129245344066 0.03107565920334621 0.018757286529614083\nleaf_weight=309.89703315326187 26.406262868535123 9.9948631381557806 117.33937818394043 222.36733800196089 106.34984504175372 19.646246336400509 13.45594111987157 40.103051968442742 68.774898649993702 16.04484850159497 43.458105004858226 21.921031643927563 10.276867572218178 11.680620099417863 9.434192775981499 136.76869240851374 24.192330488935113 18.980718663020525 4.0271022564265868 11.425659470492972 80.127786764875054 32.326301403343678 9.0902442125661747 11.827639404102227 4.3886872611474272 5.8751874081790438 3.0739406980574113 6.1688046001363546 9.4963553630514053 20.203433443617541\nleaf_count=195572 7642 414 332 641 15578 84 4369 2467 4067 5861 101 760 82 461 126 1861 126 641 204 77 1405 318 1279 1184 53 74 11 324 61 174\ninternal_value=-0.000309574 -0.0202204 0.00830135 0.00590331 -0.00102595 0.0231747 -0.0269407 0.00126417 0.00355972 -0.0153823 0.0108776 0.00925275 0.00425426 0.000723358 -0.00794107 0.0090765 -0.0068228 -0.00223478 -0.0116736 0.0055222 0.00619406 0.00170537 -0.00272136 -0.0194486 -0.00277652 0.0252193 0.00280994 -0.0221928 0.0063873 0.00205619\ninternal_weight=1425.12 429.848 995.275 858.29 358.997 136.986 335.034 332.59 94.8146 26.0397 499.293 449.96 227.592 25.1366 115.784 158.69 68.9027 58.6258 34.4335 15.4528 216.806 136.678 104.352 25.3066 13.4789 49.3333 79.0456 9.24275 69.8028 60.3065\ninternal_count=246349 210744 35605 35189 30622 416 200402 22980 10342 6275 4567 4392 3751 4830 15704 2621 1130 1048 922 281 7276 5871 5553 2516 1332 175 3037 335 2702 2641\nis_linear=0\nshrinkage=0.03\n\n\nTree=78\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 13 1 4 2 5 17 2 22 1 4 4 4 13 18 21 4 6 13 9 2 2 7 5 6 18 14 19 4\nsplit_gain=243.573 85.1923 40.5293 42.7553 23.2085 21.109 19.3436 16.197 15.7881 15.5136 12.8265 12.679 11.2074 23.6556 9.94783 9.03495 8.69976 8.64873 8.38258 10.273 6.93427 13.9374 7.30967 7.0447 6.82989 10.6096 6.31773 8.99018 9.5107 10.0192\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 3.3033179733375269 0.84667968750000011 -10.210010528564451 4.4121093750000009 3.9281938047451082 -30.420168876647946 0.93750000000000011 1.5661550609387962 6.4848632812500009 1.4716796875000002 1.0693359375000002 -0.48398449089866569 0.89587973461402759 5.5000000000000009 1.9753417968750002 1.4980468750000002 -0.56995612852957833 1.0000000180025095e-35 23.600493431091312 -18.451059341430661 4.1191406250000009 5.7207031250000009 3.3837890625000004 2.8617925509761908 0.52141881878679586 263.50000000000006 3.2895507812500004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 4 -2 10 -1 12 -3 -4 -6 -8 13 14 -7 -5 17 -11 19 -17 21 22 -10 -22 25 -14 -12 28 29 -28\nright_child=2 8 9 15 5 7 11 -9 20 16 26 -13 24 -15 -16 18 -18 -19 -20 -21 23 -23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.028405321927953892 -0.0253431843954291 -0.028287294632016818 0.027993473907677598 0.022728923491017156 -0.026524280000368099 -0.023409220279271857 0.023009967035855821 -0.031748741774386191 0.0085645098534198345 0.0036612789167099189 0.0028618137049725317 -0.022802087887305502 0.0081013083338845257 -0.029129631131823874 0.015939030399849639 -0.01293937528456918 0.022954256023922877 -0.044680338065343175 0.028590815345344509 0.011102192846005722 0.021811389141909936 0.015752972840177615 -0.033199031186101929 -0.0060843134332766368 -0.035460660736579518 0.030605155097817836 -0.027469996029861086 -0.028749622587667695 0.021649580082859273 0.01084534331480452\nleaf_weight=306.24195535424224 29.19465344256605 10.217444448950117 109.40746261493769 80.492272456176579 26.417074580444023 6.0749825654202132 10.819510432076642 9.1239713630056922 4.9598463107831803 8.1912364922463912 61.451972178881988 9.0752413321897603 256.46152723720297 21.796827069192659 32.35256327613024 16.466743671102449 7.719325875863432 4.4605560842901459 21.693448989186436 212.5149909745669 41.489066106674727 20.226916967891157 11.182557875406927 9.6145034094806743 2.0766934026032677 19.56534137390554 14.271028539369583 16.590797867742367 9.6165778485592437 9.554832229157908\nleaf_count=195489 6865 6164 321 153 8179 266 1144 1929 651 27 568 3730 6500 2883 956 162 25 34 63 1382 695 456 2087 292 31 42 1727 2812 195 393\ninternal_value=-0.000243697 -0.019206 0.00800617 0.00559512 0.000178355 0.00176246 -0.0265488 0.00570116 0.00532812 0.0235551 -0.0081309 0.00202108 0.00680169 -0.00486858 0.00928254 0.0139855 -0.000476445 -0.0154066 0.0110988 0.00932455 0.00952998 -0.000508944 -0.0206159 0.0165471 0.00933732 0.00980115 -0.00357543 -0.0113829 -0.00218277 -0.0122866\ninternal_weight=1399.32 423.827 975.495 845.716 514.549 485.354 326.137 347.452 97.6903 129.779 137.902 19.8948 338.328 60.2244 38.4275 331.167 20.3711 12.6518 250.675 228.982 87.4729 36.3693 16.1424 51.1036 278.104 276.027 111.485 50.0332 33.4424 23.8259\ninternal_count=246221 210708 35513 35106 33346 26481 200363 12607 10345 407 13874 4874 10678 4105 1222 1760 86 61 1607 1544 4181 3194 2738 987 6573 6542 5695 5127 2315 2120\nis_linear=0\nshrinkage=0.03\n\n\nTree=79\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 1 4 2 4 3 18 4 2 22 14 17 4 2 2 12 18 21 21 4 18 21 19 21 0 19 2 2\nsplit_gain=232.809 78.8041 37.5612 34.4079 20.3754 16.567 14.9544 20.5352 14.901 22.7738 14.8747 14.124 10.0569 10.3455 9.42146 8.92426 8.77477 8.15999 7.69272 7.63739 11.1282 7.58245 7.57916 9.28638 9.88248 7.51904 7.16544 10.3848 7.0801 6.88342\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 2.1468275719512682 3.8417438268661503 -13.633526802062987 4.6120605468750009 0.47260169401963864 3.6242520361853052 5.9760742187500009 -30.420168876647946 0.93750000000000011 0.33900667957710101 2.1810207941665634 2.6340332031250004 -19.576506614685055 -19.986686706542965 0.79622554818162328 1.4978661367769956 5.5000000000000009 3.5000000000000004 0.84667968750000011 1.2424533248940002 5.5000000000000009 249.50000000000003 4.5000000000000009 1.3170161875856263 251.50000000000003 23.600493431091312 10.479034423828127\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 3 4 12 -6 7 14 9 18 -3 -4 13 15 16 -2 25 22 -1 20 -13 -18 -9 -24 -25 -5 27 -14 -8 -29\nright_child=2 10 11 6 5 -7 28 17 -10 -11 -12 19 26 -15 -16 -17 21 -19 -20 -21 -22 -23 23 24 -26 -27 -28 29 -30 -31\nleaf_value=-0.0012571452017600454 -0.027705772623731511 -0.028201841584975674 0.027756271892223031 -0.0037867733466677533 -0.032165610006334049 0.011072889696573 0.010058455893912552 0.018724383515955276 -0.029761347154801738 0.024259257699590218 0.0089420057547004907 0.0091671064011559431 -0.03389180076753099 -0.030096407758396483 -0.018574261960821736 0.0095565759000543395 -0.029055077515949979 -0.021932712779874994 -0.027166052554598835 0.022372166227734493 -0.04539684040372273 0.0077553750798436292 0.012232159227596323 -0.030210861636338088 0.004554453102474515 -0.037835420955318118 -0.0018797592073702203 -0.018043925250601299 0.028351389419102969 0.0063605473213402411\nleaf_weight=13.503264335624406 5.6147197231184682 9.9542360639461531 103.05032606050372 29.599117958103307 7.7666389383666674 91.028958066366613 32.59557086735731 120.86577727273107 245.45246725279139 11.058232647716066 85.253251727233874 7.3281495012342948 28.948632686791825 7.7744269342074395 21.908167858520756 46.277873727871338 4.305542064481414 5.328688556328415 44.083729833015241 7.3578939251601687 4.8256758693605661 110.12576391390758 158.88376275094924 8.8656079341308196 28.199344369466417 6.3598013842711216 26.926117790164426 35.180343737927615 50.915456918766722 14.002229671226813\nleaf_count=1525 1669 6191 317 1149 1747 1923 351 276 174726 230 4181 25 9729 2011 832 2401 552 24 23310 21 35 1600 1687 185 343 357 839 6654 138 623\ninternal_value=-0.000136805 -0.019066 0.00791148 0.00563572 -0.00333967 0.00738552 0.00972678 0.00771343 -0.0262941 -0.0136471 0.00480573 0.0233641 -0.00977799 7.36799e-05 -0.000647244 0.00507725 0.00206972 0.0121733 -0.021216 -5.54831e-05 -0.0143017 0.00613065 0.0128142 0.00909202 -0.00433631 -0.0106695 -0.0153212 -0.0198711 0.0214149 -0.0111077\ninternal_weight=1373.34 409.305 964.035 841.473 263.52 98.7956 577.953 494.442 314.098 68.6452 95.2075 122.562 164.724 59.667 172.298 51.8926 150.39 322.143 57.587 19.5117 12.1538 114.431 316.814 195.949 37.065 35.9589 105.057 78.1312 83.511 49.1826\ninternal_count=245651 210163 35488 35090 27596 3670 7494 7005 199791 25065 10372 398 23926 6081 4490 4070 3658 2515 24835 81 60 2152 2491 2215 528 1506 17845 17006 489 7277\nis_linear=0\nshrinkage=0.03\n\n\nTree=80\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 1 13 7 17 2 2 5 0 1 1 17 12 14 17 7 18 14 5 7 7 13 19 13 13 18 14 0 1\nsplit_gain=243.173 79.8153 35.3436 31.2226 22.9009 14.0163 13.8559 13.4846 13.2123 12.1628 11.5562 11.2935 10.337 9.56263 11.1502 8.93044 8.77506 9.04143 8.76335 9.43798 13.3603 10.5413 9.20996 8.86511 8.51597 7.99711 7.20801 6.89624 6.72952 9.4472\nthreshold=0.95935769349451683 1.0000000180025095e-35 3.3033179733375269 0.14796868535883259 0.89599609375000011 3.9281938047451082 -30.420168876647946 -10.988907814025877 4.4121093750000009 1.6765212893675572 0.44640776988282155 1.4475910192202701 0.068176441062887241 0.86223072954082747 0.80526343422453606 1.6214838777669847 2.1386718750000004 0.89587973461402759 0.84677952177506588 0.89599609375000011 1.0000000180025095e-35 1.0000000180025095e-35 -0.67453178091691768 244.50000000000003 -0.51158128144345905 0.14796868535883259 0.89587973461402759 0.58961150764511439 1.1210156985015984 1.8223781763416411\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 3 4 11 7 -3 9 26 28 -10 12 -2 14 -8 18 17 -4 19 20 -13 -21 -23 -22 -25 -19 -1 -9 29 -6\nright_child=2 6 16 -5 5 -7 13 27 10 -11 -12 15 -14 -15 -16 -17 -18 25 -20 21 23 22 -24 24 -26 -27 -28 -29 -30 -31\nleaf_value=0.0038825951936303587 0.016776782745890905 -0.02815108619198102 0.022379818238320876 0.021624236023215095 -0.022910281640185715 -0.025837388637594116 0.0072089314322501029 0.014726682497106725 -0.028362497211605862 0.027648856596422591 0.01867598808432858 -0.014949812107320656 -0.031511408529466954 0.026141606836034715 -0.02896464761454565 -0.020705310747926291 0.027288140321778512 0.0085111826336244107 -0.02333644067432588 0.012093559416672036 -0.029141153363003762 0.017359386354592447 -0.022339081355071979 -0.0091366359064224186 0.024825333480563608 0.024234780401916501 -0.029303763002501055 0.0043689049732978432 -0.027509391312327301 0.0046643748549685853\nleaf_weight=5.791982101975008 4.0716183071490404 9.6274967427889333 79.813342120498419 74.830134518910199 15.081686991456079 11.173975798941681 59.173858927330002 139.43847893347265 7.2010944177600349 8.4093601885251683 10.017800036090192 15.132838859339243 24.699581034990842 19.403359358664602 7.7960521905333726 23.716524638526607 49.027375357691199 215.27037760405801 13.211412088538053 16.147012409113813 2.7416613451205185 5.4835159075446418 32.155647734150989 7.3652682285755864 40.776437880063895 33.258162628859282 279.49161060745246 100.3341621287982 12.77712521178182 35.902038493513828\nleaf_count=475 457 6171 177 246 3739 2104 1976 1685 4309 17 503 1873 10074 58 2180 3213 164 1486 1963 808 260 119 2198 158 492 120 195322 2511 173 1501\ninternal_value=0.000132717 -0.019541 0.00830432 0.00366593 0.00098658 0.00580413 0.00425992 0.00701649 -0.0270835 -0.00430713 -0.00147293 -0.00739102 -0.0243438 0.00815342 0.00263373 -0.00414436 0.0154471 0.0136015 -0.00105905 0.00157047 0.0091092 -0.00767983 -0.016256 0.0164146 0.0195493 0.010704 -0.0286188 0.0104286 -0.00888363 -0.00377639\ninternal_weight=1359.32 398.503 960.818 583.448 508.618 323.117 96.0008 311.943 302.502 72.1702 17.2189 185.502 28.7712 86.3733 66.9699 156.73 377.369 328.342 133.014 119.802 66.0162 53.7862 37.6392 50.8834 48.1417 248.529 285.284 239.773 63.7609 50.9837\ninternal_count=246532 210994 35538 33591 33345 11730 10385 9626 200609 5430 4812 21615 10531 4214 4156 11084 1947 1783 7871 5908 2783 3125 2317 910 650 1606 195797 4196 5413 5240\nis_linear=0\nshrinkage=0.03\n\n\nTree=81\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 17 13 4 0 2 13 14 0 22 19 2 4 3 3 17 1 5 17 13 5 2 12 12 0 17 19 4 1\nsplit_gain=233.468 68.7861 36.7602 34.3889 30.486 22.5844 17.305 17.2736 15.1372 13.9964 12.8431 12.708 12.5361 11.592 10.7866 10.1347 9.47876 9.19792 10.4889 8.70489 8.5555 8.11761 7.93107 7.77515 10.1846 7.16287 11.5965 6.97018 6.61631 6.59554\nthreshold=0.98142480879535021 1.0000000180025095e-35 3.9281938047451082 0.14796868535883259 0.84667968750000011 1.3771184689138856 -35.542255401611321 0.14796868535883259 0.42891595223756424 1.1851094193663554 0.93750000000000011 263.50000000000006 -30.420168876647946 3.8073730468750004 0.36709515926886871 0.42638606281916264 1.0329219259280322 1.5275240303550033 4.2558593750000009 0.15621648314530187 -0.55973954933580472 2.4414062500000004 -41.62170219421386 0.86223072954082747 0.026890617030159995 1.0917760095905165 0.96280603246018803 243.50000000000003 5.7263183593750009 1.4475910192202701\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 7 3 4 22 6 11 9 13 -1 -5 -6 -3 21 20 -7 -11 19 25 -16 -10 29 -2 24 28 27 -27 -19 -14 -8\nright_child=2 12 -4 10 5 15 8 -9 14 16 -12 -13 23 -15 17 -17 -18 18 -20 -21 -22 -23 -24 -25 -26 26 -28 -29 -30 -31\nleaf_value=-0.029773711886984997 0.0054418519967157708 -0.028023345459950676 -0.026325643207528483 0.028731611000883631 -0.026043581541024673 0.0055314718630641128 -0.01773708995350802 0.016008724950886614 0.005155273518097068 0.0085095912439573911 0.0015627966973574034 0.0058069568103335687 -0.011890782231332137 0.022610009686248312 0.013555943061980592 0.017791344825767597 -0.029303960303709124 -0.018774752040573505 0.024763351308090674 -0.027233118197742663 -0.026317557070837602 -0.032789325534279844 -0.030236694778638034 0.025188544586277513 -0.022622725750466696 0.0018666911387697489 -0.031707714787259665 0.010013805240104572 0.012939939767335248 0.0098302139156525286\nleaf_weight=262.47351000775234 6.9352821296197389 9.3614730082626902 26.014259224539273 97.466195697430521 27.371793332655216 85.068895088508725 7.4326599779014932 7.7325705394614479 11.565444812382337 15.40321741870139 18.761972148902714 18.195892674499191 11.462331659684422 60.849316185544012 6.0284843377303359 221.52868395263795 8.4271555187733593 7.4049918190867166 21.266115284524858 14.56370854450506 21.049182869421202 3.5393995512276879 23.273914379416965 15.853052147896959 11.149138777051119 35.182689909590408 11.517715372610835 82.4074691766873 43.833063246740494 123.45026879571378\nleaf_count=196442 808 6213 5055 322 4706 907 1572 45 1281 1889 77 925 1793 557 288 737 2651 754 93 2227 4253 256 4977 50 609 320 132 1942 1790 2220\ninternal_value=7.15688e-05 -0.019525 0.00820337 0.00922402 0.00695982 0.00814629 0.00386751 -0.0265779 0.00584088 -0.0277798 0.0243967 -0.0134736 0.00314318 0.0118375 0.000261192 0.0144187 -0.00550408 0.00318263 0.00566474 -0.0152171 -0.0153357 0.00683017 -0.0221339 0.00698229 0.00232938 0.0025311 -0.00690489 0.0074596 0.00767292 0.00814208\ninternal_weight=1316.57 385.696 930.874 904.86 788.632 758.423 451.825 294.036 406.257 286.304 116.228 45.5677 91.6591 195.272 210.986 306.598 23.8304 178.371 157.779 20.5922 32.6146 134.422 30.2092 82.2976 66.4445 136.513 46.7004 89.8125 55.2954 130.883\ninternal_count=245891 211482 34409 29354 28955 23170 21526 201027 15895 200982 399 5631 10455 4605 11290 1644 4540 5756 3241 2515 5534 4048 5785 4242 4192 3148 452 2696 3583 3792\nis_linear=0\nshrinkage=0.03\n\n\nTree=82\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 17 1 11 7 3 22 19 18 5 2 4 4 18 10 10 1 3 11 4 7 22 14 5 7 7 5 17 2\nsplit_gain=198.604 64.8173 38.9801 26.7635 23.1024 21.5475 19.7443 15.7876 15.5602 15.059 12.3468 11.4486 11.1227 10.4068 9.90798 8.99029 8.3333 9.13057 8.10935 9.03385 7.30991 7.1129 9.70667 8.90475 7.45924 7.06222 6.83654 6.82425 6.61975 6.58375\nthreshold=0.95935769349451683 1.0000000180025095e-35 3.325091732892195 2.9477339485081675 0.042545688445982228 0.89599609375000011 0.38657275885365588 1.0000000180025095e-35 238.50000000000003 2.4414009612931857 1.4980468750000002 -1.7340650558471677 5.9760742187500009 3.9514160156250004 2.249904835165133 2.524928003624769 1.7005986908310777 -0.13637628801644239 0.42638606281916264 0.28596955977216088 1.0693359375000002 3.2382812500000004 0.93750000000000011 0.38774031279192267 2.2500000000000004 1.0000000180025095e-35 2.3505859375000004 3.4169921875000004 0.95146732080301388 -10.988907814025877\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 4 5 9 14 -6 15 20 11 -9 -8 18 28 16 17 -3 19 -5 -2 22 23 -7 29 -22 -26 -25 -1 -24\nright_child=2 8 -4 13 7 21 12 10 -10 -11 -12 -13 -14 -15 -16 -17 -18 -19 -20 -21 25 -23 24 27 26 -27 -28 -29 -30 -31\nleaf_value=0.0046535317387481441 0.0070447560634916539 -0.018497925101821654 -0.024388241512830321 -0.0042280062534822103 0.020311099298505386 0.005438893212657149 -0.018680726180856599 -0.032742988427853163 -0.027795978264101139 -0.020840671343230633 -0.04108173494867122 0.029774401627732639 0.021271451200914025 0.025092643175963176 -0.029437083615450187 -0.028772093248396153 0.026932500562355688 0.011199475088459528 0.014202912866798466 0.019225052728994493 0.0092200142303883077 0.020107176015590977 0.0031475910065908706 -0.0236025925530969 -0.024380403406009273 -0.020778012948878471 0.013048831946268274 0.0010663235480591858 -0.028626209880878703 0.020261256759120647\nleaf_weight=13.969324964302361 44.647101365902927 10.973370233652529 32.427600306051318 49.178481402923353 92.776036928553367 44.582520793192089 11.65159337030491 2.9965341226779856 11.136410023871575 58.67262107921124 6.2643122704466796 6.9370717315468955 11.439504147332626 65.05507854768075 241.94304825377185 4.4462893522140794 20.856651442591101 41.525642862310633 288.9846017381642 19.980745315086097 9.5583205987932178 26.10738165769726 34.150380719162058 24.182049258728512 9.1329797079961299 21.719726902316324 6.9029894173436324 16.70372031518491 7.523883368499809 50.202096475288272\nleaf_count=1944 925 1870 5968 783 928 1388 4342 238 5726 15648 295 14 469 241 191156 348 335 2006 1386 105 271 243 1451 3211 639 619 156 672 2404 297\ninternal_value=-0.000123916 -0.0184565 0.00744239 0.00864341 0.00358479 -0.000216202 -0.0253806 0.0155648 0.0038848 -0.00956334 -0.0112204 0.00964577 0.00117327 0.0140673 -0.0277038 0.0087147 0.0113433 0.00476238 0.0120003 0.00272524 -0.000690934 0.00573629 0.0036208 -0.00378827 0.00990386 -0.0116092 -0.00844809 -0.0137394 -0.00775106 0.0134508\ninternal_weight=1286.63 375.466 911.162 878.735 455.536 346.562 286.527 108.974 88.9384 134.598 16.1979 9.93361 23.0911 423.199 263.436 77.802 73.3557 52.499 358.144 69.1592 75.9251 211.964 185.857 85.4683 100.388 31.278 16.036 40.8858 21.4932 84.3525\ninternal_count=246078 210600 35478 29510 26995 25520 200315 1475 10285 17463 547 252 4811 2515 195504 4559 4211 3876 2274 888 1815 8057 7814 5271 2543 890 795 3883 4348 1748\nis_linear=0\nshrinkage=0.03\n\n\nTree=83\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 4 13 3 2 18 4 7 11 22 18 2 9 18 1 1 17 17 17 17 1 3 12 18 2 21 12 12 6\nsplit_gain=191.362 61.8646 35.3302 31.813 26.0192 31.9015 17.1472 16.7614 14.3849 14.0749 12.7962 12.7104 11.5096 9.98513 9.27438 11.2871 8.96188 9.40354 9.96251 8.22767 7.8763 9.22373 7.59826 7.5175 8.32328 7.42744 6.92003 6.8565 7.83806 6.83374\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.84667968750000011 0.14796868535883259 0.47260169401963864 -19.986686706542965 3.6242520361853052 5.9760742187500009 2.5244140625000004 0.30499968189655369 0.93750000000000011 1.4978661367769956 -30.420168876647946 1.0000000180025095e-35 3.1984648276080736 1.5275240303550033 4.0016611463566001 0.28904399529622732 1.490403556610195 1.6895709195314625 1.0864924032472716 2.5144622974930591 0.066193923190483769 0.76965903234489019 1.4978661367769956 -1.7340650558471677 1.5000000000000002 0.86223072954082747 0.026890617030159995 6.3203125000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 9 4 5 -4 7 -1 11 -2 -5 19 -3 16 23 -16 17 -13 20 -7 21 22 -19 -6 -25 -12 -17 28 29 -14\nright_child=2 12 3 10 14 8 -8 -9 -10 -11 25 13 27 -15 15 26 -18 18 -20 -21 -22 -23 -24 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.01837147903827982 -0.031084804474887599 -0.027858211865267605 -0.01917494935781678 0.028311844057264533 0.014630561908762404 0.01279296444421164 -0.029720077987556728 0.022296780254056184 0.01668453759327726 0.014087876406340478 -0.026340271929711997 -0.031782595431718623 0.0022410762467459967 0.025436847821837968 -0.032834910610375447 -0.014330574718296577 0.012295712294152415 0.020333528140935173 -0.021064857653454248 -0.023148890511599747 0.012943026699279873 0.0063078897994179699 -0.02722597475744892 -0.041777652849822058 0.0087065567281002475 0.013233318470234956 0.0098672569579709036 0.02504915208467777 -0.01832683189661944 0.028962617723498448\nleaf_weight=54.579697335095261 28.309779870585771 8.6067696012905781 61.875195567234186 88.241957615944557 303.76644390041474 74.70532234525308 212.16603027825477 10.030815776961388 65.23064186249394 7.2426333155017337 5.3985350076109198 14.540770989668088 43.389341354981298 8.2284754457650688 6.5083323865546836 11.916871216031721 17.964321052771993 2.7886113125132423 28.60144645976834 5.2941984094213685 24.865043292607879 21.082875325693749 19.848934754292713 3.6003716159611923 7.4173962995409966 11.82200191169977 61.0986145575298 14.21116325818002 12.014612138213122 9.8102071629837138\nleaf_count=24974 6917 6122 10056 317 1597 415 175262 225 721 51 49 2028 3552 92 1124 605 170 45 4711 132 1143 328 3282 26 35 32 1572 52 601 37\ninternal_value=-0.000411667 -0.0186518 0.00715946 0.00840031 0.00618263 0.000162153 -0.0255977 -0.0118141 0.00444038 -0.0217487 0.023743 0.000715833 0.00322866 -0.0047947 0.0114369 0.00224904 -0.00689273 -0.0100597 -0.00651898 0.010158 -0.000237527 -0.00785548 -0.020755 0.0137436 -0.0098942 6.24835e-05 0.00577767 0.00690183 0.00262867 0.00761209\ninternal_weight=1245.16 364.809 880.349 844.796 739.334 345.026 276.777 64.6105 283.151 35.5524 105.462 217.92 88.0321 137.92 394.308 79.5238 129.692 111.728 97.1869 79.9995 68.5855 43.7204 22.6375 314.784 11.0178 17.2205 73.0155 79.4253 65.2142 53.1995\ninternal_count=246273 210825 35448 28480 28082 23123 200461 25199 13067 6968 398 12346 10364 11799 4959 3301 11707 11537 9509 547 4798 3655 3327 1658 61 81 2177 4242 4190 3589\nis_linear=0\nshrinkage=0.03\n\n\nTree=84\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 1 13 9 18 4 22 18 4 17 1 4 0 19 4 13 3 11 19 10 10 1 18 6 14 0 3 14 12\nsplit_gain=189.437 58.5911 32.4122 24.1009 19.3142 20.6558 20.3038 16.1803 15.5703 15.2103 14.0066 11.1257 12.3509 11.0413 10.3073 9.29243 9.01589 8.72713 8.06381 8.20439 7.79054 9.91621 8.41622 8.42011 7.79822 7.5056 11.3215 7.82917 8.43631 7.1655\nthreshold=0.95935769349451683 1.0000000180025095e-35 4.0016611463566001 0.14796868535883259 1.0000000180025095e-35 1.2424533248940002 8.2534179687500018 0.93750000000000011 3.6242520361853052 5.9760742187500009 3.2718994203396332 1.4475910192202701 3.5791015625000004 -1.131379951360411 263.50000000000006 2.6540527343750004 0.14796868535883259 0.3261289997719386 0.74021522701736098 237.50000000000003 2.524928003624769 1.7005986908310777 0.26032702032866095 2.9174053685313028 1.2426757812500002 0.33900667957710101 0.69651904584641833 0.36709515926886871 0.41892379644680228 0.276797460766049\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 3 4 5 -2 10 -5 9 -1 11 12 -7 -3 18 -16 -10 -14 19 -13 21 22 23 -15 -24 -21 -27 -28 -29 -6\nright_child=2 13 -4 7 29 6 -8 -9 16 -11 -12 14 17 20 15 -17 -18 -19 -20 25 -22 -23 24 -25 -26 26 27 28 -30 -31\nleaf_value=-0.018006068048860319 0.01115793704147159 -0.02358548856733482 0.01750154736590245 0.027127591641333198 0.024671099657193739 -0.028740182257673264 0.022473679163956265 -0.0079503530479754017 -0.029916665850460256 0.021600265566040434 -0.028069135804468761 -0.018325157302399562 0.018075162174691243 0.011471226197492638 0.015269529285440762 -0.034849399877716396 0.015016847835013277 -0.021900496112078387 0.017543545928867199 0.0046287548590846972 -0.027053486777918391 0.026589922138273169 -0.016748457223218393 -0.027317885371462541 0.01924316102281939 -0.0221078674725463 -0.023026183427224033 -0.037347612387735964 0.0026298329884210307 -0.0085020355423853344\nleaf_weight=53.299870824776008 129.43383294169325 11.08591739820258 223.35768904560246 59.755934260669164 42.174031622591428 27.8707494574046 26.447554172249511 14.214645414613186 202.91904467801214 9.5212099234922771 19.987052639466128 32.084928082098486 11.508323513335201 7.4780713117797886 45.098614579823334 2.6520849477383299 3.5343578608008093 7.1062532795476718 15.09767460497096 86.690959273284534 4.599239209550432 21.077950207632966 6.0092144866357584 12.039055904708219 25.103358786203898 31.469906701793661 13.132338587834964 3.9849732806906095 100.2868421239109 6.2763360585668115\nleaf_count=24900 685 5920 967 219 404 11163 231 67 175160 212 5719 2172 955 163 1155 162 29 767 62 2859 285 272 981 2524 265 5061 1065 115 1637 103\ninternal_value=0.000277136 -0.0181914 0.00761938 0.00432964 0.00232577 0.000717757 -0.00249245 0.0203707 -0.0250965 -0.0117589 -0.00420807 -0.00287208 -0.0164512 0.00312799 -0.000918281 0.0118282 -0.0290788 0.00247566 -0.00310501 -0.00433073 0.00726383 0.00980671 0.00243991 -0.0126209 0.0119865 -0.00235522 -0.00642582 -0.00207107 0.000761748 0.0202888\ninternal_weight=1255.3 356.667 898.631 675.273 601.302 552.852 423.418 73.9706 269.274 62.8211 396.971 376.984 46.4853 87.3928 330.498 47.7507 206.453 18.6146 282.748 267.65 76.3069 71.7077 50.6297 19.5171 31.1126 235.565 148.874 117.404 104.272 48.4504\ninternal_count=246279 210711 35568 34601 34315 33808 33123 286 200301 25112 32892 27173 12885 10410 14288 1317 175189 1722 12971 12909 4490 4205 3933 2687 1246 10737 7878 2817 1752 507\nis_linear=0\nshrinkage=0.03\n\n\nTree=85\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 13 1 3 7 18 9 0 17 5 5 7 0 7 17 0 13 21 18 0 13 7 3 14 18 0 19 7 19\nsplit_gain=171.685 53.4942 29.1652 28.3451 19.5045 17.5229 10.4446 10.1764 9.97609 9.64523 9.27072 18.3207 9.32535 9.14671 8.87131 8.58331 8.05527 10.02 7.42717 6.84806 6.68951 9.58703 8.49549 6.55846 10.7691 7.14179 10.1211 6.03118 6.19821 7.08699\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 2.9477339485081675 0.38657275885365588 0.89599609375000011 2.4414009612931857 1.0000000180025095e-35 -1.131379951360411 2.5380284121401901 0.89599609375000011 1.4980468750000002 1.0000000180025095e-35 1.1851094193663554 0.89599609375000011 3.9281938047451082 1.2661233611507157 -0.070829593282168909 5.5000000000000009 5.0271592211193568 0.29896707506516013 -0.32019508363135163 2.6738281250000004 0.47260169401963864 0.30268741420124606 2.4414009612931857 1.5447487307370646 234.50000000000003 2.2500000000000004 246.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 5 13 6 10 16 -3 -4 12 -12 -2 -1 -6 20 18 -18 -5 -8 21 -7 -23 24 -22 26 -26 -19 29 -29\nright_child=2 8 9 7 14 15 19 -9 -10 -11 11 -13 -14 -15 -16 -17 17 27 -20 -21 23 22 -24 -25 25 -27 -28 28 -30 -31\nleaf_value=-0.029231492357533449 -0.020413278658452817 -0.023288832217412316 0.023464005954561697 0.0076003746536282037 -0.028550874651517226 -0.034069513784864135 -0.021787763263332042 0.03170564958686925 0.006505718284902119 -0.017489991913665567 -0.038505309307256513 0.0072534500221176617 0.012586600388318931 -0.0071239606567189834 0.012858489660904599 -0.023187400399251341 0.01744515215040909 -0.021826185295494453 -0.0090971295877446245 0.0026272833475885844 0.012963142006428263 -0.019711987412784085 0.0171056329623273 0.011566725415550289 -0.039221373060165883 0.0019835202702160881 -0.0021488478319320412 0.0096445908959205875 0.033554314336912562 -0.022538481511482081\nleaf_weight=218.8036129885586 8.525379899539983 10.715538969569023 100.14142945082858 124.75961027352605 5.3358106094819986 8.0905359211028571 56.356595805977122 19.024967947043478 73.3613736294501 4.7577195409685364 13.806860004086049 16.25938016443979 40.453544695628807 19.061762751924107 16.595421657548286 9.1457687444926687 144.22399256727658 7.6439959346316746 28.729710482992232 12.367973018393966 41.939768698808621 8.3777853872161376 13.182754371431654 118.5426231113961 10.67918123630807 37.942382205161266 15.610417975811286 19.981138657080013 5.1284753680229178 7.7165726199746123\nleaf_count=191217 748 5814 403 1528 2671 1805 16416 48 4480 28 661 369 372 4349 2222 2045 432 86 414 2504 770 1940 288 1182 317 2839 144 77 9 64\ninternal_value=-0.000377028 -0.0183018 0.00669325 0.00465246 -0.0250109 -0.000719351 -0.0089547 0.0108222 0.00247932 0.0214589 -0.00146964 -0.0143227 0.00654906 -0.0274971 0.00197798 0.0039152 0.00955733 0.0137663 0.00443633 -0.0174193 0.00497666 -0.00822026 0.00255979 0.0067467 0.00130735 -0.00638527 -0.0180936 0.000551421 0.00625353 0.000205458\ninternal_weight=1217.26 343.874 873.389 768.489 259.797 411.281 147.77 357.208 84.0769 104.899 79.0452 30.0662 48.9789 237.865 21.9312 263.511 338.183 184.694 153.489 68.7246 254.365 29.6511 21.5605 224.714 106.172 64.232 26.2896 40.4702 32.8262 27.6977\ninternal_count=246242 210753 35489 35058 200459 32400 21070 2658 10294 431 2150 1030 1120 195566 4893 11330 2610 668 1942 18920 9285 4033 2228 5252 4070 3300 461 236 150 141\nis_linear=0\nshrinkage=0.03\n\n\nTree=86\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 13 1 7 0 22 13 0 2 2 4 0 0 7 17 5 3 4 13 0 4 1 10 10 14 17 0 17 1\nsplit_gain=162.145 61.3084 29.9307 29.5747 22.1927 13.8898 13.5381 12.757 11.786 11.3875 11.0543 11.0069 11.2558 10.6125 9.5184 9.41073 8.58382 7.59294 7.03104 6.83319 9.42443 8.37055 6.48268 6.37106 8.6417 6.21161 5.93569 6.89286 6.34569 5.88709\nthreshold=0.95935769349451683 1.0000000180025095e-35 0.14796868535883259 3.3033179733375269 0.89599609375000011 1.5447487307370646 0.93750000000000011 0.14796868535883259 0.69651904584641833 -34.252307891845696 -30.420168876647946 3.9514160156250004 0.31303196667935901 1.0212340148387515 2.2500000000000004 3.9281938047451082 1.4980468750000002 0.48555671035752257 6.1582031250000009 -0.1126198999233307 1.3771184689138856 2.2978515625000004 -0.90051303354301027 3.6242520361853052 1.7005986908310777 0.51224307552198323 1.0329219259280322 1.1851094193663554 0.42938087594322377 2.6981674883069502\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 7 3 4 13 9 -4 8 -1 -6 -3 12 -5 17 16 29 -8 -2 26 20 21 -14 -12 24 -24 -19 27 -10 -29 -11\nright_child=2 10 6 11 5 -7 14 -9 18 15 22 -13 19 -15 -16 -17 -18 25 -20 -21 -22 -23 23 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.030028504057795384 -0.025496336424051677 -0.027588020175172603 0.027237593299702541 -0.033575781774389558 -0.021671539299196338 0.01584152631270784 0.0029140736844980764 0.013328384439203167 -0.021627886440330535 0.0015365787866442676 -0.024132689494155757 0.026068121311289116 0.011164510838259032 -0.0023129998945062734 0.03223543730591337 -0.030869405216163152 -0.044431965572442167 0.0084790388577022837 0.012792869504282595 0.0028736827430191985 0.020032258030706417 -0.011819101504065939 0.0038290252251621361 -0.032716645934008221 0.025625616803811233 -0.019937979581123377 -0.02869680905458151 -0.02472030889098209 0.019149036787501534 0.014307937571328295\nleaf_weight=186.41421728538262 42.704183202557033 7.9490275552961966 82.24588453117758 4.3695076261647037 17.348387270052626 75.225760469795205 9.7244764789938944 6.9510525700170538 14.494879669677177 154.72013260982931 4.4773253952735095 42.578391070244834 61.591788890073076 92.405297546152724 4.6455016061663619 6.1322620985156382 4.2449393682181826 14.07445412874222 6.9235765488701864 74.682045955094509 95.963726358488202 17.581745213828981 47.080023105314467 2.1047908270265898 24.462664906983264 11.896617754944598 23.842386862830608 3.1351583435171042 9.8659938528144249 40.220966287888587\nleaf_count=173374 15629 6108 336 223 3791 395 35 47 10161 5896 603 192 513 3292 7 1278 37 603 235 458 225 140 3026 120 447 2003 14631 1076 785 294\ninternal_value=-0.000172207 -0.0177344 0.0067985 0.00472286 -8.96795e-05 0.00481521 0.0220655 -0.0251857 -0.0263297 0.000962789 0.0040879 0.0120725 0.00963518 -0.00900576 -0.00100973 0.00301324 -0.0134802 -0.0178487 -0.0141879 0.0105218 0.0137483 0.00597687 0.00761879 0.00984652 0.0115325 -0.0047946 -0.0180015 -0.0080457 0.00756016 0.00423119\ninternal_weight=1190.06 337.701 852.356 751.495 454.728 293.648 100.861 251.627 244.676 218.422 86.0738 296.767 254.189 161.081 18.6149 201.073 13.9694 68.6753 58.262 249.819 175.137 79.1735 78.1248 73.6475 71.5427 25.9711 51.3384 27.496 13.0012 194.941\ninternal_count=245960 210613 35347 34932 33181 11654 415 200309 200262 11259 10304 1751 1559 21527 79 7468 72 18235 26888 1336 878 653 4196 3593 3473 2606 26653 12022 1861 6190\nis_linear=0\nshrinkage=0.03\n\n\nTree=87\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 2 4 1 9 13 17 3 0 5 19 2 10 5 0 0 14 4 2 5 2 1 14 0 11 19 0 10 2\nsplit_gain=166.309 42.2324 40.1986 35.8918 26.0157 26.8111 22.4821 18.7871 13.6175 13.6164 11.4746 10.8416 10.2004 9.24142 9.01026 8.81354 8.78759 7.98721 9.97345 7.49084 10.1198 7.93126 7.7192 10.6588 7.41115 6.26637 6.23399 10.9531 6.03691 6.43802\nthreshold=0.24961073363565869 1.7005986908310777 -34.252307891845696 0.84667968750000011 4.0016611463566001 1.0000000180025095e-35 1.0000000180025095e-35 3.9653415636860969 0.37248792739977959 0.49000999047351379 3.8603515625000004 240.50000000000003 18.240050315856937 2.6735537653587351 6.3300781250000009 1.8987378677022575 1.4460303622695712 0.56086626140189244 7.2277780771255502 -6.3279094696044913 2.1386718750000004 -8.3227787017822248 1.9634439907166827 0.63637684326820176 1.3771184689138856 0.97194359927659313 264.50000000000006 0.96046990436194013 2.0126758453675748 -19.986686706542965\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 16 8 -4 5 6 7 9 -2 12 11 15 17 -3 -7 28 25 18 -5 20 21 24 23 -21 -13 -1 27 -10 29 -11\nright_child=2 13 3 4 -6 14 -8 -9 26 10 -12 19 -14 -15 -16 -17 -18 -19 -20 22 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.030164987340761975 -0.028863055977774787 0.023142035841529882 -0.021306978851499861 -0.016263657491898936 0.017924295174805531 0.023114683027847194 0.017904513539729969 -0.030094249388333667 -0.025372734513784589 0.015934835183998457 0.01672675310245289 -0.0045934310097692707 0.0088152330988479008 -0.024420766900247456 -0.0311266289802999 0.0068138429978871788 0.00095863663625947969 -0.032245028977142536 0.022859263453338694 0.0053482039640200593 -0.036590470182355901 -0.025809307882076003 0.011362891418478982 -0.025058847527406818 0.012900172378663319 0.01943367842552042 0.010774216939410986 0.0054804282673791404 0.029384825286540867 -0.025855618602741371\nleaf_weight=184.25692972566321 33.058206578440149 18.790005323098743 37.217897853261093 24.967637471774655 198.26527031580918 80.142085963510908 79.075003656442277 16.151696909466409 20.996722953219432 3.0377918553131158 53.857237633084878 49.491954996265122 17.651021494879387 3.6542221168201641 2.0186776004265985 17.184289814031217 8.798122656051417 18.78841667243978 6.6363618329050942 43.44229467283003 6.244276642202748 8.531516218223258 139.37254158314317 12.540947458561275 37.440250068728346 1.6855168938636769 15.863908936589722 19.010645293397829 1.3644164586439718 30.622306974473759\nleaf_count=163404 19891 657 12450 6245 771 978 377 6830 2134 163 600 4106 2832 368 64 196 1060 9516 100 3038 756 373 1256 2724 215 10 658 177 10 4147\ninternal_value=-7.81348e-05 -0.0237522 0.0052253 0.00716244 0.00843733 0.00551403 0.00317251 0.000750277 -0.0138973 0.00186912 0.00396897 0.0019596 -0.010449 0.0146868 0.0214237 -0.0105225 -0.0282348 -0.0172369 -0.00759929 0.00418127 -0.00240603 0.000186108 0.0076056 -0.00178358 0.0030018 -0.0296138 -0.0046905 -0.0109358 -0.0189814 -0.0217331\ninternal_weight=1190.16 217.185 972.973 884.044 846.826 648.561 566.4 487.325 88.9295 471.173 403.13 349.273 68.0434 22.4442 82.1608 52.2088 194.741 50.3924 31.604 297.064 101.708 95.4637 195.356 55.9832 86.9322 185.942 55.8713 40.0074 35.0245 33.6601\ninternal_count=246106 165499 80607 57747 45297 44526 43484 43107 22860 36277 17584 16984 18693 1025 1042 4516 164474 15861 6345 12468 5450 4694 7018 5762 4321 163414 2969 2311 4320 4310\nis_linear=0\nshrinkage=0.03\n\n\nTree=88\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 1 7 2 9 18 11 4 4 4 18 2 2 2 22 18 14 10 10 2 2 1 5 17 19 5 2 7 2\nsplit_gain=154.32 49.2556 26.8717 17.0397 24.8006 15.7158 14.2421 21.9836 10.855 17.3464 10.6762 10.3916 9.59273 9.23583 10.0017 8.17311 7.90326 7.68671 7.28427 8.77868 7.20245 9.42047 8.35474 7.67482 12.7944 6.61992 7.97388 6.53771 6.67038 6.46001\nthreshold=1.211689956917019 1.0000000180025095e-35 4.0016611463566001 2.5244140625000004 -2.5585746765136714 1.0000000180025095e-35 1.2424533248940002 0.75683518236239655 1.6286621093750002 1.6572265625000002 1.0693359375000002 2.8617925509761908 -30.420168876647946 5.8049912452697763 1.2144355773925783 0.93750000000000011 2.6735537653587351 0.57916335608139347 3.6242520361853052 1.8688348091416842 9.3619499206542987 11.076014518737795 1.6575619461558879 4.8398437500000009 1.2596354777617942 237.50000000000003 2.6738281250000004 18.240050315856937 1.4980468750000002 -19.19599628448486\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 5 16 -5 6 -2 8 10 29 13 -6 -3 14 17 -9 -1 -8 19 -14 23 -22 -23 24 -11 -24 27 -27 -29 -10\nright_child=2 12 -4 4 11 -7 7 15 9 20 -12 -13 18 -15 -16 -17 -18 -19 -20 -21 21 22 25 -25 -26 26 -28 28 -30 -31\nleaf_value=-0.01196983052036938 0.01081099150010372 -0.027476944818569979 0.016131026057589464 -0.026551751159808571 -0.032712492170714355 0.020236966165640927 0.0034833449494571043 0.027804431479135097 -0.0092670787553016092 -0.0030723100365681964 -0.025527510492406202 0.023013379149211814 0.001082727167842492 -0.023747727853618977 0.025839314084647048 -0.013142743917629901 -0.02904654914940924 -0.020726487713937994 -0.035820525314882688 0.022862805493947192 0.026825262108945626 -0.020680095447902492 -0.0099306573160962974 0.019837280913734861 -0.02557757064700469 -0.020408280785281074 0.02405504663099077 0.013323363290181878 -0.032940459543442102 0.029814265710833641\nleaf_weight=30.345355744022527 133.90265605971217 7.7503327004669691 217.49188446067274 25.844780268344039 2.6113593768095624 47.178987588733435 34.009999591304222 29.822995552793145 4.0811325343383968 103.79905719932867 29.72214729845291 20.227689805324189 58.195790055498946 16.59762488247361 10.766452087700598 4.5083235297352067 185.0759144403637 16.894185724115232 2.6955272019840768 22.394025564892218 13.286408022628164 13.077685668744378 20.152838029258419 8.5797536823665705 28.240582058686414 8.6314180474728328 20.688818158116192 23.952828854788098 2.2001139846397555 21.844266396947205\nleaf_count=8484 660 6236 980 23698 166 371 2292 108 251 5176 4535 2028 3897 2475 409 44 174370 2092 138 446 155 1201 611 74 3461 208 142 472 83 191\ninternal_value=-9.64471e-05 -0.0165561 0.00713683 -0.0230944 -0.00689141 0.00381052 0.00235343 -0.000422542 -0.00253929 0.00069902 -0.0105244 0.0155369 0.00245707 -0.00457684 0.000879285 0.0222075 -0.0267075 -0.00479599 0.0055356 0.00734094 -0.00182911 0.00421647 0.000568736 -0.00621272 -0.0080418 0.00446514 0.00977822 0.000851816 0.00836161 0.0235465\ninternal_weight=1164.57 355.141 809.43 264.105 48.6838 591.938 544.759 410.857 376.525 268.535 107.99 22.839 91.0357 78.2683 61.6706 34.3313 215.421 50.9042 83.2853 80.5898 242.61 101.99 88.7037 140.619 132.04 75.626 55.4732 34.7844 26.1529 25.9254\ninternal_count=245454 219463 25991 208746 25892 25011 24640 23980 23828 12025 11803 2194 10717 7268 4793 152 182854 4384 4481 4343 11583 2872 2717 8711 8637 1516 905 763 555 442\nis_linear=0\nshrinkage=0.03\n\n\nTree=89\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 13 0 12 18 6 0 10 2 7 2 12 12 11 12 18 7 5 0 7 5 5 17 17 19 5 18 9 14\nsplit_gain=153.582 51.6656 20.4947 20.4075 16.7343 14.8383 15.2993 11.6937 10.3074 9.63207 8.6649 8.0466 7.84283 7.56102 8.32657 7.50082 8.7117 6.79419 9.29675 10.6287 10.2016 11.4541 8.27419 7.43361 6.51951 6.47064 10.1771 8.75533 7.41221 6.15906\nthreshold=1.4475910192202701 1.0000000180025095e-35 0.14796868535883259 0.83245446090221253 0.21094178584231701 2.249904835165133 2.5244140625000004 -0.064617101914726832 3.7839313027319412 -60.293891906738274 0.89599609375000011 -5.9238033294677725 0.79622554818162328 0.43172124950159557 0.74021522701736098 0.76965903234489019 1.4978661367769956 2.4414062500000004 1.8691406250000002 1.6765212893675572 0.89599609375000011 0.89599609375000011 1.2426757812500002 1.2596354777617942 3.0352616308921099 240.50000000000003 3.7255859375000004 5.2067464199591091 1.0000000180025095e-35 0.050734622491290134\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 5 4 -1 13 9 12 -9 -7 29 -6 -3 17 -15 -16 -17 18 20 -20 21 -2 -23 -24 25 28 27 -27 -11 -5\nright_child=2 7 -4 10 11 6 -8 8 -10 24 -12 -13 -14 14 15 16 -18 -19 19 -21 -22 22 23 -25 -26 26 -28 -29 -30 -31\nleaf_value=-0.029143424510005005 0.012935603232737406 -0.018154846754593827 0.02176796556238798 0.0061453018795649276 -0.021052699465032659 -0.032697005916184768 0.025180655176658368 0.012809980517876712 -0.038483548729338096 -0.016739170136011745 0.0032852575045693379 0.016827665897826256 0.024253629840845274 -0.041639043686978149 0.026525056334274273 -0.041813361389126134 0.015223903255097933 0.024448211111875683 -0.010212745481336882 0.011429054995272395 0.019593852115733754 -0.01985655237974801 0.014403207288203093 -0.022571152264908665 -0.017423910276541832 0.0024930109646289188 0.024889299599257859 -0.031602060729645373 0.014589645121472391 -0.024083880013642853\nleaf_weight=198.02693915243435 94.213445326196961 25.821618442598268 82.275614488637075 7.2329256107768733 7.8481537467450861 7.4554526260471894 20.396610741736367 64.193398123563384 2.7706260692793867 30.084096112375846 33.270115321749472 10.748968399362637 3.70798853971064 4.0757788085611564 5.8557086102664453 4.0226160958409309 3.8770016636699438 29.743415960576385 39.724773150868714 39.974294924642891 102.88778466032818 19.052105646347627 19.659172145184133 5.4537904815515494 17.277645874288282 178.0396544499381 16.937431302387267 6.046551061270292 7.8634938109898931 27.858206935823546\nleaf_count=192611 803 8514 396 1403 3975 1395 92 2512 72 2557 4250 457 13 58 15 30 16 84 357 169 338 326 131 92 2622 8404 99 1098 91 13595\ninternal_value=-0.000275742 -0.0156962 0.00773914 -0.0221016 -0.0266004 0.00593999 0.000782005 0.00325419 0.0101527 -0.00119314 -0.00766765 0.000583259 -0.0122097 0.00990331 -0.00877633 0.00312312 -0.0152552 0.0108753 0.00954548 0.000649149 0.0124695 0.00706972 -0.00545282 0.00581661 -0.000153013 0.00116308 0.003317 0.00120842 -0.0100393 -0.0178555\ninternal_weight=1116.4 381.479 734.916 284.985 216.624 652.641 284.101 96.4936 66.964 263.704 68.3612 18.5971 29.5296 368.54 17.8311 13.7553 7.89962 350.709 320.965 79.6991 241.266 138.379 44.1651 25.113 256.249 238.971 201.024 184.086 37.9476 35.0911\ninternal_count=246575 227402 19173 216291 197043 18777 16358 11111 2584 16266 19248 4432 8527 2419 119 61 46 2300 2216 526 1690 1352 549 223 14871 12249 9601 9502 2648 14998\nis_linear=0\nshrinkage=0.03\n\n\nTree=90\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 11 3 4 4 2 14 3 11 17 13 13 0 14 4 4 4 4 22 2 19 0 2 4 13 22 4 11 2 12\nsplit_gain=109.603 51.5048 35.0833 31.9395 29.4607 18.2853 15.6029 13.1896 12.912 11.8832 11.77 10.2149 10.1951 9.33649 10.7242 10.177 21.8705 16.9328 12.4554 10.5326 9.38091 9.02728 8.81675 7.91679 9.12615 7.27132 7.00973 6.03478 5.78517 6.05667\nthreshold=0.75738199555027408 0.77813183983917178 0.31840721315068282 6.9193702936172494 0.84667968750000011 -25.182852745056149 0.69550561433971103 0.53650715206675215 8.8817841970012543e-16 3.5426598534814651 0.14796868535883259 0.14796868535883259 0.58526177796276302 0.91125284794847328 4.5744628906250009 1.0693359375000002 1.4716796875000002 1.6572265625000002 0.93750000000000011 -19.986686706542965 240.50000000000003 1.6765212893675572 -11.357147216796873 2.1750488281250004 -0.055817889458273311 0.93750000000000011 2.3220214843750004 0.30499968189655369 -58.62553405761718 2.2204460492503136e-16\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 10 27 -5 28 9 -7 11 -1 13 -3 14 15 20 -17 -18 21 -20 -6 -19 -15 24 -21 -13 -25 -2 -4 -30\nright_child=4 12 6 5 7 8 -8 -9 -10 -11 -12 25 -14 22 -16 16 17 18 19 23 -22 -23 -24 26 -26 -27 -28 -29 29 -31\nleaf_value=-0.028078761872503251 -0.027514723272077858 0.03032962183507093 -0.022747557588737626 -0.028871136284337504 -0.025522872613128918 0.016777732785259663 -0.02018682900675297 0.014644806536170885 -0.024465580908173599 -0.022172788892146199 0.016838870166607411 0.026376551149092261 -0.018508588169651848 0.0097869173037771606 0.015718518623622047 -0.029966910509542814 0.022634451068536007 -0.020876128509467955 -0.011345731544647599 0.027041038555727835 0.016685702996974135 0.0070320938938797217 -0.032933334851643445 -0.039862759016069692 -0.031160586965920847 -0.0047260239360385507 0.0057711672200665603 0.013196991121151401 0.00078473335224238036 0.01902550239718331\nleaf_weight=190.32947654939198 23.644050288887232 28.936220390256494 6.7680157980066715 14.769696749935973 4.2666310454369514 34.209976898797322 35.175198207813082 245.48428588639945 7.5469527323439243 12.264713158059747 4.7763889766065395 34.291747814044356 3.7353304616408414 6.5904712963965748 78.960031075956067 19.114069099523476 31.407785551535198 36.54563194286311 24.239395172684453 28.021255828934958 58.632020383127383 13.869129627943037 9.9708231338299793 2.3196890403924035 1.8205344046727976 8.0290884831920248 33.864974525436992 3.1734467772766939 92.163502903058543 18.950060990464408\nleaf_count=174762 8728 135 1016 14279 543 2037 11640 1579 542 4279 38 168 30 683 883 3685 584 4072 1667 371 2074 206 2123 64 80 35 608 30 8492 275\ninternal_value=0.000322442 -0.0113666 -0.014304 -0.0211718 0.00788497 -0.00128031 -0.0029489 0.00916738 0.00892574 0.00581289 -0.0268989 0.00674282 0.0243621 0.00502347 0.0061317 0.00310272 -0.000359385 0.00308329 -0.00141057 0.00525274 0.0134679 -0.0132111 -0.0163438 0.0114247 0.0225952 0.0205029 0.00185237 -0.0223959 0.00232134 0.00403759\ninternal_weight=1113.87 437.361 404.689 251.632 676.51 56.5266 153.057 649.692 41.7569 404.208 195.106 391.943 32.6716 349.622 333.061 254.101 191.202 172.088 140.681 90.2658 62.8987 50.4148 16.5613 66.0265 29.8418 42.3208 36.1847 26.8175 117.882 111.114\ninternal_count=245708 213246 213081 191658 32462 16858 21423 23704 2579 22125 174800 17846 165 17643 14837 13954 11337 7652 7068 2790 2617 4278 2806 1123 451 203 672 8758 9783 8767\nis_linear=0\nshrinkage=0.03\n\n\nTree=91\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 2 13 4 1 6 18 0 10 18 4 4 13 21 21 17 18 3 17 2 3 18 4 4 19 17 3 4 19\nsplit_gain=126.683 35.2612 31.3125 29.1144 28.6923 20.3428 16.8908 13.0117 12.6422 9.99878 9.46206 9.06185 10.7524 13.3103 9.36382 10.1751 8.82887 7.63849 9.50235 7.05365 6.93205 6.73123 9.47635 5.93029 7.78992 5.92602 5.81999 5.79688 5.67966 5.57494\nthreshold=0.24961073363565869 0.89587973461402759 -34.252307891845696 0.14796868535883259 0.84667968750000011 2.9477339485081675 3.7958984375000004 4.2696729980286863 1.1495945197163422 3.6242520361853052 4.7317931335534196 3.9514160156250004 2.2978515625000004 -0.091653201177799501 2.5000000000000004 5.5000000000000009 3.9653415636860969 3.4811217321331038 0.016076142967085191 2.1810207941665634 -15.262049674987791 0.47260169401963864 4.2121001622835372 1.6286621093750002 1.0693359375000002 264.50000000000006 0.2328254379290535 0.58795853567030554 2.6030273437500004 253.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 17 8 4 -4 6 7 16 21 -3 -9 12 13 23 -14 -16 -6 18 -1 20 -12 -2 -23 24 29 28 -22 -25 -17 -7\nright_child=2 9 3 -5 5 11 -8 10 -10 -11 19 -13 14 -15 15 25 -18 -19 -20 -21 26 22 -24 27 -26 -27 -28 -29 -30 -31\nleaf_value=0.015238846256143215 -0.027906968665005277 0.01538034463209754 -0.023870755719293413 0.022505401834396872 0.0029089133964837061 0.0062454235476960586 0.022187527505596234 -0.032020687155268387 0.0081701354404958952 -0.037100925732327178 -0.02244754452400527 0.023340174341386598 -0.026044294888546017 -0.00078704744350833631 0.010756458496007294 -0.045556410808003857 -0.028085822969271954 -0.030246999340571901 -0.022073889274438582 -0.026058881474864577 -0.029569333499814682 -0.027270363314080663 0.010297581875837535 0.015877735972045798 -0.011904016630654228 0.017892256274920533 0.0088770798634217364 0.034396523361850367 -0.0024056148177221399 0.026318849235157524\nleaf_weight=7.2914186487323631 37.430295973237662 24.926276764570506 30.576338892322383 85.855631778365932 292.38904263281438 27.738371544051915 30.319851834792644 15.520343628930275 17.726242546108551 2.775699246703879 9.4205667108471953 47.439951221225783 10.396966347005216 46.830751594505273 43.189381832256913 5.8095084447413665 7.5157887747482155 138.8792046802846 24.669102797503001 11.609945443953618 2.9590006188955149 10.036797566746829 12.513154784159267 39.131295517552644 10.463690849486737 2.5060237422585478 34.433254128351109 28.103148975875229 4.7961005908437073 22.901526945643127\nleaf_count=1225 21397 2041 12491 398 16508 179 183 7662 274 146 4292 227 173 401 209 44 2604 148682 13390 5024 1119 342 655 207 173 10 5687 69 57 73\ninternal_value=-0.000512679 -0.0221133 0.0043384 0.00608758 0.00411245 0.00537887 0.00102464 -0.000746331 -0.0137384 0.00901238 -0.0119295 0.0114373 0.00901623 0.0129006 -0.00122506 0.00373278 0.00204141 -0.0271834 -0.0133574 -0.00614307 -0.000762083 -0.0202409 -0.00687155 0.0178627 0.0108219 -0.0185258 0.00514372 0.024009 -0.0279313 0.0156574\ninternal_weight=1086.15 198.542 887.613 809.906 724.051 693.475 404.168 373.848 77.7065 27.702 73.9431 289.307 241.867 175.169 66.698 56.301 299.905 170.84 31.9605 58.4228 46.8128 59.9802 22.55 128.338 61.1036 13.1116 37.3923 67.2344 10.6056 50.6399\ninternal_count=245942 165484 80458 57790 57392 44901 43079 42896 22668 2187 23784 1822 1595 1102 493 320 19112 163297 14615 16122 11098 22394 997 701 425 111 6806 276 101 252\nis_linear=0\nshrinkage=0.03\n\n\nTree=92\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 8 18 2 13 1 17 4 4 17 3 13 1 18 1 18 0 17 4 11 6 10 0 1 17 0 3 18 2 0\nsplit_gain=141.832 48.4961 27.6724 19.8911 17.0377 13.7267 10.6739 10.327 10.2343 10.9821 7.72746 11.1073 6.51556 6.09861 6.04932 5.97288 5.96044 7.27244 6.38833 5.72099 5.56246 5.47847 5.33098 9.67456 6.27034 5.3768 5.13695 4.95584 7.14842 4.76194\nthreshold=1.6109634896136267 1.0000000180025095e-35 3.6242520361853052 -42.103004455566399 0.14796868535883259 4.0016611463566001 2.6250469244966204 2.9518512487411503 5.5515136718750009 2.8667313866239854 0.58795853567030554 1.0000000180025095e-35 1.9634439907166827 1.2424533248940002 1.4475910192202701 6.8062759569045816 1.5447487307370646 0.48516060896247715 0.84667968750000011 0.93735200266714214 6.5410156250000009 3.7012257604091228 0.96046990436194013 1.075454090076706 0.46867156328060416 1.8987378677022575 0.58795853567030554 3.1984648276080736 -10.615592002868651 1.6765212893675572\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 26 19 8 15 14 9 10 11 16 13 -12 -1 21 18 -18 -5 -4 -8 -3 23 -9 -24 -26 -2 -14 -29 -13\nright_child=3 6 4 5 -6 -7 20 22 -10 -11 12 29 27 -15 -16 -17 17 -19 -20 -21 -22 -23 24 -25 25 -27 -28 28 -30 -31\nleaf_value=-0.018673826681315186 -0.022711678764263266 0.011371526016578318 -0.028945890863334454 -0.028787650592935345 0.018109126897762569 0.0164605516863374 -0.02500868698250755 0.026530705844268865 0.017114090305982887 -0.021611164898802788 0.015628196774988709 0.020542763579829287 0.017089333703040766 -0.035340491494388573 0.002365685767025485 -0.018943922710863759 -0.0054450499413858933 0.015987871410823907 -0.0012249886278717836 0.020045698399525785 0.014932377205935436 -0.030277183277883177 0.014493115610756481 -0.030992524205170648 -0.035221584665555382 0.015331061787320126 0.0064818753753792882 0.012071570608739887 -0.037589903705298945 -0.0070703441779296668\nleaf_weight=51.717151270306204 21.392701336139002 79.372554693036363 186.85239868058125 6.9536548809846854 6.7041042769560582 187.33141420257743 14.321724003297279 14.917233070991648 72.048336004198063 12.674203671340363 2.955486223101615 38.153011154383421 85.911783333169296 3.8582720452686772 15.770319864590418 6.0490187820760175 21.532936519826762 38.887168849469163 158.82011771181715 1.5138258934020985 3.2304014631372402 2.0078230183571568 6.4163183784112325 2.2810593854228491 6.1568687141116225 1.9262377757113425 6.6602357635856597 6.7967143527348526 2.9415468932129425 6.0553448740392914\nleaf_count=26100 2456 3960 189796 1757 42 873 4315 1296 554 1825 8 166 413 250 824 2900 210 294 6165 3 133 122 283 176 975 44 142 219 151 35\ninternal_value=0.000148732 -0.0139972 -0.0202249 0.00854901 -0.0267641 0.0096308 0.00348502 -0.00722541 0.00680498 0.00483916 0.00579364 0.00311236 0.0127711 -0.0140619 -0.0137946 0.00779553 0.000397339 0.00838517 -0.00254683 -0.0284484 -0.017249 0.00998891 0.00675517 0.0176174 -0.00643433 -0.0228131 -0.0157961 0.0148083 -0.00503282 0.0166878\ninternal_weight=1072.21 399.237 294.256 672.973 195.07 644.92 104.982 99.1852 457.589 385.54 372.866 270.402 102.464 6.81376 67.4875 87.4294 226.194 60.4201 165.774 188.366 17.5521 81.3804 31.6977 17.1983 14.4994 8.08311 28.0529 95.65 9.73826 44.2084\ninternal_count=246487 230969 219539 15518 189841 12920 11430 29698 12047 11493 9668 8627 1041 258 26924 6982 8426 504 7922 189799 4448 4082 2774 1472 1302 1019 2598 783 370 201\nis_linear=0\nshrinkage=0.03\n\n\nTree=93\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 1 4 1 10 0 0 3 17 9 13 5 12 17 13 0 21 4 0 1 21 19 0 5 17 3 1 13 3 4\nsplit_gain=117.237 29.2703 26.137 24.869 22.6743 12.7522 11.4849 12.3994 10.4504 9.20649 7.74343 17.5798 12.3427 7.71247 7.10472 8.4044 7.30673 6.60012 6.4113 6.23019 5.7879 13.2814 8.44658 5.78161 6.08792 5.58904 5.29126 7.49366 5.23341 6.16617\nthreshold=0.18336881478606726 2.6981674883069502 3.7858886718750004 1.4475910192202701 1.8688348091416842 0.92717806344988862 0.72961209499321422 0.016076142967085191 2.4286772172182522 1.0000000180025095e-35 -0.05385377781229627 3.1562500000000004 0.084607650078150978 4.2643260401066021 1.0000000180025095e-35 1.5447487307370646 3.5000000000000004 1.6286621093750002 -0.83739793879099134 4.0016611463566001 5.5000000000000009 251.50000000000003 0.94290057586474996 1.8691406250000002 0.30925796400737565 0.51817819236196982 1.075454090076706 -0.62487823464398462 0.51817819236196982 7.58411729335785\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 3 5 6 -2 -1 -8 14 13 11 -4 18 19 15 17 -16 -5 -13 20 -3 -22 -23 -19 -25 -24 -20 -28 29 -29\nright_child=1 9 10 8 -6 -7 7 -9 -10 -11 -12 12 -14 -15 16 -17 -18 23 26 -21 21 22 25 24 -26 -27 27 28 -30 -31\nleaf_value=-0.029621440226972148 -0.02745466888819327 0.0083039178248687159 -0.030458770280737364 -0.014285964971469589 0.013734492189362292 -0.0033465310273944116 0.017960881442664816 -0.020902917450074866 -0.028913901750698783 0.028276063041376007 0.015409201821131055 -0.025325128700213549 -0.034124530924540313 -0.025881726573045615 0.023779469466598081 0.0099446919619327421 -0.017477192031904693 0.0057617431328628307 0.020712496045807837 0.013591437327328296 -0.026097626803089753 -0.031640911818605386 -0.010308807045878009 0.0080691598514715523 -0.019763929212377081 0.024804713224717972 -0.0194239235645404 -0.020626956479220367 0.027700224661061763 0.012558322513117519\nleaf_weight=128.39102061508311 58.459230656902946 166.20058740186505 11.580209979707432 43.54757044967846 13.776030440159955 30.025852261940599 9.6270860160002467 23.886353364934621 10.010325807626939 23.077741850167513 77.809970219532261 3.3555921545848824 5.0114619635132831 4.6816398586379355 19.358920895494524 45.412602906348184 3.9985329108312717 54.620801542856498 35.680140876596852 180.43580574681982 14.596719580818897 2.9280550524126765 4.7816647756844786 10.03798864176497 19.781224390957505 15.63140326249413 8.123783116519915 6.2549007854686378 10.501910952618344 15.561698772944508\nleaf_count=143451 50122 2010 3632 5785 774 6159 1517 13951 2594 97 2386 2116 593 402 95 417 53 2414 3001 849 318 114 34 552 1510 51 460 495 51 128\ninternal_value=-0.000102056 0.00436407 -0.000762836 -0.00619023 -0.0224084 -0.0194028 -0.0255492 -0.00954195 -0.000469813 0.0101849 0.00843575 0.00268716 0.00753332 0.00904267 0.00111621 -0.00100931 0.0162892 -0.00494298 0.0105514 0.00953384 0.0058986 -0.00458139 0.00939348 -1.37943e-05 -0.0104367 0.0164822 0.0124408 0.00481939 0.0113134 0.00255694\ninternal_weight=1057.15 881.466 469.133 295.253 175.68 88.4851 161.904 33.5134 206.768 412.334 173.88 96.0697 84.4895 389.256 196.758 173.4 23.3575 127.988 79.478 384.574 204.138 37.9378 23.3411 84.44 29.8192 20.4131 76.1224 40.4423 32.3185 21.8166\ninternal_count=246131 86438 82563 69701 159693 56281 158919 15468 13420 3875 12862 10476 6844 3778 10826 10678 148 10261 6251 3376 2527 517 199 4476 2062 85 4135 1134 674 623\nis_linear=0\nshrinkage=0.03\n\n\nTree=94\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 12 3 16 4 6 14 3 13 4 4 0 14 3 13 0 4 2 2 3 0 2 10 2 4 4 21 3 17 21\nsplit_gain=99.5461 49.442 30.4065 25.0781 16.5168 15.9118 19.2343 12.2635 14.5891 10.2458 10.8603 10.0954 10.008 9.32339 7.85512 7.66542 12.4438 7.48049 7.26137 7.25632 7.19162 6.82748 6.80719 6.52644 6.46713 16.4554 6.31262 5.62172 7.98009 6.24719\nthreshold=0.72961209499321422 0.71769025269686726 0.28904431203898151 1.0000000180025095e-35 0.84667968750000011 4.3378906250000009 0.52010779807184393 0.53650715206675215 0.14796868535883259 4.4806598424911508 2.1750488281250004 0.095700972623321071 0.86227501112960836 0.24463700593768659 -0.05277688766252419 1.2661233611507157 2.5541992187500004 13.964207649230959 50.658309936523445 0.38657275885365588 1.0410863859294157 20.814876556396488 1.2424533248940002 -42.103004455566399 1.6286621093750002 1.0693359375000002 6.5000000000000009 0.62297257604742995 0.65522739201539382 3.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 -2 6 23 8 9 10 13 22 18 -6 24 16 17 27 -11 -16 -21 -12 -5 -4 25 26 -15 29 -29 -9\nright_child=4 -3 5 11 7 -7 -8 15 -10 12 21 -13 -14 14 19 -17 -18 -19 -20 20 -22 -23 -24 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.028836278250021358 -0.014835115587566637 0.025132902700454388 -0.022796615909348915 -0.026641480785230015 -0.011296134581069119 0.026772944001761802 -0.021905596180070343 9.9391043086129538e-05 0.020933972397057535 0.0078323902826716654 -0.016326902997042207 0.013194990822190309 -0.018697001720641652 0.015735536763869667 0.0092798988969356126 0.017455419938771581 0.01847341056838285 0.013808024694616046 0.027207859587400386 0.0081035609210236802 -0.038093556891262009 0.0015079857295066531 0.0021873756344265127 0.00423007584094929 0.015541912967996079 -0.024555788475963486 -0.017900234954166816 -0.020987400708236851 0.01682122061747824 -0.032490474360105347\nleaf_weight=156.15182981402177 29.481066406813625 28.134220636158716 8.044256130786378 16.614752696114014 30.56719668443111 11.939402386429718 53.09481790351856 8.6513795572100225 42.117945705074817 51.040035181489657 50.149967916382593 18.282967035542242 8.1372862433490791 37.059970125963446 9.9240279697114619 144.84363208757713 43.536824997514486 16.564048558415379 26.465703106950969 4.0054938624380148 8.5157349451910687 30.788464355515316 12.432828892051473 71.697162711105193 70.435988004086539 14.553063755214678 4.9359685338858972 7.2225103743840036 11.938830541446803 12.615280467085539\nleaf_count=173834 9644 206 1526 6843 10915 66 20566 115 226 910 4781 1104 478 933 343 626 341 257 112 59 121 1944 1590 4739 1369 1844 159 113 66 184\ninternal_value=-0.000207784 -0.0125125 -0.015617 -0.0230687 0.00678027 -0.0050572 -0.00809678 0.00780706 0.00447353 0.0024187 -0.000547273 -0.00383736 0.0113574 0.00355714 0.0066504 0.0130481 0.00658869 -0.00265217 0.0147108 -0.0094071 -0.0238089 -0.00960702 -0.0146401 0.00125507 0.00953997 0.00192394 0.0114578 -0.00956671 0.00223585 -0.0198237\ninternal_weight=1039.94 376.392 348.258 203.482 663.55 144.776 132.836 634.069 388.697 346.579 260.936 47.3305 85.643 179.997 149.43 245.373 100.529 56.992 77.5057 22.4453 12.5212 80.9384 29.0476 79.7414 126.985 56.549 41.9959 40.428 19.1613 21.2667\ninternal_count=246014 210474 210268 183371 35540 26897 26831 25896 24194 23968 22468 9537 1500 15743 4828 1702 1076 735 1022 523 180 6725 8433 6265 4305 2936 1092 478 179 299\nis_linear=0\nshrinkage=0.03\n\n\nTree=95\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 5 0 17 0 13 22 7 12 21 0 13 13 5 4 4 4 5 5 21 1 0 13 21 7 3 22 14 0 5\nsplit_gain=118.874 43.2404 21.9609 19.4215 15.222 14.4377 16.961 14.9108 11.12 12.9447 9.9192 12.013 9.65064 9.53542 13.2478 9.18306 9.04635 18.3115 9.52647 8.82117 7.8198 7.66025 7.56257 7.29124 7.25938 6.96863 6.85132 8.24755 6.68546 6.43954\nthreshold=1.211689956917019 3.3837890625000004 -0.7405159439372101 3.2718994203396332 0.84505259813102007 0.14796868535883259 0.93750000000000011 0.89599609375000011 0.28848765710462548 2.5000000000000004 0.44282757256233729 -0.18763083470077987 -0.030094928075697355 0.89599609375000011 0.84667968750000011 2.7826617956161503 5.5515136718750009 2.1386718750000004 0.89599609375000011 7.5000000000000009 -0.32709731491006455 0.96046990436194013 0.14796868535883259 5.5000000000000009 2.6738281250000004 0.44825707527455844 0.93750000000000011 0.48434979059990024 0.45494749303197096 2.0131835937500004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 -3 5 22 7 -7 13 10 25 11 -9 -6 14 -2 -15 17 18 -12 -16 -4 -21 -1 -8 -13 -10 27 29 -11 -20\nright_child=3 2 20 -5 12 6 23 8 9 28 16 24 -14 15 19 -17 -18 -19 26 21 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.029838839704255004 -0.014969460037538301 -0.025523308987230559 -0.013630459918778808 -0.019895881036702211 -0.020697429904656194 0.027352399304682268 -0.023426903069269867 -0.023278689877988281 -0.034030855443204437 0.0057875263271830605 -0.034966342396303071 -0.011649745995318215 0.0065441909222688551 -0.010272575916001456 0.011401860296064758 0.016130447230201253 0.023610817954675789 -0.004607931656833074 0.015566339677161069 0.020036228533338644 0.0108307454005842 -0.030464455018655537 0.012986301432704964 0.019860413013016141 0.025228200799852302 0.015594542138566116 0.018100808543171136 -0.018336038925556051 -0.043275871702872074 -0.03151504071352737\nleaf_weight=145.53827639916199 23.942925363473478 23.3272142194146 13.206502542765518 23.196190917609783 25.641275341236906 57.064551054267213 9.3357840646058303 15.090043810196219 2.0426158818299944 3.0826248926168764 2.5322415707341852 6.9385144753905506 20.202398770117725 87.658487143809907 116.14426701593038 12.772955404128878 57.178397049196064 56.804014125344111 29.638508241740059 3.0765137115668022 70.048530389707594 7.8755972742801532 3.2824947994668028 4.3933913670480242 11.797955342917701 15.701408823719246 162.35581881311373 9.9409633161849325 8.6409329254529421 1.9315784513019014\nleaf_count=169625 4325 24016 3852 4261 13524 322 51 2350 98 203 121 755 3701 7013 2086 497 335 1084 320 134 5308 120 43 16 96 96 1014 559 148 48\ninternal_value=2.50796e-05 -0.0157971 -0.00041869 0.00657478 -0.0241719 0.00747148 0.0202411 0.00601859 0.00973215 -0.00776837 0.0112058 -0.00413959 -0.00880916 0.000336598 0.00508795 -0.00678869 0.0128339 0.0104122 0.0145482 0.00895381 0.00683481 -0.015789 -0.0288139 -0.00916687 0.0116732 0.00837109 0.0153309 0.00432656 -0.0309341 0.0118066\ninternal_weight=1030.38 301.247 106.582 729.136 194.664 705.94 70.7937 635.146 383.676 29.4676 354.208 33.8265 45.8437 251.471 151.039 100.431 320.382 263.203 206.399 127.096 83.255 10.9521 148.821 13.7292 18.7365 17.744 203.867 41.5111 11.7236 31.5701\ninternal_count=246121 220069 33176 26052 186893 21791 389 21402 7227 545 6682 3201 17225 14175 6665 7510 3481 3146 2062 2340 9160 254 169668 67 851 194 1941 927 351 368\nis_linear=0\nshrinkage=0.03\n\n\nTree=96\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 12 4 3 4 0 2 14 16 0 12 6 3 13 17 4 2 10 4 4 4 11 5 9 3 22 13 13 9 2\nsplit_gain=82.0015 39.4123 24.227 27.2532 15.7806 12.6801 12.3894 11.9081 9.15639 15.1319 7.15581 6.93311 7.15726 7.65596 6.89277 6.55547 6.27324 5.36919 5.32967 15.3373 9.0357 5.26204 4.94623 5.63009 5.0291 5.10959 7.05769 4.81657 4.80044 5.33584\nthreshold=0.69651904584641833 0.76965903234489019 8.7147907018661517 0.31033658837561479 0.84667968750000011 1.3771184689138856 -3.0241031646728511 0.52722272431259987 1.0000000180025095e-35 0.13844441961022361 0.49742347178520357 0.89599609375000011 0.51817819236196982 1.0000000180025095e-35 3.9281938047451082 5.2885742187500009 -18.836229324340817 2.8617925509761908 1.0693359375000002 1.6286621093750002 1.6572265625000002 0.081550465899991945 1.4980468750000002 1.0000000180025095e-35 0.22136347059745312 1.0000000180025095e-35 -0.087621146745077524 0.14796868535883259 1.0000000180025095e-35 26.206565856933597\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 8 -2 6 22 11 10 -10 -1 12 13 -5 15 18 -4 21 -8 -20 -21 -18 24 -24 -6 -26 -27 -9 29 -22\nright_child=4 -3 16 7 5 -7 14 27 9 -11 -12 -13 -14 -15 -16 -17 17 -19 19 20 28 -23 23 -25 25 26 -28 -29 -30 -31\nleaf_value=-0.028871110734487962 -0.015070669247250489 0.027367180238450119 -0.0218330240224969 -0.015122344016761028 -0.026882381917551934 0.011331036904204839 0.015424911652690398 -0.021310106641454527 -0.028665591922879791 0.017380675404315597 0.011265225961161533 0.025677850732607518 0.011930889633816135 0.01866924367304101 -0.021202180291967437 0.017903743249649166 0.017511051601220681 -0.029416596914275474 -0.027348444463854107 0.029925092131042608 -0.0055769310412911305 -0.019630506888359264 -0.015808990816138901 0.0096015984744297865 0.015082003353886878 0.0011434864035384561 -0.037868507034198959 0.012151323902034621 0.029314155673163249 0.010147470741664775\nleaf_weight=146.76601482097976 29.7827984305477 20.549526899238117 5.1393267757230197 33.562788460185402 4.5351058538944917 304.42215226183907 33.553932630544296 39.49603463398671 12.825845004059376 11.085045978834385 3.615553397248731 8.3708951274165866 20.733994617097778 6.4211393255973226 6.3774987891374613 48.484016379487002 32.01869082232588 1.8505058473674569 13.119129939135744 11.513258126331491 48.147070098129916 3.0011912148911497 44.280908140113752 8.7884542899846547 20.080638320388971 41.172567576439178 3.6381120815931345 3.6403325521387151 4.6340665648458517 31.116995071904967\nleaf_count=173976 10798 131 3131 4215 3925 3699 1571 16231 6384 863 245 245 559 50 2713 670 579 107 3282 146 2962 187 5138 152 719 1443 120 29 49 1779\ninternal_value=-0.000333969 -0.0120537 -0.0145951 -0.0177076 0.00593376 0.00695901 0.00277019 -0.00610095 -0.0251035 -0.00747784 -0.0278381 0.00165229 -0.00202239 -0.00937254 0.00741526 0.00847863 0.00655095 0.0109794 0.00515982 0.00187435 0.00613449 0.0138714 -0.0046976 -0.0115012 0.000592488 0.00289148 -0.00278133 -0.0182749 0.0025557 0.000645575\ninternal_weight=1002.72 349.077 328.527 286.518 653.647 623.864 319.442 112.225 174.292 23.9109 150.382 69.0888 60.7179 39.9839 196.946 190.568 42.0097 36.8704 142.084 108.531 95.4114 35.0199 122.496 53.0694 69.4264 64.8913 44.8107 43.1364 83.8981 79.2641\ninternal_count=246098 206932 206801 202797 39166 28368 24669 21329 181468 7247 174221 5069 4824 4265 13172 10459 4004 873 9789 8218 4936 766 11497 5290 6207 2282 1563 16260 4790 4741\nis_linear=0\nshrinkage=0.03\n\n\nTree=97\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 1 4 10 18 18 18 12 18 18 7 13 9 18 3 4 4 3 0 4 21 3 7 3 5 0 18 13 0 13\nsplit_gain=107.422 33.7933 37.5589 23.6098 10.0116 9.20287 8.62919 8.77977 8.50423 8.85867 7.73651 9.39159 7.59492 7.43162 10.0385 7.32883 7.96018 7.13704 6.66875 8.49764 6.66318 6.64936 6.4146 6.66828 7.51018 7.33771 7.08348 6.19256 6.46503 6.10554\nthreshold=0.18336881478606726 1.9634439907166827 3.8073730468750004 0.89587973461402759 3.3842466058243152 2.9174053685313028 3.6762205501217919 0.37771650667025564 2.0126758453675748 5.1619206365690227 3.4804687500000004 -0.026449291562135987 1.0000000180025095e-35 2.4414009612931857 0.47260169401963864 11.379394531250002 4.2863769531250009 0.016076142967085191 0.48517496193768689 8.4741210937500018 3.5000000000000004 0.29242024582541543 2.2500000000000004 0.53650715206675215 2.1386718750000004 1.2661233611507157 1.8688348091416842 -0.032724337246802453 1.0212340148387515 -0.28983840832261526\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 2 13 5 12 17 -6 18 15 -10 11 -11 22 21 -15 16 -4 -1 19 -8 -16 -2 23 -3 25 -25 -26 28 -5 -23\nright_child=1 4 8 27 6 -7 7 -9 9 10 -12 -13 -14 14 20 -17 -18 -19 -20 -21 -22 29 -24 24 26 -27 -28 -29 -30 -31\nleaf_value=0.01852762405302338 0.0065061127062145631 0.0051436442656318656 0.011496244511200822 -0.029788764324091135 -0.020954812169432145 -0.029828538298666239 -0.016862933525138207 0.022473735936360005 0.015242195898683975 -0.032198947743704097 0.020066522985286883 0.0081885242370744616 0.02567769723810856 -0.023391929601465137 -0.017653204176400823 0.011739345574943881 -0.042321317506770498 -0.018862618354876928 0.0083104310080879991 0.016578160430680871 0.013694147162428907 -0.005303627931462265 0.0224030168501149 0.0057909811553988541 -0.024374198261730297 0.019864387976254817 0.01157630940110186 0.019725129951682196 0.025160578367079921 -0.034228168744686004\nleaf_weight=5.899275152944031 25.813733314906131 200.8409597427235 3.0494048637338 5.1262511449749608 13.123129708197665 117.1234810633905 28.882920088624815 17.895798651792575 92.848049745851313 10.407907586974037 7.4725871307746266 8.8923990284965839 29.78975741378963 76.988286026346032 10.236681060108824 6.8077647374011567 7.0708200108492747 13.046623740963694 37.979409851919627 7.9972758196527129 12.358919544160015 23.753640781913418 34.249093261314556 50.397445995389717 8.0238811112940329 104.16054660431109 9.9658641992136818 12.891713179298675 1.9273514966480432 8.3442667500348744\nleaf_count=653 2675 2596 48 1810 590 151140 3358 204 3952 6308 470 1130 114 56916 2320 58 162 6072 1603 51 1112 1028 133 446 34 391 64 254 27 406\ninternal_value=0.000511854 0.00477649 -0.00340829 -0.0222616 0.00920433 -0.026706 0.000915918 0.00420071 0.0080756 0.0108949 -0.00418765 -0.0141041 0.0112039 -0.013353 -0.0182904 -0.011343 -0.0265296 -0.00687504 -0.000406273 -0.00936519 -0.000653699 -0.00469632 0.0100859 0.00889926 0.0132159 0.0153421 -0.00489768 0.00788607 -0.0135144 -0.0136299\ninternal_weight=993.365 837.351 294.044 156.015 543.306 136.069 105.879 92.7554 136.549 119.621 26.7729 19.3003 437.428 157.496 99.5839 16.928 10.1202 18.9459 74.8596 36.8802 22.5956 57.9116 407.638 373.389 172.548 154.558 17.9897 19.9453 7.0536 32.0979\ninternal_count=246125 86169 76585 159956 9584 157865 5806 5216 12128 11860 7908 7438 3778 64457 60348 268 210 6725 5012 3409 3432 4109 3664 3531 935 837 98 2091 1837 1434\nis_linear=0\nshrinkage=0.03\n\n\nTree=98\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 2 1 0 12 17 6 7 13 22 3 0 3 17 2 0 5 11 1 3 2 2 4 4 4 1 17 4 13\nsplit_gain=104.7 36.9805 33.649 16.9716 12.6938 9.83976 8.80947 8.76607 7.48514 8.94319 12.4401 9.74762 8.51035 7.45345 7.29848 6.88167 6.47621 6.33625 6.72695 6.23822 8.14956 6.06062 6.02913 5.94052 15.8304 6.57627 5.77992 5.71222 6.84186 5.42459\nthreshold=1.4475910192202701 3.2580566406250004 -15.262049674987791 4.0016611463566001 0.84505259813102007 0.86223072954082747 6.2913755499196116 3.3837890625000004 3.5400390625000004 -1.0000000180025095e-35 0.93750000000000011 0.47260169401963864 1.4460303622695712 0.32220753877234559 2.1677456597340812 -1.7340650558471677 0.62323935301024946 4.4238281250000009 5.5511151231257837e-16 0.18336881478606726 0.18960663291921473 37.549415588378913 -1.3165187835693357 1.6286621093750002 1.0693359375000002 2.3505859375000004 0.7474413184595442 0.45436599119170595 1.9753417968750002 -0.45698050410493757\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 5 6 -1 -3 7 8 9 11 -11 12 -2 -10 19 -12 -16 18 21 20 -4 23 -17 24 -13 -25 -18 28 -14 -6\nright_child=3 2 14 -5 29 -7 -8 -9 13 10 15 17 27 -15 16 22 26 -19 -20 -21 -22 -23 -24 25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.030041530063535198 -0.0084518805039503868 -0.028665171183977545 0.0058866766950534718 0.014664227001816851 0.0036689732292854646 0.022884954071240686 -0.030811967934386064 0.024693806775182185 0.01366849660894346 0.01610859342807142 -0.036196028462515663 0.0079507637785316831 -0.037612520877402369 -0.042836222235107323 -0.02182441124281459 0.02915480123235702 0.01919859009023326 0.028294957322868953 -0.032826481338684828 0.015062121040163591 -0.034052864219176061 -0.015139125788862963 -0.011556572424770333 0.016957876471767049 -0.027345251572408545 0.0027720851915162188 -0.017742518066069982 0.014979013916551268 0.004883504468524512 -0.01802407615849759\nleaf_weight=111.40330853611522 110.14230254170252 43.807537434688129 20.761282100123939 174.1027092745062 15.056602569857203 2.8723447471857062 5.5533964435744556 17.174166244454682 2.61927312798798 88.92271104772226 5.3475824725464944 44.375505120508024 3.7716920275706789 5.0569249973632395 13.876878756113003 3.8572237084154049 9.4639234143687627 9.8586338567547482 3.2326628425507797 56.35505441998248 4.836670486314687 11.113690569764001 9.0610691979527438 65.681040098512312 14.291118666384135 53.444718728162115 5.0582648750569197 27.987430008593947 11.306528225773944 30.384712018858409\nleaf_count=140873 10212 52948 4131 971 4569 10 1388 77 26 1591 240 1200 98 46 5735 7 721 43 109 3634 1629 159 104 694 948 845 308 250 130 12734\ninternal_value=-8.54943e-05 -0.0143465 -0.00401082 0.00663787 -0.0246035 -0.0250797 0.00377825 0.0042348 0.00343649 0.00394656 0.0114995 0.00164147 -0.00404995 -0.0242025 0.00504669 -0.0106103 -0.00786676 0.0059569 0.00467017 0.00956246 -0.00265636 0.00548131 0.00182062 0.00684775 -0.000961502 0.0106696 0.00601743 0.00714449 -0.00742473 -0.010912\ninternal_weight=980.777 313.877 157.032 666.9 156.845 46.6799 492.798 487.244 470.07 462.394 107.189 355.205 153.208 7.6762 110.352 18.2659 28.3991 201.997 192.139 81.953 25.598 188.906 12.9183 177.792 58.6666 119.126 14.5222 43.0657 15.0782 45.4413\ninternal_count=246430 227292 69116 19138 158176 52958 18167 16779 16702 16630 1942 14688 10690 72 16158 351 6764 3998 3955 9394 5760 3846 111 3687 2148 1539 1029 478 228 17303\nis_linear=0\nshrinkage=0.03\n\n\nTree=99\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 4 0 1 18 1 13 13 6 17 18 17 13 3 7 10 4 1 18 18 0 2 5 14 3 3 1 5 1\nsplit_gain=93.7674 34.1202 27.7179 13.9689 13.0534 12.463 11.4695 10.7444 9.15192 14.4364 14.865 9.02581 8.75945 8.61997 8.18816 7.0705 6.97878 6.87955 6.66351 6.30194 6.13998 8.27775 5.96567 10.9454 5.5372 5.46078 5.38214 11.1183 5.24656 5.91748\nthreshold=1.7622051334857016 -15.657813072204588 3.2580566406250004 1.5447487307370646 1.4475910192202701 2.9174053685313028 2.6981674883069502 0.14796868535883259 -0.05277688766252419 3.4492187500000004 1.9205778057410925 1.8688348091416842 2.7033165568765605 -0.45698050410493757 0.048385965359893195 0.89599609375000011 2.740319461670996 5.0736774206161508 0.20426862894419642 2.249904835165133 2.740319461670996 0.32384722065649935 26.206565856933597 3.3496093750000004 0.23441483525608195 0.215496103286093 0.27708310882964732 1.5661550609387962 3.7500000000000004 3.3033179733375269\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 7 4 6 5 13 15 11 9 10 18 -1 17 -3 -15 19 -10 -8 -4 -2 22 25 -17 -24 -23 -22 -6 -28 29 -5\nright_child=3 2 8 28 26 -7 12 -9 16 -11 -12 -13 -14 14 -16 20 -18 -19 -20 -21 21 24 23 -25 -26 -27 27 -29 -30 -31\nleaf_value=0.00070656640938689932 -0.0019782923312958215 0.01380418285880476 -0.028331541889001079 0.0054184950255214545 0.013479773655435599 -0.030214687419025551 0.0069657249602562143 0.016121860541232793 0.014310038101348104 0.017071323049986275 -0.036183943738154228 -0.02918226129769113 -0.019198680182705975 0.0090304321121611915 -0.030065004448049399 -0.017766092546051802 -0.027191585432376351 0.023794660005692823 0.004783646093604055 -0.02511275319482649 0.0082620749122632384 -0.018764756859535065 0.019544883862874396 -0.039064442223843795 0.014866489876732154 -0.03434892359625448 0.0090587741734035228 -0.03552781051328445 0.02855060275167914 0.016528346576888096\nleaf_weight=9.8534162445721432 21.798044321709312 11.188947646820454 5.6894043250504174 76.914399915491231 14.219169174000852 57.423746236258012 206.48469466529787 4.8717999557266003 59.489508478953212 20.262574753141962 15.126844665268434 110.02547039013007 9.5948049855651316 6.5267876672878655 14.076000236658727 22.181184081127867 2.9729432458989313 23.777379411854781 29.014965634079999 19.972744593731477 4.0654483339458221 4.0016702245338811 11.5695617479505 2.7349466215819112 46.942411752068438 5.5510138479003217 11.055309793679045 7.7907161444745716 18.350629866123199 101.51790402946062\nleaf_count=463 438 2075 4411 678 1311 70137 2656 46 4082 249 5268 133663 562 1242 5855 282 58 137 2962 4845 249 138 74 25 1171 973 789 1094 52 335\ninternal_value=-0.000600809 -0.012157 -0.00582074 0.00702984 -0.0160629 -0.0219728 0.00364339 -0.0249513 0.00367698 -0.00374847 -0.0123836 -0.02674 0.00760387 -0.00661848 -0.0178352 -0.00320523 0.0119374 0.00879889 -0.00128668 -0.0133686 0.00126103 0.00707777 -0.0083251 0.0065188 0.0119144 -0.0172533 5.99552e-05 -0.0102342 0.0134951 0.0117963\ninternal_weight=955.044 379.588 254.837 575.457 122.281 89.2155 378.674 124.751 132.556 70.0938 49.8312 119.879 239.857 31.7917 20.6028 138.817 62.4625 230.262 34.7044 41.7708 97.0462 60.5605 36.4857 14.3045 50.9441 9.61646 33.0652 18.846 196.783 178.432\ninternal_count=246320 233705 99533 12615 82503 79309 11550 134172 17030 12890 12641 134126 3355 9172 7097 8195 4140 2793 7373 5283 2912 2531 381 99 1309 1222 3194 1883 1065 1013\nis_linear=0\nshrinkage=0.03\n\n\nTree=100\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 8 0 2 4 7 6 7 7 4 0 0 4 4 4 4 12 2 13 9 0 17 3 3 2 3 2 19 13 7\nsplit_gain=68.907 35.3797 32.7403 13.2873 15.3552 12.9858 9.58873 10.0208 8.88999 10.0948 7.95709 7.93678 9.08863 15.1733 10.2166 8.1322 7.49856 7.46234 7.86089 8.24326 7.34946 7.30627 6.49611 6.42716 11.6558 6.50828 6.25711 6.23245 6.09842 7.0608\nthreshold=0.28904431203898151 1.0000000180025095e-35 0.70465417701170696 -39.733196258544915 0.84667968750000011 2.3505859375000004 4.3378906250000009 3.8603515625000004 1.0000000180025095e-35 1.1459960937500002 0.96046990436194013 1.3771184689138856 1.0693359375000002 1.6286621093750002 1.6572265625000002 2.8471679687500004 0.49742347178520357 18.240050315856937 0.14796868535883259 1.0000000180025095e-35 0.29549219484135919 1.9512128003148403 0.18038095578113905 0.47260169401963864 5.317040443420411 0.41780940659066851 29.286581993103031 207.00000000000003 -0.015587343939122865 0.89599609375000011\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 16 10 -5 8 7 11 -4 -10 -2 12 -6 26 -15 21 -1 18 19 -3 28 23 -11 24 25 -16 -14 -19 29 -17\nright_child=3 17 5 4 6 -7 -8 -9 9 22 -12 -13 13 14 15 20 -18 27 -20 -21 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.027941766814596671 -0.02096472340108364 -0.021682876303660406 -0.028741451452690022 -0.016353398806140111 0.015108554889315553 0.025961756373805305 0.028012376648840584 -0.025481794639913669 0.013864085396046175 -0.028076150783543388 0.0058323382113709809 0.012287886779204515 -0.025740025234927539 0.022074881382856203 0.005991695246338768 0.022154784310404702 0.013571839351106552 -0.0038422620063658553 0.025131119862139426 0.0038046156519848393 0.01156055858147439 -0.03384021594138769 -0.0030800191505587888 0.0044451547181414506 -0.041880691931021903 -0.028230104063143354 0.0076038739226066444 0.023382308737427856 0.011341861057778787 -0.033464938373677186\nleaf_weight=126.83434141198086 23.72619477778062 18.375619752696366 13.497778116819971 22.603978650113277 64.915814225343638 11.033935450119314 20.689332500449382 7.1762686519650734 21.193630023975857 12.747458556150375 16.119241290318314 236.22941351143527 20.971375921872099 27.125454116117908 21.550603960458833 1.8533550991851395 3.4793295368144745 10.273343529202974 7.211447136476635 27.222475527487404 97.157202373091422 5.5670320475473991 30.228855727633345 64.392190660815686 7.5732197684701523 5.4307267292751922 6.0791451181285074 25.273652616422623 7.2250582792330524 8.7443616091622953\nleaf_count=171450 5943 7403 10702 6209 2301 344 103 198 3579 8511 218 1404 5286 389 2650 343 225 516 36 2104 2941 1535 3845 1181 1180 494 771 406 323 3436\ninternal_value=0.000942677 -0.0107831 -0.0172254 0.00636949 0.00744301 -0.00310732 0.00836113 0.00761569 -0.00751457 -0.00267711 -0.0102467 0.00807243 0.0051038 0.00268426 0.00504403 0.00284876 -0.0267385 0.00519988 -0.00197698 -0.00671151 0.00832179 -0.00319416 -0.0109641 -0.00114859 -0.0114043 -0.0016591 -0.0182426 0.0156244 -0.00900991 -0.0226664\ninternal_weight=972.502 307.372 219.015 665.13 625.285 88.7017 602.681 581.991 77.6677 64.1699 39.8454 574.815 338.586 273.67 246.619 219.494 130.314 88.3565 52.8095 45.5981 114.98 104.514 42.9763 98.9467 34.5546 26.9813 27.0505 35.547 17.8228 10.5977\ninternal_count=246026 209121 198656 36905 30744 26981 24535 24432 26637 15935 6161 24234 22830 20529 14472 14083 171675 10465 9543 9507 7043 7040 12356 5505 4324 3144 6057 922 4102 3779\nis_linear=0\nshrinkage=0.03\n\n\nTree=101\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 2 4 11 1 13 18 21 4 3 10 0 5 10 3 7 7 0 21 1 1 4 4 4 11 21 11 13 14\nsplit_gain=83.8728 36.0675 16.2372 14.188 13.4133 12.3362 13.9828 8.95513 8.19804 7.16268 7.01339 7.00167 6.9331 9.30602 6.73318 6.64344 6.24356 5.97305 7.06568 6.36565 6.35316 5.8701 7.79305 9.9042 5.53607 5.47807 5.2891 5.26267 5.13527 7.7304\nthreshold=0.95935769349451683 1.0000000180025095e-35 -51.587459564208977 0.84667968750000011 0.66750229771629133 2.9477339485081675 -0.060596817301233889 3.5972184255501678 2.5000000000000004 3.7858886718750004 0.2352687277455576 1.7005986908310777 0.010815665348252143 5.3300781250000009 2.524928003624769 0.47260169401963864 1.4980468750000002 2.5244140625000004 0.96046990436194013 2.5000000000000004 2.1468275719512682 1.7622051334857016 1.6286621093750002 1.0693359375000002 3.5249023437500004 0.026362180702627195 6.5000000000000009 0.30499968189655369 -0.45698050410493757 0.050734622491290134\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 -2 27 7 6 15 28 -8 10 -7 12 25 -14 -13 17 -12 18 20 -19 -5 -17 23 -23 -20 -3 -21 -4 -1 -30\nright_child=2 11 3 5 -6 9 8 -9 -10 -11 16 14 13 -15 -16 21 -18 19 24 26 -22 22 -24 -25 -26 -27 -28 -29 29 -31\nleaf_value=0.0016775557280142707 -0.02360659840020406 -0.031160910050349395 -0.021145179688047466 -0.028342516081891596 0.016135397931204672 -0.031053269301447205 0.017099324882525281 -0.029071673842699885 0.0011571005706886989 0.021645882095100145 0.0105032219450675 0.024328567755903883 0.011586824379155466 -0.031626457402243298 -0.012270025977589501 -0.0077661692624921442 -0.021622078328270337 0.018578840360036035 -9.6098011319921465e-06 -0.032607605307039954 -0.0011588644535202551 0.0049257309748916515 0.013114091759916312 -0.036088373265047428 -0.032097081558309468 0.0059033908806648619 0.0099072685457421462 0.014309818358970586 0.0047779559339732316 -0.030339178955749547\nleaf_weight=15.000740352181309 15.911929584333846 10.412124066548129 21.503873419722371 24.192654655242222 6.9074777521891511 2.8803325355402185 78.468098663841374 114.72866896191772 45.981651255075121 50.403056107461452 257.15037398680579 18.081245945097184 25.657268732029483 4.4071452091448 5.246795024839229 28.684964670304907 4.6598944299621499 14.807882601890016 52.405200570705347 4.3683645144046732 11.29491771783796 18.694986312533732 64.170867395558162 6.2934282007045104 4.3315794106165404 4.8264753328403449 4.4699510511854896 3.7933597035589619 7.6495031984377446 18.153993167179578\nleaf_count=6162 4406 6630 6348 10554 100 178 2401 175956 1916 318 1768 689 1507 130 521 1772 56 318 1926 276 1241 230 812 559 240 798 196 33 1882 16228\ninternal_value=0.000394283 -0.0152779 0.00547721 0.00617249 -0.0229428 0.00700808 0.00318533 -0.0247703 0.0112652 0.0113296 0.00929101 0.00293287 -0.00395983 0.00443801 0.015931 -0.00114673 0.00985133 -0.00622122 -0.00965319 0.00701264 -0.0199594 0.00385313 0.00763605 -0.00639499 -0.00297283 -0.019781 -0.0122843 -0.0154867 -0.0122702 -0.0201388\ninternal_weight=945.539 231.071 714.467 698.555 162.44 673.258 358.165 155.533 124.45 315.094 264.691 68.6311 45.303 30.0644 23.328 233.715 261.81 115.871 92.2244 23.6462 35.4876 117.844 89.1593 24.9884 56.7368 15.2386 8.83832 25.2972 40.8042 25.8035\ninternal_count=246151 210603 35548 31142 200328 24761 22441 200228 4317 2320 2002 10275 9065 1637 1210 18124 1824 14751 13961 790 11795 3373 1601 789 2166 7428 472 6381 24272 18110\nis_linear=0\nshrinkage=0.03\n\n\nTree=102\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 0 2 1 11 17 7 7 0 10 10 12 13 17 6 18 14 7 18 13 2 13 13 8 7 10 3 0 13\nsplit_gain=86.4441 30.8022 15.0588 15.0444 13.6584 9.23679 8.80843 9.01854 8.79587 7.7977 7.34867 12.5336 5.85513 10.8543 5.83238 5.69281 10.141 9.23913 5.64955 4.96445 4.7253 4.50239 4.93915 4.4318 9.41978 4.74364 4.09131 4.07021 3.9216 3.81509\nthreshold=1.0240454605967615 1.0000000180025095e-35 0.72961209499321422 -47.402034759521477 2.5144622974930591 0.042545688445982228 3.1006363674515387 2.2500000000000004 0.89599609375000011 1.5447487307370646 2.524928003624769 1.7005986908310777 0.21283935326158318 -0.041888724317900632 2.6250469244966204 6.3203125000000009 2.9174053685313028 0.83758563423804033 3.0205078125000004 2.3502401828962087 -0.54808102951419191 -19.19599628448486 -0.45067951570476578 -0.26322344360915401 1.0000000180025095e-35 1.8691406250000002 1.2424533248940002 0.31033658837561479 0.96046990436194013 -0.48398449089866569\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 28 5 9 7 -6 -4 18 11 15 -7 -14 -15 16 27 -18 20 -20 -5 -22 -23 25 -25 -24 -21 -3 -2 -17\nright_child=3 10 8 4 6 12 -8 -9 -10 -11 -12 -13 13 14 -16 29 17 -19 19 26 21 22 23 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.028563960085895686 -0.026572293595078861 0.01653943447519151 -0.020306504855047942 0.0021770713445779886 0.008816843029044277 0.02745461604473895 -0.017392507745512605 0.023230482535402969 0.0048589202435371714 0.0088443087308427964 -0.029836562347651408 0.025268494827330169 -0.03391530577950895 0.014637177872363164 -0.020382832433233796 0.028195679970105633 -0.032869213975066924 0.0056567050061287708 -0.031345155899595245 0.019594246017011801 -0.025205499105360529 -0.029891867652784909 0.00461144613852673 -0.017641158934127963 0.018031427088156013 -0.030130377367547187 -0.024679262020079962 -0.025231934789334942 0.0018038605902268307 -0.020046642954328502\nleaf_weight=115.35419763789832 15.207687745009027 12.915722188772632 25.381952032108529 38.002324655593839 343.62913709442364 14.321128973184384 9.5938786051992775 43.738749383483082 23.157751799350081 46.590665989846457 5.2305139695381504 17.319286135483708 4.5215911292179944 37.290972588642035 3.9387808741303152 5.9344067488855217 15.980691244287302 7.9716237654647548 1.3871232345700288 15.940044663351729 14.372848955194057 7.1698368566867421 39.530547463742543 38.387451792514192 7.1729887956171288 2.8778461338661137 1.2549167128745455 1.5584082528948773 5.8417547487770198 1.1042684792773787\nleaf_count=178347 4568 255 19195 4143 3505 125 589 223 5328 747 546 701 340 291 61 39 5559 3164 14 333 5154 1334 3937 6729 198 300 42 43 164 127\ninternal_value=0.000122634 -0.0156857 -0.0226927 0.00544487 0.00623946 0.00106425 0.00978383 0.0105 -0.00844463 -0.00188045 0.0012831 0.00430278 0.01135 0.00582954 0.0108882 -0.00405575 -0.00866899 -0.0203345 -0.00492372 0.0102876 -0.00689624 -0.010036 -0.00750601 -0.00536599 -0.0117583 0.00161405 0.0151843 0.010702 -0.0189725 0.0190749\ninternal_weight=922.679 231.909 163.894 690.77 669.721 272.759 396.962 387.368 48.5397 212.687 68.0149 62.7844 60.0725 45.7513 41.2298 45.4651 38.4264 23.9523 166.096 18.5821 147.514 109.512 95.1387 87.9688 45.5604 42.4084 17.195 14.4741 21.0494 7.03868\ninternal_count=246101 213304 202870 32797 28065 23748 4317 3728 24523 22931 10434 9888 817 692 352 9187 9021 8723 22184 389 21795 17652 12498 11164 6927 4237 375 298 4732 166\nis_linear=0\nshrinkage=0.03\n\n\nTree=103\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 17 4 13 22 1 10 4 0 9 12 3 0 7 4 14 3 1 1 0 12 4 0 9 1 14 5 5 14 13\nsplit_gain=73.8214 20.7763 17.0304 17 20.4563 17.7994 15.3958 11.8201 9.36217 9.21292 11.9064 11.4005 11.1326 13.0742 8.29768 10.961 8.04415 7.44742 7.87007 7.03022 6.38896 9.82078 5.97911 5.69428 5.65258 5.01794 4.9847 8.6581 5.22903 4.88837\nthreshold=0.18336881478606726 3.325091732892195 11.379394531250002 0.14796868535883259 0.93750000000000011 3.3033179733375269 0.89587973461402759 0.84667968750000011 0.72961209499321422 1.0000000180025095e-35 0.084607650078150978 0.47260169401963864 1.8987378677022575 2.5244140625000004 1.0693359375000002 0.55141373229880208 0.016076142967085191 1.5275240303550033 1.1267314367267358 0.80892202233486332 0.027767680665956421 2.7826617956161503 -0.049143614581278028 1.0000000180025095e-35 0.64953354656755413 0.72273155396328503 3.8818359375000004 2.3505859375000004 0.79531390260019708 -0.081478796350041446\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=6 3 -3 5 -5 7 8 -2 -1 11 22 12 13 14 15 19 -10 18 25 -9 -16 -22 -11 -7 -15 -13 27 28 -19 -8\nright_child=1 2 -4 4 -6 23 29 9 16 10 -12 17 -14 24 20 -17 -18 26 -20 -21 21 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.029868823667689826 -0.020789217747534946 -0.030517043977381267 0.018356355239502575 0.026333365361144091 -0.016052975448694815 0.0081887603346435228 -0.018306025520019684 -0.024578888503960922 0.016221137501602124 -0.0094413601432828329 -0.016554755993919288 0.01448397659408255 0.014016759889411494 -0.027064608872905287 -0.017665496286615314 -0.032229418800592394 -0.019070206178143225 0.011168472517467051 -0.036511913136579782 0.019103541846117397 -0.031666389708705528 0.023173743294935972 0.026078674203162015 0.029032323462204131 0.012478501651511471 -0.026679121672183548 0.021742235928915641 -0.01835017319901508 -0.0031767797789401856 0.012907879725581415\nleaf_weight=88.374307622641936 24.462246951152338 26.745368469106324 7.6107028245751271 53.726412047864869 11.894511505030094 216.4511543828994 5.5190035457126241 3.0829597853662625 7.2938034488033763 4.4397042344644424 8.7310131027043081 8.2406321886228415 21.46542062307708 2.6890969167870926 110.7284333627249 5.3828916044803909 19.168943499487796 84.718577008548891 6.698243257473222 18.997776883319602 3.3951020133972616 7.9570658679876942 29.400183326390106 11.698297681286929 30.178193927975371 2.7542358591163056 16.972653032047674 12.159111080574801 30.677898822905263 13.823640591246656\nleaf_count=142192 13683 16609 89 365 71 1464 1722 1994 1474 496 324 531 237 1445 37499 2965 14036 1372 897 602 1129 331 380 40 1311 1138 93 328 875 394\ninternal_value=-0.000634128 0.00298998 -0.0194774 0.00407717 0.0185247 0.00261769 -0.0210652 -0.000957478 -0.0252043 0.000277465 0.0133195 -0.00127125 -0.00598059 -0.00839738 -0.0121518 0.00266856 -0.00910655 0.00464773 -0.0132346 0.0122038 -0.0154805 0.00553682 0.0212813 0.00937954 0.0085396 0.00280798 0.00691279 0.00478579 0.00736029 0.0035396\ninternal_weight=895.438 761.258 34.3561 726.902 65.6209 661.281 134.18 433.131 114.837 408.669 42.5709 366.098 203.877 182.412 149.544 27.4636 26.4627 162.221 17.6931 22.0807 122.081 11.3522 33.8399 228.149 32.8673 10.9949 144.528 127.556 115.396 19.3426\ninternal_count=246086 86268 16698 69570 436 69134 159818 67630 157702 53947 1200 52747 47513 47276 44520 5561 15510 5234 2566 2596 38959 1460 876 1504 2756 1669 2668 2575 2247 2116\nis_linear=0\nshrinkage=0.03\n\n\nTree=104\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 4 1 1 1 12 1 13 3 2 13 14 5 4 14 3 4 5 19 13 19 3 3 7 1 17 4 9 4\nsplit_gain=67.0767 35.989 25.3512 14.9654 10.5908 10.3437 8.92282 7.73407 7.47168 8.34243 9.58141 7.27807 9.45112 6.89963 7.96103 6.85951 6.83056 7.86078 8.3623 9.0766 8.97782 6.0148 6.50069 5.9723 5.64504 6.42997 6.78401 6.08047 4.97302 6.13371\nthreshold=1.7622051334857016 -15.657813072204588 3.5791015625000004 2.9477339485081675 1.6575619461558879 1.4475910192202701 0.71769025269686726 -0.32709731491006455 0.14796868535883259 0.37248792739977959 29.286581993103031 -0.45698050410493757 0.69882437336200276 4.8945312500000009 5.5515136718750009 0.043928974430191954 0.53650715206675215 3.6923828125000004 1.4980468750000002 238.50000000000003 -0.57929579963106315 242.50000000000003 0.27708310882964732 0.56033297298897644 0.89599609375000011 4.0016611463566001 0.48975197669932524 1.0693359375000002 1.0000000180025095e-35 2.3505859375000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 5 8 7 11 23 15 9 10 -2 12 -3 14 28 -4 17 18 19 -5 -21 -7 -23 -1 25 26 -22 -27 29 -11\nright_child=3 2 4 16 -6 21 -8 -9 -10 13 -12 -13 -14 -15 -16 -17 -18 -19 -20 20 24 22 -24 -25 -26 27 -28 -29 -30 -31\nleaf_value=-0.028307528900833431 -0.020429176670378968 0.013798993906793063 0.011879445644108111 -0.024473813572634118 -0.035479968618771354 -0.030515478365337592 0.017411604029113289 0.0098395794331106838 0.017555797272364542 0.0034323822773850263 0.0070892055764079946 -0.025024208017441787 -0.023437239571078225 -0.037529309694926467 0.0209733196853134 -0.028197165869941171 0.014898059156179456 0.018962273572668921 -0.0081083266433980804 -0.034989936777737186 -0.025736599992353719 0.020925552570444546 -0.0067363471683819752 0.0034020988184097962 0.022727744759197126 0.022904843080554699 0.0056539247380604724 -0.0086349965216664312 0.029886251947763801 -0.011397820920531867\nleaf_weight=97.277615835075267 35.209630208613817 11.397760814856158 5.5707333278332944 6.3418949553742996 4.5235350880539036 3.9911720709351348 3.6623560623847871 106.99866643876885 19.272179808467627 86.145094090577913 16.002223220450105 60.714089218228764 11.208291484625077 2.8550168536603442 18.505526328925043 8.8521586511196784 156.46602825605078 33.661339023150504 41.893700575979892 2.8370245312107718 7.8472077656188075 12.209679945954123 17.333875090902438 5.4392122830613507 29.277285872609355 23.373488275567073 19.840709046518896 6.5243654246442011 3.9313168790249611 34.260162983642658\nleaf_count=133932 4147 4561 492 165 164 1088 45 9329 131 3357 810 62791 14258 29 125 4615 799 268 657 69 188 746 1424 132 153 165 389 88 30 1018\ninternal_value=-0.000237042 -0.0104825 -0.00406829 0.00634348 0.00525377 -0.0140749 -0.0249481 0.00704872 0.000210506 -0.00156727 -0.0118594 -0.0196797 -0.00487384 0.00210655 0.00315416 -0.0129513 0.010372 0.00618254 0.00294978 0.0078072 0.0103342 1.762e-05 0.00500651 -0.0266118 0.0121959 0.00654481 -0.00382967 0.0159654 0.00035588 -0.000846535\ninternal_weight=893.423 349.179 242.8 544.244 125.945 116.855 106.379 121.422 216.181 196.909 51.2119 83.3201 22.6061 145.697 142.842 14.4229 328.063 171.597 137.936 96.042 89.7001 33.5347 29.5436 102.717 86.8631 57.5858 27.6879 29.8979 124.337 120.405\ninternal_count=246165 233577 99468 12588 14600 84868 134109 14436 9647 9516 4957 81610 18819 4559 4530 5107 2941 2142 1874 1217 1052 3258 2170 134064 983 830 577 253 4405 4375\nis_linear=0\nshrinkage=0.03\n\n\nTree=105\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 18 5 11 22 1 1 18 0 17 5 7 18 19 5 7 6 2 18 17 17 0 7 0 2 7 10 5 7\nsplit_gain=78.8261 20.2327 20.0018 25.5649 21.3884 24.3569 14.9894 8.4431 8.76604 9.22572 7.52467 7.7389 6.51675 6.42809 5.92641 5.35793 5.60923 5.26895 5.16345 5.16215 5.32067 5.15069 4.89073 6.32868 5.72899 4.87068 4.78137 4.71443 4.47863 5.93834\nthreshold=0.18336881478606726 0.89587973461402759 3.1984648276080736 3.4169921875000004 0.042545688445982228 1.0000000180025095e-35 1.9634439907166827 1.4475910192202701 4.0342014792848504 0.49902597412172423 2.2468301507771291 2.6738281250000004 0.89599609375000011 2.0126758453675748 240.50000000000003 0.89599609375000011 1.0000000180025095e-35 3.3837890625000004 -12.855450153350828 4.9593358439305844 1.2428969707451916 0.92594565335601631 0.94290057586474996 1.0000000180025095e-35 0.72109735700429223 -25.182852745056149 3.9980468750000004 3.6242520361853052 0.89599609375000011 0.89599609375000011\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 13 6 4 7 -6 10 -4 -9 18 11 14 15 -1 -2 16 -8 -12 -10 -20 -21 -11 24 -24 -16 -5 -13 -3 -25 -30\nright_child=2 27 3 25 5 -7 12 8 9 21 17 26 -14 -15 22 -17 -18 -19 19 20 -22 -23 23 28 -26 -27 -28 -29 29 -31\nleaf_value=-0.0029858678430538955 -0.029501831486410768 0.010727328742639802 -0.026080243686701345 -0.020167965944883667 0.023788490911068209 -0.037794284133711116 -0.01143093786242841 -0.021784750552263445 -0.024492697103204827 0.020405721212689985 -0.032659878324720303 0.014887539749129711 0.012813317984307068 -0.029864041453555527 0.021684977690857057 -0.0031938810209624999 0.011088263476160621 0.0077631890116051684 -0.016592079990508251 0.019765405343208663 -0.024107510149038548 -0.010046120650937358 -0.029604500995738227 0.016798340499943359 -0.035503820886407016 0.011923265754310353 -0.021655407267259622 -0.030812698622391554 -0.024049968720169141 0.0049679919900749853\nleaf_weight=8.9902202578959969 7.2521636166929957 17.590806401916783 43.080506972630246 3.612697610835311 27.034283834669623 6.3339596045843782 10.343435833638067 35.186528049525805 15.847733409638748 16.276283559593143 10.290080626175039 32.203841182563337 231.77688793948619 95.45323511274546 1.3058838910656039 50.24327347293729 91.399337603448657 3.4583868365734807 11.246991687628908 10.044238652626516 2.2987670051224987 6.3240363156364756 4.4926893890005877 16.05634693504544 5.9494517869388801 77.864348161310772 2.6883482676930717 1.8554965668445209 7.3813334197839131 25.238613531328156\nleaf_count=2006 1517 1970 51787 2616 1273 1315 618 3630 4200 746 2354 534 1295 155578 25 957 598 28 2030 481 554 736 1413 554 1949 3282 56 138 1117 792\ninternal_value=-0.000330902 -0.0224107 0.00331597 -0.00350991 -0.00999785 0.0113393 0.00679871 -0.0151167 -0.0100427 -0.00314292 -0.00261821 0.000260313 0.00965234 -0.0276007 -0.00564226 0.00477963 0.00870993 -0.0226546 -0.0117979 -0.00262034 0.0104001 0.01182 -0.00234494 0.000972741 -0.0238305 0.0102731 0.0115471 0.00545058 0.00433407 -0.00211831\ninternal_weight=879.12 123.89 755.23 255.15 173.673 33.3682 500.08 140.305 97.2246 62.0381 116.317 102.569 383.763 104.443 67.6765 151.986 101.743 13.7485 39.4377 23.59 12.343 22.6003 60.4243 53.169 7.25534 81.477 34.8922 19.4463 48.6763 32.6199\ninternal_count=246149 159692 86457 72650 66752 2588 13807 64164 12377 8747 10339 7957 3468 157584 7367 2173 1216 2382 7265 3065 1035 1482 5850 3876 1974 5898 590 2108 2463 1909\nis_linear=0\nshrinkage=0.03\n\n\nTree=106\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 11 16 3 17 7 5 7 5 5 18 14 7 5 6 2 7 13 13 11 0 18 3 18 0 14 2 13 11 3\nsplit_gain=42.6049 31.3235 23.1621 20.2118 13.6799 10.7846 14.5483 17.1023 11.9391 8.75465 8.72064 8.51528 8.09516 7.91015 9.49431 7.88373 10.0651 7.26292 6.08653 6.05387 5.90099 7.21039 10.8135 5.42192 6.36568 5.28235 5.26133 5.0609 4.98033 4.72933\nthreshold=0.52258841453381233 0.77813183983917178 1.0000000180025095e-35 0.41780940659066851 3.9653415636860969 0.89599609375000011 0.89599609375000011 1.0000000180025095e-35 0.89599609375000011 1.7006835937500002 6.8062759569045816 0.0095533965833058416 1.0000000180025095e-35 1.4980468750000002 3.3837890625000004 -5.5186538696289054 3.7958984375000004 -0.044585747678852421 -0.22415943818484466 0.084305138127048276 1.4460303622695712 2.8617925509761908 0.43673077644757391 4.7317931335534196 0.31303196667935901 0.92062330376573798 28.11904335021973 -0.21703879450755229 0.83269824500278011 0.53650715206675215\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 5 6 7 -2 -7 -8 19 -9 -11 20 15 27 25 18 -13 -4 21 22 -10 -5 -25 28 -14 -15 29 -17\nright_child=4 -3 10 23 -6 8 9 11 13 12 -12 17 26 14 -16 16 -18 -19 -20 -21 -22 -23 -24 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.027111948069761551 -0.015076358062000061 0.028248776829281268 0.0099968160574828626 -0.018301848478092221 -0.022247700837652432 -0.039173038030713689 -0.017455961244900248 -0.030799444264472606 -0.032575780826982086 0.020312881073532633 -0.027309112626566375 0.0097581051285605194 -0.021323371877062777 -0.0006990548382113564 0.021114124221161892 0.0065430357697785846 -0.029480205281234462 0.022078196450211099 -0.0105429346536612 -0.029271665969272616 0.02589977472607239 0.020979226691590268 0.0092057311955868765 -0.0091904062033086129 0.01836686378315602 -0.0085538105161737862 0.0079286819192934035 -0.020236802632524874 -0.017731202106091495 0.017037002608682343\nleaf_weight=113.51241446704807 30.63894804505253 15.493372130673377 43.620763898579753 12.210998156355343 17.119612410322588 3.9323211950613759 55.90613217698774 3.6151993935200144 6.3434048744384226 12.277137086915898 6.7598821044302886 73.882018763142696 12.297580427057257 53.672291122718889 30.613031662098365 110.9219873473412 5.7237472978304131 36.306529401568696 15.296395497640331 2.8739902717934447 32.314343477366492 24.798211509969406 22.157428708393127 14.025854175532006 14.273461057367966 15.273585826842462 8.767980192351386 14.371765451171084 5.0127940553938961 60.311983237217646\nleaf_count=163413 15306 117 3639 2317 12122 574 14114 226 370 1255 6453 4288 3309 3720 238 3773 100 841 1116 309 176 1554 86 2726 777 973 564 1231 49 344\ninternal_value=-0.000400946 -0.0116965 -0.0148036 -0.020668 0.00347309 0.00420106 -0.000665418 0.00474937 0.00733826 -0.0103021 0.00222565 0.00953427 0.00185177 0.00791881 0.0055609 0.00369357 0.00672327 0.0109306 0.00619766 0.0070043 0.0159101 0.00948802 -0.000882554 -0.00239347 0.00486127 0.00795275 -0.00934044 -0.00505901 0.0094234 0.0103167\ninternal_weight=874.325 222.771 207.277 154.023 651.554 634.435 248.988 159.739 385.447 89.2488 53.2546 129.1 33.3427 381.515 295.901 265.288 197.244 125.485 89.1784 46.4948 85.6134 53.299 28.5008 40.5103 28.2993 191.52 21.0656 68.0441 176.247 171.234\ninternal_count=246080 179751 179634 169233 66329 54207 41019 21777 13188 19242 10401 6471 5128 12614 10428 10190 5239 6245 5404 3948 2186 2010 456 5820 3503 5139 3873 4951 4166 4117\nis_linear=0\nshrinkage=0.03\n\n\nTree=107\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 1 9 2 17 4 18 4 5 18 10 3 12 12 13 21 18 3 4 17 4 4 22 2 5 14 14 13 14\nsplit_gain=60.3768 30.9932 24.5684 14.1889 11.5928 8.4775 7.81074 7.65138 7.01572 6.88054 6.67257 6.57663 6.41836 5.92836 7.8413 5.8496 8.06636 6.08761 5.65102 5.329 9.72194 6.32617 15.3755 8.50807 6.67172 6.53745 6.23842 5.83996 5.39744 5.07219\nthreshold=0.26032702032866095 0.89587973461402759 3.3033179733375269 1.0000000180025095e-35 -30.838275909423825 5.817026225953116 0.84667968750000011 0.89587973461402759 8.9774329662323016 1.7006835937500002 2.9174053685313028 3.6242520361853052 0.016076142967085191 0.086834563006374169 0.88192242588002812 0.14796868535883259 3.5000000000000004 5.2755571088971331 0.40911314460322756 3.7858886718750004 1.849927262362129 1.0693359375000002 1.6286621093750002 0.93750000000000011 9.8937015533447283 2.2500000000000004 0.97222286375224221 0.84677952177506588 -0.38940835686028247 0.32282440459636236\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 10 3 4 9 6 -6 -4 -9 18 12 -3 -1 -5 -15 17 -17 19 -2 20 21 27 -23 24 -24 28 -22 -8 -25 -26\nright_child=2 11 7 13 5 -7 15 8 -10 -11 -12 -13 -14 14 -16 16 -18 -19 -20 -21 26 22 23 25 29 -27 -28 -29 -30 -31\nleaf_value=0.017529984290098978 -0.022518703694788356 0.015876816009398075 0.020877726635789552 0.020121638704331164 -0.018126191096719643 -0.029676604938140876 0.012668189788980612 0.0065902416817490948 0.031391547499646989 -0.032307978125539843 -0.029198449262932535 -0.032664677648069407 -0.019815526091194805 -0.02250584708303224 0.018789826192309955 0.022694679217967553 -0.016989082078635753 -0.019063453779075142 0.0014451394452841498 0.0070379420540930913 -0.02810440814311034 -0.025253316563266841 -0.025289557373709227 0.015237058999880735 0.011463585435089385 -0.024173063188453525 0.028547842845078442 -0.023113431901341915 -0.00086786210225412201 -0.011540271224966403\nleaf_weight=5.1852196647669215 13.916477322763965 22.758117311488608 52.414057312533259 36.884147393750027 19.42944857744078 7.9412947931268709 44.153166886477266 188.52439592889277 9.9684059685096127 15.173997993351803 88.820386040153608 1.8716008291594324 11.962879431684994 7.3878741900989535 7.1791423580143601 20.518644094583582 5.076217862777411 13.637939335472764 21.524890246873838 85.605545440092101 15.81937500150889 22.379440202526894 14.266406562965129 48.023441679091775 14.658278644346863 4.7482971544523016 1.1472140115220089 3.6003728554642285 30.141013848035072 18.22383895162784\nleaf_count=629 11597 2055 190 1029 10467 5912 2196 1684 48 11318 157071 147 6576 347 41 140 40 5006 1211 3282 6372 7468 5093 1040 605 355 34 915 1054 2222\ninternal_value=-0.000212516 -0.0189034 0.00318822 -0.000851574 -0.00268303 -0.000819767 -0.000105592 0.010752 0.00798689 -0.016001 -0.0259053 0.0110576 -0.00817934 0.0138643 -0.00225422 0.000968044 0.0144813 -0.000167266 -0.00832428 0.000743857 -0.00175818 0.000138345 -0.00285351 0.00116407 -0.00889807 0.00692377 -0.0228981 0.00955169 0.00909432 -0.00124987\ninternal_weight=852.942 130.598 722.343 471.436 419.985 369.37 361.429 250.907 198.493 50.6154 105.968 24.6297 17.1481 51.4512 14.567 341.999 25.5949 316.404 35.4414 302.766 217.161 200.194 152.441 130.061 47.1485 82.9128 16.9666 47.7535 78.1645 32.8821\ninternal_count=246144 166478 79666 77744 76327 52201 46289 1922 1732 24126 164276 2202 7205 1417 388 35822 180 35642 12808 30636 27354 20948 17837 10369 7920 2449 6406 3111 2094 2827\nis_linear=0\nshrinkage=0.03\n\n\nTree=108\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 1 10 4 1 17 2 18 5 6 4 13 2 9 18 18 6 2 10 13 14 14 13 21 18 2 21 18 4 5\nsplit_gain=67.0837 23.6528 18.732 18.3821 22.2522 13.0637 9.49886 7.11138 6.5189 7.52392 6.98522 7.10263 6.49441 6.23005 6.60174 6.17085 7.13261 6.01984 5.99404 5.32856 5.77598 4.87554 4.73406 11.3532 5.95861 4.61509 4.87493 3.90849 5.8184 3.8174\nthreshold=0.24961073363565869 3.3033179733375269 1.7005986908310777 3.7336425781250004 1.4475910192202701 1.9205778057410925 21.425971984863285 3.4811217321331038 6.0800781250000009 3.3837890625000004 9.9511616230011004 -0.05385377781229627 -16.901534080505368 1.0000000180025095e-35 1.4978661367769956 1.4978661367769956 1.4980468750000002 -43.067142486572259 3.6242520361853052 -0.18869652365325987 0.92413204406002747 0.032916404154818342 1.0000000180025095e-35 3.5000000000000004 2.249904835165133 42.119770050048835 5.5000000000000009 5.2496487501035594 1.0693359375000002 3.8818359375000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 3 6 4 13 17 -1 -8 9 10 11 19 -12 14 -2 -3 21 -6 -4 20 29 -17 24 -24 -19 26 -23 -16 -29 -5\nright_child=1 15 18 8 5 -7 7 -9 -10 -11 12 -13 -14 -15 27 16 -18 22 -20 -21 -22 25 23 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.029160538576990237 0.00091892995707996148 0.016090239340388311 0.01722688864935323 -0.0016725762414060825 -0.020061186625538782 -0.025124829338286719 0.0070819838891878206 -0.027409057628863226 -0.029414658124196021 0.023403189962305934 0.017676695378336819 0.014825271811772714 -0.039384355873408876 0.024605005623830889 -0.031379660879257289 -0.016687781904972481 0.025848109449490796 0.0047254662329577913 -0.031358442385102575 -0.023381357970299752 -0.026129430133954142 0.0098070070218421908 0.023755409144133079 -0.025523881181949301 -0.0058600129695779999 -0.026396711165091086 -0.0053708684744041384 0.023604540652870026 -0.02565249032480112 0.014100340711749197\nleaf_weight=86.529436738430377 8.4110246695927326 111.28583080973476 13.583643317033419 26.907244369143882 10.627473895583533 17.614674885116983 13.221417624023163 7.7371530249220077 3.1925485552346791 25.144942732178606 1.9407479187066199 58.518432048746035 4.6279027800774193 1.7523918847073208 39.680680094101263 8.3139005533885193 14.594829192850737 119.16131445614155 1.7670024279359484 6.4931680160807437 4.4328306680399683 96.497513434267603 20.062648878432814 4.384401708841323 78.545480954533559 2.9075083074858403 22.983260474866256 2.6914074325759421 4.2750372531882022 26.965680919223814\nleaf_count=146269 796 402 882 4640 1962 3600 4155 14114 127 243 96 2146 146 159 44201 116 85 1294 153 1502 1904 982 124 53 8187 96 263 368 5678 1474\ninternal_value=-1.28208e-06 0.00349319 -0.0204032 -0.000542325 -0.00481725 -0.000940149 -0.0248061 -0.00631923 0.007736 0.00868465 0.00567822 0.00734305 -0.0224173 -0.021594 -0.0234771 0.0107871 0.00664314 0.000989561 0.0100584 0.000463403 0.00346833 0.00431864 0.00208198 0.0142618 0.000511712 0.0058693 0.00686705 -0.0278029 -0.00604835 0.00633552\ninternal_weight=844.852 722.013 122.839 465.43 307.207 250.396 107.488 20.9586 158.223 155.031 129.886 123.317 6.56865 56.8105 55.0581 256.583 145.297 232.781 15.3506 64.7989 58.3058 130.702 222.154 24.4471 197.707 122.388 119.481 46.6471 6.96644 53.8729\ninternal_count=246217 80644 165573 78700 66422 15220 164538 18269 12278 12151 11908 11666 242 51202 51043 1944 1542 11620 1035 9520 8018 1457 9658 177 9481 1341 1245 50247 6046 6114\nis_linear=0\nshrinkage=0.03\n\n\nTree=109\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 1 4 10 11 17 18 3 7 3 5 4 18 0 1 18 4 4 13 18 21 13 7 3 22 5 5 6 6 18\nsplit_gain=62.5755 20.8513 16.0592 14.9125 12.8762 11.2844 10.0023 9.06259 7.47246 6.40288 9.95067 6.58612 6.84529 6.3629 6.42574 6.42004 6.33608 6.76878 6.08476 5.17327 4.98574 4.73411 4.89704 4.65592 4.50155 7.09438 4.44759 7.01869 9.92856 6.34899\nthreshold=0.24961073363565869 2.5144622974930591 5.5515136718750009 1.7005986908310777 0.042545688445982228 1.2596354777617942 3.2769667020129059 0.016076142967085191 2.2500000000000004 0.58795853567030554 1.2426757812500002 2.5541992187500004 3.0677824455408698 0.94290057586474996 1.048858409861958 4.2121001622835372 3.4438476562500004 4.1906738281250009 -0.051402772639976051 1.7005986908310777 5.5000000000000009 1.0000000180025095e-35 0.89599609375000011 0.47260169401963864 0.93750000000000011 1.4980468750000002 6.0800781250000009 3.3837890625000004 2.0131835937500004 6.3465127852423047\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 2 4 6 5 13 7 -1 9 10 21 -12 16 14 -2 -16 -13 -18 -6 -11 -21 22 -3 -7 -20 -26 27 28 29 -4\nright_child=1 8 26 -5 18 23 -8 -9 -10 19 11 12 -14 -15 15 -17 17 -19 24 20 -22 -23 -24 -25 25 -27 -28 -29 -30 -31\nleaf_value=0.020157729549943222 -0.029370403188818409 0.002540034676757751 0.011892444358009941 0.0081294725275308111 -0.026705842990414546 -0.026131821289771453 -0.029180989025501518 -0.018682738495691964 0.019585240492253599 0.018119797973700596 -0.0091362348319956148 0.020425033218398628 -0.015802718585819539 0.00024397233792294461 -0.018671334588843867 0.0034300457550606824 -0.036119236537662053 0.01083061772771522 0.019325814612713134 0.0097851429627426722 -0.012869423011605784 0.021402439320447813 0.020003669856114639 -0.0067713926866602575 0.013484350478908398 -0.036672370483959459 -0.021589068936945805 0.02365211489872138 -0.021572765756707513 -0.025973737796894086\nleaf_weight=7.0074599796498687 14.247538026327673 69.023520540766185 54.669568058758159 13.989542858980711 2.9021778617898226 40.57189458486755 82.092484708777192 15.899780983283561 42.54356344952248 59.305564070120454 64.921046995048528 23.501012283843011 9.2538880625506845 85.272693119120959 20.703755348360573 25.433317699556937 3.4632750782184285 6.4179368657059968 25.563250285529648 38.640083542093635 10.291356260189785 23.189269181340933 17.410587504971772 16.48608132780646 6.2103407867980405 2.9353276872279821 3.4509318771597455 22.169477605959401 10.177254834477933 3.3612578034080789\nleaf_count=984 21503 1776 2325 1011 1627 26859 153249 10331 283 187 1085 135 515 4759 4567 7297 48 61 553 378 101 173 137 1928 188 229 133 241 1053 2575\ninternal_value=-0.000455491 0.00296213 -0.00246002 -0.0204784 -0.00656393 -0.00955305 -0.0243307 -0.00645503 0.00787946 0.00629378 0.00329592 -0.0031757 0.00590166 -0.00505497 -0.01242 -0.00667353 0.012203 -0.00743201 0.00937695 0.0122238 0.00485807 0.00961687 0.00624632 -0.0207509 0.0130574 -0.00464149 0.00800983 0.00936392 0.00444521 0.00929631\ninternal_weight=821.105 702.116 334.155 118.989 240.326 202.715 105 22.9072 367.961 325.418 217.181 107.557 42.6361 145.657 60.3846 46.1371 33.3822 9.88121 37.6111 108.237 48.9314 109.623 86.4341 57.058 34.7089 9.14567 93.8285 90.3776 68.2081 58.0308\ninternal_count=246291 80716 75837 165575 69510 66913 164564 11315 4879 4596 3930 1844 759 38126 33367 11864 244 109 2597 666 479 2086 1913 28787 970 417 6327 6194 5953 4900\nis_linear=0\nshrinkage=0.03\n\n\nTree=110\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 1 9 18 0 11 7 3 19 7 5 5 11 11 11 17 0 7 10 14 1 1 17 19 1 18 3 0 11\nsplit_gain=64.5623 21.1166 18.9281 14.9376 17.0395 10.7498 10.4484 7.7579 7.46787 7.07468 7.10189 6.24854 6.41982 6.20463 7.54763 5.37239 5.18647 4.99459 5.72915 4.96412 4.69764 5.64784 4.69962 4.10038 4.84276 4.37934 3.96883 5.93416 5.78686 5.72259\nthreshold=0.24961073363565869 1.7005986908310777 4.0016611463566001 1.0000000180025095e-35 2.524928003624769 1.4460303622695712 0.75683518236239655 4.1972656250000009 0.29586597258655462 234.50000000000003 0.89599609375000011 2.5244140625000004 2.2500000000000004 0.11679098783484217 0.77813183983917178 0.49644476220754463 2.7033165568765605 0.29896707506516013 3.3134765625000004 3.6242520361853052 0.22768215226658742 1.5661550609387962 1.7090960825073716 0.19767305700411134 255.50000000000003 2.5144622974930591 1.2424533248940002 0.65908129556435835 1.4460303622695712 0.59460817960898271\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 3 4 11 -1 7 8 15 -10 23 12 16 -5 -15 20 26 18 -12 -3 21 -6 -23 24 -11 -25 29 -28 -29 -2\nright_child=2 19 -4 13 6 -7 -8 -9 9 10 17 -13 -14 14 -16 -17 -18 -19 -20 -21 -22 22 -24 25 -26 -27 27 28 -30 -31\nleaf_value=-0.028671907657197432 0.010432599067325212 0.017455635294202212 0.012723104542073429 0.022187210306194558 -0.025585235175117984 0.0066615571736111084 0.016049569203323327 0.028740559751786375 -0.025132991988397448 -0.024251068533059941 -0.028465784761419671 0.013727593611793057 -0.024992390153480339 -0.025175519287191136 0.015116091983345906 0.017606751451073165 -0.02926265746721661 0.0044119313385700184 0.0217798728226118 -0.029174635746761726 -0.027939738070987569 0.023238087432794267 -0.012080065508205062 -0.027231860962706869 0.014444341730563715 -0.0073950754593369297 -0.0028722506106563877 -0.0053893010798049049 0.029041702275072048 -0.018293679281741259\nleaf_weight=91.928970205863152 83.623414522618987 13.801782510417977 150.8834463453386 37.46451601600711 9.163039267852584 8.0857032194762706 15.665892859688027 3.9338484822656019 14.620292012710705 3.1619776679144698 7.7404460824545804 49.816002754261717 6.6512133268843163 5.7317787220308665 10.057259566383435 2.7284594754601121 3.4045101367664747 73.87983010186872 1.8592508312431162 1.4630664535507083 32.618052023361088 5.3604043795930902 6.3372374922619201 18.279469917128154 8.7486735661295807 23.048982571024681 98.745995715435129 6.7567494222894338 10.264546607155351 5.8364355415105811\nleaf_count=163206 772 853 979 1046 7238 1070 131 28 3675 1246 3286 549 254 354 65 70 828 3809 116 146 38023 268 738 10876 1104 1153 3730 75 40 44\ninternal_value=-9.44261e-05 -0.0207896 0.00335637 0.00074702 -0.000897813 -0.0257777 -0.00691196 -0.00868042 -0.00952411 -0.00598487 -0.0037696 0.00426192 0.00201824 0.0156872 -0.000138988 -0.0187824 0.00299446 0.0017026 -0.0175976 0.0114674 -0.0209281 -0.00918793 0.00465987 -0.0122233 0.00308898 -0.0166049 0.00366887 -9.91156e-06 0.0158334 0.00837696\ninternal_weight=811.661 115.28 696.382 545.498 492.245 100.015 227.146 211.48 207.546 151.339 136.719 265.099 215.283 53.2536 15.789 56.2072 208.632 83.4795 9.5997 15.2648 53.4787 20.8607 11.6976 53.2391 11.9107 41.3285 205.227 115.767 17.0213 89.4599\ninternal_count=245772 165275 80497 79518 78053 164276 71761 71630 71602 25265 21590 6292 5743 1465 419 46337 5489 7211 3402 999 46267 8244 1006 14379 2350 12029 4661 3845 115 816\nis_linear=0\nshrinkage=0.03\n\n\nTree=111\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 5 17 1 11 7 12 2 6 5 2 9 2 17 5 1 18 18 14 18 14 0 1 14 2 2 21 2 18\nsplit_gain=62.469 22.406 13.3315 11.2465 10.0133 9.07667 9.00674 8.83913 8.02316 7.35037 6.52445 6.00132 5.79922 5.14298 5.15208 4.8194 4.79493 4.8513 6.56501 4.61877 5.41085 4.44856 5.31502 4.26503 6.17532 4.18626 7.02597 4.06621 3.93882 3.90053\nthreshold=1.7622051334857016 -19.576506614685055 4.9648437500000009 1.6722174619049508 0.60367420583093068 0.72257959311131259 1.0000000180025095e-35 0.93735200266714214 -42.103004455566399 2.9179687500000004 4.8710937500000009 21.425971984863285 1.0000000180025095e-35 132.99933624267581 5.817026225953116 1.7006835937500002 1.7090960825073716 5.2600927804663344 5.0271592211193568 0.032916404154818342 3.2769667020129059 0.39695441677226773 0.53713645201195115 2.9477339485081675 0.69550561433971103 30.514518737792972 7.3050365447998056 7.5000000000000009 -30.0182991027832 2.0126758453675748\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 3 4 11 -1 15 19 -8 10 12 -3 13 14 23 -2 17 18 21 20 -5 22 -6 24 28 26 -4 -25 -10 -13\nright_child=6 2 25 7 16 -7 8 -9 9 -11 -12 29 -14 -15 -16 -17 -18 -19 -20 -21 -22 -23 -24 27 -26 -27 -28 -29 -30 -31\nleaf_value=-0.027559223178953197 -0.016044454610806849 -0.024592704466815069 0.010076869874060595 0.011381435496993515 -0.024236846721540334 0.020965920335216558 -0.012412283501188184 0.024059294500535972 -0.028659885894651058 0.027144267638899208 -0.025842408444960573 0.015036107875659966 0.023780104642838869 -0.025611048740499173 -0.028510438124618662 0.012806886126380087 -0.031390470176192134 -0.023344800354824703 0.025978146012467278 -0.02943394537443805 -0.024751620280867345 -0.0054640157248871139 0.014979930717000612 0.010774163780624893 0.011025909406599416 0.025657787436913104 -0.02711004378325672 -0.0029536601648274519 9.6803835318473504e-05 -0.014613508392062961\nleaf_weight=74.346544713633193 23.633267939570946 28.007052700031636 23.531421772015168 6.1980916881293506 2.647832880800709 2.9367734529077998 18.004490459119552 2.8617372235748908 3.4783054627187076 15.706498145824296 4.4963145977817467 6.5717458034050633 17.276801574975252 3.5048685416404615 2.7324906984285908 5.8017096188996211 2.922685308585641 5.0441483329268513 10.051868961541912 38.52280589143993 6.9469783490058035 38.746525118905993 23.587469608966785 221.35121215257095 57.858492867759196 9.4041670240694639 4.6313811895961399 20.861587887629867 100.0834443779022 7.4513714677086682\nleaf_count=124941 3059 37108 789 817 1417 30 1664 11 382 84 52 562 138 125 547 300 370 1399 1039 40174 5996 9020 1909 1560 2036 91 204 151 2418 7829\ninternal_value=-0.00077614 -0.0116958 -0.00669843 -0.0100941 -0.0050576 -0.025471 0.00573178 -0.0213432 0.00675425 0.00755161 0.00677759 -0.0168025 0.00718052 0.00642464 0.00676313 -0.0101238 0.00101908 0.00257455 0.00460109 -0.0242669 -0.0081142 0.000950731 0.0102773 0.0070718 0.00326564 0.00937281 0.00324156 0.00958451 -0.0011339 -0.00064265\ninternal_weight=789.2 294.411 217.127 179.56 125.031 77.2833 494.789 54.5296 465.355 447.35 431.644 42.0302 427.147 409.87 406.366 29.435 83.0005 80.0778 75.0337 51.6679 13.1451 64.9818 26.2353 403.633 161.42 37.567 28.1628 242.213 103.562 14.0231\ninternal_count=246222 233706 108735 107651 60653 124971 12516 46998 9157 7493 7409 45499 7357 7219 7094 3359 15154 14784 13385 46987 6813 12346 3326 6547 4836 1084 993 1711 2800 8391\nis_linear=0\nshrinkage=0.03\n\n\nTree=112\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 4 18 2 13 4 3 13 11 18 11 13 18 2 14 3 18 13 17 4 5 11 12 3 18 3 4 11 4 4\nsplit_gain=38.3261 31.8028 24.8097 14.4685 10.3904 10.2879 9.70619 8.07177 6.97713 6.57604 6.55912 7.96825 6.34764 5.77658 9.01703 6.98365 5.73909 7.22503 5.23988 5.20425 5.40639 4.82847 5.69139 4.76911 5.05489 7.31564 6.55618 5.30089 4.58196 8.38367\nthreshold=0.31840721315068282 6.9096149206161508 3.1567740231385479 -25.182852745056149 0.14796868535883259 0.84667968750000011 0.58795853567030554 -1.0000000180025095e-35 0.75683518236239655 1.2424533248940002 8.8817841970012543e-16 0.14796868535883259 0.89587973461402759 -13.247784614562987 0.60348287794770183 0.47260169401963864 3.0201273556387074 -0.4958236943477588 0.29595754201755942 2.5541992187500004 1.4980468750000002 0.63309421653839248 0.60365083802632757 0.32220753877234559 1.7005986908310777 0.40911314460322756 1.4716796875000002 0.41772787082845381 1.6286621093750002 1.0693359375000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 -3 -4 -2 8 12 13 -10 -5 -12 -1 15 23 -7 28 -18 -19 20 -17 22 -16 -15 26 -26 -25 -27 29 -8\nright_child=5 3 4 10 -6 6 16 -9 9 -11 11 -13 -14 14 21 19 17 18 -20 -21 -22 -23 -24 24 25 27 -28 -29 -30 -31\nleaf_value=0.0093773330046965694 -0.016203812518499817 -0.027348655801814951 -0.026307984332516934 0.020293862912444977 0.016822201334429465 -0.013746500155672114 0.011950734166746487 0.0092158751610748411 -0.011967507258699685 0.022689606760332558 -0.016109065103630214 0.022352568241368342 -0.015614357141518483 0.030861336510150269 -0.0048051576801661153 0.003015197448953775 -0.028994400624647282 -0.029162766474510902 0.012075738164996928 0.018454747091480758 -0.023083417946209578 -0.032437675537711423 0.026676230755720936 -0.0030262862566246196 -0.013339627436857376 0.0086874731001046323 0.017324871567808365 -0.036429552445915017 0.017803610128334015 -0.023150853597340988\nleaf_weight=11.108131319429956 19.434522577394091 7.9345902133463815 108.48943823130321 40.518055510416161 4.6189597343327469 41.828220487295766 30.08702427719254 30.866805348734488 5.4153634698595878 24.912395460298285 11.435389727048461 6.947646288201212 39.18006180163502 6.8927873408392761 66.932438962598098 30.729713291919325 5.3536434726265751 2.2179828988155377 20.60419454483781 12.775151663838185 8.2398149735527095 4.1994118876755229 4.6009003518265663 18.598581636906601 21.005753217352321 56.332098672923166 56.357646560180001 1.4689493554178614 93.891316152119543 6.6585807501687659\nleaf_count=481 5555 14518 182716 2248 48 7227 130 2447 50 188 716 59 13548 137 8266 716 1114 149 341 412 339 132 44 127 1808 1519 228 28 464 135\ninternal_value=0.000138263 -0.00927339 -0.015395 0.00848433 -0.0244147 0.00471005 0.00552269 -0.00267985 0.00278426 0.0163167 0.0136874 -0.00116969 -0.0100188 0.00149491 0.00398879 -0.00478082 0.0116789 -0.000588802 0.00701201 0.00258987 -0.00294262 -0.00450488 -0.00244047 0.00799578 0.00677799 0.00143571 0.0123019 0.00694083 0.0143092 0.00514524\ninternal_weight=799.636 261.099 194.263 66.8357 113.108 538.536 519.102 81.155 360.289 30.3278 58.9011 18.383 50.2882 329.961 236.389 93.5729 158.813 28.1758 22.8222 51.7447 38.9695 75.7328 71.5333 160.656 153.763 78.8068 74.9562 57.801 130.637 36.7456\ninternal_count=245890 216781 199240 17541 182764 29109 23554 16476 21221 238 3023 775 14029 20983 12289 8694 2333 1604 490 1467 1055 8442 8310 3847 3710 3355 355 1547 729 265\nis_linear=0\nshrinkage=0.03\n\n\nTree=113\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 11 7 4 17 17 3 4 5 19 3 2 12 2 5 13 12 17 4 4 3 3 2 19 16 3 18 0 18 17\nsplit_gain=42.7314 29.2584 21.3666 16.1674 10.331 8.55052 7.32479 8.0789 8.3857 8.65147 6.70276 6.65381 6.49675 6.41506 5.34323 5.93959 5.28367 6.96334 6.35371 7.09889 5.23604 5.06119 4.80307 4.7065 4.58809 7.15354 5.48418 4.395 4.26499 14.5016\nthreshold=0.52258841453381233 0.77813183983917178 3.4169921875000004 0.84667968750000011 3.9653415636860969 0.29595754201755942 0.53650715206675215 4.4806598424911508 1.4980468750000002 240.50000000000003 0.23327427011668259 -41.62170219421386 0.0027888146193529866 -6.3279094696044913 3.5400390625000004 0.14796868535883259 0.34440402210285487 2.3696055953329416 1.6286621093750002 1.0693359375000002 0.0037166969630965979 0.41780940659066851 -22.80348968505859 217.50000000000003 1.0000000180025095e-35 0.53650715206675215 5.1387309967764621 1.2661233611507157 0.89587973461402759 0.43370896730050829\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 -2 6 24 7 8 9 20 -10 -11 -9 -12 -14 23 18 -18 19 -8 -5 -7 -4 -16 25 -1 -26 -20 29 -15\nright_child=3 -3 22 4 -6 21 16 12 10 11 13 -13 14 28 15 -17 17 -19 27 -21 -22 -23 -24 -25 26 -27 -28 -29 -30 -31\nleaf_value=-0.021034891619404545 -0.017304886732341759 0.028486464910606187 -0.023718868550762513 0.020645771388451892 -0.02062162583856969 -0.026854637804409911 0.0092260704593802867 0.016840126390239282 -0.024714412331609634 -0.021366551682302688 -0.014851337390947099 0.0092327694849418419 0.021083851423989126 -0.043333097196464485 0.00063210548552642863 0.013753461017935004 0.0051101340633552903 -0.032462363152222465 0.0094428605433373603 -0.016070782830871409 -0.021965579488618357 -0.011243598926933886 0.013192991398984775 -0.03636648139857368 0.023039170934514741 0.018088392387038146 -0.023318044222865226 0.020642571968987153 0.0059167491251821093 0.01470891762620396\nleaf_weight=18.594406343727318 28.20982654377076 14.090662906877695 2.690005851043682 2.3262649783282532 12.281494690170801 89.408671190081805 44.817487328255083 58.83975046613341 13.167443156406078 5.788800112604803 26.520755427700351 132.53514578107206 5.7686068961338632 6.1136124515905959 5.1089114330243337 5.6025414059404275 25.04755820136052 4.353192747978027 58.943592177645769 11.801223321002906 13.656876225242739 26.228921500922297 24.820114670670591 6.7183706653595436 7.2933957799978071 4.5274954165797672 2.2731747439320316 75.019120783777907 56.095540972673916 7.7079313627909869\nleaf_count=26390 18907 120 3406 201 8423 141458 321 1636 9485 2296 2484 14058 76 26 75 53 200 62 782 550 3765 5088 1235 223 385 326 1344 339 2534 49\ninternal_value=0.00060389 -0.0117766 -0.015146 0.00449437 0.00558595 -0.0196314 0.00619006 0.00345225 0.000880902 0.00537369 -0.00543564 0.00780803 0.0116139 -0.00257673 -0.00178989 -0.0100935 0.0104605 -0.00133782 0.0122781 0.00376596 -0.0149132 -0.0234405 0.00887831 -0.0215812 -0.00586057 -0.0129419 0.0108596 0.0158212 0.00222301 -0.012156\ninternal_weight=796.351 189.927 175.836 606.424 578.214 148.326 565.933 345.951 263.912 154.307 109.605 138.324 82.0382 96.4378 23.1984 17.4298 219.982 29.4008 190.581 56.6187 15.9831 115.638 27.5101 11.8273 32.6885 23.1219 9.56657 133.963 69.9171 13.8215\ninternal_count=246297 179752 179632 66545 47638 174991 39215 36961 34898 20320 14578 16354 2063 5093 427 351 2254 262 1992 871 3966 146546 4641 298 28445 26716 1729 1121 2609 75\nis_linear=0\nshrinkage=0.03\n\n\nTree=114\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 3 1 2 4 0 17 3 6 17 4 2 2 3 1 11 4 21 2 12 3 2 1 19 4 6 1 0 21\nsplit_gain=55.8817 18.4052 13.851 12.5028 17.2144 10.0152 8.32815 8.09107 7.19188 9.01692 6.78807 6.61071 6.22407 7.91323 12.3163 7.42315 6.16325 8.244 5.57386 5.24775 4.77716 4.76045 5.38138 4.5611 4.42835 4.34629 4.0957 3.83864 4.73818 4.81788\nthreshold=0.18336881478606726 0.84667968750000011 0.0054559557070609941 2.1468275719512682 -18.836229324340817 2.9518512487411503 1.4460303622695712 1.6722174619049508 0.58795853567030554 1.2426757812500002 0.27410064895469471 5.9760742187500009 18.240050315856937 -17.667204856872555 0.25176753112663547 0.6973310998193073 5.5511151231257837e-16 1.6572265625000002 2.5000000000000004 -47.402034759521477 0.90934753668501156 0.22136347059745312 -10.210010528564451 1.4475910192202701 265.50000000000006 3.6123046875000004 0.89599609375000011 4.0016611463566001 0.40053340798327824 3.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 11 4 10 7 20 12 9 19 -3 -1 13 -6 18 -16 -10 -18 -15 24 -4 22 -21 -14 -5 -8 27 28 -23 -30\nright_child=1 3 6 8 5 -7 25 -9 16 -11 -12 -13 23 14 15 -17 17 -19 -20 21 -22 26 -24 -25 -26 -27 -28 -29 29 -31\nleaf_value=-0.018255283422386744 -0.020598438799420346 0.0008291712883286334 -0.029663112731542682 -0.031930771351480279 0.024691501608029295 0.0080066645522323451 -0.023539935691320728 -0.030001443970831432 0.01575795283320746 0.022216981122659931 -0.026695374757466282 0.019030981590813876 -0.016171494851974059 0.019109144051332946 0.0092812820082868713 -0.033240424477386635 0.01406891939738055 -0.032722901704089292 -0.024154189274312374 0.014638203414296148 0.018311031491294434 -0.023558361534836664 -0.034661494142877079 0.012471825443429502 0.014512950470258034 0.021395839905760553 -0.031344210562593448 0.010206296923974735 0.0078432431772588931 -0.0029877896575325352\nleaf_weight=6.131360297324135 27.078634927573148 11.068986050227975 74.45420681493124 5.6809517219371619 4.4569426173402453 117.96428916087461 2.0468616713624206 9.5971828620531578 109.61160519113764 23.39620206534164 28.131940556595509 9.7853437882586132 5.1551694135196149 10.217740356158172 4.0020612826410824 20.58851704562403 6.9265881450846774 4.9744444671086967 2.5951822885035645 2.3399982959526824 1.286671608686446 5.1761887682951047 4.812092276843031 27.846439444721909 1.8833894212730227 4.4378879610740114 1.8333419024711464 83.106767346616834 86.279469094442902 63.386635363800451\nleaf_count=6658 16814 6096 151309 640 516 9029 762 8676 716 243 23707 1061 3104 2879 1831 4103 46 48 2183 139 5 901 475 838 34 100 71 536 1533 983\ninternal_value=7.36e-05 0.00318798 -0.020946 0.00422329 -0.00116146 0.00238853 -0.0258539 -0.0054602 0.00747559 0.00478407 -0.0190961 0.00443767 -0.00196005 -0.0095408 -0.0141566 -0.0262279 0.0135417 -0.00650976 0.00923213 0.00310077 -0.0286201 0.00388701 -0.0187147 0.00765345 -0.0200227 0.00596216 0.00463609 0.00504333 0.00222455 0.0032667\ninternal_weight=766.253 668.111 98.1423 641.032 241.624 202.424 82.2256 84.4592 399.408 277.895 39.2009 15.9167 74.8621 41.8604 37.4035 24.5906 121.513 11.901 12.8129 254.499 75.7409 246.934 7.15209 33.0016 7.56434 6.48475 239.782 237.949 154.842 149.666\ninternal_count=246036 86141 159895 69327 62962 33159 152176 24130 6365 5555 29803 7719 15454 11512 10996 5934 810 94 5062 5312 151314 4638 614 3942 674 862 4024 3953 3417 2516\nis_linear=0\nshrinkage=0.03\n\n\nTree=115\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 10 4 1 4 2 11 19 21 19 2 7 4 14 19 0 2 0 4 5 3 4 5 3 14 21 2 2 21\nsplit_gain=47.4891 18.0276 17.3422 14.2409 9.38259 9.10936 7.85972 7.72512 7.88184 6.21098 6.49871 5.55595 4.75794 6.41176 6.01963 5.49061 4.59922 5.04172 4.55418 4.44633 13.9014 5.40035 4.42183 4.47043 6.07059 4.71846 4.64292 4.39478 4.37066 4.32435\nthreshold=0.24961073363565869 -39.283819198608391 0.89587973461402759 0.84667968750000011 2.9477339485081675 4.9616699218750009 21.425971984863285 0.042545688445982228 246.50000000000003 3.5000000000000004 262.50000000000006 24.399772644042972 0.89599609375000011 1.0693359375000002 0.63637684326820176 254.50000000000003 0.96046990436194013 -41.14759445190429 1.6765212893675572 8.9774329662323016 4.4707031250000009 0.28287969384316608 4.9233398437500009 1.2426757812500002 0.58795853567030554 0.032916404154818342 2.5000000000000004 -10.988907814025877 -18.054428100585934 6.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 16 6 -3 5 7 -1 8 18 -9 -11 -8 13 14 -10 -15 -2 -18 -5 20 21 28 23 25 26 -23 -25 -14 -6 -20\nright_child=1 3 -4 4 19 -7 11 9 12 10 -12 -13 27 15 -16 -17 17 -19 29 -21 -22 22 -24 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.029422199586177547 -0.024305826487013509 -0.018850860246284015 0.0078321850408827297 -0.01610534041123924 0.013851916469331593 0.008704591907858647 0.017925053980733324 0.01816851084157815 0.014841164966474156 -0.032615666378634868 0.017469390301319762 -0.016869670490158809 -0.0060548993112539487 -0.031526765902201201 -0.013020962606605584 -0.0048794075211306773 0.0059722292897474447 -0.035641644834991944 0.010813164944721926 0.027349223302790451 -0.045214963607444071 -0.01729027304941692 0.024810860249938511 -0.014761521729342548 0.013211224343564273 0.014560309544348716 0.0032772445806577366 0.0099158273514826437 -0.02926911710599292 -0.027595942870290574\nleaf_weight=67.52423412310236 25.230803321377607 22.95932724704835 17.141343791263353 54.820712683584134 2.5480983857996762 105.80579342833516 5.0216185526442159 25.717764207889559 16.842034924404292 4.3252106872205323 3.2977345136605436 12.486774513243288 21.175172484428913 11.126292263867073 10.517407498002283 17.389102351382462 11.707148814311948 2.3076559879409606 11.544239229609955 10.642408087849615 3.3392594289034596 3.5587959629483512 15.97815460572019 15.598427591961807 44.352085418999195 98.190932146157138 55.776863340986893 51.773323649773374 4.9231373551301649 2.3492650632397263\nleaf_count=144830 18671 13139 2184 21614 54 4935 2194 630 1124 266 72 16020 3551 5929 1188 3947 402 73 322 64 23 31 90 278 186 707 427 2428 271 154\ninternal_value=-0.000406125 0.00257515 -0.019323 0.00384332 0.00475439 0.00145859 -0.0248245 -0.00188529 -0.00415022 0.0112641 -0.0114354 -0.00646083 0.000245109 -0.00628049 0.00404905 -0.0159927 -0.0168854 -0.00279726 -0.0122758 0.0090848 0.00818078 0.00910468 0.00992463 0.00872237 0.00462796 0.0132889 -0.000814455 0.00526073 -0.0146634 0.00290172\ninternal_weight=755.971 653.797 102.174 614.552 591.592 336.684 85.0326 230.878 197.538 33.3407 7.62295 17.5084 128.823 55.8748 27.3594 28.5154 39.2456 14.0148 68.7142 254.908 244.266 240.926 233.455 217.477 115.727 101.75 71.3753 72.9485 7.47124 13.8935\ninternal_count=245804 80576 165228 61430 48291 46160 163044 41225 40257 968 338 18214 18167 12188 2312 9876 19146 475 22090 2131 2067 2044 1719 1629 891 738 705 5979 325 476\nis_linear=0\nshrinkage=0.03\n\n\nTree=116\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 1 5 1 17 2 9 3 3 21 8 13 18 9 2 2 2 13 5 17 2 5 11 12 13 7 5 18 17\nsplit_gain=47.784 13.3015 12.6472 13.0988 12.7088 7.10195 6.7836 6.06758 9.00012 5.83568 4.97621 4.36991 4.26222 8.22407 4.48829 4.36634 4.81241 4.25867 4.17016 4.8352 4.17004 4.07995 3.7812 3.75874 8.08666 6.2957 5.42238 5.29551 3.70204 3.69098\nthreshold=0.18336881478606726 1.0000000180025095e-35 2.9477339485081675 2.6015625000000004 1.9634439907166827 1.490403556610195 -26.006598472595211 1.0000000180025095e-35 0.42638606281916264 0.024199563577468081 3.5000000000000004 1.0000000180025095e-35 -0.056686766081373456 2.0126758453675748 1.0000000180025095e-35 -4.3113665580749503 34.417465209960945 44.569120407104499 -0.66255419986965325 0.89599609375000011 0.56338221076126505 2.0782871246337895 1.8691406250000002 0.91125284794847328 0.083740223399813532 0.14796868535883259 3.2382812500000004 5.1621093750000009 0.89587973461402759 1.2428969707451916\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 4 5 12 -5 8 10 -3 -4 -7 13 21 -15 -14 -17 23 19 -10 -12 22 -2 24 26 27 -8 -26 -22 -29\nright_child=2 9 7 6 -6 11 17 -9 18 -11 20 -13 15 14 -16 16 -18 -19 -20 -21 28 -23 -24 -25 25 -27 -28 29 -30 -31\nleaf_value=-0.026979938735226403 0.015518391877038438 0.010013005705402221 0.0023637074933221926 -0.02594667988907956 0.0015117492721485309 -0.031364578373515764 0.00078726676815460081 0.02519595933943438 0.0085003306983753441 -0.016719676383466366 -0.03074514996886351 0.0082455414148364849 -0.01830668606160071 -0.022039058359914682 0.018332003174606721 0.016286645516926358 -0.021649581647976195 0.01868145573980657 0.010261390617693546 -0.031177743115325784 0.027045924374750292 -0.012242819250228111 -0.024893273209384576 0.023683173095513942 -0.041069227333789662 0.014875695767030685 0.016298764879962058 0.013421576644487163 -0.012002093526368832 -0.027782262720038579\nleaf_weight=67.44778580525599 14.503318134389703 15.146713224396082 33.384207527618855 4.5666065725308718 133.63529266759724 19.920200596519866 47.484695843315421 15.838487820699809 5.5473910938017053 12.511917432453627 8.6046520440722833 2.4322476132729198 5.7501722154083863 41.345062085641985 1.9019397994998133 15.584443309664495 2.7706616273353566 25.913120872297441 196.35204488300951 3.9383707458619028 1.9006173573434333 11.190030718018532 1.4929605727666047 8.1376981865614635 6.7807054026197893 4.1691462891176334 34.546681431733305 3.9492578310309909 7.9291581416036934 2.251538107811939\nleaf_count=151464 443 1706 739 5714 6134 23182 5877 88 37 6594 192 296 2457 33906 152 1878 829 463 1468 77 8 732 89 44 532 35 690 54 275 91\ninternal_value=-0.000114906 -0.0198837 0.00275167 -0.000730624 -0.00481134 -0.0119917 0.00666953 0.00768071 0.00651077 -0.00224148 -0.00459574 -0.0268859 -0.00822558 -0.0119098 -0.0198894 0.00258505 0.00973757 0.00797243 0.00931548 -0.00937579 -0.0166722 0.0010391 0.010504 0.00523807 0.00350339 -0.014686 0.0074374 -0.024565 -0.00265183 -0.00332022\ninternal_weight=756.927 95.1064 661.821 388.326 250.526 116.891 137.799 273.495 257.656 27.6586 51.8186 22.3524 94.5386 70.4333 43.247 24.1053 18.3551 133.233 205.838 9.48576 18.4344 27.1863 15.9963 107.32 99.182 17.1506 82.0314 12.9815 9.82978 6.2008\ninternal_count=246246 159764 86482 83598 70098 63964 13500 2884 2796 8300 1214 23478 40486 35322 34058 5164 2707 7786 1582 114 475 1264 532 7323 7279 712 6567 677 283 145\nis_linear=0\nshrinkage=0.03\n\n\nTree=117\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 16 18 7 7 0 11 12 18 4 3 22 4 5 21 21 3 4 7 4 14 13 13 4 14 4 7 6 14 21\nsplit_gain=36.5643 27.9222 14.2882 11.2657 9.03327 7.51645 6.66903 9.79997 6.42833 6.35965 6.28893 5.90609 5.87236 5.84781 6.08984 5.81799 5.43898 5.63974 7.77918 7.77399 7.52359 6.19865 5.33301 5.18824 5.08923 4.92461 4.89765 5.57531 4.83339 4.78328\nthreshold=0.31840721315068282 1.0000000180025095e-35 3.5694334999727624 1.0000000180025095e-35 2.6015625000000004 1.3771184689138856 0.74021522701736098 0.38126483565173674 5.2755571088971331 1.6572265625000002 0.0037166969630965979 0.93750000000000011 2.1186523437500004 4.2558593750000009 5.5000000000000009 1.5000000000000002 0.32220753877234559 1.1459960937500002 0.89599609375000011 2.3220214843750004 0.30493411065509307 -0.17415164680393 0.14796868535883259 6.3167884349823007 0.79531390260019708 3.2895507812500004 3.6503906250000004 7.1152343750000009 0.63263761892177561 1.5000000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 9 29 6 7 13 -3 -2 -10 -8 -9 14 16 23 -5 20 25 28 -18 -21 -4 -15 -22 -19 -7 -28 -20 -1\nright_child=3 8 22 5 -6 26 11 12 10 -11 -12 -13 -14 15 -16 -17 17 18 19 21 24 -23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.00038715438210181879 -0.019784293882220229 0.011205095393291297 -0.026463625821220648 0.027593384240328492 0.019535529588060128 0.011076953995042065 0.020032838273281443 0.0016147085166771313 0.01832465643919727 0.014956282619021653 -0.019952454575945022 -0.019519451952264462 -0.035750402676353724 0.02218519739763997 -0.011265074882580717 0.022990573738033957 -0.010983554901867977 -0.017834395942319808 0.018380417810876286 -0.0096942665533613862 0.023560860378742759 0.0099684961883022283 0.010540074420886777 -0.023634452050978451 -0.012802556471025628 0.026245585266718167 -0.036034526551357797 0.018784948119785254 -0.0052159602906691853 -0.017154527977317265\nleaf_weight=24.138248450923129 24.319438346872634 50.687959483868326 85.181513388533858 6.5689505279951836 9.0736047689279058 173.0312769964803 26.910145622212443 5.8960109944164278 4.4818438462243675 5.0586052156213546 12.786613628617486 3.0626423002686343 9.1737773459317378 2.6223498206236426 32.285855668604199 20.296049253229285 9.2855580181349051 27.848413576444731 30.850546090921853 46.805651969181781 25.492212603239754 19.743326877622167 3.1487245444441205 4.9016859670809936 3.3408397180028251 1.6035103012982279 3.3435414917767039 1.9179734177887438 9.8530447558805445 41.101554768334609\nleaf_count=8270 7387 2636 180324 535 218 1155 217 264 457 629 7638 46 693 179 3347 662 141 5594 600 4104 865 802 42 333 638 107 40 16 893 17384\ninternal_value=-0.00038387 -0.0102227 -0.01694 0.00421831 -0.00710593 0.00536516 0.00234136 0.00075094 0.00586667 -0.0135069 -0.00955876 0.0154916 -0.0219406 0.00225156 0.00053807 0.0149371 0.00268915 0.00160176 -0.00126363 0.00251409 0.0116475 -0.00379963 -0.0250413 -0.00693575 0.0190596 -0.0146514 0.0101846 -0.0162426 0.0126804 -0.0110502\ninternal_weight=724.811 230.6 162.644 494.211 74.3134 464.833 286.541 256.568 67.9564 29.378 17.2685 29.9728 15.0698 241.498 213.678 27.8201 181.392 174.823 136.704 107.253 38.1186 66.549 88.3302 7.52404 28.8331 29.4519 178.293 5.26151 40.7036 65.2398\ninternal_count=246216 216969 206238 29247 25872 21231 20020 19757 10731 8016 8095 263 957 18800 17626 1174 14279 13744 12100 6399 1644 4906 180366 512 1503 5701 1211 56 1493 25654\nis_linear=0\nshrinkage=0.03\n\n\nTree=118\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 10 2 3 1 6 17 13 5 21 17 9 2 2 13 5 21 7 14 21 14 21 12 1 5 4 13 17 14\nsplit_gain=48.8522 12.4851 12.2701 11.873 10.0498 8.44072 6.23481 8.49507 6.21691 5.10899 8.99759 3.91655 4.31455 3.94266 4.63393 9.11702 5.24681 5.063 4.05311 9.72986 4.72424 4.17887 3.95851 3.90214 3.89647 3.85985 3.82947 3.80852 3.72589 3.68702\nthreshold=0.18336881478606726 0.84667968750000011 0.89587973461402759 -34.252307891845696 0.53650715206675215 2.9477339485081675 3.3837890625000004 3.5426598534814651 0.14796868535883259 5.7089843750000009 6.5000000000000009 3.1006363674515387 1.0000000180025095e-35 132.99933624267581 22.867874145507816 -0.072294345214240982 1.7006835937500002 2.5000000000000004 0.89599609375000011 0.28824823456061044 1.5000000000000002 0.42891595223756424 1.5000000000000002 0.30400123877355156 4.0016611463566001 2.9179687500000004 3.1247558593750004 1.0000000180025095e-35 1.2596354777617942 0.60840604842079971\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 23 -1 4 -3 6 7 8 29 -8 -11 12 13 14 15 16 18 25 19 24 -21 -20 27 -2 -7 -18 -24 -17 -16 -5\nright_child=1 3 -4 5 -6 11 9 -9 -10 10 -12 -13 -14 -15 28 22 17 -19 21 20 -22 -23 26 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.02562299747442361 -0.021460185995733072 -0.02073183805112698 0.0036125373615333006 0.0034286705532875447 0.006639648825345775 -0.040728519990716222 0.02571632529185696 -0.024091175989384896 0.016759972562452541 -0.034056323662892754 0.023606851459709245 -0.012001859372805153 0.026931242142597893 -0.021984256421458861 0.022670315446635069 -0.015227373284438827 -0.038171812813806778 0.0064575211078670711 0.033031983126995339 -0.015709985428195114 0.017994673490182841 0.013111689468050778 -0.018120112103578327 0.010514781046084467 0.00018846478404447938 0.0054161240971903595 0.0082062215479275797 0.021545105680829523 0.002626256818059146 -0.0036196118950406224\nleaf_weight=73.962918710885788 22.577833234538051 30.088947102987731 15.334639934681034 159.54216945725057 19.102433171588928 3.3785660513676694 15.903349734173391 10.901413104376841 21.941515358048491 3.5137736928299992 4.3025672547519207 6.6860629842267363 12.09733283985406 2.7308531142771235 34.441464552655816 2.9721849146299046 3.4543698013294479 40.063193911453709 16.769448329694569 3.4166557649150482 34.355259421397932 25.442933472106233 20.314724879397549 3.4011190953460746 3.9980646371841431 2.72814114484936 5.8402740589808664 6.5089524537324905 11.290511734900063 113.40439720689756\nleaf_count=157517 16705 20742 2104 16552 401 33 149 8349 185 106 19 207 79 73 236 61 101 208 50 95 237 110 224 71 16 23 52 53 124 20820\ninternal_value=4.31758e-05 0.00294836 -0.0206181 0.00381872 -0.0101822 0.00505136 0.00193723 0.000763151 0.00176028 0.0164301 -0.00323781 0.00936397 0.0100349 0.00897718 0.00947045 0.00710533 0.0106862 0.00244101 0.0149738 0.00843862 0.0146194 0.0216066 -0.00624512 -0.0170395 -0.0211768 -0.0208617 -0.0121559 0.00966453 0.0178838 0.000497713\ninternal_weight=730.466 641.169 89.2976 615.19 49.1914 565.998 329.509 305.789 294.888 23.7197 7.81634 236.489 229.803 217.706 214.975 169.243 133.607 46.2457 87.3609 45.1485 37.7719 42.2124 35.6361 25.979 7.37663 6.18251 26.155 9.48114 45.732 272.947\ninternal_count=245702 86081 159621 69305 21143 48162 46180 45906 37557 274 125 1982 1775 1696 1623 1263 873 332 541 381 332 160 390 16776 49 124 276 114 360 37372\nis_linear=0\nshrinkage=0.03\n\n\nTree=119\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 4 0 7 5 2 5 5 6 12 4 7 0 12 4 4 4 11 12 4 21 13 17 13 4 6 18 2 0 13\nsplit_gain=32.7558 20.9544 27.5354 9.05982 9.78389 8.65114 8.01343 9.56685 7.80742 8.5841 9.83694 9.02876 7.53586 6.21176 6.04405 7.68046 7.0903 5.84985 7.27727 5.78163 9.6648 17.8885 5.12984 5.10359 5.06567 10.2675 4.90901 4.27737 4.77343 4.66171\nthreshold=0.28904431203898151 7.0112304687500009 0.96046990436194013 0.89599609375000011 0.89599609375000011 -25.182852745056149 1.4980468750000002 0.89599609375000011 4.3378906250000009 0.24770387456944362 2.9858398437500004 3.6503906250000004 0.45911736242885776 0.59482941130694844 4.9616699218750009 1.0693359375000002 0.84667968750000011 8.8817841970012543e-16 0.79622554818162328 5.5515136718750009 2.5000000000000004 -0.051402772639976051 1.6895709195314625 -0.041888724317900632 7.0112304687500009 1.7006835937500002 0.89587973461402759 28.11904335021973 0.010815665348252143 -0.56995612852957833\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 13 4 22 -3 7 -5 9 12 -11 23 -8 -1 15 16 -4 -7 -19 20 21 -14 -2 -12 -21 -26 -17 28 -6 -30\nright_child=3 5 14 6 27 17 8 -9 -10 10 11 -13 19 -15 -16 26 -18 18 -20 24 -22 -23 -24 -25 25 -27 -28 -29 29 -31\nleaf_value=-0.027895063900150573 0.0054415875046261086 -0.026196565232844662 -0.022850762422147328 -0.033442815468882336 0.021780156433787511 0.017429148944572394 -0.011483787997692346 0.016951541130544107 0.02615139319124642 -0.038863307159066218 -0.029168451511096354 -0.03911364482685703 -0.029566382103423296 0.016250973958734351 0.02068143192981993 0.0066872677316968014 0.017720617627443858 -0.019441927534904063 0.019046922590630781 0.026527976748060057 0.0082374771115808643 0.01587386016689709 -0.019940769530456755 0.0097420417411400468 -0.023970800030790256 0.019100867787870849 -0.021446344569963277 0.0023790615879328916 0.00042761304805982606 -0.020908166350737955\nleaf_weight=87.18085599164624 112.41385964338406 5.8225829077528024 4.0433728585048838 2.6368374137600759 2.3172359991222029 31.084640017084894 20.876772837698809 78.073598737581051 12.692162977473343 7.4408188229645011 2.4936134888266706 3.9661897131591095 17.531782701931661 2.349478060641558 9.405027023953151 6.9373688991763656 16.702448790303606 10.121423811076967 6.3717016512528053 21.827371543331537 112.35296599526919 12.619983143158608 6.6756287390453499 22.992097551206829 6.291599306510764 14.893856289330868 21.077461785225751 18.458063283571391 11.812076420974334 40.315288638688799\nleaf_count=182520 7791 12698 3368 370 587 1987 6767 1667 111 503 419 106 1161 64 261 283 1556 691 58 345 1906 162 2553 251 110 129 6086 1402 2889 7441\ninternal_value=-0.000152731 -0.010428 -0.0162219 0.00376803 -0.00142862 0.00560044 0.0067268 0.014916 0.00409674 0.00284382 -0.0103042 -0.00204969 0.00523237 -0.0265699 -0.000114209 -0.00446031 0.00912605 0.00995094 -0.00433391 0.00716496 0.00419517 -0.0106484 0.00386456 0.00497735 0.0166852 0.00580565 -0.0144892 -0.0100106 -0.0141781 -0.0161564\ninternal_weight=729.778 201.096 147.696 528.682 191.992 53.4003 336.69 80.7104 255.979 243.287 36.8927 29.4519 206.394 89.5303 58.1657 48.7607 20.7458 47.5778 16.4931 185.518 142.505 30.1518 119.089 25.4857 43.0128 21.1855 28.0148 72.9027 54.4446 52.1274\ninternal_count=246242 209572 194138 36670 22663 15434 14007 2037 11970 11859 1279 776 10580 182584 11554 11293 4924 2736 749 3813 3229 1323 10344 670 584 239 6369 12319 10917 10330\nis_linear=0\nshrinkage=0.03\n\n\nTree=120\nnum_leaves=31\nnum_cat=0\nsplit_feature=18 1 8 4 1 17 8 2 13 11 10 22 18 2 17 2 14 1 2 4 4 5 4 4 18 1 2 4 18 14\nsplit_gain=34.4977 19.7926 14.782 13.2193 10.9405 7.29696 9.61404 7.43853 8.16763 7.25785 6.7066 6.68119 10.2027 5.73985 5.43945 5.20737 4.98417 4.58111 4.17949 4.06838 5.92884 5.83948 5.94357 4.83678 4.14788 4.11956 3.96779 3.92272 6.81052 5.46287\nthreshold=3.7494348669884658 -0.041910702480567281 1.0000000180025095e-35 0.84667968750000011 1.0240454605967615 3.5426598534814651 1.0000000180025095e-35 22.129880905151371 -0.39543742724656211 0.77813183983917178 1.7005986908310777 1.0000000180025095e-35 2.740319461670996 1.2144355773925783 1.5107988686755358 18.821963310241703 0.72685412594425991 4.0016611463566001 1.0000000180025095e-35 1.6286621093750002 1.6572265625000002 1.7006835937500002 2.1186523437500004 1.0693359375000002 2.1383330595080281 1.9634439907166827 53.951957702636726 4.9616699218750009 3.1984648276080736 0.57616929401185935\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 10 4 -1 -3 7 -7 8 11 14 -2 12 -5 18 16 -15 -6 -16 -13 23 -21 22 24 -9 -22 -23 -24 28 29 -10\nright_child=1 2 -4 5 9 6 -8 19 27 -11 -12 13 -14 15 17 -17 -18 -19 -20 20 21 25 26 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.020635744435398562 -0.030464268671357337 -0.026365717861704219 0.011404705859533576 -0.031048190840477559 0.00019590126338362237 -0.031500081643849806 0.018244669963341165 0.0093155280957230024 0.0025305606104844064 0.02194722094484676 0.0030938464654226686 0.0082731650964315041 0.0095145837295639649 0.02526538332415515 -0.027037784868679984 -0.020141445771251085 -0.023499325437223561 0.0062273023880348825 -0.031989116162853382 0.037455244346137097 0.01641624418901971 0.0068121596560438051 -0.040906267466459777 -0.015133494138262146 -0.025940177341466249 0.019801632711676082 0.0058351769926045126 0.0090371666937432617 -0.021730603232254814 -0.011084383650883401\nleaf_weight=17.691459456193115 44.289723620146106 27.526962182859279 36.818725244065718 8.1694711955933617 56.898262728191185 9.5598379516122787 4.435670760809443 21.857563599405697 107.66331744384661 7.2450280312914392 5.8812351890665004 85.861222808933235 13.42299746822391 29.626451389034631 16.997391775763393 1.7285265978134692 8.2914370809958182 4.2837071975227436 1.4181116127292615 7.9942976135353083 9.6959843615477457 38.362635404126195 3.8963254309492177 9.6960355142000463 1.6434178815252369 55.676095454196911 2.1170553782867501 28.589334306569071 14.532426421312264 34.09420531088108\nleaf_count=11421 120699 58794 3490 1044 8346 5662 96 889 3875 75 750 3477 2431 508 7364 42 4519 119 59 68 172 1908 157 1658 517 474 135 617 4869 1787\ninternal_value=-6.47277e-05 -0.0103187 -0.00503311 0.00415266 -0.0100774 0.00508778 -0.0155689 0.00572644 0.00315384 -0.00506516 -0.0265469 0.0085864 -0.00652725 0.0113688 -0.00755345 0.0222171 -0.00312738 -0.0203628 0.00726403 0.0112103 0.0137244 0.0117032 -0.00409119 0.00156855 0.0086728 0.0146301 -0.0259649 -0.000990663 -0.0028708 -0.000798687\ninternal_weight=715.965 208.232 158.062 507.732 121.243 490.041 13.9955 476.045 325.106 93.7158 50.171 140.227 21.5925 118.634 86.4708 31.355 65.1897 21.2811 87.2793 150.939 119.386 111.392 17.3528 31.5536 11.3394 94.0387 6.01338 184.879 156.29 141.758\ninternal_count=246022 204156 82707 41866 79217 30445 5758 24687 18709 20423 121449 7561 3475 4086 20348 550 12865 7483 3536 5978 3431 3363 981 2547 689 2382 292 11148 10531 5662\nis_linear=0\nshrinkage=0.03\n\n\nTree=121\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 11 17 2 4 4 17 2 0 18 3 18 10 4 17 14 4 3 18 21 2 2 2 18 5 2 9 19 3 21\nsplit_gain=34.1532 21.6449 12.0852 11.7745 8.7429 8.07926 6.15835 5.90625 6.94471 5.76829 4.69175 5.25184 4.40794 6.81964 7.48541 4.77057 9.34175 5.69661 9.47698 7.09726 5.25347 4.26436 7.05626 5.59513 5.46051 4.77865 5.6332 4.25997 4.1988 5.51047\nthreshold=0.52258841453381233 0.77813183983917178 4.4301796584464421 -29.210144996643063 0.84667968750000011 2.7826617956161503 2.4775242203633088 -5.1300392150878897 1.3771184689138856 4.2121001622835372 0.48555671035752257 4.9593358439305844 2.249904835165133 11.379394531250002 3.8863394171308765 0.72273155396328503 7.1799265146255502 0.50207005315878861 2.4414009612931857 3.5000000000000004 -1.3165187835693357 -35.542255401611321 -23.591764450073239 1.7005986908310777 1.4980468750000002 -7.1578526496887198 1.0000000180025095e-35 245.50000000000003 0.048385965359893195 6.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 4 -1 -2 10 -7 8 9 21 -5 -12 13 14 15 17 -17 20 -19 -20 27 -6 23 -23 -24 26 -26 -9 -22 -30\nright_child=2 -3 -4 5 7 6 -8 12 -10 -11 11 -13 -14 -15 -16 16 -18 18 19 -21 28 22 24 -25 25 -27 -28 -29 29 -31\nleaf_value=-0.02741278155364409 -0.013224501478253367 0.026757182532949009 -0.027987719095213397 -0.023822146679961222 -0.018130883655391734 0.0030826472673551558 -0.018513400691001932 -0.015938493311721882 0.0089666026133355028 0.019847248052645147 -0.019978320037435549 0.019266943507565179 -0.018858846467912888 0.027353064706906068 -0.035475200979201406 0.0046875612676923785 -0.02914173060590387 0.020142270950563524 0.008244115756828773 -0.029837506032788463 0.023739481333471535 -0.019115761267129831 -0.00086097561806288217 0.013042434797910847 -0.0354989020360214 0.0073372156567773107 0.0037515138280529098 0.028907593155746772 0.004461328469521372 -0.017166733670766609\nleaf_weight=44.338617296976736 22.628336783756822 11.438615727005525 9.4187929132094705 34.529982037252921 18.556295121081348 48.939427322467964 14.610871266828324 1.578227878686447 60.898202993994346 7.7871348607586688 5.240258657751836 5.039805688909837 4.6211972389137363 14.958236395847051 2.6956119942587966 84.611351681582164 7.0395438210689454 76.713972627767362 13.726426608714975 5.2988474350422612 7.8900829116901141 4.9038249010918653 29.066738361161697 28.844075662906107 13.780350370347156 4.0889522038851274 4.0032520873501189 12.005089249512819 104.62712421964534 10.79905925218918\nleaf_count=92484 15447 124 11083 60390 5863 15757 9365 359 681 1454 1140 442 102 113 733 8218 167 315 227 131 1047 68 4379 3825 2948 460 122 692 7559 428\ninternal_value=0.000464863 -0.0115044 0.00404774 -0.014526 0.00465595 -0.00905051 -0.00209239 0.00546025 0.000906517 -0.00355486 -0.0186575 -0.000735926 0.00771022 0.00812324 0.00716283 0.00762522 0.00180271 0.00990154 0.0154932 -0.00332258 0.00590709 -0.00549343 -0.00254508 0.00795396 -0.00952005 -0.0204865 -0.0269339 0.0229614 0.00385896 0.00230778\ninternal_weight=714.678 164.138 550.541 152.699 541.122 108.36 63.5503 518.494 171.929 111.031 44.81 10.2801 346.565 341.944 326.985 324.29 91.6509 232.639 95.7392 19.0253 136.9 103.243 84.6872 33.7479 50.9393 21.8726 17.7836 13.5833 123.316 115.426\ninternal_count=246123 179702 66421 179578 55338 87094 25122 39891 19800 19119 61972 1582 20091 19989 19876 19143 8385 10758 673 358 10085 17665 11802 3893 7909 3530 3070 1051 9034 7987\nis_linear=0\nshrinkage=0.03\n\n\nTree=122\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 4 6 17 1 0 10 0 4 1 5 14 12 2 2 7 3 6 2 3 18 0 2 3 0 18 4 13 14\nsplit_gain=41.7073 16.5627 13.6453 10.7784 19.6266 10.3925 8.95274 7.05944 6.15979 6.15832 8.28438 6.39738 6.93986 5.8363 5.66872 5.22111 4.98808 5.08996 3.67664 3.57097 3.37776 4.46393 5.11681 3.1957 3.15379 4.69448 5.14203 3.32572 5.29273 4.29321\nthreshold=0.18336881478606726 -39.283819198608391 0.84667968750000011 3.4492187500000004 3.9281938047451082 4.0016611463566001 1.4460303622695712 1.0000000180025095e-35 0.96046990436194013 3.7336425781250004 1.4475910192202701 2.1386718750000004 0.66452579377975118 0.90934753668501156 30.514518737792972 -41.14759445190429 3.6503906250000004 0.30295287721642777 2.2500000000000004 68.466045379638686 0.47260169401963864 1.4978661367769956 1.3771184689138856 15.769127845764162 0.58795853567030554 1.1210156985015984 3.5694334999727624 1.0693359375000002 -0.44041049690472206 0.14925710185700605\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=6 8 -3 4 5 9 13 -6 -2 10 20 19 -13 -1 -5 -10 18 -18 -11 -12 21 22 -4 -14 27 26 -26 29 -29 -7\nright_child=1 2 3 14 7 24 -8 -9 15 16 11 12 23 -15 -16 -17 17 -19 -20 -21 -22 -23 -24 -25 25 -27 -28 28 -30 -31\nleaf_value=-0.025255374054300051 -0.025893228462721341 -0.020341959043010421 0.023331255577178536 0.011836082039525801 -0.035719601644849663 -0.010851535898370663 0.007870844879459573 0.010717540108260645 0.0082985634947406912 0.010748293239334756 0.0037514966175462459 -0.02700084965062197 0.018690347822685437 0.020535971173436928 0.040262332476969635 -0.034590554889128745 0.011195302284229375 -0.034563584771743872 -0.014049004758126222 -0.023187550604331413 0.003393430132056131 -0.026088371523311223 -0.02786085959096183 -0.017387369372154862 -0.017999744864601534 0.026334143414597711 0.023402059446868569 -0.019668568559893549 0.0082255885475950811 0.023079149051671199\nleaf_weight=69.999901030441833 23.158326165601466 19.529185542531195 3.5359980562643676 19.524515935074309 14.043614449819872 3.3554183649830547 8.5542120809113822 3.1745288881938896 11.367279402678831 109.25763210750301 228.95850280710147 12.551865126479244 5.7238744048954668 1.8838533870875824 9.3920763097703439 2.2198323009652077 3.2748438251146572 3.893187797133578 4.821413315657991 3.5305570421041912 7.2037757129728552 18.820144327299204 1.8824791682709472 2.3752535480889492 4.8923842404037696 26.09985501691699 3.9850966520607471 6.0506220269016913 47.177555115660653 20.81518501136452\nleaf_count=158818 20343 14031 157 277 8288 19 1045 98 426 6968 7145 604 321 16 45 83 124 149 223 349 1514 24074 299 125 27 96 15 54 366 122\ninternal_value=-0.000180562 0.00246446 0.00370377 0.00457039 0.0035839 0.00465153 -0.0203677 -0.0270396 -0.0168042 0.00240155 -1.75264e-05 0.00180495 -0.0138066 -0.02379 0.0221061 -0.000599511 0.00803314 -0.0148467 0.00958729 0.00324485 -0.0142935 -0.0194762 0.00397636 0.00736167 0.0126841 0.0201308 0.00107316 0.00910507 0.00475054 0.0181248\ninternal_weight=701.053 620.615 583.87 564.34 535.424 518.206 80.438 17.2181 36.7454 405.83 284.582 253.14 20.651 71.8838 28.9166 13.5871 121.247 7.16803 114.079 232.489 31.4424 24.2386 5.41848 8.09913 112.376 34.9773 8.87748 77.3988 53.2282 24.1706\ninternal_count=246221 86342 65490 51459 51137 42751 159879 8386 20852 42052 34588 8544 1050 158834 322 509 7464 273 7191 7494 26044 24530 456 446 699 138 42 561 420 141\nis_linear=0\nshrinkage=0.03\n\n\nTree=123\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 16 6 3 7 0 7 7 14 0 4 0 4 12 4 13 5 12 5 14 9 4 17 10 21 12 5 4 7 17\nsplit_gain=28.5582 15.626 11.0408 12.5304 9.91156 9.52337 8.89957 7.55186 7.64559 6.79432 6.2306 5.06568 5.07892 6.37308 4.67239 4.53827 4.51208 4.21052 3.91009 3.6532 3.57853 3.42419 4.75262 3.513 3.2142 5.06191 5.44061 4.33511 5.77987 4.27194\nthreshold=0.17302227103784359 1.0000000180025095e-35 3.3837890625000004 0.37248792739977959 3.6503906250000004 0.52258841453381233 2.5244140625000004 2.6015625000000004 0.2621461414509263 0.13844441961022361 0.84667968750000011 1.3771184689138856 1.6286621093750002 0.63959556986474675 2.1433105468750004 -0.081478796350041446 5.7089843750000009 0.88192242588002812 6.0488281250000009 0.46513657297606553 1.0000000180025095e-35 3.2895507812500004 1.2596354777617942 1.0000000180025095e-35 5.5000000000000009 0.1157423781038089 4.4472656250000009 1.0693359375000002 0.89599609375000011 0.13727533581160231\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 7 3 5 10 6 -2 -1 -9 -8 -5 17 -13 -14 -15 18 -4 20 -3 -6 24 22 -7 -23 27 26 -26 -12 -29 -30\nright_child=2 15 16 4 19 21 9 8 -10 -11 11 12 13 14 -16 -17 -18 -19 -20 -21 -22 23 -24 -25 25 -27 -28 28 29 -31\nleaf_value=-0.026433255921876741 -0.027704450971832151 -0.021415543517450231 0.027416455475988623 -0.013125208313512585 0.012798009784776987 -0.0001260505671879125 -0.020815117429945584 0.019116188393914789 -0.025065627444763386 0.023098012689666648 0.012080712859193418 0.0015215123774699739 0.015401666912960311 0.012318571964759856 -0.027624840444216054 0.009448275060874034 -0.00064116191299208038 0.020268431935458735 0.014870884463855193 -0.041132786284539327 0.021472582061565312 0.011973576117829478 -0.018218281804022934 -0.014587048841249143 -0.0075136013898113646 -0.034867496207376447 0.02119021376267869 -0.010035394510369505 -0.014396943971687481 0.0061883114711762171\nleaf_weight=55.898112331533412 25.156300410546464 9.0972629803945875 22.355880603528934 14.371960907956234 0.91570643009617825 62.320721888316257 4.6555904418346454 6.9679245271036034 5.3222399157421023 6.261972151689406 40.283878661255585 43.324257519212551 113.92318880843231 3.9991270974278432 4.8435079745249823 25.010379016115621 6.5629197683301745 11.868628178257493 2.8351923984955638 4.6147358733287547 7.7052031631465061 26.204697799432328 15.539181446183646 4.4845238541893186 29.050332439983322 3.7696492514587581 6.4147080299735526 31.819324576805229 9.039168171817435 98.260475333328941\nleaf_count=129356 42536 6385 281 3565 9 13860 4925 3007 15162 454 1124 363 631 25 62 1779 172 87 208 172 215 1498 7563 295 2670 229 131 4838 280 4346\ninternal_value=0.00093053 -0.0134137 0.00347157 0.00253253 0.00513 -0.00506921 -0.0181969 -0.0218268 -0.000463092 0.00419678 0.00568949 0.00638904 0.0103992 0.0134946 -0.010144 0.002281 0.0212404 0.00356121 -0.0123106 -0.0316108 0.00259984 -0.00058022 -0.00392211 0.00775451 0.00184 -0.00584005 -0.00188309 0.00354266 0.00100177 0.00433724\ninternal_weight=702.877 105.131 597.746 568.827 424.204 144.623 36.0739 68.1883 12.2902 10.9176 418.673 404.301 166.09 122.766 8.84264 36.9428 28.9188 238.211 11.9325 5.53044 226.343 108.549 77.8599 30.6892 218.638 39.2347 35.465 179.403 139.119 107.3\ninternal_count=246228 155897 90331 89878 18747 71131 47915 147525 18169 5379 18566 15001 1081 718 87 8372 453 13920 6593 181 13833 23216 21423 1793 13618 3030 2801 10588 9464 4626\nis_linear=0\nshrinkage=0.03\n\n\nTree=124\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 17 2 1 1 18 1 7 9 19 6 2 7 7 2 2 3 5 7 7 5 14 3 17 1 17 0 7 1\nsplit_gain=30.6583 12.1685 12.0807 10.6361 8.46966 8.29724 6.36279 6.2887 5.96871 6.24551 5.19241 4.66505 4.88951 6.51804 6.05683 5.81719 5.81296 4.72858 5.20666 6.26247 4.57267 4.29426 4.29185 4.27488 4.04627 3.946 4.02028 3.78553 3.72118 3.71047\nthreshold=0.24961073363565869 0.89587973461402759 3.9281938047451082 -34.252307891845696 0.28463941411212784 3.3033179733375269 3.4811217321331038 2.2520445226790948 3.2382812500000004 1.0000000180025095e-35 263.50000000000006 2.0131835937500004 8.8705892562866229 1.2426757812500002 2.6015625000000004 11.076014518737795 14.537487030029299 0.27378847774872456 2.5244140625000004 0.89599609375000011 2.0131835937500004 5.9863281250000009 0.043928974430191954 0.016076142967085191 0.86664248603069383 1.4475910192202701 0.068176441062887241 0.96046990436194013 1.0000000180025095e-35 1.4866206336930319\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 7 -4 8 23 -2 9 11 27 12 13 -5 -15 -14 -17 18 -18 -20 -7 -10 -6 -1 -16 -19 -27 -9 29 -28\nright_child=2 -3 4 5 22 20 -8 10 21 -11 -12 -13 15 14 24 16 17 25 19 -21 -22 -23 -24 -25 -26 26 28 -29 -30 -31\nleaf_value=0.014669887149423098 -0.027383310971373653 0.0054726404429474929 0.02502642608445908 -0.0019404077211582751 0.0018713517915386491 0.0068670375726663053 -0.028584584310286217 -0.024793652737065942 0.013835160309942649 0.016874583564348435 0.012316732342095172 -0.031266859126116743 0.019534451107535281 -0.026539656801409406 0.015889090739179764 -0.020209136934347349 -0.023813772983129505 -0.0097731695930600698 0.027586537660387846 -0.022083069181041089 0.021670246397668302 -0.022563750117944138 -0.034694219244270338 -0.015956907726935039 -0.026265387550065657 -0.027101042420789454 0.021662359287374512 5.8931295279861274e-05 0.010249880707856214 -0.025933190119331474\nleaf_weight=4.8680604610781328 14.049370095290213 16.128240583173465 2.3840868260886081 116.82687579427511 3.4367844070220626 164.70340607524849 50.286202292827511 9.9362182158802188 38.942475028554327 16.231288182360004 10.968369920534313 3.7307567974203257 15.385769999742477 19.606848728213667 5.9997956039660485 10.113411814785648 10.6647242466388 13.047525372327071 4.4813125531363758 2.9774960496579297 20.610858664498664 2.2578358905011546 14.350533121765691 13.2573670986294 1.9130330442421835 1.9907395606860507 1.1937846496293776 11.120033090352079 78.476548205624567 4.2374871358188102\nleaf_count=1255 17027 2243 590 24202 1251 1306 148929 968 808 471 201 284 1265 5123 323 2046 5031 4666 108 273 144 101 11743 13419 432 60 40 140 1327 652\ninternal_value=-0.000893162 -0.0176956 0.00150124 0.0023107 -0.0207629 0.0035113 -0.0231577 -0.0113677 0.000774053 -0.000678507 -0.0037252 -0.00171327 -0.00122172 -0.00509694 -0.0179592 0.00270999 0.000538164 0.00248646 -0.0112116 0.0074922 0.00862094 0.011353 -0.0279056 -0.00739433 0.00389421 0.00508256 0.00742556 -0.0122611 0.00855425 -0.0137301\ninternal_weight=684.177 84.5399 599.637 579.466 20.1714 533.392 68.4116 46.074 348.078 306.877 32.0246 290.646 286.915 144.347 27.5197 142.569 127.183 117.07 18.1235 7.45881 185.314 41.2003 17.7873 18.1254 7.91283 98.9461 85.8986 21.0563 83.9078 5.43127\ninternal_count=246428 165846 80582 66998 13584 48662 163603 18336 47212 46303 1309 45832 45548 30080 5878 15468 14203 12157 5412 381 1450 909 12994 14674 755 6745 2079 1108 2019 692\nis_linear=0\nshrinkage=0.03\n\n\nTree=125\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 19 17 18 4 1 2 3 2 18 2 2 1 12 1 12 3 1 1 1 6 4 1 4 19 18 1 17 21\nsplit_gain=36.3042 17.8268 11.9049 11.4459 10.886 7.62433 6.199 5.84091 6.1872 5.15983 10.4941 5.12837 4.8956 4.51959 4.14342 4.01516 3.94855 3.63793 4.5961 4.07244 7.14831 4.6069 6.33097 4.64215 4.87338 4.97958 7.29845 6.02682 4.76955 4.01294\nthreshold=0.98142480879535021 5.7263183593750009 240.50000000000003 3.9281938047451082 2.9174053685313028 0.84667968750000011 4.0016611463566001 -10.988907814025877 0.37248792739977959 -22.378742218017575 4.0163424379838117 -34.692844390869134 -18.836229324340817 1.6575619461558879 0.30400123877355156 -0.79535375801325292 0.66389400333341531 0.58795853567030554 1.7090960825073716 1.075454090076706 1.211689956917019 3.0683593750000004 7.6279296875000009 1.2421711908324335 5.5515136718750009 242.50000000000003 0.89587973461402759 2.3713805701640611 1.0796935653640076 6.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 15 5 -1 14 7 8 -7 13 12 -8 -11 -10 -2 -3 -6 19 -19 -9 -21 22 23 -22 25 26 28 -28 -25 -24\nright_child=3 2 -4 -5 16 6 11 17 9 10 -12 -13 -14 -15 -16 -17 -18 18 -20 20 21 -23 29 24 -26 -27 27 -29 -30 -31\nleaf_value=-0.0035029634190337595 -0.021666862277740492 -0.025424721611688874 -0.023779476800511755 -0.022482216660237826 -0.030331981720616196 -0.016130998634916443 -0.014024515817184764 0.020086557059874482 -0.023237582958181405 -0.042592113746231154 0.016993786111237186 0.012996655739551848 -0.0096109867393819222 0.0085643917566468782 0.01498039101217773 0.0095494457137068924 0.011539035478621272 -0.012187055817024405 0.013840621932941376 -0.032271775746128536 0.027006520213132504 0.021798681175919751 -0.035901678115416197 -0.011458283203965129 0.014804933499911977 0.0035879289992677582 -0.032487427301455547 -0.0040529915271857331 0.029700469891181556 0.010886851174929393\nleaf_weight=20.671898188396881 13.376622879075514 2.2235045063789576 13.825773925078463 13.282696968673916 56.325906427247901 22.156721509683848 5.9080748750129706 12.090427321760215 3.3816244020126751 5.8288061353378025 8.3357657591404841 109.34418587014079 11.994721656752519 54.504559788154438 2.6882484027009914 41.492416066525038 1.6122402909968503 5.9473075542482539 66.753169373201672 4.1122223890342857 6.3529124062479232 10.894797510001807 5.0308801289647809 3.5532964100129885 24.509138731169514 119.6906632106984 11.22394922449894 15.607372642349217 5.2434589369222522 1.7388028101995585\nleaf_count=10774 5829 2703 31334 5023 161399 7766 94 1147 1017 119 635 712 323 1646 46 5183 50 371 515 1487 310 69 100 27 312 4391 2057 532 22 15\ninternal_value=0.000574692 -0.0132212 -0.000490707 0.0040483 -0.0224165 0.00475393 0.00539953 0.00361202 -0.0023982 0.00136926 -0.0094662 0.0115028 -0.0220865 0.0063435 -0.0150159 0.00724177 -0.0289908 0.00579313 0.0115749 0.00383919 0.00280297 0.00366463 0.00253359 0.00362267 0.00265179 0.000651592 -0.00904152 -0.0166875 0.0136025 -0.0240295\ninternal_weight=679.702 136.152 57.5417 543.55 78.61 530.268 514.203 398.951 106.202 84.0455 26.1593 115.252 17.8235 57.8862 16.0649 43.7159 57.9381 292.748 72.7005 220.048 207.957 203.845 192.95 186.181 179.828 155.319 35.6281 26.8313 8.79676 6.76968\ninternal_count=246008 211443 39220 34565 172223 29542 23667 22861 11506 3740 1077 806 442 2663 5875 7886 161449 11355 886 10469 9322 7835 7766 7651 7341 7029 2638 2589 49 115\nis_linear=0\nshrinkage=0.03\n\n\nTree=126\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 2 7 3 12 3 17 5 7 5 2 21 5 0 18 7 2 13 14 17 5 0 7 13 2 2 2 13 18 2\nsplit_gain=30.1626 17.1673 11.6616 9.54685 8.49345 8.36386 7.17675 5.86488 15.439 6.50277 5.47344 5.44599 5.34086 5.31487 5.83515 6.88767 5.7103 5.05862 4.99035 4.69343 5.06935 4.59486 5.96214 4.39629 4.33203 4.13417 3.95722 3.69127 4.15097 3.59184\nthreshold=1.2661233611507157 -34.692844390869134 1.0000000180025095e-35 0.47260169401963864 0.88192242588002812 0.48555671035752257 3.9653415636860969 0.89599609375000011 0.89599609375000011 1.2426757812500002 27.147908210754398 5.5000000000000009 1.7006835937500002 1.0212340148387515 2.249904835165133 1.2426757812500002 44.569120407104499 -0.81210969460121862 0.47889823060390885 0.29595754201755942 1.0000000180025095e-35 0.29549219484135919 3.4169921875000004 -0.31142306388867069 31.708583831787113 34.417465209960945 -34.252307891845696 -0.044585747678852421 5.5664324555418956 -49.116760253906243\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 19 -2 6 -1 7 11 10 -10 26 24 29 16 17 -16 21 -15 23 -3 -21 22 27 -19 -4 -12 -9 -11 -29 -7\nright_child=3 2 4 -5 -6 12 -8 8 9 13 25 -13 -14 14 15 -17 -18 18 -20 20 -22 -23 -24 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.027289331781053465 0.0011545916402665238 -0.0020651309999597245 0.014655815282632378 0.01266607244603732 0.02071996128236964 0.019307529548427349 -0.020457776870476604 0.016065488784391211 0.016914536883490119 -0.028204972445825306 0.017645290247160435 -0.017022222942287855 -0.0255448124652016 0.01653328546157002 -0.02673147398428748 0.009361921126493181 0.024019738898198657 0.021409975859111117 -0.042565312872451848 0.018649629263845415 -0.030697285387600342 0.0034564948169002034 0.0079562456942719201 -0.02537818385545508 -0.025594705896214077 -0.020138595473068951 -0.029364697758285308 -0.028097045839489819 0.0099982488043720494 -0.01135205740185717\nleaf_weight=41.464999197984071 113.229530134573 9.9351509172429342 40.138113098575559 151.4079103426775 14.672001052182166 7.9892025740991821 16.715796580985625 1.2318158481211892 18.908228154381504 12.633781639206061 4.7365204159577825 5.5911665998391973 5.267700090102152 1.7966591304866621 5.5735738492730915 16.235914217591926 8.5415320778192818 2.2508543074363843 8.9303847764385846 1.3961300698283619 20.819584235408911 89.988047970400658 11.173364615809986 3.1322242461901615 1.6625327516012465 3.7768281640892383 26.200072546620504 2.9340322241369012 6.5086254996640491 4.7340716424514548\nleaf_count=92711 5595 7914 7762 1103 129 259 22541 178 1524 15534 1144 2208 356 19 275 609 868 24 261 112 37223 12317 1392 123 1044 2321 26874 1912 1615 273\ninternal_value=-5.70142e-05 -0.00530375 -0.00264345 0.00776358 -0.000794461 -0.0200226 -0.00195575 -0.000747689 -0.00277828 0.000691935 -0.0205126 0.00873225 -0.00271926 -0.00120412 -0.0109477 -0.000629565 0.00166093 -0.0240629 -0.0299056 -0.0195379 -0.0269264 -6.96086e-05 -0.00943502 -0.00552518 0.0124943 0.000528096 -0.0268314 -0.0182005 -0.00339584 0.00790378\ninternal_weight=659.576 394.939 335.483 264.637 303.332 59.456 288.66 271.944 224.552 188.607 35.9452 47.3918 17.991 169.699 37.9196 21.8095 131.779 16.1101 14.3135 32.1509 22.2157 123.238 33.2498 5.38308 41.8006 8.51335 27.4319 22.0764 9.44266 12.7233\ninternal_count=246220 239522 145923 6698 100674 93599 100545 78004 66990 36473 30517 11014 888 34949 1311 884 33638 427 408 45249 37335 32770 20453 147 8806 3465 27052 19061 3527 532\nis_linear=0\nshrinkage=0.03\n\n\nTree=127\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 9 3 2 7 11 11 17 1 6 4 12 22 6 4 21 10 7 17 4 19 19 2 1 3 14 3 3 3\nsplit_gain=27.6004 11.7232 8.40192 10.0368 14.2289 8.24093 8.03519 7.1475 6.73217 6.205 6.84185 6.72929 5.18757 5.6466 5.1634 4.92165 5.52733 4.56326 4.11796 4.02342 3.3994 3.30045 3.25008 3.18569 3.11944 4.5207 4.17569 4.01901 3.52805 3.48844\nthreshold=0.18336881478606726 1.7005986908310777 1.0000000180025095e-35 0.47260169401963864 -26.006598472595211 3.2382812500000004 0.11347059467310934 0.86534941175790137 1.6722174619049508 1.7090960825073716 1.8691406250000002 1.0693359375000002 0.027767680665956421 1.0000000180025095e-35 4.7578125000000009 4.1025390625000009 2.5000000000000004 3.6242520361853052 2.6015625000000004 0.12494407617242172 0.84667968750000011 263.50000000000006 48.500000000000007 -42.103004455566399 2.9477339485081675 0.42638606281916264 0.39695441677226773 0.12808136320582797 0.22136347059745312 0.41780940659066851\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 4 21 8 22 14 12 10 11 20 24 18 -8 -10 -17 -3 -14 -13 -5 -2 -4 -23 27 26 -26 -6 -29 -30\nright_child=2 17 6 9 5 -7 7 -9 15 -11 -12 19 13 -15 -16 16 -18 -19 -20 -21 -22 23 -24 -25 25 -27 -28 28 29 -31\nleaf_value=-0.022527088942763493 -0.025892635164537933 0.018255829936182123 -0.019022141284720705 -0.021745972216151283 0.010492798253675068 0.018281051218825711 -0.036460233768875236 0.01902190993556261 -0.024150775485149011 0.0061922586050189837 0.01637316707871336 0.0032926308101451783 0.020651426636720729 -0.019148245834886703 0.0071423895216456952 0.014653100357724729 -0.02483234691195893 -0.027695733337036191 -0.016282802530147023 -0.032105752345939924 0.02127339823615015 -0.025508236406415745 0.02296309069172206 0.0098383794201700424 0.0022280417927078161 0.013994908919437757 -0.025136148319944204 -0.023171475152891689 -0.0024195783933276521 -0.020235735931952385\nleaf_weight=62.739116807597384 27.182822510663755 9.4846364708719175 1.0192961734137487 1.4067446512926896 11.373061352362127 17.768118950494681 5.5450016603353998 5.9266887588892123 19.141725637109634 246.8996509288263 6.9921100384381125 3.3896102359867681 22.261263466141834 3.8283444215267073 3.4503102275775746 7.7366977546625986 4.0264061794077861 1.4585851083684236 2.3293733547470756 13.521319092265911 4.7767070699628675 2.6384000902871776 33.642190757731441 5.278763271871866 19.94202074222267 20.540429887128994 5.646519055706448 10.806953195657114 61.571232657712244 10.873181623464914\nleaf_count=159318 26175 838 61 736 3436 485 340 65 14901 3977 106 178 538 134 78 720 1093 155 80 2670 227 2443 1062 1092 259 155 239 8133 15205 1713\ninternal_value=-0.000408746 -0.017575 0.00180037 0.000683987 -0.00362894 -0.000734718 0.0134825 -0.00471596 -0.00250596 0.00462147 -0.0080535 -0.0158216 -0.000121374 0.0113434 -0.0206943 -0.0151462 0.000250899 0.010325 0.0166532 -0.0252228 0.00982163 -0.0210738 0.0212296 -0.00348051 -0.00249988 0.00384155 -0.00452857 -0.00559915 -0.00789593 -0.00533272\ninternal_weight=653.197 73.6823 579.515 529.931 252.945 217.845 49.5835 14.922 200.077 276.986 30.0865 23.0944 169.172 28.419 8.99531 30.9048 11.7631 10.9432 24.5906 16.9109 6.18345 35.1 34.6615 7.91716 140.753 46.129 25.5885 94.6244 83.2514 72.4444\ninternal_count=246612 160311 86301 84695 76801 47091 1606 483 46606 7894 3917 3811 29892 752 418 16714 1813 993 618 2848 963 29710 1123 3535 29140 653 498 28487 25051 16918\nis_linear=0\nshrinkage=0.03\n\n\nTree=128\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 9 17 1 0 11 7 0 18 3 7 19 18 3 10 17 4 2 18 4 21 5 1 3 4 19 0 18 3\nsplit_gain=28.0167 13.8654 10.3746 11.5194 8.37282 8.77309 6.39794 5.99891 5.36031 5.21477 4.97606 4.81992 6.30408 7.53106 6.66805 5.60918 4.95268 4.48673 5.58017 6.07203 5.03957 4.99719 4.84178 4.42199 4.76635 5.9235 4.26704 4.17214 4.01326 3.99128\nthreshold=0.18336881478606726 0.89587973461402759 1.0000000180025095e-35 3.9653415636860969 4.0016611463566001 0.52258841453381233 5.5511151231257837e-16 2.5244140625000004 1.4460303622695712 2.1383330595080281 0.50207005315878861 3.1132812500000004 238.50000000000003 0.89587973461402759 0.65908129556435835 1.8688348091416842 1.0055080915597503 3.7858886718750004 47.592866897583015 1.2424533248940002 3.8664550781250004 7.5000000000000009 3.1562500000000004 1.4475910192202701 0.56033297298897644 1.4160156250000002 248.50000000000003 -0.049143614581278028 2.524928003624769 0.47260169401963864\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 3 4 5 7 27 29 -1 -8 -9 12 13 16 17 28 -7 18 19 21 -19 -14 -22 -21 -25 -26 -27 -4 -15 -2\nright_child=2 -3 6 -5 -6 11 9 10 -10 -11 -12 -13 14 15 -16 -17 -18 20 -20 23 22 -23 -24 24 25 26 -28 -29 -30 -31\nleaf_value=-0.026807587807130103 -0.022048756322266869 0.0072836862470677935 -0.0080442662846963147 -0.025875129112797494 0.0093018350474285052 -0.013085533832652924 -0.022314083916189397 0.010449520803168208 0.0039712188062824179 0.010613773684619942 -0.032657189648877948 0.014275672294466793 0.011165020034883899 -0.010476020043015578 0.015547991066873431 0.025930087030537907 0.030276987032656195 0.032896569086138978 -0.023284324922388355 -0.016970699419734999 -0.02534073295797171 -0.015641454490573376 0.010325788160907354 0.0020848744752271259 0.0049613064106188536 0.0058529612090236355 -0.027772784382236099 0.026878865545064971 -0.032421215967459789 -0.0035151826963468408\nleaf_weight=52.449293236884841 30.367429392657868 15.863155771265154 2.9947136207993053 12.91944382855945 105.07540314714424 3.0556928461883199 4.9956235290737849 15.832673358836475 5.3641340308749923 14.59571588016115 1.8268368009012181 24.576387405570131 54.974299105699174 23.390889530259301 30.021424906793982 1.9142387892352406 5.3569965257775038 6.5972266193566638 8.3931362321236502 16.686516352669059 3.16664178692736 6.165184195386245 18.702481129468651 104.73548198867502 12.387915319181049 3.8943991504493196 15.396056705154477 28.147855228220578 11.305316437275904 16.34080284157244\nleaf_count=156556 35417 2087 626 14159 821 42 93 2641 995 491 257 298 586 1148 307 30 39 66 755 14327 322 39 363 5084 361 72 327 409 4249 2949\ninternal_value=9.98304e-05 -0.0172042 0.00230924 0.00106241 0.00177983 -0.000142068 0.0151365 -0.0102343 -0.0239329 0.00154155 0.00447905 0.00173399 0.00074817 -0.00954969 0.00242544 -0.0153922 0.0148098 0.000801052 -0.000652121 0.000338585 0.0118026 0.00825369 0.00427904 -0.0028553 -0.00101773 -0.0110255 -0.0210305 0.0233748 -0.0183345 -0.0157744\ninternal_weight=657.493 73.6766 583.817 533.083 520.163 415.088 50.7339 64.3677 57.8134 19.5913 17.6595 350.72 326.144 45.0231 281.121 36.6104 8.41269 251.099 222.633 214.24 28.4663 61.1395 21.8691 153.1 136.414 31.6784 19.2905 31.1426 34.6962 46.7082\ninternal_count=245916 159638 86278 84659 70500 69679 1619 41264 157551 584 2898 28415 28117 5508 22609 5427 81 22302 21551 20796 751 625 685 20171 5844 760 399 1035 5397 38366\nis_linear=0\nshrinkage=0.03\n\n\nTree=129\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 4 17 2 1 8 3 1 1 2 4 12 2 10 4 21 12 21 5 12 3 3 1 3 19 17 4 2 1\nsplit_gain=25.857 16.0552 9.00283 7.06177 6.50379 6.39259 6.01942 5.87231 5.27423 6.09743 5.21611 4.3991 4.3182 4.1483 3.95715 3.94217 4.97742 3.91817 3.85974 3.83224 3.78146 3.6854 7.32415 3.79053 3.64381 3.53791 3.35812 5.16474 5.12769 4.7523\nthreshold=0.60367420583093068 6.9096149206161508 0.84667968750000011 3.9281938047451082 -20.765821456909176 4.0016611463566001 1.0000000180025095e-35 0.58795853567030554 1.7622051334857016 0.6180242047922454 -6.7439365386962882 1.6286621093750002 8.8817841970012543e-16 -19.986686706542965 0.89587973461402759 1.0693359375000002 6.5000000000000009 0.88192242588002812 7.5000000000000009 6.1093750000000009 0.79622554818162328 0.37248792739977959 0.26802264518526847 1.6109634896136267 0.40160889672495398 209.50000000000003 0.61464819824310435 6.3966703414917001 62.792827606201179 1.4866206336930319\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 20 13 5 -3 6 7 10 9 -9 21 15 -6 -2 -5 16 -10 -14 -17 -8 -1 22 23 -4 -23 -12 29 -28 -29 -27\nright_child=2 4 3 14 12 -7 19 8 11 -11 25 -13 17 -15 -16 18 -18 -19 -20 -21 -22 24 -24 -25 -26 26 27 28 -30 -31\nleaf_value=-0.023722190393333238 0.00049162336724802621 -0.02588001885491592 -0.018621777227106704 -0.02517040972205847 0.016136775041559574 0.010233349262227676 0.013709323292999847 0.026276979322897149 0.010926577765857388 -0.031309925624189862 -0.023589782615063195 0.013392056123751645 -0.019493730244103998 -0.027118360528733773 0.012651539244709499 -0.035304000219521757 -0.024870250117414609 0.01585798652288941 0.01777357966493865 -0.024399008023527423 0.017393134612840179 0.011206438701905746 -0.030738930457762541 0.010947060008773475 -0.0068036609748835615 0.012515884373896221 0.0073497029268889498 -0.027954104860391887 0.020103256221493636 -0.0075579862042279853\nleaf_weight=65.233659628909663 7.7102687274527879 5.1216249509234322 5.4530181081136133 10.812052005050644 19.220861206667909 105.35477213014383 45.406538947754598 1.4083006465225469 18.009151147445671 5.1843726040533511 3.9697525066367225 60.754254877276253 6.8369131879589968 12.259950132236552 2.4905323177517849 3.8046517775510429 3.3064922660123548 3.5023434655740857 0.97051105392165471 1.6098109412123438 1.3949975678697217 11.663582354463868 14.202688205532466 8.8739394697622611 49.953846366981452 12.472452744787004 96.709615366471553 6.4784094911592529 1.9170788545161483 49.660136781167239\nleaf_count=174358 3596 14667 8195 7032 1776 804 2095 15 127 613 724 521 571 7095 68 198 28 28 22 56 24 982 5567 1050 3397 2617 8010 194 8 1586\ninternal_value=-0.000273636 -0.0141046 0.00234039 0.00311138 0.00242419 0.003691 0.00195444 0.000603559 0.00701698 -0.0171678 -0.00170733 0.00902746 0.00801459 -0.0169434 -0.0176998 -0.001255 0.00450811 -0.00717618 -0.0233068 0.0119236 -0.0226169 -0.00750599 -0.0162342 -0.000787897 -0.00327 0.00137173 0.00210095 0.00535061 -0.0160087 -0.00339395\ninternal_weight=641.747 101.31 540.436 520.466 34.6817 507.163 401.809 354.792 93.4377 6.59267 261.355 86.8451 29.5601 19.9702 13.3026 26.0908 21.3156 10.3393 4.77516 47.0163 66.6287 90.1471 28.5296 14.327 61.6174 171.207 167.238 105.105 8.39549 62.1326\ninternal_count=246024 191424 54600 43909 17042 36809 36005 33854 1524 628 32330 896 2375 10691 7100 375 155 599 220 2151 174382 19191 14812 9245 4379 13139 12415 8212 202 4203\nis_linear=0\nshrinkage=0.03\n\n\nTree=130\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 4 3 4 1 5 21 12 10 2 18 0 18 1 2 1 2 10 18 13 2 4 6 6 1 8 13 3 3\nsplit_gain=29.7241 9.36454 9.15338 8.37485 10.5492 8.34402 8.14673 7.47892 6.53542 5.74362 5.66116 5.31743 6.77416 5.23298 7.92322 5.30431 4.98501 4.85797 5.25685 4.85777 4.57081 4.30324 5.00169 4.23634 5.11749 4.094 4.08807 4.01195 4.37828 3.68306\nthreshold=0.15052965826911111 21.425971984863285 0.84667968750000011 0.53650715206675215 3.6923828125000004 1.4475910192202701 1.4980468750000002 3.5000000000000004 0.33344852869475661 2.0126758453675748 30.514518737792972 0.89587973461402759 1.5447487307370646 2.0126758453675748 2.9477339485081675 -26.006598472595211 1.4866206336930319 12.256900787353517 2.0126758453675748 2.6735537653587351 -0.30355183526111729 9.8937015533447283 5.6958007812500009 1.0000000180025095e-35 1.4980468750000002 1.8223781763416411 1.0000000180025095e-35 -0.15532138408334048 0.39370400768236996 0.21010136461729736\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 9 -2 4 5 19 7 29 11 -1 13 20 16 14 -6 -15 -13 21 -19 -4 -8 22 -17 -24 -25 -5 -18 28 -9 -7\nright_child=2 -3 3 25 10 6 8 27 -10 -11 -12 12 -14 15 -16 17 26 18 -20 -21 -22 -23 23 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.028627252973409279 -0.016628392151751326 0.00085378582789349348 -0.0027877847786877906 -0.0047556923188603988 -0.031037067711682305 -0.018206485618786867 -0.035807239215066677 -0.01885667150615282 -0.037023691193575951 0.0029481054428925259 0.01677860275763431 0.024360180961497419 0.012742456770485805 -0.024356987764840882 0.019684065460062714 -0.011127285945085478 -0.012882022590504218 -0.01818142880860367 0.027354370371631558 -0.02615763359226142 0.00022192644150775991 0.030635250784099004 0.026964756102598134 -0.019970741599362896 0.011397535416244759 0.009183395813120979 0.022573012852713039 -0.021319368199699405 0.0051730557504284106 0.011513093250471791\nleaf_weight=45.572868563505835 21.955766990307893 16.267695845101116 13.187700260150448 20.644885852525476 7.751466384273952 3.0981281176791517 7.8322144721169025 7.9828073599128393 5.6347577549895496 5.6353883916744953 36.756662763662462 3.0087259554857146 18.344603867270052 3.9973375472309263 3.2481561033055186 9.4548466671694751 44.119466709624248 9.934470986692757 1.9439457179978479 20.455515874818957 4.9232755644479749 7.9206876583775729 11.430420711410987 4.7876922388604752 24.393488321511541 174.33677842054749 2.232773760915733 10.081836448378452 27.40836707654671 66.455254266096745\nleaf_count=138993 17291 17388 2930 1794 209 500 27 987 196 661 756 226 208 6181 20 2823 4363 1123 14 40668 34 207 540 601 991 1863 97 593 653 3254\ninternal_value=-0.000286816 -0.0189694 0.00194119 0.00270941 -2.98352e-05 -0.00374164 -0.00139208 0.00382086 -0.00832062 -0.0251625 0.00710726 -0.0058807 -0.00238853 0.00277166 -0.0156671 0.00569015 -0.0081532 0.00772873 -0.00918393 -0.0173419 -0.0228958 0.0112902 0.00768974 0.0122327 0.00575984 0.00769077 -0.0107335 -0.00556636 -0.000616297 0.00995\ninternal_weight=640.798 67.476 573.322 551.366 356.385 234.765 201.122 115.026 86.0958 51.2083 121.619 80.4611 67.7056 84.8625 10.9996 73.8629 49.361 69.8656 11.8784 33.6432 12.7555 57.9871 50.0664 40.6116 29.1812 194.982 46.3522 45.473 35.3912 69.5534\ninternal_count=246191 157042 89149 71858 68201 54736 11138 5987 5151 139654 13465 4955 4894 12709 229 12480 4686 6299 1137 43598 61 5162 4955 2132 1592 3657 4460 2233 1640 3754\nis_linear=0\nshrinkage=0.03\n\n\nTree=131\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 0 4 1 16 12 12 3 5 4 21 0 13 14 3 17 17 17 1 18 7 3 13 3 18 17 19 18 18 0\nsplit_gain=30.4549 10.6271 12.4361 11.4704 9.05851 8.03722 8.69063 7.76312 7.41998 9.08023 6.78302 6.25624 5.99 6.74924 8.33776 5.81687 5.51482 5.509 5.48255 6.0659 5.24821 4.8345 4.41094 3.85484 3.79129 6.29629 3.7824 3.75942 7.7788 3.75115\nthreshold=0.18336881478606726 1.8987378677022575 5.5515136718750009 2.1468275719512682 1.0000000180025095e-35 0.084607650078150978 0.76965903234489019 0.58795853567030554 1.4980468750000002 2.6030273437500004 1.5000000000000002 0.20945686230155949 -0.056686766081373456 0.62855356909667792 0.39370400768236996 1.667689311232593 1.6214838777669847 0.78716740761650772 1.5275240303550033 2.9174053685313028 3.4804687500000004 0.50207005315878861 -0.0094969182612588115 0.26534039634031065 0.89587973461402759 0.71600374927799881 244.50000000000003 3.1984648276080736 4.0342014792848504 0.49902597412172423\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 3 12 -1 11 17 8 15 -10 -11 20 13 14 18 -5 -14 -7 19 -2 22 -3 -4 -12 -23 -26 -8 -9 -29 -30\nright_child=1 21 5 7 -6 6 26 27 9 10 23 -13 16 -15 -16 -17 -18 -19 -20 -21 -22 24 -24 -25 25 -27 -28 28 29 -31\nleaf_value=-0.025429655644949634 0.0089705112422699459 0.0055551004651341563 -0.031360123169538977 0.0066366371488843562 -0.0019590505942387875 0.0014328885816211855 0.021268598554604647 0.012435644795379083 -0.01867919668627455 -0.021329639971928747 -0.025105251117009262 0.020878691001230316 0.0073642067364414927 -0.023596509493412278 0.0047835459091685253 -0.017461046935531892 -0.018388874641908172 -0.037115865969208205 -0.033154038253144252 -0.022167691028454742 0.0157452928788397 0.02885623539298704 0.0083834789828458375 0.010965494903252737 0.02330090432921781 -0.015336255887879867 -0.015756891033479575 -0.029142873594427868 -0.015893268040812529 0.025471461661414958\nleaf_weight=45.681596597636599 10.199458761791908 33.447795146494173 4.9988110801423318 95.375299573483062 21.92925070120873 5.7801377483046981 10.216099210781975 77.848556729964912 35.536859666201053 7.948930661979829 1.9718207516125392 56.200373259162006 32.486002304376598 29.789166057009425 30.795555367323686 8.9861727623938332 8.6441613664683228 6.6671740440797294 10.920736735406534 10.528448608929464 8.9400119523779704 22.646596793085337 3.7364397990531888 30.454338398107211 8.583645331207661 5.4062483885209067 2.3992108625825486 5.5011077381204796 2.0010670148185454 5.7684036926948465\nleaf_count=151747 2194 515 4031 2764 8281 204 92 462 1126 361 284 1251 5965 29182 2948 1032 6686 447 2751 21539 477 102 511 630 58 39 34 192 168 62\ninternal_value=0.000936702 0.00319035 0.00165441 -0.000683961 -0.0179566 0.0111318 -0.00344819 0.00285603 -0.000760253 -0.00784082 0.00191967 0.0160581 -0.00785543 -0.0120637 -0.00627669 0.00441565 0.00164378 -0.0204397 -0.0168557 -0.00713802 0.000433343 0.0140505 -0.0152452 0.0080866 0.0214529 0.00834294 0.0135869 0.00994121 -0.00453777 0.0142203\ninternal_weight=641.389 573.779 503.694 404.756 67.6108 98.9383 25.0626 271.393 180.273 75.9119 40.3751 73.8756 133.364 92.2334 62.4442 104.361 41.1302 12.4473 31.6486 20.7279 17.6753 70.0843 8.73525 32.4262 36.6365 13.9899 12.6153 91.1191 13.2706 7.76947\ninternal_count=246135 86107 85393 78346 160028 7047 777 7081 6197 2401 1275 6270 71265 58614 29432 3796 12651 651 26484 23733 5019 714 4542 914 199 97 126 884 422 230\nis_linear=0\nshrinkage=0.03\n\n\nTree=132\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 18 2 2 17 6 14 2 3 14 18 3 19 1 8 3 2 17 19 1 18 18 5 17 14 2 4 3 4\nsplit_gain=21.7145 16.817 13.9967 13.6364 6.99956 5.32962 5.09103 8.31772 6.2998 5.32277 4.9453 4.96998 7.59634 5.9928 5.4199 4.04114 4.5215 4.0266 4.01643 4.01635 3.53388 4.04329 4.72337 6.99249 4.51018 4.079 3.93924 5.52777 3.99636 6.55332\nthreshold=1.7622051334857016 3.5791015625000004 2.4414009612931857 -25.182852745056149 -55.708036422729485 4.4301796584464421 1.0000000180025095e-35 0.0095533965833058416 5.8049912452697763 0.30295287721642777 0.032916404154818342 3.5694334999727624 0.22136347059745312 226.50000000000003 -0.041910702480567281 1.0000000180025095e-35 0.48555671035752257 31.708583831787113 1.849927262362129 209.50000000000003 4.0016611463566001 0.89587973461402759 2.4414009612931857 1.4980468750000002 1.0055080915597503 0.28287814140777068 34.417465209960945 2.8684082031250004 0.65908129556435835 1.6286621093750002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 18 -3 -2 6 7 17 -8 -9 -5 12 -12 -14 -13 16 -4 -6 -1 -16 26 -22 24 -24 -23 -26 28 -28 -11 -30\nright_child=4 3 15 10 5 -7 8 9 -10 20 11 14 13 -15 19 -17 -18 -19 -20 -21 21 22 23 -25 25 -27 27 -29 29 -31\nleaf_value=0.0026182929656427081 -0.024864682602397652 -0.026168929248714484 -0.028424926840206612 0.021407359603573119 -0.038073856528132499 -0.027518602968029281 -0.0035605245660674358 -0.0079067574258847289 0.02115111691930209 0.0034860498310538344 0.0094311093361269405 -0.015574473376167278 -0.0046743714288456769 -0.038689773412272248 -0.023608416712726541 0.0063865404547169807 -0.0051141077441279109 -0.0048262478664696847 -0.027355224536241649 0.013453046413025119 0.02298578873869701 0.0069070699230984007 0.024417857782789369 -0.0094155712033451161 0.011011985757634042 -0.026323842222601351 -0.01933895437727318 0.011916839830862856 -0.0083278515353668002 0.022680681878935679\nleaf_weight=33.655195383362297 6.2682164073557933 13.697461705076419 56.727961506503561 12.88306520104379 4.7117789919284396 3.5432271441677576 12.354316428769378 26.015016924022348 33.456328324711649 178.15890636069525 13.890745211581814 6.8954625035894415 11.539713733349346 7.205378303086035 1.9005704819064693 3.4482296068963469 9.1742801496875455 8.5084458713827171 3.5531154959789992 44.049809863601695 19.296959456056356 27.17866288125515 21.801896601216871 6.6472636780235907 3.1487242467701417 7.4998196447268128 13.152811213367388 7.1210697168717152 7.6095582369889625 23.372859597788192\nleaf_count=4680 1119 45871 157582 1341 163 1244 374 2383 374 5306 1032 8541 427 518 532 1036 2614 140 2399 6637 113 142 198 101 15 58 336 76 205 131\ninternal_value=0.000299214 -0.00731624 -0.01579 0.000809743 0.00436939 0.00488378 0.00523868 0.00399649 0.0145531 0.00492103 0.00479212 0.00206944 -0.00700572 -0.0190467 0.00769997 -0.0236606 -0.0253048 -0.0185201 -0.000884992 0.0114499 0.00600215 0.0111799 0.00745195 0.0164613 0.000426757 -0.0152862 0.00403067 -0.00831645 0.00526146 0.0150425\ninternal_weight=628.467 218.621 106.559 112.062 409.846 403.578 400.034 354.224 45.8106 341.004 98.3647 85.4817 32.6358 18.7451 52.8458 69.3505 65.9022 13.2202 37.2083 45.9504 314.989 85.5733 66.2764 28.4492 37.8272 10.6485 229.415 20.2739 209.141 30.9824\ninternal_count=245688 233210 168311 64899 12478 11359 10115 9367 748 9064 19028 17687 1977 945 15710 161232 160196 303 7079 7169 6681 627 514 299 215 73 6054 412 5642 336\nis_linear=0\nshrinkage=0.03\n\n\nTree=133\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 4 17 9 17 0 3 11 13 12 5 21 21 4 4 4 4 2 11 21 4 21 2 14 0 2 14 3 3 13\nsplit_gain=20.298 13.5735 8.75383 8.0892 7.97942 7.96867 6.74511 6.06712 5.95959 5.58871 5.30212 4.19982 5.51935 4.19708 16.6973 9.25556 5.64461 8.35456 6.11065 6.51783 8.11459 6.33458 5.29835 5.27781 4.66606 4.64188 4.02739 3.8242 6.85513 4.15372\nthreshold=-39.283819198608391 0.84667968750000011 3.9653415636860969 1.0000000180025095e-35 2.5789308873416652 0.52258841453381233 0.44825707527455844 5.5511151231257837e-16 0.14796868535883259 0.79622554818162328 6.0488281250000009 1.5000000000000002 5.5000000000000009 1.0693359375000002 1.6286621093750002 1.6572265625000002 3.7858886718750004 -5.1300392150878897 0.33353935825991382 2.5000000000000004 2.3220214843750004 1.5000000000000002 -19.576506614685055 0.92990689048105823 1.8987378677022575 -37.900318145751946 1.0000000180025095e-35 0.53650715206675215 0.51817819236196982 -0.6866788914798736\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=6 -2 3 5 7 8 -1 -5 11 -9 -4 -3 -13 -7 25 22 18 -18 19 20 -17 -20 -16 -19 -22 -15 -27 28 -21 -29\nright_child=1 2 10 4 -6 13 -8 9 -10 -11 -12 12 -14 14 15 16 17 23 21 27 24 -23 -24 -25 -26 26 -28 29 -30 -31\nleaf_value=-0.027192150224006197 -0.020179306473205427 -0.022358231547506928 -0.022351845997105178 0.024635872741750209 -0.03116359344016649 0.0092377382991416148 -0.0042606188415963827 -0.011817747480452405 0.021859448248122058 0.019102074264946891 0.018152844948994763 -0.00054162152798990944 -0.026716196633510339 0.022469066146300145 -0.016048462436217072 0.0022087865089777826 -0.025143237263223223 0.012960579175543403 0.0058387430254654791 0.0024854030512805655 -0.03160476312496599 -0.035248666280044888 0.023173832888812435 -0.015413466132847182 0.0091079934699132437 0.010588370764282947 -0.026729352653936278 -0.019773070459258968 -0.027994080024771387 0.011655375049760778\nleaf_weight=29.528097032749429 23.788309611201839 16.481169135004166 17.556642008567 32.17517303216664 2.4064515981081049 91.603298287507641 19.254286173483706 9.9455331463250349 4.9284530102740964 9.1690982365980727 2.6715926057440802 41.819217035770635 7.7824689263725295 1.3992416925284512 2.7620606104210337 21.835672830464318 5.2961304461787213 76.275934275705367 5.0443260538158921 60.236407155294728 13.97544333451515 7.8839854899633792 28.337839062252897 5.5580645700320011 2.5856011103605843 2.3785447238478836 23.597877768779654 3.1659765637741648 6.449925214634276 54.92929924314376\nleaf_count=82888 34631 24280 28414 1611 412 5514 1071 656 65 92 51 27035 11421 100 227 3519 670 1699 63 6528 3431 140 481 492 51 339 9328 130 104 334\ninternal_value=-2.12346e-05 0.00154879 0.00250909 0.0032491 0.0141882 0.00200957 -0.0184085 0.0168713 -0.00720833 0.00322647 -0.0163992 -0.00966974 -0.00509521 0.00360686 0.00197989 0.00408598 0.00222241 0.00842111 -0.00087467 0.000748866 -0.00990434 -0.0199491 0.019299 0.0108706 -0.0250918 -0.0200102 -0.0230465 0.00407966 -0.000832615 0.00963703\ninternal_weight=630.822 582.04 558.251 538.023 53.6963 484.327 48.7824 51.2898 71.0113 19.1146 20.2282 66.0829 49.6017 413.316 321.712 294.337 263.237 87.1301 176.107 163.178 38.3967 12.9283 31.0999 81.834 16.561 27.3757 25.9764 124.782 66.6863 58.0953\ninternal_count=245777 161818 127187 98722 2771 95951 83959 2359 62801 748 28465 62736 38456 33150 27636 17869 17161 2861 14300 14097 7001 203 708 2191 3482 9767 9667 7096 6632 464\nis_linear=0\nshrinkage=0.03\n\n\nTree=134\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 18 7 18 4 19 4 4 5 0 4 14 14 3 3 18 14 13 17 3 3 17 17 3 0 1 1 19 13\nsplit_gain=19.9585 13.3846 9.78902 8.42262 7.92287 5.6354 6.30182 5.1893 9.30964 7.43843 5.53371 4.92921 4.65766 4.62196 5.08352 4.57643 4.56507 6.3829 4.52474 4.24354 3.97368 3.98859 3.89306 3.8874 3.86167 3.83268 3.7869 3.76249 7.3784 3.97329\nthreshold=1.6109634896136267 3.4438476562500004 6.9032449973715968 0.89599609375000011 2.4414009612931857 1.0693359375000002 243.50000000000003 2.1750488281250004 3.7858886718750004 2.3505859375000004 0.70465417701170696 2.7826617956161503 0.088129815934729419 0.45494134147108911 0.58795853567030554 0.37248792739977959 2.0126758453675748 0.59843054868820844 -0.25172795495308953 1.2428969707451916 0.50207005315878861 0.44825707527455844 1.2428969707451916 0.72612140026470018 0.18185704705590336 1.6765212893675572 1.5275240303550033 3.3033179733375269 264.50000000000006 -0.030094928075697355\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 -3 5 19 6 -2 12 9 10 -9 -7 -5 15 -15 -14 17 20 -19 23 21 -12 -18 24 -1 -10 -6 28 29 -8\nright_child=3 2 -4 7 26 11 27 8 25 -11 16 -13 13 14 -16 -17 22 18 -20 -21 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=0.0093164732870934085 -0.018865742037922035 0.0021168983825509322 -0.025907596444212758 -0.024836930573678893 -0.02594498550726267 -0.013361130212008021 -0.0031676609716431076 -0.023155504339898724 0.0065355932097125929 -0.031394866207542539 0.0040631109208775877 0.012152810476438531 0.0007858643349473182 -0.0005808084311802375 0.01921650690252967 0.027087560135843555 0.014396912439659276 -0.00789320036486316 -0.041182570046573018 -0.02373819015033328 0.013987375915162829 -0.029555689006910399 -0.013382511917705085 0.016157608707065479 -0.024709103629282411 0.022019302132490856 0.00065144343128658926 0.010100596267346844 -0.03856890205824047 0.018447913317143332\nleaf_weight=4.8805865529502679 11.133031746809136 84.557759858053032 11.914632561705728 2.1120371810102361 55.227532410581262 51.61878599874035 31.937370241605091 8.0719706965683127 61.123234499806131 6.9071386788782538 8.4496351859997976 7.0350251848576582 7.6522990114463019 30.569158438680461 17.82862321997527 26.82401078986004 27.383134226023689 13.233212702907624 4.4750011612195513 6.0640285527297229 15.856768954545258 3.9040085263550273 4.5886322507867581 9.5714033467011159 5.4187909210886573 18.480195418931544 5.2168471346558354 45.719534604460932 3.4909338194993316 8.9830845776887163\nleaf_count=1519 1185 28057 38590 39 155660 4699 2564 1280 1576 533 69 325 196 757 142 129 369 91 63 3129 55 13 148 971 1167 114 1460 435 321 298\ninternal_value=-0.00105807 -0.0092952 -0.0015745 0.00256418 -0.0178013 -0.00283238 0.00146306 0.00591418 0.00289323 -0.00354806 -0.000975263 -0.0101485 0.01194 0.0131622 0.00695344 0.0214364 0.00158771 -0.00444567 -0.0180573 -0.00373847 0.0044243 -0.00797883 0.010125 0.00320986 -0.00918778 0.0103588 -0.0236763 0.00415356 -0.00205626 0.00190438\ninternal_weight=600.228 182.852 96.4724 417.377 86.3792 159.918 101.264 257.459 172.473 92.8695 85.9624 58.6538 84.9861 82.8741 48.3978 34.4763 77.8904 45.9186 17.7082 25.9348 28.2104 12.3536 31.9718 19.8708 10.2994 79.6034 60.4444 90.1309 44.4114 40.9205\ninternal_count=245954 230553 66647 15401 163906 9827 4803 5574 4311 2621 2088 5024 1263 1224 899 325 808 291 154 6786 137 82 517 3657 2686 1690 157120 3618 3183 2862\nis_linear=0\nshrinkage=0.03\n\n\nTree=135\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 3 7 5 4 2 2 14 3 21 3 17 1 17 3 5 17 3 1 13 19 0 19 2 1 10 4 19 1 14\nsplit_gain=20.7352 12.2784 9.13919 9.22133 9.83779 8.02565 6.73587 6.31412 8.10054 6.19196 5.89379 5.63066 5.50683 5.35787 5.33564 5.33295 4.93116 4.45658 4.45429 5.00737 5.35561 5.31263 4.3313 4.26746 4.24992 4.18555 3.91541 6.82234 3.79991 3.67587\nthreshold=0.15052965826911111 0.024199563577468081 0.89599609375000011 0.89599609375000011 0.84667968750000011 18.240050315856937 -55.708036422729485 0.21770771341770703 0.33504387972412514 6.5000000000000009 0.58795853567030554 0.35042734261365599 2.5144622974930591 3.9653415636860969 0.43673077644757391 1.7006835937500002 3.2718994203396332 0.53650715206675215 3.3033179733375269 -0.39822854168506638 252.50000000000003 0.24908287076908558 264.50000000000006 -24.354135513305661 0.45969269984116679 1.8688348091416842 5.9760742187500009 243.50000000000003 4.0016611463566001 0.73406812255247111\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 16 3 4 -2 11 -4 8 -6 28 14 12 -5 18 -11 17 -1 -7 19 21 23 24 -22 -21 -8 -3 27 -23 -9 -13\nright_child=2 25 6 5 7 15 13 9 -10 10 -12 29 -14 -15 -16 -17 -18 -19 -20 20 22 26 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=0.012070898917142487 -0.0161715834040396 -0.028069613083328126 -0.028095089382035941 -0.023507613110396862 0.012851458006646018 -0.019936771484137847 0.013099275589218723 0.0071930158712016044 -0.031797670871798675 -0.028941042595661753 -0.03895546546640076 -0.031857906765694721 0.010207722066531371 -0.017374719017089306 0.017200427072274038 0.01362388231117344 -0.020129825932871489 0.0087879785853097209 0.011242742513485959 -0.025716409613533362 -0.020085835102373831 -0.026046345197675659 0.013789508470989621 0.0036306191958394005 -0.031603823347445707 0.0050449518119064984 0.021580431758378954 0.0098882621757565518 0.021720350904204353 -0.0070596188632183615\nleaf_weight=14.474010926184748 21.469302607480131 38.561613949974344 4.5130434054362913 6.4711925239589645 5.6399800140025018 12.468392105172827 2.372879700989869 60.957392894084478 7.5532659625750966 2.2781602595168788 3.4893990207347079 23.288207774883634 9.5469925226643664 8.1189161144211521 6.2742070519452682 14.362809199403271 4.9358691952747895 6.8811298337968756 100.82513035205193 3.6415442488323597 16.546335184841158 4.2231853428484092 3.4606630503039915 117.74794445720181 4.0376993345962546 3.4565060141831045 22.838918735527841 57.158397275929019 22.048199652228504 7.7834288688427469\nleaf_count=8417 17327 137123 6329 5620 1006 3834 1182 6154 112 949 53 12428 258 4801 72 1665 11036 295 705 3532 1704 1031 215 7545 2481 529 446 3456 154 5790\ninternal_value=4.52785e-05 -0.01635 0.00188278 -0.00303564 0.0019251 -0.0109252 0.0048798 0.00562972 -0.01315 0.00834625 -0.0110906 -0.0187929 -0.00399391 0.00539713 0.00316635 0.000289351 0.00329729 -0.00979173 0.00600295 0.00368758 0.000146368 0.00915046 -0.0138488 0.00254732 -0.0155222 -0.0252916 0.0111886 0.00703798 0.0112644 -0.0260598\ninternal_weight=617.425 61.428 555.997 210.512 129.71 80.8022 345.485 108.241 13.1932 95.0474 12.0418 47.0898 16.0182 340.972 8.55237 33.7123 19.4099 19.3495 332.853 232.028 141.396 90.6311 20.007 121.389 6.41058 42.0181 84.2205 61.3816 83.0056 31.0716\ninternal_count=246249 157105 89144 55717 25827 29890 33427 8500 1118 7382 1074 24096 5878 27098 1021 5794 19453 4129 22297 21592 12996 8596 1919 11077 3663 137652 4933 4487 6308 18218\nis_linear=0\nshrinkage=0.03\n\n\nTree=136\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 18 1 4 4 4 10 19 19 19 4 3 7 18 5 4 4 1 13 19 5 17 11 4 13 3 0 4 3 0\nsplit_gain=21.2649 9.72385 9.54193 7.76902 6.29794 14.963 5.88158 5.88133 6.3795 5.34676 4.77401 4.73253 4.28513 9.99145 4.08397 4.39182 5.4493 3.94854 3.88697 3.79026 3.77026 3.68558 3.60446 3.50706 4.53573 3.97747 4.01713 3.36497 4.33032 3.35804\nthreshold=0.24961073363565869 3.3140206880897671 3.3033179733375269 0.84667968750000011 1.0693359375000002 1.4716796875000002 1.0000000180025095e-35 264.50000000000006 240.50000000000003 267.50000000000006 3.9514160156250004 0.024199563577468081 3.4804687500000004 5.2755571088971331 4.7832031250000009 4.8846709132194528 7.3568274974823007 0.68185604854186199 -0.53246711157653859 244.50000000000003 6.0800781250000009 0.67728060322190908 0.66750229771629133 1.6572265625000002 -0.04883439202386685 0.29586597258655462 0.75738199555027408 2.2978515625000004 0.65908129556435835 -0.049143614581278028\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 11 3 -2 7 9 -3 8 -5 -6 27 -1 13 14 15 17 21 -7 -13 29 -14 -17 -15 25 -25 -19 -27 28 -4 -18\nright_child=2 6 10 4 5 12 -8 -9 -10 -11 -12 18 20 22 -16 16 19 23 -20 -21 -22 -23 -24 24 -26 26 -28 -29 -30 -31\nleaf_value=0.017864473592258599 -0.020878094294632128 -0.029161841965024408 0.0060544833996972781 -0.028272323138855752 -0.03117812172974109 -0.026251839428694631 -0.0029749482069790622 -0.021352149109473622 0.01479934119168381 0.015631612293883015 0.020489878525693013 0.012819105867365921 0.013748823310143876 -0.02936134135533653 -0.0314904055965122 0.023465056857892637 0.016700878061176549 -0.024145657084868301 -0.020918117248632705 0.017847833644370405 -0.020077516810865881 0.0047367473454286019 0.012100486814869291 0.00078982909478089161 -0.012267872393736822 -0.020171470419216783 0.022602983546130048 -0.0020902686117072588 0.024248591655344126 -0.028675542475656807\nleaf_weight=8.1107700699903962 15.711616762608172 35.42116915457882 107.13670754616032 2.3939121905386846 17.954298804363138 4.5377107016283835 10.104809390100852 4.2514299557005861 48.0235514577289 1.8737940679420706 21.731497698638123 3.7656380511543821 28.339388634864012 13.082973361690163 2.5346260463411445 18.015315735327022 1.1394960576435549 1.5143556737912174 8.3766753749259788 2.348280873906333 2.4159575420053434 20.088224900260684 1.6217012372362649 110.36083257183964 29.53790710094836 1.5011054823989969 13.326766601385314 37.649206886766478 12.462041241582481 6.2057819043693581\nleaf_count=1381 15485 149760 1210 1655 12829 10073 3683 492 5120 177 217 1956 828 12948 131 552 14 719 8537 44 130 1244 57 13085 2425 362 171 407 73 221\ninternal_value=-3.6358e-05 -0.0159846 0.00198553 -0.000920793 8.63501e-05 -0.00176194 -0.0235383 0.00929483 0.0122441 -0.0262164 0.00759045 0.00130371 0.000222379 -0.00122355 0.000491495 0.00102854 0.00891337 -0.0013561 -0.0102761 -0.010393 0.010543 0.0139651 -0.0243094 -0.000470699 -0.00203465 0.0122629 0.0172719 0.00568955 0.00813564 -0.0204784\ninternal_weight=591.538 65.7791 525.758 346.779 331.067 276.399 45.526 54.6689 50.4175 19.8281 178.979 20.2531 256.57 225.815 211.11 208.576 47.7971 160.779 12.1423 9.69356 30.7553 38.1035 14.7047 156.241 139.899 16.3422 14.8279 157.248 119.599 7.34528\ninternal_count=245986 165317 80669 78762 63277 56010 153443 7267 6775 13006 1907 11874 43004 42046 29041 28910 2075 26835 10493 279 958 1796 13005 16762 15510 1252 533 1690 1283 235\nis_linear=0\nshrinkage=0.03\n\n\nTree=137\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 18 5 1 17 4 1 10 3 14 1 0 13 3 4 4 17 18 17 0 13 14 2 14 1 21 14 2 3\nsplit_gain=21.3567 10.8119 10.2537 12.4627 7.75631 8.14462 7.0722 6.88808 6.32049 6.12375 5.20602 4.94948 4.69342 3.99502 3.72976 3.54457 5.46549 3.47348 4.82043 4.67306 4.45645 4.22596 3.72799 4.47535 3.61705 3.55963 4.83449 4.61512 3.4657 3.34656\nthreshold=0.24961073363565869 1.7005986908310777 1.7005986908310777 4.2421875000000009 1.7622051334857016 1.6214838777669847 3.8417438268661503 1.4130033503388202 2.6735537653587351 0.37963534769016083 0.66452579377975118 1.8912989126705502 0.84505259813102007 0.14796868535883259 0.11110445317776836 1.0693359375000002 0.84667968750000011 0.68882141110442896 2.3502401828962087 0.45436599119170595 1.4460303622695712 -0.74650584208841086 0.69550561433971103 -33.804542541503899 0.43080354720652592 2.6981674883069502 5.5000000000000009 0.60348287794770183 10.479034423828127 0.45898141303402712\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 12 10 4 5 6 15 28 -3 13 -2 -12 -1 -6 -5 16 -4 18 21 25 -19 -11 23 -21 -23 27 -27 -20 -8 -16\nright_child=2 8 3 14 9 -7 7 -9 -10 17 11 -13 -14 -15 29 -17 -18 20 19 22 -22 24 -24 -25 -26 26 -28 -29 -30 -31\nleaf_value=-0.027448064569943698 0.01063044140436854 0.019135902080134017 -0.025970716416217999 -0.023223747661546026 -0.013840977964597484 -0.031318687949646894 -0.0014555058921958562 -0.028252712942105539 -0.028964252045187631 0.016663287446101353 -0.023140381557986069 0.0031022659267936092 -0.0058051519774175398 0.017705826508428248 0.01871605985750948 -0.020202203110218964 0.010480405211977446 0.0031649683667188109 -0.027330885662654594 0.0055762449394151222 0.022530412228521552 -0.0021177120024892227 0.0055668014642665268 -0.036406349217774407 -0.028454823794856015 0.017359054038668803 -0.018487047535301348 0.0089867531943429134 0.024040877557526361 0.002833196457731931\nleaf_weight=38.780027763217731 132.98342266441614 9.7464603671833157 4.5798025441144974 1.7577205924717407 37.300031667335134 14.561318257849051 7.8463293212516936 3.2838245636739876 2.2383106364213736 2.718834369210529 6.405166030075633 50.127737431670539 12.479583499294675 3.1350978985428801 34.156319668894866 27.472238826867397 10.023522779778432 72.843194513974595 4.3973179021268134 3.057992042915429 11.639000661496537 7.5689926306949955 6.4637261407333417 5.9162342591444022 12.134989521466194 25.504440852528209 3.0511530406656666 6.7296369316172786 10.926579551625762 18.694756349403178\nleaf_count=153576 1594 664 8673 222 5614 22502 3641 434 367 19 666 464 10704 42 1416 25075 3856 2832 445 140 100 70 237 232 184 408 104 659 548 514\ninternal_value=-0.000144072 -0.0164728 0.0018495 -0.00130122 -0.00385585 -0.0117842 -0.00693608 0.00662799 0.00861378 -0.000731454 0.00740911 -0.000221552 -0.0223908 -0.0110267 0.0116842 -0.0140354 -0.00188296 0.00188972 -0.00267879 0.00212991 0.00606528 -0.0140859 -0.0114137 -0.0229759 -0.0189287 0.0075499 0.013033 -0.006436 0.0138497 0.0132548\ninternal_weight=588.524 63.2444 525.279 335.763 281.154 78.6936 64.1323 22.0567 11.9848 202.461 189.516 56.5329 51.2596 40.4351 54.6088 42.0756 14.6033 162.026 77.5433 55.1205 84.4822 22.4228 15.438 8.97423 19.704 39.6825 28.5556 11.127 18.7729 52.8511\ninternal_count=246002 165311 80691 77967 75815 64729 42227 4623 1031 11086 2724 1130 164280 5656 2152 37604 12529 5430 2498 2225 2932 273 609 372 254 1616 512 1104 4189 1930\nis_linear=0\nshrinkage=0.03\n\n\nTree=138\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 1 3 0 3 0 18 2 19 13 14 2 9 4 4 0 7 2 3 3 6 3 14 0 7 3 2 1 21\nsplit_gain=28.1764 12.7287 6.8072 8.97429 5.97317 6.02242 5.74211 5.55729 4.9246 4.56733 4.52947 6.60372 4.48645 6.5414 4.32264 4.27147 3.87277 3.8656 3.78531 4.01163 3.65421 3.57995 4.89516 3.33041 3.884 3.67271 5.43744 4.79836 3.53995 5.2252\nthreshold=1.7622051334857016 1.0000000180025095e-35 0.95935769349451683 0.3261289997719386 1.2661233611507157 0.53650715206675215 0.72961209499321422 5.0271592211193568 -55.708036422729485 247.50000000000003 -0.48625410865680269 0.74140086107412606 32.956295013427741 1.0000000180025095e-35 5.9760742187500009 3.2895507812500004 0.29896707506516013 2.2500000000000004 10.479034423828127 0.25176753112663547 0.48555671035752257 2.5244140625000004 0.36709515926886871 0.57616929401185935 1.3771184689138856 0.89599609375000011 0.35479072494832037 47.592866897583015 4.0016611463566001 2.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 15 8 17 -1 9 -2 -5 11 -8 13 -3 -12 18 -17 23 19 -4 -11 22 -10 25 -25 28 27 -27 29 -6\nright_child=4 12 3 7 5 -7 10 -9 21 20 14 -13 -14 -15 -16 16 -18 -19 -20 -21 -22 -23 -24 24 -26 26 -28 -29 -30 -31\nleaf_value=-0.0278375835937512 -0.024840214719591133 -0.016896561604159671 0.011200617321660982 -0.036276695489208872 0.0054386880517623666 0.013780792548421406 0.026721435452956986 0.0073906481107303551 -0.010669697715674012 -0.027233089309433526 -0.025312302605248565 -0.024577233504024455 0.014562647739695171 0.0063713841486187882 0.01021907011110885 -0.019234226131807645 0.024355370213381067 0.017271568927522615 -0.027343050623045661 -0.023851587206356828 -0.0013978995956821696 0.021576126314106877 0.0030054587761677736 -0.034658661318397337 -0.0029269980811057763 -0.020166580409381959 0.018705444173682105 0.018925405550475331 0.010130869864299777 -0.025916853990049032\nleaf_weight=42.065812665154226 5.4989913820754728 17.282503611409993 11.450960398928151 11.922703828018713 8.5697779975889734 93.784021808940452 5.2673226732040375 6.428496748001634 27.000973985312157 7.5191447684082986 12.791144147289744 2.5977041006372019 17.315670878102537 25.941954439080291 3.367735748179256 1.4179422271809006 10.321586559155547 14.270063042407854 3.8742230675379679 2.8588822394121953 11.96103255136404 6.8365720552392295 147.63353316303983 3.0779850091785219 25.012134894001065 6.8992369740735722 24.296004578471184 3.4635521131567657 10.275393624790015 9.0595939705381152\nleaf_count=176227 1135 8522 5505 2889 166 535 1931 1620 4424 2887 18219 3723 624 2324 131 1407 454 87 2731 3162 1132 72 4934 154 482 86 109 28 72 275\ninternal_value=-5.93054e-06 -0.00929148 -0.0144232 -0.00757235 0.00468599 0.00829398 -0.021229 -0.0170999 0.000827773 -0.0221867 -0.00900743 0.00891709 0.00210384 -0.00310357 -0.0177431 0.00463646 0.0179939 0.0033059 -0.00447549 0.00309722 -0.0122128 0.00173744 0.000842374 0.000943067 -0.0074759 0.00478121 0.0111711 -0.00658518 -0.00326716 -0.0112009\ninternal_weight=580.063 194.385 133.845 67.755 385.678 198.708 66.0897 37.8314 186.97 31.4029 24.0239 7.86503 60.5401 43.2245 16.1589 29.9236 11.7395 104.924 18.1841 14.3098 19.4802 181.471 174.635 90.6537 28.0901 62.5636 34.6588 10.3628 27.9048 17.6294\ninternal_count=246047 233488 222018 21787 12559 1994 200231 8528 10565 6908 24004 5654 11470 10846 18350 13259 1861 1459 11398 8667 4019 9430 9358 1372 636 736 223 114 513 441\nis_linear=0\nshrinkage=0.03\n\n\nTree=139\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 2 17 3 18 4 18 4 4 18 18 2 2 2 1 13 19 5 4 5 3 4 13 22 1 13 19 4 2\nsplit_gain=15.6983 13.6962 8.5282 7.00542 6.12174 5.8672 6.99662 5.79149 5.67251 5.21411 7.50179 4.96602 4.52414 6.38584 6.08644 6.03149 5.50838 4.14442 4.94933 8.02802 7.04238 4.29594 7.30445 4.41427 5.40824 4.53964 4.08789 4.08565 7.28077 4.91137\nthreshold=1.5661550609387962 1.0000000180025095e-35 -9.1231012344360334 1.6214838777669847 0.33966381556979741 1.4978661367769956 8.3627929687500018 4.8713658868283671 3.3325195312500004 3.8417438268661503 3.5114340430413216 0.89587973461402759 -1.3165187835693357 -3.0241031646728511 1.2144355773925783 3.3033179733375269 -0.30024263418623959 237.50000000000003 3.7734375000000004 2.3220214843750004 4.4589843750000009 0.65908129556435835 1.6286621093750002 1.0000000180025095e-35 0.93750000000000011 4.0016611463566001 0.14796868535883259 264.50000000000006 2.6540527343750004 32.956295013427741\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 11 4 9 8 17 -6 12 10 -4 -1 13 15 -14 16 -2 -7 19 21 -20 23 -23 25 -25 -19 -5 29 -29 -21\nright_child=5 -3 3 26 7 6 -8 -9 -10 -11 -12 -13 14 -15 -16 -17 -18 18 20 27 -22 22 -24 24 -26 -27 -28 28 -30 -31\nleaf_value=0.013174938448650803 0.0025641044557353728 0.002416236401949756 0.008284531604394409 -0.025820957662629228 -0.030492761163752612 -0.014262185624108676 0.01761027706566605 0.013087997783809702 0.017776903847177606 0.022018128053227112 -0.026929924468150936 -0.029093877626474723 -0.031392040941598788 0.033675759695551873 0.00091023233052555492 0.014829213091866269 -0.032967409283727919 -0.0060971131715405929 0.023493542649897507 -0.017736250588782417 -0.025507207118475192 -0.014129884092348249 0.030784852811528096 0.020003600583857181 -0.019058660796377597 0.0079917669199671016 0.016947519151605674 0.022866528524034441 -0.032815377044970812 0.0100829133820607\nleaf_weight=2.1196442226355421 12.924294920172542 56.819224358749125 15.974382135493217 20.93452789417643 9.1460117231872555 23.384630636326619 15.589304367022121 3.1123379528980868 29.605373892583884 9.7852629845001484 7.0323310368166849 37.840175056693624 4.714555960148572 7.6189727187156668 63.972849781217519 23.165658422745764 4.5626338960137227 82.02849023959061 12.87239423401479 46.876681838839431 2.3907432576525016 4.0694095786893731 9.2641989646945131 15.502067099558191 3.1133551658131173 26.428428183076903 1.5347599759697903 6.3131622871296695 2.0038518546498381 5.8326328856492173\nleaf_count=44 91 11384 7910 38380 3809 2282 174 1039 171 2703 25273 139061 29 18 609 119 75 8213 148 3146 38 199 51 161 53 327 11 116 100 219\ninternal_value=-0.00169231 -0.00947612 -0.0157274 -0.00900741 -0.00203629 0.00150273 -0.00124204 -0.0192752 0.00626671 0.00478771 -0.00315267 -0.0265848 0.00322718 0.0101666 -0.00172558 0.00505792 -0.00798847 -0.00253416 -0.00120883 -0.00248271 0.0147218 0.00147819 0.0170475 -0.000275835 0.012832 -0.00262233 -0.022302 -0.0114726 0.00736868 -0.0145275\ninternal_weight=566.532 164.299 107.479 67.5196 45.0503 402.234 255.669 12.2583 146.564 32.792 23.0067 39.9598 116.959 48.2716 68.6874 40.6526 17.4869 240.08 216.695 201.432 15.2631 140.406 13.3336 127.072 18.6154 108.457 22.4693 61.0263 8.31701 52.7093\ninternal_count=245953 229614 218230 79125 40734 16339 15227 4848 1112 35886 33183 139105 941 303 638 285 166 15053 12771 12585 186 9004 250 8754 214 8540 38391 3581 216 3365\nis_linear=0\nshrinkage=0.03\n\n\nTree=140\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 1 2 16 1 6 9 5 10 2 5 4 4 21 4 2 2 1 5 5 4 13 21 18 18 17 0 21 0 11\nsplit_gain=16.6032 11.1566 10.2781 6.71101 5.83573 5.04768 4.70162 3.70463 4.54005 3.46013 5.85456 8.21354 5.35795 4.5121 4.06092 4.39407 4.97279 4.11554 4.75616 4.03519 6.19358 3.72334 6.32487 4.05523 3.71774 3.47231 3.32889 3.32846 3.83407 3.31162\nthreshold=3.0352616308921099 2.5144622974930591 -33.804542541503899 1.0000000180025095e-35 0.24961073363565869 2.1386718750000004 1.0000000180025095e-35 5.3300781250000009 1.4978661367769956 22.129880905151371 1.7006835937500002 1.6286621093750002 1.1459960937500002 1.5000000000000002 5.0736774206161508 15.769127845764162 15.156654357910158 2.9477339485081675 2.1386718750000004 1.7006835937500002 1.8393554687500002 -0.074019523483382962 1.5000000000000002 3.5694334999727624 2.6735537653587351 1.3781395569682247 1.6765212893675572 5.5000000000000009 1.4460303622695712 0.91125284794847328\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 4 -4 26 9 -5 -9 10 11 12 -3 -12 15 16 17 -15 27 20 25 -13 -23 -24 -21 -11 -2 -19 -29 -6\nright_child=5 6 3 7 29 -7 -8 8 -10 19 13 21 -14 14 -16 -17 -18 18 -20 24 -22 22 23 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.021911215470357848 -0.027049029136132767 0.002527355308175857 -0.01982564283075116 0.010695234778743372 -0.0033124645854743324 0.0038062406111692001 0.024390939024163574 -0.028984586786590681 0.0086583072821338515 0.011147900462339606 -0.028094056616131964 0.017513074129495598 -0.019866414471162965 -0.024474871566244092 0.01785570178176428 -0.026853877493471692 0.035330008253756713 0.011856480456952758 -0.019052109973651594 0.022234402769570182 -0.034855057161167186 0.021837378639802917 -0.028535893228940245 0.013883149852524772 -0.006201192539133888 -0.026375637189781975 0.0035627077912615498 -0.032625203853580191 0.005522594499503884 0.019257771188440541\nleaf_weight=23.663258850543571 29.196499725443573 61.587835058453493 18.645008011424579 47.940743422303058 163.59715785748449 6.881166820659927 10.067349608754737 4.2792345745401672 5.6032774890190922 20.116297498956563 5.7531578897323916 50.075340583280195 10.363714987470301 6.4765356425341443 6.9728728038608088 6.237075498327612 2.0691086351871482 20.985751172760505 8.1539690839708765 26.13391442340799 2.6129988045722703 5.751633901614694 5.7644379732664675 2.187153238337487 4.4354189528967245 1.5274369377584651 3.332764277176465 2.8656579183880222 5.5301898140460253 5.0797375068068495\nleaf_count=62303 68888 1232 61118 4372 40489 3709 97 272 195 302 271 436 414 277 61 36 3 143 160 156 83 68 71 59 154 89 377 62 21 32\ninternal_value=-0.000546028 0.000854282 -0.00344847 -0.00158453 -0.00434158 -0.0190879 0.00520451 0.00726906 -0.00880709 0.0043567 0.00250539 0.00603794 -0.000926294 -0.00484591 -0.00216153 -0.00511423 -0.00166391 -0.00414934 -9.81712e-05 0.0109699 0.00169493 0.0137876 -9.13791e-05 -0.0166207 0.0180428 0.00745193 -0.0239 0.00561588 -0.00958448 -0.00252327\ninternal_weight=573.887 534.476 268.808 245.145 187.322 39.4104 265.668 57.8233 9.88251 255.601 200.774 135.73 71.9516 65.0443 59.2912 52.3183 46.0812 44.0121 37.5356 54.8261 24.2567 63.7786 13.7032 7.95159 30.5693 21.6437 32.5293 29.3816 8.39585 168.677\ninternal_count=245950 172976 168781 106478 101639 72974 4195 4839 467 4098 3314 2280 1646 1034 763 702 666 663 386 784 474 634 198 130 310 391 69265 226 83 40521\nis_linear=0\nshrinkage=0.03\n\n\nTree=141\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 4 17 9 18 1 12 11 1 17 18 13 3 2 0 3 17 6 2 19 18 5 7 11 3 2 1 0 7\nsplit_gain=16.9624 12.0745 9.15315 6.61632 5.87224 6.43966 7.00011 5.30919 5.57698 4.93775 4.90415 4.77762 4.27849 3.96513 3.89891 3.74622 3.50594 4.8548 3.5767 3.3096 5.66064 3.70725 3.11403 5.12226 3.68403 3.5711 3.55964 7.27997 4.07675 3.10317\nthreshold=2.6981674883069502 -42.103004455566399 0.84667968750000011 3.9653415636860969 1.0000000180025095e-35 3.7494348669884658 1.0240454605967615 5.5511151231257837e-16 0.8867719982102783 1.2421711908324335 2.5789308873416652 3.2384861814448418 0.14796868535883259 0.53650715206675215 25.282213211059574 -0.58440677301489119 0.43673077644757391 1.5700365906568221 1.2426757812500002 7.8238990306854257 253.50000000000003 3.0677824455408698 1.7006835937500002 0.89599609375000011 0.28781742082600997 0.44825707527455844 -1.7340650558471677 4.0016611463566001 1.6765212893675572 1.7006835937500002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 13 -3 4 5 10 12 -6 -9 -8 -4 -12 -7 -1 19 -5 18 -18 -16 22 21 -21 23 25 -25 -2 27 -27 -28 -11\nright_child=14 2 3 15 7 6 9 8 -10 29 11 -13 -14 -15 16 -17 17 -19 -20 20 -22 -23 -24 24 -26 26 28 -29 -30 -31\nleaf_value=-0.028956079199843318 -0.009261134409737538 -0.024446254111197236 -0.00098473300945449383 0.0037306140701472159 0.018562095199526629 -0.025746024083738848 0.017157137691335738 -0.018114345204746723 0.017996018351091561 -0.0012775115078975999 -0.00028775105742979507 0.029363442008293671 0.015026102700525524 0.0064935195070046419 -0.0059148043918655357 -0.029942914562063508 0.0237503484456182 -0.0064949464766739838 0.023302054052740349 0.0063528598166573353 -0.028216016881269443 -0.018428178462094772 -0.00064417321298092418 0.019744567013880412 -0.0047128435469897178 -0.013732533458108334 0.026196985299638001 0.013964950483291511 -0.011670407561395023 -0.021288903934715837\nleaf_weight=20.878944524576585 14.935684487107208 15.579882365582305 154.24036477465597 3.7944020227660085 24.158177980143591 23.403909785560248 8.284288397362614 8.8034299419159634 5.3713526502251616 22.098866443189767 9.4800046167947603 9.0604223204500141 1.912880707066505 2.8876975908351588 12.869427388926853 10.008101065557636 28.113260550424457 5.2670548883615984 4.1889124009758225 26.52439749496989 6.000177130335941 5.7749953759193877 44.241611556935823 38.727334501745645 6.0447277249477329 14.591521480833761 11.769729844818356 17.899631695589051 2.5811825394630423 9.2859072792925854\nleaf_count=77335 707 35324 18827 3356 1996 66781 2891 734 52 5863 1031 836 16 218 384 25653 207 84 44 253 48 240 867 275 94 402 120 162 13 1074\ninternal_value=6.44834e-06 -0.00440719 -0.00276422 -0.00151896 -0.00046965 -0.00223735 -0.0102102 0.0102729 -0.0041466 -0.00219314 0.000802866 0.0149635 -0.0222115 -0.0245606 0.00606637 -0.0210592 0.0134067 0.0189279 0.00215288 0.00404816 -0.00375346 0.00150176 0.00604137 0.00879752 0.0164117 0.00310577 0.00716014 0.0014876 0.0190695 -0.00767289\ninternal_weight=568.778 329.249 305.482 289.902 276.1 237.767 64.9859 38.333 14.1748 39.6691 172.781 18.5404 25.3168 23.7666 239.53 13.8025 50.4387 33.3803 17.0583 189.091 38.2996 32.2994 150.791 106.55 44.7721 61.7778 46.8421 32.4912 14.3509 31.3848\ninternal_count=245887 241987 164434 129110 100101 97319 76625 2782 786 9828 20694 1867 66797 77553 3900 29009 719 291 428 3181 541 493 2640 1773 369 1404 697 564 133 6937\nis_linear=0\nshrinkage=0.03\n\n\nTree=142\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 12 4 1 1 3 4 18 18 5 18 12 21 14 14 4 4 18 5 3 5 18 18 1 14 2 19 6 3 12\nsplit_gain=17.1532 10.939 7.9723 8.1075 6.88131 5.6001 4.52105 4.31025 11.9534 5.63016 4.72697 6.14867 5.60959 5.12425 4.82427 4.14579 6.33993 4.60847 3.99014 3.98712 7.0806 3.85331 7.94975 4.8267 3.66457 4.89501 5.78179 3.60784 3.52719 3.45318\nthreshold=0.52258841453381233 0.76965903234489019 3.7858886718750004 1.9634439907166827 -0.041910702480567281 0.22136347059745312 7.0112304687500009 4.0689902272260703 3.1984648276080736 1.4980468750000002 2.8029010331479989 0.57536747852547443 1.5000000000000002 0.80526343422453606 0.85380319876708544 1.4716796875000002 1.0693359375000002 1.8688348091416842 4.2265625000000009 0.51817819236196982 2.2500000000000004 4.7317931335534196 5.2600927804663344 1.0240454605967615 0.28287814140777068 2.0782871246337895 253.50000000000003 4.1835937500000009 0.50207005315878861 0.25412329628254166\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 -2 -1 -5 21 8 10 -10 11 15 -13 -14 -15 16 17 24 -8 -17 -21 -6 23 -23 -7 -26 -27 -20 -19 -4\nright_child=2 -3 29 5 6 7 18 -9 9 -11 -12 12 13 14 -16 19 -18 28 27 20 -22 22 -24 -25 25 26 -28 -29 -30 -31\nleaf_value=-0.02580172006950586 -0.0080283559987744512 0.024400176380973016 0.012579727932989237 -0.02760277274483711 -0.023625025395960734 -0.0089859335064807477 0.019925305910679092 0.021134847601791368 -0.0064230497747698751 -0.037771329151548863 0.016720236150267614 0.013282018923286933 -0.045053733945522083 0.017958323319390273 -0.023801120257645261 0.0015908538007131281 -0.021142258568243217 0.002134572645231998 -0.021717642009928973 0.015052231923763029 -0.021382003348700218 -0.020745424169696255 -0.022956378539909505 0.0090459966946604981 0.019928910804900665 0.011187975307373246 -0.031040067568228621 0.0090771964002895143 -0.028554341257539379 -2.0882310705648372e-05\nleaf_weight=26.353879304460861 78.523786950216163 6.7373945589642963 83.487341237376313 4.3300131762734955 17.701624552139037 8.5888680783100408 7.582239023147852 10.908669278695013 15.037069000391059 7.294442199956392 25.154621787631186 4.5503372663806685 4.6796891493722788 3.3189702201634628 5.0242120400071144 51.524286577478051 10.955020534922367 6.8359072150196862 5.8237669126647216 66.389690838987008 4.3050396451726547 5.2124058331901351 16.404449327354087 22.272542775510374 22.328868819284253 7.2114103286876325 3.5487546608783305 6.0435560999176232 5.4116159030236304 25.471986756747356\nleaf_count=110527 57174 107 3548 460 22445 84 1782 422 1164 434 622 50 47 32 51 566 323 302 1313 333 39 18381 19086 5966 152 122 44 171 124 498\ninternal_value=1.09826e-05 -0.0103357 0.00262399 0.000386112 -0.0127198 0.00287781 -0.00820303 0.00348214 0.00263799 -0.0178428 0.00469671 0.00314489 -0.0133522 -0.022942 -0.00704487 0.00475712 -0.00136125 0.00296905 0.00424182 0.00798977 0.0125662 -0.0121379 -0.00708182 0.00286623 0.00771897 0.0121914 -0.00419421 -0.0065484 -0.0125573 0.00966062\ninternal_weight=569.012 114.132 454.881 345.921 107.394 267.397 81.0406 263.067 252.159 22.3315 229.827 204.673 17.5732 13.0229 8.34318 187.099 64.8804 53.9254 19.4496 122.219 70.6947 61.591 43.8894 27.4849 41.6779 33.089 10.7602 11.8673 12.2475 108.959\ninternal_count=246369 179778 66591 62545 179671 5371 69144 4911 4489 1598 2891 2269 180 130 83 2089 1151 828 3266 938 372 65878 43433 24347 402 318 166 1484 426 4046\nis_linear=0\nshrinkage=0.03\n\n\nTree=143\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 17 4 3 2 17 9 7 5 3 3 3 21 3 3 0 13 5 4 17 3 12 19 2 18 21 18 2 1 17\nsplit_gain=16.1534 11.5558 9.17923 7.14523 8.12442 4.78489 4.97887 6.00032 4.74639 4.63773 4.35808 4.26906 6.72228 4.07881 6.7671 4.27151 4.15047 3.9613 3.85698 5.14959 3.68261 3.66466 3.62246 3.58475 5.63389 5.18413 5.48451 4.54755 5.18523 4.52124\nthreshold=-51.587459564208977 3.9653415636860969 0.84667968750000011 0.58795853567030554 29.286581993103031 2.1810207941665634 1.0000000180025095e-35 4.1191406250000009 5.0468750000000009 0.56033297298897644 0.37248792739977959 0.50207005315878861 1.5000000000000002 0.33966381556979741 0.31840721315068282 0.70465417701170696 0.14796868535883259 4.3242187500000009 1.9406738281250002 1.1019619337295856 0.33061855659870582 0.032945457735612756 242.50000000000003 -21.166955947875973 1.2424533248940002 6.5000000000000009 3.3497501700808394 -30.0182991027832 3.3033179733375269 0.15621648314530187\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 -2 4 5 6 10 -8 -6 -7 13 -12 -13 14 15 16 17 -4 -17 -20 -3 -5 -16 27 29 26 -26 28 -14 -25\nright_child=1 20 3 21 8 9 7 -9 -10 -11 11 12 23 -15 22 18 -18 -19 19 -21 -22 -23 -24 24 25 -27 -28 -29 -30 -31\nleaf_value=-0.025585616926967703 -0.017879491611513094 -0.027566917323325588 -0.026677660242335484 0.010689823768906211 0.0057999755072611946 -0.019461231642508067 0.016669847170273776 -0.019905577761716788 0.02687444794016779 0.015555713598696239 0.0049166762144244973 -0.03061607205697308 -0.031998549440318769 -0.022976911043445684 -0.025658511885048443 0.0067133407003857025 0.016567086962960009 0.0059160377762554573 -0.023458961191957627 0.0093680368454388038 0.0020494434219913214 -0.005917737861094341 0.018830999973275527 -0.015020265489012039 -0.035131446744015346 0.0086503745356184547 0.00020438510733619282 0.023061804054015537 0.010475701979138016 0.018231719965037443\nleaf_weight=20.682840495651362 19.713749018055751 15.508555909664212 19.922009669784536 106.64517499114299 74.732030712888445 21.012526877644635 24.187651501462827 3.9328573200391466 10.235030213865683 3.2542696084128684 94.853085849666968 7.1156266317557302 3.1023262313829054 11.961577903020951 0.99804526932712079 19.851485843853879 2.1257121674716464 3.6125932622780956 11.234132838842926 5.9491795423509748 4.7291419597859194 12.812022642363443 12.441453058390833 4.6248430535197249 11.171039899520116 5.358760460454504 5.8378594793757648 9.1704399452428316 6.1170537259895355 10.050430706702171\nleaf_count=61421 29797 39776 60897 2023 9687 13060 1919 346 96 80 5980 643 229 2887 478 6116 21 1538 3659 1056 1068 218 1976 20 190 168 408 171 53 54\ninternal_value=-0.000193929 0.000819872 0.00169926 0.00250165 0.000495326 -0.00184368 -0.000687627 0.0110466 0.00862166 -0.0143355 -0.00207109 0.000924518 -0.00511159 -0.00737363 -0.00465609 -0.0088075 -0.0179697 -0.0216373 -0.00215373 -0.0122035 -0.0208755 0.00887429 0.013979 -0.0013532 -0.00668196 -0.0158845 -0.0236654 0.00917163 -0.00554949 0.00746282\ninternal_weight=562.944 542.261 522.023 502.309 382.852 297.885 273.618 28.1205 84.9671 24.2668 245.498 157.401 62.5484 88.0962 76.1346 62.6951 25.6603 23.5346 37.0348 17.1833 20.2377 119.457 13.4395 55.4328 37.0429 22.3677 17.0089 18.3898 9.21938 14.6753\ninternal_count=246035 184614 143770 113973 111732 101949 88809 2265 9783 13140 86544 7916 1936 78628 75741 73287 62456 62435 10831 4715 40844 2241 2454 1293 840 766 598 453 282 74\nis_linear=0\nshrinkage=0.03\n\n\nTree=144\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 2 9 17 12 4 5 1 5 4 17 1 4 2 17 11 11 5 6 13 13 0 2 19 2 19 13 6 1 2\nsplit_gain=14.4542 15.347 8.20826 7.09046 5.72808 5.27649 4.55994 4.09931 3.93328 3.99601 3.99168 3.91464 3.76368 4.39378 4.19457 3.75371 4.92032 3.69912 4.23814 3.41637 3.29494 5.79842 4.10231 4.80314 3.26461 3.21912 2.83274 2.7609 2.75339 4.23557\nthreshold=0.47260169401963864 -43.562467575073235 1.0000000180025095e-35 5.817026225953116 0.4270257964607716 0.84667968750000011 3.8818359375000004 1.7622051334857016 2.3505859375000004 0.84667968750000011 0.48516060896247715 -0.32709731491006455 1.0693359375000002 6.3334450721740732 1.3227116620644523 5.5511151231257837e-16 0.91125284794847328 2.8623046875000004 1.8691406250000002 -0.62487823464398462 0.14796868535883259 1.2661233611507157 -20.765821456909176 263.50000000000006 -18.451059341430661 245.50000000000003 -1.0000000180025095e-35 4.3886718750000009 2.5144622974930591 -30.420168876647946\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 5 6 -3 8 -8 9 25 26 -7 -6 -14 -15 -4 27 -12 19 -19 22 -22 23 28 -24 -2 -10 -17 -13 -30\nright_child=4 2 15 -5 12 11 7 -9 10 -11 17 20 13 14 -16 16 -18 18 -20 -21 21 -23 24 -25 -26 -27 -28 -29 29 -31\nleaf_value=-0.029696531745606495 0.012937001295122196 -0.021887515291607637 0.017382699598675437 -0.030030088355711942 0.01113703347866871 -0.01889099259127943 0.0020292597708377875 0.0290010927986411 -0.038263033740690441 0.0076175366769146834 -0.020664645762099858 -0.031271610679357034 -0.028772584600812418 0.0064793766272989509 -0.021378956340579951 -0.027551190334222923 0.023944530496772046 -0.020143021194349839 -0.029622214931829961 0.022429270966747003 0.022122479495803378 -0.018071966162618122 0.019354075286006689 0.010936511850729037 -0.0016992076377921987 -0.027876060307764208 0.015371831170615642 0.0025230175616440002 0.011086869144279672 -0.022707288091856789\nleaf_weight=19.414958917369404 1.8783760453734419 12.510546628524024 23.218125693110778 8.6074554029946722 8.0424399631447141 11.441558128439281 7.9694336424581733 13.759132826700805 3.6463217033306137 177.84434613904159 4.2771875598555189 9.1610955121032074 7.8475095900648766 13.342199414852077 6.3885040722088879 4.7113526592074759 3.5623944059479973 1.2285918344132358 1.2315116095705878 9.149340467003638 10.995705428184008 3.5719295758754006 5.9153662493172297 5.1677025039753053 159.87883534566754 4.2879143598838709 0.52156750485301273 4.9981760935697821 4.9439880491845543 6.9560350794345123\nleaf_count=74556 414 29208 2611 25968 111 37011 508 85 215 4630 376 18626 238 130 111 579 47 344 23 148 149 48 2380 1769 43951 1163 2 246 240 202\ninternal_value=-4.7743e-05 -0.00459153 -0.00272229 -0.00475158 0.00507873 -0.00368218 0.00684156 0.0196331 0.00539043 0.00676818 -0.00697213 -0.00254241 -0.00597722 -0.0111832 -0.00313689 0.0103494 -0.0023106 0.000449954 0.00925096 0.0160619 -0.00155034 0.0117392 -0.00261413 -0.0134071 -0.000836516 -0.0153732 -0.0298755 -0.0132799 -0.0196052 -0.00889609\ninternal_weight=556.47 295.055 275.64 239.15 261.414 230.543 225.794 21.7286 204.065 184.011 20.0545 218.032 35.6207 27.5782 19.7307 36.49 13.2719 15.8866 11.6094 10.3779 206.591 14.5676 192.023 26.2288 165.794 6.16629 4.16789 9.70953 21.0611 11.9\ninternal_count=246089 237591 163035 159552 8498 133584 7908 593 7315 6207 1108 104376 590 479 241 3483 872 891 515 492 67365 197 67168 20837 46331 1577 217 825 19068 442\nis_linear=0\nshrinkage=0.03\n\n\nTree=145\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 3 1 2 2 2 9 1 1 2 2 1 1 12 2 18 11 18 4 18 18 12 12 14 18 18 3 4 4\nsplit_gain=21.1864 11.4188 6.81357 5.68853 5.54385 4.54429 5.07466 4.50274 4.37315 7.04944 4.87814 5.89658 4.03256 7.44798 4.21109 4.64162 6.10141 4.32108 3.99883 3.91979 3.88113 3.82318 3.96732 4.40837 3.81219 4.60312 3.74912 3.73979 4.34606 4.32804\nthreshold=1.0240454605967615 5.6958007812500009 0.10728613048115772 3.3033179733375269 44.569120407104499 8.8705892562866229 11.076014518737795 1.0000000180025095e-35 2.1468275719512682 1.8223781763416411 4.837390899658204 1.6481127738952639 1.075454090076706 1.3779212206518803 2.2204460492503136e-16 22.129880905151371 3.0201273556387074 0.79782821954032401 3.4811217321331038 8.9774329662323016 4.3630783395783075 6.3465127852423047 0.026890617030159995 0.88192242588002812 0.65389599190343672 1.7005986908310777 3.4498615536424366 0.27378847774872456 5.6752929687500009 4.0207519531250009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 18 -2 5 21 7 -7 8 9 10 11 -4 -8 -14 15 16 24 -16 26 -5 -10 22 -3 -24 25 -15 -1 -17 29 -29\nright_child=2 4 3 19 -6 6 12 -9 20 -11 -12 -13 13 14 17 27 -18 -19 -20 -21 -22 -23 23 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.014035712770869433 -0.027842747631499649 0.0060769569986628301 -0.0070183651813119563 0.0066027888910384115 0.020048860229490286 0.021576768215071429 0.023286008122486827 0.015577633491577187 0.0016597341595751816 -0.030258620499089093 -0.03518116996179875 0.021970988878802441 -0.033460602215183904 0.012196436734050499 -0.028304475946038524 -0.0089667336796487029 0.012150879492237253 0.0019200793657619508 -0.028434397278635143 0.031022579166864187 -0.020647998051538215 -0.022786006096655868 -0.03126867385329353 0.012507147669678063 -0.03405788575121748 -0.023434288182893701 0.03092852838631396 0.011790872971972555 0.03049891327572523 -0.01853273848852233\nleaf_weight=18.930909395340681 5.4918984724572502 20.639218755219193 31.554890318942853 157.11925808098749 7.7755269957269766 13.29383368542767 6.863231283688946 10.006703695893519 81.909198281828139 12.091372821261755 3.2964507280994431 6.8382702683229573 4.3654693594926348 6.1146481976029516 6.2967056558118228 7.1651999587083983 11.068725156714207 9.9386202767491323 32.710447729285079 5.2491951957345 6.6685952964326125 8.595230140423153 5.3693018675985504 2.4065159913152456 5.4288120349956435 5.0800816147093428 0.88524722235524578 36.306597899907501 9.5960977943614107 3.8763261092244639\nleaf_count=20842 1529 4706 12807 1840 170 618 410 309 2233 2640 872 732 2091 84 119 1216 1214 97 152018 46 1726 34095 677 27 235 197 797 1456 42 85\ninternal_value=-8.36525e-05 -0.0126914 0.00269182 0.00313509 -0.00147974 0.00052045 0.00473047 -0.00295258 -0.00435469 -0.0110175 -0.00485883 -0.00142836 0.00255958 0.00100141 0.00280665 0.00550519 -0.00446166 -0.0107643 -0.0220446 0.00757732 -0.000231463 -0.00641117 -0.000851715 -0.0178392 -0.0155849 -0.00456829 -0.0106377 0.0103455 0.0132471 0.00848667\ninternal_weight=542.933 97.3124 445.62 440.128 44.7858 277.76 125.394 152.365 142.359 53.781 41.6896 38.3932 112.101 105.237 100.872 84.6365 27.6923 16.2353 52.5266 162.368 88.5778 37.0103 28.415 7.77582 16.6235 11.1947 19.8162 56.9442 49.779 40.1829\ninternal_count=245930 213332 32598 31069 39675 29183 7864 21319 21010 17051 14411 13539 7246 6836 4745 4529 1730 216 173657 1886 3959 39505 5410 704 516 281 21639 2799 1583 1541\nis_linear=0\nshrinkage=0.03\n\n\nTree=146\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 4 3 5 2 1 2 0 4 7 4 4 13 4 5 2 3 3 2 7 2 2 10 14 18 4 4 4 0 18\nsplit_gain=14.2597 10.2495 7.9388 4.83098 7.36065 6.46487 5.26557 5.03185 4.77766 10.3713 7.582 7.25438 4.65766 4.50183 7.51513 4.80584 4.3965 4.30359 4.72942 6.08141 3.90589 3.55676 3.55509 3.49518 3.46341 3.24319 4.78702 5.17827 3.76473 3.66936\nthreshold=-42.103004455566399 0.84667968750000011 0.58795853567030554 4.9648437500000009 30.514518737792972 -0.17729835697704338 7.3050365447998056 0.84505259813102007 1.0693359375000002 0.89599609375000011 2.7826617956161503 1.7380371093750002 -0.29868446770448304 3.8073730468750004 4.2832031250000009 -1.3165187835693357 0.50207005315878861 0.31033658837561479 29.286581993103031 1.0000000180025095e-35 -18.836229324340817 -19.986686706542965 2.524928003624769 0.58611746946415877 1.4978661367769956 9.2039954662323016 5.5515136718750009 3.4626464843750004 0.96046990436194013 2.4414009612931857\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=16 21 3 5 6 7 22 -3 -7 10 17 20 25 -14 -15 -16 -1 -10 19 -19 -11 -2 -5 -18 -4 26 27 28 -13 -30\nright_child=1 2 24 4 -6 8 -8 -9 9 11 -12 12 13 14 15 -17 23 18 -20 -21 -22 -23 -24 -25 -26 -27 -28 -29 29 -31\nleaf_value=-0.025404488930040915 0.0010642176115506607 -0.028784512220358008 0.015981690776341297 0.012787720236899454 0.029410176197229028 0.0079406746588369867 -0.027263268866327944 0.005000717804120495 -0.032009932753975269 -0.0098060265553249707 0.011975226203296697 -0.009952715330119221 -0.010816842890491004 0.0074465967372957607 0.0081058698342732997 -0.027576288966105256 0.010950497105158798 0.0073390742828965145 0.0068720555951158561 -0.025249747572467247 0.023173613991942652 -0.02646510661669926 -0.018102488240286724 -0.024780929160148363 0.0051568613358506958 -0.02619512879301971 0.019552594232281874 -0.016185251581579796 0.0052604109771328339 0.02710112040394368\nleaf_weight=22.552836973544803 5.7748834419944641 11.259697380925902 46.286380190169439 17.697420099601011 11.769083785038672 62.157484851607478 3.7862152239904381 5.4776234225928411 12.261311399932309 3.2974071614262384 8.8688323850456054 9.2920609473476343 48.183183822879982 45.167764997196173 4.3847948836373698 9.9128942461174994 6.5168661456264072 6.2968603454610301 9.0257792400043382 20.27169777340896 18.306717367117017 13.999371917592724 3.1695087192638303 2.6662980030669132 64.223903617290489 1.8473739343462501 18.220491297928675 10.229278275655814 30.075615848778398 8.2875786341865005\nleaf_count=77383 9307 51407 241 1465 97 8610 212 4179 23497 911 2012 5563 10050 4969 307 231 151 1637 921 5157 1104 26509 206 349 2012 182 2260 3950 495 341\ninternal_value=0.000656738 0.0018863 0.00275649 0.000685605 0.0109206 -0.000430373 0.0012915 -0.0180691 0.000527455 -0.0012418 -0.0124898 0.00188265 -7.12104e-05 -0.0041004 0.00143064 -0.0168188 -0.0185389 -0.0171761 -0.0113311 -0.01754 0.0179288 -0.0187612 0.00748248 -0.000838723 0.00979342 0.00547437 0.00657288 0.00222455 0.00642768 0.0105473\ninternal_weight=541.267 509.531 489.757 379.247 36.4222 342.824 24.6531 16.7373 326.087 263.93 56.7245 207.205 185.601 107.649 59.4655 14.2977 31.736 47.8556 35.5943 26.5686 21.6041 19.7743 20.8669 9.18316 110.51 77.9524 76.105 57.8845 47.6553 38.3632\ninternal_count=245715 167832 132016 129763 1980 127783 1883 55586 72197 63587 33224 30363 28348 15557 5507 538 77883 31212 7715 6794 2015 35816 1671 500 2253 12791 12609 10349 6399 836\nis_linear=0\nshrinkage=0.03\n\n\nTree=147\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 1 4 4 2 9 13 1 11 12 4 19 2 4 21 18 2 13 19 19 18 18 18 18 4 18 2 21 14 7\nsplit_gain=12.5636 7.08728 10.5217 8.19944 7.91955 4.45447 4.14648 7.03764 4.15898 5.35121 6.10289 7.15208 5.28984 8.0758 4.22822 3.9977 3.62751 7.18714 3.58374 3.57844 3.57795 3.48839 3.84068 3.31637 6.27121 5.11079 3.9101 6.72482 4.19065 3.66569\nthreshold=-0.32709731491006455 2.5144622974930591 1.6286621093750002 1.0693359375000002 -34.692844390869134 1.0000000180025095e-35 -0.53246711157653859 2.3713805701640611 0.91125284794847328 0.21199991874952331 3.7336425781250004 250.50000000000003 -10.988907814025877 2.6340332031250004 1.5000000000000002 2.0126758453675748 32.956295013427741 0.14796868535883259 207.00000000000003 249.50000000000003 5.589117819455554 1.2424533248940002 5.2496487501035594 1.7005986908310777 2.2978515625000004 2.0126758453675748 7.8238990306854257 1.5000000000000002 0.88948809600489676 2.1386718750000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=15 2 3 21 -4 23 18 8 9 10 11 14 -13 -14 -8 -1 17 -11 -6 20 -12 -2 -23 29 26 -26 -25 -28 -29 -3\nright_child=1 5 4 -5 6 -7 7 -9 -10 16 19 12 13 -15 -16 -17 -18 -19 -20 -21 -22 22 -24 24 25 -27 27 28 -30 -31\nleaf_value=0.0062587679836841057 0.0091541418799729567 0.0059301754338526077 -0.022255406211457215 -0.030580632367468719 -0.020250752087567048 0.023876369016640468 -0.029660824892949623 -0.0259236951067566 0.0203082498098579 -0.03494921218618368 0.014848799077940344 -0.018109766440008362 0.024477029316238044 -0.021015641832647248 -0.005469607600092621 -0.024453497191148334 0.0040352292318806583 0.0081603558729400148 0.012257620851302668 -0.020876767853918086 -0.0077425888821787752 -0.01394410772306052 0.020660174022909178 0.0082391273292773386 -0.034420260887413595 -0.0042701715083824513 0.012652972962660125 -0.034783688609227599 0.01344040026689319 0.023674172746245239\nleaf_weight=4.1421787867875564 9.7736874621477892 99.8972809278348 11.290833736231887 16.528499944420673 2.382346487040194 9.7374390952754748 9.3224617956493621 7.8271984488674198 9.0320229339995404 9.686693126648608 44.677500670505196 4.6796664433350115 15.05082005312943 3.6654474180577372 19.202313988002061 22.080840277773859 7.196737100603058 4.681653066945727 44.622487584991177 2.3018784481500907 6.6845572801062181 28.095658909716352 2.3198306851172665 73.805388818844222 4.9004800388356662 38.599537525689811 6.8941797677980494 6.5874057039036398 1.4848919198848305 10.936479791998861\nleaf_count=990 495 711 29175 23838 918 92 9231 244 81 1232 3638 3328 1400 454 12148 101108 116 59 9955 1010 3393 37847 758 1983 83 1214 268 341 84 48\ninternal_value=0.000310163 0.00136377 -0.00212047 -0.0134193 0.00109768 0.00492745 0.0025868 4.11904e-05 0.00171165 0.00024248 0.00305039 -0.00424597 0.007869 0.0149536 -0.0141126 -0.0195505 -0.0130313 -0.0212858 0.0102225 0.0100583 0.0118169 -0.00579038 -0.0107312 0.00407376 0.000851666 -0.00836388 0.00541034 -0.00818184 -0.0254092 0.00787709\ninternal_weight=538.088 511.865 259.022 56.7177 202.305 252.843 191.014 144.009 136.182 127.15 105.585 51.9207 23.3959 18.7163 28.5248 26.223 21.5651 14.3683 47.0048 53.6639 51.3621 40.1892 30.4155 243.106 132.272 43.5 88.7719 14.9665 8.0723 110.834\ninternal_count=246242 144144 139320 62938 76382 4824 47207 36334 36090 36009 34602 26561 5182 1854 21379 102098 1407 1291 10873 8041 7031 39100 38605 4732 3973 1297 2676 693 425 759\nis_linear=0\nshrinkage=0.03\n\n\nTree=148\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 3 9 0 1 10 0 7 1 12 19 0 4 2 2 4 12 12 19 1 12 2 2 2 2 19 4 18 18 4\nsplit_gain=15.2665 9.91068 7.19015 6.39001 5.77287 5.47238 4.39962 4.17798 4.80267 4.51658 4.04926 3.98844 4.42343 5.30846 5.28005 4.34402 3.85381 4.66605 3.8383 3.7092 3.5284 3.48733 5.00582 3.48114 5.56303 4.15403 4.02073 4.58328 4.06848 3.40857\nthreshold=0.24961073363565869 0.58795853567030554 1.0000000180025095e-35 0.84505259813102007 -0.12981918606627474 2.1383330595080281 1.1851094193663554 3.2763671875000004 2.6981674883069502 0.65514425837065626 259.50000000000006 1.5447487307370646 1.6286621093750002 -23.200188636779782 32.956295013427741 5.7535560131073007 0.026890617030159995 0.79622554818162328 263.50000000000006 1.6575619461558879 0.29026868663126176 30.514518737792972 6.8277702331542978 7.8238990306854257 2.9882335662841801 254.50000000000003 1.9406738281250002 4.8242976514536702 5.2755571088971331 1.9753417968750002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 2 7 5 -5 -1 29 8 11 21 -10 12 20 -14 18 -13 -4 -18 23 -20 -2 22 -9 24 25 26 -15 -28 -29 -3\nright_child=1 6 16 4 -6 -7 -8 9 10 -11 -12 15 13 14 -16 -17 17 -19 19 -21 -22 -23 -24 -25 -26 -27 27 28 -30 -31\nleaf_value=-0.026938054170103014 -0.021253699359702671 -0.011836132804564244 0.019464919473961721 0.010928239344440285 -0.026590909918040018 0.0089284563279950994 0.013181071765379167 0.014067766368662317 -0.0020374717603929811 -0.021089205397896001 0.01047482256513005 0.0049492239923678519 -0.023000970765389431 0.020975220518669881 -0.021707859174582282 -0.034453563671483368 -0.022216038055173856 0.016153040830154002 -0.021705948698165611 0.0319101018023824 0.0081590998194887472 0.025146233650131744 -0.02669967703595803 0.0056591815238866364 -0.036245381155846947 -0.026081675001251695 -0.019384679991413824 0.024455099410114145 -0.0092987803976307109 0.0063733508286234128\nleaf_weight=31.354786171200431 32.563204074746864 12.075488623493582 20.738946625289831 11.639268175007599 4.2132066879603363 3.8586050126923501 73.379035842546728 7.620553934699279 112.30918593705428 2.9520959544461212 28.299408230028348 32.584105166708468 12.874089843625823 6.2942173304145461 10.501875082616605 1.7253294854890544 4.6828927938840916 4.7529058371437714 0.71178107522791734 4.6400104900822043 3.5967475560755693 10.931326906429602 2.9563053113452042 29.166834867580292 4.4017562962690127 4.8399051006999789 9.2745352029487567 5.4416582885887674 5.6214350837472011 30.675784954626579\nleaf_count=153948 33559 1075 985 7085 3746 583 593 560 2842 44 331 1204 17398 1064 2454 38 365 68 455 48 347 73 130 5136 1488 1369 3793 1424 2332 920\ninternal_value=-0.000492896 0.00119301 -0.00127305 -0.0159163 -2.96076e-05 -0.0228699 0.00877239 -0.00256953 -0.00353463 0.00912485 0.000536986 -0.00700261 -0.00938998 -0.00584137 -0.00286294 0.00204812 0.0125044 -0.00319328 0.000213234 0.0224846 -0.0181872 0.0140246 0.00135096 -0.00193207 -0.00809019 -0.00315736 0.00180188 -0.00483692 0.00795485 0.00107694\ninternal_weight=526.677 475.611 359.481 51.0659 15.8525 35.2134 116.13 329.306 304.846 24.4603 140.609 164.237 129.928 93.7681 80.894 34.3094 30.1747 9.4358 70.3921 5.35179 36.16 21.5082 10.5769 65.0403 35.8735 31.4718 26.6318 20.3376 11.0631 42.7513\ninternal_count=245457 80095 77507 165362 10831 154531 2588 76089 75282 807 3173 72109 70867 36961 19563 1242 1418 433 17109 503 33906 763 690 16606 11470 9982 8613 7549 3756 1995\nis_linear=0\nshrinkage=0.03\n\n\nTree=149\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 4 9 4 7 13 4 4 4 1 2 4 12 11 19 13 3 2 7 3 11 3 2 2 22 10 1 11 7\nsplit_gain=12.0941 12.6151 8.83514 6.24567 5.62614 4.46644 8.52024 4.49936 6.43561 5.38023 4.51395 4.39368 4.37466 4.24295 5.50348 3.64542 6.34601 3.19188 7.75417 3.3077 3.15011 2.93285 3.35293 2.7876 3.42465 2.6528 2.44978 5.08977 2.68095 2.43406\nthreshold=2.1468275719512682 -15.657813072204588 2.7826617956161503 1.0000000180025095e-35 2.3505859375000004 0.89599609375000011 -0.39822854168506638 1.0693359375000002 1.7380371093750002 0.84667968750000011 1.7090960825073716 22.129880905151371 2.1433105468750004 0.21199991874952331 0.91125284794847328 207.00000000000003 0.14796868535883259 0.29586597258655462 -8.3227787017822248 2.7412109375000004 0.26802264518526847 0.86534941175790137 0.38657275885365588 -29.596046447753903 -30.420168876647946 0.55000000000000016 1.8688348091416842 0.24961073363565869 3.8857805861880484e-16 3.6503906250000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 21 4 17 5 7 -7 9 -9 -3 -11 -8 -13 25 15 -15 -17 18 20 -20 -2 22 -1 24 -24 26 27 -4 -29 -19\nright_child=3 2 13 -5 -6 6 11 8 -10 10 -12 12 -14 14 -16 16 -18 29 19 -21 -22 -23 23 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.025015173640085653 0.019628127606808581 -0.020570986444646271 -0.015360209450789702 0.022879817100689117 -0.03346985369608943 0.027150926923343286 -0.02594794240992828 -0.033491461333743901 -6.3387254444768987e-05 0.01396661439142843 -0.02274049336015822 0.02273617085632057 -0.021674398984293769 0.004330455425795055 0.01420985083015913 -0.038975976667321603 0.012106377410934394 0.0041028806410509759 -0.028325491972269252 0.013826963871102176 -0.023265980679646085 0.017153540972393178 -0.0077686476895359813 -0.021948096111494116 0.029854783674046526 -0.0065358516342027446 0.023691638875878919 0.0068225125817889089 0.024001715626009348 -0.021858238692095862\nleaf_weight=33.275757380253708 7.5996480327303306 8.8759727620945359 7.260126401713932 13.88637574756285 6.8340232947411996 8.1792057159054838 6.4581267952171384 13.260283842388164 8.1492170891406186 14.24587842433357 2.8118985276960293 4.3695445463181386 2.231156602529154 4.5038335195567951 6.2955193234374738 7.6812323126760003 2.3973754865583032 270.53328271763166 10.71286661011254 1.3401668276637781 0.9973167909629409 1.2638119682669628 6.1117522750319031 5.2259201075903539 2.1619318137745713 10.52502568780619 8.2561374139040691 49.058094042238736 9.2422979602124524 2.3116170279681674\nleaf_count=132941 505 23618 11524 143 8036 1458 3127 23683 8537 9593 297 735 520 89 59 853 39 5767 1051 22 198 13 1968 1439 53 2415 188 7042 127 21\ninternal_value=2.66052e-05 -0.00518518 -0.00151305 0.00390812 -0.00928532 -0.00642898 0.00484532 -0.0114822 -0.0213382 -0.00285551 0.00699138 -0.010063 0.00684799 0.00409263 -0.00775791 -0.0175451 -0.0268291 0.00293585 -0.00817967 -0.0229384 0.0127463 -0.0186805 -0.0199875 -0.00672411 0.00422163 0.00708264 0.00908582 0.00691818 0.00990474 0.00380391\ninternal_weight=536.055 228.674 180.635 307.381 75.4153 68.5813 21.238 47.3433 21.4095 25.9337 17.0578 13.0588 6.6007 105.22 20.878 14.5824 10.0786 293.495 20.65 12.053 8.59696 48.0392 46.7754 13.4996 8.27368 84.3417 73.8167 65.5605 58.3004 272.845\ninternal_count=246061 238354 101940 7707 79604 71568 5840 65728 32220 33508 9890 4382 1255 22336 1040 981 892 7564 1776 1073 703 136414 136401 3460 2021 21296 18881 18693 7169 5788\nis_linear=0\nshrinkage=0.03\n\n\nTree=150\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 2 10 1 7 5 12 2 19 17 10 3 11 7 0 17 6 4 3 19 1 1 19 3 14 19 0 0 1\nsplit_gain=12.6104 8.48908 7.29245 5.94562 5.55153 4.51148 5.51733 4.03892 5.89373 4.97039 3.8893 5.45609 3.58098 4.89745 3.71639 3.713 3.81207 3.38373 4.48591 4.15769 3.75097 3.9473 3.30689 3.35447 4.22818 5.75644 3.27175 3.0679 3.30076 5.01152\nthreshold=0.24961073363565869 1.8688348091416842 23.600493431091312 3.6242520361853052 2.9477339485081675 3.2763671875000004 4.4589843750000009 0.091687229702130682 34.417465209960945 209.50000000000003 3.0352616308921099 1.4978661367769956 0.58795853567030554 0.48368796551400856 1.7006835937500002 1.2661233611507157 1.0434576633023316 2.0131835937500004 5.5515136718750009 0.12808136320582797 245.50000000000003 1.7090960825073716 4.0016611463566001 258.50000000000006 0.53650715206675215 0.40422649757387208 251.50000000000003 1.5447487307370646 0.010815665348252143 2.6981674883069502\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 -3 5 6 10 9 -9 -7 17 -12 14 15 22 -14 -17 18 19 -2 21 -21 23 -6 25 -25 -18 28 -23 -30\nright_child=4 3 -4 -5 12 7 -8 8 -10 -11 11 -13 13 -15 -16 16 26 -19 -20 20 -22 27 -24 24 -26 -27 -28 -29 29 -31\nleaf_value=-0.028375425334679703 0.013503889505950844 0.01909859354627513 0.00077920650762410557 -0.032369052299650918 -0.010530658255725063 -0.012672764260265127 -0.03400706758464022 -0.02654324548702738 0.018878547071757484 0.020033558476537448 -0.029383199037832104 0.021673069924499529 0.0014120089940308962 -0.01741676520341531 0.012977228138315847 0.023731665403639451 0.013948525889457485 -0.02912245365696059 0.011147333001353347 -0.022863167823597933 -0.0015847024958677066 0.019583066251599938 0.0047355594781049986 0.02089763157202159 0.019434009756781185 -0.026855643829159496 -0.018335561445657525 0.007960793458040006 -0.028327543533546048 0.0056185755359761157\nleaf_weight=29.78181419340217 12.242020483763552 8.5805215678119549 10.236894420433599 1.603346271898771 37.492259506871051 4.4420427765435333 4.1950019058240278 5.3557818976551044 3.4131076482590288 20.913249989192991 8.4649632574455591 1.57814141842391 17.162671847268939 4.2953635584563008 21.310922223812668 28.178063381928951 6.6186053764540684 3.0349380099596592 22.122069483870291 16.098280795158644 149.05398880415305 1.7969731839548391 53.785594089946244 2.7225271601346348 8.2592861140146834 5.0812357721733861 3.5004410073161125 10.53424391732551 9.9372659994551196 5.6485499857080859\nleaf_count=148237 4894 691 16715 150 1262 202 541 312 31 839 15485 26 219 54 285 105 39 489 2517 15411 34196 249 680 28 53 124 21 247 1911 259\ninternal_value=-0.000825763 -0.0149599 -0.0210804 0.0088258 0.000721591 -0.0019647 -0.00339604 0.00810395 -0.00874102 0.0140402 -0.00272352 -0.0201243 0.00468652 0.0106923 0.00183605 0.0131393 0.0182251 -0.00188158 -0.00139217 -0.00279077 -0.00388824 -0.0114805 -0.00047523 -0.0056993 0.00553369 -0.00971443 0.00213599 -0.00435299 -0.0118424 -0.0164279\ninternal_weight=517.44 50.2026 40.0187 10.1839 467.238 278.831 244.706 34.1242 8.76889 25.3553 240.511 10.0431 188.407 59.7551 128.652 55.4598 38.2971 230.468 227.433 205.311 193.069 44.0153 107.341 53.5553 16.063 7.80376 10.119 27.917 17.3828 15.5858\ninternal_count=246272 165793 164952 841 80479 77609 76225 1384 343 1041 75684 15511 2870 438 2432 384 165 60173 59684 57167 52273 18077 2147 1467 205 152 60 2666 2419 2170\nis_linear=0\nshrinkage=0.03\n\n\nTree=151\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 1 4 17 2 10 2 2 2 14 17 3 9 6 6 1 4 4 3 3 1 17 11 14 1 10 3 21 1 2\nsplit_gain=16.3045 6.68648 7.0461 5.91698 5.51036 5.69641 4.78323 4.65914 4.08891 4.77422 4.49323 3.85025 3.54942 3.40922 6.36533 3.30751 3.2114 3.79024 3.4115 5.72794 3.12421 3.04991 2.94971 2.86814 2.85309 2.8289 2.70154 3.52979 6.90398 2.54295\nthreshold=0.27261778724294111 2.1468275719512682 2.7826617956161503 0.32406367513646889 -14.08206272125244 1.8688348091416842 24.399772644042972 -14.450672149658201 34.417465209960945 0.60348287794770183 1.0587051740042546 0.016076142967085191 1.0000000180025095e-35 4.3378906250000009 2.2500000000000004 0.60367420583093068 2.3505859375000004 1.7380371093750002 0.50207005315878861 0.37248792739977959 1.8223781763416411 0.16595947670675196 3.8857805861880484e-16 0.50603332636230081 1.9634439907166827 3.7839313027319412 0.65908129556435835 5.5000000000000009 2.6981674883069502 21.425971984863285\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 6 15 -1 11 16 -5 10 20 21 29 26 14 -9 -4 17 23 19 -19 -10 -8 -14 -2 -21 -7 -3 -28 -29 -6\nright_child=1 12 3 7 5 25 8 13 9 -11 -12 -13 22 -15 -16 -17 -18 18 -20 24 -22 -23 -24 -25 -26 -27 27 28 -30 -31\nleaf_value=-0.02927979041765074 -0.0096545762810759904 0.0021714335144707702 -0.017883816881719378 -0.027435090293800795 -0.021186041831377442 0.020956700930465529 -0.018922588383204382 0.0012734509030919265 -0.015588521717731691 -0.033625711522051696 -0.024036740099319179 -0.023172788214683462 0.028198038556024881 0.017141045649836404 -0.028179703579418767 0.018807450554062896 -0.033599372438795051 -0.012830475361716007 -0.029602720131522146 0.025696012046318592 0.020726710059326938 0.027266821411987184 -0.0014632697225753072 -0.029148817373231758 -0.019377393048365728 -0.021983424081850951 0.016391322665001151 -0.03468370580476661 0.013093373408585475 0.012902307279346803\nleaf_weight=18.382021985961728 17.770817074830124 256.69444320801995 1.6777045789749561 5.0028242146654511 1.8638867882709806 6.2712887374509583 0.7695463089512512 51.311914938045447 2.9186883194470266 3.8500190250506421 1.4734595749778225 16.170293805112124 8.5131785080302489 8.2065408523194474 6.567230942338937 20.322931857481308 6.3682733804816953 8.3780651292345336 2.19214732236287 7.8232467746638559 3.7540484502824256 8.5724610661382012 4.2802127346512853 11.862420982441106 0.72967498788057117 0.84624457248719398 26.42170850362163 3.2178507114876975 6.7303023452404886 5.3606177011697582\nleaf_count=104428 13324 7389 1196 7318 3752 131 222 6696 1832 2738 1378 53911 70 86 212 2312 6199 7444 323 1049 165 1107 76 17264 67 55 199 52 41 5262\ninternal_value=-0.000363524 0.00134598 -0.00341983 0.0021218 -0.0166792 -0.00839976 -0.0101067 -0.00212938 0.00189971 -0.0109033 0.0142248 -0.0154333 0.00399044 0.000166713 -0.00249028 0.0153536 -0.0147054 -0.0116119 -0.00107417 0.00406451 0.00488082 0.0220002 0.0188878 -0.0181551 0.0202248 0.0137721 0.00327809 0.0108216 -0.00411713 0.00260103\ninternal_weight=524.304 475.41 169.552 93.0891 48.8944 30.5123 76.4629 71.0885 21.3382 10.5228 10.8155 23.3948 305.858 66.0857 57.8791 22.0006 55.1246 48.7564 19.1231 16.931 6.67274 9.34201 12.7934 29.6332 8.55292 7.11753 293.064 36.3699 9.94815 7.2245\ninternal_count=246298 78759 70932 17820 167539 63111 53112 14312 7442 4735 2707 62925 7827 6994 6908 3508 45670 39471 8883 8560 1997 1329 146 30588 1116 186 7681 292 93 9014\nis_linear=0\nshrinkage=0.03\n\n\nTree=152\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 0 12 7 12 2 12 0 7 4 0 12 13 2 7 18 19 19 2 2 13 19 22 4 3 4 19 17 5 18\nsplit_gain=13.5531 8.14537 7.75274 6.86061 4.70864 4.63953 3.96998 3.59371 4.2887 12.2099 6.04782 4.36893 5.91046 4.09479 3.31887 6.73377 5.6932 3.25155 3.14261 3.08877 2.96055 4.24314 3.54659 2.85907 5.34141 5.17266 4.43977 3.48187 2.8666 2.79785\nthreshold=3.9653415636860969 1.3771184689138856 0.63107571507209059 1.0000000180025095e-35 0.027767680665956421 28.11904335021973 0.88192242588002812 -1.035541008535932 0.89599609375000011 1.4160156250000002 0.010815665348252143 0.28848765710462548 0.14796868535883259 -7.5318784713745108 3.1132812500000004 5.2390944889447963 213.50000000000003 213.50000000000003 -8.7471351623535138 34.417465209960945 -0.025789737757327676 254.50000000000003 0.81250000000000011 1.6286621093750002 0.37248792739977959 1.0693359375000002 264.50000000000006 0.16255022815751183 4.5058593750000009 2.8029010331479989\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 23 4 -1 6 7 -5 9 20 -11 14 18 -14 15 29 -16 -7 -13 -19 21 -9 -22 25 28 26 27 -3 -25 -10\nright_child=-2 2 -4 5 -6 17 -8 8 11 10 -12 12 13 -15 16 -17 -18 19 -20 -21 22 -23 -24 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.021119490511803347 -0.023957621361078048 -0.0082888559935787275 -0.013896605506819159 -0.024859700040898582 0.016851312995061579 -0.016759255088726253 0.016096632525492986 -0.014070172206988295 -0.0035804765172911981 0.020294022283409926 -0.02664592462262208 -0.0049404635889781683 -0.019499951918296909 0.020932177854427223 -0.022778660625795529 -0.023211298969729153 0.017068833090978279 0.022593379724892657 -0.039231083372823237 0.0044812710581995235 0.022442204086016958 0.0064434776223581186 -0.021507279524452416 -0.0067917533041888748 0.01328415271057751 -0.019337738702740258 -0.025763388839330301 0.013455861897236581 0.019041375133385871 0.0062324510929581368\nleaf_weight=22.407180040083407 19.298547870790571 7.713632234153919 15.266763121820984 5.518912765287495 2.6062399783641004 3.1680480800550859 9.3711582683026773 14.885561433584369 42.678618177109456 1.9859039067278015 27.262684703667219 4.1388968471874259 2.6430119415745139 5.0730197341181329 2.8540560354158542 9.1641921118743976 22.601794053183767 13.172525145495454 6.0631479135263371 22.713191137707327 8.4961244831574696 20.239540385140572 1.1719936025801878 20.726586922828577 99.290643951615493 7.8092699681292279 2.6797795335005494 29.75792160992205 3.7137561619747421 64.508249425801296\nleaf_count=45686 61497 203 230 21718 355 1354 108 7899 1647 2072 36983 1216 50 47 266 21268 1978 1945 578 4647 1022 3571 395 1114 670 541 48 1006 57 26068\ninternal_value=7.38685e-05 0.001048 0.0059849 -0.00191313 -0.0166675 -0.000577113 -0.00209582 -0.00287232 -0.00224954 -0.00816538 -0.0228933 0.000526533 -0.0130246 0.00643446 0.00231801 0.000136024 0.0119091 0.00890346 -0.0270025 0.0115619 0.0017414 -0.00239878 0.0155979 0.00782299 0.0102178 0.00156234 0.00599846 0.00888054 -0.00227219 0.00232826\ninternal_weight=518.981 499.682 186.958 312.724 25.0134 287.711 248.657 239.286 233.767 74.0418 29.2486 159.725 17.9181 7.71603 141.807 116.351 25.4559 39.0538 10.202 35.8857 44.7932 35.1251 9.66812 171.692 123.731 47.9606 40.1513 37.4716 24.4403 107.187\ninternal_count=246239 184742 3869 180873 46041 134832 126886 126778 105060 51942 39055 53118 1891 97 51227 48983 2244 7946 1794 6592 12887 11470 1417 3639 1841 1798 1257 1209 1171 27715\nis_linear=0\nshrinkage=0.03\n\n\nTree=153\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 4 4 3 0 3 0 3 4 4 13 8 14 2 2 0 2 7 1 4 18 0 1 3 11 13 2 22 18\nsplit_gain=11.1115 8.81006 5.62303 5.57761 4.65485 4.31183 3.99214 3.89132 3.45352 3.71064 4.86151 3.35199 3.63212 3.259 3.2196 7.07006 6.96049 3.96124 3.82455 3.19788 3.10295 3.81259 4.00684 3.96546 3.78698 3.13694 3.09048 4.00824 5.39955 3.54735\nthreshold=1.7622051334857016 -15.657813072204588 0.84667968750000011 5.5515136718750009 0.31840721315068282 0.32384722065649935 0.48555671035752257 1.8987378677022575 0.35479072494832037 3.2895507812500004 1.0693359375000002 -0.45698050410493757 1.0000000180025095e-35 0.33900667957710101 -0.44610977172851557 -3.0241031646728511 -0.53258533561495902 25.282213211059574 1.0000000180025095e-35 2.9477339485081675 4.0444335937500009 0.89587973461402759 1.8987378677022575 2.2520445226790948 0.48555671035752257 0.64604618140914571 -0.21069274056236062 7.3050365447998056 0.93750000000000011 3.0201273556387074\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 -3 4 -2 -5 7 -1 9 10 13 -12 -13 -4 15 16 -10 18 -16 20 21 22 24 -23 -6 -22 -21 28 29 -28\nright_child=3 2 8 5 19 -7 -8 -9 14 -11 11 12 -14 -15 17 -17 -18 -19 -20 26 25 23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.025572177815714917 -0.0097961276378325298 -0.028189433750911329 0.019326240094700776 -0.010861493722470281 -0.00060842658728002151 0.017716175707764029 0.0058591401275682794 0.016925718403581445 0.022937176419139154 0.0078888162693743647 0.0091122630013440835 -0.023947464359691435 0.0096641245487484578 -0.017278453278730652 0.014063251214824926 0.02781847815934782 -0.031761823118101377 -0.0030531460289804054 -0.027931887959044475 0.0081684586578305036 -0.025679523208216339 0.0089445250207650408 0.013024449799538566 -0.0036225959357908625 -0.041094956179653683 0.017927705617268606 -0.012383783754153686 0.0071684330167463067 -0.018433652382369832 0.016449424632839712\nleaf_weight=30.808292035600061 30.22162870712782 6.6382812724932583 8.2126809525107074 4.8976506449980652 3.3488921776879561 30.674602414481342 4.4451557092106659 1.3984064955438942 2.3313071518205106 44.177984897893566 4.4020453956100019 15.338202850317435 2.9421257727262846 2.021090145317884 1.6199745451640399 5.5327486575415588 5.0717557052876145 19.474670726856857 14.301363095084524 104.05592781250016 6.9221420052781459 39.753101194401097 2.5122690809657797 49.849911307937873 4.6194427906302735 1.0777688138186929 5.2795124155236435 32.14764293259941 17.399230009672465 9.1397298185038363\nleaf_count=133430 4156 22972 3356 294 37 283 642 18 177 13855 9327 35674 222 5772 538 498 1939 1729 3556 1204 252 2887 23 2430 36 8 94 394 295 219\ninternal_value=-0.000591318 -0.00686845 -0.00319228 0.00251724 0.00119112 0.0135921 -0.0196779 -0.0232728 -0.00165678 0.00228328 -0.00525088 -0.0128339 -0.0183172 0.0112025 -0.00786046 0.00468519 -0.0138568 -0.0124481 -0.0230804 0.00242476 -0.00159245 -7.44227e-05 -0.0163608 0.00199009 -0.0260446 -0.0184654 0.00500305 -0.000192712 -0.00762182 0.00577386\ninternal_weight=510.616 168.716 132.064 341.899 306.327 35.5723 36.6519 32.2067 125.426 77.0941 32.9161 22.6824 18.2803 10.2338 48.3318 12.9358 7.40306 35.396 15.9213 276.106 108.084 100.084 10.4806 89.603 7.96833 7.99991 168.022 63.9661 31.8185 14.4192\ninternal_count=246317 233705 99615 12612 12035 577 134090 133448 76643 68206 54351 45223 35896 9128 8437 2614 2116 5823 4094 7879 5673 5413 96 5317 73 260 2206 1002 608 313\nis_linear=0\nshrinkage=0.03\n\n\nTree=154\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 4 4 13 1 13 1 18 18 4 11 0 4 18 13 5 1 1 2 3 3 4 14 11 6 21 3 12 1\nsplit_gain=13.3382 9.32847 8.07166 6.33777 8.44709 5.8012 4.7181 4.29036 3.98598 4.07612 3.92453 3.79748 3.48368 5.21502 5.0938 4.84034 3.43795 3.14124 3.08743 2.99155 3.58081 8.42236 7.31386 4.6268 5.71367 2.84398 3.87807 3.6591 3.08985 2.77637\nthreshold=-34.692844390869134 2.9477339485081675 1.6286621093750002 1.0693359375000002 -0.31142306388867069 0.98142480879535021 -0.35100884762992762 2.5144622974930591 1.7005986908310777 1.8688348091416842 2.7487792968750004 0.60365083802632757 1.6765212893675572 1.6572265625000002 1.2424533248940002 -0.30310226004743962 2.1386718750000004 2.6981674883069502 -0.32709731491006455 53.951957702636726 0.58795853567030554 0.41780940659066851 16.653947710990909 0.032916404154818342 0.91125284794847328 1.4980468750000002 7.5000000000000009 0.29586597258655462 0.88192242588002812 1.211689956917019\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=7 2 3 4 -2 -6 18 -1 11 -10 12 -3 13 -8 -14 -15 -12 -9 -4 20 21 23 -23 29 -25 26 27 -11 -28 -18\nright_child=1 8 6 -5 5 -7 10 17 9 25 16 -13 14 15 -16 -17 19 -19 -20 -21 -22 22 -24 24 -26 -27 28 -29 -30 -31\nleaf_value=-0.025939179436242559 -0.020383905556865076 0.012151029581699233 -0.023129778418244422 -0.025962988410424329 -0.01799533200979293 0.011202761403295734 0.010376594314949002 0.016746704442941882 -0.01994067069661401 -0.024402126952017281 0.016207880664725682 -0.01153587182771168 -0.01908352267315595 0.0061106589630110543 0.022919126651108103 -0.025940264146900448 -0.0031209937763230127 -0.011568110631868722 0.0064600968351184824 0.011074078355637201 0.013937871120649434 -0.034272380226476599 0.026336156429054985 -0.010058173800115594 0.022882846315401204 0.020758502416061385 -0.031478432042546779 0.0052013601292679309 0.01752401794191577 0.029568139219657452\nleaf_weight=20.753397395394131 19.987949320946427 85.59695697017014 2.3247756592481883 18.415225825566267 7.1497379405336714 23.959892106979169 5.3568773916304044 3.3375725223304498 6.2948274553054935 2.9979098443582179 9.7048942519340908 5.7797510334348763 3.5529927176830807 4.7527888655058623 5.1402160908764927 22.760733571019959 3.4242786235718068 19.618957343904185 89.907134850359398 12.388494350632753 7.4406802342855363 12.157325652224241 1.3256005449220545 31.69455941117667 4.578105302527546 6.7603352596051982 4.0689866798711583 65.814311369453208 0.79857909679412831 5.3479141604766482\nleaf_count=92503 18345 508 9527 28879 24178 4349 988 230 67 294 2727 86 122 2354 185 20962 1766 834 18651 920 144 555 12 15305 53 70 211 1085 3 125\ninternal_value=-0.000673131 0.000816246 -0.00248847 -0.0113303 -0.00565965 0.00414089 0.000320281 -0.0163308 0.00620609 0.00156781 -0.00336068 0.0105441 -0.0108403 -0.0152122 0.00555453 -0.0203835 0.000251642 -0.00692431 0.00547417 -0.00190395 -0.00447012 -0.00700517 -0.0269065 -0.000593264 -0.00539818 0.00347513 0.0016326 0.00358057 -0.0218226 0.0177941\ninternal_weight=513.192 469.482 291.37 69.5128 51.0976 31.1096 221.857 43.7099 178.112 86.7349 129.625 91.3767 41.5636 32.8704 8.69321 27.5135 88.0619 22.9565 92.2319 78.357 65.9685 58.5278 13.4829 45.0449 36.2727 80.4401 73.6798 68.8122 4.86757 8.77219\ninternal_count=246038 152471 150147 75751 46872 28527 74396 93567 2324 1730 46218 594 24611 24304 307 23316 21607 1064 28178 18880 17960 17816 567 17249 15358 1663 1593 1379 214 1891\nis_linear=0\nshrinkage=0.03\n\n\nTree=155\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 1 3 10 11 17 12 21 4 4 4 12 9 19 4 3 0 0 5 12 19 0 21 2 19 2 19 17 2 2\nsplit_gain=12.8741 7.12229 6.58763 5.64911 4.74797 4.7202 4.36562 3.7104 3.27131 6.608 4.74753 5.04541 5.4621 4.92609 5.47298 7.38942 4.24674 4.00381 5.54929 4.57082 4.44849 3.32722 3.44484 4.47416 3.91924 3.07043 3.04165 2.96989 2.89561 3.55953\nthreshold=1.3771184689138856 -0.17729835697704338 0.53650715206675215 1.7005986908310777 0.77813183983917178 1.8572459393501972 0.63107571507209059 2.5000000000000004 1.6286621093750002 1.1459960937500002 1.6572265625000002 0.28848765710462548 1.0000000180025095e-35 226.50000000000003 5.5515136718750009 0.47260169401963864 0.13844441961022361 0.94290057586474996 2.5244140625000004 0.027767680665956421 247.50000000000003 0.97942481616843191 1.5000000000000002 -7.5318784713745108 261.50000000000006 16.375147819519047 267.50000000000006 1.9205778057410925 1.6481127738952639 7.8238990306854257\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 6 -1 8 -4 -2 -6 9 -3 -10 12 13 -12 15 17 27 19 20 -15 -19 -17 -23 24 -24 -20 -11 -16 -25 -30\nright_child=2 4 5 -5 7 -7 -8 -9 10 26 11 -13 -14 14 16 21 -18 18 25 -21 -22 22 23 28 -26 -27 -28 -29 29 -31\nleaf_value=-0.025609626814901729 0.0030532640511021082 -0.0040247332051083348 0.014691796524975138 0.0096726083747889335 0.02230816048068136 -0.019302367157336599 -0.018766993314576645 -0.002762271230961893 0.012444822479523795 -0.027434607485635393 -0.024602044361579662 -0.020200333672865053 0.012759134495601877 -0.022977129243687926 0.0057500581665798026 0.0080072536208517422 0.016751921119991263 -0.03483061102737061 -0.015888107757092602 0.015232375560074103 -0.00042935532755285422 -0.029723787902850438 -0.036424467954804918 0.028330969089973068 0.0012334319992711967 0.028422985505345531 0.013392395272195439 -0.030139274795556972 -0.030615029094707327 0.0092273365249916518\nleaf_weight=20.369827534901564 98.579766715105507 64.9420854024238 68.352044219849631 4.4881808860300216 10.739335306512659 3.0430455273599355 8.0017262180335802 9.3447151633445156 18.590961563413657 16.983810265652213 8.459774924494921 13.319551444588795 19.852645283259335 29.561808419573939 4.4484091997755995 34.637740880920319 19.586772879058117 4.3344237907367669 1.1263980805888412 2.4202323363497262 9.3175358307635161 3.5723068862571408 3.6845731568755635 3.9351003604533572 5.1686599496169947 5.0433696594336652 1.2184044070017979 2.5123754627893504 1.6351517173461596 7.6186863495677244\nleaf_count=118629 4022 38502 460 727 140 93 132 168 2023 16969 5501 1721 1955 40655 3731 1929 941 1130 207 1049 1125 85 54 29 49 159 228 3681 31 92\ninternal_value=-0.000460637 -0.00399193 0.00601181 -0.0191116 -0.00268868 0.0129963 0.00125587 0.0110656 -0.003704 -0.0086645 -0.00159667 -0.0031057 -0.0016294 -0.00364482 -0.00219862 -0.00510606 0.00981344 -0.0133015 -0.00250916 -0.0197115 -0.0129833 0.00204234 -0.00602162 -0.00105702 -0.0165453 0.0192456 -0.0241466 -0.00936237 0.009773 -0.000111931\ninternal_weight=504.889 326.913 177.977 24.858 302.055 71.3951 106.581 20.0841 281.971 83.1443 198.826 180.236 166.916 147.063 138.604 112.056 26.5476 51.8038 19.8217 31.982 13.652 60.2522 25.6145 22.0422 8.85323 6.16977 18.2022 6.96078 13.1889 9.25384\ninternal_count=246217 241510 4707 119356 122154 553 4154 308 121846 55699 66147 64124 62403 60448 54947 46594 8353 44325 2621 41704 2255 2269 340 255 103 366 17197 7412 152 123\nis_linear=0\nshrinkage=0.03\n\n\nTree=156\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 2 2 4 1 2 5 0 2 4 12 12 21 17 1 3 17 7 17 17 2 7 7 1 2 2 18 2 4 7\nsplit_gain=10.8402 8.29304 6.23899 5.37356 4.88564 4.55571 4.13857 3.86783 3.90623 3.23251 2.6713 4.46689 3.78765 4.2007 2.84663 3.86066 3.38539 5.69168 5.8473 4.71937 3.03516 4.95249 4.79864 4.71436 4.63986 6.22794 6.29596 4.37821 4.25562 4.00029\nthreshold=3.9653415636860969 -51.587459564208977 22.129880905151371 0.84667968750000011 0.15052965826911111 20.814876556396488 6.0488281250000009 0.60985949876859957 22.867874145507816 8.1645507812500018 0.52292651861841544 0.7283458988905912 2.5000000000000004 0.12980400503841663 2.9477339485081675 0.62297257604742995 0.38405652560789177 0.89599609375000011 0.84061941967369014 0.46867156328060416 12.256900787353517 2.6015625000000004 1.0000000180025095e-35 1.8223781763416411 8.8705892562866229 5.8049912452697763 3.0677824455408698 -15.262049674987791 2.9458007812500004 1.2426757812500002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 4 -4 9 10 -2 -5 -9 -3 14 -12 -13 -14 20 16 -16 18 -18 -19 21 24 -22 27 25 26 -6 -23 -27 -26\nright_child=6 2 3 7 5 -7 -8 8 -10 -11 11 12 13 -15 15 -17 17 19 -20 -21 22 23 -24 -25 29 28 -28 -29 -30 -31\nleaf_value=-0.021733863890928563 -0.026420066756937911 -0.023164240635548113 -0.025924986175858082 -0.0097084043317061421 0.0041636678783899417 -0.025553341985592126 0.01416694233847586 0.03317623196655986 0.008272339129537877 0.0019632933994781013 -0.024894147267191736 0.01650306806641744 0.021230718875972873 -0.016773003714575251 0.008321389806517674 0.014224610322722779 -0.030389307829363808 -0.025321450954982033 -0.00094345099026081842 0.0095060907011640348 0.015657799258136276 -0.022148082404749518 -0.018172577802707153 -0.019846805014499366 -0.004002901313718412 -0.040772030934994111 -0.011581714896193324 0.021144538741064756 0.0086916792903517712 0.029805952004782355\nleaf_weight=13.367443159528191 17.137259487150008 12.872065031228887 3.3552316667155528 9.2869067774390697 57.430609954437386 5.0704608990417901 1.9474028604126932 5.142489018609921 109.12472222383076 6.6795612352070739 8.4535231908084842 7.0661026341840651 2.4139199946075669 11.276904385653323 41.632093929685652 29.542450845590793 9.6949249892204552 3.0088022673153345 14.187601560872279 28.254207511956338 3.9402395398647085 1.5232882222517279 17.136418169913668 2.7118039733031756 5.2379705007024295 5.4123079955497806 36.584171528170373 20.182449572108453 1.663895330733794 6.1767468911093601\nleaf_count=40947 61345 68630 5421 7639 4333 703 151 166 6828 1637 433 100 25 185 484 199 214 61 378 297 1535 1110 3464 95 1665 1627 35016 954 386 365\ninternal_value=-0.000187612 0.000717321 0.00140789 0.00705183 -0.000724836 0.000189075 -0.0217794 0.0081483 0.00967121 -0.0149019 0.000685305 -0.0078007 -1.49445e-05 -0.00908326 0.00158142 0.00492033 0.00196367 -0.00289882 -0.0136377 0.00547328 -0.00110217 0.000558186 -0.0114253 0.0125244 -0.00212635 -0.0042148 -0.00202078 0.0172647 -0.0295051 0.015219\ninternal_weight=497.514 478.429 465.062 126.909 338.153 318.601 19.0847 123.554 114.267 19.5516 313.53 29.2105 20.7569 13.6908 284.32 126.32 96.7776 55.1455 23.8825 31.263 158 136.923 21.0767 24.4175 112.506 101.091 94.0148 21.7057 7.0762 11.4147\ninternal_count=246393 184897 143950 20054 123896 53629 61496 14633 6994 70267 52926 743 310 210 52183 1633 1434 950 592 358 50550 45551 4999 2159 43392 41362 39349 2064 2013 2030\nis_linear=0\nshrinkage=0.03\n\n\nTree=157\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 10 0 18 4 10 7 4 2 19 4 18 7 21 5 2 4 19 4 0 18 7 18 3 18 3 18 4 4 18\nsplit_gain=9.84218 8.00305 4.82983 4.64517 4.96541 4.22284 3.78983 3.94117 5.22431 5.29102 4.61691 4.62094 4.38603 3.92743 3.70402 4.79 3.9371 3.65083 3.43272 3.43252 3.42563 3.24339 3.72576 3.01916 2.94116 2.86595 5.56057 4.03454 4.02456 3.74454\nthreshold=0.18474805855422213 0.89587973461402759 1.3771184689138856 3.2769667020129059 3.4714355468750004 2.9174053685313028 0.89599609375000011 1.0693359375000002 18.240050315856937 261.50000000000006 1.9511718750000002 4.0689902272260703 1.0000000180025095e-35 5.5000000000000009 1.4980468750000002 -7.9326825141906729 4.9990234375000009 245.50000000000003 1.0000000180025095e-35 0.010815665348252143 2.1383330595080281 2.6738281250000004 5.2496487501035594 0.29586597258655462 3.1132683346437333 0.3261289997719386 3.7494348669884658 7.3568274974823007 11.379394531250002 5.987706711087152\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 6 4 -1 -3 7 12 9 19 11 -10 18 17 23 16 -16 -14 -2 -9 -19 22 -18 -8 -12 -17 27 -27 -29 -28\nright_child=2 5 -4 -5 -6 -7 14 8 10 -11 24 -13 13 -15 15 25 21 20 -20 -21 -22 -23 -24 -25 -26 26 29 28 -30 -31\nleaf_value=-0.01547345225509222 0.017468299659048021 0.014547579721025199 0.0062852314197274503 -0.023417605424293882 0.016011001994783197 -0.023962372518775973 -0.020532222899900164 0.011787962340279618 -0.024724283236555501 0.0004770487904259288 0.027289697179127791 0.017960354017906785 -0.011874290293223729 -0.01547985011968039 -0.014598407836483951 0.013173577152483195 0.032709256789671043 -0.0037404116584493739 -0.022606850522904579 -0.030394605354897977 0.016882783075991521 -0.0089029811764328465 -0.017588525585952142 0.013000329820998476 -0.016975439695289033 0.0089111774093434386 -0.017638498283129307 -0.031756554678551943 0.012277431856672114 0.017931875212020144\nleaf_weight=10.934701365125877 1.577880078690211 13.216191975251602 160.83006469445536 30.079391768651931 6.2795356363531019 2.1863319555595799 1.712568581952836 1.3886719962811245 6.0073508364494028 8.967910272689549 5.8303383207094148 2.5321967011532251 6.3829987247354429 6.8002376083823028 27.277567238492338 21.485008574325548 4.1874878309390606 9.4553817705309466 9.4918160152701603 18.48265428110517 26.159171135721408 5.7158617317300013 1.0707372516071698 34.381191548265633 0.96397130339755666 46.99706017842982 16.686810405470172 3.7604325737338504 2.3753812583163381 2.3038940322221597\nleaf_count=11727 185 2028 2610 148735 747 357 2446 1260 1483 4331 241 1582 1730 1143 11058 4617 268 142 15664 17041 3562 3406 2517 1779 869 1118 2995 110 50 268\ninternal_value=0.000565743 -0.0104475 0.00218145 -0.0164809 -0.00373984 0.0079536 -0.000259243 -0.00473306 -0.0114229 -0.0187423 0.00264765 -0.0115113 0.000309605 0.00423715 0.00252371 0.000166488 -0.00864527 0.00771549 -0.0159205 -0.0270232 0.0114549 0.00613545 0.0212932 0.0108987 0.0196372 0.00382059 0.000892361 0.005874 -0.0153779 -0.0126421\ninternal_weight=495.521 62.6962 432.825 47.2936 17.2142 15.4025 271.995 104.041 44.1731 28.8392 15.3339 8.53955 59.8675 48.7978 167.954 131.86 38.2517 41.9976 11.0697 19.8713 35.6146 10.9741 5.25823 36.0938 6.79431 93.6086 72.1236 53.1329 6.13581 18.9907\ninternal_count=246069 163594 82475 161209 12474 2385 79865 49233 26807 22632 4175 3065 22426 6577 30632 26407 17249 5434 15849 18301 3704 6191 2785 4225 1110 9158 4541 1278 160 3263\nis_linear=0\nshrinkage=0.03\n\n\nTree=158\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 4 2 1 3 7 12 11 19 19 4 18 3 17 3 18 2 2 14 13 4 3 17 1 7 14 2 4 4 19\nsplit_gain=10.2815 7.16156 5.83071 5.23674 4.78823 4.60443 7.49435 4.69426 3.77617 3.15057 4.67208 3.8943 3.11953 4.3069 3.77024 5.22606 3.22727 4.65902 3.80364 3.36666 2.9732 2.79774 3.36747 3.05089 2.64619 7.02362 4.76661 2.70421 5.08622 4.43723\nthreshold=3.2718994203396332 0.84667968750000011 -34.692844390869134 2.6981674883069502 0.37248792739977959 2.4414062500000004 5.5511151231257837e-16 0.91125284794847328 207.00000000000003 263.50000000000006 2.1750488281250004 2.524928003624769 0.53650715206675215 0.72612140026470018 0.48555671035752257 1.7005986908310777 22.129880905151371 7.3050365447998056 0.88948809600489676 -0.28983840832261526 11.379394531250002 0.36709515926886871 0.09506205666563787 2.9477339485081675 1.7006835937500002 0.82532919407277039 44.569120407104499 2.5541992187500004 2.3505859375000004 240.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 4 5 -3 24 -7 8 -8 11 -11 -6 13 14 21 -16 17 -14 19 -19 -2 -5 -23 -24 27 26 -26 28 29 -4\nright_child=20 2 3 12 9 6 7 -9 -10 10 -12 -13 16 -15 15 -17 -18 18 -20 -21 -22 22 23 -25 25 -27 -28 -29 -30 -31\nleaf_value=-0.017908339425735009 -0.022588473395088852 -0.025665287974361316 -0.021340798140932305 -0.010506024705050259 0.0042048898402674275 0.012305946584983577 0.0072716515794226484 0.012983731902131914 -0.026519267718582956 0.017897116752375458 -0.026752808525667054 -0.02785953576027015 0.011220071251307254 0.010397516445940037 0.00061968147505336879 -0.034518805259940145 0.020910815001442137 -0.031595628413528004 0.021160804941657438 0.00019340309986397725 0.00438493398739677 -0.016946667672095291 -0.020862175115688254 0.013241468968336209 -0.031881024446369338 0.011819615660194557 0.0093078741916537977 0.0054620155757128235 -0.028146454180452737 7.2472785448522473e-05\nleaf_weight=15.971546519269394 20.84906463973039 10.547241813860635 8.6446364910734683 10.608740508410845 6.3964793460909295 50.756399118023182 3.5277445881365592 5.3602342423982909 10.449270954974052 9.2825906910002214 1.7168536862518511 5.7849122854204325 54.492544581764378 39.064619766737451 8.4129049926996213 5.9096759136882602 20.637398049468175 4.9245800022035828 2.3044080697000018 6.0462670689448705 4.0704730140314487 3.7860826721880612 1.7004403664032022 25.609395511331968 11.4678455706935 6.5859180425368313 2.6880588058265849 37.743439876705452 6.206562016704992 89.746972801173797\nleaf_count=31721 69301 53594 13235 320 80 7071 81 58 1191 339 288 1585 429 714 49 108 134 55 16 71 584 61 79 274 5238 3303 149 13959 4329 37825\ninternal_value=0.000208534 0.00122806 0.00194482 0.00292294 -0.00987504 -6.48123e-05 0.00631264 -0.00925015 -0.0180745 -0.00192623 0.0092792 -0.0119808 0.00669937 0.00288091 -0.00245041 -0.0151855 0.0107285 0.00736865 -0.00811923 -0.0155234 -0.0181833 0.00217636 0.00669563 0.0104561 -0.00282763 -0.0125793 -0.0239616 -0.00132878 -0.00380771 -0.00200467\ninternal_weight=491.293 466.374 450.402 416.674 33.7281 233.177 70.0936 19.3372 13.977 23.1808 10.9994 12.1814 183.497 95.0919 56.0272 14.3226 88.4052 67.7678 13.2753 10.9708 24.9195 41.7047 31.0959 27.3098 163.083 20.7418 14.1559 142.342 104.598 98.3916\ninternal_count=246241 176356 144635 88749 55886 86439 8401 1330 1272 2292 627 1665 2310 1605 891 157 705 571 142 126 69885 734 414 353 78038 8690 5387 69348 55389 51060\nis_linear=0\nshrinkage=0.03\n\n\nTree=159\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 17 3 4 17 8 1 18 1 3 1 12 11 10 0 18 10 5 2 4 5 2 3 5 5 21 4 0 18 19\nsplit_gain=10.5367 6.18125 5.15091 5.14799 5.90885 4.44955 4.44766 4.48956 7.06462 4.43784 3.80632 3.4506 6.13867 7.04247 5.79351 3.74345 3.65955 3.13665 3.02556 2.91351 3.55851 3.08617 3.59701 2.79285 3.80242 2.64188 6.66806 9.2667 4.65998 5.10425\nthreshold=-51.587459564208977 3.9281938047451082 0.58795853567030554 3.2580566406250004 1.9205778057410925 1.0000000180025095e-35 1.4475910192202701 2.9174053685313028 0.6973310998193073 0.56033297298897644 1.7622051334857016 5.5511151231257837e-16 0.77813183983917178 1.0000000180025095e-35 0.56830236175476889 6.5213569628644246 1.8688348091416842 1.4980468750000002 -29.210144996643063 3.3457031250000004 3.6767578125000004 10.479034423828127 0.51817819236196982 4.0332031250000009 4.4355468750000009 3.5000000000000004 2.0893554687500004 1.8987378677022575 2.0126758453675748 262.50000000000006\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 3 4 6 -3 7 8 17 -9 23 16 13 14 15 -13 18 -2 -5 -20 21 -21 -23 -4 -25 26 -8 28 -28 -30\nright_child=1 5 10 11 -6 -7 25 9 -10 -11 -12 12 -14 -15 -16 -17 -18 -19 19 20 -22 22 -24 24 -26 -27 27 -29 29 -31\nleaf_value=-0.022948009043616633 0.021543365748749148 -0.027072217001686407 -0.02270317303641026 -0.023074054750077597 -0.023352546650036929 0.0040023649330476795 0.0088043642376772081 -0.026206146369514682 -0.023313580210618596 0.02057912634531725 0.0094207756561874058 -0.039826672218778919 0.012667001036961249 -0.035563275717432487 0.01439815566414664 0.0083729168578312498 0.024960685711485277 -0.019107378438916387 0.027869073711937292 -0.012613553963138326 0.010799979684601264 0.010021720764505214 -0.025259605099112851 0.025523603912481006 -0.018977766138342757 -0.0034671834038555566 -0.028142952279060229 0.026804603532031766 -0.013143432101993891 0.025315536592740048\nleaf_weight=15.930787481827791 8.7298334958095456 10.699276141854172 5.2330627590490604 2.2133016782829591 10.736891343024579 6.0666347209607911 68.679551575831283 15.785495965438713 6.0643490363327137 1.307164505080437 96.045700284535997 3.0370174823838196 12.409249272663144 6.9800095111640976 8.5682289927863149 1.8007681072485864 9.1527403836007561 1.2621612157072366 4.5519688996073446 13.672621489756692 42.937968931748856 17.835178336840276 2.046969005430582 2.8685953227977734 2.1805315142555628 86.603090354408778 11.877527525764895 4.8057590257376424 10.302548028641466 3.2916853385977438\nleaf_count=61459 4161 40922 1200 7719 18444 867 4473 76717 1969 135 1221 616 194 545 180 501 367 1478 796 8941 4674 2621 100 77 197 3774 157 23 1268 75\ninternal_value=0.000311616 0.00113354 0.00179904 5.62418e-05 -0.00261122 -0.0162357 -0.00149133 -0.0114413 9.93251e-05 -0.0216886 0.00755737 0.00492646 -0.00335863 -0.0132498 3.01594e-05 -0.0235244 0.0078712 0.0151373 0.00571857 0.00677654 0.00517792 -0.00212064 0.00535365 -0.00914088 0.0063449 0.000337959 0.00366943 -0.00788741 -0.0151981 -0.0027345\ninternal_weight=493.677 477.746 460.98 354.652 229.446 16.7659 218.709 33.149 16.0563 17.0927 106.328 125.206 32.7953 20.386 13.406 4.83779 92.4107 9.99199 83.258 81.0447 76.4927 33.5548 19.8821 10.2822 5.04913 185.56 98.9571 30.2775 25.4718 13.5942\ninternal_count=245871 184412 142623 139928 112674 41789 94230 84460 7608 76852 2695 27254 2036 1842 1297 1117 25218 5639 24851 17132 16336 11662 2721 1474 274 9770 5996 1523 1500 1343\nis_linear=0\nshrinkage=0.03\n\n\nTree=160\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 2 3 0 18 4 17 17 3 2 17 1 14 3 3 5 3 18 11 11 7 3 4 6 5 1 17 5 1 1\nsplit_gain=15.7215 7.18872 5.51769 5.13652 4.6803 6.60999 3.53189 4.35432 3.4852 3.83744 3.28982 3.23999 3.25235 3.01256 3.30382 2.92603 2.75667 2.84986 2.52408 2.9884 3.50687 2.70571 2.67213 2.40467 2.23788 2.16048 3.24938 2.36536 2.13099 3.86436\nthreshold=0.40160889672495398 -34.252307891845696 0.58795853567030554 1.2661233611507157 0.89587973461402759 3.8417438268661503 1.2596354777617942 0.89189169605791641 0.41780940659066851 -2.5585746765136714 0.59049165877606857 2.3713805701640611 0.35914742995396232 0.082806484040480313 0.22516767725953754 1.0000000180025095e-35 0.29242024582541543 3.9219243190762363 0.77813183983917178 8.8817841970012543e-16 3.4492187500000004 0.3261289997719386 3.9514160156250004 1.7006835937500002 1.2426757812500002 1.6575619461558879 1.5316304627735702 2.8037109375000004 2.6981674883069502 1.1267314367267358\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 8 -4 24 6 7 11 9 10 -2 -6 -13 -9 -15 -8 -16 -18 19 20 21 23 -21 25 -3 26 -7 -27 29 -10\nright_child=2 4 3 -5 5 18 15 13 28 -11 -12 12 -14 14 16 -17 17 -19 -20 22 -22 -23 -24 -25 -26 27 -28 -29 -30 -31\nleaf_value=-0.024908187778141987 0.013060987587244463 0.019724355207329201 0.0019956265016777505 0.015324675215890473 -0.021030958961005718 0.018057858940112331 0.015893263193095274 0.026664414665240783 0.01068543047928339 0.023656558046030899 -0.027598383659228166 -0.018226267998194552 0.0069306730842373758 -0.026951570651193284 0.020353550813969449 -0.023099066964007233 -0.019019706540066197 0.015428343393714569 0.017646991109365143 0.017573066528420826 0.014243826769931708 -0.019637102064856908 -0.020091211455960503 -0.016401483825087021 -0.00065076005618277448 0.010642876463884498 -0.011189117869085807 -0.026027935650208143 0.0029787367865153343 -0.0067101432066605923\nleaf_weight=16.950415072432406 3.0232341154987834 8.2798450790724036 45.039780986378901 61.734480165934656 20.661788572391742 12.165335450500619 1.1719511057672232 3.2315789857411827 13.001956920234077 10.889933017628207 2.2908235367722227 6.7773433475522351 10.516508856773724 2.6806764320226639 5.0148719903136207 25.876459850120227 4.7042100695143745 2.5624875947869441 6.0762049069162449 1.2188699708785851 13.705886467603703 6.2124181779436176 9.6647633243046567 4.6099194793450797 10.037533938084378 2.067596724300528 3.8551857647089482 2.6008410688809818 94.27793350337015 64.560956308825553\nleaf_count=92509 282 413 2277 511 45829 5641 82 2264 3478 839 728 258 471 6101 1922 60786 570 720 107 8 1277 1141 1432 3595 307 135 5417 275 1571 5351\ninternal_value=9.44355e-05 -0.00685286 0.00435998 0.00977287 -0.00484333 -0.00665083 -0.0121295 -0.00761119 0.00125104 0.0141247 -0.00606171 -0.0132844 -0.00338555 0.00429969 -0.00189855 -0.0209032 0.00518725 -0.00647527 0.000756007 -0.00135044 0.00212398 -0.00342354 -0.0147493 0.00115317 0.00910352 0.00564057 0.010776 -0.0107745 -3.75987e-05 -0.00369521\ninternal_weight=475.462 180.643 294.819 106.774 163.692 145.375 83.1979 56.1495 188.045 16.204 5.31406 37.9556 17.2939 18.1938 14.9622 27.0484 12.2816 7.2667 62.177 56.1008 45.2172 31.5113 10.8836 25.2989 18.3174 20.689 16.0205 4.66844 171.841 77.5629\ninternal_count=246297 231260 15037 2788 138751 138031 119003 58135 12249 1849 1010 46558 729 11577 9313 60868 3212 1290 19028 18921 17481 16204 1440 15063 720 11468 11058 410 10400 8829\nis_linear=0\nshrinkage=0.03\n\n\nTree=161\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 1 4 18 1 8 18 12 18 11 7 18 18 14 17 3 19 18 19 19 19 6 19 5 11 22 4 3 13 13\nsplit_gain=10.159 5.73881 5.70896 6.25933 6.74874 5.69497 4.46291 4.17755 5.24429 3.94152 3.71903 3.71851 3.44022 3.40242 2.83493 2.72411 2.69987 2.59633 4.25431 2.83888 2.5582 2.63841 2.47043 2.42147 2.40539 10.4233 2.44508 3.92018 3.52515 3.63054\nthreshold=3.2718994203396332 2.5144622974930591 5.5515136718750009 3.1132683346437333 0.85642036656791032 1.0000000180025095e-35 4.9593358439305844 5.5511151231257837e-16 6.8062759569045816 0.60365083802632757 2.5244140625000004 2.8617925509761908 2.249904835165133 0.35710832892811234 0.92169800002628566 0.048385965359893195 229.50000000000003 2.9174053685313028 254.50000000000003 257.50000000000006 264.50000000000006 0.89599609375000011 246.50000000000003 5.2226562500000009 0.54338456430551896 0.93750000000000011 2.2978515625000004 0.42638606281916264 -0.096683568280545792 -0.277954964655938\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 11 -5 -2 13 8 22 23 24 15 -11 -6 -8 -1 -14 -13 -19 -20 21 -15 -4 -9 26 -26 27 -3 29 -28\nright_child=5 10 7 4 6 -7 14 9 -10 12 -12 17 16 20 -16 -17 -18 18 19 -21 -22 -23 -24 -25 25 -27 28 -29 -30 -31\nleaf_value=0.020967749614817982 -0.027570404860554995 -0.0019837529760687424 0.014473564371090515 -0.026838382736189385 0.0024448487831542393 0.0016685508671562728 0.01728285415105868 -0.037491508356321028 -0.02216149679224496 -0.016753226879580207 0.018872219859282915 0.027808083079093517 0.024043176501349083 -0.027040515699484052 -0.012305854643547342 -0.003192967091307986 -0.005899583244484489 -0.025583845835144881 0.031276192996482717 -0.016789508434818912 0.00071035403781140999 0.013363816344027868 -0.011004508943366272 -0.0023810514124180788 0.011501073015004921 -0.034611495653945987 0.0007646380023465844 0.010368949565309571 0.011404715602330057 -0.018424976865413781\nleaf_weight=3.4254132758760525 16.019901322975784 29.713901209674077 45.278390836541803 18.112957708289287 10.549374929192711 9.0873898050576845 9.3203941495776217 3.4100271937750248 3.3112775599129227 4.2250953086186191 15.942650715995113 5.8808272130281676 6.0546705979213566 14.590900205070286 3.242059910488023 84.205295394072891 3.8675455463817343 2.2857849532792898 2.8030344952458113 0.82909190485952411 4.9217562928552061 1.0108640067264776 2.9848662745825996 3.1920936195801914 10.619033962371761 6.1600874508731058 35.350764178539976 98.869368604660849 17.572098550386727 10.814176777596002\nleaf_count=691 65966 1233 4484 109729 3789 4096 3381 1472 16664 94 181 763 93 11302 2619 10412 170 2245 153 568 1086 91 1625 494 196 83 546 1414 314 246\ninternal_value=0.000877584 0.00191002 -0.00138319 -0.00451862 -0.0119376 -0.017329 -0.00528967 0.00557062 0.0100889 -0.00560737 0.00531802 0.000161783 0.00352293 -0.0112952 0.00930242 -0.00202263 0.0129003 0.0152761 0.000603054 0.0190485 -0.0181433 -0.0237754 0.0127065 -0.023062 0.00419968 -0.0064227 0.00515487 0.00752112 0.000338736 -0.00402589\ninternal_weight=483.651 458.544 233.502 161.178 61.7483 25.1073 43.6353 72.324 51.5745 20.7494 225.042 99.4294 14.1473 31.0729 12.5625 87.6307 9.92222 11.7987 5.91791 3.63213 20.5235 15.6018 48.2633 6.60212 209.099 16.7791 192.32 128.583 63.737 46.1649\ninternal_count=246200 176138 171925 146829 131997 70062 22268 25096 22773 2323 4213 14832 357 16268 6000 11103 263 3729 2966 721 12479 11393 6109 1966 4032 279 3753 2647 1106 792\nis_linear=0\nshrinkage=0.03\n\n\nTree=162\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 17 0 7 12 9 5 6 21 1 6 11 0 5 7 2 2 2 8 5 2 7 10 0 19 2 0 17 2 3\nsplit_gain=10.2696 6.38304 4.7081 5.39253 3.96438 3.94689 5.09598 5.02908 4.33931 3.90524 3.83968 3.14002 2.99244 2.96314 7.08258 6.87663 3.66453 3.33504 4.02651 3.06444 3.86544 2.84757 4.03862 4.50763 3.50387 3.50614 3.06818 2.81537 2.79354 2.73718\nthreshold=-39.283819198608391 3.9653415636860969 1.3771184689138856 1.0000000180025095e-35 0.45050241529338036 1.0000000180025095e-35 4.2695312500000009 2.4414062500000004 5.5000000000000009 1.6575619461558879 2.0131835937500004 1.0000000180025095e-35 1.0212340148387515 0.89599609375000011 0.89599609375000011 27.147908210754398 31.708583831787113 -34.252307891845696 1.0000000180025095e-35 1.2426757812500002 -10.615592002868651 2.5244140625000004 0.89587973461402759 0.40053340798327824 232.50000000000003 30.514518737792972 0.78681748362969484 0.29595754201755942 -6.3279094696044913 0.53650715206675215\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=12 2 3 9 29 8 11 -8 13 -2 28 -7 -1 16 15 17 -5 -15 -19 20 -16 22 23 -21 25 -23 -25 -11 -10 -4\nright_child=1 -3 4 5 -6 6 7 -9 10 27 -12 -13 -14 14 19 -17 -18 18 -20 21 -22 24 -24 26 -26 -27 -28 -29 -30 -31\nleaf_value=-0.021976063189359667 -0.028709333906583534 -0.017190660243079709 0.0044326515588641166 0.010696377934533759 -0.005821791553829541 0.031147148577325083 -0.033946031049554883 0.0073044162134161565 -0.025576480896478664 0.013852201229988868 0.018165301419103361 -0.00016182702112045954 -0.00059534858794620723 0.0079261107040970863 -0.011992063826529672 0.0078738031848753689 -0.02699719635187527 -0.032428660725762136 0.015836914945756352 -0.023708792745359947 0.022850915763200315 -0.023291625921188157 0.026368286670708247 0.0076801538558443194 0.014183300017782986 0.012585212519795183 -0.0058513930707240194 -0.014834646625962837 -0.0059865111103204549 0.012730629216793448\nleaf_weight=21.353601526484457 8.7502349752358004 14.577819810016079 86.678765640521306 37.957219100558177 20.754000368033303 10.531574330949299 2.3941638642063472 11.331000687474441 10.608604524541084 4.6049982841723232 2.8770390116515037 3.8302327063865951 8.0769935114476521 2.1812562159157123 3.0472002334208801 8.3037038434067636 1.5259889186418139 15.27451534320608 1.1290155851893366 8.1235461143223784 11.256910714271724 4.5531660435481189 3.1800567390091592 22.146852321305232 22.728370226730476 3.5333839638333302 42.33732267006053 5.8564879133591603 16.98756555185355 62.09216557230684\nleaf_count=83318 34666 36261 3026 8409 317 1012 774 495 8535 598 315 295 596 1633 793 3049 1055 22701 157 15449 666 1322 230 5958 3704 180 2676 1940 5648 440\ninternal_value=0.00054775 0.00168685 0.00235811 -0.0001431 0.00624662 0.00107731 0.0114569 -0.0017078 -0.000307762 -0.0150405 -0.0106174 0.0233302 -0.0163207 0.00141726 -0.000548858 -0.0141841 0.00860867 -0.0239384 -0.0283876 0.00257496 0.0151297 0.000765392 -0.00236886 -0.00396861 0.0083061 -0.00795557 -0.00115772 -0.00210025 -0.0141484 0.00795784\ninternal_weight=478.584 449.153 434.575 265.05 169.525 245.839 28.087 13.7252 217.752 19.2117 30.4732 14.3618 29.4306 187.279 147.795 26.8885 39.4832 18.5848 16.4035 120.907 14.3041 106.603 75.7878 72.6077 30.8149 8.08655 64.4842 10.4615 27.5962 148.771\ninternal_count=246218 162304 126043 122260 3783 85056 2576 1269 82480 37204 14498 1307 83914 67982 58518 27540 9464 24491 22858 30978 1459 29519 24313 24083 5206 1502 8634 2538 14183 3466\nis_linear=0\nshrinkage=0.03\n\n\nTree=163\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 1 21 2 4 4 2 1 13 2 3 18 14 0 12 11 1 3 21 3 2 19 18 2 19 0 18 2 1 13\nsplit_gain=10.3574 6.1767 4.52339 4.39876 4.32365 7.0581 3.60802 5.88517 3.5075 4.77164 4.28579 4.16557 4.03824 3.10738 3.04107 5.28622 2.79903 2.74259 2.7081 4.62005 2.68676 2.67463 2.6622 2.54102 4.25839 3.36099 4.80415 2.73569 2.62544 2.46258\nthreshold=3.2718994203396332 2.6981674883069502 7.5000000000000009 -58.62553405761718 1.6286621093750002 1.0693359375000002 -15.262049674987791 1.8223781763416411 -0.070829593282168909 -18.054428100585934 0.47260169401963864 0.89587973461402759 0.5668027336126189 1.0212340148387515 0.21199991874952331 0.60365083802632757 2.5144622974930591 0.33061855659870582 3.5000000000000004 0.20268037713056589 39.736396789550788 267.50000000000006 1.2424533248940002 -11.719409942626951 262.50000000000006 1.5447487307370646 0.89587973461402759 -17.275529861450192 4.0016611463566001 -0.53246711157653859\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 23 -1 5 8 7 -6 9 10 11 -5 13 -11 -8 16 -16 -2 22 -20 -10 -7 -17 25 -25 27 -27 -3 -29 -9\nright_child=17 2 -4 4 6 21 14 29 20 12 -12 -13 -14 -15 15 18 -18 -19 19 -21 -22 -23 -24 24 -26 26 -28 28 -30 -31\nleaf_value=-0.025722030766340401 -0.024832881809415695 0.0082362423553228575 -0.0091038769247001085 0.022955019924438786 -0.021659879958347204 -0.027106433752549251 0.0060031688064151227 0.015053698170230485 0.015107755352672019 -0.024685401005234198 0.023300330941988918 -0.02684523184625447 -0.028416014456068658 0.0070636106153742217 -0.03163823266262402 -0.013883453726402142 0.013694599881646003 -0.0034086855649099112 0.015677393659169658 -0.027331220293999651 -0.01913108173590557 0.011462916944479264 0.01704406375099209 0.0079808633311213347 0.02536842648346891 0.014243611123944185 -0.025898361441986949 -0.028168472862131939 0.0049531238433577051 -0.0063661703832967424\nleaf_weight=5.1624670798546495 15.11270252983468 48.640032084687846 15.011715315835316 1.3776448888820541 12.924018018180506 14.501366206239252 128.85806846372179 8.4881844830233621 13.554492408048644 2.9130734518767047 6.4702038150426233 3.4972048293011539 9.2558772941474654 8.5927360469868272 6.3605350475281766 2.3593616604339358 0.88335383066441853 8.6098856138323736 2.951784586563007 4.5078338533057831 1.5281960745669483 1.2560572871152533 10.960274964862036 108.83385778874799 13.761989342514424 3.3355361707508546 6.6825658855959773 2.9821759965852825 4.1516312262974679 8.9992549361340917\nleaf_count=30813 67939 1181 359 5 37709 25999 33857 272 6570 6189 344 13428 12048 889 746 36 9 1989 69 200 2119 442 180 1523 70 24 58 141 44 937\ninternal_value=0.0014113 0.00242557 0.00631248 -0.000682472 -6.5579e-05 -0.00680512 0.00222361 -0.00708077 -0.000847497 -0.00641437 0.00874104 -0.0112598 -0.014717 -0.00230584 0.00406069 -0.00477042 -0.0251195 -0.0175096 0.00315116 -0.0104712 0.0106648 -0.0235314 0.0109788 0.00758252 0.0101222 0.0027695 -0.0124533 0.00567501 -0.0106537 0.00428265\ninternal_weight=482.524 458.801 203.4 255.402 250.24 62.9469 187.293 30.4115 47.1894 32.1067 11.3451 4.87485 20.7617 11.5058 156.881 28.0231 7.24389 23.7226 20.7793 7.45962 15.0827 15.7574 13.3196 188.388 122.596 65.7919 10.0181 55.7738 7.13381 17.4874\ninternal_count=246189 176261 3400 172861 142048 68033 74015 38918 41592 32903 13777 13433 19126 7078 35097 1240 755 69928 485 269 8689 26441 216 3041 1593 1448 82 1366 185 1209\nis_linear=0\nshrinkage=0.03\n\n\nTree=164\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 1 4 17 4 4 4 3 13 13 17 19 14 3 0 14 13 1 22 0 3 17 13 13 14 19 0 5 7 17\nsplit_gain=7.64316 5.05653 4.82369 6.69492 4.75004 5.7001 7.31939 6.58212 4.52779 4.38399 5.61942 3.97658 4.13033 3.97441 5.57724 3.42865 3.91316 3.42187 3.23864 5.54271 3.44986 3.16116 3.1242 2.88175 6.534 2.84955 2.74274 2.74133 2.71044 2.59137\nthreshold=-0.29568658027461547 4.0016611463566001 3.2895507812500004 2.775573723725536 2.2978515625000004 1.6286621093750002 1.0693359375000002 0.65908129556435835 -0.39320322527632096 -0.46714757665816348 0.19767305700411134 264.50000000000006 1.0000000180025095e-35 0.33504387972412514 1.2661233611507157 0.42891595223756424 -0.52657366547836626 1.6109634896136267 0.55000000000000016 0.94290057586474996 0.50207005315878861 0.068176441062887241 -0.52938858054062321 -0.277954964655938 0.088129815934729419 242.50000000000003 1.4460303622695712 2.2500000000000004 3.4804687500000004 0.091221764698383404\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=26 2 3 4 5 6 8 13 9 10 -2 12 -8 -7 29 -16 -17 -6 -4 28 -21 -3 -12 -19 25 -25 -1 -26 -20 -15\nright_child=1 21 18 -5 17 7 11 -9 -10 -11 22 -13 -14 14 15 16 -18 23 19 20 -22 -23 -24 24 27 -27 -28 -29 -30 -31\nleaf_value=-0.023283624930342833 -0.033818785771913128 -0.015754285513626981 0.0052909049400999778 -0.030338583281468226 -0.029873715917708391 -0.011502464418857255 0.016827038018429705 0.026416303561819548 0.0051125609208947454 -0.032208321034040095 -0.0044814996807989267 0.0044777934055246242 -0.031438997386101723 0.017641325691987987 0.032015594470525446 0.023526545804803047 -0.0083249827094103515 -0.0016704452687142379 -0.035384698044695032 -0.022920962029004562 0.012621512408088353 0.008275399420926241 0.02085012381707109 -0.021137584704570885 -0.031288938011844437 0.020548852756269227 0.0062280692729166278 0.0027037940574278175 0.0075962702651164345 -0.0063063995041506117\nleaf_weight=16.329165920013114 3.4996366897230535 4.3472848366945973 108.35757438294422 7.3565802081056964 6.150877163576296 14.205323438246522 1.1518381424957684 10.794977674697291 45.019957527028055 5.3072308958508065 13.495803494952275 4.1850659490464741 14.919614267856558 3.8547514306483235 7.897127253701913 5.4869452219863897 6.6638231198448894 32.063855001571937 5.5995371289027416 2.2269481890361922 9.5988685600459558 77.650523382995743 5.3148442681667758 1.1189368828636386 12.333682524898903 4.4883862848728313 2.9492273265786926 2.362291615419962 1.261308196784739 24.478056390398706\nleaf_count=105122 1680 63 21236 26164 15826 14224 572 113 19190 2639 4595 1474 16534 152 52 73 101 1218 7138 206 70 869 1735 77 650 57 472 201 195 2968\ninternal_value=-0.000675653 0.000151735 -0.00138683 -0.00395279 -0.00295874 -0.000353274 -0.00526557 0.00585246 -0.000724297 -0.0100853 -0.00366662 -0.0207499 -0.0271967 0.0019459 0.00604751 0.0177529 0.00675354 -0.0102317 0.00329624 -0.00795138 0.00466317 0.00682695 0.00336699 -0.00740805 -0.0159654 0.0102902 -0.0186844 -0.0260007 -0.0275219 -0.00255791\ninternal_weight=460.47 441.192 359.194 232.15 224.793 166.275 92.894 73.381 72.6375 27.6175 22.3103 20.2565 16.0715 62.586 48.3807 20.0479 12.1508 58.518 127.044 18.6867 11.8258 81.9978 18.8106 52.3672 20.3033 5.60732 19.2784 14.696 6.86085 28.3328\ninternal_count=245666 140072 139140 110295 84131 66102 48419 17683 29839 10649 8010 18580 17106 17570 3346 226 174 18029 28845 7609 276 932 6330 2203 985 134 105594 851 7333 3120\nis_linear=0\nshrinkage=0.03\n\n\nTree=165\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 1 4 13 18 3 2 18 12 2 22 3 5 3 4 7 3 18 7 18 3 5 4 5 3 12 12 5 13 4\nsplit_gain=12.5512 5.05044 5.04635 5.40524 4.67657 5.51679 4.17239 3.83936 3.51984 3.41357 4.41995 3.26963 3.37491 3.41652 3.1906 3.08756 3.00231 2.83828 4.78019 2.81345 4.42217 3.45273 3.92775 10.7895 5.79711 2.80089 3.36682 3.10202 2.96208 2.94763\nthreshold=0.24961073363565869 2.5144622974930591 1.6286621093750002 -0.34166011882750691 5.0271592211193568 0.25176753112663547 22.129880905151371 1.2424533248940002 0.90934753668501156 16.994721412658695 0.81250000000000011 0.53650715206675215 1.7006835937500002 0.37963534769016083 1.0693359375000002 1.0000000180025095e-35 0.29586597258655462 5.0657496176235997 3.4804687500000004 0.89587973461402759 0.3261289997719386 5.4863281250000009 5.5515136718750009 2.3505859375000004 0.36709515926886871 0.90934753668501156 0.21199991874952331 1.4980468750000002 -0.26241780919347202 6.9193702936172494\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=8 2 4 17 5 14 9 -7 -1 -3 -11 12 13 16 15 -2 -8 19 -19 20 -4 22 23 24 -21 26 28 -28 29 -5\nright_child=1 6 3 25 -6 7 11 -9 -10 10 -12 -13 -14 -15 -16 -17 -18 18 -20 21 -22 -23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.020676993824270479 -0.019876002799420926 0.0019271642578425642 0.019394841038229527 -0.024622373936742582 0.015499947572614395 -0.0047408919595740154 -0.020192048028843074 -0.0297552677402406 0.014231985105677136 0.0059643715224984113 -0.034103118826154355 0.018580898430123491 0.011605754980338926 -0.021829716111120896 -0.026545091396437261 0.016445701712663179 0.022510406783142201 -0.029897913175833481 0.01700039655628785 -0.024893883561644538 -0.032003725418593185 -0.025869990609849181 0.019479470842102872 -0.034396026415541951 0.01417104973656364 0.013000146243704231 0.0085273838130180673 -0.022862488978933764 -0.00050535359164603799 0.011738613348123334\nleaf_weight=33.242831908453809 1.814782665736854 166.45879327423609 1.5907803742447857 8.4964517369571677 4.1135936523351111 8.1976610441925022 1.3110434246191305 19.174079543023254 2.1269880142062894 4.0412637429399165 4.5060088180471212 23.582703377440339 17.024625376434415 6.4152329649077728 1.922618165210223 8.2668901602410187 3.4607501915597823 4.0633686662013124 2.3622367354400922 2.919710511538141 2.9701815214939415 1.5830422051658377 22.959718876038096 4.7291369309532447 30.342348443553419 5.5727397825685321 2.8396912410244139 16.505422861362604 47.877615213771605 1.8978890098805994\nleaf_count=165747 7558 3748 19 5422 3045 258 179 15351 35 135 38 195 324 193 5541 3532 79 6551 276 4973 33 96 762 2113 2429 62 394 1436 15691 313\ninternal_value=-0.00132637 0.000114337 -0.00334948 -0.000812613 -0.0122671 -0.0154799 0.00317318 -0.0226948 -0.0182441 0.000937561 -0.0165182 0.0105662 0.00345977 -0.00887204 0.00145127 0.00870391 0.00931229 0.00507752 -0.0126892 0.00694037 -0.0138129 0.00864761 0.00994723 0.00383465 0.0101155 -0.00601661 -0.00752996 -0.0180621 -0.00378777 -0.0175357\ninternal_weight=462.37 427 200.2 156.71 43.4896 39.376 226.8 27.3717 35.3698 175.006 8.54727 51.7944 28.2117 11.187 12.0043 10.0817 4.77179 73.5205 6.42561 67.0949 4.56096 62.534 60.9509 37.9912 33.2621 83.1898 77.6171 19.3451 58.272 10.3943\ninternal_count=246528 80746 75855 40570 35285 32240 4891 15609 165782 3921 173 970 775 451 16631 11090 258 17252 6827 10425 52 10373 10277 9515 7402 23318 23256 1830 21426 5735\nis_linear=0\nshrinkage=0.03\n\n\nTree=166\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 4 1 12 11 5 2 1 4 0 12 14 18 4 21 19 7 13 21 4 14 19 19 2 1 2 19 19 4\nsplit_gain=8.69114 5.05021 4.52305 8.5113 4.73869 6.50559 4.08311 3.50085 4.47225 3.40051 2.77627 2.55013 2.46517 2.48166 2.44692 5.05721 4.76801 4.38902 3.05814 4.47657 3.05784 2.4752 2.761 2.40176 3.3135 4.32361 3.81352 3.5936 2.37364 3.81732\nthreshold=2.9477339485081675 -58.62553405761718 5.5515136718750009 0.85642036656791032 8.8817841970012543e-16 0.91125284794847328 5.0585937500000009 21.425971984863285 -0.12981918606627474 4.9233398437500009 0.84505259813102007 0.66389400333341531 0.96902457618464355 6.8062759569045816 2.2978515625000004 2.5000000000000004 238.50000000000003 0.89599609375000011 -0.014071314328338634 2.5000000000000004 2.4313964843750004 0.75822041467236145 241.50000000000003 258.50000000000006 25.282213211059574 4.0016611463566001 34.417465209960945 250.50000000000003 245.50000000000003 0.84667968750000011\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 7 12 6 -6 11 10 14 -9 -3 13 -4 17 16 -16 23 -19 -20 -17 22 -22 24 25 27 -26 -2 -5 -30\nright_child=9 2 4 28 5 -7 -8 8 -10 -11 -12 -13 -14 -15 15 20 -18 18 19 -21 21 -23 -24 -25 26 -27 -28 -29 29 -31\nleaf_value=-0.026971567672362244 -0.029506056801034394 -0.029766247395716688 0.010373491499513795 -0.0087818186913313223 -0.031751490346230413 0.0172455259433698 -0.0024373205258198275 -0.01806350938371264 -0.026304259668687907 0.016311052337503091 0.021122661162278542 0.010980484358240252 -0.020640910534285756 -0.015687832895101721 0.011043277842841197 -0.016496552226311974 -0.035600999251420087 0.017285862598535366 0.018693698743537351 -0.022297146632997693 -0.016933679976003514 -0.01302953708908825 0.01209292413073278 0.010583238323496846 0.025385783045573533 0.0016305780661224855 -0.0044167443509634313 -0.00093221547383768525 -0.023494780709240576 0.00039891339020328926\nleaf_weight=7.3251999656013131 9.1890019451020617 18.610245420259169 51.094671172575545 44.96970323003643 8.1335802507674142 5.6257981394883236 8.3805146860231599 1.4149478749532134 4.0619815601394276 18.759467174881138 4.662781170272865 0.99342390727542706 1.7505276810734347 2.6387671598622537 2.2266767743276414 5.7180719507741733 5.8066855084471172 32.024288471962791 3.5372685005422673 4.1125265960581601 2.4381269991863546 4.5087351195688816 25.519081058970187 16.325961845752317 6.3007486004789808 22.969061127689201 7.6818303902982734 6.5907187214615979 5.3352182036151126 119.5337150243522\nleaf_count=51586 533 119259 6505 13229 1169 72 615 10717 10479 225 2267 32 1243 7570 33 93 235 236 64 49 52 97 241 194 86 351 221 181 4418 14105\ninternal_value=-0.000699367 -0.00392271 -0.00321888 -0.00560098 0.00289916 -0.0085827 -0.0178638 -0.0201343 -0.00513129 0.00457265 0.0107366 -0.0272687 0.00755163 0.00883042 0.00305601 -0.00268587 -0.0228809 0.00549889 0.0133175 -0.00337308 0.00203838 0.00569824 0.00906417 0.000881058 -0.00226386 -0.0067582 0.00981071 -0.0183384 -0.00295284 -0.000800532\ninternal_weight=458.239 284.531 277.206 199.582 77.6239 22.1399 16.5141 29.7434 10.1397 173.708 6.07773 19.6037 55.484 53.7334 154.949 46.2174 8.03336 108.731 39.6741 7.6498 38.184 32.4659 27.9572 69.0573 52.7314 38.7488 13.9826 15.7797 169.839 124.869\ninternal_count=246157 243266 191680 174506 17174 1856 1784 142754 23463 2891 12984 119291 15318 14075 2666 751 268 1915 349 113 483 390 293 1566 1372 1065 307 714 31752 18523\nis_linear=0\nshrinkage=0.03\n\n\nTree=167\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 2 4 3 9 3 11 17 0 4 11 11 3 2 0 0 4 0 7 0 5 7 2 21 19 5 17 0 2 2\nsplit_gain=7.24919 6.23271 5.48838 4.37234 3.40237 4.59301 3.89953 3.16692 2.78323 3.27705 3.35897 4.59149 2.44729 2.71881 2.61112 5.23395 4.58631 3.23152 3.53701 3.08964 2.75523 3.34613 2.68287 5.50957 3.09813 2.8284 2.59933 3.9774 3.30946 2.46749\nthreshold=0.58795853567030554 -51.587459564208977 0.84667968750000011 0.53650715206675215 1.0000000180025095e-35 0.31033658837561479 5.5511151231257837e-16 2.1677456597340812 1.8987378677022575 2.1186523437500004 0.11679098783484217 0.8867719982102783 0.3261289997719386 8.8705892562866229 1.3771184689138856 1.0659450432993143 2.6030273437500004 0.69651904584641833 3.3837890625000004 1.3170161875856263 4.2558593750000009 1.8691406250000002 8.3473243713378924 2.5000000000000004 235.50000000000003 1.4980468750000002 0.69259390652607344 1.2225960941547209 -42.581604003906243 57.827308654785163\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -2 -1 5 -3 7 -6 9 -4 -11 -12 13 -7 15 17 26 22 20 -18 21 -19 23 -14 -25 -24 -17 28 -28 -23\nright_child=2 4 8 -5 6 12 -8 -9 -10 10 11 -13 14 -15 -16 16 19 18 -20 -21 -22 29 25 24 -26 -27 27 -29 -30 -31\nleaf_value=-0.027904998278812973 -0.030528348208518401 -0.007537251888720644 -0.00028415146845528288 0.013834713313067281 0.02040565240885844 -0.0073510751951294815 -0.005038800338699489 -0.010843374893828693 0.021725807798322765 0.013991057402022215 -0.028588549344865665 0.017492572818817619 0.011909708892140767 0.022324003899542318 0.0040879208838003681 -0.030232016069994413 0.0075772722760808825 0.0071779295711944663 -0.025328932666717343 -0.028591695420800249 0.024207620461600948 -0.02905895481356394 0.0013133144586061694 0.014185470815192691 -0.017648309296554091 -0.029551349694331489 0.019547432492901428 0.0060883337616023693 -0.031352790102484547 0.014905548094427425\nleaf_weight=11.321923542196599 2.365971819330297 85.864503376778885 36.557441132084932 2.122572473017498 18.066299237845673 3.4383559048983425 11.993674531695431 2.7974594563011115 11.966695827664806 47.304101843539684 3.3504614288685834 2.6253846921026707 15.233882695656574 7.9841262600366463 80.215784526400967 10.616396451740004 12.172787921415871 42.644707880364876 2.174331694841384 1.5380549296387478 6.6239984567509955 3.2239793836051822 4.1960260250853008 2.5694294683489742 14.233788365983854 5.9823525498832169 0.67121153423795288 6.8597301941481419 6.161718384395499 0.87475881678983558\nleaf_count=61096 478 152876 984 109 1612 2357 1002 1220 91 1135 135 18 4083 826 1099 494 304 3920 66 62 81 249 2454 1003 5309 1741 9 290 421 11\ninternal_value=0.000517315 -0.00150515 0.00744653 -0.0208137 -0.000697376 -0.00166888 0.00834117 0.0159114 0.00861745 0.00665142 0.0113277 -0.00873816 0.000570119 0.0135186 -0.000173789 -0.00270304 -0.0119929 0.000994816 0.00570613 0.00185214 0.00741321 0.00457609 -0.00519235 -0.000752964 -0.0122571 -0.0178482 -0.0195664 -0.00980213 -0.0244956 -0.0185929\ninternal_weight=463.752 359.582 104.17 13.4445 346.137 313.28 32.8574 20.8638 101.804 89.8374 53.2799 5.97585 227.415 11.4225 215.993 135.777 38.0199 97.7573 55.5418 13.7108 53.3674 46.7434 42.2155 32.0371 16.8032 10.1784 24.3091 13.6927 6.83293 4.09874\ninternal_count=245535 242694 2841 61205 181489 177655 3834 2832 2363 2272 1288 153 24779 3183 21596 20497 1580 18917 4327 366 4261 4180 14590 10395 6312 4195 1214 720 430 260\nis_linear=0\nshrinkage=0.03\n\n\nTree=168\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 4 17 2 12 18 2 1 17 2 17 12 11 4 2 19 13 1 9 17 12 2 19 13 3 2 1 3 5\nsplit_gain=10.9577 8.16894 4.90338 4.22926 4.82215 3.9194 3.37713 4.37835 5.0607 3.50469 3.26553 3.83079 2.84132 4.28286 3.33365 4.55712 2.80169 2.76469 2.7393 2.67579 2.9059 2.98497 2.6231 4.50336 3.08839 2.60226 3.63845 2.52265 2.51697 2.40155\nthreshold=-51.587459564208977 2.6981674883069502 1.6286621093750002 2.5380284121401901 27.147908210754398 0.29026868663126176 2.9174053685313028 -10.988907814025877 0.39417715631504779 1.0329219259280322 -15.262049674987791 3.9653415636860969 0.24649637830053225 0.91125284794847328 2.0893554687500004 53.951957702636726 264.50000000000006 -0.38214989186364662 1.7090960825073716 1.0000000180025095e-35 2.1677456597340812 5.5511151231257837e-16 6.8277702331542978 241.50000000000003 -0.26992354677691938 0.65908129556435835 12.778713226318361 0.52819557638054959 0.23756184665784369 1.2426757812500002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 5 4 22 6 7 -2 9 -9 18 12 19 14 -14 27 -8 -10 -4 -12 21 -21 24 -24 -3 26 -25 -16 -29 -28\nright_child=1 3 10 -5 -6 -7 16 8 17 -11 11 -13 13 -15 15 -17 -18 -19 -20 20 -22 -23 23 25 -26 -27 29 28 -30 -31\nleaf_value=-0.026073183390976924 0.0095068389864808121 0.010452357647092727 -0.018759687439095722 -0.015118335352463656 0.016525541610257106 0.012446843592470844 -0.027656172518085657 0.030662597369541751 -0.032549186030332314 -0.017868526975456878 0.0025787856731085248 -0.021237003266883051 0.012677629258764016 0.013381510634244462 -0.0016903348492262734 0.01122468753031477 0.0026819970970786552 -0.0095824652379403654 -0.00060431099443720429 0.022478297226067083 -0.012272708951530107 -0.015605559528737189 0.015645043927893641 -0.030387230714429465 0.00026482684229146044 0.010319766660563638 0.0096863486136728062 0.0048767077500377638 -0.038944406637667524 -0.012647657490213576\nleaf_weight=12.500445670686529 10.10258255431472 70.947589088609675 12.854602649193113 7.2861253606752134 36.513308626395883 5.6630348202816085 14.842515917531275 2.7827024282378288 7.5587296048961461 1.2689909937307673 115.01646783556498 5.2727509066535267 2.8717651759143399 6.2899217910598955 2.9347603398691708 2.9852138356072819 3.1104991662468811 13.410089067849183 16.40477041909071 15.70789649588915 2.4942465369385891 1.3771915916986461 8.2979784604394791 6.2192454672185766 42.746607132328791 4.7179692189674816 5.9854077657219049 1.1136651608685486 8.292010720484539 10.687016994925214\nleaf_count=61507 2091 1015 43335 544 552 293 73864 1482 1086 2278 34547 11792 76 79 496 48 4289 2272 1571 693 255 139 135 125 903 39 107 43 430 125\ninternal_value=0.000556635 0.00136011 -0.00219288 0.00598358 0.00688607 -0.00970644 -0.0122566 -0.00652407 -0.0131135 0.0149344 0.000124747 0.00178224 0.0026731 -0.00654629 -0.0137726 -0.0192018 -0.0225357 -0.0189661 -0.00893693 0.00438631 0.0143631 0.0187153 0.00443925 -0.00257529 0.00665777 -0.00840678 -0.0125367 -0.0268518 -0.0337858 -0.00453533\ninternal_weight=458.256 445.756 252.354 193.401 186.115 58.7391 53.0761 35.1231 25.0205 4.05169 193.615 164.356 159.083 24.4873 18.1974 15.3257 17.953 20.9688 29.2594 134.596 19.5793 17.0851 149.602 35.9076 113.694 27.6096 22.8917 12.3404 9.40568 16.6724\ninternal_count=246211 184704 181159 3545 3001 87655 87362 9209 7118 3760 93504 48598 36806 1172 1093 1017 78153 3358 44906 35634 1087 832 2449 531 1918 396 357 969 473 232\nis_linear=0\nshrinkage=0.03\n\n\nTree=169\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 3 6 17 3 6 4 10 3 9 4 18 18 1 13 4 4 4 1 14 5 3 3 19 0 19 0 0 14\nsplit_gain=7.04265 4.63925 4.35561 6.60513 3.16475 3.40442 2.99029 3.14343 2.64692 2.60818 2.37322 4.72545 3.31639 3.82506 3.16109 2.99762 2.5937 2.31758 4.88942 4.06593 4.11311 3.99857 3.48863 3.85243 2.32943 2.27991 3.20707 2.40392 3.98465 3.58816\nthreshold=0.60367420583093068 0.84667968750000011 0.016076142967085191 3.4492187500000004 3.2718994203396332 0.063303256847330999 3.7011718750000004 12.005371093750002 1.4978661367769956 0.56033297298897644 1.0000000180025095e-35 7.3007230758667001 5.2600927804663344 5.1503930356556173 2.1468275719512682 0.14796868535883259 7.6279296875000009 5.5515136718750009 6.6723632812500009 1.211689956917019 0.29138017841767527 5.1210937500000009 0.58795853567030554 0.38657275885365588 235.50000000000003 1.3771184689138856 230.50000000000003 0.97942481616843191 0.92717806344988862 0.62855356909667792\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 -1 -4 5 -3 7 10 9 -6 12 -12 13 17 -13 16 -14 19 -19 20 -7 22 23 24 -20 26 -21 28 -28 -29\nright_child=1 4 3 -5 8 6 -8 -9 -10 -11 11 14 15 -15 -16 -17 -18 18 21 25 -22 -23 -24 -25 -26 -27 27 29 -30 -31\nleaf_value=0.0075260669046737193 -0.014745215082478759 -0.028455267676871513 -0.0195543688108053 0.0081551009884329052 -0.027045072954496558 0.0082643006916771813 0.021174339715767685 -0.030037389685019601 0.012591676309514023 0.0051615372478027262 0.023627056385865007 -0.023152123659964385 -0.030895531975533876 0.02513291063357696 0.018605352017789786 0.016046818995854289 0.008494514329704694 0.019526886214078061 -0.020063234484064694 -0.02733450020485027 -0.02994217262308756 0.017779663458448895 0.013178492940670356 -0.038143947323513713 0.015941159572563673 0.0058795993340577902 0.001806820326856194 0.0018793304131811159 0.020152708686390446 -0.011749952592957637\nleaf_weight=10.579187739956977 12.79873455263805 1.9953742547622813 33.170460062572602 9.3953899087737209 6.9575720138227535 3.4430666079388157 7.6621481000693157 1.5398049006180361 2.4894568707677527 2.767348863533698 12.244007149745814 3.4962470317695979 5.5015909499288664 6.1742921482300508 1.6580135654658077 1.5653480812907217 1.4798774235405288 20.210077317766263 1.9360466540092618 2.6634016771604356 5.9138506761892131 5.284688253508647 3.0636963665019712 4.2099556755856611 2.6008088113740087 117.64265273455385 76.887744368912536 49.265074642506079 11.419292894985118 25.67997846846265\nleaf_count=14129 10687 529 176164 1149 7930 1912 134 52 47 104 356 267 5862 1364 14 35 222 517 105 1088 10701 73 20 117 60 1368 8960 926 338 877\ninternal_value=0.000991023 0.00236489 -0.00916539 -0.0133914 0.00296924 0.00347145 0.00371884 0.00329425 -0.0114897 -0.0182535 0.00351818 0.0140379 0.00294854 0.00343414 -0.00887912 -0.014522 -0.0225278 0.00295378 0.0099001 0.00203843 -0.0164428 -0.00193094 -0.01145 -0.0205435 -0.00027191 0.00270392 0.00043046 0.0010471 0.00437828 -0.00288398\ninternal_weight=451.695 398.55 53.145 42.5658 385.751 373.537 371.542 363.88 12.2144 9.72492 362.34 17.3983 344.941 336.395 5.15426 8.54682 6.98147 330.22 37.3053 292.915 9.35692 17.0952 11.8105 8.74681 4.53686 283.558 165.915 163.252 88.307 74.9451\ninternal_count=246107 54665 191442 177313 43978 35897 35368 35234 8081 8034 35182 637 34545 28426 281 6119 6084 27062 892 26170 12613 375 302 282 165 13557 12189 11101 9298 1803\nis_linear=0\nshrinkage=0.03\n\n\nTree=170\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 0 12 17 18 17 22 21 14 2 2 4 2 3 0 18 2 4 4 12 18 2 17 14 4 2 21 4 19 2\nsplit_gain=9.79053 5.50467 4.05651 3.06377 2.86495 2.95719 3.10654 3.29194 3.25418 2.85322 2.51503 5.98209 4.35871 4.30249 3.27472 3.21509 3.03428 2.77679 10.0965 5.03155 2.8151 3.89921 3.72053 2.80875 3.80935 3.9569 3.92198 3.74156 3.29043 5.27135\nthreshold=-51.587459564208977 0.69651904584641833 0.83259803690661693 3.9281938047451082 3.8393944990995772 2.4775242203633088 0.81250000000000011 3.5000000000000004 0.67646516896599407 29.286581993103031 -10.210010528564451 0.84667968750000011 -17.275529861450192 0.25176753112663547 1.8987378677022575 3.1132683346437333 -19.19599628448486 1.6286621093750002 1.0693359375000002 1.0000000180025095e-35 2.249904835165133 14.537487030029299 1.2428969707451916 0.96868206101997334 1.6572265625000002 5.317040443420411 2.5000000000000004 3.2895507812500004 254.50000000000003 7.8238990306854257\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 4 10 -2 6 8 -8 -6 -9 12 -12 15 -14 -15 -3 -17 18 -13 23 21 -21 -22 24 -19 26 -26 28 29 -27\nright_child=1 3 -4 -5 5 -7 7 9 -10 -11 11 17 13 14 -16 16 -18 19 -20 20 22 -23 -24 -25 25 27 -28 -29 -30 -31\nleaf_value=-0.02469386343281638 -0.023319537174830599 0.0049612918442906405 0.019663460001147694 -0.019237561127812382 0.0052037924600584406 -0.016518466203781279 0.0023996771748515402 -0.025666449164561272 -0.011287597855607311 0.01592511107285164 -0.024114582098448613 0.0063777881149123423 0.013900499417263094 -0.024399629673470032 0.015396013103236116 -0.028060767764878054 0.012831516612258139 0.025121237993433312 -0.023836791297355546 -0.036322146153542997 0.015862986145168227 -0.0032264919434269955 -0.016723918739500045 -0.017216371974983009 -0.01486672717806861 -0.037562231200629043 0.018983497283403344 0.010969241828659411 0.010788756547151876 -0.0012882297562370069\nleaf_weight=12.567350834298848 7.0931414782289712 57.171581360104028 4.8409413839690378 4.4827158815769499 45.25423854341534 14.641269362681667 8.507076363182934 11.197068964035678 13.048710692807616 1.0127242311464204 5.2713451236350002 44.609693121265082 3.4790542187092823 14.020672956330596 1.4395817839540548 4.3567439882463068 1.6584874041955115 12.897387428103682 11.732349766334663 3.4941183370392563 9.0992891889254661 11.967553648864849 3.651711445687396 2.622195416406611 2.6263672937075162 3.0387812489279886 36.997515739238224 51.757423253933666 15.781628155531505 34.391018016642192\nleaf_count=61504 6681 3421 86 7003 34349 45651 6778 18389 34815 91 5708 2334 1850 786 14 3501 295 227 4345 110 217 199 136 577 1044 170 1192 1465 379 2784\ninternal_value=0.000410437 0.00117811 -0.00476862 0.00305256 -0.00612422 -0.0045894 -0.0022003 -0.0118889 0.00138478 -0.0212723 0.00341032 0.00490212 -0.0011341 -0.0133745 -0.0199838 0.00268598 -0.0165642 0.0056229 -0.000219898 0.00736285 -0.00419906 -0.0124578 0.00599302 0.00941354 0.00996275 0.00844739 0.0164291 0.00530783 -0.000299651 -0.00513397\ninternal_weight=454.71 442.142 105.595 336.547 100.754 93.6611 79.0198 20.7169 58.3029 12.2098 332.064 249.938 82.1261 18.9393 15.4603 63.1868 6.01523 244.667 56.342 188.325 28.2127 15.4617 12.751 160.112 157.49 144.593 39.6239 104.969 53.2114 37.4298\ninternal_count=246101 184597 146840 37757 146754 140073 94422 25258 69164 18480 30754 20887 9867 2650 800 7217 3796 15179 6679 8500 662 309 353 7838 7261 7034 2236 4798 3333 2954\nis_linear=0\nshrinkage=0.03\n\n\nTree=171\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 18 7 11 13 7 1 5 1 5 2 13 17 2 14 2 17 21 11 11 2 2 7 18 19 14 1 9 17\nsplit_gain=8.67475 7.23601 4.18232 3.72876 4.07016 3.91447 3.39969 4.39964 3.38085 3.40282 2.92897 2.72801 2.57818 2.45521 3.16454 2.45191 3.2026 2.9841 2.73872 2.44647 2.44176 2.35978 3.08888 3.08176 2.56907 2.24599 2.20999 2.19509 2.14648 2.19984\nthreshold=2.5144622974930591 -15.657813072204588 1.2424533248940002 0.89599609375000011 0.59460817960898271 -0.31426444304395862 1.4980468750000002 2.1468275719512682 1.7006835937500002 1.3779212206518803 4.9648437500000009 39.736396789550788 -0.36167213950920335 0.67078623592565922 -18.836229324340817 0.43714636302263388 25.282213211059574 1.1976019298503242 3.5000000000000004 8.8817841970012543e-16 0.91125284794847328 1.0000000180025095e-35 16.375147819519047 2.6015625000000004 6.2567088148180039 209.50000000000003 0.043928974430191954 1.8912989126705502 1.0000000180025095e-35 3.1006363674515387\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 12 4 8 6 11 21 15 -10 20 -5 -1 -4 -15 17 -17 18 26 -12 -7 23 24 -8 -23 -21 -3 -18 29 -2\nright_child=28 3 13 5 -6 10 7 -9 9 -11 19 -13 -14 14 -16 16 27 -19 -20 25 -22 22 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=0.021393733687302897 0.0036881547970064672 -0.020886553149563499 -0.01072066944981849 0.018108932578637296 0.020577313026694039 -0.0086036613541510751 -0.020690632801922286 -0.034028165984963145 -0.018446248420592153 0.014729435400819899 0.022459999909830507 -0.021488333357080829 -0.015606308493874718 -0.031951676613166495 0.0001480262583569606 -0.032246964472341051 0.0048314528517423129 -0.01991408954731802 -0.023533776847569351 0.013889425942849877 0.012920392812297845 -0.024567296067031313 0.005982763274231937 0.019608391162990537 0.017957107479216496 -0.016325470645450334 0.012491776397303299 -0.028256313936507672 0.018270438309365616 -0.015306982729195423\nleaf_weight=3.5297019218705827 200.82484643081261 1.1856192773520913 20.598693393145368 21.252879694643529 3.1672924702579612 51.248883910841869 1.1725005966281994 2.0057909083261611 3.16342163502304 7.5232997423809138 5.3458139850517892 0.86017611239731095 1.8217463714536268 15.212128489922179 3.3211182892155202 10.676849897216927 5.2594427211533903 8.986226857971813 2.1646687015099806 3.0722114980453625 4.3334601684473446 5.8221218160233574 13.237625145475247 10.300071369154464 0.81191961279182567 3.8928923885650875 10.419893196098199 1.5832262128678847 7.9218799562659106 4.6700805111358923\nleaf_count=59 4062 1090 46050 4428 49 18971 2111 97 9000 921 127 168 64 86690 4683 29437 10320 13969 2620 62 54 2339 1670 1393 123 230 4432 312 108 680\ninternal_value=-0.000482498 -0.00462454 -0.0153033 -0.00187737 -0.00836453 0.00101363 0.00686226 0.000712091 -0.0105246 0.00417088 -0.00379446 0.0157056 0.00832412 -0.0186614 -0.0264864 -0.0144305 -0.0226406 -0.00742278 0.00152524 0.00895572 -0.00672686 0.00391853 -0.00239003 0.0141611 -0.0176924 -0.00292753 0.00769542 -0.00545098 0.00382701 0.00318443\ninternal_weight=435.386 221.97 44.4834 177.486 54.1299 123.356 55.4631 33.35 50.9626 10.6867 67.8933 22.1131 5.35145 39.1319 18.5332 40.2759 17.5195 22.7564 13.7702 12.3109 55.5823 31.3442 19.8717 11.4726 6.63404 6.9651 11.6055 6.84267 213.417 205.495\ninternal_count=246319 241469 137546 103923 72150 31773 12329 7733 72101 9921 19444 4596 123 137423 91373 62180 40069 22111 8142 419 19025 7636 4132 3504 2462 292 5522 10632 4850 4742\nis_linear=0\nshrinkage=0.03\n\n\nTree=172\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 4 3 4 3 2 2 12 11 0 19 18 4 17 17 22 17 5 0 7 3 19 17 11 0 18 13 3 0 3\nsplit_gain=8.73423 7.05801 4.06713 6.52345 3.64925 4.58406 3.3308 3.13603 4.9029 4.8405 3.36299 3.09997 3.07619 2.90561 3.27406 2.7706 2.7651 2.45892 2.35706 2.32609 3.40047 3.55549 2.86532 2.5024 2.19315 6.12775 2.33856 2.32782 3.8744 3.29992\nthreshold=-39.283819198608391 0.84667968750000011 0.39370400768236996 3.2895507812500004 0.50207005315878861 -49.116760253906243 50.658309936523445 0.032945457735612756 0.74021522701736098 1.8987378677022575 207.00000000000003 1.2424533248940002 1.0693359375000002 1.2596354777617942 0.24468021674538498 0.81250000000000011 0.33136032425088474 1.2426757812500002 1.4460303622695712 2.6015625000000004 0.31840721315068282 219.50000000000003 1.8807230532061787 5.5511151231257837e-16 1.3771184689138856 0.89587973461402759 -0.43755180013342404 0.45898141303402712 1.4460303622695712 0.53650715206675215\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 18 3 13 -1 -6 15 24 9 10 -9 16 -12 14 -3 19 -10 -15 -2 20 21 -5 23 -21 25 -4 -27 28 -26 -29\nright_child=1 2 7 6 5 -7 -8 8 11 -11 12 -13 -14 17 -16 -17 -18 -19 -20 22 -22 -23 -24 -25 27 26 -28 29 -30 -31\nleaf_value=-0.023426955598247139 -0.025992306210807805 -0.018302071291254753 -0.030431066416386876 0.0092438742805559283 0.013771673320405258 -0.023388644253809217 0.018694173401524926 0.010079659122366981 0.014981446556515555 0.020607885173421468 0.00039850218323937215 0.012446018166087331 -0.039864890801021402 -0.006755871795381784 7.512623089535134e-05 -0.020688938619339011 -0.024395813398488807 -0.02882688972694784 0.0011533459053753089 0.018810891765765166 0.011329111313570044 -0.023096587194377641 -0.0045805522833976837 -0.0039496571590050458 -0.015638635827572914 0.00011561517402037174 0.0083858583473995739 -0.0018154127661369366 0.024156214305521416 0.011695111489903261\nleaf_weight=16.575338631819253 9.3743765760555089 9.962343359120494 3.5292780832387498 3.8217497411424146 6.5675511646841187 3.9384356893569912 12.632956069218379 1.6322463917022076 1.5958145223557973 1.7777954693883664 2.1425032873521541 15.209678255254401 7.0528359922973323 10.69380309963981 42.712935925892452 3.4455938908686212 3.8382365135475993 8.1236633663511366 3.8705355867241442 19.075539791230771 7.2009139289002624 8.6521294224348804 10.029731786381488 5.1563545029239313 1.8302719869534509 50.879942972405843 76.972526734967687 23.341372676106403 16.186375671823043 50.931154514604714\nleaf_count=83445 34072 12922 27 1293 340 228 954 37 12 11 51 250 475 19817 31192 3008 59 25191 762 3502 1049 12833 3687 573 51 6213 3421 212 160 384\ninternal_value=0.00112905 0.00222822 0.00296314 -0.00111661 -0.0150642 -0.000980835 0.00535426 0.00520583 -0.00331087 -0.0173458 -0.024793 0.00566456 -0.0313646 -0.00744017 -0.00367673 0.00214751 -0.0123008 -0.0172579 -0.0185355 0.0039564 -0.00413308 -0.0132374 0.00858727 0.014003 0.00648095 0.0039434 0.00512109 0.0100124 0.0195038 0.00748149\ninternal_weight=438.754 411.673 398.428 141.508 27.0813 10.506 70.015 256.92 33.2491 12.6054 10.8276 20.6437 9.19534 71.4927 52.6753 57.382 5.43405 18.8175 13.2449 53.9364 19.6748 12.4739 34.2616 24.2319 223.671 131.382 127.852 92.2892 18.0166 74.2725\ninternal_count=246231 162218 127384 116021 84013 568 26899 11363 895 574 563 321 526 89122 44114 25945 71 45008 34834 22937 15175 14126 7762 4075 10468 9661 9634 807 211 596\nis_linear=0\nshrinkage=0.03\n\n\nTree=173\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 9 1 4 2 2 13 2 2 14 14 0 0 18 2 7 17 7 0 13 14 1 18 1 18 18 13 1 5\nsplit_gain=7.98319 4.68686 5.57885 5.75457 5.53307 4.0196 6.88221 2.84935 2.72408 3.06194 2.8427 3.03336 2.44431 3.91634 2.92816 4.33503 2.9378 3.12514 3.00464 3.04198 2.92267 2.88934 2.79572 2.76487 2.37149 2.3663 2.3502 2.31961 2.31368 3.25628\nthreshold=2.9477339485081675 18.240050315856937 1.0000000180025095e-35 0.24961073363565869 1.4716796875000002 29.286581993103031 34.417465209960945 -0.54808102951419191 25.282213211059574 32.956295013427741 0.52141881878679586 0.69550561433971103 1.8987378677022575 1.6765212893675572 1.2424533248940002 11.076014518737795 3.1132812500000004 1.7061657746510188 3.3837890625000004 0.21782845461571179 -0.070829593282168909 0.97105577915991304 1.7090960825073716 2.0126758453675748 0.30807730392478799 0.89587973461402759 0.89587973461402759 -0.04883439202386685 0.35607680265790181 2.1386718750000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 5 -3 -7 -4 -2 -10 25 -12 13 14 22 16 17 20 19 -18 28 23 -5 -15 -17 -11 -24 -22 -16 -30\nright_child=8 4 7 12 -6 6 -8 -9 9 10 11 -13 -14 21 15 24 18 -19 -20 -21 27 -23 26 -25 -26 -27 -28 -29 29 -31\nleaf_value=-0.025206337130867208 0.0020894720838347445 -0.033500912148308241 0.019800037181719198 -0.024822578393829342 0.0049013979286187262 0.020761423060684063 -0.021073567524424717 -0.0012837663778956601 0.02313532460362093 0.027145567492777471 -0.02993044144140751 0.0042706235081627865 0.0098826327549641612 -0.042327615701959552 0.016889361377018648 0.021755952716413854 -0.018366156887690332 -0.024220520311916866 -0.01016045991579864 0.02888809628532368 0.024486775961062107 0.0099585022873191521 -0.0034771042239646406 0.0098556124166116633 -0.030284509197712323 0.0030406688879072554 0.022974822932452708 -0.00081972525510925738 -0.0067012812940027101 -0.024452304660018704\nleaf_weight=16.8227891230772 127.16416908718384 5.0511519233932658 7.9866627307528715 1.2069698176783266 74.298372144019595 5.202699521779321 7.2744810591593705 15.800854194920246 12.010303551971445 5.036136869108307 2.6785861512471438 5.8234808704000898 6.5435076576977727 5.2156394860940045 2.127456399242873 0.16103116366775694 0.80307249445467999 8.5074695037665133 3.9256989713503572 5.1839286889480718 3.0378175841688089 1.0628257840871809 4.7663817992433888 0.68160692020319391 8.6231186697277717 10.088670409386394 6.2940594675019383 16.61147491790598 52.851397254633412 10.704475511579401\nleaf_count=139795 2318 7221 1594 160 15152 2288 9861 4573 124 30 58 190 141 156 4028 249 417 16347 475 92 764 3 33 45 3992 160 49 4082 25232 6620\ninternal_value=-0.00104981 -0.00420098 -0.00701622 -0.00908671 0.00130083 -0.0134019 -0.0033832 0.00630833 0.00418436 0.0113985 0.00473508 -0.00852038 -0.00695917 -0.00786368 -0.00650247 -0.00802194 -0.00615367 -0.00780359 0.00874935 0.0208282 -0.00590527 -0.0283888 0.00699057 -0.0357418 -0.0272044 0.0122526 0.0122325 0.00408908 -0.00891158 -0.0100235\ninternal_weight=433.546 270.745 178.918 155.131 91.8267 17.5283 12.4772 23.7875 162.801 35.6372 23.6269 8.50207 138.308 131.764 124.804 112.537 103.753 93.8401 9.9127 5.987 85.3326 6.96007 12.2674 5.89725 8.78415 15.1248 11.0604 19.6493 65.6833 63.5559\ninternal_count=246249 243369 208847 202680 34522 19370 12149 6167 2880 562 438 248 62885 62744 62540 62298 58057 57073 984 509 40726 204 242 201 4241 190 82 4846 35880 31852\nis_linear=0\nshrinkage=0.03\n\n\nTree=174\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 7 0 13 10 18 5 7 0 21 5 18 0 12 18 13 14 21 13 5 3 3 19 14 0 0 5 0 13 19\nsplit_gain=6.95878 4.06087 6.41476 5.18751 3.72278 3.18829 2.83689 3.62147 3.22381 3.68045 2.55157 2.25985 3.03373 4.15598 2.26079 3.61209 5.09026 5.59248 4.35846 2.25808 3.83602 3.66801 2.12026 2.57791 2.11349 5.39974 3.38004 3.12356 4.14188 2.87699\nthreshold=0.44825707527455844 2.5244140625000004 0.53713645201195115 0.14796868535883259 2.9174053685313028 7.6016521472294505 3.8173828125000004 3.4804687500000004 0.72961209499321422 2.5000000000000004 5.0722656250000009 4.2121001622835372 1.1210156985015984 0.21094178584231701 2.249904835165133 -0.45067951570476578 0.32930624672166869 3.5000000000000004 -0.48998133262703347 1.4980468750000002 0.048385965359893195 0.26534039634031065 258.50000000000006 0.0095533965833058416 1.3771184689138856 0.97942481616843191 4.5488281250000009 1.6765212893675572 -0.028657236793878823 262.50000000000006\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 5 6 -3 8 -8 -10 -11 12 13 22 15 18 19 -18 -14 20 -17 -21 23 -4 25 26 -2 28 -26 -30\nright_child=24 4 11 -5 -6 -7 7 -9 9 10 -12 -13 14 -15 -16 16 17 -19 -20 21 -22 -23 -24 -25 27 -27 -28 -29 29 -31\nleaf_value=-0.022059204371259997 0.0076953632111168976 0.021750737692900684 0.0063428060138452907 0.017909722507631502 -0.02634555737159866 -0.022237301180724606 -0.019901178434670722 0.012140005181244981 -0.022839345234435687 0.022910712738285408 -0.019814414833745991 0.018953137791675016 -0.0055775634622368692 0.011976927911667544 0.012035197534270159 0.023109810162455474 -0.0049663483657760524 -0.037549540016104256 0.028545437095929247 -0.012804361923687675 -0.012928752395664192 0.020614141949691491 0.0023222635489406605 -0.019457994019003916 0.015923884824327307 -0.0059397221228590178 -0.016389803141167114 0.0013918635537374965 -0.016024398065959181 0.019394716781966047\nleaf_weight=28.868313691596544 74.820464077001816 8.8960437073237681 3.4658300889441298 2.4947080851270576 2.5687835006538071 2.526790541265199 7.5412988939983761 18.655626305675924 2.0680624038941442 5.9106661002224428 0.73290714356699016 3.0050595473958301 6.2296297586472065 5.8723132628329031 10.616623008275381 2.3995198520569856 13.123087471449255 6.765360091780166 5.7004063068961823 3.3954702610990344 11.693580930459573 9.086969841795506 6.0721044483425421 28.993928145779137 41.084201231918996 56.723884243067005 4.7064865879947311 48.715881336014718 6.3816637959826021 1.9260672321543095\nleaf_count=147372 7751 4693 1897 61 384 20521 3797 1651 135 159 27 1861 912 335 3226 216 1748 797 205 386 926 93 7260 37154 393 1314 332 451 103 9\ninternal_value=1.95013e-05 -0.00413011 -0.00659894 -0.0184248 0.00331372 0.00548001 0.00753947 0.00341302 -0.00672713 0.00555202 0.0162217 -0.00328332 -0.004033 -0.0100555 -6.95906e-05 -0.00243594 -0.00622836 -0.017317 0.0116738 0.00239717 -0.00566815 0.0112482 -0.0136313 -0.0165959 0.00350478 0.00107157 0.00608415 0.00683907 0.0120775 -0.00661133\ninternal_weight=431.042 196.683 147.783 31.363 48.9002 46.3314 43.8046 34.9086 16.2529 8.71164 6.64357 116.42 113.415 44.4042 69.0106 58.394 46.464 19.8884 11.93 26.5755 14.0931 12.4824 38.5319 32.4598 234.359 136.251 79.527 98.1078 49.3919 8.30773\ninternal_count=246169 235816 204449 147433 31367 30983 10462 5769 4118 321 186 57016 55155 46646 8509 5283 4166 2545 1117 1621 1142 479 46311 39051 10353 9397 8083 956 505 112\nis_linear=0\nshrinkage=0.03\n\n\nTree=175\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 2 13 0 4 4 4 4 14 3 19 2 2 14 4 0 14 17 12 4 0 14 12 13 4 19 9 14 13 4\nsplit_gain=7.06226 7.66001 5.04602 3.85604 4.08646 3.54306 7.8164 4.72707 5.47444 4.01343 3.73872 3.33223 3.15206 3.12468 2.86944 2.80254 2.5715 2.73635 2.34132 2.32598 2.28766 5.71876 4.80091 4.03591 3.36447 3.66375 4.25316 3.54949 2.80456 3.25122\nthreshold=0.37248792739977959 -26.006598472595211 -0.45067951570476578 0.83245446090221253 2.7826617956161503 1.6286621093750002 1.0693359375000002 2.2978515625000004 0.28824823456061044 0.65908129556435835 263.50000000000006 44.569120407104499 -37.900318145751946 0.70873219185162306 4.9616699218750009 0.29896707506516013 0.48434979059990024 0.077783200051907503 0.81392317852017215 2.9518512487411503 1.3771184689138856 0.72685412594425991 0.65514425837065626 -0.26133395848386182 6.9907226562500009 262.50000000000006 1.0000000180025095e-35 0.60348287794770183 -0.44620633588241487 2.4204101562500004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 18 13 4 -4 6 -2 9 -9 16 11 12 -8 -3 -15 -16 17 -7 -1 -6 21 23 -22 24 25 26 -10 -26 29 -27\nright_child=5 2 3 -5 19 7 10 8 20 -11 -12 -13 -14 14 15 -17 -18 -19 -20 -21 22 -23 -24 -25 27 28 -28 -29 -30 -31\nleaf_value=-0.027060519157886133 0.0017897135420640129 0.017815256397052143 -0.025706612972250752 -0.0016472732429704989 0.018178900864082703 -0.01542400513710547 0.0050937783643305642 0.012313315292456125 -0.028209006410240441 0.024228718965776597 0.0060674199667745286 0.0060958358186010446 -0.036270333450192573 -0.023918240064963729 -0.021530219073909434 0.016950541173307802 0.00081556055069520635 0.015258348635227022 0.013565629914486121 -0.0089124776681835625 0.0064765998328379631 0.0033012428988646856 -0.024846173514461927 -0.035028767670488591 0.026165235459512746 0.014441260178089025 0.017778300063489693 -0.018969653136111257 0.023714595423535569 -0.027214566312870909\nleaf_weight=16.153009013595696 67.748318378888143 13.351682570404593 14.676646197713124 59.486298788820022 2.3445149917952248 2.0741280823713195 1.6482985613765766 27.26457791235589 12.43961980603672 13.550468387562431 3.7629251909147561 2.2109116177452952 11.524813033385728 3.0776013986069284 1.3875863369139549 5.3465544405589753 33.893071508451612 28.420634629277629 0.79088464379310508 20.33325768341092 36.905582181585487 29.828039425992756 3.9936172561137928 6.0693028160785607 3.5058391294878666 2.1372827591549139 1.4017605121771346 1.539564960512507 2.518013318973316 2.8873187259305269\nleaf_count=109957 5214 6562 59988 13745 1515 25 250 419 1764 136 507 132 2582 17629 2182 444 2374 523 9 15194 356 3613 81 353 74 28 25 105 53 261\ninternal_value=-0.000610372 -0.00621623 -0.00345146 -0.00645911 -0.0138791 0.00200139 -0.00306362 0.00411756 0.00060723 0.0099127 -0.0194728 -0.025709 -0.0311004 0.00886969 -0.00382517 0.0072629 0.00657542 0.012748 -0.0245139 -0.00546219 -0.00257312 -0.00615242 0.00289807 -0.0146687 -0.00888662 -0.0140045 -0.0226676 0.011533 0.0026857 -0.0100397\ninternal_weight=432.272 136.948 120.004 96.8407 37.3544 295.324 86.8953 208.429 130.491 77.9383 19.1469 15.384 13.1731 23.1634 9.81174 6.73414 64.3878 30.4948 16.9439 22.6778 103.226 62.3267 40.8992 32.4987 26.4294 21.384 13.8414 5.0454 7.54261 5.0246\ninternal_count=246100 227225 117259 90442 76697 18875 8685 10190 7132 3058 3471 2964 2832 26817 20255 2626 2922 548 109966 16709 6713 6276 437 2663 2310 2131 1789 179 342 289\nis_linear=0\nshrinkage=0.03\n\n\nTree=176\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 4 4 4 17 1 18 13 4 2 18 19 19 18 17 4 6 4 19 1 19 4 17 6 1 18 0 18 7\nsplit_gain=8.5678 8.05287 5.74712 5.10641 4.72716 4.43507 3.92174 4.79961 3.61252 4.88647 3.79216 3.55191 3.89173 3.46666 3.34732 3.31885 3.10421 2.94556 3.34848 3.40674 2.80533 2.86382 2.71589 2.70341 2.57536 2.50719 2.45138 2.4059 2.85463 2.76641\nthreshold=2.5144622974930591 -33.804542541503899 1.6286621093750002 1.0693359375000002 0.84667968750000011 3.1006363674515387 1.7090960825073716 1.2424533248940002 -0.39782810107798666 1.9406738281250002 27.147908210754398 3.4811217321331038 246.50000000000003 267.50000000000006 2.6735537653587351 0.091221764698383404 4.9233398437500009 2.0131835937500004 3.5791015625000004 248.50000000000003 1.9634439907166827 205.50000000000003 2.3220214843750004 0.048554954793221271 7.2636718750000009 -0.041910702480567281 1.2424533248940002 0.29896707506516013 4.8242976514536702 3.4804687500000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 26 3 4 -3 16 -6 -8 9 14 11 20 25 -5 -4 -11 -2 18 19 -12 21 -10 -17 -18 -19 -13 -1 29 -29 -24\nright_child=5 2 8 13 6 -7 7 -9 10 15 17 12 -14 -15 -16 22 23 24 -20 -21 -22 -23 27 -25 -26 -27 -28 28 -30 -31\nleaf_value=0.011028592814898933 0.0031767919268238295 -0.024573972008737933 0.029439223122356216 -0.034034326077911167 0.010601011532064183 -0.024255713042127279 0.0086782680530142504 -0.032359369498588655 0.0022284703790152929 0.027892481133653352 -0.021322818826800795 -0.018842532908833524 -0.020928252342086784 0.012096457775960733 -0.0080667771373694862 -0.021260155957171831 -0.015646968434482555 -0.029030812485825577 0.014392365005945321 0.010778497276663523 -0.0032220852240224356 -0.029265353665855601 -0.02866902496515298 0.01895495132673394 0.016323953054422666 0.0060622406075719068 -0.028714664896230081 0.0018488232755008939 0.030974553014178462 0.0057803346072392578\nleaf_weight=1.0268834602320556 188.81338612621039 7.8415636486997782 9.2201560903304181 10.520872679789134 14.300181978104776 3.9969678419110997 4.1665468150749785 4.5148039802261337 3.0013232023084191 2.8162733610979549 5.2060729976255988 3.2182780084212963 6.8407752718776456 1.172639855336455 2.2422243195305773 4.3846692969618735 1.525354007724675 3.0074475437759243 19.503507822883876 4.8460420871379029 9.6660903076844971 13.500213817342226 3.6504597241337242 17.724345272959908 0.82854226697236288 26.10618553311815 13.712578528507036 32.724177005239653 2.3568392712695632 3.3097343073659431\nleaf_count=32 3976 32302 243 27841 12988 669 28 554 223 649 6105 13257 12865 389 2516 4465 31 83 1289 224 270 4154 8958 222 9 9801 94310 5781 612 857\ninternal_value=-0.000521637 -0.00475023 -0.0030993 -0.0127427 -0.00592946 0.00374166 0.00119169 -0.0138017 -0.000413982 0.00526055 -0.00402972 -0.00835359 -0.00213244 -0.0288198 0.0220439 0.000992804 0.00439294 0.00404994 0.00738897 -0.00627205 -0.0165566 -0.0237372 -0.0011927 0.0155758 -0.0178847 0.00279778 -0.025421 0.00134555 0.00460981 -0.0136193\ninternal_weight=425.745 213.685 198.946 42.5166 30.8231 212.06 22.9815 8.68135 156.429 60.7045 95.7245 62.3329 36.1652 11.6935 11.4624 49.2422 208.063 33.3916 29.5556 10.0521 26.1676 16.5015 46.4259 19.2497 3.83599 29.3245 14.7395 42.0412 35.081 6.96019\ninternal_count=245703 240805 146463 74102 45872 4898 13570 582 72361 24081 48280 40570 35923 28230 2759 21322 4229 7710 7618 6329 4647 4377 20673 253 92 23058 94342 16208 6393 9815\nis_linear=0\nshrinkage=0.03\n\n\nTree=177\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 4 18 11 7 11 1 17 1 13 18 2 0 18 17 17 1 1 11 11 21 21 3 0 21 2 13 13 14 18\nsplit_gain=9.29425 5.84602 3.31038 5.89865 4.87613 5.87368 4.25086 3.669 3.35044 4.83841 3.29717 4.74151 2.69184 2.99286 3.59118 3.12525 2.65307 3.06119 2.42131 5.17846 3.5728 3.45175 2.95523 3.51582 3.23293 3.07227 3.63171 2.89389 2.7984 2.67948\nthreshold=-39.733196258544915 0.84667968750000011 4.3882378946731615 0.66750229771629133 2.6015625000000004 5.5511151231257837e-16 4.0016611463566001 1.9512128003148403 0.45969269984116679 -0.26241780919347202 4.2121001622835372 -3.9017317295074458 0.70465417701170696 3.9796379800581989 0.45020544458187189 1.490403556610195 -0.12981918606627474 0.34439332195363109 0.62517033806334033 0.74021522701736098 2.5000000000000004 5.5000000000000009 0.58795853567030554 0.87173191741369926 3.5000000000000004 -2.5585746765136714 -0.54550535683058499 -0.37639895965301279 0.95830763268103369 2.4414009612931857\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 -2 10 4 6 7 -4 8 -6 -10 12 -12 13 -3 -15 -16 -14 -18 22 -20 -21 -22 23 25 27 -19 -27 -24 -25 -28\nright_child=1 2 3 -5 5 -7 -8 -9 9 -11 11 -13 16 14 15 -17 17 18 19 20 21 -23 24 28 -26 26 29 -29 -30 -31\nleaf_value=-0.018713537587034004 -0.019534768605687529 -0.01774359107433561 -0.022830006617166783 0.019673071668560225 0.028369636112082958 -0.026240959818532167 0.0072858026799953899 -0.016469325367395864 -0.029986584791891178 0.018120687335801529 0.026837341284987586 -0.013510842902222922 0.016559954747138759 -0.026460165740096887 0.01933749140490644 -0.014589976182765547 -0.022737262641555369 -0.005669733688574821 -0.033153342666809818 0.012798400837224087 -0.027913647455220389 0.0033519141775500269 -0.014164085694910979 -0.0003237786553680008 0.012298208521243057 -0.016831489759246504 -0.018567727365622674 0.0069545323998138299 -0.024117060316599871 0.022953185359833584\nleaf_weight=24.45813347376361 12.356923820372684 12.697642360451029 23.056131168519645 6.0805315784236873 7.5138229734984687 4.3564949229294134 4.6159805144416159 3.0334282494495701 2.1017014328026562 3.785271504893899 8.9065837281177647 2.8656925282139118 9.8349099408860621 2.0685409811094351 7.6027059571752016 2.5774428641243503 3.6481741906127345 7.3587192977129261 4.3701988363609443 9.4673241190030257 4.8040735200047475 6.6085903840139508 8.6386574337957445 157.1217194914534 47.251392693637172 1.9324825878102299 0.78212598734535532 13.844514515163608 3.5794539491762398 17.663485619419589\nleaf_count=82934 34980 8559 70432 183 3224 958 133 3060 503 193 3910 3462 7544 2064 2604 3498 4670 1850 126 144 78 100 313 7239 428 338 22 203 568 1625\ninternal_value=-0.00113699 -1.9736e-05 0.000650105 -0.00613465 -0.00970497 0.00129641 -0.0177275 0.00968263 0.016335 -0.000914338 0.00177435 0.0167267 0.00117732 -0.00809669 0.00248823 0.010211 0.0019811 0.00142588 0.00181576 -0.00687936 -0.000116294 -0.0109976 0.00268943 0.000722702 0.00791861 0.010389 0.0161819 -0.00141745 -0.000999618 0.0203283\ninternal_weight=424.983 400.525 388.168 54.5434 48.4628 20.7907 27.6721 16.4342 13.4008 5.88697 333.624 11.7723 321.852 24.9463 12.2487 10.1801 296.906 287.071 283.423 25.2502 20.88 11.4127 258.173 188.438 69.7346 27.7368 20.3781 22.4832 160.701 18.4456\ninternal_count=245945 163011 128031 78686 78503 7938 70565 6980 3920 696 49345 7372 41973 16725 8166 6102 25248 17704 13034 448 322 178 12586 11642 944 3835 1985 516 7807 1647\nis_linear=0\nshrinkage=0.03\n\n\nTree=178\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 4 0 18 18 19 12 6 7 13 6 7 3 3 18 21 17 0 5 4 4 4 12 5 19 21 3 3 1 1\nsplit_gain=5.13826 3.33226 2.70844 6.58312 3.63786 3.89086 2.62345 2.42304 3.73056 2.37468 2.2921 3.15114 2.30162 3.81151 2.2967 5.63344 3.43148 2.87804 3.88984 3.68362 8.35855 4.05782 3.51732 3.55993 3.14972 3.2943 5.09065 2.83415 2.61148 2.37187\nthreshold=3.9653415636860969 0.84667968750000011 0.45494749303197096 4.9593358439305844 5.2600927804663344 242.50000000000003 0.71769025269686726 2.1386718750000004 3.0205078125000004 0.14796868535883259 3.3837890625000004 3.6767578125000004 0.048385965359893195 0.098685488503310201 4.1510089048756029 3.5000000000000004 0.84061941967369014 0.69651904584641833 1.4980468750000002 1.6286621093750002 1.0693359375000002 1.6572265625000002 1.0000000180025095e-35 2.2500000000000004 247.50000000000003 6.5000000000000009 0.58795853567030554 0.47260169401963864 4.0016611463566001 2.1468275719512682\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 6 -5 7 -3 8 -6 -7 11 12 -4 -14 17 16 -16 18 29 20 24 -21 -23 -24 -19 -26 27 -27 -22 -15\nright_child=-2 2 10 4 5 9 -8 -9 -10 -11 -12 -13 13 14 15 -17 -18 19 -20 21 28 22 23 -25 25 26 -28 -29 -30 -31\nleaf_value=-0.013517672119274961 -0.018063259153621686 -0.024799113595812515 0.020085363056668134 0.011806755414809902 -0.0064993913067813297 -0.024898181201990312 0.013870241208436747 -0.015875347422902721 0.026475837995157452 0.008683992718895318 0.014032929260256654 -0.020874488531317761 -0.028563851777457915 -0.02121086658541043 0.030268098555038328 -0.012330913579116507 -0.005451038093472884 -0.0097729912585017339 -0.024487906973679339 0.015673554679904816 -0.027696116266777891 0.0038025779414114723 -0.02336567349804583 0.0011927830038679556 0.008823011126595481 -0.021880259357253844 -0.039549654694339083 0.021350392288007641 -0.0024458970837777133 0.011382390996264154\nleaf_weight=14.492302992905932 13.456621546343738 14.422543166282592 4.9332891210851804 16.031895450017146 5.3145800034280928 8.8311707280408847 1.1229632924078021 3.0950343097201758 5.4501099326125768 1.8134955379646278 11.996386466013062 4.7343607347575007 2.8243659123409079 1.7167365994191657 9.5541686799806502 5.3721490305088073 2.7859888915654656 15.33518031587664 8.1116199683410724 22.050414164055837 13.09710056039785 159.44702125785261 8.1105028693673393 12.523308940235436 39.668552254921451 1.1116395179797098 2.5120536447502673 3.0926118924398907 5.272379843867383 6.8592796936572995\nleaf_count=32927 61480 32101 3477 20565 9663 39205 34 1179 1748 44 259 138 5835 5184 2659 3445 2664 1356 4323 482 5118 9707 297 272 1500 228 27 33 73 316\ninternal_value=-0.000483926 0.000133064 0.000663835 -0.00538793 0.00109462 -0.00610188 -0.0213363 0.00435547 0.0110272 -0.0189225 0.00167165 0.00117993 0.00156498 0.00121718 0.00157193 0.0117699 0.0223998 0.000930398 -0.010815 0.00166063 -0.00375296 0.00381355 0.00228279 -0.00909422 0.00161752 0.00551008 -0.0126647 0.00791193 -0.0209493 0.00332425\ninternal_weight=425.14 411.683 397.191 56.0818 40.5363 24.5044 15.5455 13.8597 10.7647 10.6447 341.109 329.113 324.378 319.445 316.621 17.7123 12.3402 298.908 16.6876 282.221 80.0895 202.131 180.081 20.6338 61.72 46.3849 6.71631 4.20425 18.3695 8.57602\ninternal_count=246339 184859 151932 104539 72404 51839 32135 12590 11411 39249 47393 47134 46996 43519 37684 8768 5323 28916 9823 19093 8335 10758 10276 569 3144 1788 288 261 5191 5500\nis_linear=0\nshrinkage=0.03\n\n\nTree=179\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 4 17 1 1 1 18 2 13 4 4 5 1 13 1 7 2 1 0 4 12 12 0 1 4 19 1 1 17 14\nsplit_gain=6.11101 5.23934 3.74668 2.40536 3.66218 2.28662 2.49703 2.17281 2.05437 2.92902 6.09625 5.10938 3.53855 3.44489 3.39674 2.81248 4.48757 3.18107 2.78536 2.78472 3.98396 4.8517 2.74227 2.66381 2.95019 2.26319 2.253 3.65398 4.33507 2.17929\nthreshold=-42.103004455566399 0.84667968750000011 3.9653415636860969 2.3713805701640611 2.2520445226790948 2.5144622974930591 2.602003343538398 -19.986686706542965 -0.44272381170967801 2.7826617956161503 1.0693359375000002 2.1386718750000004 1.4475910192202701 -0.29868446770448304 1.4866206336930319 0.89599609375000011 -0.44610977172851557 1.7090960825073716 1.8987378677022575 3.7336425781250004 8.8817841970012543e-16 0.90934753668501156 -0.35455277334367286 1.7622051334857016 2.3220214843750004 207.00000000000003 1.075454090076706 1.7090960825073716 0.15621648314530187 0.91011757189720621\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 7 3 4 8 -1 -7 -2 15 10 13 -11 -12 -10 -14 16 18 -18 -3 -13 -21 -22 -17 -16 -25 -24 29 28 -28 -27\nright_child=1 2 -4 -5 -6 6 -8 -9 9 11 12 19 14 -15 23 22 17 -19 -20 20 21 -23 25 24 -26 26 27 -29 -30 -31\nleaf_value=-0.024741462433878626 0.002646351590689966 -0.027489138641826217 -0.015434404314984411 0.0045428747438209958 -0.016424113989143063 0.010742668991539847 -0.016987620472056205 -0.023806468608220423 -0.023608933776483088 0.023667610795473994 -0.028271431084316696 -0.02124374919374555 0.016035136416345713 0.014079300477139392 -0.029620919852275392 -0.020963595738002955 0.01831635329161747 -0.023443879471223773 0.014937387205655697 0.0076265297991588051 -0.018044736037724097 0.013417274277815378 -0.016126372140861094 0.0032392572290248782 -0.026852659124543866 0.024097801840356023 0.01832702144044096 0.013885405404030706 -0.027363622613247079 -0.0091076627982540469\nleaf_weight=10.16025480253302 3.5671950698924766 6.4483182754347563 10.00871615081587 204.145532422408 10.729211294259583 4.5453315291088057 5.2806907691083325 9.4030900991028812 1.6560372977728537 7.9978708612079563 10.019064783259635 4.3646650561886409 3.8108837772924771 12.954896658149663 4.7160289320036126 1.6168047050377925 6.6626952943497599 1.1753445157228268 1.0411076562013475 33.446178601536303 14.066556604563631 5.3978637480176976 1.7392775040625554 10.597526722638575 2.9659747712503295 15.367105683038973 2.0550115247642706 14.60492338840413 4.2354859817787656 1.4774839781248363\nleaf_count=77252 9338 10745 29412 3663 579 56 672 26469 2060 1489 36971 5763 230 5980 1371 3295 6436 351 8 9911 1390 79 535 1086 293 6091 110 441 915 3081\ninternal_value=0.000385498 0.0011981 0.00183496 0.00232489 -0.000213786 -0.0154026 -0.00435269 -0.0168625 0.000909761 -0.00144423 -0.00710086 0.00265065 -0.0142746 0.00856893 -0.00697694 0.00552592 -0.00520013 0.0101102 -0.0205266 -0.00063991 0.00142226 -0.00909042 0.00955488 -0.0123573 -0.00473304 0.0112917 0.012939 0.00591028 -0.0119707 0.0208381\ninternal_weight=426.257 406.271 393.301 383.292 179.146 19.9863 9.82602 12.9703 168.417 111.994 46.7204 65.2731 32.1095 14.6109 22.0904 56.4236 15.3275 7.83804 7.48943 57.2753 52.9106 19.4644 41.0961 18.2795 13.5635 39.4793 37.74 20.8954 6.2905 16.8446\ninternal_count=246072 168092 132285 102873 99210 77980 728 35807 98631 66623 47991 18632 39951 8040 2980 32008 17540 6787 10753 17143 11380 1469 14468 2750 1379 11173 10638 1466 1025 9172\nis_linear=0\nshrinkage=0.03\n\n\nTree=180\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 7 11 17 13 0 4 13 3 0 0 3 1 3 17 0 7 0 0 3 14 3 1 21 1 1 0 19 4 1\nsplit_gain=8.20759 6.34765 4.03092 4.75522 3.64061 4.56785 3.51462 3.04649 2.69971 2.46924 3.62171 3.05514 3.29989 2.24367 3.50007 2.91542 2.2663 2.19684 3.94453 2.08137 3.47229 3.19088 3.4957 2.84556 3.81216 2.03882 2.85948 2.00345 2.32901 1.94385\nthreshold=0.31840721315068282 2.6015625000000004 8.8817841970012543e-16 2.0954123253646428 1.0000000180025095e-35 1.1851094193663554 0.84667968750000011 0.14796868535883259 0.28580415008586763 0.29896707506516013 0.065478893854873294 0.3261289997719386 2.9477339485081675 0.39370400768236996 1.0401760489885838 0.97942481616843191 0.89599609375000011 0.38995551143381973 0.52258841453381233 0.42638606281916264 0.4480071257740102 0.47260169401963864 2.5144622974930591 6.5000000000000009 2.5144622974930591 1.5275240303550033 1.8987378677022575 242.50000000000003 6.4848632812500009 1.7622051334857016\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 3 -3 5 -1 -2 -4 -7 10 27 -11 13 14 15 16 -13 -15 -19 20 -20 22 -21 -23 29 -17 -27 28 -8 -25\nright_child=6 2 7 -5 -6 8 9 -9 -10 11 -12 12 -14 17 -16 25 -18 18 19 21 -22 23 -24 24 -26 26 -28 -29 -30 -31\nleaf_value=-0.020313047220860976 -0.014047960705878564 0.018578646888347421 -0.023022287999417843 -0.011279942481762679 0.0071394622742501808 0.00059338298170648438 0.023759583542012963 0.013358343447037296 -0.024650918974764666 0.022733381456347655 -0.025280135147194616 -0.016255092892611098 0.0069561476681911051 0.017628894185672248 -0.02358668474289859 0.0076214303393508523 0.021169580410416881 -0.026577072743914094 0.025759847722820337 -0.019756266255797999 -0.0032340055953110057 0.004768879281081894 0.014555633216316879 -0.021603956168379013 -0.037358484153512971 -0.03130692597597292 0.012069837438996251 -0.013518658766419179 -0.011322809721125572 0.010535401757103154\nleaf_weight=38.025942723279059 9.4249896830960989 17.205577142701266 5.8220396415754276 5.7590255417048857 7.7819409276489742 21.800985941770708 4.230273317480167 2.2049198365129996 3.6096216923215243 6.908846444262962 5.436739200559713 1.0617738242053771 144.23976525533362 6.7473697150380749 7.3289384436739047 2.9416807418165245 5.7776747468105896 3.4592403428323442 7.17608800234302 10.79218561181551 6.4514927685268049 74.568474419203994 2.6632870536413966 1.3851190638451933 2.5349325253628194 4.0364379991660826 1.2678863159380851 3.0101960990577936 1.805544456336065 4.8625756624096539\nleaf_count=184679 5608 11271 2675 12573 360 5289 1605 61 316 829 4227 1567 1730 1062 2619 302 740 1464 122 1057 964 2337 63 265 59 189 13 1827 521 109\ninternal_value=0.000744253 -0.00661168 0.00461831 0.0109325 -0.0114948 -0.0138561 0.00312105 -0.0126561 -0.00379021 0.00368893 -0.00803679 0.00429142 0.00377061 0.000532708 -0.00797573 0.000603345 0.0140273 0.0021661 0.00110458 0.0022024 0.012751 0.000602267 -0.0124277 0.00282811 -0.012453 -0.0108437 -0.0206687 0.00381146 0.0131474 0.00141239\ninternal_weight=420.322 102.21 30.9916 22.9646 71.2185 63.4366 318.112 8.02696 25.4106 308.687 14.4828 294.204 287.295 143.055 22.4144 15.0855 6.83945 120.641 113.893 110.434 13.6276 96.8066 13.4555 83.3511 8.78263 8.24601 5.30432 9.04601 6.03582 6.24769\ninternal_count=246503 217224 26580 23844 190644 190284 29279 2736 5605 23671 8180 15491 14662 12932 5430 2811 2307 7502 6440 4976 1086 3890 1120 2770 433 504 202 3953 2126 374\nis_linear=0\nshrinkage=0.03\n\n\nTree=181\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 1 10 17 4 4 2 2 17 19 0 4 1 17 3 14 3 1 2 1 18 2 2 0 5 19 18 2 5 4\nsplit_gain=8.02171 3.93835 3.87191 3.18908 2.85618 3.06307 3.50818 3.98538 3.75307 4.69801 4.2069 3.30821 3.97764 4.27008 4.64068 3.55223 2.78631 2.62479 2.496 3.99727 4.68157 3.77285 2.41171 2.53097 2.40628 5.59922 4.39017 3.11513 3.28246 2.37134\nthreshold=1.3771184689138856 -0.17729835697704338 1.7005986908310777 2.2468301507771291 0.84667968750000011 1.0693359375000002 34.417465209960945 30.514518737792972 0.42804718369210387 253.50000000000003 1.2225960941547209 2.9458007812500004 1.8223781763416411 0.16595947670675196 0.58795853567030554 0.50129856847702103 0.40160889672495398 2.5144622974930591 18.240050315856937 0.6973310998193073 2.6735537653587351 18.821963310241703 6.8277702331542978 0.57379665772197053 1.2426757812500002 261.50000000000006 2.249904835165133 -14.86118173599243 1.4980468750000002 3.9514160156250004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 29 17 6 7 8 10 -10 -6 12 18 14 -14 24 -17 -3 19 20 -7 -20 23 -11 -15 26 -26 28 -28 -2\nright_child=3 4 -4 -5 5 11 -8 -9 9 22 -12 -13 13 15 -16 16 -18 -19 21 -21 -22 -23 -24 -25 25 -27 27 -29 -30 -31\nleaf_value=-0.025582324058434077 0.0040913826690171717 -0.029698489884875466 0.0087634955374953603 -0.0098684925100516428 0.021436665113383838 0.024275404827541241 -0.029172369084500594 0.033731366255976801 -0.018295974018596697 -0.021410988240940419 -0.018214525555535148 0.00044505915837354386 -0.036863532121181916 0.013929589936824577 0.0089147049679638504 0.012213271161335902 -0.012530057339937018 -0.00058882388502587825 0.023766605906754518 -0.03339562202042394 -0.023503964482702682 -0.015469230669233709 -0.024638544840154864 0.014458081168685768 0.0098220327836503977 0.020577071754454414 0.019097320200197443 -0.029841688906714483 -0.022525165284066583 0.014032262386367417\nleaf_weight=12.053583480336782 113.25834352691163 5.6860687949647382 3.2702165021855754 11.434858910264664 12.810183204297571 2.3761894669896781 1.6339906103021338 3.4027357733120871 10.215988295662102 1.1222849961741292 2.1173286615667157 87.005164876438812 5.5284659395329063 13.954702846111102 2.3910856809234247 3.984181637941222 26.75226687102986 4.8521890309893934 2.2176684185696995 11.389504185325675 3.0799798692100921 6.0892581155230801 1.0257784642708418 13.487874937730338 6.3804689131211552 6.7321467053261577 2.7246241355896945 7.8081986490888049 2.1452200880994496 26.190721390885301\nleaf_count=118607 2864 23085 732 1448 2457 994 1198 164 5078 1191 91 29372 600 656 109 1087 1481 675 280 14733 28658 7950 81 746 68 97 102 567 424 420\ninternal_value=-0.000710066 -0.00387621 -0.0181648 0.004782 -0.00291974 -0.00219834 0.00469584 0.00656633 0.0035465 -0.00339242 0.015022 -0.00387177 -0.00746829 -0.00402959 -0.0235922 -0.00153218 -0.00903896 -0.017508 -0.0176229 -0.0239662 -0.00215876 -0.00356641 0.00685951 0.0105077 0.00435243 -0.00115606 -0.00947611 -0.0191963 4.83381e-05 0.00604505\ninternal_weight=413.121 262.237 15.3238 150.884 246.914 236.375 45.8162 44.1822 40.7794 25.8519 14.9275 190.559 103.554 78.4014 7.91955 70.4818 30.7364 10.5383 25.1526 16.8457 5.45617 8.30693 15.6359 14.6102 39.7454 25.7907 19.0585 12.678 4.86984 139.449\ninternal_count=246015 241283 119339 4732 121944 98184 11006 9808 9644 7096 2548 87178 57806 5191 709 4482 2568 23760 52615 44385 29652 8230 2018 1937 1914 1258 1161 1093 526 3284\nis_linear=0\nshrinkage=0.03\n\n\nTree=182\nnum_leaves=31\nnum_cat=0\nsplit_feature=18 1 22 11 19 2 1 1 19 0 19 0 18 1 17 4 5 6 4 4 1 0 7 1 4 1 2 1 5 19\nsplit_gain=6.32394 4.14792 3.3986 3.42424 3.87092 3.03381 2.87291 3.14937 2.82586 4.48921 3.53807 3.84998 3.93828 5.29675 3.92753 3.1091 2.97788 2.96596 2.81634 4.47468 3.42635 3.6248 3.72608 2.93579 2.72029 2.5352 2.7162 2.33024 5.57052 3.12215\nthreshold=3.3140206880897671 -0.041910702480567281 1.0000000180025095e-35 0.72257959311131259 226.50000000000003 28.11904335021973 2.1468275719512682 -0.10824782141703854 257.50000000000006 1.5447487307370646 246.50000000000003 1.1495945197163422 2.0126758453675748 1.3779212206518803 0.068176441062887241 4.1440429687500009 1.7006835937500002 3.3837890625000004 5.5515136718750009 4.0444335937500009 0.45969269984116679 0.24908287076908558 3.5976562500000004 1.211689956917019 0.84667968750000011 1.8912989126705502 8.8705892562866229 1.9634439907166827 1.4980468750000002 246.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=6 5 3 4 -3 -2 7 -1 10 -10 11 14 13 -13 -9 16 -15 -17 19 23 -20 22 -22 -6 -12 -26 -27 29 -29 -25\nright_child=1 2 -4 -5 18 -7 -8 8 9 -11 24 12 -14 15 -16 17 -18 -19 20 -21 21 -23 -24 27 25 26 -28 28 -30 -31\nleaf_value=0.013426791135314656 -0.026295250646381961 -0.021867052634017926 -0.010622103061671309 0.018265186864329479 -0.020203884019102877 0.0065447644145260704 0.0048670895111715094 0.015450752574751873 -0.030082275222352257 0.0064549467699028537 -0.024628807531833972 -0.035863228462558988 0.010653310920120222 -0.01303560627359839 -0.030861638283033285 -0.03010593426171216 0.021224736888689678 0.012202162051331946 0.026179984794423378 -0.027588803778793004 -0.030065242199703263 0.012822074872550215 0.011945220958158196 0.021182740171695764 0.013712919592383023 -0.028808274498967144 0.0060116225655498035 0.0065590453214165317 -0.026830320857954133 -0.014102413021276846\nleaf_weight=10.145759674488543 11.446493066903257 6.7059434448183302 36.427272957905643 7.5894551610108456 4.5093467926885742 2.7427582487371174 193.85572870537726 1.3819411383319655 7.0748306209025014 4.358801366732223 1.4218205859924662 4.3186223111934003 12.684743699228418 2.6229284482542408 7.772419641716624 3.5059405072825021 5.8145907310536131 1.5618425067514179 6.2623278224573395 5.1422008398167227 3.9965333985901443 8.9646685590014368 2.3668363866527216 9.105437351148792 21.556242651985489 2.0319514573566257 5.2608109619468451 15.737893683754011 5.1593718553576755 2.146391162164945\nleaf_count=7990 119141 7005 31483 170 34299 8721 3099 174 2407 121 1462 1249 1940 194 4423 54 96 17 2935 3478 2196 357 149 3490 3141 114 77 1818 1193 3068\ninternal_value=0.000321172 -0.00518644 -0.0031954 0.000377732 -0.00178862 -0.0199344 0.00280688 -0.00156277 -0.00357215 -0.0167544 -0.00121114 -0.00703018 -0.00164843 -0.0105041 -0.0230372 -0.000481528 0.0103217 -0.0172049 0.000642025 -0.00389131 0.00920264 0.0010879 -0.0149394 7.71573e-05 0.00639439 0.00869725 -0.0059941 0.00344552 -0.00253334 0.0138501\ninternal_weight=413.672 128.303 114.114 77.6864 70.097 14.1893 285.369 91.5132 81.3675 11.4336 69.9339 39.663 30.5087 17.8239 9.15436 13.5053 8.43752 5.06778 63.391 41.8006 21.5904 15.328 6.36337 36.6584 30.2708 28.849 7.29276 32.1491 20.8973 11.2518\ninternal_count=246061 219503 91641 60158 59988 127862 26558 23459 15469 2528 12941 8147 3550 1610 4597 361 290 71 52983 47346 5637 2702 2345 43868 4794 3332 191 9569 3011 6558\nis_linear=0\nshrinkage=0.03\n\n\nTree=183\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 4 2 19 3 18 19 13 18 17 18 18 7 4 12 18 1 13 5 13 22 9 3 1 7 5 12 7 6 4\nsplit_gain=3.80197 2.88397 2.50184 2.49594 2.4598 2.35398 4.13623 4.10077 4.03226 3.00614 3.40502 2.85669 2.84134 2.5031 5.52834 4.2987 5.26155 3.04659 3.67779 2.86692 4.37102 2.80066 2.95533 2.43761 2.39391 2.43097 2.37468 2.35141 2.31193 6.08438\nthreshold=5.817026225953116 0.84667968750000011 -10.988907814025877 255.50000000000003 0.38657275885365588 5.2755571088971331 242.50000000000003 0.14796868535883259 5.0271592211193568 0.42804718369210387 5.2067464199591091 5.9362152003510085 3.0205078125000004 5.5515136718750009 0.049631972434904477 3.1984648276080736 1.211689956917019 -0.52657366547836626 1.4980468750000002 -0.41589482361201985 1.0000000180025095e-35 1.0000000180025095e-35 0.048385965359893195 1.2421711908324335 2.6738281250000004 3.1982421875000004 0.74643410817732136 0.89599609375000011 1.7006835937500002 6.6723632812500009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 4 -1 8 11 -8 13 -10 -11 27 -13 15 24 21 -17 -18 19 -19 -21 22 -3 -20 25 -15 -16 -7 29 -26\nright_child=-2 5 -4 -5 -6 6 7 -9 9 10 -12 12 -14 14 26 16 17 18 23 20 -22 -23 -24 -25 28 -27 -28 -29 -30 -31\nleaf_value=-0.016423479868851763 -0.020586960652309628 0.02331726057187335 -0.024551578852608381 -0.021809810252742791 0.023689966944337275 0.014203270106508934 -0.030286742686564842 0.01387863494142417 0.027487345228957547 -0.022447421661208326 0.018110909219021924 -0.013827571111566518 0.019193984639364146 -0.0086361355703745979 -0.021648425210318709 -0.028429950371095695 -0.02248265249716673 0.01951787441552754 0.018193177654483344 0.0075950913003550056 -0.022491847178948424 0.019877430325401225 -0.00034470598463713122 -0.014461848326585015 0.013446645612311516 0.026958098113109473 0.0011903834732457401 -0.028238163877471311 0.013776717971654855 -0.021439838250455919\nleaf_weight=1.1809568397193286 7.6491063055291297 3.856146085101499 7.1939964793682503 1.9953902636179917 3.3614536669028894 0.95716268251999581 8.5936820123661128 1.7320964481623384 7.5693690943380698 2.4740638972793931 3.0186980409621356 2.432611863022883 7.4134502749775493 1.5542155418297614 6.7675369474090976 8.7380657826615771 6.1980809063570623 7.9942779897537575 1.4529251600743034 14.447009190638711 5.0487471433189048 5.6866243093754738 217.63411826832635 17.804186831060179 9.3942835699781444 10.171004239462489 8.4635810656473023 2.9567677343693504 16.794588895965717 6.9408079183485816\nleaf_count=11672 42519 1073 18661 5165 417 1374 44464 48 4822 9838 3384 5887 2370 346 464 64229 2212 696 119 2421 1083 134 13155 2997 498 453 155 4471 465 395\ninternal_value=-0.000907751 -0.000480844 -0.0135082 0.000288532 0.0121795 1.6187e-05 -0.00887174 -0.022307 0.000630344 0.0163807 -0.000801123 0.00203857 0.0105128 -4.36474e-06 0.00551469 -0.00116745 -0.00812303 -0.00430669 -0.0014764 0.00550449 -0.000913655 0.000748456 0.000170199 -0.0112216 0.0107679 0.0219321 -0.00966598 -0.0170808 0.00628982 -0.00175836\ninternal_weight=407.475 399.826 13.7318 6.5378 4.54241 386.094 24.0858 10.3258 362.008 13.0621 5.49276 13.76 9.84606 348.946 60.086 288.86 61.6833 52.9452 46.7471 27.49 19.4958 227.177 221.49 19.2571 44.8549 11.7252 15.2311 3.91393 33.1297 16.3351\ninternal_count=245987 203468 35915 17254 12089 167553 58614 44512 108939 18044 13222 14102 8257 90895 2776 88119 73757 9528 7316 4200 3504 14362 14228 3116 2157 799 619 5845 1358 893\nis_linear=0\nshrinkage=0.03\n\n\nTree=184\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 4 3 1 1 3 4 13 19 4 4 4 2 4 14 19 2 2 19 18 19 4 2 3 2 2 11 21 18 13\nsplit_gain=5.59627 3.80437 3.37979 3.26757 2.5739 5.23965 2.9852 3.25058 2.4168 3.03369 11.7615 2.81346 2.71752 2.69366 2.62831 2.74479 2.40211 2.51953 3.01296 3.83444 2.4825 2.34666 2.39368 2.2333 3.96149 2.22894 2.13196 2.63251 2.12421 2.10689\nthreshold=-42.103004455566399 0.84667968750000011 0.50207005315878861 -0.59079332554470076 2.9477339485081675 0.58795853567030554 3.6791992187500004 -0.18664372546193439 264.50000000000006 1.0693359375000002 1.4716796875000002 1.6572265625000002 39.736396789550788 1.6572265625000002 0.24669767028957582 262.50000000000006 6.3334450721740732 -10.988907814025877 261.50000000000006 1.8688348091416842 245.50000000000003 1.7380371093750002 -34.252307891845696 0.48555671035752257 -3.9017317295074458 -10.988907814025877 0.38821517471137929 2.5000000000000004 4.1820210059610323 -0.24540680895748776\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 25 -1 -3 8 6 7 16 9 12 -11 -12 14 -10 -5 28 17 18 -6 -20 -18 22 -13 -19 -25 -2 -7 -28 -16 -22\nright_child=1 3 -4 4 5 26 -8 -9 13 10 11 21 -14 -15 15 -17 20 23 19 -21 29 -23 -24 24 -26 -27 27 -29 -30 -31\nleaf_value=-0.024821788669282533 0.00035376182620097603 -0.019325158372287269 0.00096857513186424972 0.026760432363978504 -0.010496577173893584 0.014142473048627091 0.011188232324506501 -0.0092989112611423976 -0.025416020291559007 -0.032082031925868941 0.014932105926145953 0.014137145320575379 -0.01738099219380659 -0.00042250328497089862 -0.006252324920578793 0.019324637625893225 0.013094853540139208 0.022125205879567192 -0.012522042916494235 0.023837126134838262 -0.023142091219221853 0.00044175061202842802 -0.029580782788873921 0.022649263312432159 -0.015933597141336651 -0.024487145523267941 0.013686382010775959 -0.029007030260551306 0.018614843728959749 0.0088279187600748561\nleaf_weight=11.719275758371625 5.2865029577055793 5.7802319257648369 7.0268380282141143 6.5152076526810587 11.874327521058147 47.098536319914274 18.818631382047897 30.919997080622124 6.0825371280075151 9.3824872329555529 10.8305641887041 0.84283987081198564 2.4353612697141207 8.6613483557052842 13.732206716540082 8.1222981725168193 2.7091647517518131 11.007906414917676 3.3169082137756032 6.1834745213272981 6.9012685341185707 150.04751497487496 3.4318622260961993 3.2967271530069411 4.4163338667713097 7.3276920835864967 1.7451940886676305 1.5927666276693342 2.9156495127599547 1.8382246850524087\nleaf_count=77708 13853 28926 500 1623 312 314 282 798 1982 21861 3073 375 1946 1656 3455 310 51 122 17 56 113 56879 4055 25 37 22192 28 23 3968 20\ninternal_value=-9.76779e-06 0.000771117 -0.015595 0.00132531 0.00169444 0.004695 0.000727617 -0.00176498 -0.000358858 0.000495603 -0.0011095 0.000832578 0.00859613 -0.0116929 0.011094 0.00609479 0.00284538 0.00629589 -0.000802669 0.0111593 -0.00877406 -0.000246714 -0.019915 0.0140304 0.00126431 -0.0148152 0.012493 -0.00867612 -0.00108916 -0.0162017\ninternal_weight=411.86 393.114 18.7461 380.5 374.719 151.719 101.283 82.4643 223 208.256 174.535 165.153 33.7207 14.7439 31.2854 24.7702 51.5443 40.0957 21.3747 9.50038 11.4487 154.322 4.2747 18.721 7.71306 12.6142 50.4365 3.33796 16.6479 8.73949\ninternal_count=246560 168352 78208 132307 103381 2198 1833 1551 101183 97545 86243 64382 11302 3638 9356 7733 753 569 385 73 184 61309 4430 184 62 36045 365 51 7423 133\nis_linear=0\nshrinkage=0.03\n\n\nTree=185\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 17 4 12 19 11 18 1 1 1 4 18 18 19 7 18 12 18 21 11 19 7 13 18 3 18 1 18 3 4\nsplit_gain=6.7518 3.35372 2.89224 3.89364 4.34289 4.04978 3.28701 5.82577 3.59647 2.91455 3.31612 2.82042 3.82367 2.47936 2.49944 2.49099 2.45627 2.85292 4.03417 2.96374 2.40267 2.19524 2.04621 2.05703 1.99222 1.9824 2.19569 1.98135 1.85757 1.88258\nthreshold=2.5144622974930591 3.9653415636860969 5.5515136718750009 8.8817841970012543e-16 246.50000000000003 0.91125284794847328 3.1132683346437333 1.048858409861958 1.2421711908324335 0.6973310998193073 1.9406738281250002 3.3842466058243152 4.0689902272260703 207.00000000000003 2.5244140625000004 6.8062759569045816 0.65514425837065626 3.0201273556387074 2.5000000000000004 0.72257959311131259 207.00000000000003 2.2500000000000004 -0.4958236943477588 4.3882378946731615 0.17302227103784359 2.8617925509761908 1.4475910192202701 4.7405652883788738 0.22136347059745312 2.1750488281250004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 6 4 13 20 9 -8 -9 10 -1 16 -13 14 -4 -15 21 18 19 -18 -5 28 -14 -24 -16 26 -11 -25 -2 -30\nright_child=11 -3 3 5 -6 -7 7 8 -10 25 -12 12 22 15 24 -17 17 -19 -20 -21 -22 -23 23 27 -26 -27 -28 -29 29 -31\nleaf_value=0.019751785654477941 -0.020626500948196715 -0.020849996812172894 0.01558309508903535 0.0018322123797918208 -0.02137980380901093 0.013603225773121997 -0.027117533331956736 0.014788850340497305 -0.0091199819054782975 -0.020129166887098936 -0.014343937877840425 -0.017293324564730293 -0.021585999170551096 0.014885855948529435 0.012544306103770934 -0.018405947397657316 -0.02500531499368263 0.019292157866308456 -0.030505326624222718 0.019034695511562505 -0.026333437964333135 0.018756757779640205 0.017742090546683322 -0.017845072688988054 -0.027628575841351491 0.0096923368934691626 -0.0036833351351816981 0.0057736644834758995 0.0080737237922945265 0.00074549684360137453\nleaf_weight=8.9705080062412872 1.4995741270176939 9.1590075270632934 2.1280477055224756 3.6671481938246897 3.4812077192477764 4.1164944957708931 14.272466028930465 6.0138702559215735 29.462935141663365 7.5152480035721956 2.6500045644115735 11.603378627107302 1.7156222470803175 32.424707272423802 0.841165962132436 1.3240433288856355 1.0348468832380584 1.9476617984473694 5.0439113201573491 3.3979208492673934 8.3850603350838337 10.975125083699821 7.4303278499282959 3.4738235562581385 3.3318625085230451 7.3909151160059983 57.396872180608625 10.218740770353177 95.536018250415509 47.192179730554926\nleaf_count=6653 100 60717 87 104 12745 62 120379 4575 12933 2451 2928 698 297 4264 101 7377 22 26 96 74 2267 80 322 356 341 1197 2100 855 1394 529\ninternal_value=-0.000165255 -0.00402155 -0.00312132 0.0023417 0.0069868 -0.00984284 -0.00556043 -0.0115912 -0.00477274 -0.00189103 0.0114559 0.00371977 -0.00400838 0.00991324 -0.00655278 0.0130876 0.00533157 -0.00762001 -0.0144645 0.00604316 -0.0182796 0.0063276 0.00318349 0.00608009 -0.0186713 -0.00416316 -0.00586387 -0.00102518 0.00525915 0.00567274\ninternal_weight=403.601 202.532 193.373 59.6997 43.531 16.1687 133.673 49.7493 35.4768 83.9235 11.6205 201.069 34.4419 40.0498 6.30108 33.7488 166.627 11.4243 9.47668 4.43277 12.0522 155.203 22.8385 21.1229 4.17303 72.303 64.9121 13.6926 144.228 142.728\ninternal_count=246130 241281 180564 27348 24915 2433 153216 137887 17508 15329 9581 4849 2528 12170 529 11641 2321 218 192 96 2371 2103 1830 1533 442 5748 4551 1211 2023 1923\nis_linear=0\nshrinkage=0.03\n\n\nTree=186\nnum_leaves=31\nnum_cat=0\nsplit_feature=4 17 2 2 14 1 0 2 1 9 2 7 14 5 1 2 7 11 18 7 19 17 0 0 2 19 5 21 4 5\nsplit_gain=4.92235 4.00439 3.20898 2.86533 4.68828 3.75992 3.51077 3.94582 3.90747 3.3106 2.75316 4.65286 3.07981 2.91706 2.62579 2.593 2.44557 2.43652 4.0551 2.72095 4.29627 3.01637 2.57908 5.46064 4.30273 3.16368 2.43672 2.94596 5.65293 3.01396\nthreshold=0.84667968750000011 3.325091732892195 27.147908210754398 19.473560333251957 0.28481101427573913 2.3713805701640611 0.89788080022475569 2.5338792800903325 -0.17729835697704338 1.0000000180025095e-35 34.417465209960945 0.89599609375000011 0.45703470522544881 2.6015625000000004 1.5275240303550033 22.129880905151371 3.9980468750000004 0.58781685707414322 3.0201273556387074 0.89599609375000011 257.50000000000006 0.30925796400737565 1.6765212893675572 1.3771184689138856 -0.89641284942626942 245.50000000000003 1.7006835937500002 2.5000000000000004 4.1440429687500009 1.8691406250000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 3 5 -5 6 8 -8 -2 -9 12 13 -4 -12 -14 -6 -3 19 -19 20 21 -7 23 24 -23 -26 -21 28 -28 -29\nright_child=1 16 10 4 15 17 7 9 -10 -11 11 -13 14 -15 -16 -17 -18 18 -20 26 -22 22 -24 -25 25 -27 27 29 -30 -31\nleaf_value=-0.018226430549715008 -0.020638853402808778 -0.019502060146243349 0.024170796925055291 0.0073355930348325956 -0.033249304620111199 0.0072626724698972387 -0.0044446782164840907 -0.029690626033409013 0.0023720007821464717 0.0085578880119084391 -0.015910856813167902 0.0078982087296155298 -0.020304078808803163 0.014156584364568582 0.010238003592754346 -0.0076842002045459778 0.0073525886656911456 -0.022362838681465529 0.010989377723029376 0.012694184344083545 0.0095918092435548826 -0.027325790346088726 -0.029603781487632454 0.015237653902719722 -0.024691174520178411 0.015023054115561695 -0.036474625176557043 -0.011019338309637812 0.012544280933390096 0.0098197431101428936\nleaf_weight=12.975493369654485 6.3930978794069224 13.357449144518343 12.145160485441922 7.79538026894443 6.5472809910988863 9.232483650397624 30.436659984660764 11.26819804464321 64.338745286402172 1.9801253365003511 14.263989224734361 38.357135203957114 2.424882732734658 2.7646724967580658 8.8277104244043567 8.3005214631739346 3.304235207583587 7.2734493378084135 4.6476834978529951 40.822222463742946 21.220724288876227 9.9139364467773721 4.9737926060333839 6.3055577137274659 1.6214439621544503 4.8543319823511411 3.5299849594448451 6.873328470697742 3.2529081557877362 31.124724634370068\nleaf_count=42538 59162 57933 965 931 1903 333 4780 2116 60064 66 4854 2203 1890 154 276 2615 564 134 108 504 351 693 38 65 154 103 229 115 78 445\ninternal_value=-0.000734273 -0.000104195 0.00055788 -0.000893014 -0.0109076 -1.62116e-05 -0.00412483 -0.0107077 3.34369e-05 -0.0237745 0.00589205 0.00215094 0.0142966 -0.0105148 0.00253636 -0.0203441 -0.0140617 0.00301111 -0.00951567 0.0041087 -0.000897863 -0.007024 -0.0118782 -0.00690809 -0.0158144 0.00310649 0.00747914 0.00255167 -0.0143751 0.00586434\ninternal_weight=401.127 388.152 371.49 292.707 22.6432 270.063 114.417 43.685 70.7318 13.2483 78.7836 55.3858 23.3978 17.0287 11.2526 14.8478 16.6617 155.647 11.9211 143.725 58.1223 36.9015 27.6691 22.6953 16.3897 6.47578 85.6032 44.7809 6.78289 37.9981\ninternal_count=246364 203826 145329 134987 5449 129538 126188 6962 119226 2182 10342 7211 3131 5008 2166 4518 58497 3350 242 3108 1737 1386 1053 1015 950 257 1371 867 307 560\nis_linear=0\nshrinkage=0.03\n\n\nTree=187\nnum_leaves=31\nnum_cat=0\nsplit_feature=7 1 0 1 12 11 13 4 13 0 1 13 4 21 4 0 4 3 19 13 7 4 21 17 19 19 13 7 19 4\nsplit_gain=5.15053 4.65994 5.45033 4.64585 3.90909 5.099 3.69157 4.06359 4.84292 3.53773 4.87725 4.2967 3.50788 7.61103 5.21346 4.27676 4.25364 2.98552 2.87838 3.96548 2.8612 4.42589 2.84078 2.80889 2.79562 4.42448 2.76796 2.75506 2.73623 2.42895\nthreshold=0.89599609375000011 2.3713805701640611 0.17863154480046692 1.1008244052526537 8.8817841970012543e-16 0.91125284794847328 -0.52657366547836626 1.4160156250000002 -0.23926212085043433 1.8987378677022575 1.7090960825073716 -0.45067951570476578 5.6752929687500009 2.5000000000000004 2.1433105468750004 0.13844441961022361 2.9518512487411503 0.34948566548243237 249.50000000000003 -0.0081313880310213676 3.3837890625000004 8.2534179687500018 4.5000000000000009 1.3029497703631001 241.50000000000003 246.50000000000003 -0.26241780919347202 3.0205078125000004 243.50000000000003 2.1433105468750004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 12 29 -3 28 -9 10 11 -4 13 14 23 26 -16 -15 19 -18 -17 -22 24 -2 -10 -26 -14 -28 -8 -6\nright_child=4 6 9 -5 5 -7 7 8 22 -11 -12 -13 15 17 16 20 18 -19 -20 -21 21 -23 -24 -25 25 -27 27 -29 -30 -31\nleaf_value=-0.020224930344627582 0.016693553283306763 0.011308421513825988 0.0049543324396758188 0.02772011339567431 0.0063446185715135151 0.015334158393382773 -0.014846796441596396 -0.031954522746031264 0.021533364549391955 0.0090440661214563171 -0.029194263678064006 -0.017169533405947585 -0.016628825344328184 -0.0064646398802580118 -0.030305252892415962 0.019746617234436142 -0.025821995864077477 0.0072435711069436517 0.014494193839926249 0.0029860385032182226 -0.025599278044955147 0.011404317232909661 -0.028742558460853287 -0.014965376567586248 -0.027322353765701666 0.014953225165973687 -0.01223194819752864 0.022343294970325868 0.0044994669988739399 -0.014715601796562708\nleaf_weight=1.8239454881879726 7.8550278523925936 18.452406552023604 11.811805901537349 4.6955950731867233 5.3597248225398735 7.140537512721493 6.8463202868897488 7.4874998307859668 3.9667625824113184 4.7651897513242139 16.960032900093211 20.805824028821693 6.9076144657487912 15.799764205669133 10.525363521581314 29.734937799714316 8.0890000487052021 102.3679617178841 3.6745310724481888 7.809110167449945 2.7345935184184773 12.33613212656928 1.9615511739793863 2.6459333089615029 3.5541388759993424 3.4621354670998699 2.6933156281268111 4.034030711595733 39.958988812726602 27.632813605925094\nleaf_count=20604 2043 539 28637 1548 1185 126 534 533 166 443 4473 76913 21641 32006 9353 1800 9461 3577 2760 2635 149 274 174 1797 263 106 11486 2785 1292 6859\ninternal_value=-0.000284357 -0.00475532 -0.01103 0.0134009 0.00227035 -0.00622785 -0.000222806 -0.00350674 -0.0143286 -0.0141502 -0.0165142 -0.00924877 0.0038619 0.00152404 -0.00953529 0.0100807 -0.015754 0.00537183 -0.00683618 -0.0123323 0.0144829 0.00351487 -0.00257289 0.00810881 0.00410953 -0.00719867 -0.00438634 0.00870938 0.00141827 -0.0112083\ninternal_weight=403.893 146.552 60.8624 6.51954 257.34 40.1331 85.6898 67.2374 20.4321 54.3429 49.5777 32.6176 217.207 158.767 40.599 58.4406 30.098 118.168 19.5726 15.8981 44.8057 15.0707 12.9446 10.501 10.983 7.01627 13.635 6.72735 46.8053 32.9925\ninternal_count=246162 136225 132618 22152 109937 8170 3607 3068 1242 110466 110023 105550 101767 63632 28049 38135 24209 35583 14856 12096 2223 423 709 3840 535 369 35912 14271 1826 8044\nis_linear=0\nshrinkage=0.03\n\n\nTree=188\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 1 12 18 11 18 18 21 4 17 7 13 4 1 13 3 0 0 19 21 6 18 0 8 21 4 13 7 18 17\nsplit_gain=4.31846 6.71085 3.82452 5.23188 5.51339 3.5088 4.42601 4.24862 3.6499 3.13391 3.11145 3.39526 2.95276 3.74166 4.29433 4.35447 2.93224 3.98371 2.84192 2.73156 2.33736 2.1526 4.15917 2.72671 2.11835 3.9174 4.98968 2.82924 2.76992 2.17077\nthreshold=0.69651904584641833 -0.041910702480567281 0.52292651861841544 2.249904835165133 0.74021522701736098 4.8242976514536702 5.2755571088971331 3.5000000000000004 8.9774329662323016 1.1384572374241533 3.4804687500000004 0.14796868535883259 4.8846709132194528 1.7622051334857016 -0.20019894947294922 0.35479072494832037 0.7126865886964483 0.75738199555027408 264.50000000000006 1.5000000000000002 0.89599609375000011 5.1503930356556173 0.31303196667935901 1.0000000180025095e-35 2.5000000000000004 2.1433105468750004 -0.099843025280253614 2.6015625000000004 1.4978661367769956 1.490403556610195\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 12 4 -4 7 21 18 -9 -5 11 -8 16 14 15 -14 -2 -18 20 -6 -3 22 23 -7 25 29 27 -27 -28 -19\nright_child=2 5 3 9 19 6 10 8 -10 -11 -12 -13 13 -15 -16 -17 17 24 -20 -21 -22 -23 -24 -25 -26 26 28 -29 -30 -31\nleaf_value=-0.025947792272990718 0.024972630197178703 -0.0089381687648268978 -0.037755118452841532 0.0149191290250994 0.014998364167712799 -0.025555844079659402 -0.020786732027263086 -0.026238506463688506 0.01136374652167658 -0.015177864855257257 0.013125012334123134 0.014516414131303751 0.0085483984761988499 0.019234431043090677 0.014552288105220228 -0.02705922080477699 -0.025149587615828455 0.0057965636418985068 0.022326932624933949 -0.013940366872403694 0.010946442066002503 0.024467408502280213 0.019919364271617942 0.0086997394899381939 0.0035487692879553607 -0.030913254995847156 0.019874097055648295 0.014422133796186117 -0.010750814681199464 -0.012870129761698088\nleaf_weight=11.34610333782325 3.8037309699720927 22.390957337762764 6.6433757999038852 9.4803736689427733 3.5428449760656786 3.7016551981994343 10.749170962069231 12.799794256897256 2.2020781913161036 3.5125598922022609 4.3786905674305663 2.3334238341631126 4.9619555043427699 20.37006683673826 10.248372508962346 5.8370074497361202 4.0267068012894915 31.400459333474828 3.03889440434068 7.4500703022349626 5.9706496892904388 8.0454738221660573 7.0964556443805122 3.0686198824460007 167.83422890805741 9.873501523122286 3.7768970775650796 0.81341782555682662 5.0385993594745742 5.80833474224164\nleaf_count=122821 1824 25202 143 191 60 12136 20687 15675 71 91 656 54 495 220 333 369 4671 5870 787 145 507 4483 3315 322 17115 2215 89 3 806 4604\ninternal_value=0.000216592 -0.00525927 0.00197509 -0.00791582 -0.0181494 -0.00225519 0.00427495 -0.00776946 -0.0203538 0.00627939 -0.00678404 -0.0139064 0.00310637 0.0103894 0.00134486 -0.0113602 0.00176933 0.0012761 -0.00131456 -0.00414093 -0.00452184 0.012999 0.00514689 -0.0109081 0.00185938 -0.00311513 -0.0136552 -0.0265238 0.00305809 0.0026221\ninternal_weight=401.544 97.122 304.423 30.6292 17.6363 85.7759 39.3735 46.4024 15.0019 12.9929 17.4613 13.0826 273.793 41.4174 21.0473 10.799 232.376 228.572 31.4005 10.9929 28.3616 21.9122 13.8667 6.77028 224.545 56.7112 19.5024 10.6869 8.8155 37.2088\ninternal_count=245960 206716 39244 630 348 83895 41653 42242 15746 282 21397 20741 38614 1417 1197 864 37197 35373 26496 205 25709 20256 15773 12458 30702 13587 3113 2218 895 10474\nis_linear=0\nshrinkage=0.03\n\n\nTree=189\nnum_leaves=31\nnum_cat=0\nsplit_feature=4 1 2 17 9 11 1 1 19 19 13 0 0 4 2 13 19 4 0 13 13 1 2 14 13 1 0 4 0 1\nsplit_gain=6.36319 4.78386 2.93508 2.54672 2.46666 2.55983 2.03692 4.49728 3.79524 2.88076 2.48944 2.29953 1.90832 1.88313 3.56069 3.45585 3.61816 2.40317 3.80769 6.56432 2.92989 2.21644 4.47876 3.56808 1.98156 1.97451 1.91952 1.8656 1.85596 1.82468\nthreshold=0.84667968750000011 2.6981674883069502 -58.62553405761718 1.8807230532061787 1.0000000180025095e-35 5.5511151231257837e-16 0.98142480879535021 1.1267314367267358 233.50000000000003 243.50000000000003 -0.6105545148247048 0.88405423821904405 0.17393699858242903 1.6286621093750002 -38.806066513061516 -0.26766225074600319 246.50000000000003 2.3505859375000004 1.3170161875856263 -0.44620633588241487 -0.40579127137408294 1.8912989126705502 25.282213211059574 0.43714636302263388 -0.43125657894431851 1.4475910192202701 1.0659450432993143 2.6030273437500004 1.4460303622695712 1.4866206336930319\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 -2 4 6 -6 9 -8 -9 12 11 -11 -4 14 24 21 -17 25 19 -19 -20 22 -16 -23 -10 -15 -18 -21 -22 -5\nright_child=1 -3 3 29 5 -7 7 8 13 10 -12 -13 -14 17 15 16 26 18 20 27 28 23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.020176811990445305 -0.024607391993206374 0.0045018298109123726 -0.017828554342083004 -0.014489524783057185 0.017139371393322101 -0.0053809456689604875 0.018818331196904042 -0.023732667651030263 0.030905154960564746 -0.020380386481559622 -0.02745252922798138 0.020878543376403522 0.0065955631901951674 -0.018161348634098368 -0.024445474457772936 -0.022640711246931157 -0.011994255419469595 -0.035873949404189642 0.026763096040447042 -0.025133165709546167 -0.025229611872848438 0.00075184620985570478 0.018170169236613371 -0.036460983328005014 -0.010058795964603445 0.0085331146482506212 0.021882839666734846 0.0023018621088450662 0.0059083866811573014 -0.0019029243444646836\nleaf_weight=12.553299957593707 4.1615936732614491 168.51961709120951 2.8443011367044191 20.196415205910398 12.765728949298138 6.1964570845257176 9.7209752898279458 5.5171373416014822 2.3284513749604234 1.1920769786092926 8.13690722637193 1.7547926446513882 10.473258940839058 1.7297813430232043 3.5063381673362626 2.2694827719733386 1.24825666088509 5.5300034537267448 4.4842647173209116 1.5339171104187794 1.2964667738997357 3.177772051705686 3.4015240639328113 7.2005230898430455 0.98115560886708664 40.681294815592992 5.6341853292833548 17.736047383154983 5.9579331740969792 20.970400266892284\nleaf_count=42427 48365 3311 10519 54055 1600 668 2639 1520 239 7593 46316 554 12415 1043 1722 831 534 916 46 450 45 128 202 619 361 1975 105 1865 106 2800\ninternal_value=2.40335e-05 0.000740418 -0.00224784 -0.00168613 -2.52687e-05 0.00987929 -0.00134906 0.000235252 -0.00149599 -0.00908835 -0.019695 0.00325338 0.000498288 -0.000161266 -0.00646936 -0.0102724 0.0043524 0.00226677 -0.00332637 -0.00987655 0.010201 -0.0178299 -0.00381685 -0.026002 0.019245 0.00705108 0.0149965 -0.00101419 -0.00203877 -0.00827509\ninternal_weight=393.7 381.147 212.627 208.466 167.299 18.9622 148.337 123.936 114.215 24.4013 11.0838 2.94687 13.3176 108.697 29.7477 26.4381 9.15192 78.9497 36.5386 24.8 11.7387 17.2862 6.90786 10.3783 3.30961 42.4111 6.88244 19.27 7.2544 41.1668\ninternal_count=245969 203542 200231 151866 95011 2268 92743 15346 12707 77397 54463 8147 22934 11187 4741 4141 1470 6446 3428 3231 197 2671 1924 747 600 3018 639 2315 151 56855\nis_linear=0\nshrinkage=0.03\n\n\nTree=190\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 2 2 1 4 4 2 2 13 3 2 1 13 12 0 3 19 19 2 3 7 1 12 18 13 0 21 21 3\nsplit_gain=4.02123 2.80208 1.9253 4.36289 3.3051 3.13504 10.9564 6.19044 2.94736 4.60157 2.68501 2.56541 2.4084 2.91062 2.13172 2.39234 2.89204 2.11501 2.10613 4.08473 2.26885 2.1523 2.09871 2.3374 2.09497 1.83667 1.79763 1.78865 1.78496 6.60665\nthreshold=-60.293891906738274 -0.90051303354301027 22.129880905151371 22.867874145507816 3.3033179733375269 1.0693359375000002 1.6286621093750002 -37.900318145751946 -18.054428100585934 -1.0000000180025095e-35 0.65908129556435835 -39.283819198608391 2.2520445226790948 -0.28078353421895563 0.50308954086190749 1.3170161875856263 0.37248792739977959 264.50000000000006 253.50000000000003 30.514518737792972 1.0000000180025095e-35 3.1132812500000004 1.9634439907166827 0.38126483565173674 1.7005986908310777 0.14796868535883259 0.52258841453381233 6.5000000000000009 2.5000000000000004 0.45898141303402712\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 -2 5 -4 12 8 7 11 14 -10 28 -7 18 17 15 16 -3 -14 19 20 -5 22 -21 -24 -18 -16 -11 -28 29 -8\nright_child=1 2 3 4 -6 6 10 -9 9 26 -12 -13 13 -15 25 -17 24 -19 -20 21 -22 -23 23 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.022685899550602039 -0.025665097724946063 -0.015634921096760387 0.028042640879040341 0.016539405080216497 0.010985007669228077 -0.016150216786648242 -0.00067694077289699728 -0.033183758836390458 -0.005464648127315324 -0.012376265212817742 0.01012143414199935 0.029134716489051245 -0.027959440932122822 -8.018451354889365e-06 -0.027237766232536098 0.0265529265240163 -0.019265058111029274 0.012485927004546774 0.013788017260791964 -0.0061777290786568882 -0.024348150388942214 0.014904056611715573 0.021531393016910653 -0.01845320507533647 0.015940534733376824 0.01078574199632827 0.027216624045156668 -0.011580479533581331 0.00016400796161203602 -0.026830376016839432\nleaf_weight=5.9985143464325565 2.7181436613318501 3.1113049106388742 5.3760566626915525 0.72086883236079202 27.718011129327351 1.0866624069590216 39.239676414605583 13.778405877810654 34.209924770360431 0.89220281730194795 18.115023765371006 1.7225258948515174 6.938962219352109 11.669003798275297 1.4088107839579598 8.6433989807846938 0.84827939604292524 0.77668934990651894 10.353081672710685 18.045821419394546 7.0231853702539411 9.8976851332117786 4.7380476331600212 0.87638705258723337 14.110463217120921 1.4023084039799869 6.8491268620664414 0.67215768759251648 128.86986965033066 10.18683524561129\nleaf_count=49866 20461 17680 813 230 467 4115 35970 22467 20135 1495 324 425 261 523 57 197 16 26 1004 17742 5878 202 320 19 1274 27 818 70 42142 1125\ninternal_value=-0.000145622 0.000256145 0.000502436 0.00397316 0.00239492 -0.000774506 -0.00260468 -0.0252896 0.00459037 -0.000494365 -0.000570462 0.0119763 -0.00106268 -0.00980575 0.0117067 0.0143267 0.00739284 -0.0229226 0.00234225 -0.000784951 -0.0190858 0.00386836 -0.00116532 0.0134437 0.012862 -0.0104169 0.0181488 0.0227973 -0.00170716 -0.00649236\ninternal_weight=397.997 391.999 389.281 104.134 98.7577 285.147 212.999 16.5876 72.148 42.6234 196.411 2.80919 71.0397 19.3847 29.5246 26.7134 18.07 7.71565 51.6551 41.302 7.74405 33.5579 23.6603 5.61443 14.9587 2.81112 8.41349 7.52128 178.296 49.4265\ninternal_count=246149 196283 175822 27485 26672 148337 106568 27007 41769 22518 79561 4540 26205 810 19251 19167 18970 287 25395 24391 6108 18283 18081 339 1290 84 2383 888 79237 37095\nis_linear=0\nshrinkage=0.03\n\n\nTree=191\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 19 18 17 6 11 2 17 5 0 7 7 18 13 7 3 21 3 1 13 0 3 17 2 17 2 2 0 5\nsplit_gain=7.49133 5.15486 3.93745 3.68208 3.19828 2.94304 2.62666 2.48277 2.45476 4.77778 2.86212 2.78188 3.51872 2.72977 2.52304 2.42403 2.38538 2.28355 2.62056 2.06745 2.01944 2.93944 3.63649 3.41897 2.31782 1.98565 1.98163 4.00614 2.34045 3.73048\nthreshold=-39.283819198608391 2.6981674883069502 261.50000000000006 1.2424533248940002 3.1006363674515387 1.0000000180025095e-35 0.91125284794847328 -1.0000000180025095e-35 1.3227116620644523 3.5107421875000004 1.4460303622695712 2.6738281250000004 3.2382812500000004 1.8688348091416842 -0.6866788914798736 3.3134765625000004 0.45898141303402712 6.5000000000000009 0.28904431203898151 1.4866206336930319 -0.54808102951419191 0.24908287076908558 0.40160889672495398 0.84061941967369014 -30.838275909423825 1.9887490991681982 20.814876556396488 15.769127845764162 1.8987378677022575 1.7006835937500002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 6 4 7 5 26 8 -4 10 -10 20 -11 25 14 -12 -15 -5 -16 -19 -14 21 -2 -23 24 -24 -13 27 -3 29 -28\nright_child=1 2 3 16 -6 -7 -8 -9 9 11 13 12 19 15 17 -17 -18 18 -20 -21 -22 22 23 -25 -26 -27 28 -29 -30 -31\nleaf_value=-0.016807381198703841 -0.023496010650506223 0.0016868973656196137 -0.017097378284842667 -0.0044674405088718089 -0.023889550581453631 0.01562446185856404 0.01673067701449335 0.013017512384481416 -0.014823492749002179 0.016136643107503956 -0.021870325211069006 0.0012780315007084218 -0.0056067143829369708 0.025031004031106977 0.0090233266273167488 -0.018536994744687479 0.024313626411614497 0.010580342980024257 -0.029938142810268006 0.015131319742690504 -0.0051746961280704469 0.019636341717615982 0.010527045037228597 0.018329003870739023 -0.023254561363441045 -0.032272319536029985 -0.013067892129886001 -0.021050591518627992 0.01923752895807173 0.013134000082480791\nleaf_weight=20.861436577969471 2.0046112929703694 77.390741565875942 4.026347329840064 2.6405741303169661 2.8923179768462424 15.42727015580749 5.6357891050865865 3.8578652096912291 33.850411244592578 7.5266505957738472 2.5394895459467071 2.246539801327347 8.8705260632941627 8.1580372199387021 21.726892987106119 0.58402077085338611 19.677406876638997 1.8523694348405104 2.2828205623663962 6.801958931668195 77.149396379839345 10.943718949858523 1.8473472608166046 3.2351746733002074 5.6966748594059009 3.8680907318816935 7.6782851417883675 6.6553246806142843 10.548602097958794 10.227664750542315\nleaf_count=83761 5161 1603 31 61 410 242 102 21 69675 1565 70 467 1865 513 851 15 170 126 51 113 65133 9321 129 487 1291 1835 278 117 95 161\ninternal_value=0.000256459 0.00126714 0.00527564 0.0148637 0.00296611 0.00375387 -0.00186409 -0.0025551 -0.00246515 -0.00731336 -0.000209757 0.00156196 -0.00396025 0.00680144 0.00202574 0.0205164 0.0208628 0.005099 -0.0132619 0.00376145 -0.00283275 0.00468332 0.00820332 -0.00408581 -0.0147184 -0.0214987 0.00200274 -0.000339957 0.00869896 0.0018014\ninternal_weight=388.704 367.843 161.022 30.2022 130.82 127.928 206.821 7.88421 201.185 63.1642 138.021 29.3138 21.7871 37.1436 28.4016 8.74206 22.318 25.8621 4.13519 15.6725 100.877 23.7275 21.7229 10.7792 7.54402 6.11463 112.501 84.0461 28.4546 17.9059\ninternal_count=245720 161959 3189 283 2906 2496 158770 52 158668 75520 83148 5845 4280 1626 1098 528 231 1028 177 1978 81522 16389 11228 1907 1420 2302 2254 1720 534 439\nis_linear=0\nshrinkage=0.03\n\n\nTree=192\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 3 4 12 11 12 4 4 4 2 13 2 19 22 12 0 3 1 0 7 4 5 2 2 2 2 22 2 19 17\nsplit_gain=5.09127 3.18027 3.46548 2.61792 4.40085 3.22639 2.89629 4.47445 3.10574 3.78505 2.93376 2.87854 2.67899 2.58039 2.84304 2.56597 2.43415 2.33677 2.31663 2.29448 2.52781 5.95492 5.22555 3.71558 5.23152 4.49498 4.46509 3.52835 3.25671 3.22917\nthreshold=5.817026225953116 0.28904431203898151 4.9990234375000009 0.28848765710462548 0.74021522701736098 0.66389400333341531 1.6286621093750002 1.6572265625000002 1.0693359375000002 -37.900318145751946 -0.54808102951419191 29.286581993103031 262.50000000000006 0.93750000000000011 0.74643410817732136 0.70465417701170696 0.2352687277455576 -0.57436713779905635 0.61663808233260153 1.0000000180025095e-35 5.5515136718750009 2.1386718750000004 5.8049912452697763 11.680273056030275 -10.988907814025877 24.399772644042972 0.93750000000000011 -17.667204856872555 245.50000000000003 0.15621648314530187\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 6 5 -5 8 10 -3 -10 -8 -11 15 -6 -15 -12 17 -4 -13 -9 21 23 -23 24 27 -25 -26 -21 29 -24\nright_child=-2 3 16 4 13 -7 7 19 9 11 12 18 -14 14 -16 -17 -18 -19 -20 20 -22 22 28 25 26 -27 -28 -29 -30 -31\nleaf_value=-0.0099630643651846887 -0.024389976801823157 0.0019250508383523968 -0.015393270748137755 -0.0071580140961648511 0.01022154223261572 -0.036279541711653068 -0.0076913616837687169 0.022441494504195133 0.015718507926755762 -0.025007506290909802 -0.0048179045934569238 0.023941106167854651 -0.011930300449096065 0.019146462049623489 -0.021931397380910954 0.033375060801304694 -0.016385908889143409 0.0093236999032718874 -0.011001084422805273 0.0044956448321409395 0.010177907921664203 -0.026153659328251606 0.0082913893226155347 -0.017141689328095049 -0.010851800263716191 0.0064546544165554245 0.020193417518800171 -0.019690111421143543 0.0099007239418424133 -0.022295142246499929\nleaf_weight=44.290765030707917 6.3461528076013556 60.110271576221749 3.0537798419582094 11.898182072957754 15.215457371203231 4.2407669143140074 2.4928206106596944 4.7510620068060225 3.0699913253238291 11.388018966243182 1.5512300319132908 1.5495014599146077 1.0865106342380375 1.2150544459000219 4.6179813967319205 11.21120643627364 4.0838341743474293 24.76570505710697 4.1739895672708371 29.694857777226389 34.793631567488774 10.440507060224261 4.1676347861575795 12.326431871884781 4.1411233400667689 15.137026378277367 31.622296335126521 5.5952232118797829 11.158682595396384 7.7112786933976167\nleaf_count=135155 42524 9398 18041 1360 318 183 171 797 800 5072 144 341 41 4 100 94 1903 13212 496 3210 2545 3401 122 924 1578 860 578 610 530 1415\ninternal_value=0.000349786 0.000823946 -0.00462091 0.0021932 -0.00517214 -0.0164805 0.00322997 0.00525982 -0.00152611 -0.0114109 0.0198077 -0.0168774 0.0249422 0.00385275 -0.0118249 0.0287201 0.00287289 0.0061705 0.000612537 0.00376714 0.00310462 0.0011776 -0.0095772 0.00489182 0.00848117 -0.00436625 0.0164012 0.000224097 -0.00128934 -0.0117535\ninternal_weight=387.901 381.555 76.1941 305.361 37.1874 16.1389 268.173 187.882 80.2918 20.1815 16.3418 17.1115 13.8489 21.0485 5.83304 12.7624 31.9033 27.8195 5.72349 171.54 166.789 131.995 33.4781 98.517 71.0535 27.4635 35.7634 35.2901 23.0376 11.8789\ninternal_count=245927 203403 168311 35092 1965 1543 33127 17020 16107 6709 450 5909 279 422 104 238 33156 31253 837 16570 15773 13228 5468 7760 5976 1784 2156 3820 2067 1537\nis_linear=0\nshrinkage=0.03\n\n\nTree=193\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 4 2 2 13 19 1 17 1 1 19 5 2 1 18 13 18 14 4 2 2 18 19 1 19 2 7 4 5\nsplit_gain=5.93599 6.03596 3.74569 2.93095 2.5598 2.85515 2.54433 2.18091 1.89336 2.63053 2.27451 2.46155 1.95282 1.88207 2.68558 3.15844 2.28081 2.46654 2.92881 2.06399 2.70953 3.07616 2.20496 1.75378 1.66044 1.79024 1.64259 1.64109 4.0536 3.57413\nthreshold=0.24961073363565869 1.0000000180025095e-35 0.84667968750000011 -51.003999710083001 -51.003999710083001 1.0000000180025095e-35 264.50000000000006 2.3713805701640611 0.24468021674538498 0.45969269984116679 1.6109634896136267 253.50000000000003 5.7089843750000009 22.129880905151371 2.6981674883069502 1.4978661367769956 -0.07062347187798422 2.3502401828962087 0.22768215226658742 2.2978515625000004 -22.378742218017575 -16.072356224060055 2.9700856263602158 257.50000000000006 0.98142480879535021 229.50000000000003 -18.054428100585934 3.1132812500000004 5.9891631007194528 2.1386718750000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 4 6 -2 -6 -4 8 10 -10 24 -12 27 16 15 -15 19 -18 26 -9 22 -22 -21 -20 25 -5 -19 29 -29 -11\nright_child=2 -3 3 7 5 -7 -8 13 9 12 11 -13 -14 14 -16 -17 17 18 23 20 21 -23 -24 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.025582859670264853 0.014611199838839883 0.0022600660680803931 -0.028584459893663785 0.014033693783998717 -0.029587872020702823 0.01003041678470471 0.0066050296560283255 0.0071325930734112359 0.01258444161269115 -0.000195213588223159 -0.019033306064279717 0.0055618435552575556 -0.028732350612090892 0.0097975197989673742 0.012804619343795271 -0.0267809627686076 -0.014422777916814605 0.0055149954306504441 0.016688574408752273 0.017186597093073401 -0.03032617861033713 -0.0021074716539502374 -0.011334412250424577 -0.0095562150700678333 0.016617637358011372 -0.021184944177428833 -0.023713396921777707 0.031441164899283551 -0.0017048885553090987 -0.01312463318356125\nleaf_weight=14.511804167462742 1.3566491428459801 12.507817358875855 4.2000068228194314 1.514896270691682 6.9162232450430547 1.532247935363557 2.3773944380973262 82.541344682149429 8.6989866426670215 51.949016852001478 4.6246930110028179 9.6239950890885648 1.7595279400302377 3.9541922174394122 37.07645315505215 2.8176583691601991 12.330505120684395 2.2672719876863994 9.1888692233260354 8.9663539645262045 3.0665240781672791 25.290898534978624 2.4595667568682975 2.2424436969449735 14.764058285392819 1.7716483071990294 3.5980680971224528 3.0335088996944277 18.292659326209105 29.591698959439782\nleaf_count=157183 1493 8455 9485 374 14224 28 1187 1677 9056 22528 816 505 149 46 671 217 215 69 312 69 86 441 281 41 2411 3924 204 67 1025 8963\ninternal_value=0.000226088 -0.0130733 0.00126402 0.00179921 -0.0161397 -0.0221014 -0.0166722 0.00220271 -0.000576821 -0.00240669 0.00572294 -0.00314568 -0.00376045 0.00426238 0.00967378 -0.00690978 0.00264779 -0.00470914 0.00262536 0.00445339 -0.00114395 -0.00608805 0.010629 0.0111856 0.0125213 -0.00546453 -0.0132574 -0.00306079 0.00420701 -0.00498955\ninternal_weight=384.827 27.0196 357.807 348.002 9.80512 8.44847 6.5774 341.425 145.625 113.325 32.2993 14.2487 104.626 195.8 43.8483 6.77185 151.952 29.6272 17.2967 122.325 39.7833 28.3574 11.4259 11.4313 18.0506 3.28654 5.86534 102.867 21.3262 81.5407\ninternal_count=246202 165638 80564 64819 15745 14252 10672 54147 49818 41788 8030 1321 32732 4329 934 263 3395 841 626 2554 877 527 350 353 6709 4298 273 32583 1092 31491\nis_linear=0\nshrinkage=0.03\n\n\nTree=194\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 8 4 3 0 0 4 4 3 12 11 0 13 3 0 22 3 0 7 3 3 18 3 4 5 5 21 17 18 11\nsplit_gain=3.56111 3.45841 2.7974 2.64195 3.18266 5.08545 4.42287 2.44464 3.39836 3.26947 4.33213 2.69754 4.08447 3.20488 2.98057 2.96183 3.90476 2.90397 3.93451 2.51955 2.9044 2.20768 2.14374 2.03218 4.25322 1.8956 1.83449 1.79164 3.31361 2.95402\nthreshold=3.8863394171308765 1.0000000180025095e-35 0.84667968750000011 0.53650715206675215 1.1495945197163422 0.97942481616843191 2.4904785156250004 5.6752929687500009 0.50207005315878861 5.5511151231257837e-16 0.91125284794847328 0.69651904584641833 -0.42002175472665648 0.048385965359893195 0.72109735700429223 1.0000000180025095e-35 0.31033658837561479 0.97942481616843191 1.2426757812500002 0.3261289997719386 0.44825707527455844 6.8062759569045816 0.082806484040480313 2.7487792968750004 3.9023437500000004 1.4980468750000002 6.5000000000000009 0.12980400503841663 1.2424533248940002 0.38821517471137929\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 -1 7 5 -5 -7 8 11 21 -11 15 17 -14 -15 16 -4 18 26 -18 25 -9 -17 27 -25 -21 -13 28 -6 -29\nright_child=1 -3 3 4 23 6 -8 9 -10 10 -12 12 13 14 -16 22 19 -19 -20 20 -22 -23 -24 24 -26 -27 -28 29 -30 -31\nleaf_value=-0.013724542684362654 -0.026524229786466475 0.0031544003670872325 -0.015551063875296728 0.0044208701282459672 0.012387599298552843 -0.02983432677925376 0.003003885730771658 0.010434972775856526 -0.013158804465514286 -0.016906635839891736 0.017811786138385969 0.031008949774466294 0.02308675944168501 0.021901863929883157 -0.0049092860358434026 0.0077528568892991706 0.024357544949287459 0.0031854754375329358 -0.016018339447583883 0.0024713525091134649 0.014227797910023068 -0.015757236796482901 -0.025315092019468619 -0.018832495461667858 0.014649279410641817 -0.026358171132861697 -0.011165899941925369 0.014898197312656075 -0.019879790807207757 -0.010878380052940482\nleaf_weight=11.433224599142248 7.8982465060630584 5.5413433690700913 10.963292511751208 37.231675537830597 5.9171775355935079 7.8175395143480255 6.0607543414662359 38.661591847624095 23.54006516483787 10.678001728635079 3.560855622868984 9.4206572890193421 3.2295674956485518 2.9334186166797727 79.708056770019539 1.6374745822190551 4.1378172588392799 23.166446266584899 1.5195110470212956 3.6534585246245097 5.5712853582099342 2.2488320744032526 9.5719063907272339 6.0853759356541541 5.616509442217648 3.2684736796572906 0.52591648051566231 46.248509647048195 3.8914645919285249 3.6632042666897169\nleaf_count=32528 59902 3666 54256 2665 35 181 91 5668 1290 2254 77 2966 1340 827 11293 8013 1311 2549 640 2143 1088 17602 29560 119 77 2741 683 392 79 69\ninternal_value=-0.000298845 -0.0149163 0.000267885 0.000748353 0.00430891 -0.00139778 -0.0162556 -0.00109515 -0.00276433 0.00438966 -0.00762516 -0.00115588 0.00107236 -0.00244527 -0.00370713 -0.00792871 -0.00239248 0.00961814 0.0213168 0.0066718 -0.000556865 0.00865358 -0.0202452 0.00835554 -0.00287424 -0.0127505 0.0279627 0.0105671 -0.00106897 0.0128298\ninternal_weight=385.402 13.4396 371.962 360.529 122.532 51.11 13.8783 237.997 182.847 55.1493 14.2389 159.307 120.504 85.871 82.6415 38.8037 27.5943 34.6325 11.4661 16.631 12.4932 40.9104 11.2094 71.4222 11.7019 6.92193 9.94657 59.7204 9.80864 49.9117\ninternal_count=246105 63568 182537 150009 3708 2937 272 146301 120700 25601 2331 119410 20298 13460 12120 99112 61539 6838 4289 7283 5972 23270 37573 771 196 4884 3649 575 114 461\nis_linear=0\nshrinkage=0.03\n\n\nTree=195\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 3 1 4 9 11 11 3 2 3 4 17 1 3 17 17 7 1 0 0 17 2 1 3 21 17 0 11 3 1\nsplit_gain=6.52994 6.17219 3.77253 4.10086 3.68033 3.27501 3.04528 2.69421 2.76925 2.97083 2.55155 3.54924 3.23704 2.88565 2.69383 3.39939 2.40323 2.26496 2.25741 3.07145 2.49849 2.45052 2.55613 2.345 2.23785 2.17694 2.106 2.03963 2.027 2.02635\nthreshold=-39.283819198608391 0.58795853567030554 2.9477339485081675 2.6030273437500004 1.0000000180025095e-35 5.5511151231257837e-16 0.38821517471137929 0.31840721315068282 28.11904335021973 0.33966381556979741 2.2978515625000004 1.9887490991681982 2.9477339485081675 0.39370400768236996 0.25672585397575204 1.0329219259280322 1.4980468750000002 2.5144622974930591 0.83245446090221253 0.22637654874239863 1.2133651393498066 -8.3227787017822248 1.6109634896136267 0.10824180149310623 2.5000000000000004 0.61464819824310435 1.1851094193663554 0.71017588656534381 0.37963534769016083 0.95935769349451683\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=17 4 3 -3 7 -6 -4 18 9 24 -11 12 13 28 -14 -16 -13 -1 19 20 23 26 -23 -2 -9 -19 -20 -7 -12 -10\nright_child=1 2 6 -5 5 27 -8 8 29 10 11 16 14 -15 15 -17 -18 25 21 -21 -22 22 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.026331181649814902 0.019511396418533781 -0.0083377441278648934 0.015705732395306555 0.011686629172702743 0.015676270063252621 -0.016228183246908653 -0.011769832774141335 0.021656194387484125 -0.022173541058271479 -0.00060250891655093267 -0.018669655336985228 0.02416576042431829 -0.019965593418748918 -0.029283749718182484 0.014888902340900102 -0.016622326247779654 -0.0036766502499261493 -0.021116266574025711 -0.010646260748063864 -0.023964847572342785 -0.016112334756763953 0.0016699017087461999 -0.015102031408675255 -0.011899767303769222 -0.0063164197390259873 0.0029944628400892293 0.020424597193066711 0.0092945033605906918 0.009951066893100536 0.0075159325339253808\nleaf_weight=9.251327578977909 4.5612750081963913 19.725448940062051 42.83760907931719 15.560864912171381 18.830250519024961 5.3848346159920748 3.2437542332336298 6.5173397483840727 1.2256026767154242 80.843170400034978 3.5188833718630148 3.0169984713793374 5.038865891168828 13.239953955995587 8.0480393940233608 3.7141817393712691 8.462224086513741 4.5426701923206556 2.0950605388643444 13.281707724910573 6.4396818471082042 16.24009807650555 15.117403564881895 2.630206348975662 3.3499067852799271 8.069110205746254 5.3988003183349065 4.1702575406525284 3.4956286949727655 39.809904524667218\nleaf_count=83121 6777 1323 339 890 2503 786 61 1677 1341 9844 1048 576 70 1988 122 72 1450 328 3873 74684 25259 11660 1232 10088 1592 541 922 103 273 1464\ninternal_value=-0.000195652 0.00080081 0.00800424 0.000571488 -0.00135629 0.00868818 0.0135628 -0.00254556 -0.000635337 -0.00265641 -0.00389537 -0.00925732 -0.0136864 -0.0214839 -0.00360735 0.00441574 0.00499106 -0.0157377 -0.00769415 -0.0142164 -0.0037291 -0.00293275 -0.00663266 0.00797287 0.0124522 -0.00660335 0.0115074 -0.00526374 -0.00494474 0.0061228\ninternal_weight=377.661 355.798 81.3677 35.2863 274.43 28.3853 46.0814 246.045 180.281 139.245 129.378 48.5348 37.0556 20.2545 16.8011 11.7622 11.4792 21.8631 65.7642 26.9129 13.6312 38.8514 31.3575 7.19148 9.86725 12.6118 7.49386 9.55509 7.01451 41.0355\ninternal_count=246007 162017 2613 2213 159404 3392 400 156012 21517 18712 15443 5599 3573 3309 264 194 2026 83990 134495 116808 42124 17687 12892 16865 3269 869 4795 889 1321 2805\nis_linear=0\nshrinkage=0.03\n\n\nTree=196\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 0 4 4 5 6 1 21 4 10 18 13 13 1 1 3 21 4 21 19 2 0 13 1 18 2 0 4 19 18\nsplit_gain=6.48968 2.86503 3.49391 2.63269 4.76045 3.87451 3.65344 2.84298 5.09128 2.70566 2.61282 2.73588 3.35097 3.5193 2.48048 2.45189 2.77631 2.58775 2.31529 2.43148 2.83037 2.24618 5.27556 2.29639 2.4227 2.26609 2.4762 2.22061 2.21781 2.15221\nthreshold=-34.692844390869134 1.3771184689138856 0.84667968750000011 5.5515136718750009 4.4589843750000009 2.0131835937500004 0.7474413184595442 2.5000000000000004 7.3568274974823007 2.8029010331479989 2.0126758453675748 -0.74086763132968725 -0.36630902128874782 1.6109634896136267 -0.27208026740318675 0.10728613048115772 5.5000000000000009 1.0693359375000002 2.5000000000000004 264.50000000000006 1.2144355773925783 1.3170161875856263 -0.54550535683058499 2.6981674883069502 1.2424533248940002 13.964207649230959 0.60985949876859957 6.4848632812500009 246.50000000000003 5.0271592211193568\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 -2 15 7 6 27 8 28 -7 11 -3 18 14 -14 17 21 -4 -13 -20 -21 23 -23 29 -25 -26 -27 -6 -5 -17\nright_child=1 10 3 4 5 9 -8 -9 -10 -11 -12 12 13 -15 -16 16 -18 -19 19 20 -22 22 -24 24 25 26 -28 -29 -30 -31\nleaf_value=-0.014834314623001938 -0.021468759894832774 -0.017806920283790831 0.0070163866833754416 0.029460365025134529 0.020910078864459843 0.0076693896491021318 -0.03424412508516407 0.00079476241758096272 -0.013481695631564854 -0.026191398129139773 0.014767709480498041 -0.010945115853391548 0.0071200789518214985 0.0017005421955314894 -0.027847513993677225 -0.0050456233154499222 -0.011929587151452358 -0.026444485708639637 0.014547394147315229 -0.026898019316426734 0.0087751374418333587 0.018077921391806723 -0.032945419471548852 -0.023332024128702753 0.0028105417029399109 -0.01616905151465009 0.02058578712963367 -0.019677880593940964 -0.014004099376947785 0.0093471973365161273\nleaf_weight=25.049235510742619 6.5807016080730127 4.7441759590947177 2.4202486948579445 14.269928281055403 1.9197323666121549 16.086914111016998 4.9236691854293895 10.66054889620318 2.7490768426896457 1.4394780940419867 16.914836154309342 3.7707795953829182 2.0452683599432913 60.521421200377517 5.59602985449692 58.060557177710962 19.689633057628726 6.246151910622757 36.237383358391526 2.1282486581185358 4.6646858653984964 2.012222611985635 4.2467850746106697 1.5737891360186029 39.351250681720558 1.1300367700896448 12.658340225185382 1.1093165280044557 0.56577762933920972 10.161947169333869\nleaf_count=93755 32292 175 6228 2033 536 983 448 3464 674 215 1285 256 355 915 1063 38524 9729 33753 464 40 42 49 289 27 990 106 223 1439 917 14866\ninternal_value=-0.000381432 0.000678737 -0.00145608 -0.000734537 0.00494296 -0.00438659 -0.0204749 0.013221 0.0203062 0.00362471 0.00406872 0.00244713 0.0034293 -0.000829648 -0.0187472 -0.00268951 -0.00172667 -0.0173408 0.00951976 0.011521 -0.00441878 -9.20392e-05 -0.0163112 0.000858668 0.00538621 0.00665198 0.0166838 0.00485101 0.0270231 -0.00279771\ninternal_weight=379.528 354.479 217.856 211.275 53.7244 25.4791 7.95272 28.2453 17.5848 17.5264 136.623 119.708 114.964 68.1627 7.6413 157.551 148.885 8.6664 46.8011 43.0303 6.79293 129.195 6.25901 122.936 54.7134 53.1396 13.7884 3.02905 14.8357 68.2225\ninternal_count=246135 152380 147785 115493 10709 3621 2423 7088 3624 1198 4595 3310 3135 2333 1418 104784 64803 39981 802 546 82 55074 338 54736 1346 1319 329 1975 2950 53390\nis_linear=0\nshrinkage=0.03\n\n\nTree=197\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 1 9 5 6 21 0 13 0 7 7 7 7 1 0 0 1 1 7 3 1 14 3 0 3 3 0 1 1 7\nsplit_gain=4.28226 3.6731 3.68655 3.21257 4.20683 3.10378 4.12696 3.39426 3.12935 4.18041 3.34451 3.19168 3.15773 2.94051 3.39847 2.86672 3.91925 3.05448 2.82609 2.74444 2.75058 2.66329 3.46063 2.57215 2.51618 2.49014 2.0808 2.10984 3.40522 3.14124\nthreshold=1.3771184689138856 -0.70140436206657719 1.0000000180025095e-35 4.2695312500000009 2.4414062500000004 3.5000000000000004 0.69651904584641833 0.14796868535883259 0.13844441961022361 3.4169921875000004 2.5244140625000004 2.6738281250000004 2.8623046875000004 1.7622051334857016 0.46743596317556629 0.97942481616843191 1.8912989126705502 2.1468275719512682 3.2763671875000004 0.47260169401963864 -0.58235837231666587 0.67646516896599407 0.45898141303402712 0.17863154480046692 0.48555671035752257 0.33504387972412514 1.3170161875856263 2.6981674883069502 2.2520445226790948 1.2426757812500002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 5 -4 -5 7 13 8 9 -3 19 -12 -13 -7 24 16 -8 -17 -14 20 -10 -18 -23 -22 -15 -20 27 28 29 -21\nright_child=-2 2 3 4 -6 6 15 -9 10 -11 11 12 18 14 -16 17 21 -19 25 26 23 22 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.02622621769460394 0.0041555070084444883 -0.027208474329504989 0.020237153868093501 -0.033031437335842573 0.0072215470326056829 -0.030057012080776366 0.015163649920513209 0.015494650566615148 0.024980633988840643 0.016650838655061579 0.03060321019820951 -0.026052575935524999 0.020554189140898006 -0.017676178991152796 -0.025002119883781246 -0.023597246691903481 -0.01906580617154182 -0.0038380426668323283 0.01391317834718349 -0.0091959321534382035 0.016616181081327057 -0.027647399073351807 0.014967768618869089 -0.010453163023759289 0.014576251712161025 -0.021216231056723701 -0.023793880851739006 0.011235880796303662 -0.025243938685243907 0.017871051934574094\nleaf_weight=4.6954050448334774 140.76129465904432 7.2081606819060635 11.096983225617352 1.9895884707279936 9.7331470141748451 9.5357943386484312 13.167437450179024 10.55283510412846 1.3386557976368667 1.8252356763105124 4.4843940325454268 1.7704217740993033 7.0750106226751068 2.2613828067187551 4.977485489072933 8.388563497800531 6.8472262004088389 30.369566244800808 2.3230274182133144 6.8485780515638925 2.5125160814445797 1.2706134194231706 6.252697331350646 44.363842018223295 5.7065823390003061 3.1496872913266998 1.4226131955219887 20.437151104735676 3.5657207223121068 6.6915918146078184\nleaf_count=54031 4721 30042 2189 777 633 58990 7293 223 4399 655 254 236 403 1360 889 3710 435 646 81 2086 3472 122 153 66616 471 135 64 488 310 381\ninternal_value=1.22894e-05 -0.0024062 -0.00182648 0.00938584 -0.00166949 -0.00305821 -0.00731768 -1.27433e-05 -0.00155685 -0.0175702 -4.04836e-05 0.0111012 0.00339413 -0.0181428 -0.00784068 -0.00343178 0.00388943 -0.00860056 0.00916526 -0.00254209 -0.00726221 -0.00675536 0.00537117 -0.00868552 0.00447237 -0.00645894 0.00333404 0.00495265 -0.00276015 0.00448976\ninternal_weight=382.623 241.862 237.167 22.8197 11.7227 214.347 88.7773 125.569 115.017 9.0334 105.983 18.8025 14.3181 22.4812 12.9455 66.2961 27.538 38.7581 12.5477 87.1807 48.215 14.3705 7.52331 46.8764 7.96797 5.47271 38.9657 37.543 17.1059 13.5402\ninternal_count=246265 241544 187513 3599 1410 183914 74069 109845 109622 30697 78925 1109 855 61710 2720 12359 8003 4356 619 77816 74487 710 275 70088 1831 216 3329 3265 2777 2467\nis_linear=0\nshrinkage=0.03\n\n\nTree=198\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 7 5 5 3 19 18 4 12 21 13 18 13 4 18 1 18 21 5 3 18 4 18 21 18 14 7 18 17 17\nsplit_gain=3.00709 2.66326 4.64963 4.56595 2.99784 2.93631 2.85135 2.77305 2.47308 3.41566 2.78736 2.69415 3.60255 2.68909 2.46913 2.29575 2.16929 2.1601 2.04766 2.18727 2.76625 3.10321 3.24322 2.70308 2.34964 2.34374 2.04511 2.02453 2.33549 2.28448\nthreshold=3.9281938047451082 0.89599609375000011 0.89599609375000011 1.7006835937500002 0.56033297298897644 267.50000000000006 2.3502401828962087 0.84667968750000011 5.5511151231257837e-16 1.5000000000000002 0.14796868535883259 5.2390944889447963 -0.044585747678852421 3.9316406250000004 4.8242976514536702 -0.69268724735312648 2.249904835165133 1.5000000000000002 0.89599609375000011 0.58795853567030554 1.2424533248940002 2.2978515625000004 2.1383330595080281 5.5000000000000009 2.602003343538398 0.96902457618464355 3.4804687500000004 1.7005986908310777 0.72612140026470018 0.29595754201755942\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 7 4 5 15 27 -1 11 -10 -11 14 26 -12 18 -4 -6 -16 -3 25 -21 22 -22 24 -23 -20 -13 29 -29 -5\nright_child=-2 8 3 6 16 -7 -8 -9 9 10 13 12 -14 -15 17 -17 -18 -19 19 20 21 23 -24 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.010587659091711993 -0.013604294575536571 -0.023253724661363297 0.012152628211352703 0.016707136174647919 0.004467944321545703 0.014435656068297239 -0.012725936139025704 0.00460836512614448 0.0058927800168263643 -0.02141235374962213 -0.01772382822751736 -0.02919652728413847 0.0093660433496217076 0.013364000513345306 -0.0074756757453177981 -0.026131722637490102 -0.02305318985117762 0.023341270602650663 0.0030255261903174993 0.022704907329281307 0.033756283137026263 -0.0074936784744615882 0.00070602657714953921 -0.020631384390387503 0.014042792021193335 -0.026198833425489888 0.0048919156729099337 -0.0066494075654356692 0.032290547138047486 -0.01267647678866538\nleaf_weight=11.905949991334181 12.435869579161588 1.4548090737453083 0.94420229951492984 3.134367249025674 9.2085735886648745 1.2876482564206515 6.528229025167092 66.265980949826371 15.651635609878666 11.428555831798803 3.2084075693855993 5.5048743069464745 5.1080727746948469 5.1881158776814118 2.023638441705315 21.255197125083217 2.4644007517672426 9.724318380361181 124.80374658846722 10.918294155038891 4.7668606303632277 7.5962078995071414 5.0609688226832077 4.0742144596442804 8.8771238579101901 1.5266252394776518 1.7531666368636249 1.4990595095587194 4.0584045252726355 4.6164592276218173\nleaf_count=32893 62352 1074 5605 89 134 552 17459 15629 1593 4621 73 34393 4894 109 3272 32088 752 6277 18929 65 16 88 66 89 305 876 1534 268 140 202\ninternal_value=0.000333768 0.000848974 -0.00251202 -0.00911442 -0.0154638 -0.0214365 0.00234176 0.00218944 0.0028087 -0.00438338 -0.0123897 0.00414518 -0.00892129 0.000862967 0.0050829 -0.0240497 -0.00270293 0.0178629 0.00409511 0.00446897 0.0103833 0.00537186 0.0183737 -0.00141097 0.00425135 0.00246939 -0.0213672 0.0101059 0.0223747 -0.00024266\ninternal_weight=374.274 361.838 133.168 54.9965 35.16 23.487 19.8365 78.1719 228.67 35.4767 19.8251 193.193 12.3661 8.39652 180.827 22.1994 11.673 11.748 169.079 167.624 41.2937 30.3754 9.82783 20.5475 16.4733 126.33 7.25804 13.3083 5.55746 7.75083\ninternal_count=246437 184085 105811 57289 39131 38245 18158 48522 78274 6396 4803 71878 40821 182 31057 37693 886 9549 21508 20434 629 564 82 482 393 19805 35927 699 408 291\nis_linear=0\nshrinkage=0.03\n\n\nTree=199\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 16 3 18 14 7 2 2 0 19 1 0 19 5 18 18 17 2 1 2 14 21 14 4 3 4 14 11 14 10\nsplit_gain=4.10689 5.17754 4.13186 3.20171 2.51754 2.18104 2.1471 2.5276 3.08223 3.06722 2.07487 6.3768 3.48894 2.63094 2.43748 2.88973 2.73426 2.02687 2.89857 2.32612 2.50847 2.2858 1.99455 1.98522 1.97467 2.33736 1.86566 2.54116 1.85969 1.84748\nthreshold=2.3713805701640611 1.0000000180025095e-35 0.21366020901822882 2.524928003624769 0.050734622491290134 1.4980468750000002 28.11904335021973 32.956295013427741 1.1851094193663554 240.50000000000003 0.7474413184595442 0.85792061509615258 227.50000000000003 4.7402343750000009 2.6735537653587351 2.1383330595080281 0.2328254379290535 28.11904335021973 2.054932859494814 -10.988907814025877 0.40875666244379477 7.5000000000000009 0.46513657297606553 6.9096149206161508 0.53650715206675215 2.4204101562500004 0.87121128816919835 0.56850430734128443 0.032916404154818342 3.6242520361853052\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 4 28 -3 -2 9 -9 -8 11 -4 -12 22 15 17 -16 19 -19 20 -14 -20 -13 -7 25 -10 27 -21 -1 -25\nright_child=6 5 10 -5 -6 23 7 8 24 -11 12 13 14 -15 16 -17 -18 18 21 26 -22 -23 -24 29 -26 -27 -28 -29 -30 -31\nleaf_value=-0.011071729075766963 0.0010971349875288743 0.017530902620512781 -0.018750145614582773 -0.026498776961962987 -0.015256196553350953 -0.014283554488904612 -0.011822188463824514 -0.014845894027962015 -0.013355836443205111 0.027706254439388539 -0.028740584614791235 -0.01230344538481541 0.019514598670944606 -0.01957707309226105 0.01138083572711659 -0.02559554506101722 -0.0042607972132994876 0.011336264305521164 -0.032279575857753338 -0.026763701330938625 -0.0099794607559640213 0.013051117781582444 0.029574706402523205 0.0062921552446913338 0.018319386824771463 0.010688867716630095 0.0048075314219662661 0.012862767218036643 0.027089325313729033 -0.023577350031664869\nleaf_weight=1.0987013351641528 159.96519297714258 8.9447531783753202 4.9070336246345496 13.937499293963354 4.9611247601897048 5.0045747660369662 1.5721936807822192 5.3074388131863115 5.1698590625564993 8.57298089120013 4.6247148707534498 0.61050788719785121 3.1720919116414716 0.5927461492246956 13.689021399645751 7.0482928954006629 30.922917831921495 10.16732440767373 1.9984351152088486 10.650231493846151 5.8786464934019023 0.83929090946912754 7.6588474917271014 29.170385197011914 9.7634581025340577 7.8762529725790955 2.3708088689199931 1.0643774606287468 2.2312694235333765 1.0121065632820307\nleaf_count=464 4851 2831 29017 164538 4953 2994 40 356 195 100 1929 475 79 49 4426 1198 17738 298 64 565 250 4 2396 5760 78 141 179 11 83 196\ninternal_value=-0.000292372 -0.00366531 -0.00669974 -0.0181876 -0.00286806 0.00511768 0.00264741 0.00893573 0.0039844 0.0210694 -0.00414788 0.0063869 -0.00576886 0.0207954 -0.00424886 -0.0092434 0.000683072 -0.0054511 0.00384235 -0.0106178 0.0012707 -0.0189652 0.0254892 0.00156492 0.00880543 0.000888037 -0.0178347 -0.0224348 0.0148492 0.00456656\ninternal_weight=370.783 172.556 128.424 22.2286 8.2911 44.1318 198.227 38.2622 28.117 10.1452 106.195 13.7691 92.4262 8.8621 87.8014 43.1895 44.6119 36.1412 13.0051 23.1362 9.05074 2.83773 8.26936 35.1871 22.8096 13.0461 14.0854 11.7146 3.32997 30.1825\ninternal_count=246258 240497 228716 170038 5500 11781 5761 910 770 140 58678 31937 26741 2920 24812 2648 22164 1450 366 1084 329 68 2871 8950 414 336 755 576 547 5956\nis_linear=0\nshrinkage=0.03\n\n\nTree=200\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 3 6 3 11 5 12 4 18 2 17 18 1 19 2 5 1 1 5 17 17 13 17 2 18 1 4 17 19\nsplit_gain=5.35212 2.53476 3.74311 2.29952 3.19531 2.10717 2.03531 2.00615 3.27079 3.89502 5.08374 4.30113 3.57847 3.53909 3.65104 3.20436 3.16225 3.08319 2.8541 3.52429 2.91311 2.69457 2.55945 2.3083 2.16344 2.56274 2.08151 2.0803 2.02679 2.13847\nthreshold=2.6981674883069502 -31.266788482666012 0.53650715206675215 2.9179687500000004 0.29938127432470335 0.91125284794847328 1.7006835937500002 0.083740223399813532 5.5515136718750009 3.1567740231385479 7.8238990306854257 1.2428969707451916 5.2496487501035594 0.39417715631504779 259.50000000000006 12.256900787353517 1.4980468750000002 1.7622051334857016 1.6575619461558879 2.7412109375000004 0.10726386778582141 0.81373174126422521 -0.28983840832261526 1.1891760317644442 -1.3165187835693357 1.2424533248940002 1.6109634896136267 1.0693359375000002 0.28904399529622732 249.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 4 -2 7 -4 8 9 11 12 13 -11 -3 18 26 -9 -13 20 28 -15 24 -17 -22 25 -16 -12 -14 29 -20\nright_child=3 5 6 -5 -6 -7 -8 16 -10 10 15 17 27 14 21 22 -18 -19 19 -21 23 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.022672706688516713 -0.015530728698183246 0.023041993632111773 0.018390062699391283 0.025832420601944933 0.0038131660475064768 0.014315781271789604 -0.018891120076335852 0.0073042349197353562 0.0047404377237138971 -0.027473346658694046 -0.0047879827269163714 -0.00035648859013127698 0.024519083482700266 0.010519512568001165 0.015974580459914123 0.0045376279141066825 -0.018771654840391162 -0.024417660185348515 -0.021381236901489707 -0.023634776714087792 -0.025965317324796268 0.0081763582338341256 -0.022302588217406612 0.012902628870048278 -0.036938840340386296 -0.01971726667033993 0.031937315400880836 -0.011289000109311205 0.017331453785826754 0.0098723167294497063\nleaf_weight=12.274684452268732 7.1019182842555884 6.0787753970171243 3.3420134114421662 3.0509461557958266 157.36723810377225 5.3370034061372271 0.97341531456913788 5.0808183770604982 43.837286135845716 17.383277286864544 1.8279905974586643 9.044208563673239 1.3395236061919593 4.1963264614023483 1.9348400750895951 6.2574821426596818 14.790995642692904 9.2695551911128842 2.0774384257383707 1.7002454265020777 5.6243112234196815 3.4137963988614493 4.9997551940282383 1.0802985151094615 3.3924568653219458 3.9944435630459338 2.789036745483827 2.9141726279240174 20.124905427095655 4.493574805521348\nleaf_count=100216 702 4817 243 55 3183 105 144 537 8896 61206 5676 7251 65 443 23 10988 1961 722 231 62 2410 160 22827 139 211 312 380 11316 683 131\ninternal_value=-0.000484856 -0.00379498 -0.0146902 0.00345773 0.00288907 -0.00273269 0.00800874 -0.00332214 -0.00215072 -0.00482029 -0.012601 -0.000887892 -0.0217288 0.00313973 0.000423982 0.000390712 -0.0120741 -0.0131688 0.00496915 0.00989042 -0.00760429 -0.0129414 -0.0082301 -0.0188408 -0.020718 -0.00744578 0.0191331 0.00250793 0.0128019 -0.00152755\ninternal_weight=367.093 199.573 16.5901 167.52 164.469 182.983 4.31543 177.646 157.774 113.936 37.5112 76.4252 21.637 58.1114 52.0326 15.8743 19.8718 18.3138 39.2971 28.3962 10.9009 12.7355 11.2572 6.70461 9.32174 5.92928 4.61703 4.2537 26.6959 6.57101\ninternal_count=246095 242155 100603 3940 3885 141552 387 141447 138949 130053 112458 17595 72587 9622 4805 39871 2498 7973 4099 1107 2992 706 33815 2549 546 335 6056 11381 1045 362\nis_linear=0\nshrinkage=0.03\n\n\nTree=201\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 1 5 14 5 14 3 3 1 17 4 19 14 3 13 0 3 6 6 19 4 19 3 3 4 0 3 19 4 13\nsplit_gain=4.95909 3.34689 2.49157 2.89419 2.18814 2.40606 2.16076 1.73328 3.82118 1.83577 1.71488 3.90139 2.8397 2.28673 2.08815 1.75462 2.0387 1.71483 1.89636 2.42139 2.77604 2.27739 3.43869 2.31098 1.68946 2.64596 2.20824 2.15172 2.03672 4.16553\nthreshold=3.5426598534814651 3.3033179733375269 3.7255859375000004 0.043928974430191954 1.2426757812500002 0.74790089217118771 0.50207005315878861 0.016076142967085191 -0.32709731491006455 3.5112638802509566 1.6286621093750002 254.50000000000003 0.4896816566336385 0.62297257604742995 -0.36167213950920335 1.4460303622695712 0.58795853567030554 4.2832031250000009 2.2500000000000004 200.50000000000003 5.5515136718750009 241.50000000000003 0.56033297298897644 0.62297257604742995 2.2153320312500004 1.8987378677022575 0.44825707527455844 238.50000000000003 1.6286621093750002 -0.38940835686028247\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 7 3 -3 10 6 -6 -1 9 -9 11 13 -12 15 -13 -5 -17 18 19 -10 24 23 -23 -22 27 -26 -27 -21 -29 -30\nright_child=-2 2 -4 4 5 -7 -8 8 17 -11 12 14 -14 -15 -16 16 -18 -19 -20 20 21 22 -24 -25 25 26 -28 28 29 -31\nleaf_value=0.012945387939260517 -0.016979031037996608 -0.015323638390030002 0.021967124951875604 -0.0046338081863922962 -0.0040308771173436439 -0.01330556895639767 0.0095541799452497408 -0.02640857356508464 -0.020409824912037206 0.0188000826380612 0.032611652502409674 0.025857999571338112 0.0035657468038362812 -0.018883233311569197 -0.0017805619764245995 0.0016908824953466568 0.027183724579299837 0.013313385928911296 -0.014904309712259454 -0.01613760339059583 0.014885598529461697 -0.018022511494181364 0.019620797537015025 -0.013423304613236875 -0.0067097880642797009 0.020195736644776228 -0.012699972511854202 -0.0015440143206585854 0.014493243184022741 -0.0035030107822912049\nleaf_weight=7.4314973587079285 12.519194980191285 5.3006368287897194 7.043369890889152 10.845206071884609 17.018891509738751 7.6156594164349363 24.893427557428367 5.9480170078536494 4.2014205483110336 0.203200023383033 5.9998296932317299 9.6771911998512206 5.8258796777226962 3.854810945224016 3.0368217392824581 7.4657707102596742 3.6866990104317665 7.1241249563790907 7.1712244720997651 5.1315447386295991 24.042365710184644 5.4147503353997299 2.3585043905768543 2.1194873634667593 64.154113514610799 5.4073039239010532 1.7921476893243378 58.743115393223434 26.007148056386086 19.694941974901212\nleaf_count=6562 66305 118 98 381 272 265 192 62802 1462 286 33 83 75 50 46 98 16 241 863 9644 2803 4783 24 227 31384 112 21 42540 5466 8787\ninternal_value=0.000862759 0.00152999 0.00580003 0.0045212 0.00571535 0.00118988 0.00407246 -0.00042678 -0.000893783 -0.022495 0.0100535 0.00667374 0.019696 -0.000180482 0.0196079 0.00377552 0.0116615 -0.000230839 -0.000714051 -0.000183913 0.000304296 0.00804873 -0.00566647 0.0121824 -0.0011845 -0.00474379 0.0114569 0.00115516 0.00214976 0.00682896\ninternal_weight=371.728 359.209 112.264 105.221 99.9202 49.528 41.9123 246.945 239.513 6.15122 50.3922 38.5665 11.8257 25.8525 12.714 21.9977 11.1525 233.362 226.238 219.067 214.865 33.9351 7.77325 26.1619 180.93 71.3536 7.19945 109.577 104.445 45.7021\ninternal_count=246039 179734 1727 1629 1511 729 464 178007 171445 63088 782 674 108 545 129 495 114 108357 108116 107253 105791 7837 4807 3030 97954 31517 133 66437 56793 14253\nis_linear=0\nshrinkage=0.03\n\n\nTree=202\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 4 3 2 19 2 13 2 17 4 14 4 14 3 2 14 6 17 10 19 19 21 4 14 2 2 1 13 4 1\nsplit_gain=4.77926 2.56927 2.79373 2.37948 2.61076 2.13482 4.38306 3.04664 2.15548 2.81462 1.99623 1.98701 5.46382 3.87518 2.68396 2.15285 1.8954 2.0922 1.86357 1.84342 1.95119 1.81657 1.80291 1.77566 2.59506 5.18186 2.57887 2.55667 3.08689 3.34139\nthreshold=3.3033179733375269 3.9514160156250004 0.58795853567030554 -34.692844390869134 264.50000000000006 -2.5585746765136714 -0.51158128144345905 1.6481127738952639 0.86664248603069383 1.6286621093750002 0.41892379644680228 1.0693359375000002 0.54439270847295596 0.44825707527455844 -3.4835662841796871 0.70437716445572607 4.2421875000000009 3.5426598534814651 3.7012257604091228 233.50000000000003 227.50000000000003 1.5000000000000002 3.3457031250000004 0.075876460218247857 22.867874145507816 26.206565856933597 -0.17729835697704338 -0.44620633588241487 1.6286621093750002 0.30807730392478799\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 2 11 4 -1 6 -4 -7 10 -10 -9 13 22 -2 15 -14 17 23 -18 20 -3 -19 -13 -5 26 -26 -25 28 29 -28\nright_child=1 19 5 16 -6 7 -8 8 9 -11 -12 12 14 -15 -16 -17 18 21 -20 -21 -22 -23 -24 24 25 -27 27 -29 -30 -31\nleaf_value=-0.021040110931287773 -0.010054479402067539 0.014741538507448736 -0.022996582933649298 0.0045524366324738337 0.0042737849091022771 0.0323597800010266 0.010171685572994865 -0.0024109800908994131 -0.028276981261600616 0.0075606846250690545 0.025834444267454532 0.0036534789762896016 -0.024365093353355575 0.01511381686104846 -0.036294862317739672 0.0055718837990174058 0.013567494973813017 0.0010278570926205359 -0.019761893164721333 0.021456011679770563 -0.022091208065759042 -0.027483584268736184 -0.020498504660435436 -0.02457382892782041 -0.036420867185424095 -0.0059446949869106586 0.027210753858421793 -0.004338085325175412 0.008024330543744131 -0.012929816227914311\nleaf_weight=12.162453271615506 8.0167637080667209 1.7431526966684026 4.1895141187123945 25.436997711620734 4.6712972958534911 5.3390362993814078 10.602449719794093 3.2132739946246129 2.0132159877102813 4.8680551112629473 5.4607537630945444 29.496852170268543 2.8012241594842635 13.468100377620429 5.0786203154566447 4.3431794237549184 9.4240380769570002 2.9666237190995153 0.86347074988225436 10.176264998328405 1.475420029601082 4.8190844629980347 2.0653736981912507 3.3285445856238303 4.8749443013211922 39.294874347373934 1.2075436163404436 83.42538025725753 41.152733331843137 12.766721837786461\nleaf_count=84843 287 40 55 9258 8587 23 109 23 36 41 33 464 70 205 155 176 332 10837 103 155 36 25990 37 36075 2616 11818 3089 35502 9803 5251\ninternal_value=-0.00124677 0.00380053 0.0020876 -0.0035951 -0.0141699 0.00871494 -8.38795e-05 0.0146633 0.00700161 -0.00518167 0.0162034 -0.00161272 -0.00524468 0.00569313 -0.0214375 -0.00771642 -0.00276155 -0.0033654 0.00926709 0.0156689 -0.00337714 -0.0177393 0.00149204 -0.00275525 -0.00377336 -0.0100401 -0.00176579 -0.0010457 0.0039253 -0.00787566\ninternal_weight=360.746 114.351 100.956 246.395 16.8338 35.6863 14.792 20.8943 15.5553 6.88127 8.67403 65.2701 43.7852 21.4849 12.223 7.1444 229.561 219.273 10.2875 13.3948 3.21857 7.78571 31.5622 211.488 186.051 44.1698 141.881 138.552 55.127 13.9743\ninternal_count=246049 1945 1714 244104 93430 320 164 156 133 77 56 1394 902 492 401 246 150674 150239 435 231 76 36827 501 113412 104154 14434 89720 53645 18143 8340\nis_linear=0\nshrinkage=0.03\n\n\nTree=203\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 7 4 4 3 4 22 3 18 0 0 2 4 1 0 4 22 14 14 13 1 2 3 1 14 2 3 19 10 9\nsplit_gain=4.11238 4.1961 5.96783 3.27947 3.10338 3.19335 3.02041 3.49267 2.95364 3.99556 4.3246 3.69025 3.61706 5.31298 7.15915 2.94023 2.82614 2.73516 2.65142 3.11966 2.53912 2.43393 2.29638 2.90351 2.55299 2.17695 2.05994 1.99494 1.88448 1.87546\nthreshold=2.6981674883069502 3.1132812500000004 6.6144968271255502 8.2534179687500018 0.51817819236196982 3.1247558593750004 1.0000000180025095e-35 0.30295287721642777 3.1567740231385479 0.89788080022475569 1.1495945197163422 18.821963310241703 3.8417438268661503 1.5661550609387962 0.7126865886964483 1.6572265625000002 1.0000000180025095e-35 0.36880189058923429 0.51224307552198323 -0.39822854168506638 0.39417715631504779 30.514518737792972 0.45898141303402712 4.0016611463566001 0.57616929401185935 -6.3279094696044913 0.084693112351350075 235.50000000000003 2.249904835165133 1.0000000180025095e-35\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 8 -3 -4 5 6 7 -2 9 11 29 25 13 -10 18 -9 17 -14 19 -15 -12 -5 23 26 -24 -1 -8 -21 -7 -11\nright_child=4 2 3 21 -6 28 22 15 12 10 20 -13 16 14 -16 -17 -18 -19 -20 27 -22 -23 24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.0066565082045090904 -0.0241508343377498 0.025677198358013961 -0.022616986951952807 -0.0026057997161301255 0.0075954578889837381 0.013184444820260514 0.018218694945075015 0.013730415094351713 -0.028026963484959044 -0.017159341452477328 0.015759788274191111 -0.01933654078252291 -0.012446230681378723 0.029027465304187785 -0.03377149446516111 -0.0079726892646733005 -0.023012207849362122 0.0099994780810171661 -0.014863255876052868 0.023717896032112856 -0.0020431492720011115 0.017745433542210646 -0.03005403042090243 0.0098295008772409827 -0.0029174410096834487 0.026375298151993635 -0.016374621268742617 -0.0077334589902749561 -0.020779923410265553 0.014618326424880812\nleaf_weight=1.7761709253049947 2.6713137334627417 9.464360521739307 4.1599459793330906 16.485189786077175 86.115611031715162 15.820200216097874 0.94242091604610401 17.24849649480166 16.151921332356778 20.895715043433942 7.098348307622472 1.7113943063532122 5.9908673678996882 4.3760927085950234 4.8379898061948543 7.3393819210723441 3.3292433006620277 15.482507290170586 3.7555064236976241 1.3715204672116721 64.738979890191331 6.6492078704613951 7.2136507513932866 8.1039870720705949 5.4115107012767112 8.6291866260776242 10.816996607038162 6.7381405255264326 0.72239484172314483 1.0642808776428854\nleaf_count=1799 468 1489 4138 4261 1176 447 10 616 152502 8732 3660 895 9131 258 1320 350 18265 23128 3961 256 4229 96 125 110 101 1101 495 2545 16 38\ninternal_value=0.000173787 -0.00264896 0.00641358 -0.000937531 0.00372619 -0.000683148 -0.00393059 0.00337613 -0.0046587 -0.00158222 -0.00371557 0.0139565 -0.00980293 -0.0155919 -0.00522575 0.00719372 -0.000845609 0.00346216 0.00474309 0.0113052 -9.17976e-05 0.00373639 -0.00997897 -0.00328915 -0.0194903 0.0206476 -0.0123917 -0.000394649 0.0106013 -0.0150496\ninternal_weight=367.113 204.707 36.7587 27.2943 162.406 76.2904 59.7478 27.2592 167.948 105.914 93.7973 12.1168 62.0338 37.2312 21.0792 24.5879 24.8026 21.4734 16.2413 12.4858 71.8373 23.1344 32.4886 19.8634 12.6252 10.4054 11.7594 8.10966 16.5426 21.96\ninternal_count=245718 241804 9984 8495 3914 2738 2275 1434 231820 20454 16659 3795 211366 160842 8340 966 50524 32259 7020 3059 7889 4357 841 615 226 2900 505 2801 463 8770\nis_linear=0\nshrinkage=0.03\n\n\nTree=204\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 4 3 17 4 4 2 17 4 17 14 2 17 4 12 2 18 3 4 14 21 2 3 18 3 18 7 18 6 3\nsplit_gain=4.60249 3.05429 3.04945 3.10659 2.43721 5.73428 4.72816 2.89872 3.37659 2.81208 2.2819 2.2126 2.14383 2.01379 2.13757 3.33517 2.83932 2.37733 3.27351 4.25372 2.41211 2.43356 3.20347 3.89724 2.17504 2.06788 2.38702 2.04194 1.99526 1.87766\nthreshold=-39.283819198608391 0.84667968750000011 0.50207005315878861 0.61464819824310435 1.0693359375000002 1.6286621093750002 -37.900318145751946 3.2718994203396332 6.9907226562500009 3.5112638802509566 0.043928974430191954 29.286581993103031 0.011353531392579346 1.6572265625000002 0.28848765710462548 16.375147819519047 3.7256208424938388 0.37248792739977959 4.8846709132194528 0.24319921965375105 1.5000000000000002 -14.86118173599243 0.53650715206675215 0.89587973461402759 0.3261289997719386 5.589117819455554 3.4169921875000004 4.2555875595453383 5.7285156250000009 0.53650715206675215\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 -1 -4 10 6 -6 13 -9 11 -3 12 -8 -7 17 16 28 18 19 -15 21 -19 23 -23 25 -20 -27 -21 -16 -13\nright_child=1 4 3 -5 5 7 9 8 -10 -11 -12 29 -14 14 15 -17 -18 20 24 27 -22 22 -24 -25 -26 26 -28 -29 -30 -31\nleaf_value=-0.022592059383630205 -0.014323379455936751 -0.015958385347418239 -0.02390360682773254 0.011216168735546232 0.027907507201467247 0.011935104051770326 0.014882691853304179 -0.028535038861261575 0.0035075174322257889 0.013922873013206324 0.0083674662458336803 -0.017419310999547372 -0.030734919432234793 -0.025274138239131841 -0.030537914608252115 0.0052535872882451878 0.0019384613875494359 0.019499665035279359 0.01331666929570339 0.0025247104698761114 0.0071248391810025506 0.015764938074367553 0.0045686618348174 -0.02472406284237913 -0.014830306153015059 -0.024027908445054359 0.010263899056387183 -0.022221343048223374 0.0045302804378705396 0.013065526910400334\nleaf_weight=11.141218860852861 9.4853756119673545 2.7580486799474775 2.2569723271008097 6.4595950190378062 1.3393323205418735 18.987973228628562 0.42335285110130527 4.8852826224622259 5.5649996026445478 1.5717362843363378 65.150518897341954 3.1412947170679208 11.884652544745448 8.6264823717383496 7.4142069179160917 12.577419270121025 5.3607876152182135 2.8003856613831912 18.389087251340705 19.058539251501319 110.85282557687424 1.9572167932055915 11.633662281907162 9.4230214219332975 3.2101341218804 2.4896731188441654 2.9803010892246675 2.5323306337832028 1.480350218596868 2.3023359245380561\nleaf_count=83411 34865 1586 246 346 474 1496 138 20082 1056 6730 13671 2449 20742 6277 278 353 870 380 2750 10961 4411 22 676 1538 521 5292 994 23460 50 113\ninternal_value=0.000820915 0.00163625 -0.012872 0.000598431 0.00212392 0.000839166 -0.0139745 0.00211319 -0.0126556 -0.0181078 0.00716224 -0.021535 -0.0281654 0.00280676 0.00197143 -0.00582247 -0.0152986 0.00307402 -0.00203113 -0.00868594 0.00521315 -0.00295566 -0.00638631 -0.0170497 0.00544242 0.00865688 -0.00664549 -0.00123285 -0.0248334 -0.00449929\ninternal_weight=368.139 348.281 19.8578 8.71657 338.796 270.887 20.6627 250.225 10.4503 19.3234 67.9086 17.7516 12.308 239.774 220.786 26.8328 14.2553 193.954 57.2865 30.2174 136.667 25.8143 23.0139 11.3802 27.0692 23.8591 5.46997 21.5909 8.89456 5.44363\ninternal_count=246238 162235 84003 592 127370 112113 30646 81467 21138 30172 15257 23442 20880 60329 58833 1551 1198 57282 50255 40698 7027 2616 2236 1560 9557 9036 6286 34421 328 2562\nis_linear=0\nshrinkage=0.03\n\n\nTree=205\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 3 7 18 12 17 3 18 1 13 1 2 13 1 5 2 2 0 2 0 1 0 18 18 13 1 13 21 2 5\nsplit_gain=4.12914 3.63553 2.79923 2.75287 2.63119 2.51337 2.60881 2.65001 2.99166 2.10037 2.31948 2.54515 1.88334 2.02995 2.43998 1.85184 2.93403 1.8953 2.27005 3.93995 2.32171 2.14608 2.08885 1.84673 1.82364 2.81742 1.79557 2.09956 1.82154 3.56469\nthreshold=-58.62553405761718 0.53650715206675215 2.5244140625000004 4.3757372435704527 8.8817841970012543e-16 2.9431225651686121 0.016076142967085191 4.1820210059610323 1.5661550609387962 0.14796868535883259 1.6575619461558879 -11.719409942626951 -0.42002175472665648 0.6973310998193073 1.4980468750000002 -21.560756683349606 25.282213211059574 1.3170161875856263 6.3334450721740732 1.0212340148387515 1.7622051334857016 0.64273501842566294 5.2496487501035594 2.3502401828962087 -0.74650584208841086 4.0016611463566001 1.0000000180025095e-35 1.5000000000000002 53.951957702636726 2.2500000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 3 5 -4 6 -2 12 -9 10 -5 -12 13 14 -8 -3 17 18 22 20 -20 -22 24 -6 -17 -26 28 -28 29 -14\nright_child=1 15 4 9 23 -7 7 8 -10 -11 11 -13 26 -15 -16 16 -18 -19 19 -21 21 -23 -24 -25 25 -27 27 -29 -30 -31\nleaf_value=-0.025370066719156606 0.021947082873792614 0.010929588025054712 0.0078157013080456163 -0.02471976227497728 -0.022685452017339142 -0.022123634381378564 0.02497311563495801 -0.017398898834584144 0.025873550519197267 0.012238061355808667 -0.023345029310492302 0.0053926517996576891 -0.0056800608239801825 -0.00056793922376207956 -0.016339733226127695 0.0070906978453330691 0.010749281956284553 0.0041818457090749481 -0.01328161078348539 -0.01776011448306326 -0.016216195425786867 0.023056647787616388 0.013991080299228856 0.00071710545102690398 -0.027461296680452971 -0.00067988593574527583 0.016091431450318359 -0.0066582101934224173 0.005192194475205694 -0.024384301751205657\nleaf_weight=4.7694942990992777 3.1528545588844272 28.744508322641195 39.65537492055023 11.232441010677354 3.7335221700777756 4.5388743092207724 4.9428419432902038 1.144890170680809 4.7029117019374098 1.9534632791546744 3.2649411186794168 7.1522589615597081 67.372929910821085 38.048914301334207 0.89592216127994095 2.2880171762954005 25.424082206454841 31.863773176504765 1.8282853398559371 4.785655305357067 0.72752767773636051 8.2335981865617196 1.6682745696161863 8.2929606263703572 10.264488724587865 5.2277963396627447 6.3031624525610805 6.4773244120297022 9.4184209624108899 9.8539744842137207\nleaf_count=51695 1527 760 12190 100811 176 18130 6878 8087 621 58 2214 1656 24509 4452 2956 231 989 299 747 161 103 155 201 1513 832 60 135 166 1560 2298\ninternal_value=-0.000195241 0.000215385 -0.0019813 -0.00374167 0.00411968 -0.00226228 -0.00152998 -0.00216907 0.016098 -0.0130813 -0.0158054 -0.0048701 -0.00302083 0.00212207 0.0171716 0.00440949 0.00228385 -0.00104482 -0.00578402 0.00268861 0.0124958 0.0185603 -0.0123688 -0.00773179 -0.0154554 -0.0190132 -0.00528949 0.00491512 -0.00683362 -0.00834788\ninternal_weight=357.963 353.194 232.138 180.456 51.6819 156.853 152.314 149.161 5.8478 23.6031 21.6496 10.4172 143.313 43.8877 5.83876 121.056 92.3115 66.8874 35.0236 15.5751 10.7894 8.96113 19.4486 12.0265 17.7803 15.4923 99.4258 12.7805 86.6453 77.2269\ninternal_count=246170 194475 189937 176058 13879 71319 53189 51662 8708 104739 104681 3870 42954 14286 9834 4538 3778 2789 2490 1166 1005 258 1324 1689 1123 892 28668 301 28367 26807\nis_linear=0\nshrinkage=0.03\n\n\nTree=206\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 3 2 0 3 21 0 19 13 13 12 19 11 4 9 17 14 0 17 5 7 4 2 2 4 1 1 19 2 17\nsplit_gain=4.1525 3.70492 2.39501 2.0877 2.53429 2.15386 2.48383 2.08956 2.19828 1.86988 2.49725 3.79287 2.69896 2.53598 2.41515 2.47877 2.45921 2.42784 2.27033 2.13954 2.07428 7.29248 2.6649 5.8005 4.52377 2.89141 3.33549 2.84085 2.61122 2.17362\nthreshold=2.9477339485081675 0.36709515926886871 -58.62553405761718 1.2661233611507157 0.50207005315878861 5.5000000000000009 1.0659450432993143 253.50000000000003 -0.17886242654860576 0.14796868535883259 0.24649637830053225 207.00000000000003 0.91125284794847328 1.9406738281250002 1.0000000180025095e-35 1.490403556610195 0.91125284794847328 -0.43599614227863787 5.817026225953116 4.2265625000000009 3.1132812500000004 6.6144968271255502 8.8705892562866229 3.4383306503295903 2.1750488281250004 1.6109634896136267 2.5144622974930591 241.50000000000003 11.076014518737795 0.13249526821743748\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 -1 5 -5 6 -3 8 -8 10 14 -12 13 -13 16 19 18 -17 20 -16 22 -22 23 24 -4 -26 -27 29 -24 -25\nright_child=1 3 9 4 -6 -7 7 -9 -10 -11 11 12 -14 -15 15 17 -18 -19 -20 -21 21 -23 28 27 25 26 -28 -29 -30 -31\nleaf_value=-0.023798221695664437 -0.010404831639529356 0.0082378117886176706 0.0022275487265032519 0.00050738234594570775 0.012962713777047124 -0.010718391735039705 -0.031323920358528871 0.0067672482995619551 0.0003044228039973117 0.0097579154666416913 0.014000911218472394 -0.00095393182804932842 0.0082306756042444303 -0.032694658115400989 0.023799650940651378 0.013090884304520059 -0.018980064838517625 -0.018775692105234814 -0.023876113046318648 -0.00083042052099119973 0.028581736886183776 -0.01122514962336537 0.017852425225698864 0.024470542925674847 -0.029247478393773674 0.0026278717322232433 -0.026701195286710395 -0.038587645941087587 -8.6964784874124138e-05 -0.019550136753429051\nleaf_weight=3.495651678128751 12.790569133976531 35.676601807143015 46.803404554579402 20.701144590377226 50.455454549897695 9.4385685363376961 3.3104523308575153 5.1220178743824363 3.6193487567361426 11.237530151440295 3.0297586628075832 3.05530395294545 2.1509610149078062 8.5018038026623781 8.4297205087814309 2.8224922156505272 6.7697211545423661 4.0758559561095344 3.0501485152524301 4.6859326036210396 7.3857135806297247 7.3842081530087853 7.2690443130613849 0.88391672808938837 6.5960094402310032 11.422836389259826 3.9312439209970753 5.6831701282967515 62.391832591226375 1.1774142799467879\nleaf_count=51377 838 950 54565 253 387 292 46 45 63 274 87 644 34 1728 1020 990 30865 825 25161 836 484 1564 2682 228 31376 1368 204 4868 29640 2129\ninternal_value=0.000443872 0.00445128 -0.0021093 0.00600106 0.00939641 0.00168593 0.00430858 -0.00706299 -0.0168496 -0.00165596 -0.00231793 -0.0130891 -0.0196214 -0.0250477 -0.00130953 0.00830111 -0.0024778 -0.00573068 -0.0016856 0.0155644 -0.00112016 0.0092305 -0.00222175 -0.00605255 -0.00307804 -0.0139951 -0.00605503 -0.0290875 0.00201115 0.00115365\ninternal_weight=363.348 141.114 222.234 128.324 71.1566 57.167 47.7284 12.0518 6.9298 218.738 207.5 16.7378 13.7081 11.5571 190.763 20.014 170.749 6.89835 163.979 13.1157 160.929 14.7699 146.159 76.498 68.7535 21.9501 15.3541 7.7445 69.6609 2.06133\ninternal_count=245823 2874 242949 2036 640 1396 1104 154 109 191572 191298 2493 2406 2372 188805 3671 185134 1815 154269 1856 129108 2048 127060 94738 87513 32948 1572 7225 32322 2357\nis_linear=0\nshrinkage=0.03\n\n\nTree=207\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 13 18 18 1 13 1 13 7 7 14 2 3 14 3 2 2 7 19 6 13 19 2 18 18 18 19 2 17\nsplit_gain=4.07044 6.74259 3.29328 3.67826 3.14696 2.83233 2.12275 2.01985 1.98905 2.52299 2.28408 1.94135 2.57416 2.18029 2.11199 3.96627 2.37179 2.03557 2.21837 2.83263 2.01778 1.83147 2.56787 2.1418 1.78899 2.57128 2.04396 2.00501 1.66662 2.33964\nthreshold=21.425971984863285 0.15052965826911111 -0.76787006514439582 1.7005986908310777 2.249904835165133 2.9477339485081675 -0.66255419986965325 2.054932859494814 -0.05385377781229627 3.2763671875000004 3.2382812500000004 0.6618576200563534 -11.357147216796873 0.65908129556435835 0.58665773875949134 0.50207005315878861 -33.394275665283196 -15.657813072204588 1.7006835937500002 235.50000000000003 4.3886718750000009 -0.2925558868519148 253.50000000000003 -1.3165187835693357 3.6242520361853052 3.1567740231385479 2.9174053685313028 213.50000000000003 15.769127845764162 0.59049165877606857\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 -3 -5 6 -4 -6 11 10 -10 13 -13 14 17 -16 -17 -8 19 -19 -20 22 -21 -24 25 26 -2 -27 -7 -30\nright_child=24 2 5 4 7 28 8 -9 9 -11 -12 12 -14 -15 15 16 -18 18 20 21 -22 -23 23 -25 -26 27 -28 -29 29 -31\nleaf_value=-0.022756820296945889 0.0044859062916181051 -0.02038823153342826 0.0047677859845924511 0.032437748699280698 0.010779178698003161 0.0019493707982451694 -0.021163279360519913 -0.022456169754866247 -0.00020422616609065371 -0.019691030190671286 0.027810485948108633 -0.004764016229538314 -0.029129908610416751 0.013189863424245534 0.014670679458686046 0.015143850097516414 -0.034541149196961093 0.022592932772069772 -0.027612079878477323 0.012888710674254848 0.0080948839049922866 -0.015030790577494346 0.0060209817548527991 -0.027632313860726988 -0.0069353776912544716 -0.013860425705423418 -0.023249829011208124 0.023168885429379192 0.0041054890224019799 -0.026919988371093235\nleaf_weight=13.589200285925925 74.787440787083597 1.8943404930105305 14.404781538401492 5.6170242589068931 5.8802857263999613 100.09423611460079 12.81895605914996 1.1768827066698575 25.596164379333011 4.4133171301609755 1.9160820068082092 7.2963215516870177 8.7102651779150602 3.5528509411087716 7.3274136498916942 0.54733157901500851 2.0956788761905045 2.5611788416799888 5.6966158599770997 6.9692581337653845 1.321129575371742 10.054248174070375 2.6092471025513078 2.6953811908297212 10.595832937594649 0.83925502461170154 1.4874700849686622 9.340719754077325 3.8518833664711556 3.4525727349391673\nleaf_count=139818 5871 72 4432 88 4211 2100 18509 221 9827 226 16 11960 7511 66 5958 45 243 1125 954 1316 15 5246 873 710 20040 680 1347 2626 39 88\ninternal_value=-0.000963407 -0.00294406 -0.00174516 0.0116203 0.0175455 -0.00264624 -0.00578253 0.0031375 -0.00724494 -0.000939324 0.00265381 -0.00990746 -0.0189564 -0.0071429 -0.00867674 0.00352521 -0.0229194 -0.0114144 -0.00697364 -0.00244749 -0.0207188 -0.00616515 0.00168246 -0.0127495 0.00424077 0.00567415 0.00365115 0.0191494 0.00089601 -0.012035\ninternal_weight=353.193 256.143 242.553 14.5685 12.6742 227.985 120.586 7.05717 106.181 31.9256 27.5122 74.2559 16.0066 58.2493 54.6964 9.97042 2.64301 44.726 31.9071 24.8893 7.01775 22.3281 12.2739 5.30463 97.0507 86.4549 76.2749 10.18 107.399 7.30446\ninternal_count=246233 215669 75851 4592 4520 71259 69032 4432 64600 10069 9843 54531 19471 35060 34994 6246 288 28748 10239 9270 969 8145 2899 1583 30564 10524 7218 3306 2227 127\nis_linear=0\nshrinkage=0.03\n\n\nTree=208\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 2 4 0 19 2 2 4 13 21 1 10 7 1 9 1 2 1 4 5 10 10 2 13 13 18 2 21 0 19\nsplit_gain=4.87402 2.95132 3.31276 2.1683 4.02896 1.89014 2.82844 2.65586 6.0451 2.05049 1.86419 1.93354 2.16219 2.84975 2.6385 2.12231 2.26021 3.41075 2.01423 2.78989 1.95944 1.78235 1.74907 3.768 3.60479 3.20414 2.76306 2.85549 2.40907 2.26647\nthreshold=-51.587459564208977 -50.338066101074212 0.84667968750000011 1.3771184689138856 254.50000000000003 -1.7340650558471677 -29.210144996643063 2.4904785156250004 -0.55639320842146012 4.5000000000000009 -0.17729835697704338 2.249904835165133 2.6015625000000004 0.45969269984116679 1.0000000180025095e-35 0.71428742788045685 12.256900787353517 1.8912989126705502 3.9514160156250004 4.2558593750000009 0.89587973461402759 1.7005986908310777 0.82504272460937511 -0.27202883008418016 -0.055817889458273311 1.4978661367769956 -14.08206272125244 2.5000000000000004 1.5447487307370646 248.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 -2 -3 10 22 6 -6 8 -8 -9 21 12 14 -14 18 -15 -17 -18 -12 20 -20 -4 24 -24 25 26 -5 -28 -25 -27\nright_child=1 2 3 4 5 -7 7 9 -10 -11 11 -13 13 15 -16 16 17 -19 19 -21 -22 -23 23 28 -26 29 27 -29 -30 -31\nleaf_value=-0.021618395825098381 0.030253861316345037 -0.015066603480790305 -0.021823944909621799 -0.01475150686644181 0.022616183416387468 0.018463071788034743 0.011426232075406932 -0.0073966906293040551 -0.032149461077267176 0.025405471699723001 -0.00077400823196518162 -0.024358836230664458 0.020624757974360593 -0.02151925451025366 0.018552815952499347 0.01010246943249129 -0.024627908243633267 0.0086227282446839047 -0.021956475643239021 0.016300722227100769 0.0077964063809294647 0.0058440742495331184 -0.014520989993208248 -0.013219286640012887 -0.013215628562261232 0.031393817658891404 -0.017354618263208523 0.021913662936547659 0.0054035023650969155 -0.0076345270017459327\nleaf_weight=7.7578365089182162 2.0861783390532755 10.483800273242876 5.7730584948953929 4.4117413267995289 5.0732076127351347 20.014829787323833 5.0072030789306154 2.1949126787221749 4.6819370935409079 3.4628676874563098 137.85260155795709 1.8777772994872055 8.856740792393337 2.1409009658063978 4.8805166743122745 19.446406853682678 4.183293409239921 5.1798736425116658 10.709872460483437 1.8426208395303536 1.8720346389054601 2.5206711545233702 15.27033411404045 6.5123193961871939 5.3040858262374959 7.3313614284706983 1.3967111297533898 5.6782184392868658 36.827219055805472 1.1870438472469684\nleaf_count=61551 1962 37960 61931 86 47 201 48 28 100 35 65594 366 2216 720 713 1373 346 98 6622 155 511 267 750 605 206 184 56 69 1220 51\ninternal_value=0.000485189 0.00104479 0.000778907 0.00132385 0.00443833 0.0120929 0.00527871 -0.00148542 -0.010668 0.0134804 -0.000558265 5.52661e-05 0.00040989 0.00656492 -0.0011808 0.00202197 0.00443458 -0.00717602 -0.00193018 -0.012213 -0.0172815 -0.0136909 0.000651129 -0.00220046 0.00708736 0.0128433 0.00199564 0.0129723 0.00237016 0.0257245\ninternal_weight=351.818 344.06 341.974 331.49 124.354 40.435 20.4201 15.3469 9.68914 5.65778 207.136 198.843 196.965 39.8072 157.158 30.9505 28.8096 9.36317 152.277 14.4245 12.5819 8.29373 83.919 58.6099 25.3092 20.0051 11.4867 7.07493 43.3395 8.51841\ninternal_count=246071 184520 182558 144598 3686 459 258 211 148 63 140912 78714 78348 4753 73595 2537 1817 444 72882 7288 7133 62198 3227 2575 652 446 211 125 1825 235\nis_linear=0\nshrinkage=0.03\n\n\nTree=209\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 13 2 2 13 18 2 1 18 1 4 17 2 1 18 1 4 7 18 19 3 18 10 3 18 1 19 18 5 19\nsplit_gain=3.74179 2.65623 4.2221 4.13965 4.00036 3.85821 2.90535 2.85906 2.59041 2.47728 2.8055 2.47883 2.29929 2.46948 2.18872 4.59495 4.18615 4.68782 3.10397 2.34478 2.3561 2.23401 2.0272 1.9883 1.98342 1.92461 1.84468 2.00508 1.78854 1.77541\nthreshold=3.8863394171308765 -0.48625410865680269 30.514518737792972 -10.210010528564451 -0.44620633588241487 1.2424533248940002 -21.99824237823486 1.8912989126705502 0.89587973461402759 2.6981674883069502 3.9857867956161503 0.13727533581160231 -1.3165187835693357 0.6180242047922454 5.1503930356556173 1.4475910192202701 3.2580566406250004 2.6738281250000004 5.2600927804663344 230.50000000000003 0.16358559074890858 5.4868673298236716 2.0126758453675748 0.2708708835714248 3.0677824455408698 0.64953354656755413 242.50000000000003 6.0438190738591979 6.4160156250000009 248.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 4 6 5 -3 7 -1 -8 10 29 -11 25 -14 15 16 -6 -18 -16 -17 -21 -20 -10 -22 -9 -5 27 -23 -19 -4\nright_child=-2 2 9 12 14 -7 8 24 22 11 -12 -13 13 -15 18 19 17 28 21 20 23 26 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.01841085584774638 -0.017323007021427653 0.0050425040657606389 -0.019042573099582086 -0.0095660277060228231 -0.029470223611588742 -0.034332778450857061 0.016063726181301766 0.012050906743320128 -0.027154264913785881 -0.016035336407972226 0.0078801919497465955 0.015469251586862267 0.02087991450559475 0.0021351032395638173 0.023947733368679679 -0.021005534989444203 0.01355556978562574 -0.025743827545829736 -0.025124212954164414 -0.02508075262459631 0.012354566676221951 0.027061774504400827 0.0097486217665528254 -0.0012311701221317519 -0.010421026159438213 0.022219992885962374 -0.0077379914193533668 -0.0046890862273316521 0.0048212898979710416 0.0023117055641348108\nleaf_weight=3.9643118505296142 10.537196373954655 2.9454383014090109 7.268237989296269 1.3801415479902051 10.030863082487715 6.0453641887077083 1.1511368204664894 14.508243489282906 7.8286154495754072 1.6806872102315562 16.537097584122421 22.308206874367897 6.6844546508950762 37.492984305196842 5.4086333853474597 4.383848297482472 6.3650250678360853 6.1261096591706501 1.8754486938992148 2.4987826389569809 9.5545092966312932 3.4019275777800413 1.057998232419777 118.09856888566446 3.7409062294809692 13.580164773352406 5.4276271258312363 2.5857402338042448 1.9025803674303459 5.7424833400200441\nleaf_count=23813 63455 40 8538 3813 62778 7011 18 151 6078 45 627 427 8703 4414 4569 532 4151 890 6935 550 1669 1794 125 5784 770 1265 21655 4336 81 736\ninternal_value=-0.000527849 5.6369e-05 -0.00160068 0.00438158 -0.00373018 -0.0222196 -0.00418122 0.00198532 -0.0167712 0.00575591 -0.000120015 0.0127086 0.00904324 0.00537076 -0.00267536 -0.00382458 -0.0155245 -0.00452276 0.0067922 -0.00160162 -0.00079287 -0.00137643 -0.0222191 -0.000126186 0.0071423 0.018872 0.00423455 0.0146415 -0.018769 -0.0101252\ninternal_weight=342.113 331.576 240.187 91.389 186.65 8.9908 32.2512 22.2135 10.0378 53.5367 29.5478 23.9889 59.1377 44.1774 177.66 158.96 24.4246 14.3937 18.6994 134.536 130.152 13.2907 8.88661 127.653 18.2491 14.9603 11.4153 5.98767 8.02869 13.0107\ninternal_count=245753 182298 133148 49150 122775 7051 30955 24734 6221 10373 9901 472 18195 13117 115724 76435 67900 5122 39289 8535 8003 34720 6203 7453 921 5078 27785 6130 971 9274\nis_linear=0\nshrinkage=0.03\n\n\nTree=210\nnum_leaves=31\nnum_cat=0\nsplit_feature=4 2 17 2 4 2 17 17 0 10 9 17 18 18 14 12 7 4 4 19 0 17 11 18 8 17 21 4 0 2\nsplit_gain=4.11232 3.44276 2.60614 2.58772 3.0131 2.10034 3.03184 2.05577 1.72964 1.64932 1.92465 2.46223 2.0494 3.902 2.13895 2.50088 2.11878 2.66159 3.73351 2.29159 3.69244 1.75753 1.49357 2.3613 2.59021 1.51117 1.4593 1.43391 2.08439 1.57782\nthreshold=0.84667968750000011 -39.283819198608391 0.69259390652607344 -49.116760253906243 2.1750488281250004 -10.988907814025877 0.96280603246018803 3.9653415636860969 1.5447487307370646 2.524928003624769 1.0000000180025095e-35 2.1677456597340812 5.2600927804663344 5.1503930356556173 0.10679678813632677 0.66389400333341531 3.1132812500000004 6.6144968271255502 8.2534179687500018 233.50000000000003 1.4460303622695712 0.92169800002628566 0.19364712777950585 0.89587973461402759 1.0000000180025095e-35 0.45020544458187189 4.5000000000000009 7.0991159677505502 0.38644940889434637 28.11904335021973\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 2 3 4 26 6 -1 9 -5 10 12 -12 13 16 21 -16 19 -18 -19 20 -3 -14 23 -22 -25 -24 -2 28 -21 -30\nright_child=1 7 -4 8 -6 -7 -8 -9 -10 -11 11 -13 14 -15 15 -17 17 18 -20 27 22 -23 25 24 -26 -27 -28 -29 29 -31\nleaf_value=-0.022164494707879023 0.028960006799782077 -0.023801030289606732 -0.024692302596896844 -0.02297712367503674 -0.01951697625708889 -0.026588691966928285 0.016368043473422272 -0.012311481537329468 0.012518832117071709 -0.016122520851697787 0.013642283952852913 -0.015275091043705055 0.019155036615227199 0.02462750070542704 -0.029272921148968654 0.0074775480698611192 0.023675450128582402 -0.020107703292729033 0.012464068783621086 -0.014134408314391793 0.025776128637358165 -0.016801593860557369 0.0085233419881405488 -0.01504492145506795 0.014723571694075002 -0.031933702207615169 -0.0028996531890490502 0.020868614446498283 0.00097384282549233968 0.0074929904997764144\nleaf_weight=2.4116080039839289 3.5386872933407858 7.4107978395636573 6.6139221373547334 4.4790471609785545 1.4853798298692025 6.3159464214734253 2.9908651324385573 7.822913010585693 0.90078741591423739 3.2957080025898895 22.144578640340118 2.1577409250912742 2.4242083001167871 5.9540134394856059 4.9037945261945879 1.8404068352538159 7.7385785450442226 3.7256296706073035 8.6323886549434956 6.3055787387428373 2.7625544450565931 0.98631208660162872 0.80616229859879318 5.1050778458866244 3.6408946544106584 1.1296126956585792 1.5734220997662758 2.5600515956102745 184.87889829157416 39.898318426498918\nleaf_count=9340 3283 12389 51785 6385 12824 22118 11050 28442 9 377 1914 627 555 4615 17280 73 258 221 193 18472 65 1257 24 546 179 25 1958 57 35445 4473\ninternal_value=0.000652353 0.0012669 -0.0109555 -0.00205172 0.00993371 -0.0161038 -0.0016078 0.00199502 -0.0160031 0.00238409 0.00262812 0.0105726 0.00193001 0.00242358 -0.0107008 -0.0195736 0.00185452 0.0109936 0.00187432 0.00109264 -0.00820556 0.00730472 0.00144364 0.00587313 -0.00241329 -0.017921 0.0202841 0.00195404 0.00165548 0.00215906\ninternal_weight=356.434 344.715 18.5912 11.9773 6.59749 11.7184 5.40247 326.124 5.37983 318.301 315.006 24.3023 290.703 280.549 10.1547 6.7442 274.595 20.0966 12.358 254.498 20.8551 3.41052 13.4443 11.5085 8.74597 1.93577 5.11211 233.643 231.083 224.777\ninternal_count=246239 203731 76244 24459 18065 42508 20390 127487 6394 99045 98668 2541 96127 76962 19165 17353 72347 672 414 71675 13228 1812 839 790 725 49 5241 58447 58390 39918\nis_linear=0\nshrinkage=0.03\n\n\nTree=211\nnum_leaves=31\nnum_cat=0\nsplit_feature=18 1 18 18 1 3 4 19 4 4 12 18 14 4 1 13 0 13 3 3 4 9 1 1 4 19 4 5 6 19\nsplit_gain=2.86083 4.38115 3.74992 2.34206 2.23451 2.25117 2.87202 3.48476 3.58814 3.44102 3.16346 3.07665 2.89791 2.68301 2.55898 2.55439 4.0826 5.87993 2.60326 2.38707 2.10301 2.08353 3.58289 2.33124 2.02341 3.1916 2.91657 2.65146 2.60614 2.49385\nthreshold=3.7494348669884658 -0.041910702480567281 3.9796379800581989 3.6762205501217919 -0.12981918606627474 0.26802264518526847 1.6286621093750002 263.50000000000006 1.0693359375000002 1.6572265625000002 0.25233898660012938 0.89587973461402759 0.67646516896599407 2.9858398437500004 3.3033179733375269 -0.46714757665816348 1.2225960941547209 -0.07178845268160132 0.40160889672495398 0.53650715206675215 0.84667968750000011 1.0000000180025095e-35 1.4475910192202701 0.98142480879535021 1.0693359375000002 246.50000000000003 2.7826617956161503 4.3496093750000009 4.1835937500000009 266.50000000000006\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 -2 -3 4 -1 -6 7 8 11 -8 21 12 -7 -12 -9 -13 18 -18 -17 -19 -4 22 23 -11 -22 26 -26 -28 -29 -27\nright_child=1 2 20 -5 5 6 9 14 -10 10 13 15 -14 -15 -16 16 17 19 -20 -21 24 -23 -24 -25 25 29 27 28 -30 -31\nleaf_value=0.01530153868944919 -0.024749389906801228 -0.020691518221715437 -0.022015698918197352 0.019408496502541162 -0.0072540015731986564 0.026326064066318721 0.015895447961288087 -0.031611375006185133 -0.016246352490784072 -0.00067421388416768189 -0.024536030226637327 0.0095986383941467114 -0.0067299174525909744 -0.00038823911422747922 0.0096237317610722538 -0.021444763004129144 -0.034931911892318764 -0.011855784305239564 0.010013609516371822 0.019437766041384873 0.013533963524700134 0.017694085512445955 0.0041236460735595802 -0.031753947024453992 -0.013450022027208419 -0.015754760237972718 0.014011340102257202 -0.021023502700025981 0.010590347709710922 0.013371692638101037\nleaf_weight=8.7070768466733153 7.8703371745964423 7.775901108623656 2.6071915300387891 5.498682969409173 27.946487352896384 6.7327601545548514 17.672398772218003 5.0163259354321772 10.602086006646458 4.535339476723947 5.8076256887561639 11.978686180360446 2.9541834740593913 10.655748166536794 1.2421300116693599 1.9039533010545806 7.0081542339758016 3.8801197395114269 12.459935452345233 3.0046434632677119 9.5138471507532341 8.055763722270056 113.07360780076488 3.2483772040468466 5.7709527227727895 14.602146022531999 20.144787081422013 3.8341246209129158 3.6887228288942424 2.3891916640204718\nleaf_count=12342 121580 6170 12622 3019 15216 100 241 793 1896 1366 180 1102 84 283 16 1110 340 145 621 28 6165 124 2096 773 15405 28260 11984 1335 154 818\ninternal_value=0.00015267 -0.00486772 -0.00232695 0.00160861 0.00117014 0.000635531 0.00162213 -0.00358213 -0.00120603 0.00376258 0.00218915 0.00224441 0.0165587 -0.00982384 -0.023229 -0.00151983 -0.00640017 -0.0174416 0.00471937 0.00253447 0.000245671 0.00378713 0.00272256 -0.0157832 0.00155927 -0.000919047 0.00444915 0.00852152 -0.00609832 -0.0111436\ninternal_weight=350.181 78.1972 70.3269 271.984 266.485 257.778 229.832 66.783 60.5245 163.049 145.376 49.9224 9.68694 16.4634 6.25846 40.2355 28.2568 13.8929 14.3639 6.88476 62.551 128.913 120.857 7.78372 59.9438 50.4299 33.4386 27.6676 7.52285 16.9913\ninternal_count=246368 204493 82913 41875 38856 26514 11298 6235 5426 5063 4822 3530 184 463 809 3346 2244 513 1731 173 76743 4359 4235 2139 64121 57956 28878 13473 1489 29078\nis_linear=0\nshrinkage=0.03\n\n\nTree=212\nnum_leaves=31\nnum_cat=0\nsplit_feature=7 3 18 18 14 2 1 2 3 2 6 19 22 5 2 10 12 1 18 18 2 21 5 5 14 18 14 14 14 18\nsplit_gain=3.15342 2.75246 2.58696 2.97727 3.03145 2.20258 2.43096 2.25337 3.70008 3.61786 2.73468 2.99966 2.53116 5.01993 3.22007 2.54798 2.44698 2.40083 3.20732 2.18224 2.03553 2.69207 2.02986 2.42746 3.32511 2.08848 4.11269 2.00143 2.80458 2.17642\nthreshold=0.89599609375000011 0.58795853567030554 2.249904835165133 3.5694334999727624 0.67646516896599407 50.658309936523445 -0.29568658027461547 -1.3165187835693357 0.56033297298897644 -5.1300392150878897 0.89599609375000011 253.50000000000003 1.0000000180025095e-35 3.5107421875000004 0.82504272460937511 1.0000000180025095e-35 0.061140206337120091 1.0240454605967615 6.2567088148180039 2.9700856263602158 -25.182852745056149 2.5000000000000004 0.89599609375000011 1.7006835937500002 0.67646516896599407 1.8688348091416842 0.75822041467236145 0.032916404154818342 0.064908752851949539 2.4414009612931857\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=22 5 20 4 -4 6 -2 8 9 -8 11 -11 13 15 -14 16 -9 18 -15 -6 -3 -22 27 24 -24 -26 -27 -1 29 -29\nright_child=1 2 3 -5 19 -7 7 12 -10 10 -12 -13 14 17 -16 -17 -18 -19 -20 -21 21 -23 23 -25 25 26 -28 28 -30 -31\nleaf_value=-0.023528643412394034 -0.020741566123395304 -0.009003819396541211 -0.027086567166410169 0.011865042426723116 0.013875643260391941 0.010065665805109375 0.00063516599610212519 -0.027803243344613374 -0.02401660737522724 0.026073275717481102 -0.013531228056229116 -0.01152916182503397 -0.026981016628659579 -0.02595042192639262 0.0057831091491534325 0.010663040925799713 0.0025029676386340476 0.0040924351242863644 0.022152517976489108 -0.017808964326764879 -0.0035071268781028575 0.020700923022017686 -0.019429195499100226 -0.00017339959544158818 -0.016354592546583641 0.025433018832527857 -0.018316887876380663 0.030792633387618965 -0.002022651198892245 -0.013366113515547694\nleaf_weight=2.5761013924202407 4.8179203648303419 2.6609627641737488 4.5119601952028479 8.3814571767397847 4.4407699006260373 14.751840567814041 53.174468612151259 14.995513010949031 3.3769680250552474 12.139696371745684 1.6846935496296271 1.6486501787176155 2.0105806254691769 4.6312303737340725 25.699755267271939 1.6381206271762483 2.6361441274775634 22.151628877720213 0.77780344702841819 2.0712322373001371 4.8307415311574031 19.820449284277856 24.668853239780674 17.673619498893231 3.0208712803832878 4.3539910352876632 2.1523793019669029 3.1210926314743115 70.959355346857819 0.6988671564076866\nleaf_count=4455 54857 25 68 691 63 2332 37953 8012 151 1297 226 229 160 1178 1210 99 161 668 116 91 78 132 29915 22955 490 2860 17008 90 57479 1200\ninternal_value=-0.00104295 0.00120168 0.00755279 -0.000842422 -0.0106871 -0.000618776 -0.00171917 -0.000957536 0.00280756 0.00445301 0.0167383 0.0211016 -0.00458454 -0.00877055 0.00254888 -0.0201036 -0.023381 -0.000420279 -0.0166562 0.00276873 0.0133367 0.0160052 -0.00472101 -0.00925476 -0.01381 0.00120323 0.010448 -0.00160164 -0.000545751 0.0216168\ninternal_weight=342.078 212.853 46.7176 19.4054 11.024 166.135 151.383 146.565 72.0245 68.6475 15.473 13.7883 74.5408 46.8304 27.7103 19.2698 17.6317 27.5607 5.40903 6.512 27.3122 24.6512 129.225 51.8697 34.1961 9.52724 6.50637 77.3554 74.7793 3.81996\ninternal_count=246249 109797 1148 913 222 108649 106317 51460 39856 39705 1752 1526 11604 10234 1370 8272 8173 1962 1294 154 235 210 136452 73228 50273 20358 19868 63224 58769 1290\nis_linear=0\nshrinkage=0.03\n\n\nTree=213\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 7 0 18 10 2 19 19 2 18 2 2 5 2 19 19 18 18 17 2 7 17 0 18 2 18 2 6 19 17\nsplit_gain=3.37534 2.53756 3.23035 5.61976 2.38032 1.89203 1.87297 2.10715 5.6739 2.45171 3.03948 2.79065 2.67223 3.22653 2.32595 2.21745 2.14972 1.99208 3.83032 2.42069 1.98772 1.91263 1.83811 1.75577 1.86627 2.11474 1.73605 1.73516 1.52768 1.52391\nthreshold=4.4301796584464421 0.89599609375000011 0.52258841453381233 4.1820210059610323 3.2769667020129059 -34.692844390869134 264.50000000000006 259.50000000000006 -1.7340650558471677 3.1984648276080736 -18.054428100585934 31.708583831787113 1.7006835937500002 13.964207649230959 262.50000000000006 240.50000000000003 2.0126758453675748 1.7005986908310777 0.49969040966739248 25.282213211059574 1.0000000180025095e-35 0.48975197669932524 0.17863154480046692 2.8617925509761908 -46.244852066040032 1.7005986908310777 -26.006598472595211 6.5410156250000009 202.50000000000003 0.92169800002628566\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 22 6 5 23 7 9 14 10 26 -11 15 -14 -9 20 17 18 -17 -20 -12 -19 -1 24 -3 -26 -4 -2 -15 -23\nright_child=27 4 3 -5 -6 -7 -8 8 -10 11 12 -13 13 28 -16 16 -18 21 19 -21 -22 29 -24 -25 25 -27 -28 -29 -30 -31\nleaf_value=0.0011870090832436936 -0.024106980020999671 0.019422757256561937 5.5465410496525787e-05 0.026819681754784715 -0.026048693933037995 0.0038542044819478718 -0.01578695023651638 0.0048318120250396834 0.022653346899784148 -0.024218211279395808 -0.00050461760141046435 0.01288849543276826 -0.019100743031998855 -0.016077648169158399 -0.021932139733467335 0.0098551558868821137 -0.017291670141103152 -0.016795376947548096 -0.022917547526864417 0.0022532829311895704 -0.034514163006540262 0.029524722992628788 -0.021639836044342848 -0.019218116419245059 0.014184759168332844 -0.022856025694976041 0.020520403156244465 0.011911822932622957 0.02002971630262532 -0.0034700045775525193\nleaf_weight=4.2672819743656865 6.1578937419935746 2.4209507463474438 7.1998505526144072 5.2860497171218404 1.5525571550915618 198.46517776517499 7.6203139989040656 5.5581316974876263 9.7000591406631411 7.6542299923789852 2.3384703791894017 1.6303456070413629 2.0982154959519894 0.44710427583413359 4.5519346727696757 9.0203265763593716 8.3428474346051171 0.75117349577703763 8.8086270226685865 5.0391347852419122 3.7480369622367107 3.4120428117803385 10.361213248606871 5.3999171870311784 1.4236334372981203 2.5540194930563302 6.7545835564915233 0.81677562593313258 7.9443028891882932 1.4931646701998031\nleaf_count=16134 55879 131 1522 3094 240 40088 2479 1504 1029 17978 1540 2427 612 125 1122 433 4333 174 428 328 1068 127 50881 40555 21 155 746 106 556 306\ninternal_value=0.000367767 0.000823406 -0.00257165 -0.00074208 0.0028141 0.00314951 -0.00238329 -0.00117369 0.00731923 -0.00343373 -0.0013312 -0.0170827 -0.00460356 0.00841695 -0.0081078 -0.00790241 -0.0048005 -0.000685143 -0.00471047 -0.0142231 -0.0233626 0.0138516 -0.0153311 -0.00807923 0.00264809 -0.00941249 0.0106712 -0.0189242 0.0165988 0.0205942\ninternal_weight=342.818 335.844 124.027 109.399 211.816 210.264 104.113 96.4926 19.8101 76.6825 67.3979 9.28458 53.4434 10.4896 10.1101 42.9538 36.8673 28.5245 22.8681 13.8478 6.08651 5.65638 14.6285 11.7985 6.3986 3.97765 13.9544 6.97467 8.39141 4.90521\ninternal_count=246121 190136 108946 41931 81190 80950 38837 36358 3655 32703 12298 20405 10030 1293 2626 8737 6129 1796 1189 756 2608 607 67015 40862 307 176 2268 55985 681 433\nis_linear=0\nshrinkage=0.03\n\n\nTree=214\nnum_leaves=31\nnum_cat=0\nsplit_feature=4 2 12 13 2 0 18 4 17 21 6 4 5 22 2 2 2 13 2 17 17 0 18 18 18 0 14 13 13 21\nsplit_gain=3.7098 3.15618 2.12068 3.37319 2.95678 2.33462 2.6661 2.09862 1.83717 1.72889 2.50801 2.01294 1.54053 1.81808 3.82014 1.86961 4.24017 2.80859 1.81354 1.49993 1.43078 1.38935 1.23765 3.39115 2.76033 2.14046 3.13589 2.99779 3.59322 2.65406\nthreshold=0.84667968750000011 -41.62170219421386 0.21199991874952331 0.14796868535883259 32.956295013427741 1.3170161875856263 0.89587973461402759 1.0000000180025095e-35 3.9653415636860969 2.5000000000000004 1.0000000180025095e-35 3.9514160156250004 4.5058593750000009 0.93750000000000011 5.317040443420411 7.8238990306854257 20.814876556396488 -0.40579127137408294 50.658309936523445 0.78716740761650772 0.61464819824310435 1.8987378677022575 5.0271592211193568 5.2600927804663344 5.5664324555418956 0.69651904584641833 0.20058490075496901 -0.10757639301670423 -0.18869652365325987 3.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=7 19 5 4 -4 6 -3 -1 22 10 -5 -11 13 14 -7 -15 17 -17 -18 20 -2 -9 25 -24 -25 26 -8 28 -27 -28\nright_child=1 2 3 9 -6 12 8 21 -10 11 -12 -13 -14 15 -16 16 18 -19 -20 -21 -22 -23 23 24 -26 27 29 -29 -30 -31\nleaf_value=0.011629267966772451 -0.013640933145654593 -0.027011484485160747 -0.019446812917241072 0.018877757902575083 0.0065137585409734767 -0.025367110203209486 -0.0240489591487947 -0.024519979390600181 -0.013764972047007615 -0.017304298812595525 -0.016825920096833331 0.010465956505073941 0.02036383794387267 0.01099704327277792 0.0034923033097858588 0.0032437515261979728 0.014778973131330104 -0.026485774164308321 -0.0079435368459377775 -0.024168048692442778 0.0085001909954286175 0.0077082254256889875 0.020900906876916862 -0.026043819390270687 0.0051571546356531574 0.0017242944489485102 0.0043233050622991072 0.012613903007956709 -0.027712327663710516 -0.014065991405704702\nleaf_weight=1.4964173242492478 4.7884683196014048 2.0603747908462529 14.345684323607204 8.3277268722886237 4.757356748887104 3.8561228643957319 5.606395893012631 8.2437059054609545 6.0388576998025192 4.3221857440657905 1.3658837936818589 3.3165196055779234 5.721509702387265 52.32960578381244 26.572061379382831 4.8620727525849361 13.882290290610401 5.3279464065472002 3.3263872023235299 4.7455783888201486 3.9016263528956188 0.91354454748579961 9.5295548598171553 2.2980495798533402 10.355381877662692 74.713666689008733 22.396217733981761 21.988661345364562 2.9256693343977522 9.1279017297511\nleaf_count=321 20003 64 2074 227 122 556 5547 41833 28305 89 32 96 64 712 2131 51 168 111 138 47130 5012 179 11001 8475 12398 12476 27682 3699 1098 14006\ninternal_value=0.000951072 0.0015293 0.00214331 -0.00458295 -0.0129793 0.00302264 0.000742338 -0.0156924 0.0012496 0.00487435 0.0127439 -0.00543139 0.00627527 0.00536689 -0.000855786 0.00772295 0.00130262 -0.0132784 0.0101917 -0.0122027 -0.00384881 -0.0208625 0.00190256 0.00831407 -0.00200187 0.000809772 -0.00525005 0.00309794 0.000276747 -0.00127013\ninternal_weight=343.443 332.79 319.354 36.4354 19.103 282.919 167.041 10.6537 164.98 17.3323 9.69361 7.63871 115.878 110.156 30.4282 79.7283 27.3987 10.19 17.2087 13.4357 8.69009 9.15725 158.941 22.183 12.6534 136.759 37.1305 99.628 77.6393 31.5241\ninternal_count=245800 203467 131322 2640 2196 128682 124751 42333 124687 444 259 185 3931 3867 2687 1180 468 162 306 72145 25015 42012 96382 31874 20873 64508 47235 17273 13574 41688\nis_linear=0\nshrinkage=0.03\n\n\nTree=215\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 7 22 17 13 2 2 5 3 3 12 12 7 3 18 17 21 10 1 17 5 17 14 5 7 10 0 18 7 1\nsplit_gain=3.77613 3.02975 3.07063 3.48319 3.59107 3.96234 4.24718 2.9575 2.61137 2.80891 2.59367 2.45236 4.0867 2.32133 2.87947 2.27331 2.23781 3.87916 2.59016 2.44158 2.11358 2.09834 2.09 2.08894 2.06811 2.04768 2.04459 3.00757 2.02079 1.98556\nthreshold=-34.692844390869134 0.89599609375000011 0.93750000000000011 0.48975197669932524 -0.20399465096770289 1.6481127738952639 -1.7340650558471677 1.4980468750000002 0.62297257604742995 0.53650715206675215 0.7283458988905912 8.8817841970012543e-16 1.2426757812500002 0.47260169401963864 4.2121001622835372 0.24468021674538498 1.5000000000000002 1.0000000180025095e-35 2.9477339485081675 3.9281938047451082 0.89599609375000011 2.1810207941665634 0.88948809600489676 2.1386718750000004 1.2426757812500002 2.249904835165133 1.8987378677022575 5.0271592211193568 2.8037109375000004 0.95935769349451683\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=13 2 21 8 5 6 -5 20 9 15 -10 16 -13 -1 -15 23 17 18 24 -18 -3 -2 -7 -4 -9 -14 27 -11 -19 -26\nright_child=1 7 3 4 -6 22 -8 11 10 26 -12 12 25 14 -16 -17 19 28 -20 -21 -22 -23 -24 -25 29 -27 -28 -29 -30 -31\nleaf_value=-0.021389689713524441 0.0034936151215292739 -0.02030499822096429 -0.022303942362358394 -0.0081619376676406868 -0.033825290260996378 -0.030475504180980405 0.026471008355244127 -0.02756635000518546 -0.025835776292706242 -0.010553389974568426 0.015245283241698871 -0.033556732871381438 0.0015456162081773802 -0.012979787031130834 0.013769532959844935 0.0087299840372518991 0.0053588282170188763 0.022666863411869218 -0.02741562942903299 -0.021420818585397482 0.011364607816866015 -0.013515483669828901 0.0083758134393262917 0.014192824013269512 -0.018791431856548992 -0.022443796162159236 0.035006900375661038 0.028337699236004179 -0.0064703741434057535 0.0047699100262250452\nleaf_weight=10.688310285970603 51.936723440646119 1.1010219534500709 5.5202948924782103 12.360607471623554 6.2959652100455505 6.8536194704565778 3.1851562992369535 1.9825916411025541 4.5538796990076653 3.707041662768459 1.1064693629741666 3.4647820400409755 20.527287100846937 8.3927896881723409 4.9224723400657231 5.7384155685035694 103.64598659812418 5.0859256221156093 5.0864207296981467 2.1877006853726462 38.021529457531869 6.4546688248690307 1.0382852178245228 1.0628043179913218 3.1256901097913241 2.7722503850891362 2.6276123439893126 1.9250803132817962 2.7434419060446089 13.654545648966176\nleaf_count=92658 50545 1036 4158 8633 5374 1447 599 3401 246 291 5 493 1880 385 659 2594 19036 409 156 7486 5177 27538 171 153 9776 269 12 22 165 1469\ninternal_value=-0.000387726 0.000494016 -0.00339464 -0.00835127 -0.0152056 -0.00902434 0.000104973 0.00268643 -0.000311796 0.00476716 -0.0167221 0.000897289 -0.00724722 -0.0116134 -0.00281984 -0.00430319 0.00252335 -0.00437539 -0.0103307 0.00460994 0.00998947 0.0014175 -0.024997 -0.0153171 -0.00454621 -0.00211488 0.0173014 0.00500817 0.0128806 -0.000428946\ninternal_weight=341.769 317.766 114.367 55.9752 29.7336 23.4377 15.5458 203.399 26.2416 20.5812 5.66035 164.277 26.7643 24.0036 13.3153 12.3215 137.512 31.6786 23.8492 105.834 39.1226 58.3914 7.8919 6.5831 18.7628 23.2995 8.25973 5.63212 7.82937 16.7802\ninternal_count=246243 152541 101788 23705 16224 10850 9232 50753 7481 7230 251 44540 2642 93702 1044 6905 41898 15376 14802 26522 6213 78083 1618 4311 14646 2149 325 313 574 11245\nis_linear=0\nshrinkage=0.03\n\n\nTree=216\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 2 4 4 13 18 2 4 4 1 19 13 4 19 18 3 3 19 21 18 1 2 2 7 18 4 14 14 21 18\nsplit_gain=3.52652 3.13587 2.27553 1.99829 4.11066 3.13033 2.6266 3.67081 2.56925 3.40839 2.98435 2.36806 2.39786 3.06309 2.10601 3.79234 3.08877 2.41385 2.12523 2.06615 2.00084 1.90965 2.73789 2.50126 1.89087 1.86794 2.72927 3.37282 1.74748 1.74077\nthreshold=0.53650715206675215 -51.587459564208977 0.84667968750000011 3.1247558593750004 -0.39782810107798666 4.297262267176281 53.951957702636726 3.4626464843750004 1.0693359375000002 1.4475910192202701 242.50000000000003 -0.68251518906484077 1.9406738281250002 264.50000000000006 5.0005792898547883 0.22136347059745312 0.50207005315878861 245.50000000000003 4.5000000000000009 3.3140206880897671 -0.041910702480567281 5.8049912452697763 2.0782871246337895 1.0000000180025095e-35 4.6298509007569537 1.6286621093750002 0.48434979059990024 0.82532919407277039 3.5000000000000004 3.8815106545092593\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 -2 4 5 11 7 28 10 -10 -6 -3 13 25 15 19 17 18 24 -9 -16 22 -14 -23 -17 26 -13 -28 -5 -11\nright_child=2 3 -4 6 8 -7 -8 14 9 29 -12 12 21 -15 20 16 -18 -19 -20 -21 -22 23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.022193781883696868 -0.017841587029296512 -0.015007244330116893 0.0049762117685289006 -0.0046384711522212928 -0.01955250683337903 -0.021863620128810858 0.014718013296653931 0.014858738769281558 -0.029308025236307194 -0.010825077121225759 0.0033954258690803321 -0.020125912048530707 -0.0034022174506278279 -0.013728601082047447 -0.015500102512660232 -0.030199157107047361 0.0056879386199809139 -0.0034555325046915007 0.00032331225990632793 -0.0087766161130447707 0.0096622748858570372 0.014659349396043457 0.029620008225251591 -0.028249409490398721 0.006402772221248812 0.02614932063015268 0.019497393030645811 -0.022648999531770259 0.028005671207653321 0.0032786487628676747\nleaf_weight=5.8567566744763999 3.1110790057446129 3.6337086843773241 112.43530039610596 1.6501900528844533 5.2294379293884949 3.0980467014394426 11.53726278430622 9.2321934987985497 7.3320628784954343 40.125052387199503 27.17589485272083 1.672268662167659 8.3214796313968673 2.6374955228225181 2.4589060796047297 12.456035696654913 7.334372802903089 10.02139718775288 3.056539411720677 4.1400923103024061 16.774007689093654 1.0122607160819814 2.0209413328848305 3.1345681653174324 1.0553085578512753 9.7813348784241025 10.222924353129654 1.1540537068703987 4.6782814328396398 9.2503888366231859\nleaf_count=61198 834 4829 3878 3002 13145 25606 1052 942 55388 2274 25174 755 2452 1613 14763 1231 193 960 724 4015 8442 82 144 693 264 872 4015 2824 1762 3104\ninternal_value=-2.81711e-05 -0.00220557 0.00421403 -0.00157599 -0.00381821 0.00334593 0.00203449 -0.00014608 -0.00756721 -0.0113637 -0.000782269 0.00558772 0.00778131 0.0131568 -0.00233264 -0.0057074 -0.0109544 -0.0155848 -0.0223072 0.00743966 0.00583821 -0.00190675 0.00509297 -0.016962 -0.0270961 0.0167085 0.00827438 0.0137377 0.0200221 -0.00817014\ninternal_weight=341.6 226.053 115.546 220.197 135.802 46.6891 84.3946 72.8573 89.1128 56.7075 32.4053 43.591 39.9573 25.4681 66.5289 47.2959 33.9237 26.5893 16.5679 13.3723 19.2329 14.4892 10.3424 4.14683 13.5113 22.8306 13.0492 11.377 6.32847 49.3754\ninternal_count=246230 241518 4712 180320 142970 43885 37350 36298 99085 60766 38319 18279 13450 10079 31534 8329 3372 3179 2219 4957 23205 3371 2596 775 1495 8466 7594 6839 4764 5378\nis_linear=0\nshrinkage=0.03\n\n\nTree=217\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 4 2 17 1 3 7 9 4 7 7 4 2 2 17 3 17 18 3 4 4 14 0 1 17 18 4 2 4 2\nsplit_gain=3.53557 3.12897 1.85827 3.1473 2.45031 2.89254 3.18111 2.05533 4.0394 6.31295 1.97454 1.84757 2.59287 4.71417 4.33797 3.02006 2.91992 4.24841 4.10771 2.69312 2.68854 2.55568 2.84626 2.41157 2.24396 3.1334 2.19098 2.06599 3.13932 2.57506\nthreshold=-51.003999710083001 0.84667968750000011 21.425971984863285 2.4775242203633088 -0.10824782141703854 0.2708708835714248 3.5693359375000004 1.0000000180025095e-35 6.6144968271255502 3.1132812500000004 3.6503906250000004 2.1750488281250004 3.4383306503295903 -5.1300392150878897 1.1976019298503242 0.48555671035752257 0.38405652560789177 1.7005986908310777 0.65908129556435835 2.1433105468750004 1.0693359375000002 0.2247238965590572 1.8987378677022575 1.7622051334857016 0.077783200051907503 2.1383330595080281 2.1186523437500004 -7.9326825141906729 2.1433105468750004 -9.4683709144592267\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 -2 3 7 -4 6 -6 8 9 11 -7 12 13 24 16 21 17 -14 26 20 -17 -15 -23 -16 25 -3 -18 29 -29 -26\nright_child=1 2 4 -5 5 10 -8 -9 -10 -11 -12 -13 14 15 23 19 18 -19 -20 -21 -22 22 -24 -25 27 -27 -28 28 -30 -31\nleaf_value=-0.019074258702858012 -0.015889872714914332 -0.016261065301090016 0.019947998441595977 -0.013382026983795836 -0.0174046376683317 0.006423442799406316 0.020261578681591143 0.0098465376326762003 -0.020103805586425529 0.029308044621297595 -0.014503166879683489 -0.0046573499714958836 0.017359384414001748 -0.01129786332611276 -0.017326829894965388 0.0066943766606443267 -0.036239515233424044 -0.022185406997420082 0.0052898595433158958 0.029036781976062562 -0.02284164828893679 0.028567774191396337 -0.017165333791474859 0.020769529339000978 -0.0023234486517823218 0.027652264833664862 0.0039877499834236729 -0.027404255462186649 0.017644072957130844 0.020267195852950819\nleaf_weight=7.7579867605140853 9.4837058534480985 1.2852335301504343 7.3851799391711657 15.211974182767333 9.5264388626078453 64.285714767628633 1.6124145450594358 17.641517727726296 8.094007717530074 5.5491148427299768 3.3956030278932294 65.907696686496365 4.9057002335321211 1.3336696205678951 1.1050454861256174 6.5238252196213544 8.0220609208761271 3.1745822160546595 3.4700748390168874 1.6524159640539444 3.4511282250132354 14.218373242289314 0.68577511439798478 5.6846770276169991 49.893626878587149 4.1438458166248893 1.1937201924056342 4.7965572507960195 1.0756064970114492 3.986363281840112\nleaf_count=62475 38228 59 10116 35313 8211 4378 32 2337 4228 429 101 34116 121 731 3088 107 3541 2295 21 12 320 5273 16 217 25396 2287 223 1819 67 1051\ninternal_value=-0.000261938 0.000238584 0.000765883 -0.000629702 0.00450201 0.00282299 -0.0107323 0.000318082 -0.000566533 0.000356701 0.00517767 -0.000656378 0.00155042 0.00394303 -0.00638372 0.0140365 -0.013078 0.00109077 -0.021523 0.00203636 -0.00467462 0.0220128 0.0255165 0.0131409 -0.000518858 0.0163443 -0.0311486 -0.00229485 -0.017786 -0.000313213\ninternal_weight=336.454 328.696 319.212 233.007 86.2054 78.8202 11.1389 217.795 200.153 192.059 67.6813 186.51 120.602 93.0464 27.5559 27.8652 20.7661 8.08028 12.6859 11.6274 9.97495 16.2378 14.9041 6.78972 65.1812 5.42908 9.21578 59.7522 5.87216 53.88\ninternal_count=246608 184133 145905 123067 22838 12722 8243 87754 85417 81189 4479 80760 46644 37138 9506 6459 6201 2416 3785 439 427 6020 5289 3305 30679 2346 3764 28333 1886 26447\nis_linear=0\nshrinkage=0.03\n\n\nTree=218\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 4 3 18 1 3 1 1 4 0 12 18 3 1 17 1 1 17 4 18 18 0 0 7 3 7 18 13 17 17\nsplit_gain=3.78299 2.38001 2.01523 2.42815 4.66807 2.42705 2.83802 2.28335 1.99925 2.61456 2.83037 2.41301 2.37607 2.73184 2.35596 2.52096 2.34507 2.73183 2.28658 2.26987 2.01264 4.47087 2.87073 1.96557 1.96031 1.95843 1.87185 1.83912 1.79955 1.69904\nthreshold=3.2718994203396332 0.84667968750000011 0.53650715206675215 0.89587973461402759 4.0016611463566001 0.016076142967085191 -0.32709731491006455 -0.27208026740318675 3.1247558593750004 0.47605733498249619 0.61117045150363192 4.0342014792848504 0.50207005315878861 1.4475910192202701 0.41124616060114544 0.73033385732376643 1.4866206336930319 0.17669220786083642 8.2534179687500018 5.589117819455554 2.8029010331479989 0.69651904584641833 0.63578092066124181 3.9980468750000004 0.2352687277455576 2.6738281250000004 2.249904835165133 -0.67231277955111934 1.0587051740042546 0.077783200051907503\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 5 4 -4 -3 -7 -8 12 18 11 14 13 -9 -11 -16 28 29 19 -10 26 22 -22 -2 -23 -21 27 -19 -15 -18\nright_child=23 2 3 -5 -6 6 7 8 9 10 -12 -13 -14 16 15 -17 17 20 -20 25 21 24 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.012850850442898727 -0.020170704003131704 0.016000932560954355 -0.018989975544149836 0.0063998017059624735 0.013971869176373686 -0.022519283032879936 0.02381216669161387 -0.013230012156083083 0.024855180305162834 0.014463470256229946 -0.019282882916345469 -0.023268287686317373 -0.012817188792588393 0.026986752644075372 -0.025120991691893819 0.0022209413325327779 -0.0029417263639097647 -0.023651404594550399 -0.004363122301093072 -0.016790985255713257 0.0022216987764510062 -0.016338319057064704 -0.030647198336676309 0.0069787946533904849 0.021572161524429662 0.014269154697055423 -0.01621556552657585 0.0021243794196489081 -0.015790130077179557 -0.0244001627222112\nleaf_weight=9.7198140651257727 10.529398045629817 6.9615623126001074 8.9621549682924506 98.764714829079821 5.4172019488178185 4.3146557214230361 2.4734238956922345 13.741733097228119 10.442545716624865 12.174934894022043 5.3773179727795641 2.1599748763746893 17.934776958873044 3.6759719913006839 2.4093858834053234 26.702309346243055 7.8618429313319194 1.6643373720289663 5.4850182802201735 2.193657784986498 11.847494141235986 0.62276473499150498 1.9615777565759334 2.4903485373258709 13.930000502723486 3.2759059801093144 6.1531191509347982 37.43875600885076 0.39948736570295285 5.1851212771616702\nleaf_count=31518 69565 5362 71 3536 64 51618 3671 51243 5922 1012 176 782 1046 278 942 1055 470 41 1016 6311 3660 238 303 542 867 2538 462 727 142 775\ninternal_value=0.00041431 0.00106106 0.00152302 0.0047217 -0.00647106 -0.000244839 -0.000888951 -0.000298233 -0.000729178 0.00327759 -0.000603464 0.00209029 -0.00303548 -0.00125082 0.00393584 -0.000801082 0.000687742 -0.00046766 0.0117649 0.0172536 0.00182248 0.00794699 -0.00420153 -0.0148503 0.0190036 0.0011421 -0.00210455 0.000464881 0.0205087 -0.0125997\ninternal_weight=342.271 329.252 319.532 113.144 14.3794 206.388 199.426 195.111 192.638 70.2211 48.8239 43.4466 122.417 104.482 41.2866 29.1117 90.7405 86.665 21.3971 15.9121 73.618 28.3618 13.8091 13.0197 14.5528 5.46956 45.2562 39.1031 4.07546 13.047\ninternal_count=245953 175846 144328 3671 135 140657 135295 83677 80006 19754 3967 3791 60252 59206 3009 1997 7963 7543 15787 14771 6298 5068 3963 70107 1105 8849 1230 768 420 1245\nis_linear=0\nshrinkage=0.03\n\n\nTree=219\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 22 3 7 12 1 3 3 5 5 5 7 7 22 18 3 3 0 19 18 21 5 1 1 18 1 5 1 5 14\nsplit_gain=3.16914 4.82495 4.24501 3.08234 2.58963 2.38792 3.63887 1.96406 2.85855 3.83181 1.78783 1.76676 1.75402 3.57088 2.87202 2.66774 2.80589 2.71684 1.94757 1.73971 1.69668 1.69434 2.30087 1.85554 1.68728 1.47924 1.457 1.43939 1.42099 2.17502\nthreshold=1.3170161875856263 1.0000000180025095e-35 0.47260169401963864 1.0000000180025095e-35 0.63107571507209059 2.1468275719512682 0.37963534769016083 0.40911314460322756 4.5488281250000009 4.9648437500000009 1.0000000180025095e-35 1.8691406250000002 3.2763671875000004 0.93750000000000011 0.89587973461402759 0.099577421794159224 0.53650715206675215 1.5447487307370646 245.50000000000003 2.0126758453675748 1.5000000000000002 4.3378906250000009 1.6109634896136267 0.26032702032866095 5.0530509062989237 1.9634439907166827 1.2426757812500002 1.1267314367267358 1.2426757812500002 0.73406812255247111\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 5 10 12 20 -7 -5 21 -10 -1 -8 13 14 18 -16 19 -18 25 -17 -3 22 23 -9 -12 -2 -20 -4 29 -29\nright_child=4 2 27 7 -6 6 11 8 9 -11 24 -13 -14 -15 15 16 17 -19 26 -21 -22 -23 -24 -25 -26 -27 -28 28 -30 -31\nleaf_value=0.013138447102879992 -0.0054005015803318297 0.0078951833928750572 0.015466012196724548 -0.00058518764634070007 -0.010027893061661713 0.0098127918735510591 -0.02556445977653277 0.016132219817595175 -0.030812553330996598 0.0087165373995772844 -0.028428843292241331 0.00764564605262473 0.018067349923117801 0.0075091830920287136 0.013823240752019813 -0.023928130555715792 -0.01643564371854481 0.023891706761569896 0.0078833784097889856 -0.00076469221121936251 -0.026968485219594506 0.025164792026224261 0.0097878535404047253 -0.021936612763035535 0.0071917384064666967 0.022765363496012672 -0.025204795588637231 -0.0035713465074121912 -6.9223515932642743e-05 -0.026572726800773775\nleaf_weight=0.90819913295626875 1.7953226401750706 0.93961528320415055 2.8280924743157803 59.35216855875251 9.9521599206491356 6.055057167690391 6.17861006567 0.7878235228645275 2.9896005575646987 4.2034220532295876 4.8870529317798628 1.3155829302959317 7.9995993800694114 84.065356808594515 2.9504744118203261 12.523944066037989 1.7544591098994704 2.4394709953630809 1.695174382759433 3.8072076602938969 14.068965426002135 4.9449031738913609 39.364021532859624 3.0496301887178561 1.0644848392075799 5.7165044896537438 1.3804666098458254 13.45325361839423 35.39481245580464 4.3218945381267977\nleaf_count=231 259 1664 322 109032 248 772 509 524 301 169 34862 29 128 1683 929 1186 96 65 151 856 81733 155 1843 2813 2648 66 73 994 1162 603\ninternal_value=0.000510439 -0.00183313 -0.00729355 0.00199386 0.0040475 -0.0164984 -0.00663837 0.00321367 0.00722606 -0.0094683 -0.0162148 -0.0195208 0.00522827 0.00421444 -0.00389902 -0.00963275 -0.0134901 0.00710513 0.0085485 -0.0188838 -0.024429 0.00986209 0.00758082 -0.0124019 -0.0219397 0.0161901 -0.00950822 -0.00239217 -0.00361001 -0.0102814\ninternal_weight=342.187 206.107 84.5559 121.551 136.08 28.5578 13.5493 114.692 55.3394 7.19302 6.85974 7.49419 126.128 118.128 34.063 23.4756 20.5251 4.19393 10.5875 16.3312 15.0086 48.1464 43.2015 3.83745 5.95154 7.51183 3.07564 55.9981 53.17 17.7751\ninternal_count=246106 240366 87788 152578 5740 84707 1310 114837 5805 470 37741 538 5492 5364 3681 3132 2203 161 549 2042 83397 5335 5180 3337 37510 325 224 3081 2759 1597\nis_linear=0\nshrinkage=0.03\n\n\nTree=220\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 17 6 1 4 4 19 2 4 2 2 2 4 2 14 13 4 14 4 5 2 21 1 1 13 17 1 1 14\nsplit_gain=3.07837 2.47022 2.57665 2.54927 2.19569 2.1505 2.69961 1.86461 2.76124 2.13611 1.78295 1.64013 2.60984 3.04922 2.46274 2.45041 1.98919 1.98484 2.75344 3.1136 2.90243 2.23989 2.39515 1.98583 1.91294 1.89701 1.8745 3.35297 1.86421 3.60794\nthreshold=-51.587459564208977 1.7622051334857016 1.9512128003148403 2.1386718750000004 1.2421711908324335 1.0693359375000002 0.84667968750000011 255.50000000000003 18.240050315856937 1.6286621093750002 32.956295013427741 22.129880905151371 16.994721412658695 5.5515136718750009 22.867874145507816 0.33900667957710101 -0.30495351492633627 1.6572265625000002 0.30493411065509307 2.9458007812500004 2.4414062500000004 -14.08206272125244 1.5000000000000002 2.2520445226790948 4.0016611463566001 0.14796868535883259 0.38405652560789177 3.3033179733375269 2.9477339485081675 0.70437716445572607\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 5 4 10 6 -2 9 -9 -7 -4 12 16 15 -13 -14 28 23 21 -20 -21 -19 -23 -18 -24 -25 27 -16 29 -3\nright_child=1 11 3 -5 -6 7 -8 8 -10 -11 -12 14 13 -15 26 -17 17 18 19 20 -22 22 24 25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.018290131283798489 -0.01445208619781958 -0.010235066931617059 -0.01866969144005011 0.0032682666855378445 -0.034325015975688621 -0.024201441994889987 0.015125915258875657 -0.026031706807306099 0.0084970394063571576 0.00037515286384812965 0.0040607408641206251 0.027847118199480519 0.00019189694275325997 0.013407355842348864 -0.021329508837170674 -0.032322294403600522 0.019503693623763828 0.012040933500357379 -0.023758968523487922 0.022015137655550993 -0.01388099524447328 -0.02652373809699592 -0.011377445747073058 -0.0036385515090053234 0.012076296179363104 0.016072825050764815 0.0072461325725289537 0.010491996166969754 0.0081036070761320526 0.0077792275276577219\nleaf_weight=7.2426880616534381 2.6944511540016238 17.65297049703986 6.8582001908208721 6.4110985251877528 4.5776959599415923 2.4061407129904628 11.756684894259706 5.7570214080367332 2.4346271968638575 48.153156497680129 4.6351970903222082 3.4799037266202495 3.1466919012655108 2.2715925566153592 5.3148983624596422 5.1215081718655711 4.6265740703465772 8.3023591463552311 11.469526090082583 2.5778019009849222 3.6906287399579014 3.0160787474123927 5.1318518903753993 23.151414127858519 5.3958152802661061 4.3714842163026324 42.008222520705203 4.7890950041619362 51.641898134257644 20.590159991708788\nleaf_count=61558 23487 1488 48748 1140 2944 16452 9443 14663 726 45147 10812 58 138 44 353 277 893 357 893 110 270 138 278 1303 47 119 1810 102 739 1710\ninternal_value=-0.000185551 0.000274992 -0.00374624 -0.0123949 -0.0184687 -0.000958763 0.0090326 -0.00352641 -0.0159615 -0.00124116 -0.0099114 0.00197341 0.000513854 -0.013276 0.00640445 -0.0212625 0.00148925 -0.00221974 -0.00664417 -0.0150821 0.0018787 0.000567301 -0.00681038 0.00326394 0.000648262 -5.40351e-05 0.00447522 -0.00666017 0.0044411 -0.000585061\ninternal_weight=330.677 323.435 95.6843 22.4822 16.0711 73.2021 14.4511 58.7509 8.19165 50.5593 11.4934 227.75 172.158 10.5398 55.5921 8.2682 161.619 71.7335 39.5841 17.738 6.26843 21.8461 13.5437 32.1495 10.5277 27.5229 52.1122 10.104 89.885 38.2431\ninternal_count=246247 184689 173562 63644 62504 109918 32930 76988 15389 61599 59560 11127 8804 459 2323 415 8345 4408 2093 1273 380 820 463 2315 325 1422 2265 455 3937 3198\nis_linear=0\nshrinkage=0.03\n\n\nTree=221\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 3 1 1 17 3 10 10 17 3 6 2 19 19 2 14 2 3 2 2 6 6 5 5 12 5 3 18 1\nsplit_gain=3.18969 2.09157 1.54811 1.77759 2.87436 2.33248 1.80721 1.38611 1.48526 1.51528 1.66622 2.19959 1.21567 3.83926 1.51624 2.14143 1.48896 1.36324 1.17266 1.81173 3.17403 1.76495 3.66581 2.00601 3.46608 1.84251 4.09129 2.371 3.64175 4.00267\nthreshold=29.286581993103031 -0.59079332554470076 0.53650715206675215 2.6981674883069502 2.054932859494814 1.2783113269342083 0.62297257604742995 3.6242520361853052 1.8688348091416842 2.6250469244966204 0.56033297298897644 1.8691406250000002 22.867874145507816 261.50000000000006 244.50000000000003 27.147908210754398 0.56086626140189244 22.129880905151371 0.082806484040480313 11.076014518737795 10.479034423828127 1.0000000180025095e-35 1.4980468750000002 3.7734375000000004 2.1386718750000004 0.49742347178520357 1.2426757812500002 0.44825707527455844 0.89587973461402759 2.6981674883069502\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 -2 4 6 -5 -4 8 9 12 11 -11 17 14 -14 16 -16 18 -3 20 21 23 -23 24 25 27 -27 28 -20 -30\nright_child=2 7 3 5 -6 -7 -8 -9 -10 10 -12 -13 13 -15 15 -17 -18 -19 19 -21 -22 22 -24 -25 -26 26 -28 -29 29 -31\nleaf_value=-0.019579857696728261 0.0031599826416008033 0.012462744581874855 0.022090689309363466 0.028379708815118979 -0.025654730677013533 -0.0015738160096279124 -0.0051332055497775933 -0.025897710306357773 0.012802863901784398 -0.023611518847093362 0.0075033357421092585 0.0034423541044479341 -0.030202764250446237 0.019844646924874861 -0.028989555209094998 0.013078711260741891 0.0053455251542406795 0.017752114118628436 0.017756506882364732 -0.0071468067343314293 0.024886019233681958 -0.022670973281504091 0.003904126809530584 0.014395017209761241 -0.013342929297497844 0.0048435523818255227 -0.030754320007662185 0.0050947756717894324 0.00029738933363121132 -0.019765838176155501\nleaf_weight=4.3371068869495266 45.641323869828739 5.5712615687591542 4.1384707967263257 7.9663045464549196 1.5285686461720613 3.0290633242402691 3.2861574134112743 0.93703085326478586 6.431743395311968 7.4436554281173795 3.165139165472282 3.6419839790445394 3.7289130444605671 2.3652366933511066 2.4501237258832593 2.3550544652173357 1.2910154446628892 2.9456649122962508 5.4733876293030326 29.888818542086256 3.7528870922731121 7.8702723979370175 9.3473241735559878 8.501859921934054 14.408608525707789 6.5479949448554162 3.9196662239792204 97.703525869763325 26.183150316652018 12.573775296721577\nleaf_count=63959 21175 11299 301 60 49 60 302 130 517 36797 292 4797 1947 378 458 458 782 572 131 9074 472 5560 5758 1636 13503 290 280 4194 60296 816\ninternal_value=0.000451846 -0.000981536 0.0063395 0.013116 0.00267662 0.0206847 0.0108623 -0.000609575 -0.000424911 -0.000798316 -0.0099756 -0.0151741 -0.000230461 -0.00909267 -0.0172485 -0.00618516 -0.0185126 0.000266492 -3.10853e-05 -0.000392351 0.000667194 6.89566e-05 -0.00882117 0.000987146 0.000222073 0.00158318 -0.0100057 0.00250188 -0.00321175 -0.0065451\ninternal_weight=338.425 272.835 65.5899 19.9486 8.9532 10.9954 7.42463 268.498 267.561 261.129 14.2508 11.0856 246.879 12.1903 9.82511 6.09619 3.74114 234.688 231.743 226.171 196.282 192.53 17.2176 175.312 166.81 152.402 10.4677 141.934 44.2303 38.7569\ninternal_count=246343 224396 21947 772 652 120 603 160437 160307 159790 41886 41594 117904 4023 3645 1698 1240 113881 113309 102010 92936 92464 11318 81146 79510 66007 570 65437 61243 61112\nis_linear=0\nshrinkage=0.03\n\n\nTree=222\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 1 19 18 18 21 1 1 7 7 11 5 1 10 1 1 18 18 1 0 19 14 1 1 5 4 4 1 10 11\nsplit_gain=2.01004 4.50853 3.00584 4.09801 2.69861 3.1972 2.17583 2.23864 2.16274 2.21419 1.98462 1.90882 1.8204 1.6622 1.58217 1.57976 2.013 1.57156 1.56121 1.52219 1.51464 1.42011 1.379 1.33973 1.3258 2.07835 5.20288 2.22484 2.08807 2.05021\nthreshold=0.69651904584641833 -0.041910702480567281 264.50000000000006 4.8242976514536702 4.516324041782946 1.5000000000000002 -0.10824782141703854 0.29645739473604366 2.6015625000000004 1.2426757812500002 0.66750229771629133 4.3496093750000009 4.0016611463566001 1.8688348091416842 1.1008244052526537 2.9477339485081675 4.135134555718313 6.1147871526576338 2.9477339485081675 0.17863154480046692 245.50000000000003 0.53481096685124585 0.45969269984116679 1.5275240303550033 4.4003906250000009 6.6611328125000009 8.2534179687500018 1.0240454605967615 1.0000000180025095e-35 0.11347059467310934\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 4 5 21 -2 -8 9 14 -11 17 -6 15 -5 -7 -17 -10 19 -16 -18 -3 -13 -4 29 -26 -27 -28 -29 -9\nright_child=6 2 23 8 12 13 7 24 11 10 -12 22 -14 -15 18 16 20 -19 -20 -21 -22 -23 -24 -25 25 26 27 28 -30 -31\nleaf_value=-0.02574830336830624 0.014619324526021731 0.011043840057058146 -0.012569316380365399 -0.017745307527569604 -0.034676838593639563 -0.02564355415350144 -0.021868704784076008 0.0010036148129561647 0.028472575721843883 -0.023993443141790517 0.013496439402205357 0.01206027622016591 0.0029191912221595393 0.011981880912128533 0.023609356682973918 -0.021117917161641433 -0.016124292493089398 -0.0023139689412585954 -0.010693696494809895 -0.0036100221029856049 0.018302775457990722 -0.011999599024120605 -0.013746721934050432 0.019734367864821609 0.020239252191503482 -0.032428566188372421 -0.015413471801596905 0.022046498390352668 -0.0073308109307380675 -0.0084906019195285631\nleaf_weight=7.122203423264863 9.0405714535152111 8.5953831785993788 0.67071955290589258 1.3805171042769249 4.9768452401971208 8.3418166324297527 3.0089003828556988 200.34444094028231 5.5727204672511998 4.8101607628918819 0.92592057562433172 2.739091692546026 1.2556367250508627 1.2010280221584242 4.9998472928918982 2.6693636220952603 0.76964700622556881 1.7465031590843407 1.9160532288642569 2.3289103036886445 2.78017598635779 2.3095245539600446 2.719530852711614 6.2361124902563461 6.7712648584856678 2.357822130637941 1.8227330191875797 7.6273310047108662 2.3608563307207078 21.786881794041619\nleaf_count=122571 12746 4274 4120 17049 14221 16357 6491 18894 869 12427 28 687 57 47 2077 165 63 2742 295 2386 112 4760 779 589 134 104 111 95 57 704\ninternal_value=-0.000349372 -0.00459627 -0.00207434 -0.00423377 -0.0113405 -0.006642 0.000929899 0.00036984 0.0039001 -0.00329489 -0.0168366 0.0126834 -0.0277461 -0.0149159 0.00483502 -0.0177619 -0.0051025 0.0217312 0.00960991 0.0155648 0.00893516 0.00556551 -0.000934701 0.0154044 0.000733596 0.00776899 0.000868046 0.00952754 0.0150675 3.8533e-05\ninternal_weight=331.189 76.0677 68.9455 62.0387 32.8994 26.6669 255.121 246.08 29.1393 16.3614 5.73608 12.7778 6.23248 15.762 10.6253 14.561 6.21919 7.31922 9.24481 7.32876 3.54982 10.9049 5.45862 6.90683 243.071 20.94 14.1687 11.8109 9.98819 222.131\ninternal_count=246011 206675 84104 79395 40056 25778 39336 26590 39339 34262 12455 5077 14278 16744 21807 16697 340 3611 4758 4463 175 9034 1466 4709 20099 501 367 263 152 19598\nis_linear=0\nshrinkage=0.03\n\n\nTree=223\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 19 13 16 19 17 10 21 1 0 0 3 1 4 18 0 0 18 0 7 18 6 17 1 19 7 4 17 1 13\nsplit_gain=2.64804 2.38004 3.09075 2.37355 2.71817 2.5945 2.24639 1.94889 3.10619 3.43858 2.70356 2.01241 3.07869 2.90902 1.91762 2.92837 2.58907 3.24098 2.58314 2.47217 2.16099 1.90334 1.70337 1.70136 1.65759 1.64866 1.63465 1.59337 2.70049 2.02269\nthreshold=2.6981674883069502 262.50000000000006 -0.53713070123026629 1.0000000180025095e-35 228.50000000000003 0.068176441062887241 3.6242520361853052 6.5000000000000009 2.9477339485081675 1.8987378677022575 1.3170161875856263 0.58795853567030554 2.9477339485081675 1.9406738281250002 4.2696729980286863 -0.064617101914726832 0.7384456241152747 2.249904835165133 0.69651904584641833 2.1386718750000004 4.2121001622835372 0.89599609375000011 1.3029497703631001 1.8223781763416411 244.50000000000003 1.0000000180025095e-35 1.0693359375000002 0.63207041009378873 2.054932859494814 -0.21865947078897738\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 25 4 -4 26 -5 11 -9 10 -10 12 13 -2 16 21 18 27 20 -19 22 -16 -7 -20 -21 -1 -6 -18 -29 -30\nright_child=7 -3 3 6 5 14 -8 8 9 -11 -12 -13 -14 -15 15 -17 17 19 23 24 -22 -23 -24 -25 -26 -27 -28 28 29 -31\nleaf_value=-0.018385655435385572 0.023700834105798173 -0.0119094026018664 -0.022781156084542613 0.0059037470636098023 0.024682993678437522 -0.024671773605960855 -0.026765991765135503 -0.033257608188667361 -0.0075093564131954943 -0.020576089942645728 0.016426917411757409 0.0099227624531590623 -0.00081312819085564779 -0.0051280243908165069 0.014577015827838179 -0.024524963889994464 -0.011601100800055379 -0.024146328248177071 0.029271673353402868 -0.01998961167385092 0.020351828725650844 -0.01929002572184061 0.0037179017343064442 -0.012694936057691571 0.013324001377669087 0.0075167256928803791 -0.0037694604570596662 0.0089187504808771623 -0.022890710123956791 0.0032321559338122047\nleaf_weight=1.4104268890932861 8.7177311574105243 20.293147054986548 6.9348871063923516 27.628831231524263 3.3602955663709499 2.8373051464634482 1.0438955069812426 2.0514432444160802 6.6912921102575664 4.8076753180939695 8.8113292633788642 39.535040513408603 69.990437843769541 4.2066775792645776 3.3789842251296678 9.249935340670504 9.1212211773909058 11.417727362851108 4.2411599545184799 1.0433621624097202 1.7515302154425851 1.2872398547787041 3.3245825994813458 0.45721387439516536 2.9659188614568848 36.790747083341785 2.7174351802625551 17.959530133277443 4.3986017531133257 4.8473820579238218\nleaf_count=8961 467 22173 16865 7813 1429 9626 146 116 334 48 106 478 2008 309 11730 75272 1325 14748 2799 155 2547 22031 8312 151 132 29236 3872 2861 101 122\ninternal_value=-5.03731e-05 -0.00249011 -0.00121437 -0.00357978 -0.00593044 -0.00429454 0.00385238 0.00295274 -0.00503258 -0.000754847 0.00626647 0.00443997 0.0017386 0.0146231 -0.00580452 -0.0155313 -0.0034961 -0.00646109 0.00825183 -0.0175228 -0.00253506 0.00347879 -0.010974 0.0236766 0.00239473 0.00611577 0.0132263 -0.0014199 0.00228385 -0.0102165\ninternal_weight=323.273 178.461 158.168 119.967 91.2943 84.3594 28.6727 144.812 22.3617 20.3103 15.5026 122.45 82.9148 12.9244 78.2817 13.9162 64.3655 51.7537 12.6118 15.427 7.91342 4.66622 6.16189 4.69837 4.00928 38.2012 6.07773 36.3267 27.2055 9.24598\ninternal_count=246273 242407 220234 182037 174078 157213 7959 3866 604 488 440 3262 2784 776 151912 109033 42879 19444 23435 15035 20485 33761 17938 2950 287 38197 5301 4409 3084 223\nis_linear=0\nshrinkage=0.03\n\n\nTree=224\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 2 2 2 2 4 18 18 1 17 2 1 14 0 4 0 17 3 19 13 1 8 17 3 1 14 4 4 0 1\nsplit_gain=2.33545 5.57359 6.00732 7.16777 3.79291 3.42944 2.6235 4.74954 2.60702 2.59628 2.48869 2.3983 3.24204 3.07957 3.83 3.17761 2.26579 2.17985 5.17696 3.99987 2.14282 3.94596 3.28795 2.15571 2.10876 2.05061 1.92508 3.32261 2.882 2.83742\nthreshold=-5.1300392150878897 -3.9017317295074458 -2.5585746765136714 -1.3165187835693357 -7.1578526496887198 0.84667968750000011 4.2555875595453383 4.9593358439305844 -0.10824782141703854 1.2133651393498066 -1.7340650558471677 1.7622051334857016 0.11320561501600769 1.0212340148387515 4.6818847656250009 1.0917760095905165 2.4775242203633088 0.37963534769016083 223.50000000000003 -0.038818642250367956 1.6109634896136267 1.0000000180025095e-35 0.043095813018339015 0.26802264518526847 0.76426874490803853 0.43080354720652592 2.1750488281250004 1.6286621093750002 1.1495945197163422 3.3033179733375269\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 9 -3 10 20 -5 8 -8 -7 -2 -4 12 -10 25 15 -15 -9 19 -19 24 21 22 -1 -22 -14 -13 27 29 -28 -17\nright_child=1 2 3 5 -6 6 7 16 11 -11 -12 13 17 14 -16 26 -18 18 -20 -21 23 -23 -24 -25 -26 -27 28 -29 -30 -31\nleaf_value=0.01521356618987898 0.029442365439419891 -0.030951401513030323 0.0017326232853397485 -0.02313321499721686 -0.020580769368085564 0.018835961069898939 -0.025636610244078574 0.0088396829876804875 0.010870735195116978 -0.014964414315529107 0.031627472830457894 0.00049003685077685013 -0.01434299413852244 -0.027249724876426754 0.013485183061256468 -0.018538906857413492 -0.023704638779098359 0.0036401546652109385 -0.02976900899248712 -0.02323999000100933 0.015593978648301888 0.0040889089869694138 -0.024403337975799078 -0.0007530811323056731 0.014523547151680576 0.017600038523589809 0.016253165129152425 0.012397867257873124 -0.011000122471493012 0.0035608220818881004\nleaf_weight=1.5072894633171934 7.8632724783565191 4.1517687675518564 3.5910285219047173 4.4378872513782417 9.2056720215789465 7.1837853165859533 7.8541646234327827 10.054670689604789 8.3985669655835391 0.74673940158368157 8.3372094957210248 9.3624018716654991 2.0717446750122681 3.3451835945961639 16.010658530052751 7.313088945142229 1.346202650451686 6.623647322567197 9.4466814663046517 4.0812594896913224 6.9212190792532047 11.02471567048741 12.247090346379649 86.214935345738468 9.1240227898224457 18.675695394200375 3.2075318469142102 18.337175157845195 21.985667002358241 13.415136889554558\nleaf_count=3070 1457 2967 1927 17114 4855 8453 16438 8370 2003 1466 958 448 4069 169 236 381 6131 593 1733 1590 1845 8932 140774 6188 2167 862 33 305 483 172\ninternal_value=0.000157016 0.00211945 0.00103311 0.00187194 -0.00302962 0.000339067 0.00105089 -0.00912794 0.00233626 0.0244346 0.0234577 0.00143489 -0.00481463 0.00368258 0.000750241 -0.00241839 0.00339524 -0.00922319 -0.0165923 -0.000928285 -0.00149785 -0.00915317 -0.019325 0.000614465 0.00844381 0.0121003 -0.000727959 0.00341826 -0.00704226 -0.00473052\ninternal_weight=334.086 206.965 198.355 194.203 127.121 182.275 177.837 19.255 158.582 8.61001 11.9282 151.398 39.7459 111.653 83.6144 67.6038 11.4009 31.3474 16.0703 15.277 117.915 24.7791 13.7544 93.1362 11.1958 28.0381 64.2586 39.0654 25.1932 20.7282\ninternal_count=246189 80525 77602 74635 165664 71750 54636 30939 23697 2923 2885 15244 12155 3089 1779 1543 14501 10152 2326 7826 160809 152776 143844 8033 6236 1310 1374 858 516 553\nis_linear=0\nshrinkage=0.03\n\n\nTree=225\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 1 3 7 0 13 4 1 19 17 19 5 13 11 17 0 0 13 0 21 5 7 0 4 19 0 19 0 0 19\nsplit_gain=3.69357 1.88719 2.64459 3.64129 2.40165 2.09247 2.01739 2.09986 2.75043 1.82265 1.71922 1.69 1.63889 1.56881 1.53519 2.24229 2.52541 1.69537 1.62199 1.64432 1.44835 4.7468 7.10676 3.95228 2.64712 2.49044 2.33963 3.01608 5.00997 2.94874\nthreshold=0.53650715206675215 2.9477339485081675 0.62297257604742995 0.89599609375000011 1.5447487307370646 -0.46491054376387103 6.1948242187500009 1.6109634896136267 264.50000000000006 3.325091732892195 262.50000000000006 1.7006835937500002 -0.54808102951419191 0.74021522701736098 1.1710359837153288 0.92717806344988862 0.59092606789592239 -0.2310853419800605 1.3771184689138856 1.5000000000000002 1.2426757812500002 0.89599609375000011 1.8987378677022575 1.0693359375000002 240.50000000000003 0.70465417701170696 240.50000000000003 0.49000999047351379 1.1210156985015984 226.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=9 2 4 13 6 -6 7 -2 -9 10 20 12 -12 -4 15 16 -3 -18 19 -16 21 23 25 24 -1 -23 27 29 -29 -22\nright_child=1 14 3 -5 5 -7 -8 8 -10 -11 11 -13 -14 -15 18 -17 17 -19 -20 -21 26 22 -24 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.018174125509596559 0.020183238773856235 0.012739934211017675 -0.028464714010898291 0.0012913152075350069 -0.0066738243639734938 0.032800344187339403 -0.023756629986905448 0.0045744410138360463 -0.026137354705638421 -0.013452072346653252 0.0078312517610677236 -0.024634504461625819 -0.012160912961088403 0.0080619581956660816 -0.0080767101182009741 0.014229327028323623 -0.029537572812532512 0.0046709212129126309 -0.0099598154404937286 0.016028731142795696 -0.014368128299334202 -0.0071703430142921699 -0.026912286384454778 -0.018908410617074142 0.0053387871451178242 0.02866127203279464 -0.0057776778624215328 -0.019324465667983206 0.0042093739085719908 0.016600112499590577\nleaf_weight=3.8838138162566329 5.06998386354645 3.9872681618435291 5.9229868966849635 16.336008888545621 1.2204352460976222 3.7866366672678842 1.4379470407875476 15.877928710408016 2.1077006673513088 10.108501348643015 5.5368661946392885 4.033974340425174 7.6559654413210874 0.80059382040053595 3.1203119939309563 38.058495465375017 2.4390021117287688 1.7140210994984952 6.720228464313549 6.4331615444098134 2.4988451616727643 1.5956730792265741 2.0227567185938815 8.340085090630053 36.580122589375037 12.754366942102932 67.173629910046088 10.303829029960751 28.23040612135901 16.259524506504022\nleaf_count=9647 980 87 987 626 49 33 167 952 155 68905 3438 7934 7484 9 78 311 71 28 80 133 4962 2993 21 20565 30670 727 65660 3892 1536 13012\ninternal_value=0.000693516 0.0050157 0.000795381 -0.00665673 0.0066477 0.0236695 0.00237607 0.00492575 -0.000211738 -0.00161042 -0.000969858 -0.00977034 -0.00380985 -0.0236347 0.00849792 0.0112378 -0.00277511 -0.0172526 0.000519491 0.00813624 -0.000123384 0.00346834 0.0169378 -0.00129975 0.00269809 0.0244694 -0.00201684 0.00241608 -0.00241307 0.0119564\ninternal_weight=332.011 115.033 52.5602 23.0596 29.5006 5.00707 24.4936 23.0556 17.9856 216.978 206.87 17.2268 13.1928 6.72358 62.4725 46.1988 8.14029 4.15302 16.2737 9.55347 189.643 65.1768 16.3728 48.804 40.4639 14.35 124.466 57.2926 38.5342 18.7584\ninternal_count=246192 4746 3958 1622 2336 82 2254 2087 1107 241446 172541 18856 10922 996 788 497 186 99 291 211 153685 64623 3741 60882 40317 3720 89062 23402 5428 17974\nis_linear=0\nshrinkage=0.03\n\n\nTree=226\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 18 1 18 12 0 4 18 3 7 18 11 7 11 4 1 21 7 3 0 1 2 17 18 2 2 19 17 4\nsplit_gain=3.02486 2.19109 2.95214 4.06949 3.23375 2.37339 3.1165 2.92121 2.27986 3.02569 2.29417 2.26024 2.04855 1.94405 1.78927 1.81338 1.62164 1.87535 1.70926 1.70058 1.58747 2.7924 2.316 4.09196 2.7319 1.98795 2.40863 1.55091 1.57093 1.52529\nthreshold=-55.708036422729485 2.1468275719512682 5.0005792898547883 -0.041910702480567281 5.2600927804663344 0.21199991874952331 0.52710038286427008 7.0112304687500009 3.4498615536424366 0.048385965359893195 0.89599609375000011 2.4414009612931857 0.91125284794847328 3.2382812500000004 0.66750229771629133 4.2542884349823007 1.7622051334857016 5.5000000000000009 3.1132812500000004 0.33966381556979741 1.5447487307370646 2.6981674883069502 22.129880905151371 0.86664248603069383 0.89587973461402759 -49.116760253906243 -42.103004455566399 261.50000000000006 1.7509292311006355 7.2512207031250009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 5 -4 -5 6 7 -2 9 -8 -10 27 -7 14 15 -6 18 -18 19 -12 25 -22 23 -23 -25 -3 -27 28 -11 -9\nright_child=1 20 3 4 13 12 8 29 10 11 16 -13 -14 -15 -16 -17 17 -19 -20 -21 21 22 -24 24 -26 26 -28 -29 -30 -31\nleaf_value=-0.019724331603505913 -0.02567032993137729 0.021328295231631423 -0.020311167579280056 0.024538301579973731 0.0031243278743436939 -0.019904965345158704 0.017546231095767112 0.023403547304839248 -0.0099055255727471016 0.0040839065715394338 0.0074735184048552285 -0.013916862930867744 0.0080476253268619569 0.014159454373437415 0.014988059815198386 -0.028035415865147732 0.0009461319669423219 0.036179676518624131 0.016569990154283034 -0.023897580258572675 0.015914401710521817 0.0049223098956960518 0.011449849586284016 0.0053338654145893699 -0.026467892364868312 -0.021315592197632887 0.00072914035680721949 -0.016462682866231221 -0.013584865553422337 -0.008596510253375475\nleaf_weight=5.7476508304359859 8.0673824479237037 3.029078897459839 3.3021789954455043 8.8605647490287556 2.5327397134337826 12.578823373680278 4.7065120867327996 1.2407626647691352 3.9863899567774324 27.804471910572829 2.6684728073685147 17.725473842979227 2.2835957043571389 4.9669741971956682 1.4248940610559655 3.1185665515199616 2.0286626436936777 3.4910636355634779 4.3968640320581471 1.8428744378106783 18.393654468694876 22.59616575401742 15.7819667483418 3.0680947331711632 7.0201080983970314 3.6315341505951428 119.94356968719967 4.0813851549846722 4.381624705480589 2.4563357560436998\nleaf_count=55606 67357 250 34692 8723 7998 1002 3084 51 24428 4387 5508 15806 58 849 48 5254 162 36 173 831 239 183 237 34 76 361 5756 751 1980 387\ninternal_value=0.00015546 0.000571036 -0.00246339 0.00678351 0.0117956 -0.00466515 -0.00272335 -0.0164545 -0.000444693 -0.0033434 0.0084918 -0.00545028 -0.0153735 0.00125777 -0.00843018 -0.0157004 0.0138875 0.0253616 0.00493576 -0.00735236 0.00258019 0.00628294 0.00237984 -0.0022096 -0.0171873 0.000590845 -8.46e-05 -0.0010499 0.00134171 0.00483902\ninternal_weight=327.158 321.411 127.947 24.2059 20.9037 103.741 88.8783 11.7645 77.1138 58.6995 18.4143 53.993 14.8624 12.0432 7.0762 5.65131 14.4279 5.51973 8.90821 4.51135 193.464 66.86 48.4663 32.6844 10.0882 126.604 123.575 36.2675 32.1861 3.6971\ninternal_count=246307 190701 183565 57564 22872 126001 124941 67795 57146 26008 31138 22924 1060 14149 13300 13252 6710 198 6512 6339 7136 769 530 293 110 6367 6117 7118 6367 438\nis_linear=0\nshrinkage=0.03\n\n\nTree=227\nnum_leaves=31\nnum_cat=0\nsplit_feature=10 2 13 17 22 1 0 1 0 17 14 7 7 4 4 19 7 14 17 14 7 5 14 4 19 21 19 2 4 17\nsplit_gain=2.76393 2.19624 1.86914 1.90078 1.77704 1.64467 4.77509 2.95298 1.62338 1.60826 1.39343 1.42128 1.53674 1.37209 5.89502 3.72014 2.99267 3.02924 2.97623 2.5702 3.35719 2.43003 4.25276 2.86064 2.21657 2.16181 2.11351 3.95172 2.30536 2.04872\nthreshold=3.6242520361853052 -69.934642791748033 -0.49896153351186062 0.17341418658717714 0.93750000000000011 1.075454090076706 0.69651904584641833 1.4475910192202701 1.2225960941547209 5.817026225953116 0.91011757189720621 3.8603515625000004 1.0000000180025095e-35 2.6030273437500004 2.3505859375000004 246.50000000000003 1.2426757812500002 0.056363018198629884 1.9205778057410925 0.5668027336126189 0.89599609375000011 0.89599609375000011 0.043928974430191954 2.1750488281250004 238.50000000000003 7.5000000000000009 259.50000000000006 -5.1300392150878897 1.7380371093750002 0.56338221076126505\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 4 -3 10 -7 -8 -6 13 11 12 -5 14 18 -16 17 -15 19 20 21 -4 -23 -24 -19 -21 27 -22 -29 -25\nright_child=-2 2 9 5 8 6 7 -9 -10 -11 -12 -13 -14 16 15 -17 -18 24 -20 25 26 22 23 29 -26 -27 -28 28 -30 -31\nleaf_value=-0.023087902593411473 -0.03065567251080176 0.007521504645493695 0.0038782019391242666 -0.012400842669451597 -0.029996081179782669 -0.023558788662590886 -0.025591558735460363 0.0083304272824593073 -0.0022177549702742778 -0.018423916101274241 -0.01058758609887743 -0.0049112011162895522 0.022314651020598198 -0.015695995690715461 0.0028325476543033926 -0.034144006155360472 -0.00013178170125799281 0.029046210810659334 -0.018596985467498019 -0.0035155636576300652 -0.0073762499174750869 0.018626391364901008 -0.018383215285542154 0.0243855287569372 0.0077242573980307481 -0.029114806378697896 0.023361700903338809 0.02712411162794133 0.0046476389189033995 -0.017609241437193449\nleaf_weight=2.4321126454143487 1.5354228386145212 4.6588347945098558 36.825293567544577 0.62425381532421553 2.5067333983347453 4.1651973772372903 1.4697601994257627 47.084349670612937 4.4982991547003621 3.3241370603114992 1.1678322726206705 1.9685705229157964 13.554752789048051 2.2633617355238611 3.3602021302751819 6.7958367069436463 72.54310996807169 6.782800481506686 6.3655301861131184 29.527804730310894 10.911419355897353 3.5085530000515091 16.791219514548178 2.0177857021656105 12.545576191176449 2.3596812081348153 8.0463999992884947 6.9655118634518685 9.5373145510757222 0.74079209428055026\nleaf_count=38922 213 3573 17840 5770 3453 3379 627 1359 114 28104 10970 243 17211 723 3123 3232 29496 1450 21618 24373 2850 1589 14538 397 6509 1912 728 313 867 492\ninternal_value=0.000549653 0.000790409 0.00104098 0.00493314 -0.00590808 0.00679644 0.0040148 0.00680787 -0.0146635 -0.00029222 0.0145839 0.0169157 0.02007 3.72068e-05 -0.0018023 -0.0227528 0.00283573 0.0123704 -5.24391e-05 0.00101315 0.00347771 -0.000877926 -0.00831917 -0.0138261 0.0162674 -0.0062376 0.0106556 0.00623523 0.0151422 0.0114235\ninternal_weight=326.878 325.343 322.911 81.6986 11.6639 70.0347 52.7193 48.5541 7.00503 241.212 17.3154 16.1476 14.179 237.888 143.753 10.156 94.1348 21.5917 133.597 127.232 95.3443 59.8836 23.0584 19.5498 19.3284 31.8875 35.4606 27.4142 16.5028 2.75858\ninternal_count=245988 245775 206853 46699 7140 39559 5365 1986 3567 160154 34194 23224 22981 132050 93872 6355 38178 8682 87517 65899 39614 34856 17016 15427 7959 26285 4758 4030 1180 889\nis_linear=0\nshrinkage=0.03\n\n\nTree=228\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 21 4 3 19 17 16 5 1 4 1 0 18 5 7 18 3 17 17 18 0 0 19 3 3 0 4 18 19 1\nsplit_gain=2.33662 2.86813 5.43115 2.2502 2.42496 2.1173 2.36297 3.66673 1.86288 2.42051 1.71139 2.85484 4.38315 4.03654 1.94479 3.32139 1.7837 3.9005 3.1509 1.69163 2.02692 2.95026 2.62766 1.74836 2.91382 1.64886 1.86006 1.64657 1.62341 1.65617\nthreshold=0.47260169401963864 7.5000000000000009 2.6340332031250004 0.45898141303402712 261.50000000000006 2.8667313866239854 1.0000000180025095e-35 5.4785156250000009 0.29645739473604366 1.0693359375000002 2.6981674883069502 0.85792061509615258 4.7317931335534196 2.0131835937500004 0.89599609375000011 2.249904835165133 0.50207005315878861 0.2328254379290535 1.4392430184994598 4.4829865921051741 0.97942481616843191 1.1495945197163422 250.50000000000003 0.50207005315878861 0.48555671035752257 1.6765212893675572 8.3627929687500018 0.89587973461402759 261.50000000000006 0.45969269984116679\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 10 -3 5 -5 6 8 -8 9 -1 11 12 16 -14 15 27 17 -2 -18 20 -12 22 -22 24 -23 26 -4 -13 29 -10\nright_child=1 2 25 4 -6 -7 7 -9 28 -11 19 14 13 -15 -16 -17 18 -19 -20 -21 21 23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=0.0037759893465413967 0.021708143252524282 -0.027717521309735524 -0.021368679510777561 -0.025607512624147722 0.0093043022664764299 -0.012924020321008091 0.011958828514655169 -0.016485824650474802 0.019006025560536651 -0.026548729629692038 0.015826149498556923 -0.022249591789093109 0.020378201790600587 -0.025451926120729339 0.0089240230905873949 -0.030507156758041203 -0.03586508814752197 -0.027643566918906765 0.00042742608922314104 -0.010293496582674679 0.0035709584878438694 0.015385846875858011 -0.030343799697981337 0.0088901524342410079 -0.024570450218252365 0.022699716888378844 0.013267990806961743 0.0066868929252907362 -0.012716242981663465 -0.00057381850687804293\nleaf_weight=3.5612054748744395 2.1794031193039731 6.452456430270102 1.7643677886626394 5.7506048477280265 1.8490773965932019 11.759943571862662 23.980363586984598 3.9705850045488651 3.0596579902046406 4.9665345535375991 20.391579961244133 1.0696241580590129 5.7413104890147215 1.3947350910839302 30.124094999002409 2.3719407900189244 6.1997494402476132 1.6372608324982136 3.038183116480468 4.0913350854098089 4.0658402246772303 1.9519793489016581 2.809265001793392 52.335223758651409 2.9131784802302718 2.8277405090630054 1.875165818390087 11.649162207053449 8.5611207724050455 91.063796056233514\nleaf_count=27119 119 544 519 748 169 74728 6624 487 7718 77526 511 38 1036 739 534 503 1250 647 744 257 74 26 49 486 43 30 77 321 6456 35778\ninternal_value=0.000514786 0.00298384 -0.0101006 -0.0020869 -0.0170189 -0.00122793 -0.000154396 0.00748824 -0.00212483 -0.0148179 0.00413991 0.000443733 -0.00868085 0.0093934 0.00461798 -0.00399814 -0.0185262 -0.000806442 -0.0250533 0.00683431 0.00777466 0.0050085 -0.0123805 0.00727535 -0.00864936 0.00883393 -0.00450977 0.00280894 -0.000937955 0.000255773\ninternal_weight=325.406 166.884 12.9197 158.523 7.59968 150.923 139.163 27.9509 111.212 8.52774 153.964 65.4055 20.1906 7.13605 45.2148 15.0907 13.0546 3.81666 9.23793 88.5584 84.4671 64.0755 6.87511 57.2004 4.86516 6.46727 3.63953 12.7188 102.685 94.1235\ninternal_count=245900 8547 1170 237353 917 236436 161708 7111 154597 104645 7377 5931 4535 1775 1396 862 2760 766 1994 1446 1189 678 123 555 69 626 596 359 49952 43496\nis_linear=0\nshrinkage=0.03\n\n\nTree=229\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 18 18 1 18 9 13 7 13 5 2 2 11 7 0 13 18 5 3 21 3 0 11 7 2 7 2 18 0 18\nsplit_gain=2.80854 3.57243 4.79737 3.24901 3.90517 2.67222 4.20423 2.41551 2.73977 4.02564 3.33731 2.79571 2.71535 2.37451 2.30887 2.23997 3.2774 2.22462 5.61569 3.98586 2.27616 2.26275 2.21117 2.00975 1.99487 1.91354 3.35514 3.13016 3.41654 2.63711\nthreshold=0.69651904584641833 4.8713658868283671 5.2755571088971331 1.9634439907166827 4.0689902272260703 1.0000000180025095e-35 0.14796868535883259 0.89599609375000011 -0.50497683684085448 0.89599609375000011 -30.838275909423825 11.076014518737795 0.63309421653839248 4.0683593750000009 0.48517496193768689 -0.64803833042221404 5.2067464199591091 1.2426757812500002 0.44825707527455844 4.5000000000000009 0.62297257604742995 1.3170161875856263 0.91125284794847328 3.4804687500000004 -10.210010528564451 1.2426757812500002 6.3334450721740732 3.4498615536424366 0.91093860112668257 2.740319461670996\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 15 13 -5 6 23 8 11 10 -10 17 25 -1 -6 -3 -17 18 -2 -20 21 -21 -14 -4 -18 -9 27 28 -27 -30\nright_child=7 2 5 4 14 -7 -8 12 9 -11 -12 -13 22 -15 -16 16 24 -19 19 20 -22 -23 -24 -25 -26 26 -28 -29 29 -31\nleaf_value=-0.026684641338419742 -0.032273482292348082 0.028736926954982155 -0.030935594040855368 -0.022224145873137875 -0.0088029819521607093 0.0057576803145004182 0.014341435550240015 0.0078943005235456339 -0.024656407501379936 -0.015408559024649765 0.0020582642859461333 -0.016029718688521349 -0.016675590380361677 0.0067758104259432496 0.013397405322377773 -0.0058818503734848794 0.02972358506310209 0.023165584770000303 0.02320276166423518 -0.021334288156150853 -0.027373796545602729 0.015053980274009941 0.0084550721231248724 0.0061502629571138483 -0.014452191742221497 -0.039004234843408127 0.0050846047207819215 0.012889749787237821 -0.015427889517645357 0.012000535269273646\nleaf_weight=13.263779149055379 2.3891874704904792 4.1527501748976219 8.1672556250866837 6.5154161651717031 6.5046970628700356 6.1134065991916478 2.2812262646039008 70.432952316145247 3.66597215911679 24.799327509577552 41.982857367416955 3.8822305400094583 9.3683448766387283 1.7926964152284188 8.7817458145309484 8.7732888171376526 4.0258306191510655 6.0956605463225051 8.3291614193185506 1.2112303972025973 1.8936577397398648 4.2242078893177677 3.7980105277965768 1.1879353656386218 0.49522219912375831 3.1415314601563322 47.685379650383766 3.7200312376603497 8.637908860318019 3.8158534047461208\nleaf_count=87038 2800 5126 68553 867 1604 4995 70 3940 1643 10668 10944 2009 285 128 729 31067 3275 1118 263 121 52 23 92 1941 1563 720 2806 691 710 415\ninternal_value=-0.000678662 -0.00585169 0.00100124 -0.0122493 -0.00447496 -0.00971337 -0.0176538 0.000835276 -0.00280126 -0.00593586 -0.000569704 0.00501133 0.00321591 -0.0225268 0.00399021 0.0118374 0.00491743 0.00889772 0.00311544 0.0101284 -0.00573175 0.00489012 -0.0093412 -0.0260872 0.0231435 0.00447768 0.000822769 -0.00943116 -0.0152114 -0.00675646\ninternal_weight=321.129 72.0553 35.1969 36.8583 21.8019 17.7498 11.6364 249.074 98.4735 70.4482 45.6488 28.0253 150.6 15.0565 15.2864 17.4471 13.2943 24.1431 18.0474 15.6583 7.3291 5.43544 13.1664 9.35519 4.52105 137.434 67.0007 19.3153 15.5953 12.4538\ninternal_count=246256 206956 116590 90366 3200 75559 70564 39300 29641 23255 12587 6386 9659 87166 2333 41031 35905 4377 3259 459 196 144 377 70494 4838 9282 5342 2536 1845 1125\nis_linear=0\nshrinkage=0.03\n\n\nTree=230\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 4 11 2 4 14 2 2 9 7 2 0 2 18 5 18 14 14 2 3 2 18 18 3 18 1 7 7 4 3\nsplit_gain=3.64872 2.64526 3.62355 3.24588 2.71216 2.63696 2.51988 3.26448 2.22823 2.51841 3.39533 3.16384 5.05726 3.3405 3.24863 2.56827 2.69543 2.529 4.17796 2.42974 2.31177 3.50887 2.20653 2.06608 2.05601 2.02289 1.7949 1.78168 3.00892 1.76781\nthreshold=2.1810207941665634 1.6286621093750002 5.5511151231257837e-16 10.479034423828127 1.0693359375000002 0.5668027336126189 29.286581993103031 -37.900318145751946 1.0000000180025095e-35 1.2426757812500002 -10.988907814025877 0.46314780808741635 9.3619499206542987 0.89587973461402759 3.7011718750000004 0.89587973461402759 0.13828491300858028 0.90851014093827787 -14.86118173599243 0.19435047505660144 12.256900787353517 1.4978661367769956 3.1132683346437333 0.37248792739977959 2.8617925509761908 2.6981674883069502 2.0131835937500004 2.6738281250000004 2.3505859375000004 0.58795853567030554\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 4 8 22 -1 -2 7 -6 9 10 23 14 13 -13 27 16 -14 18 24 -16 -17 -22 -4 -3 -15 -23 -10 28 -11 -18\nright_child=5 2 3 -5 6 -7 -8 -9 26 11 -12 12 15 17 19 20 29 -19 -20 -21 21 25 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.0010335961069781287 -0.01728472941438168 -0.018707354737494804 -0.029197096609429853 0.0031660788123965282 0.0068609765056077401 -0.00042241503864581185 0.0040213789612058917 -0.028271791195804568 0.027655646507700828 0.017458154645753586 0.0097713896057407667 0.014948431648915565 0.013591758295040451 -0.023733149179724168 -0.01204822091939668 0.024843770015108391 -0.038104651026643725 0.0086436677678903909 -0.037378914240578272 0.02815879323576417 0.017500459395831652 -0.016189417852551985 -0.0049794002865470371 0.0013127175259167276 0.010931888904575218 0.0077823538024436529 0.0054690884078984056 0.017538746892126342 -0.023470677850862907 0.0035896506304604748\nleaf_weight=70.932813710221581 17.78348360146083 4.5111444051549361 6.2326590115317249 12.736410830923885 2.6280191930184182 15.113914926208961 4.6940789100249267 9.6536829539830933 5.2420174199482963 1.4761541618695426 67.292709948561196 1.7507095795735974 1.3632618897827367 1.4625625830922206 1.0500574534626137 3.9167440615369769 2.4414770878502172 1.6134172074102924 9.1280714092190447 6.7554103285244764 6.7955526264268 7.895777485707411 7.3105153683611661 28.673353581687479 3.1378261333798037 4.1711021938244812 8.2779310282269289 1.5417145190751851 4.3450501608510876 1.028267096262425\nleaf_count=42570 45274 24811 277 498 5235 42103 2192 19488 499 1386 13765 22 12 185 5078 187 63 681 1436 1921 124 1749 4939 1915 1560 77 2017 3748 22778 5\ninternal_value=-0.000468171 0.000625285 0.00252798 -0.00766373 -0.00368646 -0.00977875 -0.0140972 -0.0208036 0.00410342 0.00306288 0.00594388 -0.00179014 -0.00579831 -0.0181333 0.00965087 0.00220422 -0.0157268 -0.0225921 -0.0266006 0.0219958 0.00650999 0.00164389 -0.0173658 -0.00187647 -0.00235878 -0.00794177 0.0153836 -0.00450134 -0.012054 -0.0277098\ninternal_weight=320.956 288.058 200.15 26.2796 87.9086 32.8974 16.9758 12.2817 173.87 160.35 100.477 59.8732 44.7048 17.0926 15.1684 27.6122 4.83301 15.3419 13.7285 7.80547 22.7792 18.8624 13.5432 33.1845 4.60039 12.0669 13.5199 7.36292 5.8212 3.46974\ninternal_count=246595 159218 89733 5714 69485 87377 26915 24723 84019 81503 40491 41012 6101 3884 34911 2217 80 3862 3181 6999 2137 1950 5216 26726 1745 1826 2516 27912 24164 68\nis_linear=0\nshrinkage=0.03\n\n\nTree=231\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 12 5 0 0 4 21 13 12 22 3 19 17 19 7 4 4 7 7 5 4 19 19 5 18 19 21 0 18 4\nsplit_gain=2.16109 2.44677 2.41512 3.12798 5.19472 2.47443 2.14348 2.0359 1.789 2.86194 1.74281 1.95456 2.00861 1.67057 1.82258 1.89757 1.61186 2.5128 2.30014 2.51755 2.10308 3.14446 2.05718 2.59943 2.55888 4.03346 2.45987 4.93742 3.01925 2.30776\nthreshold=0.36709515926886871 0.28848765710462548 2.3505859375000004 0.84505259813102007 0.87173191741369926 1.0693359375000002 1.5000000000000002 -0.0029195251563053355 0.88192242588002812 0.93750000000000011 0.011671989023333709 251.50000000000003 0.59049165877606857 239.50000000000003 2.5244140625000004 5.7263183593750009 1.6286621093750002 1.0000000180025095e-35 0.89599609375000011 1.8691406250000002 1.0693359375000002 264.50000000000006 233.50000000000003 4.3378906250000009 0.89587973461402759 226.50000000000003 2.5000000000000004 0.97942481616843191 3.0677824455408698 1.8393554687500002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 16 3 -1 -5 6 -6 13 -3 -10 -7 -12 -13 -4 -15 -16 20 -18 19 -19 -2 -22 24 26 -20 -26 27 -24 -28 -29\nright_child=1 8 7 4 5 10 -8 -9 9 -11 11 12 -14 14 15 -17 17 18 22 -21 21 -23 23 -25 25 -27 28 29 -30 -31\nleaf_value=-0.019692797381563078 0.001634863018677993 -0.011669985363052174 0.010029890798538499 0.033936882831440501 0.015839907711484663 0.015571113739041844 -0.0076177135999526621 -0.0080565063063408587 0.016851606619153491 -0.022554895898452214 -0.020982519195011071 -0.022334240642400256 0.012060956804301581 -0.017696149461080609 0.019873748210214232 -0.010090192987547153 0.024797062800442572 -0.014819707927071723 0.018689682187821358 0.0093677087927213215 -0.019115468231852964 0.013759111634945186 0.0087236306399398108 0.022068529881507086 0.0017025586969145858 -0.031886129635138885 0.010834253576454948 0.014807327301554832 -0.0034981415101054585 -0.023736688860899267\nleaf_weight=14.048030728731304 50.721560257417877 18.108999719332136 21.478070129674283 2.128226679236831 5.7412227693221229 0.9977281090355089 6.4128728286309524 13.490727565769246 4.7910809639142817 1.3670177962630985 13.258960773287074 1.2833094792995314 3.6240721920835313 4.0920489530697051 3.0141016803509046 2.7522395836450642 4.3713453162536089 10.378338978006466 2.4265178109053513 5.0453869768143695 9.8813038436087464 2.6542723556849515 12.289380951657224 8.12093600976368 8.7230953676480549 4.0126926971388448 59.069918038680044 0.94599818298593064 16.436619368171705 8.5414757375619939\nleaf_count=138796 5220 1146 12085 1579 2400 440 5259 7282 123 27 6399 734 1275 27466 1909 20650 468 1509 22 1097 3208 429 1381 269 1375 483 649 9 2147 250\ninternal_value=0.000176532 0.00174481 -0.00367181 -0.00831377 -0.00310145 -0.0065906 0.00382476 0.00132045 -0.00707642 0.00617499 -0.0130582 -0.015361 0.000808269 0.00548166 -0.00443852 0.00619455 0.00282556 0.00460752 0.00378226 -0.00681893 -0.00113861 -0.0116524 0.00518338 0.0066667 -0.00490281 -0.0104312 0.00516955 -0.00358857 0.00770937 -0.018848\ninternal_weight=320.208 227.886 92.3216 47.4944 33.4464 31.3182 12.1541 44.8272 24.2671 6.1581 19.1641 18.1663 4.90738 31.3365 9.85839 5.76634 203.619 140.362 135.99 15.4237 63.2571 12.5356 120.567 105.404 15.1623 12.7358 97.2834 21.7769 75.5065 9.48747\ninternal_count=246086 19812 226274 156882 18086 16507 7659 69392 1296 150 8848 8408 2009 62110 50025 22559 18516 9659 9191 2606 8857 3637 6585 4705 1880 1858 4436 1640 2796 259\nis_linear=0\nshrinkage=0.03\n\n\nTree=232\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 0 4 2 18 3 4 19 0 6 19 4 4 1 2 2 17 5 4 0 3 18 1 17 3 2 2 5 5\nsplit_gain=2.95836 3.38416 1.94918 1.77824 1.95627 1.82796 3.78392 2.79682 2.98544 2.56439 2.35479 2.8982 2.27877 2.66734 2.23061 2.33162 3.37025 3.62191 2.62871 2.7854 3.81954 2.85214 2.41281 2.36719 2.14707 2.1223 2.09616 3.60595 2.03619 3.47633\nthreshold=-34.252307891845696 3.3033179733375269 0.96046990436194013 0.84667968750000011 -10.988907814025877 2.6735537653587351 0.47260169401963864 1.0693359375000002 253.50000000000003 1.1851094193663554 1.0000000180025095e-35 234.50000000000003 5.5515136718750009 5.9891631007194528 0.85642036656791032 25.282213211059574 9.8937015533447283 0.17669220786083642 2.1386718750000004 3.1782226562500004 0.77696153631098397 0.50207005315878861 3.8815106545092593 4.0016611463566001 0.058793162820948631 0.65908129556435835 -20.386926651000973 -14.08206272125244 4.2695312500000009 4.6953125000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 -3 4 -2 6 9 8 -7 -5 11 -8 14 -14 -9 16 17 -16 19 20 22 -22 -19 -20 -11 -10 -13 -28 -15 -30\nright_child=3 2 -4 5 -6 7 10 12 25 24 -12 26 13 28 15 -17 -18 18 23 -21 21 -23 -24 -25 -26 -27 27 -29 29 -31\nleaf_value=-0.020536889958949766 0.0055492373766206794 -0.013027276419046064 0.012717308545871723 -0.018703554152890101 -0.022619272611560198 0.00063227592418145322 -0.022318310804076646 -0.017519586367609235 0.027193776024066647 -0.023766721994262281 0.016004581085863719 0.012946328946567921 0.023148392077608253 0.012030556870796495 -0.027104208754988358 -0.017264725993511757 0.014424891823199611 -0.027747440501447656 -0.020737557098037929 0.033178483875625372 0.022744316506810486 -0.0078888016446418581 -0.00012271113944114422 0.01476801040517289 -0.0018702619415800633 -0.014674647509802101 -0.022792587075367125 0.0024208785243300278 -0.021923151628494276 0.004702320171078158\nleaf_weight=11.889580931579305 3.0340160392495363 3.344888906900449 5.8537952605402097 10.998800805481549 4.8865016512083903 8.6768100110141422 3.4227789727301561 6.1546015383798922 10.237211747853959 3.3554124292058978 10.244261368177829 13.526300127326975 6.7983531475069858 13.999104593621722 3.7043834249002421 5.1001229133587449 13.832935177655825 3.3098211825477 6.3017686618866255 1.9609968814652337 8.0868058247115169 3.3166748709627418 8.4018622563405625 1.4161188736325128 57.700948585308652 0.5855951136909433 4.5655354550108234 60.721452649915598 4.9877537738261735 15.894783606290273\nleaf_count=94505 11003 286 56 3384 22075 10722 155 70223 1884 202 111 111 1837 4717 1654 2332 2583 863 2473 161 777 112 4904 20 4359 36 77 851 1403 2513\ninternal_value=-9.47799e-06 -0.0106007 0.00299644 0.00078031 -0.0124166 0.00118594 -0.00088102 0.00394089 0.0140701 -0.00599164 0.00312237 0.00134462 0.0019116 0.00724323 -0.0017548 0.000275012 0.00235404 -0.0024835 0.00101738 0.00575702 0.00215133 0.013917 -0.00949837 -0.0132761 -0.0034372 0.0238788 0.00263903 0.000340447 0.00358461 -0.00236836\ninternal_weight=316.31 21.0883 9.19868 295.222 7.92052 287.301 164.535 122.766 19.4996 72.0552 92.4803 82.2361 103.266 41.68 61.5861 55.4315 50.3314 36.4984 32.794 25.0762 23.1152 11.4035 11.7117 7.71789 61.0564 10.8228 78.8133 65.287 34.8816 20.8825\ninternal_count=246389 94847 342 151542 33078 118464 9250 109214 12642 7945 1305 1194 96572 10470 86102 15879 13547 10964 9310 6817 6656 889 5767 2493 4561 1920 1039 928 8633 3916\nis_linear=0\nshrinkage=0.03\n\n\nTree=233\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 17 5 2 14 18 12 0 11 18 4 13 2 2 17 13 7 18 1 1 1 2 13 14 18 14 2 2 4 5\nsplit_gain=3.75613 1.88472 1.9315 4.83555 3.96018 4.10812 3.95029 4.29864 3.98443 3.56558 3.34168 3.01406 3.01405 4.06432 3.23181 2.98957 4.35076 3.82248 3.00125 2.57235 2.43742 2.37564 2.36119 2.23135 1.94192 1.84116 1.81801 1.99497 1.73931 1.90531\nthreshold=3.3033179733375269 3.9653415636860969 1.7006835937500002 -32.965999603271477 0.33900667957710101 1.4978661367769956 8.8817841970012543e-16 1.6765212893675572 0.91125284794847328 4.7317931335534196 1.6572265625000002 -0.54808102951419191 -1.3165187835693357 -2.5585746765136714 0.85615492898235035 -0.57929579963106315 1.0000000180025095e-35 1.2424533248940002 2.1468275719512682 1.4866206336930319 1.8912989126705502 16.375147819519047 1.0000000180025095e-35 0.050734622491290134 0.89587973461402759 0.84355129891941705 -23.956354141235348 -0.44610977172851557 3.9514160156250004 2.3505859375000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 4 -4 5 21 7 24 -8 12 19 -9 13 15 22 20 18 -18 -17 23 -6 -1 -14 -7 -5 -22 -21 -28 29 -2\nright_child=28 -3 3 6 9 10 8 11 -10 -11 -12 -13 14 -15 -16 16 17 -19 -20 26 25 -23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.028232804325855169 0.0040548115032177125 -0.017068683980739137 -0.02829879932894782 -0.020867586071076583 -0.020969291966483457 0.013959492567817601 -0.01763346581670627 -0.011305819197554218 0.014781924307700001 0.0094970267036280104 -0.0089944362792498626 0.022513137962683314 -0.033263906993555137 0.023255325788399332 -0.0061669093512416971 -0.020256377520211873 0.0044565639416555801 -0.029695437757662854 0.024395522057376877 -0.0088957819094324549 0.018630340960327143 0.0034176480176580028 0.0060371835286571083 -0.021435654288112276 0.0024869543538196385 -0.010699753969620938 0.034163015169202643 0.0064823556005335408 0.015152361534536795 -0.020507074453066456\nleaf_weight=3.7538020809479375 84.189255712568411 6.4244642438674182 3.8920488422874078 2.3491899558212017 1.7607941686746915 2.505615309721299 13.923414909724444 2.1610197557893143 3.5947842382593072 7.417834919685105 5.4730823394654626 14.830289351084501 11.591208333923307 2.9720205400587956 10.008887553677821 1.1714544282676822 3.4772420704321112 12.390150579033842 2.9250536927575013 1.0049747106422717 6.1814988191490539 3.522764688328607 1.2308750911615778 1.9700426452404824 75.374362070052257 1.8902751598943721 7.840602474012484 3.9420410004049691 11.913344798784236 1.9592414970975358\nleaf_count=160 1664 61443 41811 73 5928 4282 2054 73 91 25505 4384 274 10327 676 9857 10161 75 11336 351 514 187 224 35 14112 38981 210 450 675 232 74\ninternal_value=1.06805e-05 -0.00220264 -0.00166697 0.000913815 -0.00486389 0.00401497 0.00216783 0.00460432 -0.0105427 -0.00908925 0.0102489 0.0178226 -0.0116929 -0.00558385 -0.0198588 -0.00912211 -0.015546 -0.0223907 0.0101575 0.0165082 0.00415198 -0.0143485 -0.0293273 -0.00268976 0.00152496 0.0113394 0.0229352 0.0261337 0.00484958 0.00326731\ninternal_weight=313.642 215.58 209.155 116.125 93.0302 30.0129 112.233 94.7149 17.5182 63.0173 22.7364 16.9913 55.5995 32.7685 22.831 29.7965 19.9639 15.8674 4.09651 17.2633 9.83257 7.27657 12.8221 4.47566 77.7236 8.07177 12.7876 11.7826 98.0618 86.1485\ninternal_count=246219 244249 182806 83357 99449 24801 41546 39401 2145 74648 24417 347 49143 28924 20219 28248 21923 11411 10512 20033 6325 384 10362 18394 39054 397 1639 1125 1970 1738\nis_linear=0\nshrinkage=0.03\n\n\nTree=234\nnum_leaves=31\nnum_cat=0\nsplit_feature=4 1 5 2 5 17 11 2 2 2 2 19 2 18 1 18 3 7 17 7 13 17 19 18 1 17 11 22 17 13\nsplit_gain=2.52103 3.03029 2.88411 2.65756 3.02304 2.3596 2.28455 2.2774 1.9051 2.47568 3.33007 4.27653 2.24223 2.67645 1.91588 1.89241 1.87934 2.14548 2.03314 1.9357 2.11476 1.99255 1.96362 1.88248 1.87133 2.51891 1.81826 1.99254 1.7469 1.70923\nthreshold=3.2580566406250004 2.3713805701640611 2.1386718750000004 -30.420168876647946 2.2500000000000004 1.1976019298503242 0.49644476220754463 32.956295013427741 34.417465209960945 21.425971984863285 2.9882335662841801 243.50000000000003 -1.7340650558471677 2.9174053685313028 1.1008244052526537 4.135134555718313 0.42638606281916264 1.2426757812500002 0.21173336644813048 0.89599609375000011 1.0000000180025095e-35 1.1384572374241533 257.50000000000006 3.8815106545092593 1.8223781763416411 0.29595754201755942 0.81454645780468748 0.93750000000000011 0.43370896730050829 -0.14639129621291555\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 5 -2 -5 8 16 -7 9 10 12 -12 14 -14 -1 24 19 -18 -19 20 22 28 -3 -20 -16 -26 -8 -28 29 -21\nright_child=3 6 -4 4 -6 7 26 -9 -10 -11 11 -13 13 -15 15 -17 17 18 23 21 -22 -23 -24 -25 25 -27 27 -29 -30 -31\nleaf_value=-0.019843939055796071 -0.020626727898793679 -0.020998603121514216 -0.028206996135730168 0.019999000816448263 0.0026442272020447011 -0.020253589238714616 -0.02518472087706319 0.0039299932240755191 -0.019026905089652674 0.013605778410401618 0.010569750099629996 -0.031213280250438149 0.022540749116286021 -0.01809159123597193 -0.019546834374054325 0.014819755771593184 0.0046920047853923289 0.014052779382839775 -0.024195540694588652 0.0065043130896159955 0.016653753339072288 -0.021163142655003551 0.0084831330570626466 0.0077358888678768865 -0.017773017016076666 0.013048739110405588 0.012919464403121476 -0.019221608144061365 0.015912237406643267 -0.025402303654676033\nleaf_weight=3.9218548904086665 2.979788126111429 11.065544799962483 4.4749976402363236 9.1180845626527098 89.249216684689372 13.358638112296719 4.4142841603024845 4.2539821558268605 4.2928770961864293 8.6068779998020073 2.6223315056417942 5.9947458477324744 6.0745235149879653 1.0379140573832044 4.0959299587557352 5.2741946403343709 93.542442810123248 1.6987958052195589 5.5338013848522678 3.1978023761112113 1.1719781523279369 1.6538566229428386 1.8765472812556243 1.6896718141433664 2.5963197590390319 6.1714483653995558 3.2424283009022474 1.9383176881819961 7.1787374519917648 1.4387590348487713\nleaf_count=47260 46261 1576 15269 1641 24490 64984 139 11448 3660 4696 4138 6522 403 3768 1910 4088 2123 26 79 146 30 254 117 57 299 420 107 43 168 66\ninternal_value=-0.000365415 -0.00222061 -0.00713882 0.0035067 0.00443799 -0.00537137 0.000375111 -0.0145152 -0.00197487 -2.8951e-05 -0.0034051 -0.0187211 0.00158956 0.0151121 -0.00323661 0.00120356 0.00145583 0.0033271 -0.0100135 -0.00536868 -0.0127554 0.00271832 -0.016422 -0.0166939 -0.00504544 0.00303661 -0.0130275 -0.000270079 0.0074517 -0.0061467\ninternal_weight=313.767 212.42 72.7766 101.347 98.3673 68.3016 139.643 17.6126 50.689 46.3961 37.7893 8.61708 29.1722 7.11244 22.0597 18.1379 130.048 102.465 8.92227 27.5832 14.1141 13.4692 12.9421 7.22347 12.8637 8.76777 9.59503 5.18075 11.8153 4.63656\ninternal_count=246188 173796 168865 72392 26131 153596 4931 76432 77164 73504 68808 10660 58148 4171 53977 6717 4642 2285 162 2357 1723 634 1693 136 2629 719 289 150 380 212\nis_linear=0\nshrinkage=0.03\n\n\nTree=235\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 1 2 7 0 17 7 3 17 12 19 2 18 2 5 14 1 3 3 2 2 3 7 19 17 14 2 18 0 3\nsplit_gain=2.56741 2.58534 2.79321 2.70255 4.43278 3.54266 4.04484 4.23938 3.09477 2.92411 2.90927 2.20443 2.87239 2.44819 2.12197 3.16402 2.11486 2.09904 3.35103 2.00786 2.65677 2.39842 1.92265 1.88506 2.01037 1.72051 1.59834 1.58662 3.43237 1.65068\nthreshold=0.28904431203898151 0.45969269984116679 28.11904335021973 0.89599609375000011 0.29549219484135919 0.6411330003973541 1.0000000180025095e-35 0.58795853567030554 1.3781395569682247 0.28848765710462548 245.50000000000003 11.680273056030275 0.89587973461402759 22.129880905151371 3.1982421875000004 0.29138017841767527 1.3779212206518803 0.40911314460322756 0.43673077644757391 34.417465209960945 53.951957702636726 0.51817819236196982 1.8691406250000002 254.50000000000003 1.2783113269342083 0.29138017841767527 6.8277702331542978 1.7005986908310777 0.72109735700429223 0.45898141303402712\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 -2 3 5 -5 17 7 10 -8 11 -7 -6 -13 14 15 -14 -4 -3 -19 -18 21 -21 -15 24 -23 -20 -10 -27 29 -29\nright_child=1 2 16 4 9 6 8 -9 26 -11 -12 12 13 22 -16 -17 19 18 25 20 -22 23 -24 -25 -26 27 -28 28 -30 -31\nleaf_value=-0.0061503282412371657 0.018133747832542003 -0.015931992484975534 -0.010096235455474717 -0.023017559530737548 0.0065933704056878274 0.030893387443293969 -0.024773177397387852 -0.02819401318986502 0.0057254649124557772 -0.012524641107757232 -0.0066309043198767709 -0.02774019963655832 0.007534379484556643 0.020132075917236172 0.006802152965403532 -0.028942068637723455 0.017798249900435964 0.023805780318760308 -0.019342749648815594 -0.018477144750777625 0.013979396752451774 0.018506729255890083 -0.016006117934361155 -0.02266194608771049 -0.011725052277845311 0.0094689886471540849 -0.02056751971410237 -0.019782500120426569 -0.017214545109765878 0.01379759982325158\nleaf_weight=57.704550415760991 6.5337773483920492 5.1718901348276622 5.0041728078363112 5.2918823901368333 81.813845156033267 3.7254859058489265 18.171439860138893 1.5732442058797458 6.0599967591147088 9.2131144806489811 2.5737282223187776 2.7970433399605126 2.8385592146260041 5.9579680952992922 5.4819373727514176 4.8373440772775211 14.431594903169751 4.8477836918991661 3.0019512068629437 5.5446170173891032 11.690999793017907 5.9194112199038482 0.84916222762331006 0.90510363108478475 1.9881795402325222 12.03972276201239 1.9639634373543229 0.66226775981885921 8.0894323857391992 7.236655647853695\nleaf_count=209070 8563 3354 1724 4056 3800 1099 2983 309 1749 456 3067 34 46 150 187 354 428 496 135 407 371 111 131 31 64 151 623 235 598 1097\ninternal_value=-0.000511836 0.000829164 0.000284055 -0.001291 0.00151981 -0.00570829 -0.0127107 0.00560602 -0.0182192 0.00285153 0.0167538 0.00431197 -0.0038205 0.000693915 -0.00677593 -0.0161394 0.006892 0.000267222 0.00297109 0.00918488 0.0039312 -0.00462842 0.0141512 0.00507972 0.0104427 -0.000925504 -0.00227612 0.00164547 -0.00445647 0.00907551\ninternal_weight=303.921 246.216 239.682 194.198 119.081 75.1176 34.0679 7.87246 26.1954 113.789 6.29921 104.576 22.762 19.965 13.1578 7.6759 45.4841 41.0497 35.8778 40.4799 26.0483 14.3573 6.80713 8.81269 7.90759 31.03 8.02396 28.0281 15.9884 7.89892\ninternal_count=245879 36809 28246 25110 9214 15896 9830 4475 5355 5158 4166 4702 902 868 587 400 3136 6066 2712 1412 984 613 281 206 175 2216 2372 2081 1930 1332\nis_linear=0\nshrinkage=0.03\n\n\nTree=236\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 17 9 13 17 17 2 1 18 13 4 21 6 5 4 18 17 0 13 5 17 17 18 21 2 2 21 2 13 1\nsplit_gain=2.68076 2.02349 2.45854 3.02047 2.59142 2.38446 1.97929 3.43173 2.70763 2.15131 2.18813 2.88846 2.65709 1.68018 1.54237 1.48972 3.29166 2.23292 3.62201 1.87784 1.9854 1.73002 1.58129 1.96197 2.98727 1.93017 1.84695 3.46112 1.90683 2.3611\nthreshold=0.75738199555027408 1.9512128003148403 1.0000000180025095e-35 -0.48625410865680269 1.8807230532061787 0.043095813018339015 20.132114410400394 2.1468275719512682 3.4498615536424366 -0.89390355839688218 1.0693359375000002 1.5000000000000002 3.7958984375000004 4.2695312500000009 12.900723338127138 2.8029010331479989 0.35042734261365599 1.0917760095905165 -0.46714757665816348 1.2426757812500002 0.70050815440916869 0.13249526821743748 2.1383330595080281 2.5000000000000004 8.8705892562866229 2.0782871246337895 7.5000000000000009 -6.3279094696044913 -0.074019523483382962 2.3713805701640611\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 4 5 -1 7 8 9 -2 11 -11 -12 -4 -15 17 -17 18 -9 20 -18 -20 23 24 -19 -24 28 -28 29 -25\nright_child=6 -3 13 -5 -6 -7 -8 15 -10 10 12 -13 -14 14 -16 16 19 22 21 -21 -22 -23 25 26 -26 -27 27 -29 -30 -31\nleaf_value=0.017232696695735999 0.013909620933857026 -0.014346096505397919 0.020718870051866473 -0.0012488476916235817 0.016455362487587111 -0.02919289949209505 0.0058492872511117042 0.010926607711824146 0.011340334824546969 0.020122704809872449 -0.022904537927572095 -0.012809631431093475 0.0067345197648058554 -0.022020568575554959 0.010053360843801524 -0.01142718251206986 0.0011950670876171737 -0.026853477421521048 0.0062679686192351133 0.0032314902014219248 0.031979644935781079 -0.028700488006781967 0.016372104001867498 -0.014342994741136688 0.0067319881162437581 -0.0073547293028830875 0.0099485013010126785 -0.023278278264337916 -0.0097475391148337098 0.0087273178932639681\nleaf_weight=0.46906274832690176 1.9974438400859096 14.076119148205406 5.5151448664239506 46.043203364323631 0.89519983041077544 7.5842447107690987 76.754156807370009 3.659174858184997 4.7027148973363628 2.8484273789417802 17.368455445775002 5.3565092184119294 2.7352580710430621 1.3176870259037516 2.2991000337465266 4.4493633445581535 2.6320625431253593 4.4042124031911936 1.1722022967296633 6.9309129686480437 6.443871998697432 5.9366882101603551 11.834587396995628 3.4899684243136937 3.6811171000008471 3.4162816771258795 4.2645478672347963 5.4052799342898634 6.378942337469204 43.222976509074215\nleaf_count=755 308 83755 1820 92263 872 32795 10153 85 2801 1988 9829 4994 110 896 278 244 120 130 17 225 216 209 206 53 89 111 38 58 130 386\ninternal_value=0.000317007 -0.00444463 -0.00208419 -0.00451579 -0.0195102 -0.0248841 0.00195468 -3.30306e-05 -0.00814205 -0.0114309 -0.0136325 -0.000432786 -0.0187439 0.0115592 -0.0038706 0.00243505 0.00958933 0.000841664 -0.0114345 0.0157596 0.0240566 -0.0228726 0.00249067 0.000546768 -0.0125046 0.0109316 0.00239787 -0.00906571 0.00461477 0.00673923\ninternal_weight=307.285 78.1998 64.1236 54.9917 8.94851 8.05331 229.085 152.331 35.0088 30.3061 28.3087 8.20494 20.1037 9.13193 3.61679 117.322 20.4562 96.866 10.7681 16.0068 9.07593 7.10889 86.0979 70.847 8.08533 15.2509 62.7617 9.66983 53.0919 46.7129\ninternal_count=245934 213434 129679 126685 34422 33550 32500 22347 20030 17229 16921 6982 9939 2994 1174 2317 805 1512 311 561 336 226 1201 884 219 317 665 96 569 439\nis_linear=0\nshrinkage=0.03\n\n\nTree=237\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 17 1 3 1 4 4 12 10 11 0 4 5 18 5 0 18 4 2 3 3 4 18 21 0 4 14 2 2 3\nsplit_gain=3.69841 1.98172 1.98823 1.96266 1.81487 1.78823 3.09414 1.63808 2.38658 1.97726 2.27595 1.60193 2.12401 1.59551 2.89951 2.61702 3.88465 2.33434 2.66902 2.40636 2.43284 2.04221 2.10044 1.75746 1.67307 1.66879 2.35773 1.66517 2.77918 2.51488\nthreshold=-39.283819198608391 1.4144231987897788 -0.59079332554470076 0.11110445317776836 2.5144622974930591 5.6752929687500009 6.2768554687500009 0.086834563006374169 1.0000000180025095e-35 5.5511151231257837e-16 0.34029801008043176 8.8319251537323016 4.0859375000000009 3.1984648276080736 1.2426757812500002 0.89788080022475569 2.8617925509761908 2.9458007812500004 -10.615592002868651 0.31840721315068282 0.56033297298897644 0.84667968750000011 4.0342014792848504 3.5000000000000004 0.92717806344988862 3.8664550781250004 0.7674609687686843 -23.591764450073239 -10.210010528564451 0.25176753112663547\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 -2 -3 -1 13 -7 8 9 10 -5 12 -8 15 21 16 -4 18 23 -19 25 -15 -23 -16 -17 -21 -27 -26 29 -29\nright_child=1 3 5 7 -6 6 11 -9 -10 -11 -12 -13 -14 14 17 24 -18 19 -20 20 -22 22 -24 -25 27 26 -28 28 -30 -31\nleaf_value=-0.026909728248710192 -0.023283139433972772 -0.017238870338959003 -0.019079820817546552 -0.024611021254794429 -0.0056188327154761226 0.022815398616247141 0.0038096341853131942 -0.013877492146202337 0.01871786903837289 0.02056306195045058 -0.0014389262834199766 0.0087024383192236317 -0.022709510964231124 -0.02107639353336669 0.0071381448403302593 -0.023920090598657399 0.023324921065795076 -0.0248371097988229 -0.030973874329104562 0.023895469554192618 -0.023846455045916346 -0.0054785884823123484 0.014622411918655607 -0.022242408988471969 0.0091580528607055432 -0.02222293556926231 0.017359625832934811 0.018033405182864313 0.0024488727874841885 -0.013080828748304643\nleaf_weight=5.2630367126007505 1.800894533706354 7.8127524166493449 1.6003674022131225 3.2524140350238886 10.184697745424272 8.7229983662007253 6.7574884030159428 7.8010462469151198 4.5637937531300841 2.2407246305224353 38.665995954610366 15.106381072021575 3.3029025934915808 1.5895112374930729 4.2400351073333695 1.3270022758870244 9.291664314409898 1.7825415164684262 6.6450918289638157 4.2088030406994204 1.0075836647556569 6.712513329438937 11.767152900508448 1.8500107812369604 21.758538748099454 1.6525749849188289 1.7798992415227983 1.7788973715578347 94.162786413715367 18.277836498389661\nleaf_count=83008 16807 37692 601 14698 884 1007 1561 1095 382 103 20797 892 647 12337 5579 612 1563 7072 7968 635 293 4239 12434 3658 1090 472 266 816 6309 559\ninternal_value=-0.000226439 0.000552756 0.00187034 -0.00405606 -0.0140677 0.00217337 0.00842182 -0.00196592 0.000181986 -0.00210574 -0.00376838 0.00275163 -0.00616286 0.00102917 -0.00398178 0.00250798 0.0160139 -0.0110066 -0.0191323 -0.000280564 0.00683014 0.00426478 0.00741455 -0.00366528 0.00132856 0.0131667 -0.00241174 0.00174779 0.000268917 -0.00959584\ninternal_weight=306.908 291.46 227.123 64.3367 15.4477 225.323 33.8898 56.524 48.7229 44.1591 41.9184 25.1668 10.0604 191.433 43.2357 148.197 10.892 23.1665 12.7351 10.4314 8.64886 20.0692 18.4797 6.09005 137.305 7.64128 3.43247 135.978 114.22 20.0567\ninternal_count=246076 162184 87417 74767 83892 70610 4107 37075 35980 35598 35495 3100 2208 66503 54953 11550 2164 25943 17205 8738 1666 29010 16673 9237 9386 1373 738 8774 7684 1375\nis_linear=0\nshrinkage=0.03\n\n\nTree=238\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 7 18 5 3 18 2 2 2 18 13 3 5 2 2 13 2 5 13 13 18 0 18 1 1 11 13 14 2\nsplit_gain=3.07209 1.78355 1.47682 1.48075 1.40287 2.03542 1.25411 1.50936 1.26313 1.61864 2.02517 1.17897 2.12276 1.44333 1.84871 1.35114 1.87963 1.25673 1.19036 3.25696 1.90174 1.17145 2.79273 2.84857 3.92018 2.61349 1.73324 2.10427 2.13492 1.9505\nthreshold=-39.733196258544915 3.3033179733375269 1.0000000180025095e-35 1.8688348091416842 3.7255859375000004 0.58795853567030554 2.602003343538398 -10.988907814025877 -1.7340650558471677 -30.838275909423825 1.4978661367769956 -0.23457264933257435 0.50207005315878861 1.4980468750000002 8.3473243713378924 2.9882335662841801 -0.21607906719816192 -8.3227787017822248 0.89599609375000011 -0.51158128144345905 -0.39320322527632096 3.7494348669884658 0.57379665772197053 4.297262267176281 1.211689956917019 -0.10824782141703854 0.66750229771629133 -0.0029195251563053355 0.2621461414509263 11.680273056030275\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 21 3 -1 5 11 8 -8 9 -7 -11 12 13 -3 -15 17 -17 -13 19 -14 -20 25 23 24 -23 -2 27 28 -25 -30\nright_child=1 4 -4 -5 -6 6 7 -9 -10 10 -12 15 18 14 -16 16 -18 -19 20 -21 -22 22 -24 26 -26 -27 -28 -29 29 -31\nleaf_value=0.024131756073537063 0.014949440803527569 0.014765717961690399 -0.017453572001767389 -0.01111922924448659 0.015567474379826508 0.020706439452813925 0.008436888418264777 -0.020737926110397684 0.015979392459006446 0.005988855117902132 -0.021596865904261516 -0.0049513644561565281 0.024646040471299577 0.0070288703484482221 -0.023746561558177375 -0.024321294023657992 0.004758515072562362 -0.025375221528526434 -0.028037212497570043 -0.023369111998077466 0.0081488401502802822 -0.022734050507591065 -0.022739184352416248 0.0074584139124406514 0.02171410246758259 -0.0010095579567757154 0.015526220062686748 -0.024827969896996444 -0.015289902951019988 0.0058213024485869683\nleaf_weight=0.98275989244757456 8.8744519746071528 11.709394782548776 12.098425325363335 1.4459260358189565 7.2695951404748476 2.2938989177346221 2.6866656182100996 1.8988893711939443 16.048296650871634 6.1426528668962472 2.639898129273206 8.4033595338769356 1.9043237041332761 4.9735207678750148 1.498661396486568 1.3247689724666987 13.372951077082687 3.5332607448799562 3.1434647537535056 1.2825735376391083 1.3679486390319651 1.4274725482357533 6.0930990738878554 8.4930296176639803 6.9017743869544574 141.68707047027419 2.1394819304405237 4.652975737861996 14.024436605113921 4.7331858275700815\nleaf_count=37 12651 196 71751 11112 116 8 36 52 116 46 36 281 17 90 34 15 420 102 67 38 32 11887 12675 11500 1497 24181 146 15791 60295 10598\ninternal_value=-0.000392067 0.00030177 -0.0133936 0.00602291 0.0037457 0.00255465 0.00843096 -0.00519429 0.0108372 0.00268049 -0.00366306 -0.00108495 0.00341069 0.00887914 -0.00232161 -0.0054186 0.000755047 -0.012491 -0.00904047 0.00435606 -0.0175772 -0.00129224 -0.00529763 -0.00232311 0.0124762 2.35328e-05 -0.00619672 -0.00808107 -0.00444395 -0.00993781\ninternal_weight=305.048 290.521 14.5271 2.42869 91.4941 84.2245 31.7103 4.58555 27.1247 11.0764 8.78255 52.5142 25.8799 18.1816 6.47218 26.6343 14.6977 11.9366 7.69831 3.1869 4.51141 199.027 48.4655 42.3724 8.32925 150.562 34.0431 31.9036 27.2507 18.7576\ninternal_count=245823 162923 82900 11149 1702 1586 294 88 206 90 82 1292 474 320 124 818 435 383 154 55 99 161221 124389 111714 13384 36832 98330 98184 82393 70893\nis_linear=0\nshrinkage=0.03\n\n\nTree=239\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 19 9 5 5 19 3 17 18 7 13 18 3 3 5 17 14 11 5 12 21 11 11 0 18 21 17 14 18 0\nsplit_gain=2.96734 1.1117 1.07618 3.25751 2.1766 1.93809 1.96797 1.75712 1.38896 3.96156 2.21525 1.69337 1.56152 1.97967 1.44808 1.36548 1.64819 1.33067 1.24943 1.24537 1.84805 1.68428 1.53302 1.38477 2.382 3.16617 1.70713 1.70549 1.57533 1.52768\nthreshold=3.9653415636860969 211.50000000000003 1.0000000180025095e-35 4.2695312500000009 4.9179687500000009 205.50000000000003 0.10728613048115772 2.9431225651686121 5.2600927804663344 3.2382812500000004 0.14796868535883259 5.0271592211193568 0.12604885041886044 1.0000000180025095e-35 4.8554687500000009 0.42804718369210387 0.22768215226658742 1.0000000180025095e-35 5.3300781250000009 0.61950675842039238 2.5000000000000004 0.74021522701736098 0.60365083802632757 0.45911736242885776 4.1820210059610323 3.5000000000000004 0.84061941967369014 0.52141881878679586 4.7317931335534196 0.75738199555027408\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 5 17 -5 6 -1 -8 11 10 -10 12 13 -7 -11 -13 -17 -4 -6 22 21 -21 23 27 29 26 -26 28 -14 -25\nright_child=1 -3 3 4 18 8 7 -9 9 14 -12 15 19 -15 -16 16 -18 -19 -20 20 -22 -23 -24 24 25 -27 -28 -29 -30 -31\nleaf_value=0.0094693338650635502 -0.00036403764691126262 -0.025278279278986654 0.022554025287618105 -0.030176713189675044 0.011813192165111359 0.01332903368523272 -0.030740834826297976 0.007985208548914019 -0.026986394161107114 0.026172658073345973 0.010246805388500207 0.022472042028240716 -0.016479049901908523 -0.02063876920202757 -0.012319776289336101 0.014983509205049097 -0.018340242109281635 -0.00016215305479018964 -0.0085541105720551708 -0.0187864609150174 -0.017402101613001455 0.011292814562392561 0.021320693160506877 -0.0069768243288571485 0.031042302356075573 -0.0093733632671034739 -0.0031267065930355963 -0.024078835936230798 0.0079489244395676787 0.0015914835827196172\nleaf_weight=1.6151811403154841 2.2908986548010253 4.6858962562638453 8.4483890075219978 1.4878659496723674 4.2605506915795086 1.2831610862174319 4.0032106449475577 0.88774408679182659 6.3840319227069795 2.1422237043291266 1.2248533144593237 4.6696984660545677 2.6770766060519673 5.3088715753988254 0.54478158557026279 1.7768302958772895 1.57871520870367 3.0576769915417108 4.6117203708354282 1.3146103716571795 7.1277705065185728 5.2326171995373443 2.3636277989426153 19.823234021003145 5.7679516625810248 2.8697826799433415 1.4205268214636815 3.3061368201792893 5.7217343954275748 185.82742832278518\nleaf_count=2436 4663 56833 1966 728 504 1244 2244 300 45754 1683 43 4618 5086 35930 964 1629 9687 615 795 59 277 138 34 16645 2838 3609 2962 18641 5452 18041\ninternal_value=-0.000111818 -0.0181687 0.000372325 0.00662577 -0.00536393 -0.000147355 -0.0151829 -0.0235495 0.000272172 -0.0100268 -0.0204987 0.000720175 0.000258335 -0.0131421 0.0171437 0.0134868 -0.00130583 0.0169871 0.00143236 0.000673458 -0.0078051 0.00356428 0.00120969 0.000908235 0.00146881 0.0150342 0.0247329 -0.00872502 -0.000762242 0.000735796\ninternal_weight=303.715 6.97679 296.738 21.8662 10.3601 274.872 6.50614 4.89095 268.366 10.2959 7.60889 258.07 250.045 6.59203 2.68701 8.02524 3.35555 11.5061 8.87227 243.452 13.675 6.54723 229.777 227.414 215.709 10.0583 7.18848 11.7049 8.39881 205.651\ninternal_count=246418 61496 184922 4608 2027 180314 4980 2544 175334 48444 45797 126890 110956 37174 2647 15934 11316 2581 1299 73782 474 197 73308 73274 44095 9409 5800 29179 10538 34686\nis_linear=0\nshrinkage=0.03\n\n\nTree=240\nnum_leaves=31\nnum_cat=0\nsplit_feature=7 3 2 3 19 22 18 2 5 5 0 12 11 2 5 18 2 13 3 18 21 13 2 5 18 2 18 14 14 21\nsplit_gain=1.81202 2.66097 2.76846 2.11418 1.91661 2.08337 3.02616 2.37803 3.02998 1.87561 1.7613 1.72497 1.6351 1.54449 1.50776 3.65644 3.87081 2.16297 2.08308 2.01495 2.07653 1.82289 1.51487 1.60973 1.40383 3.89031 3.13016 3.03501 2.37558 2.65566\nthreshold=0.89599609375000011 0.58795853567030554 47.592866897583015 0.56033297298897644 200.50000000000003 1.0000000180025095e-35 3.1132683346437333 2.9882335662841801 2.1386718750000004 5.0664062500000009 1.8987378677022575 0.032945457735612756 0.86534941175790137 -28.784247398376461 1.4980468750000002 2.524928003624769 14.537487030029299 -0.55639320842146012 0.29586597258655462 0.89587973461402759 6.5000000000000009 -0.55973954933580472 22.867874145507816 4.7402343750000009 1.4978661367769956 -0.89641284942626942 2.3502401828962087 0.58665773875949134 0.75822041467236145 3.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 3 14 -1 -6 7 24 -9 10 13 -3 -13 -5 15 16 19 -17 -19 -2 -21 22 23 -16 27 26 28 -7 29 -26\nright_child=1 11 -4 9 5 6 -8 8 -10 -11 -12 12 -14 -15 21 17 -18 18 -20 20 -22 -23 -24 -25 25 -27 -28 -29 -30 -31\nleaf_value=-0.025534203286541612 0.0037545000576914844 0.010482686869964043 0.011736337862504187 0.0026428405397937297 0.0024721307883460077 -0.019195991645409863 -0.020043641518393274 -0.016568983840627209 0.018568213382047782 0.014176741602198398 0.0045753019592908581 -0.020408909921348747 0.0085797052851446631 -0.034465995795818438 0.014103710597326418 -0.019637802999944609 0.018548737429100536 -0.0079037837229924846 0.023203214671899411 -0.032705982193764818 0.0057344937206547288 -0.0040143868243575076 -0.01157479411497009 -0.0094974621289384727 -0.015362801154613706 0.031695968625778261 -0.019629574337851226 0.0058632934776618158 -0.011659590057821259 0.019642508096573023\nleaf_weight=2.2609184939642537 2.940081272543468 38.607874836257906 15.089463673033604 1.077145682160334 49.135191841673532 6.7155805446418517 9.9900939251673417 14.518796012300983 1.7258010031728179 0.94295676890760649 1.6391181033104656 2.6069702926470191 2.5929982773959637 4.3529745542546143 12.866542625711874 0.64684868578706201 3.2373782624017613 1.7829809387574176 11.569262344943125 4.6517536016908707 1.2852590612401398 81.753598060003014 2.7045875265812347 2.4598342100503023 1.6147051325688178 3.3588887375453877 4.3250791062732779 9.1896546305215434 3.7306367048076927 9.4572627712732338\nleaf_count=4193 39 970 2521 68 82631 230 43831 1521 32 15 11 151 41 212 15402 1495 113 6670 1055 271 40 78829 492 1502 198 220 2378 149 191 601\ninternal_value=0.000121564 0.00190106 -2.64627e-05 -0.0014291 -0.0028266 -0.00215674 -0.00566009 -0.00271711 -0.0119758 -0.0154664 -0.0206874 0.00831126 -0.00690124 -0.0278387 -0.000420923 0.00586276 -0.00597434 0.0158217 0.0187881 -0.0158901 -0.0247572 -0.00210721 0.00629659 0.00996609 0.00140505 0.00611884 0.000275807 -0.00522698 0.00696596 0.0136877\ninternal_weight=308.83 192.808 149 133.91 116.023 113.762 64.6265 54.6364 16.2446 8.0122 7.06924 43.8078 5.19997 5.43012 125.898 26.1136 12.1145 13.9991 13.3522 8.87709 5.93701 99.7846 18.031 15.3264 38.3918 22.4866 19.1277 15.9052 14.8026 11.072\ninternal_count=246072 109897 108735 106214 136175 131982 49351 5520 1553 306 291 1162 192 280 105908 9683 463 9220 7725 350 311 96225 17396 16904 3967 3588 3368 379 990 799\nis_linear=0\nshrinkage=0.03\n\n\nTree=241\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 2 3 2 1 9 2 17 2 3 1 3 17 3 3 1 3 1 6 1 18 17 13 13 14 2 2 4 13 3\nsplit_gain=3.18427 1.63488 1.40627 2.00825 1.39072 1.24382 2.40944 2.25346 1.87267 2.3674 2.10116 1.86699 1.82017 1.5446 1.85344 1.69243 2.05591 1.42438 3.55881 1.30821 1.26563 1.61284 2.18757 1.76351 1.58161 2.09976 2.61237 1.78713 1.82674 1.64276\nthreshold=5.1405116420104324 -55.708036422729485 0.016076142967085191 21.425971984863285 -0.70140436206657719 1.0000000180025095e-35 132.99933624267581 0.65522739201539382 28.11904335021973 0.19435047505660144 -0.03325755201725291 0.28904431203898151 0.43370896730050829 0.50207005315878861 0.15729395414767636 1.5275240303550033 0.22136347059745312 1.8912989126705502 3.7958984375000004 0.98142480879535021 1.4978661367769956 1.0329219259280322 -0.070829593282168909 -0.051402772639976051 0.49330140830352004 -0.44610977172851557 5.8049912452697763 1.0693359375000002 -0.59662603815460169 0.50207005315878861\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 -3 -4 6 8 -7 9 12 -11 -8 19 14 -10 16 -16 18 -9 -6 21 22 24 -24 25 -12 -27 28 -26 -28\nright_child=-2 2 4 -5 5 7 11 17 13 10 20 -13 -14 -15 15 -17 -18 -19 -20 -21 -22 -23 23 -25 27 26 29 -29 -30 -31\nleaf_value=-0.015439280729881469 -0.022040615405711634 -0.010194616012330442 -0.021030648140074788 0.021687322245580485 -0.013161835166074769 0.017515718432081846 -0.032495834542441454 -0.028425046512240454 0.016997846718427724 0.023143387908537604 0.020677914793355286 0.0062866104270338889 -0.023870162197590417 0.010873041160562773 0.006683332901967153 0.0013863552393328575 -0.029933336778872631 0.010543997793658761 0.0096590716013795659 0.014746790402892741 0.0011245964115020929 -0.013422208390449511 -0.03480477732138354 -0.0034276220562194704 -0.0070900222225287494 -0.026037083265409742 -0.0057999003381479491 -0.012305351459827101 0.018580235611614385 0.020088510057736438\nleaf_weight=4.5709152637798516 4.6641023848609384 1.7958336094237655 1.6475270139491147 5.2692710952810558 1.9709421005028369 9.7406124598205697 2.479704729944046 3.2864804414875817 4.0347507755686847 2.498447100989778 6.3308808609872367 1.2314302433769624 5.7500936537973075 25.070509400028641 1.6729546773190147 17.531669467828578 2.8385973072190609 4.4146284245653078 3.7895556832972943 2.0845158561887729 144.92381322972233 12.124536722165429 1.8238093601539729 4.3465422473454955 4.2616261699768074 1.7621395980531778 3.6643697414256167 9.523161707329562 3.9196959564287699 3.4809134368551895\nleaf_count=40737 49487 4252 22952 3040 18334 867 893 1026 4600 8856 97 84 38322 1075 1145 1092 3094 93 175 995 44080 368 15 109 49 30 64 194 79 27\ninternal_value=0.000243911 0.000665468 0.000968091 0.0133242 0.000617742 0.000816291 0.00024007 0.00763072 0.000623544 -0.000645428 9.79024e-05 -0.0210242 -0.0143206 0.00570869 0.000525653 -0.00309619 -0.0176063 -0.00148022 -0.00935864 0.00126273 -0.000311515 -0.00431727 -0.00123062 -0.0162625 0.0019811 0.00878363 -0.000903975 -0.00403043 0.00589284 0.00772987\ninternal_weight=302.504 297.84 293.269 7.0651 286.204 284.556 263.325 21.2313 259.614 208.465 198.66 3.71113 9.80555 51.1485 26.078 22.0432 4.51155 11.4907 7.07604 4.05546 196.161 51.2377 39.1131 6.17035 32.9428 15.2383 8.90742 17.7045 8.18132 7.14528\ninternal_count=246231 196744 156007 7292 148715 125763 123602 2161 122625 111619 53968 977 57651 11006 9931 5331 4239 1294 1201 19329 45112 1032 664 124 540 218 121 322 128 91\nis_linear=0\nshrinkage=0.03\n\n\nTree=242\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 2 2 0 12 22 1 13 0 13 13 2 2 17 0 5 14 13 3 2 3 4 14 13 2 0 19 5 14 5\nsplit_gain=3.7479 2.89648 5.22551 2.78711 2.35498 2.15572 2.72978 2.12011 2.25733 2.00242 2.75509 2.21738 2.11976 2.22751 2.0615 1.80423 1.79932 1.7698 2.60726 2.21549 1.68026 1.66861 1.59298 1.55525 1.51431 2.24838 1.47984 1.83755 1.68771 1.40711\nthreshold=1.3170161875856263 -3.0241031646728511 -1.3165187835693357 0.69651904584641833 0.63107571507209059 1.0000000180025095e-35 2.1468275719512682 -0.67453178091691768 0.38995551143381973 -0.10757639301670423 -0.24218098761785664 2.0782871246337895 3.4383306503295903 0.97722272727943638 0.83245446090221253 5.7968750000000009 0.61113007853467505 0.14796868535883259 0.48555671035752257 -11.357147216796873 0.48555671035752257 1.0693359375000002 0.81584600691354858 -0.82709081186650735 -14.08206272125244 0.54195883642662512 261.50000000000006 1.2426757812500002 0.92151054123056431 1.4980468750000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 20 7 -2 17 -7 8 -4 10 12 -11 21 14 16 -12 -14 18 24 -20 -3 -5 -6 -16 -1 -26 -8 28 -28 -24\nright_child=4 2 3 9 22 6 26 -9 -10 11 15 -13 13 -15 23 -17 -18 -19 19 -21 -22 -23 29 -25 25 -27 27 -29 -30 -31\nleaf_value=-0.016065826429869503 0.0061097869276922744 0.02745630628176568 0.020455159727053491 0.030623060189595864 -0.025012851652187466 -0.022291144331392978 -0.01073230280754325 -0.012285216990506845 -0.01976539730810627 -0.021327470073603201 -0.030975335828733467 0.01169012763928659 0.015380156873300015 0.0068095454348057914 0.0060689015191051051 0.011872368586500271 -0.02001616539153565 0.014441100302019531 0.018354996839216674 -0.015558657237102636 -0.0039335768271194097 0.0019397233093381755 0.014236901922296952 -0.020578320529123003 0.0084332906581049468 -0.014311056231105325 -0.021347041645867249 0.0092268694545865945 0.015165819100562548 -0.011326160389587963\nleaf_weight=10.702844076161112 109.28866411635499 7.4239011746677743 3.1845261716794075 2.5056246909866777 2.6738949216669408 11.801207269745531 19.45025432284001 22.541913298023363 0.89107768187295722 1.0762730680985466 3.0787517797537016 22.574369574353568 3.7466039668618185 12.542366671443459 1.9473374492718001 0.55721107404679049 0.83092855845370639 4.9050126166875989 6.2599849835751229 1.4416281343274011 1.5953227251884525 4.1138089634719108 2.5178352838847786 9.5115431839549238 8.1893812811599691 5.8208244577944006 3.3031996038953357 9.1827346488389576 0.74980885447439494 3.3153750519268215\nleaf_count=70410 5450 3592 3215 635 91 76002 1543 45204 1885 251 1916 5881 327 3736 241 2 868 148 472 316 188 1014 48 2317 14422 5807 272 295 12 109\ninternal_value=0.000771005 -0.00195013 0.00152293 -0.000778284 0.00490886 -0.00608391 -0.0104689 -0.00872636 0.00949981 0.00269159 -0.00151706 0.00938336 0.00131007 -0.00236149 -0.009513 -0.0232644 0.00651818 -0.000719967 -0.00337767 0.0109484 0.0220656 0.0153912 -0.00953899 -0.0159262 -0.00809442 -0.00134019 -0.00566547 0.00201124 -0.0129276 0.000176556\ninternal_weight=297.724 179.928 98.1216 89.1023 117.796 81.8069 44.4872 26.6175 4.0756 62.4848 38.8342 23.6506 35.1982 28.5788 16.0364 3.63596 4.57753 37.3197 32.4147 7.70161 9.01922 6.61943 8.50711 11.4589 24.713 14.0102 32.686 13.2357 4.05301 5.83321\ninternal_count=246669 240971 71272 67492 5698 169699 78124 50304 5100 17188 11056 6132 9138 7489 3753 1918 1195 91575 91427 788 3780 1649 248 2558 90639 20229 2122 579 284 157\nis_linear=0\nshrinkage=0.03\n\n\nTree=243\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 19 1 0 3 18 7 19 0 18 1 3 0 12 4 4 6 7 4 9 19 21 1 3 1 19 5 11 7 17\nsplit_gain=3.96772 2.45594 1.84193 2.69562 2.07272 2.00591 1.89689 1.90779 1.99918 2.07246 1.7415 1.73885 2.78192 1.6813 2.36834 3.3854 3.0832 3.31498 2.8823 2.22905 1.69871 1.67409 2.01616 1.66526 1.85897 2.17968 1.65542 2.36792 1.57793 1.5694\nthreshold=0.37248792739977959 248.50000000000003 1.6575619461558879 0.53713645201195115 0.10628559131747423 2.1383330595080281 3.1982421875000004 246.50000000000003 1.0212340148387515 3.1567740231385479 -0.27208026740318675 0.082806484040480313 0.96046990436194013 0.28848765710462548 5.5515136718750009 6.6723632812500009 1.7006835937500002 2.5244140625000004 4.0444335937500009 1.0000000180025095e-35 230.50000000000003 2.5000000000000004 2.2520445226790948 0.47260169401963864 1.5275240303550033 248.50000000000003 5.1621093750000009 0.86534941175790137 1.8691406250000002 1.5538318668855886\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 4 -1 10 7 11 9 -9 -5 -4 -13 14 18 -16 17 -17 19 23 -18 22 26 24 -2 -26 27 -15 -6 -14\nright_child=13 -3 6 5 28 -7 -8 8 -10 -11 -12 12 29 21 15 16 20 -19 -20 -21 -22 -23 -24 -25 25 -27 -28 -29 -30 -31\nleaf_value=-0.021625212717242409 -0.025117769049355334 -0.015984002924471111 0.0065896025406748243 0.012630346644572835 -0.01793689821280001 0.012871456885854731 0.01323721902912183 0.029014762228605429 -0.012910322284095868 -0.014239018395702688 -0.01266737760640016 -0.0351246164776284 -0.016060994850004458 -0.03234807654964917 0.021650302908221015 0.013371056564505142 -0.0076614800556471076 -0.03081879406002485 -0.017302454156254993 0.022867379142200897 0.021038661540782141 -0.014753816018051719 0.012799237058921711 0.0034268614565043953 -0.0096388341958717552 0.0043886032267846506 0.011414446807267296 0.0060887934686280612 0.0065526041212061069 0.004593864688956382\nleaf_weight=5.6274079716818042 2.3231495828472353 16.382671548527242 2.4229055406813229 2.7902495723510583 2.2358392276857995 16.17209235625819 2.340504143954603 2.3263929722779717 2.6508029127198824 0.71816078325264232 6.1332361321105981 5.214701158246501 9.502231728332843 2.6839759376744059 12.262821240624364 1.6448185976259972 2.142421002019546 3.3692378837713477 6.1555940855814688 3.5537697462095812 4.6080594215300152 9.5310267797194665 6.0302067749908019 112.5699126884565 16.492953618610954 22.256202579214914 1.6706380237301344 1.9240668329875914 7.8284814443450159 4.2940659889745794\nleaf_count=56173 5225 89076 403 856 33359 27539 81 66 140 814 2168 2556 640 211 774 69 160 266 1164 60 186 763 203 6687 1401 1727 53 33 12498 319\ninternal_value=-0.000501193 -0.00586225 -0.00331085 0.000838344 -0.0085977 0.00684187 -0.00889255 -0.0112045 0.00372147 0.017814 -0.00428148 -0.0151596 -0.0181223 0.0017362 0.00266089 0.0112582 -0.000421492 -0.0165098 0.0013355 0.0021701 0.0121167 -0.00596645 0.0014351 0.00154659 -0.00356576 -0.00167424 -0.00973825 -0.018075 -1.72966e-05 -0.00975623\ninternal_weight=295.859 86.6397 70.2571 40.7873 15.6917 25.0956 29.4698 27.1293 5.69536 3.04455 8.92349 21.4339 19.011 209.219 187.379 24.0274 11.7645 5.01406 163.352 157.196 6.75048 21.8399 12.3089 153.642 41.0723 38.7492 6.27868 4.60804 10.0643 13.7963\ninternal_count=245670 226688 137612 132593 102030 30563 5019 4938 1020 880 3024 3918 3515 18982 17719 1455 681 335 16264 15100 346 1263 500 15040 8353 3128 297 244 45857 959\nis_linear=0\nshrinkage=0.03\n\n\nTree=244\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 3 7 4 2 7 0 4 2 13 14 7 4 7 0 18 0 17 17 0 17 4 2 1 0 18 17 4 18 3\nsplit_gain=2.08451 2.52034 2.13606 2.09972 2.63881 2.27472 2.06969 1.99202 1.92818 1.64316 1.56423 2.44279 4.46575 2.98208 2.6183 2.30269 1.97138 1.6783 1.48992 1.45864 1.44225 1.4177 2.90894 2.38089 2.13728 2.03027 1.84624 1.75541 2.55456 2.15691\nthreshold=30.514518737792972 0.22136347059745312 2.0131835937500004 3.4389648437500004 42.119770050048835 1.0000000180025095e-35 0.60345709839408768 1.6572265625000002 18.240050315856937 -0.51158128144345905 0.47889823060390885 1.7006835937500002 5.5791015625000009 0.89599609375000011 1.6765212893675572 3.6505739029280164 1.5447487307370646 0.86664248603069383 0.49969040966739248 -0.064617101914726832 0.92169800002628566 0.84667968750000011 -28.784247398376461 1.8912989126705502 -0.32101161284190144 4.9593358439305844 0.93623644617438528 2.6030273437500004 0.89587973461402759 0.3261289997719386\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 8 4 5 -2 19 -7 -1 -3 11 13 -13 14 16 -14 17 -11 -9 -4 -10 26 25 24 -24 -23 -12 29 -29 -25\nright_child=3 9 6 -5 -6 7 -8 18 20 10 21 12 15 -15 -16 -17 -18 -19 -20 -21 -22 22 23 27 -26 -27 -28 28 -30 -31\nleaf_value=-0.025466257330865116 -0.021946206286549797 0.0035991664371311251 -0.0016848562196845806 0.010178380725172109 -0.010076807953047133 0.02020588935821083 0.0098507751662672421 -0.022923372631955929 0.014631608289457192 -0.0107015617930078 -0.01244122667629936 -0.033830151366331898 -0.015123976758751803 0.0078464332051676935 -0.021570438198255236 0.013779582678157983 0.016610358356678845 0.0040851229365148397 0.006813506113963308 -0.025215592727210929 -0.01783367339445106 -0.030920536461830076 0.013577003042999683 0.02084602543212025 -0.02008060019302536 0.0046427220481036991 0.027076359392392359 -0.023950822693289894 0.014073932833109544 -0.0074308821442121585\nleaf_weight=8.9145378637406356 1.1491244306647583 60.515354596814177 6.2506910429439122 29.071275942661487 13.990133634319021 7.9441686997618017 6.4004766691173236 1.204446809678984 1.9511151470976522 13.12898038048275 1.0329876314040114 4.0479859069203021 3.2809157796436903 40.092099597523884 5.2854180514059399 4.9195931242400093 4.2183209307113421 12.73099544044794 3.9349646314349229 2.9816237057999606 1.117500580559067 5.6846927541621417 1.2826902471260808 1.7356028985309446 10.14541865316005 1.5400139602572833 1.603820194097352 0.92793518677353759 10.155412801892451 22.875318127937135\nleaf_count=115673 2524 19669 38274 2024 12984 1687 843 388 2979 4431 1927 866 79 3067 123 621 69 4568 1418 4293 5351 2353 690 284 10504 3974 2652 11 428 1094\ninternal_value=-0.000416707 -0.00168072 -0.0100218 0.00466042 -0.00117723 0.00765792 -0.00238775 0.0116317 -0.01903 -0.000515023 -0.00224866 0.000275826 -0.0116201 0.00233358 -0.00396028 0.00182871 -0.000271288 -0.00354093 -0.00275416 -0.0110059 0.00133135 -0.00607187 -0.00717797 -0.00420548 -0.0155144 -0.0236971 0.012431 -0.000260622 0.0091717 -0.00470066\ninternal_weight=290.114 232.82 27.6159 57.2941 28.2228 14.2327 15.6328 13.0836 11.9832 205.204 144.688 87.7043 12.2485 75.4558 35.3637 8.20051 30.0783 25.86 5.13941 9.23231 3.06862 56.9839 54.3471 47.1224 11.4281 7.22471 2.63681 35.6943 11.0833 24.6109\ninternal_count=245848 224823 167413 21025 19001 6017 43410 3493 124003 57410 37741 13824 1566 12258 9191 700 9068 8999 1806 42567 8330 23917 19338 13011 11194 6327 4579 1817 439 1378\nis_linear=0\nshrinkage=0.03\n\n\nTree=245\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 19 2 5 3 18 1 6 1 1 21 17 17 6 1 2 3 7 9 5 6 2 3 7 7 7 18 6 18 1\nsplit_gain=3.07645 1.92999 1.68601 2.54484 2.67109 1.85962 1.36367 1.26311 1.23071 1.2529 2.01344 1.20566 1.21533 1.05982 1.03117 1.01071 1.00484 0.957521 0.866751 2.25514 1.67409 1.57478 1.54792 1.53456 1.37505 2.32642 2.17393 1.80854 1.5876 1.50136\nthreshold=3.2718994203396332 233.50000000000003 -34.252307891845696 1.7006835937500002 0.51817819236196982 1.7005986908310777 4.0016611463566001 1.0000000180025095e-35 -0.59079332554470076 -0.016381246029292276 2.5000000000000004 0.61464819824310435 0.15621648314530187 2.1386718750000004 0.15052965826911111 53.951957702636726 0.0037166969630965979 3.5400390625000004 1.0000000180025095e-35 2.9179687500000004 3.7958984375000004 132.99933624267581 0.28904431203898151 3.0683593750000004 3.1132812500000004 2.7412109375000004 1.4978661367769956 2.2500000000000004 1.4978661367769956 1.5275240303550033\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 7 3 4 -1 -5 -3 -2 16 10 13 12 -6 -10 -11 17 -4 -12 21 -20 23 24 -23 -21 25 -16 -26 29 -27 -28\nright_child=1 6 8 5 11 -7 -8 -9 9 14 15 -13 -14 -15 18 -17 -18 -19 19 20 -22 22 -24 -25 26 28 27 -29 -30 -31\nleaf_value=-0.013192615816277183 -0.016512321089789395 -0.029209932835224703 0.0066340783739682399 0.0082796532219166323 0.011360946661416805 -0.029210316971753431 0.003877728669524311 0.0087101434545254087 -0.019816749485487291 -0.015191013674538068 0.026291203212655417 0.018380215485474506 -0.020055111414282782 0.011312784763022887 0.0015428499364381464 -0.010166859005268607 -0.023386659496304543 -0.0023556245347584361 0.02381474676706535 0.0049300248506609487 0.014039709570500615 -0.028269745912880187 0.0076891492834535325 -0.01889043086069889 -0.019120083685632946 0.0092962026562455006 0.021846054734536564 -0.0080541223730933205 -0.023529775064074099 -0.0013604429617031591\nleaf_weight=6.3554065345539374 1.8039477409968165 5.308070785371001 0.95106576073726445 1.0309562547772646 1.3569492461829216 4.6919538103696823 1.1005299690805257 3.9292449119648154 1.5741061475589488 2.2470583582455665 5.8240048229754393 5.3123085146944504 1.1541437499181482 0.7784788813646627 208.33332237392233 0.40291458540593783 2.0804291674153319 1.0571354954108754 4.6948495159613257 4.3272979074063169 4.4540002907888256 2.0825137199265096 1.1174356850460752 3.7523227802084875 1.2756252335384513 1.1471787637565274 10.617379773291761 2.5469236351418649 4.509577248099049 3.0565711985109365\nleaf_count=24825 10756 57381 765 33 69 38165 81 1703 15718 8687 13935 204 113 167 48099 1068 21099 147 267 259 147 608 94 325 34 20 616 140 849 83\ninternal_value=0.000540008 -0.0136571 0.00118656 -0.00702756 0.000168664 -0.0222297 -0.0237714 -0.000499801 0.00182317 0.00206722 0.0121666 0.0112878 -0.00467786 -0.00921404 0.00163812 0.0193507 -0.0146589 0.0221498 0.00184771 0.00815671 0.000965216 0.00135192 -0.0168718 -0.00699475 0.00167532 0.000992099 0.009526 0.0127582 -0.0164763 0.0169195\ninternal_weight=298.874 12.1418 286.732 19.9017 14.1788 5.72291 6.4086 5.73319 266.83 263.799 9.63664 7.8234 2.51109 2.35259 254.162 7.28405 3.03149 6.88114 251.915 17.2285 12.5336 234.687 3.19995 8.07962 231.487 213.99 17.4965 16.2209 5.65676 13.674\ninternal_count=246457 69921 176536 63409 25211 38198 57462 12459 113127 91263 31035 386 182 15885 60228 15150 21864 14082 51541 998 731 50543 702 584 49841 48968 873 839 869 699\nis_linear=0\nshrinkage=0.03\n\n\nTree=246\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 6 3 1 0 3 1 3 17 7 7 5 1 3 18 3 4 9 14 1 18 18 18 4 1 4 3 18 4 21\nsplit_gain=4.23234 2.2365 2.13897 2.73905 2.06753 2.60159 3.31988 1.9611 1.92809 1.75941 1.69149 4.50983 1.64064 1.59141 2.11642 2.58892 1.88124 1.67523 1.83872 2.44957 2.25378 1.93714 1.97598 3.33692 2.99049 1.66611 2.91074 2.77188 2.10176 1.65953\nthreshold=2.775573723725536 1.8691406250000002 0.33061855659870582 1.8223781763416411 1.1495945197163422 0.50207005315878861 0.66594449314266779 0.45898141303402712 1.0329219259280322 0.89599609375000011 2.1386718750000004 2.2500000000000004 1.211689956917019 0.50207005315878861 2.3502401828962087 0.29586597258655462 6.6144968271255502 1.0000000180025095e-35 0.53481096685124585 2.6981674883069502 5.0271592211193568 2.249904835165133 4.1026092131977068 1.0693359375000002 4.0016611463566001 5.8869628906250009 0.38657275885365588 4.297262267176281 3.8664550781250004 1.5000000000000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 -2 -4 13 6 8 12 -6 -9 11 -7 -8 14 29 -16 25 18 21 20 -20 -15 24 -24 -23 27 -27 28 -17 -1\nright_child=1 -3 3 -5 5 10 7 9 -10 -11 -12 -13 -14 17 15 16 -18 -19 19 -21 -22 22 23 -25 -26 26 -28 -29 -30 -31\nleaf_value=0.0019593264649932548 -0.031089418122554453 0.0036287699280076625 -0.026336502403056593 0.0056653590872834565 0.020734752303784847 0.0073301586557926145 -0.020116833768632502 -0.025983513494762023 -0.011336607458190959 -0.0029790821862015933 0.019081907008144015 -0.02419284169726179 0.00065453450268646864 0.015603649246415266 -0.0060220180347086167 0.011319345798465524 -0.010877613928723367 0.014439270450688362 -0.031175330273455981 -0.0045027650569951991 0.0062744200877751467 -0.026289229590554162 0.02220543645306805 -0.012705282233572401 0.0065002292388567562 -0.017615683057209727 0.03054619192157966 -0.0085713280379357912 -0.011464237711785523 -0.025365761480656824\nleaf_weight=3.14751527372482 6.3520524706020778 4.8052776578995227 3.0388369427862454 5.2797282305655244 7.5403720979425071 58.037385458093922 2.6967321540550975 5.0119602556587761 1.4212410531281419 6.5479438672482502 8.4129443382844311 3.41931307921186 45.443576017894884 3.63776046590647 25.807136452451232 31.507080355480412 4.9295249791207416 2.2539239096658994 7.5734519882007607 11.432726690873094 1.3798378982355641 5.9137979361912603 3.7971571322632363 4.1340425897578825 3.4034858467057347 0.72693240619537292 4.3867614734999734 9.7193934339527441 3.2935081338910086 3.91460784084029\nleaf_count=560 61703 11976 2884 703 1974 840 976 163 1062 96 82 89 1978 36 133206 7923 833 90 2238 524 634 415 136 392 47 251 209 12836 377 1245\ninternal_value=-0.000442057 -0.0135914 -0.0191035 -0.00759687 0.000554975 0.00309911 -0.00104837 -0.00366828 0.015021 -0.0142276 0.00713217 0.00521702 -0.000894736 -0.00213936 0.000209347 0.00165969 0.00533595 -0.00675604 -0.00822774 -0.0144174 -0.0253532 -0.00180366 -0.0061288 0.00462351 -0.0148425 0.00712928 0.0219383 0.00498369 0.00890312 -0.0144548\ninternal_weight=288.966 19.4759 14.6706 8.31857 269.49 138.531 68.6618 59.7002 8.96161 11.5599 69.8696 61.4567 48.1403 130.959 87.4325 80.3703 54.5632 43.5262 41.2723 20.386 8.95329 20.8862 17.2485 7.9312 9.31728 49.6337 5.11369 44.52 34.8006 7.06212\ninternal_count=246478 77266 65290 3587 169212 7260 6249 3213 3036 259 1011 929 2954 161952 157440 155635 22429 4512 4422 3396 2872 1026 990 528 462 21596 460 21136 8300 1805\nis_linear=0\nshrinkage=0.03\n\n\nTree=247\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 4 1 18 2 2 1 2 1 10 3 4 18 13 1 1 2 2 21 3 3 3 1 18 6 18 18 4 19 14\nsplit_gain=1.94942 2.31722 1.76643 2.22708 2.32191 1.59179 2.03441 1.89477 1.7041 2.10528 1.97417 1.61714 2.54784 1.90782 1.51449 4.91151 1.96156 2.46719 1.76924 1.64583 2.1478 1.78808 1.68449 1.50884 1.4877 1.4821 1.75573 1.47674 1.39982 2.44748\nthreshold=0.58795853567030554 0.84667968750000011 1.9634439907166827 3.5972184255501678 9.3619499206542987 -34.252307891845696 2.6981674883069502 -38.377685546874993 0.45969269984116679 3.6242520361853052 0.34948566548243237 4.8474121093750009 5.1619206365690227 -0.26322344360915401 1.4475910192202701 1.075454090076706 -3.0241031646728511 -1.3165187835693357 1.5000000000000002 0.41780940659066851 0.40911314460322756 0.42638606281916264 4.0016611463566001 2.6735537653587351 4.3886718750000009 3.1984648276080736 4.0689902272260703 1.7380371093750002 245.50000000000003 0.32341854005133885\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 -2 3 -3 -5 6 -1 23 9 -7 11 13 -13 -10 15 16 -12 -18 -19 20 27 22 -21 -8 -14 -4 -27 -16 -15 -30\nright_child=1 2 25 4 -6 8 7 -9 10 -11 14 12 24 28 19 -17 17 18 -20 21 -22 -23 -24 -25 -26 26 -28 -29 29 -31\nleaf_value=-0.021370787058049703 -0.023243235736670739 -0.018588664455725053 0.010290972278097592 -0.019594919993734181 0.020940894243692117 0.0090766387543909522 0.01829193587634104 -0.020689885790097343 -0.020279703897702693 -0.023658142853671054 -0.020633577141288249 0.0075738662566817265 -0.027030511957788508 -0.010422829408056278 -0.0089011203867105423 -0.031160427364958571 0.027642079317137028 0.0095772127822029967 -0.022138192476956378 -0.030675208234052571 0.021740192116005863 0.0013006434207317172 0.0066277658294212546 -0.0060233288803640947 0.010374830404390398 -0.0178202417614759 0.0097871281301561888 0.0082723682516719719 -0.0090221645544188833 0.018147580231969222\nleaf_weight=6.1672198272466394 1.4778595647267128 4.425423047221555 48.039501675462816 0.94749574485103938 3.0603973746583506 26.482022934845308 3.8220993673894554 1.9917244935968463 9.3639369760438722 0.94266300557137483 1.2674438221447388 13.807967021163678 2.6332222939872731 14.100725433634581 5.5446250597788156 4.2493459512570544 3.1750592530051991 3.0358695176823893 1.6322290787707059 2.0744437295152229 7.1352618183354926 86.101074576413339 1.2213204401195983 3.670484400910027 0.6993435160111402 3.1779221539763958 3.2088580086128786 13.806999636074577 5.5543272580016474 4.5362066939344459\nleaf_count=93865 462 407 496 420 587 103949 58 109 12924 117 1890 943 1331 10238 1109 2634 159 906 1366 272 308 3920 24 590 24 158 198 728 2404 3431\ninternal_value=0.000949829 0.00550722 0.00653655 -0.0057227 0.00935869 -0.000383642 -0.00930809 -0.000174048 0.000331402 0.00715873 -0.000742969 -0.00569988 0.00174064 -0.00944099 0.00123021 -0.00797847 0.00484654 0.0108318 -0.00346155 0.00235103 0.00877133 0.000372808 -0.0185268 0.00707366 -0.0185981 0.00850508 -0.0045025 0.00315623 -0.00460706 0.00372701\ninternal_weight=287.353 64.3375 62.8596 8.43332 4.00789 223.016 15.6515 9.48431 207.364 27.4247 179.939 50.6957 17.1405 33.5552 129.244 13.3599 9.1106 7.84316 4.6681 115.884 26.4869 89.3968 3.29576 7.49258 3.33257 54.4263 6.38678 19.3516 24.1913 10.0905\ninternal_count=246027 2728 2266 1414 1007 243299 94622 757 148677 104066 44611 31295 2298 28997 13316 6955 4321 2431 2272 6361 2145 4216 296 648 1355 852 356 1837 16073 5835\nis_linear=0\nshrinkage=0.03\n\n\nTree=248\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 2 2 2 2 1 1 0 1 21 2 2 3 10 4 1 2 0 18 0 17 7 2 1 1 21 17 4 21 8\nsplit_gain=2.47883 1.59315 4.3196 2.1921 2.42563 3.83078 1.67071 2.16321 2.03774 1.5868 3.15601 2.66115 2.45379 2.21361 2.17677 2.10114 3.0851 2.75536 2.40526 3.14593 1.84518 1.8016 1.72801 1.55608 2.29253 3.41314 2.2125 1.54697 1.54425 1.53882\nthreshold=5.1405116420104324 -5.1300392150878897 -1.3165187835693357 -1.7340650558471677 -3.9017317295074458 0.87590939928379707 -0.16863805028123086 0.84505259813102007 0.26032702032866095 7.5000000000000009 -7.5318784713745108 -9.8266754150390607 0.48555671035752257 1.0000000180025095e-35 1.0693359375000002 2.5144622974930591 -9.8266754150390607 1.5447487307370646 1.8688348091416842 1.0410863859294157 0.48975197669932524 0.89599609375000011 -34.252307891845696 0.6180242047922454 1.4475910192202701 1.5000000000000002 2.6250469244966204 0.84667968750000011 3.5000000000000004 1.0000000180025095e-35\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 9 3 4 -3 -6 7 -4 -8 10 12 20 13 15 -14 16 17 -1 -17 28 -11 22 -16 -10 25 26 -25 -26 -20 -27\nright_child=-2 2 6 -5 5 -7 8 -9 23 11 -12 -13 14 -15 21 18 -18 -19 19 -21 -22 -23 -24 24 27 29 -28 -29 -30 -31\nleaf_value=-0.027475239038319776 -0.0186744207153773 0.01886469207892516 -0.008476040781489903 0.027677890774478266 0.019302088015984872 -0.021509718702532925 -0.020739779501630185 0.02205641927814379 0.013819345082265665 0.0052514635010102355 -0.020908557634355281 0.024327658280974537 0.011508374826033741 0.01133775301655825 0.0042053382189912126 -0.019291455987078238 0.0086498091737936406 0.0061621182165196428 0.0022194408196253377 0.021567605011820864 -0.023055070582657718 0.0023293534367700513 -0.024760245585216677 0.0095317435699574724 -0.019423910062090658 -0.028924434754103587 -0.024159576925331678 0.0021196724694590515 -0.021316110043127091 0.0019800590833680765\nleaf_weight=12.027738127120069 4.8500815200872003 6.5635886572266608 2.3575700366914116 7.2905857203345468 2.5373570565125201 4.0800664524429067 3.0394729880792957 5.9347922740953436 6.8891254955909735 7.0507198667173707 7.2643547090642633 4.3393190293309081 16.764927269265172 4.5019951522622241 2.537825540995982 4.4032380535500151 3.0555495057606139 2.2448712878795059 6.901691103459596 4.803139245894271 1.8016433982570559 23.257430598006977 4.0119298878162253 7.6630493514537665 2.0827269174578751 6.3124953733406519 1.2106605546464377 118.47117610456917 2.8026751482329928 1.6558771791023899\nleaf_count=106852 49355 2354 18976 841 3167 896 10866 3071 7021 5613 4285 1040 729 2180 262 91 3900 157 702 109 4564 866 508 3816 1494 5659 679 5663 391 291\ninternal_value=0.000660532 0.00105529 0.0028075 0.0154231 0.00717102 -0.0053814 0.00106003 0.0133979 0.000279928 -0.00180721 -0.00318221 0.00758043 -0.00148734 -0.00678189 0.00318949 -0.00927693 -0.0165246 -0.0221275 -0.00213942 0.00397478 -0.00226484 -0.00171112 -0.0144665 0.000862429 0.000117601 -0.00992517 0.00295384 0.00159074 -0.00593411 -0.0229976\ninternal_weight=288.708 283.858 176.089 20.4716 13.181 6.61742 155.617 8.29236 147.325 107.769 94.5774 13.1917 87.313 40.7409 46.5721 36.2389 17.3282 14.2726 18.9107 14.5075 8.85236 29.8072 6.54976 144.285 137.396 16.8421 8.87371 120.554 9.70437 7.96837\ninternal_count=246398 197043 64794 7258 6417 4063 57536 22047 35489 132249 121032 11217 116747 114382 2365 112202 110909 107009 1293 1202 10177 1636 770 24623 17602 10445 4495 7157 1093 5950\nis_linear=0\nshrinkage=0.03\n\n\nTree=249\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 5 1 2 3 1 0 19 0 1 0 3 7 5 4 5 11 17 4 12 1 1 19 13 4 5 4 0 5 0\nsplit_gain=4.86919 3.24949 2.8494 2.09024 1.82047 1.80032 2.09704 4.35467 2.54264 2.34059 2.80025 2.29775 1.85021 2.40859 1.58987 1.53924 1.283 1.2676 1.2635 1.21064 1.18691 1.18111 1.65475 2.16206 1.1896 1.15554 1.13956 1.32298 1.20003 1.08379\nthreshold=2.775573723725536 3.5107421875000004 1.6109634896136267 -42.103004455566399 0.51817819236196982 2.3713805701640611 0.40404697678975493 264.50000000000006 0.69651904584641833 -0.12981918606627474 0.72109735700429223 0.37963534769016083 2.5244140625000004 4.2265625000000009 7.0266062021255502 3.5976562500000004 0.49644476220754463 0.24468021674538498 5.0736774206161508 0.44193439040807492 1.9634439907166827 2.054932859494814 255.50000000000003 -0.6105545148247048 4.6120605468750009 2.2500000000000004 2.6437988281250004 1.5447487307370646 2.0131835937500004 1.1495945197163422\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 -2 15 4 -1 6 11 8 14 17 25 12 -5 -14 -8 -3 -7 -10 -18 -4 29 -22 24 -24 26 -11 27 28 -23 -12\nright_child=1 2 19 5 -6 16 7 -9 9 10 20 -13 13 -15 -16 -17 18 -19 -20 -21 21 22 23 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.021758870110903755 -0.027754307207406372 0.01328938954634401 0.018620962111796518 -0.014811605882338257 0.0030374310584885345 0.0044500179301628345 -0.021142994189032087 -0.028306485788135997 -0.012459592242762815 0.02775251421695743 -0.011474318496271481 0.015115099337145523 0.018666115135754637 -0.0099797591822868466 0.0054044335667775157 -0.023754742839070815 -0.010984274599362685 0.019627544895067482 0.010978147664698148 -0.013397493461214206 0.013008462566721635 -0.020698534485792296 -0.017812223341106853 0.016418883477172578 0.0086679821556553856 -0.0099034920073775099 -0.030101364466561038 0.01190550876291578 0.011982871279003927 -0.0017974609183739473\nleaf_weight=5.6701677082232873 8.1405647585978063 0.58202847009465397 4.0701224103877394 5.298700423053563 4.0735229877682286 134.47042813881308 7.7026475592722345 4.9707989909871904 0.60676589841318573 2.8607546699511199 17.565114258813082 11.962976930170042 4.8345516758320155 3.9646108672968694 2.1909144644623666 3.9046140782982093 8.6081270072463649 6.3442887115295434 2.2421314981766036 0.57627129927277554 5.7807088035369842 3.3385223510067612 1.3087220389716092 5.6939202377507163 1.8659415440633891 0.37181210505332263 1.895618809663574 1.7705394072982015 0.60235649425885629 24.765857078212321\nleaf_count=49093 64469 1710 166 48091 389 3805 30349 3915 1484 904 10305 4074 3182 2493 188 10656 271 5790 74 25 254 441 44 118 42 105 136 72 16 3688\ninternal_value=-0.000107454 -0.0150973 -0.00218998 0.000900879 -0.0120745 0.00142948 -0.00135696 -0.00353458 -0.00176672 0.000203371 -0.00157979 0.00598672 -0.00224741 0.00605795 -0.0153077 -0.0174029 0.003642 0.0156119 -0.00590246 0.0129804 -0.00308415 0.00245327 -0.00178262 0.00859506 -0.00954224 0.0221048 -0.0144972 -0.00560471 -0.014289 -0.00598498\ninternal_weight=288.034 17.2736 9.13304 270.761 9.74369 261.017 115.696 89.6353 84.6645 74.7709 67.8199 26.0608 14.0979 8.79916 9.89356 4.48664 145.321 6.95105 10.8503 4.64639 64.5873 22.2563 16.4756 7.00264 9.47298 3.23257 7.60704 5.71142 3.94088 42.331\ninternal_count=246349 77026 12557 169323 49482 119841 115691 57851 53936 23399 16125 57840 53766 5675 30537 12366 4150 7274 345 191 15116 1123 869 162 707 1009 665 529 457 13993\nis_linear=0\nshrinkage=0.03\n\n\nTree=250\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 4 3 2 10 17 4 4 2 2 19 19 17 17 17 4 19 7 18 17 18 2 0 7 4 21 18 7 19 21\nsplit_gain=2.25908 1.65753 1.54579 2.74602 1.46328 1.42193 1.67658 2.48901 2.71385 2.6969 2.18407 2.15303 2.46001 2.1554 2.63873 2.02325 1.79283 2.12771 2.75259 1.62175 2.21642 2.54952 2.15217 1.97708 1.51129 1.56606 1.49226 1.80052 1.54138 1.48629\nthreshold=-34.252307891845696 0.84667968750000011 0.47260169401963864 -49.116760253906243 3.6242520361853052 5.817026225953116 3.2580566406250004 2.1750488281250004 -22.378742218017575 -10.988907814025877 235.50000000000003 260.50000000000006 0.25672585397575204 0.65522739201539382 0.49969040966739248 2.6030273437500004 264.50000000000006 1.2426757812500002 5.0271592211193568 0.84061941967369014 1.7005986908310777 37.549415588378913 0.77696153631098397 1.2426757812500002 2.9990234375000004 2.5000000000000004 3.6242520361853052 0.89599609375000011 244.50000000000003 5.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 -1 -4 5 6 7 10 -9 -10 -3 13 -13 14 15 -11 19 18 -18 23 21 29 26 -12 25 -17 -22 -28 -29 -21\nright_child=1 4 3 -5 -6 -7 -8 8 9 11 16 12 -14 -15 -16 24 17 -19 -20 20 22 -23 -24 -25 -26 -27 27 28 -30 -31\nleaf_value=-0.020244742155386408 -0.013442727281598605 -0.015433172682240509 0.01377076396477718 -0.013472440910575114 -0.024440859253151783 -0.016772745306493007 0.004526381445312198 0.0093097291776661952 -0.030174440353912927 -0.024384696810958163 0.0016015770531462204 -0.019155644553583397 0.015920998926005608 -0.01852752102491691 0.018754372073865457 -0.013407104278194798 -0.022321997242668926 0.019130926026908968 0.019256249768670554 -0.012299005093724414 -0.020377995212725909 0.020677817655272927 0.017686211372841799 -0.022772173883103413 -0.021831990421638737 0.017082135187650362 -0.0074109031037621516 -0.014680073064331844 0.023308530952746896 0.010534935681246924\nleaf_weight=7.1663907145246553 6.708578634440868 5.4567899459871105 4.4915809673166223 7.5333861514573055 1.0728387041835947 3.0518999115507759 86.225231988005476 7.2659931588323152 4.1532005815995481 3.5828527820108338 63.009719122363776 1.5176708419216982 5.7980166585307416 13.139700549727307 3.2148340951700893 1.323319975702361 6.7698226700449577 1.0543336864493538 0.95036727176693592 10.187142479278917 1.408799187428033 2.7125261263045095 20.549778579326357 2.1426026714739228 0.92708505168866384 3.9788800959140644 4.1743747158581055 0.47001185061617423 4.0750435401109826 2.494184405744817\nleaf_count=93884 32986 10617 514 540 147 18705 23808 4510 4211 1461 19132 165 371 7882 448 752 1915 25 493 453 1111 123 3414 515 453 520 14980 691 1605 45\ninternal_value=-0.000460229 0.000269373 -0.0101365 -0.00302053 0.000672233 0.00086938 0.00114352 -0.000585112 -0.00656594 -0.00979256 0.00159047 -0.00646261 0.00721571 -0.0104116 -0.00148828 -0.00924171 0.00248575 -0.00988078 -0.0155824 0.0035407 0.00778384 -0.00146474 0.0123244 0.000467899 0.00163623 0.00855271 0.000833793 0.006005 0.0174411 -0.00736622\ninternal_weight=286.607 267.416 19.1914 12.025 260.707 259.634 256.582 170.357 44.9016 37.6356 125.455 33.4824 7.31569 26.1667 13.027 9.81214 119.999 8.77452 7.72019 111.224 46.0719 15.3939 30.678 65.1523 6.22929 5.3022 10.1282 8.71943 4.54506 12.6813\ninternal_count=246476 151538 94938 1054 118552 118405 99700 75892 20773 16263 55119 12052 536 11516 3634 3186 44502 2433 2408 42069 22422 621 21801 19647 1725 1272 18387 17276 2296 498\nis_linear=0\nshrinkage=0.03\n\n\nTree=251\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 3 1 6 6 6 19 4 4 3 3 13 2 2 12 4 1 21 17 13 13 18 0 0 13 4 13 4 0 4\nsplit_gain=2.21609 1.21298 1.47171 1.24775 1.42603 1.73919 2.54936 1.71245 1.69098 1.94032 1.82637 2.02987 1.72427 1.62825 1.56221 1.46009 1.65336 2.73909 2.36601 2.02382 1.59423 1.5933 1.60085 3.54393 1.47408 1.34156 1.34043 1.30325 1.43463 1.28369\nthreshold=5.1405116420104324 0.016076142967085191 -0.70140436206657719 4.2421875000000009 2.1386718750000004 1.2426757812500002 237.50000000000003 7.1369628906250009 5.8393554687500009 0.34948566548243237 0.53650715206675215 -0.6866788914798736 -51.587459564208977 2.9882335662841801 0.34440402210285487 1.0693359375000002 2.9477339485081675 2.5000000000000004 1.5107988686755358 -0.23926212085043433 -0.07062347187798422 2.9174053685313028 1.1851094193663554 1.5447487307370646 -0.044585747678852421 1.9753417968750002 -0.58541303710262749 6.6144968271255502 1.1495945197163422 2.7487792968750004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 -3 4 5 7 -7 8 10 13 11 -4 -13 -10 29 16 17 18 -14 26 21 22 -19 -24 -21 -16 -18 -11 -29 -12\nright_child=-2 2 3 -5 -6 6 -8 -9 9 27 14 12 15 -15 25 -17 19 20 -20 24 -22 -23 23 -25 -26 -27 -28 28 -30 -31\nleaf_value=0.012342761685424678 -0.018070859936580193 -0.021229950187002706 -0.020391822848209118 0.0128963188334827 -0.011730016528018498 0.018575072099875061 -0.0092393546135175943 -0.01761779614161077 0.0081417960891776928 0.019181433748994357 0.0058583296413619217 -0.021176048191020175 0.024714765380570759 -0.025306844448985 0.001929006275381197 -0.002959319875587858 -0.018311294552044919 0.020326117881579949 -0.011384609869146408 -0.02528546474149827 -0.022742549967609462 -0.018941249064633701 -0.029415221888860373 0.014071964970494986 -6.2288964400029857e-05 -0.021632062554427098 0.012402996025669688 -0.023515871058645107 0.011914302533730995 -0.0051865709440063442\nleaf_weight=7.0013429575158161 4.6977106807592 1.7804069181858029 4.3793214807624272 6.357840401873772 7.4904745014293885 9.5088223732930128 3.4246913318214629 3.7726605017239754 2.0394886867642295 10.227301783506844 70.657527604780626 2.9216173573638562 9.0739656761768419 1.4639359732036608 4.3687171882847897 93.489786265470684 0.63907030993141223 4.2093292317072137 1.3438690450777802 3.5958576365337649 1.4713359219153961 1.3485764008460126 1.7181707900072067 3.4681510972568503 3.8263628745844454 3.0574700918987219 6.0989026522329368 0.8164631615936736 1.4303242489695547 10.124238488215267\nleaf_count=8425 49274 36761 11944 366 2758 1179 8271 1067 1708 549 3145 19663 11207 233 118 66776 26 1099 5358 125 3023 10972 448 161 182 124 206 116 21 759\ninternal_value=0.000455153 0.00082771 0.000489789 0.000708943 0.000370392 0.000764909 0.0110755 0.000179096 0.000528231 0.00965914 -0.000157631 -0.00231217 -0.00157026 -0.00835088 0.00319121 -0.000972649 0.00402007 0.00890298 0.0194688 -0.00385713 -0.000899238 0.00377381 0.00854256 -0.00276092 -0.0138335 -0.00918745 0.00749992 0.0149617 -0.00423806 0.0044276\ninternal_weight=285.804 281.106 274.105 272.324 265.966 258.476 12.9335 245.542 241.77 15.9775 225.792 137.584 133.205 3.50342 88.208 130.283 36.7936 22.6334 10.4178 14.1602 12.2156 10.7442 9.39565 5.18632 7.42222 7.42619 6.73797 12.4741 2.24679 80.7818\ninternal_count=246064 196790 188365 151604 151238 148480 9450 139030 137963 2627 135336 131190 119246 1941 4146 99583 32807 32268 16565 539 15703 12680 1708 609 307 242 232 686 137 3904\nis_linear=0\nshrinkage=0.03\n\n\nTree=252\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 3 18 15 3 9 2 2 6 3 6 14 2 2 2 13 17 3 5 1 14 2 21 5 5 2 1 19 2 17\nsplit_gain=1.86276 2.00547 3.13806 1.99576 1.98462 1.79684 1.7851 1.71139 1.76484 1.67244 1.53688 1.71455 1.46332 2.25992 3.87763 3.64617 3.38284 2.52294 2.42055 4.48415 2.59853 2.2235 2.18401 2.05932 1.86936 1.77866 2.32424 1.76823 1.73048 1.65295\nthreshold=1.4144231987897788 0.11110445317776836 3.2769667020129059 1.0000000180025095e-35 0.37963534769016083 1.0000000180025095e-35 -12.481266021728514 32.956295013427741 4.3886718750000009 0.62297257604742995 1.4980468750000002 0.5367296668668039 27.147908210754398 1.0000000180025095e-35 -1.7340650558471677 -0.27202883008418016 0.89189169605791641 0.47260169401963864 1.8691406250000002 -0.16863805028123086 0.74140086107412606 -9.8266754150390607 7.5000000000000009 2.1386718750000004 2.7412109375000004 34.417465209960945 1.0240454605967615 245.50000000000003 -3.9017317295074458 1.1384572374241533\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 3 10 -2 7 12 -7 8 -4 -6 11 -3 13 14 17 16 18 21 19 -15 -21 -1 -16 27 -17 26 -14 -23 -19 -29\nright_child=1 2 4 -5 9 6 -8 -9 -10 -11 -12 -13 25 15 22 24 -18 28 -20 20 -22 23 -24 -25 -26 -27 -28 29 -30 -31\nleaf_value=-0.011396623561695182 0.014710898850339507 -0.0043348541886115274 -0.020630527602572113 -0.023531705190527778 0.013299805526343989 -0.0087854686404668092 0.015314074569451476 0.015531844749644816 0.011983330390790626 -0.014536069795228099 0.0069858330189622037 -0.020687099248795863 -0.017991042886790379 0.023207821655786123 0.019428628770438189 0.004600325865993119 -0.032841698665318546 0.0043255508709975502 0.0019041644119761811 -0.031129242459551943 0.00010269387765629784 -0.013115642364788497 -0.015894941618505954 -0.015776396026496942 -0.015960085625475261 0.00043010887897418284 0.017476645949978553 0.02128364429981142 -0.019068066165723489 -0.016093038326119052\nleaf_weight=26.580032135061625 0.75500592342723294 12.346016810202405 5.2431735453004675 7.0084869080121734 15.422875759313685 2.8103304292561679 11.931774782774481 1.4823687893532449 1.2379146004536776 1.3726468149825439 3.5704881126230239 11.081659278947198 1.0819878216025234 1.4194652240944403 12.372074831602614 25.725710198046389 5.3453258448613488 53.204701076463607 12.088239134477133 8.0232218185731217 3.2562398479656158 1.435702339848264 1.0299027800722349 2.5339047402908355 3.6842140936637398 26.805807763027133 12.242432258793995 6.6390895195370376 2.0233498144298201 0.49430762757947366\nleaf_count=81224 104 2282 39174 62394 4931 2154 799 1975 131 274 161 3352 2026 3129 2057 8980 2839 2475 3035 2498 2590 3325 105 2112 752 5147 670 4138 174 511\ninternal_value=-0.000590928 -0.00525422 -0.00294469 -0.0185583 0.0046782 0.000684584 0.0104537 -0.00707612 -0.013632 0.0103359 -0.00985275 -0.0126 -6.17015e-05 -0.00131076 0.00131274 -0.00593932 -0.0130978 -0.000922807 -0.00773161 -0.0163734 -0.0228389 -0.00681119 0.0157989 0.00420074 0.00158457 0.00500692 0.0135416 0.0111394 0.00314462 0.0170334\ninternal_weight=280.248 59.5206 51.7571 7.76349 24.759 220.728 14.7421 7.96346 6.48109 16.7955 26.9982 23.4277 205.986 165.855 106.313 59.5424 30.1325 92.9111 24.7872 12.6989 11.2795 37.683 13.402 11.103 29.4099 40.1302 13.3244 8.5691 55.2281 7.1334\ninternal_count=245518 114778 52280 62498 46485 130740 2953 41280 39305 5205 5795 5634 127787 119944 96121 23823 14091 93959 11252 8217 5088 91310 2162 10086 9732 7843 2696 7974 2649 4649\nis_linear=0\nshrinkage=0.03\n\n\nTree=253\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 14 18 13 7 18 7 7 13 18 18 14 17 5 14 3 18 18 17 17 7 5 5 3 1 5 5 18 14 3\nsplit_gain=2.1787 1.65739 2.05733 2.29213 1.68423 2.23554 2.54833 2.47218 2.33433 3.83851 1.99277 2.11224 1.88181 2.09064 2.91373 2.14148 2.72475 2.09478 1.77569 1.46946 1.6352 1.56351 2.0064 1.44273 1.75755 2.59459 2.2719 1.86378 1.85414 1.74045\nthreshold=0.31033658837561479 0.59283710889839769 4.2121001622835372 -0.11458465830609064 0.89599609375000011 1.7005986908310777 1.8691406250000002 3.2382812500000004 -0.26766225074600319 1.8688348091416842 3.8815106545092593 0.24319921965375105 0.86664248603069383 0.89599609375000011 0.070321542204431389 0.48555671035752257 0.89587973461402759 1.4978661367769956 0.61464819824310435 3.9653415636860969 2.6015625000000004 3.7958984375000004 4.8398437500000009 0.3261289997719386 1.8223781763416411 5.1621093750000009 1.7006835937500002 4.4829865921051741 0.84146630540292722 0.47260169401963864\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=19 2 4 -4 12 -6 8 -8 9 -7 11 -10 18 15 -15 16 -14 -17 -2 20 -1 -22 -23 -3 25 26 29 -26 -28 -25\nright_child=1 23 3 -5 5 6 7 -9 10 -11 -12 -13 13 14 -16 17 -18 -19 -20 -21 21 22 -24 24 27 -27 28 -29 -30 -31\nleaf_value=-0.0075074565417037088 -0.0042088602116967314 0.015482257749622155 0.016652979011651092 -0.007470646689451005 0.01088933695335479 -0.02881907502403501 -0.025650239081776431 0.006162837496924165 0.0013350185389517256 0.014579564309507152 0.013187054725495964 -0.025760924562811108 0.022644093931019897 0.016796968040577034 -0.017190375639606788 -0.0063191163999562041 -0.01769708456218879 0.031181192270258902 -0.023857099622413058 -0.021286427469477377 0.018246929929185314 -0.01615401526706628 0.0069861989335761382 -0.020212068354393341 0.00035476483040997601 0.0096913263891418872 -0.034510766387201658 -0.021847927316848068 -0.0076405687655544795 0.0082359067751088047\nleaf_weight=38.617328326539621 19.932057683249699 3.0018997731017398 15.532989516742647 3.8820406043703306 26.324753905297257 1.1170580950856699 4.9940411556053723 2.9583064592443407 4.8590546686100442 16.546828104072119 2.7292846298623763 4.3238083568994625 1.4056552364199904 2.4127432834584379 6.1167863346709055 1.3779989207978358 3.0717492783915077 4.5347411036236736 4.4492173407061264 4.0678280734407499 3.7745966274392826 4.4109333295748439 7.9920496957361138 1.9068619301708682 70.788611095132183 3.9445822024254076 5.3810662250766637 2.5731332467630628 4.7565648454628899 4.8120136329316665\nleaf_count=138767 1179 2292 4550 1052 269 12 188 74 128 459 111 187 16 99 790 35 658 157 654 60113 7833 6182 2455 5965 2913 231 2660 1295 3448 1838\ninternal_value=-0.000641791 0.000715326 0.00296751 0.0116998 0.00129178 0.00436157 -0.000381193 -0.0144492 0.00375024 0.010438 -0.00620702 -0.0127002 -0.00323132 0.00376273 -0.00682263 0.0122886 -0.00321032 0.0227917 -0.00859305 -0.00572589 -0.00420991 0.00364042 -0.00183452 -0.00221877 -0.00293984 -0.0104199 -0.0154053 -0.000707319 -0.0237213 -0.00141037\ninternal_weight=282.597 223.734 126.569 19.415 107.154 63.8531 37.5284 7.95235 29.576 17.6639 11.9121 9.18286 43.3009 18.9197 8.52953 10.3901 4.4774 5.91274 24.3813 58.8627 54.7949 16.1776 12.403 97.1647 94.1628 20.8011 16.8565 73.3617 10.1376 6.71888\ninternal_count=246610 31260 10618 5602 5016 1428 1159 262 897 471 426 315 3588 1755 889 866 674 192 1833 215350 155237 16470 8637 20642 18350 14142 13911 4208 6108 7803\nis_linear=0\nshrinkage=0.03\n\n\nTree=254\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 1 0 4 3 18 6 3 4 6 3 1 3 1 0 14 13 8 4 14 21 3 14 13 3 13 2 4 2\nsplit_gain=2.57104 1.87978 3.79084 2.59704 3.44448 2.94386 2.91769 2.48213 2.01894 2.00716 1.76614 2.08327 2.5879 2.20548 1.50198 3.13381 2.53289 1.98408 1.8023 1.74152 2.88344 3.27224 2.04873 1.64779 1.5716 1.74873 1.4863 1.47767 1.43997 1.3897\nthreshold=-55.708036422729485 1.4475910192202701 1.075454090076706 0.85792061509615258 3.9316406250000004 0.39370400768236996 2.1383330595080281 3.4804687500000004 0.28904431203898151 1.6572265625000002 2.9179687500000004 0.36709515926886871 1.6575619461558879 0.41780940659066851 2.3713805701640611 1.1851094193663554 0.45703470522544881 -0.25939487280712015 1.0000000180025095e-35 2.2978515625000004 0.75822041467236145 2.5000000000000004 0.53650715206675215 0.2247238965590572 -0.36770112406875188 0.51817819236196982 -0.75180196954606593 22.129880905151371 12.005371093750002 -4.708568572998046\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 3 4 -2 7 9 -6 -9 -5 11 12 -3 -13 15 16 18 19 -15 26 21 27 -23 -19 -17 -26 -16 -21 -11 -8\nright_child=1 10 -4 6 5 -7 29 8 -10 28 -12 13 -14 14 17 24 -18 23 -20 20 -22 22 -24 -25 25 -27 -28 -29 -30 -31\nleaf_value=-0.020911008164459507 -0.026110400780466669 0.010227602447115248 -0.02024379129660029 0.0081940265542862888 0.010360793728277477 -0.025869541962115935 -0.0073573171928216358 -0.028795373739364853 0.011439028898098976 -0.027678748753799956 0.017392428851742116 0.010891968232725556 -0.0091000639182951659 -0.020130334993050846 0.025192795831376003 -0.023511135129889393 0.0074044875329144569 0.0039736769369359947 0.017362861132632666 -0.032008726936171752 -0.016627705074870377 -0.0057131409114764076 0.01686515199576695 -0.0079359398538921561 -0.015592210877247631 0.010381179269459978 0.0069995119625518255 0.0050430664122117887 0.0036942919820843901 0.018723151345481144\nleaf_weight=4.3766234172578029 5.6201401722306654 6.854040431442626 10.460295165492537 3.9810466913459113 12.708319210136041 2.4944684291240256 1.6145620371844422 2.1766222327340752 1.0626465091227144 3.0988396544194075 4.9686165137245544 21.970005931154446 31.154356077870943 4.9557746539685494 3.7206342282588585 7.3376890692743473 20.044998427062637 18.151704875774158 0.62079119682312001 1.7621940773970046 6.4691993919041115 4.4838654822524431 11.658420850406399 22.256947584755835 4.0551711740554293 3.4029506752049201 44.340314727600344 1.1231809204909948 1.6263713524676857 11.200489081977363\nleaf_count=55560 122363 2606 10281 2611 22377 1398 4342 483 33 1487 126 1923 6167 554 58 229 2689 331 1 44 209 94 104 765 112 91 1158 18 68 7583\ninternal_value=-0.000365238 3.57831e-05 -0.00481807 -0.00091966 -0.00754535 -0.000835744 0.00649439 0.00437536 -0.0160116 -0.00648379 0.00129229 0.00084008 -0.00544187 0.00221704 0.000920377 -0.0039854 0.00240697 0.00267971 -0.0139511 0.0055868 -0.00074524 0.00521524 0.0106261 -0.00261933 -0.0145065 -0.00391499 0.00889274 -0.0199999 -0.0181213 0.0151426\ninternal_weight=279.751 275.375 56.0438 45.5835 24.0622 18.4421 21.5213 15.9476 3.23927 8.70626 219.331 214.362 38.0084 176.354 154.384 40.4174 25.6216 113.966 5.57657 73.5578 25.4969 19.0277 16.1423 40.4087 14.7958 7.45812 48.0609 2.88537 4.72521 12.8151\ninternal_count=245865 190305 173026 162745 146654 24291 16091 22893 516 4166 17279 17153 8773 8380 6457 3676 3244 2781 555 1685 469 260 198 1096 432 203 1216 62 1555 11925\nis_linear=0\nshrinkage=0.03\n\n\nTree=255\nnum_leaves=31\nnum_cat=0\nsplit_feature=7 4 2 18 18 17 3 4 5 5 1 1 4 2 2 21 1 2 14 17 18 19 5 2 2 2 0 1 14 2\nsplit_gain=2.5366 1.93268 2.20567 2.15498 2.46814 2.10891 2.05109 1.88514 2.74908 2.12558 2.07159 2.47045 2.02455 1.83415 1.83695 1.82457 1.79676 1.75273 2.24803 1.7316 4.81035 1.57755 1.55866 1.55486 2.0999 2.60876 2.02085 3.35061 1.73353 1.41959\nthreshold=0.89599609375000011 2.1750488281250004 30.514518737792972 2.0126758453675748 1.7005986908310777 1.0055080915597503 0.24931662397384338 3.2895507812500004 0.89599609375000011 1.7006835937500002 -0.69268724735312648 2.2520445226790948 1.9406738281250002 15.156654357910158 -8.7471351623535138 7.5000000000000009 -0.041910702480567281 -27.96581935882568 0.032916404154818342 1.5700365906568221 0.89587973461402759 248.50000000000003 0.89599609375000011 34.417465209960945 -1.7340650558471677 -18.054428100585934 1.8987378677022575 2.054932859494814 0.33900667957710101 14.537487030029299\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=7 6 3 4 5 13 -2 8 15 10 -10 -12 -11 14 -3 19 -5 -13 -19 23 -21 -20 -8 24 25 -1 27 28 -27 -7\nright_child=1 2 -4 16 -6 29 22 -9 9 12 11 17 -14 -15 -16 -17 -18 18 21 20 -22 -23 -24 -25 -26 26 -28 -29 -30 -31\nleaf_value=0.0056716354914071983 -0.019692136629392547 0.0070366431517471803 0.0065486052473723109 -0.017866935788431342 -0.027250041780782941 -0.001451769545729349 -0.018720034888389237 0.013330893216397261 0.013988037481589666 0.020407763464809027 -0.031156627912523788 0.0065705727202925567 -0.0071243744105746031 0.013636862885188056 -0.01721488343700734 -0.01671171561973887 0.0013154832800749685 0.012177949196872737 -0.029934917399528745 0.024123255043272007 -0.024993992798319764 -0.0061084912215809582 0.0090488503783027596 -0.013481940115631526 0.011086957342369334 0.020324099114899772 0.012485367092808829 -0.029591057364402047 -0.016349695517717634 -0.030003527291037106\nleaf_weight=17.085673785016525 1.5938310239555438 4.3769316175166777 30.769880020527694 3.6851569172737859 4.7867853740585806 2.7338308702273926 0.98502992380463084 4.2613390451075466 1.2083801950620898 2.1363443550862931 8.076451123195687 3.6364454390909495 9.3038708344800494 5.6349185180151826 5.1862021223860237 5.3076502340735479 69.889286700764387 1.5967478089905824 5.7495426494560871 1.5758064618330525 5.603395690555999 4.9847826240365976 44.160797524590407 4.1513525395588422 19.805329276596584 1.9191227143561458 1.6662808929686441 5.9293777680795756 1.0012831252651038 3.042406793225382\nleaf_count=15578 9423 101 3949 60584 316 143 463 2142 8305 1290 40492 384 19521 76 145 6371 31277 70 647 127 22559 338 3046 1664 9755 1667 23 374 4932 112\ninternal_value=-0.000643213 0.00154871 -0.000343116 -0.00252187 -0.0098119 -0.00477316 0.00671172 -0.00431475 -0.00518488 -0.0116156 -0.0163192 -0.0183401 -0.000755844 0.00134709 -0.00649989 -0.00137869 0.000127983 -0.0104033 -0.0155263 0.000263395 -0.0125817 -0.0203339 0.00805042 0.00225735 0.00388566 -0.00148853 -0.0126041 -0.0181161 0.0067871 -0.0186987\ninternal_weight=281.844 176.845 130.105 99.3355 25.7611 20.9743 46.7397 104.999 100.738 36.6926 25.2523 24.044 11.4402 15.1981 9.56313 64.0453 73.5744 15.9675 12.3311 58.7376 7.1792 10.7343 45.1458 51.5584 47.4071 27.6017 10.5161 8.84978 2.92041 5.77624\ninternal_count=245874 109635 96703 92754 893 577 12932 136239 134097 71047 50236 41931 20811 322 246 63050 91861 1439 1055 56679 22686 985 3509 33993 32329 22574 6996 6973 6599 255\nis_linear=0\nshrinkage=0.03\n\n\nTree=256\nnum_leaves=31\nnum_cat=0\nsplit_feature=10 17 16 19 7 5 2 9 17 2 2 2 13 13 19 17 2 13 2 2 2 14 21 2 4 2 7 2 2 2\nsplit_gain=1.88227 1.60796 1.55779 2.96164 1.90892 1.37086 2.96078 3.22245 1.22976 1.14665 2.5702 3.28257 1.88404 1.86156 1.6252 1.37253 2.99777 1.34892 2.32843 1.72511 1.60487 1.72192 1.59047 1.34526 1.34463 1.81255 1.38713 1.722 1.31061 2.42761\nthreshold=3.6242520361853052 5.817026225953116 1.0000000180025095e-35 234.50000000000003 1.4980468750000002 4.9648437500000009 -15.657813072204588 1.0000000180025095e-35 2.1677456597340812 28.11904335021973 -1.3165187835693357 -1.7340650558471677 -0.53246711157653859 -0.31691436769765968 258.50000000000006 1.3781395569682247 37.549415588378913 -0.52938858054062321 -9.4683709144592267 -21.99824237823486 -6.7439365386962882 0.65891777326083356 6.5000000000000009 39.736396789550788 1.0693359375000002 -26.006598472595211 2.6015625000000004 -17.667204856872555 1.2144355773925783 -1.0000000180025095e-35\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 8 6 -6 -8 -4 10 11 13 -13 17 24 -11 -17 18 19 -5 22 -22 -20 -18 25 -15 27 -26 29 -12\nright_child=-2 -3 4 9 5 -7 7 -9 -10 15 28 12 -14 14 -16 16 23 -19 20 -21 21 -23 -24 -25 26 -27 -28 -29 -30 -31\nleaf_value=-0.012229456520807223 -0.027286149989640591 -0.017114862095445656 0.0053688065399951617 0.00042115221327614294 0.017098533846507509 0.010075793351162578 -0.021990488399076542 0.0076829535613692369 0.029818667076009834 0.008998675168630994 -0.0023967702370187756 -0.010620189709739401 0.029530067845599171 -0.012156182639417767 -0.020957254879591408 -0.024616626532121978 0.024982218324911327 0.0083778084602166032 -0.0004893027165471641 -0.01747498950710619 0.01668448787331623 -0.01885566152223259 0.03457417302858766 -0.0021594304383442896 -0.00036321599892372142 0.011473178473092392 0.011467965216808573 -0.016571891675192862 -0.0029364876685195667 -0.03219586585105004\nleaf_weight=16.561061190969895 1.2603906378953968 3.7449086188899505 4.0948104015851632 14.485987313574928 3.8085181929031924 10.904296045299132 7.2598164853525322 4.8069731532341393 3.4572953611836965 25.267471078506219 6.4232966814689689 0.75157717271895297 4.7508710389500939 3.0021142390710329 3.3376882000399055 2.9858224068661894 1.6567366856542092 34.885666569355067 1.6871778870231535 6.2519015056532234 1.2644039196770784 1.7121261304703237 2.5685407955279516 3.9397052137838955 12.453371184831438 10.637717954411299 2.8903560736529235 10.287283234237071 67.88355414493401 3.1765757230684795\nleaf_count=25088 204 42524 2249 19403 4025 1143 1343 295 407 4827 2052 151 537 7555 4502 2071 319 33012 741 3116 125 1584 80 2727 29979 6409 7057 9525 30534 2123\ninternal_value=-2.08199e-05 0.000200996 0.000499915 -0.000368263 0.00637967 0.00234493 -0.00324759 -0.0104863 0.0187397 0.000566191 -0.000356446 0.00256333 0.0232561 0.00127541 -0.00351671 0.00556558 -0.00442623 0.00452808 -0.000396924 -0.00552974 0.0132049 -0.00335934 0.0232251 0.00844459 -0.00155081 0.00579728 -0.00553203 -0.00808479 -0.00451734 -0.0143644\ninternal_weight=278.198 276.938 273.193 238.861 34.3317 26.7796 15.8753 12.0668 7.55211 222.3 188.45 110.967 5.50245 105.464 42.6085 33.8497 8.58226 62.8558 27.9701 20.7379 7.23225 2.97653 4.25572 5.59644 39.2708 13.6398 25.631 22.7407 77.4834 9.59987\ninternal_count=245707 245503 202979 193517 9462 6806 5663 1638 2656 168429 158485 123776 688 123088 65027 9944 5117 58061 25049 22519 2530 1709 821 3046 60525 13964 46561 39504 34709 4175\nis_linear=0\nshrinkage=0.03\n\n\nTree=257\nnum_leaves=31\nnum_cat=0\nsplit_feature=10 17 3 2 6 7 6 6 19 7 5 3 5 2 18 18 3 18 3 18 7 21 17 3 3 3 5 3 2 7\nsplit_gain=2.11575 1.89982 1.87868 1.53839 0.962703 2.0554 1.42163 1.30984 3.70192 1.28137 2.01268 2.37485 1.61516 3.55857 4.65919 2.2029 1.59458 1.65757 1.54409 1.52494 1.4862 1.44745 1.42601 1.29424 1.22846 1.68746 1.71236 1.35129 1.21207 1.20022\nthreshold=3.6242520361853052 5.817026225953116 0.016076142967085191 21.425971984863285 3.3837890625000004 3.6767578125000004 2.2500000000000004 1.2426757812500002 237.50000000000003 3.1132812500000004 4.4238281250000009 0.48555671035752257 2.1386718750000004 11.680273056030275 4.1510089048756029 4.3757372435704527 0.23120698015535426 2.249904835165133 0.25176753112663547 5.803626672146776 2.6738281250000004 4.5000000000000009 0.42804718369210387 0.53650715206675215 0.27986503791205852 0.53650715206675215 3.9023437500000004 0.23120698015535426 -1.7340650558471677 4.5273437500000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 5 6 7 9 -9 12 28 -12 16 14 21 -16 17 -4 -18 -17 24 -14 -19 -22 27 -26 -27 -15 -11 -7\nright_child=-2 -3 4 -5 -6 29 -8 8 -10 10 11 -13 13 20 15 19 18 22 -20 -21 23 -23 -24 -25 25 26 -28 -29 -30 -31\nleaf_value=-0.0053750471808811876 -0.028690473664248858 -0.0187403563362285 -0.022002387361015854 0.022649786963110808 0.0094657088591900539 -0.025488327866635536 -0.019578765749782106 0.019281485738090052 -0.014022220729495369 -0.0092795160336810126 0.0088818115318597055 -0.027136374035246557 -0.032405004276710475 0.002975252795899554 0.019466550707366345 -0.021094615095420287 0.017303802871942615 0.012267680402124429 0.0011827710765743105 0.0049514584955808643 -0.026357675361761444 -0.0067385181898541157 -0.010038719124251408 0.0044035309767167874 0.013253698553641649 -0.012332112802082263 0.017934550708703131 -0.025564824548920156 0.027866612480047025 0.010438061353964309\nleaf_weight=2.0194089144082445 1.2593230720979125 3.5660007067571646 4.327033988529819 5.0846656629148583 9.025083891219877 3.2166940162113629 2.1832105503070798 9.0158739020738476 3.4537366282868325 0.44128804380170539 4.6907462065606751 1.3231285393048891 8.2566360290120446 3.565555332240371 2.9749835992222398 2.9620340611306801 4.5368349383199975 3.4887709647920229 165.3051415670742 3.5150030949287725 1.9103285939490882 3.3108961772642251 5.0673116216786411 1.7056583448029412 13.91367438937232 5.1606687176190453 1.3513709584367459 1.3669574415553141 3.9906518524294352 0.4609786814560265\nleaf_count=5906 195 42493 2852 3549 974 1343 2014 1900 18052 605 1626 167 3215 2540 1081 16517 5792 24897 34630 10747 391 996 60986 141 1368 359 24 379 165 37\ninternal_value=0.00022634 0.000457057 0.000770017 0.0150032 0.000342363 -7.37519e-06 0.000348859 0.000594675 0.00970107 8.98755e-05 0.00989016 -0.00178178 -0.000390454 -0.00511272 -0.014208 0.00154587 0.000922359 -0.00886319 0.00171216 -0.00818797 0.00173936 -0.0261794 -0.000611258 -0.0140371 0.00443622 0.00764773 -0.00449988 -0.00791017 0.0231406 -0.0197164\ninternal_weight=282.45 281.19 277.624 7.10407 270.52 261.495 257.817 255.634 12.4696 243.165 10.4458 6.01387 232.719 49.9938 21.0196 9.45202 182.725 12.8831 169.842 6.47704 28.9742 11.5675 8.55608 3.61599 25.3582 20.4257 6.51204 4.93251 4.43194 3.67767\ninternal_count=245941 245746 203253 9455 193798 192824 191444 189430 19952 169478 2563 1793 166915 37758 32556 28345 129157 88735 40422 27264 5202 4211 85883 532 4670 1751 383 2919 770 1380\nis_linear=0\nshrinkage=0.03\n\n\nTree=258\nnum_leaves=31\nnum_cat=0\nsplit_feature=5 1 3 17 0 17 2 2 1 1 2 19 3 2 3 5 21 0 18 0 17 3 3 18 17 0 0 0 7 21\nsplit_gain=1.82495 1.65679 1.46046 1.5398 1.63439 1.89625 1.44884 1.23283 1.42505 1.97385 1.60757 2.28802 1.62754 1.61116 1.37306 1.31061 3.51633 1.8228 3.68866 3.38061 3.12651 2.72789 2.47786 2.35275 1.7947 1.78654 2.21297 1.77456 2.11633 1.65228\nthreshold=1.0000000180025095e-35 4.0016611463566001 0.50207005315878861 0.091221764698383404 1.8987378677022575 0.45436599119170595 -59.476831436157219 29.286581993103031 -0.29568658027461547 -0.016381246029292276 22.867874145507816 249.50000000000003 0.65908129556435835 22.129880905151371 0.62297257604742995 1.2426757812500002 3.5000000000000004 0.52258841453381233 4.1820210059610323 0.84505259813102007 0.92594565335601631 0.24463700593768659 0.44825707527455844 2.3502401828962087 1.3324302935777734 0.13255531789395347 1.5447487307370646 1.6765212893675572 0.89599609375000011 6.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 6 3 -3 5 -5 -2 8 -8 -10 13 12 -12 15 -9 16 17 -11 19 -19 22 -22 23 -21 -23 -18 -27 -28 -29 -17\nright_child=1 2 -4 4 -6 -7 7 14 9 10 11 -13 -14 -15 -16 29 25 18 -20 20 21 24 -24 -25 -26 26 27 28 -30 -31\nleaf_value=0.023507230896396752 -0.02133514148155188 -0.021490972205032457 0.0095194462790976573 0.018501896612148652 -0.016407146908721686 -0.0024656726295690275 -0.017146338601695007 0.0054201595956075198 0.015831763085467982 -0.016922348954392763 -0.028681906602401511 0.0046822387352251039 0.0095076224152740689 0.017640487135456559 -0.010011507712487894 -0.0012473371418723115 0.014012008551543659 -0.024639930861571659 0.032989598593310632 -0.013878535048455097 -0.020273194198436838 0.031823840900816774 -0.023841943442844116 0.020038311385538816 0.0020255973128455187 -0.02023487115995402 0.01122414411815085 -0.022448404437201374 0.010328823048510761 0.0087491966085823693\nleaf_weight=2.0640969737276 2.0619038396665124 1.8717727566836426 28.069037497916725 5.5778704028343808 2.5291724104899904 8.9823671782505681 4.2279767815321057 43.157996807060499 4.9053818177915209 2.726726543442342 5.194881200682401 3.7276750356520552 0.69868247343401879 3.048648320362644 4.9512199963264711 84.758214777065973 1.4216060085434516 2.9406042428714789 2.7582624218066476 2.3436681380899262 0.74226371751274101 5.3278238173825212 3.2404705227236263 3.2238773984986437 2.7541447624153079 17.402597539313437 4.0866746851024782 4.1947489148160457 1.9716494890308238 17.008672209413817\nleaf_count=387 50539 78 433 102 39 329 65635 21324 24681 12625 2783 585 63 491 347 35785 667 5638 729 439 830 342 200 922 635 13232 88 131 17 5758\ninternal_value=0.000482021 0.000225496 0.0052923 -0.00112874 0.00216625 0.00623961 -0.000834139 -0.000554937 -0.00170382 -0.00123403 -0.00181763 -0.0131241 -0.0234321 -0.00106022 0.00366047 -0.00153299 -0.0052134 0.00275758 0.00565857 0.000634857 0.00594624 0.0176766 -0.00640975 0.00582168 0.0230104 -0.0122109 -0.0140007 -0.00256167 -0.0119893 0.000492299\ninternal_weight=277.971 275.907 47.0302 18.9612 17.0894 14.5602 228.876 226.814 178.705 174.477 169.572 9.62124 5.89356 159.951 48.1092 156.902 55.1351 26.0578 23.3311 20.5729 17.6322 8.82423 8.80802 5.56755 8.08197 29.0773 27.6557 10.2531 6.1664 101.767\ninternal_count=245854 245467 981 548 470 431 244486 193947 172276 106641 81960 3431 2846 78529 21671 78038 36495 22360 9735 9006 3368 1807 1561 1361 977 14135 13468 236 148 41543\nis_linear=0\nshrinkage=0.03\n\n\nTree=259\nnum_leaves=31\nnum_cat=0\nsplit_feature=4 18 4 4 13 18 1 1 18 1 14 14 3 18 19 3 1 13 14 18 21 1 18 12 11 4 4 18 3 13\nsplit_gain=1.80147 2.61085 2.58934 2.70965 2.5874 2.22835 1.88677 1.80023 2.14424 1.97143 1.88782 1.8029 1.52895 1.48458 1.71788 1.96003 2.28817 2.26385 1.72684 1.84717 1.47611 1.32231 1.27292 1.2224 1.97209 1.50671 1.21171 3.08723 2.06539 1.91992\nthreshold=1.0693359375000002 4.3882378946731615 1.6286621093750002 1.6572265625000002 -0.51849643425434 3.5114340430413216 4.0016611463566001 0.45969269984116679 2.9174053685313028 1.4866206336930319 0.043928974430191954 0.050734622491290134 0.40911314460322756 4.0342014792848504 264.50000000000006 0.43673077644757391 2.9477339485081675 -0.36630902128874782 0.064908752851949539 2.8029010331479989 4.5000000000000009 -0.59079332554470076 2.249904835165133 0.030057938850762859 0.59460817960898271 2.9858398437500004 5.5515136718750009 3.0201273556387074 0.32220753877234559 -0.53246711157653859\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 7 4 5 -2 20 -6 8 21 10 -9 -11 -13 14 15 -14 -17 -18 -19 -20 -4 -1 -8 26 -25 -26 27 -5 -29 -30\nright_child=2 -3 3 23 6 -7 22 9 -10 11 -12 12 13 -15 -16 16 17 18 19 -21 -22 -23 -24 24 25 -27 -28 28 29 -31\nleaf_value=-0.0092520992229965706 0.0064135860666627419 -0.016104519359650005 0.019619157498432214 0.00092199293783855319 -0.027426170784095063 -0.011065631599683783 0.010280814403293924 0.015909310542540543 -0.013622765518596586 -0.020786702596635558 -0.025775194067469476 -0.0050759927567905309 0.025831416590564878 0.026383498391890615 -0.015681942198198004 -0.0040547758083979797 0.02025836662398604 0.027165248697886325 -0.012190491258926494 0.014003614747851998 -0.0044537737815788549 0.031212134082189109 -0.020277207244952671 -0.020221269659379933 -0.01607719575868035 0.0042110832088066022 0.0039975368923863292 -0.022818592493978235 -0.021405848080344618 -0.00087027361842772707\nleaf_weight=0.38331851779776105 4.0579596266587705 4.2844015799124113 11.641858866003988 83.754461191025712 7.6521765725507276 2.7494930588544539 2.4533018132206057 0.58640485597084024 0.7873817295618436 1.3006464209361173 2.5556211705408032 7.8636241822450748 4.1197700265174708 3.0586935044111661 1.9608229068653589 13.155487733794869 11.079178606509233 0.88943298716912877 6.550979425548574 2.6157075379742309 2.4013358297693221 4.7057665360030105 0.90396230531041499 6.835016526898019 3.5475216016075128 11.018799399553243 38.168509414502012 7.4596014212866093 4.2102788783447549 19.622353320372213\nleaf_count=931 7454 29898 394 6699 29981 2715 31 434 17753 215 7216 2039 195 265 152 697 142 4 136 63 109 2721 86 6220 202 589 40408 78893 4347 5061\ninternal_value=5.23483e-06 0.0042862 -0.00137511 -0.000401972 -0.012943 0.0107563 -0.0200176 0.00593868 0.0204962 0.00406963 -0.0160328 0.00554731 0.00659978 0.00878051 0.00684136 0.00835989 0.00558834 0.0115026 0.000896313 -0.00407381 0.0154803 0.0271454 -0.000712439 -0.00153019 -0.00807689 -0.00144538 -0.00056929 -0.0021058 -0.00998735 -0.00521401\ninternal_weight=272.374 65.8972 206.477 191.409 15.0674 16.7927 11.0094 61.6128 5.87647 55.7364 3.14203 52.5943 51.2937 43.4301 40.3714 38.4106 34.2908 21.1353 10.0561 9.16669 14.0432 5.08909 3.35726 174.617 21.4013 14.5663 153.215 115.047 31.2922 23.8326\ninternal_count=246050 62861 183189 145637 37552 3218 30098 32963 21405 11558 7650 3908 3693 1654 1389 1237 1042 345 203 199 503 3652 117 142419 7011 791 135408 95000 88301 9408\nis_linear=0\nshrinkage=0.03\n\n\nTree=260\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 18 17 14 5 14 4 4 1 0 1 10 1 1 0 18 18 14 3 1 18 18 17 4 14 17 14 18 17 18\nsplit_gain=1.82818 2.13687 3.10391 2.31107 2.1947 1.95643 1.69191 3.48212 2.41503 4.77916 2.72468 2.00325 1.90197 1.72125 1.56974 1.4968 3.06076 2.77489 1.32674 1.89957 1.31312 2.19349 3.86978 3.68819 3.16503 2.86821 1.71615 1.69042 1.60818 1.92844\nthreshold=0.53650715206675215 0.89587973461402759 0.32406367513646889 0.11808058947925405 1.2426757812500002 0.032916404154818342 2.8684082031250004 1.7380371093750002 0.39417715631504779 0.84505259813102007 1.4130033503388202 3.6242520361853052 1.6575619461558879 2.6981674883069502 1.5447487307370646 1.7005986908310777 0.89587973461402759 0.32729182297007969 0.45898141303402712 4.0016611463566001 2.249904835165133 4.2121001622835372 0.96280603246018803 2.3505859375000004 0.78260893289928579 0.70050815440916869 0.56086626140189244 2.9174053685313028 1.2133651393498066 2.1383330595080281\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=6 2 -2 5 -4 -3 7 8 9 -1 -10 -8 -9 -14 -11 16 17 -12 19 -18 28 22 24 25 -22 -23 -24 -26 -5 -30\nright_child=1 3 4 20 -6 -7 11 12 10 14 15 -13 13 -15 -16 -17 18 -19 -20 -21 21 23 26 -25 27 -27 -28 -29 29 -31\nleaf_value=-0.020326532524172734 0.013066335501153898 -0.0030426781094220604 0.0020947051045354672 0.0092717809843120679 -0.025917888495353075 0.029722541101801017 0.0019232716934382099 -0.025458601522032283 -0.024440080756729984 0.029539649191512054 -0.015296827248311863 -0.026931605838668816 -0.0013078642785069729 -0.015053297932674695 -0.010611888368482276 0.0029179483405877699 -0.016320345720039417 0.018449325797969061 -0.028359400652319938 0.012534497939259778 -0.023492754324044181 0.026750812408494435 0.019098608265408181 -0.020156526977860557 0.013837294042902437 -0.010671808635817256 -0.0036801129661143709 -0.013596537946414447 -0.011446407124608825 0.020192681731682052\nleaf_weight=1.9972134845669112 3.713230811757966 2.0937994855121351 3.6367490626289491 34.044650962750893 6.2654099520295858 4.4867061346376431 72.520718057678266 6.7145422393536753 3.8173864062251814 5.9171862533630701 2.5067561344112645 1.2326097727000149 16.467057412689883 15.753322846449008 0.49545581347501855 33.925684995856955 5.1117219209954765 5.1268579166244308 4.2295603330130689 2.2176517809275538 10.606834806108969 6.6561422286013094 5.1426188005971181 2.0197890826984795 3.7953783529374041 1.8231590981258703 5.2508476150214847 2.5170716286465904 6.9996567657071846 1.2502373939205425\nleaf_count=67141 46 83 50 369 56 38 79918 49886 24318 7588 48 201 2459 777 547 8511 314 114 108 31 510 471 127 867 133 591 759 407 197 29\ninternal_value=-0.000563274 0.00265974 -0.00800573 0.00440709 -0.0163813 0.0202711 -0.0023838 -0.00482571 -0.00055187 0.0135537 -0.00283337 0.00108721 -0.0118052 -0.00827917 0.0254244 -0.000857668 -0.00734562 0.00687941 -0.0165397 -0.00713304 0.00287004 -0.00118568 -0.00564515 0.00989742 -0.0141826 0.0184275 0.0082777 0.00253473 0.00643955 -0.00498698\ninternal_weight=278.336 100.302 13.6154 86.6869 9.90216 6.58051 178.034 104.28 65.3455 8.40986 56.9356 73.7533 38.9349 32.2204 6.41264 53.1182 19.1925 7.63361 11.5589 7.32937 80.1064 37.8118 27.3128 10.4991 16.9193 8.4793 10.3935 6.31245 42.2945 8.24989\ninternal_count=246694 4733 152 4581 106 121 241961 161842 108720 75276 33444 80119 53122 3236 8135 9126 615 162 453 345 4460 3865 1936 1929 1050 1062 886 540 595 226\nis_linear=0\nshrinkage=0.03\n\n\nTree=261\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 10 7 7 0 18 0 21 18 2 1 1 2 19 14 18 2 18 18 1 0 18 5 13 13 6 17 19 17 14\nsplit_gain=1.45224 1.30876 1.73142 1.88376 2.22171 3.50932 3.26664 2.69659 2.39281 2.20332 2.01801 1.80635 1.77375 2.11998 1.76682 1.81638 1.95353 2.29538 2.02739 1.94478 1.72874 1.62198 1.59966 2.30366 2.27458 1.63806 1.5918 1.51834 1.44289 1.41058\nthreshold=-0.90051303354301027 3.6242520361853052 3.1132812500000004 1.2426757812500002 0.48517496193768689 4.1820210059610323 0.75738199555027408 3.5000000000000004 4.7405652883788738 -14.450672149658201 1.5275240303550033 0.6973310998193073 2.9882335662841801 256.50000000000006 0.32729182297007969 2.8617925509761908 15.156654357910158 0.89587973461402759 0.89587973461402759 2.6981674883069502 1.5447487307370646 1.4978661367769956 4.7402343750000009 -0.64803833042221404 -0.32597037876220913 4.0507812500000009 0.69259390652607344 253.50000000000003 1.2596354777617942 0.33260346916079953\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 3 4 8 6 12 28 -2 -10 -11 27 -6 -14 20 16 17 -16 -18 21 -5 26 23 -4 -25 29 -20 -8 -7 -24\nright_child=1 -3 22 14 5 7 11 -9 9 10 -12 -13 13 -15 15 -17 18 -19 19 -21 -22 -23 25 24 -26 -27 -28 -29 -30 -31\nleaf_value=-0.0217860537470593 -0.027158709200026769 -0.023581984863765837 0.029552686221110274 -0.018400142011433418 -0.023458370696336651 0.03450095711523217 0.019771123071899082 -0.0071752784340146369 -0.016548927538318645 -0.013407492267479034 0.019019376396223826 0.0016927730257107927 -0.016924967670371282 0.01615734590279605 0.0097221853016186167 0.0043799517570402759 -0.021193066141887422 -0.026255420387696914 0.022643647794845966 0.018299503627933349 0.00068181700558894416 -0.015778362876123598 0.0087532680220008038 -0.021774080154076461 0.012177045613174887 0.012904071313625939 -0.012506215257353707 -0.010104258251550608 -0.0087559127525194957 -0.013509391544278323\nleaf_weight=1.667083590640799 3.7979080146524202 1.0232368556817153 4.7920184538764943 12.041059224819607 5.988817324335967 5.1149538436250159 7.5949084817963222 1.8588567591086755 3.9637959311089004 1.6903340044935955 4.4591831684253975 137.60619567044273 2.4110364941059856 2.5657687090194941 1.4955631752964134 23.537448240852768 2.3614060026957295 6.0004425310753504 1.9321401219349339 4.6111691547412192 6.2518952219543271 4.0232986497962884 3.0405680272117461 1.4351226911082902 5.6922545166792151 5.4115715863499636 1.100361732249439 1.2148812953610102 0.38286651975340635 6.5732197697071797\nleaf_count=35085 31756 171 726 15553 17022 2746 12170 3632 34903 19524 1103 10549 7948 467 27 40028 78 761 33 68 143 524 309 934 1924 459 86 3516 2100 1828\ninternal_value=0.000244367 0.000460303 0.000638792 -0.000176918 0.00140148 0.00239212 0.00137715 0.0213423 -0.00970349 -0.00129447 0.00947751 0.00259253 -0.0137113 -1.97972e-05 -0.00458034 -0.00129726 -0.00742426 -0.0187779 -0.000511668 0.0050169 -0.0121813 -0.0048577 0.00770925 0.0154527 0.00350289 0.00098556 0.00995084 0.0150412 0.0306046 -0.00630702\ninternal_weight=271.639 269.972 268.949 242.004 178.65 164.738 157.382 7.35668 13.9112 10.1133 6.14952 146.416 10.9656 4.97681 63.3548 45.0618 21.5244 7.49601 14.0284 11.667 18.293 7.0558 26.9448 11.9194 7.12738 15.0254 3.0325 8.80979 5.49782 9.61379\ninternal_count=246173 211088 210917 204737 147436 60150 51672 8478 87286 55530 20627 26235 25437 8415 57301 41605 1577 788 789 711 15696 643 6180 3584 2858 2596 119 15686 4846 2137\nis_linear=0\nshrinkage=0.03\n\n\nTree=262\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 1 1 6 2 3 4 5 17 7 13 1 13 6 3 1 4 13 17 4 18 2 3 19 4 2 2 4 3 3\nsplit_gain=2.00094 1.70485 3.07458 2.43503 2.65388 2.29494 2.40483 1.71897 1.52187 1.9887 2.04465 1.65366 1.51684 1.50152 1.57989 1.99863 2.39497 2.18932 1.9201 1.87429 1.86843 1.77707 1.84693 2.0036 1.73351 1.68008 1.66294 1.94198 2.69646 1.85279\nthreshold=3.8863394171308765 1.1267314367267358 1.7622051334857016 2.0131835937500004 -24.354135513305661 0.32220753877234559 1.0693359375000002 5.0527343750000009 0.077783200051907503 1.0000000180025095e-35 -1.0000000180025095e-35 1.6109634896136267 -0.79910615376814642 2.5244140625000004 0.58795853567030554 2.9477339485081675 1.8393554687500002 -0.014071314328338634 0.81373174126422521 2.3220214843750004 0.89587973461402759 -0.44610977172851557 0.43673077644757391 246.50000000000003 2.6030273437500004 -1.7340650558471677 2.5338792800903325 0.84667968750000011 0.56033297298897644 0.44825707527455844\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 5 4 -1 -3 -7 -5 -8 -10 12 -12 -11 14 18 16 25 19 21 -17 -21 22 23 -4 -24 -16 27 -20 29 -29\nright_child=-2 2 13 7 -6 6 8 -9 9 10 11 -13 -14 -15 15 17 -18 -19 26 20 -22 -23 24 -25 -26 -27 -28 28 -30 -31\nleaf_value=-0.014920909432293558 -0.013474304515772148 0.0019131517223201803 -0.006211877880657044 -0.028821097106468452 0.011231904202550613 0.0044202839174006065 0.0038163729885548822 0.0004389260523306265 0.0089319603466254133 -0.0032059811585546396 -0.022156456238632112 0.012219117764313312 -0.036313969732648471 0.018536659423645591 -0.02732421551608135 0.016299747948262524 0.0056624251998842598 -0.0089360427893212559 0.019761415269800558 0.021425287090445808 -0.0059603969367116871 -0.0093231180604053231 -0.0099163752029671447 0.014352981177552517 0.0085041805100497213 -0.001480092474051616 0.0078013619043144598 -0.010371177938329508 -0.028502560334815082 0.0070267938798509837\nleaf_weight=2.9950074559656619 8.1817912866601414 11.691811245559618 5.877532056244493 1.7045498079362449 35.924749471809548 4.4552861033062099 1.952638524905981 5.3569446847168365 1.1726701212760406 1.6667358629774778 1.2314420296497659 1.9303823810078027 8.6247636841890181 3.3709817574344916 3.5168894096568692 22.377663646882866 12.369489556775077 4.6608747341670087 2.1460996469286355 1.9667890807613719 8.1929886409779993 33.065905231961551 17.490156444871904 11.079825484394858 5.2706846633027444 4.7832340350578324 29.539087850146643 9.1213565364819313 2.810346219353959 11.102242102013404\nleaf_count=71620 63575 7107 835 5257 80220 1979 247 910 536 460 208 54 3092 111 358 238 295 83 427 15 94 1828 912 1359 213 218 1884 1505 117 481\ninternal_value=0.000298295 0.000768249 -0.000331429 0.00595915 0.008896 -0.00867667 -0.0143828 -0.00932307 -0.0193998 -0.0225452 -0.0257171 -0.00327601 -0.0317108 0.00115364 0.000739768 0.00480666 -0.00259702 0.00888088 -0.00112905 0.0116234 0.000785898 -0.00429591 1.54924e-05 0.00727608 -0.00547238 -0.0141912 0.00308481 -0.0025349 -0.00534806 -0.000939069\ninternal_weight=275.631 267.449 221.468 45.9813 38.9198 32.7257 21.0339 7.06149 16.5786 14.626 13.4533 3.16182 10.2915 188.742 185.371 57.8679 20.6696 37.1983 127.503 32.5374 10.1598 72.7841 39.7182 16.9574 22.7608 8.30012 54.7191 25.18 23.0339 20.2236\ninternal_count=246238 182663 24656 158007 151840 13683 6576 6167 4597 4350 3814 262 3552 10973 10862 1301 871 430 9561 347 109 5147 3319 2194 1125 576 4414 2530 2103 1986\nis_linear=0\nshrinkage=0.03\n\n\nTree=263\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 2 3 1 7 19 2 2 14 2 3 1 7 2 1 17 2 19 19 17 2 2 17 17 3 14 14 3 2 1\nsplit_gain=3.35862 2.39943 2.01217 1.9897 2.01211 1.84101 3.00307 2.50454 2.32178 1.84737 1.79882 1.77041 1.72999 1.545 1.48408 1.9744 1.71463 1.52277 2.13536 1.46504 1.38522 1.49979 1.34959 1.34937 1.33347 1.63738 1.33039 1.3281 1.29019 1.12115\nthreshold=3.2718994203396332 22.129880905151371 0.50207005315878861 -0.041910702480567281 3.7958984375000004 252.50000000000003 5.317040443420411 0.82504272460937511 0.73406812255247111 10.479034423828127 0.45898141303402712 1.5661550609387962 2.2500000000000004 12.256900787353517 0.45969269984116679 0.40154615600347926 12.256900787353517 241.50000000000003 226.50000000000003 0.16255022815751183 -4.708568572998046 -7.9326825141906729 0.48975197669932524 0.71600374927799881 0.56033297298897644 0.69039108051708775 0.21770771341770703 0.44825707527455844 -1.3165187835693357 1.075454090076706\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 10 4 -1 14 7 11 -8 -10 12 -7 22 -11 17 20 -17 18 -5 -9 21 -16 -3 -15 25 -2 -14 -28 -13 -22\nright_child=24 2 -4 5 -6 6 8 19 9 13 -12 28 26 23 15 16 -18 -19 -20 -21 29 -23 -24 -25 -26 -27 27 -29 -30 -31\nleaf_value=-0.027529314468546003 -0.027383950082564265 -0.0058353335752668227 0.010842844662703311 -0.0066670631555919013 0.0079490418153710732 -0.018907649642401651 -0.029089826661879061 -0.014245985485219645 -0.022708021246045977 0.022131050776769114 -0.016455401822071135 -0.00086238486491564974 0.0056082699744519123 -0.021883007905151585 0.0032207529809855488 -0.00047123494800991347 -0.01403871057080058 -0.013768865210950646 0.027754029310259932 0.017335716056444652 0.025143522035567405 -0.021763933793579701 0.0098620968973108705 0.011457804840875881 0.0059238292708667407 0.0014810184913119663 -0.023035062477818228 0.013452437336308038 -0.020850190387873169 0.0070822609786248365\nleaf_weight=3.9345110044311573 6.5690824938599786 5.6518416136044634 28.689429724217916 1.8030080383225584 1.5050897670847687 5.4381993675158347 6.182060144190471 0.79579242791259996 1.8791984115932789 1.9758589495486374 3.9926618919557759 47.196334290416416 2.9136774562398378 1.3716424601618658 13.045880690218381 52.163065105602868 9.0062095848212476 0.82579956593292525 4.7076230315988141 7.216041855456524 3.4152766067677502 1.5567422682465792 20.674230738124265 1.1706346411683624 1.57312242788248 2.1789595153853156 4.136608961280217 0.40416705282405008 2.1053988663043119 15.366628495220082\nleaf_count=80984 44442 4727 1582 2204 936 19805 805 145 323 97 487 3666 86 180 6024 17109 2841 13423 4821 782 1146 503 11103 188 118 25348 426 10 93 1546\ninternal_value=-0.000564909 0.000160692 0.00499855 -0.00161723 -0.0180028 -0.00101611 -0.00454166 -0.00174419 -0.0173406 -0.00358913 0.000394544 -0.0040912 0.00280197 0.00703696 0.00160927 0.000533682 -0.00266251 0.0137448 0.0186032 0.0129661 0.00629707 -0.000666968 0.00640186 -0.00763011 -0.016546 -0.0207799 -0.00916484 -0.0179456 -0.00211348 0.0114715\ninternal_weight=259.445 249.124 66.4626 182.661 5.4396 177.221 75.3312 62.7518 12.5794 6.39733 37.7732 54.7399 33.7805 4.51814 101.89 94.5538 61.1693 7.33643 6.51063 8.01183 33.3845 14.6026 26.3261 2.54228 10.3212 8.74804 7.45445 4.54078 49.3017 18.7819\ninternal_count=245950 176042 18421 157621 81920 75701 26084 24491 1593 788 16839 23564 16352 465 49617 29169 19950 20448 7025 927 9219 6527 15830 368 69908 69790 522 436 3759 2692\nis_linear=0\nshrinkage=0.03\n\n\nTree=264\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 14 4 1 17 7 18 1 1 18 0 4 14 4 17 17 17 4 9 14 1 4 1 4 4 7 2 4 2\nsplit_gain=1.53039 4.01785 1.88412 1.87016 1.21267 1.29194 1.18972 1.04496 1.56896 2.35174 2.94839 2.17296 4.80568 2.21207 2.00528 1.42296 1.36035 1.31039 1.1615 1.09753 1.4421 1.47417 2.32814 1.47368 1.46526 1.45469 2.61245 2.4939 1.51624 1.78788\nthreshold=0.030057938850762859 1.5000000000000002 0.82879072691359035 2.1186523437500004 1.7090960825073716 0.70050815440916869 2.0131835937500004 6.3465127852423047 0.87590939928379707 1.4475910192202701 5.1149908003866509 0.65747518837589958 5.6958007812500009 0.7674609687686843 8.2534179687500018 0.10726386778582141 3.5426598534814651 0.10726386778582141 3.2895507812500004 1.0000000180025095e-35 0.070321542204431389 1.6109634896136267 2.2153320312500004 1.7622051334857016 2.2978515625000004 5.5515136718750009 1.4980468750000002 37.549415588378913 2.6030273437500004 -17.275529861450192\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=7 3 6 -2 -5 -6 -3 8 11 10 17 12 -1 15 -14 -13 -17 18 -10 20 24 22 -22 -23 -11 26 28 -28 29 -25\nright_child=1 2 -4 4 5 -7 -8 -9 9 19 -12 13 14 -15 -16 16 -18 -19 -20 -21 21 23 -24 25 -26 -27 27 -29 -30 -31\nleaf_value=-0.025157029320803891 0.019356335038699771 -0.032143926559281645 -0.0044856013482467644 0.0065157836469555922 0.004657100909511094 -0.027586427596869287 -0.0077851099125369003 -0.014095090166322953 -0.016058068277501905 -0.019150087396353314 0.012797373151103443 -0.013216737228277467 0.022790642168769661 -0.010149634439329739 -0.0037516351392236111 0.020629683398283947 -0.011833997069299534 -0.022917664101693928 0.015578577365932366 0.012117857896938204 0.019052913363202209 -0.012349913790515987 -0.012572223739950207 0.0049430218282962993 -0.00086097680369842872 0.012344039758990993 -0.021451310766188367 0.014761101065913412 0.0079906975325312289 -0.0031097524789099502\nleaf_weight=4.426859096311488 4.5824666578597562 5.6336427224993049 8.3660455230074131 4.4731638394055153 1.4852699412931545 1.6981452784857536 3.5944972296974811 3.4607832348391971 0.90376986129535852 5.771024102059072 2.7457234783067852 0.66508496770734993 4.2348388749096539 2.5686829087391052 4.5438519387132601 15.240538446885918 0.62340391161842568 9.1304325339700636 1.3253262717771574 6.02439174788742 7.4534178232559025 7.0725414072526283 2.0132504763275847 36.782598798930394 9.6768168051353296 8.2327145536683002 7.4693451041930512 1.39041501284737 22.950530708890255 71.305899265613334\nleaf_count=125657 685 3483 2519 1354 69 65 1322 40823 962 858 2950 1202 3607 10306 572 13790 4630 14363 108 140 2192 2429 1046 3393 443 215 772 72 1213 4715\ninternal_value=-0.000529941 -0.00681577 -0.0156507 0.00610769 -0.00314167 -0.0150255 -0.0243446 0.000290027 0.000563538 -0.000437102 -0.0118221 0.00659338 -0.0026762 0.0128173 0.0100736 0.0167618 0.0187279 -0.0183269 0.00175108 0.000484175 3.03083e-05 0.000871292 0.0117687 0.000135519 -0.00844234 0.000810451 4.89345e-05 -0.0148476 0.00116093 -0.000349196\ninternal_weight=265.845 29.8332 17.5942 12.239 7.65658 3.18342 9.22814 236.012 232.551 200.248 14.1053 32.3033 13.2055 19.0977 8.77869 16.529 15.8639 11.3595 2.2291 186.143 180.119 164.671 9.46667 155.204 15.4478 148.132 139.899 8.85976 131.039 108.088\ninternal_count=245955 9497 7324 2173 1488 134 4805 236458 195635 35871 18383 159764 129836 29928 4179 19622 18420 15433 1070 17488 17348 16047 3238 12809 1301 10380 10165 844 9321 8108\nis_linear=0\nshrinkage=0.03\n\n\nTree=265\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 12 21 2 10 18 7 13 11 18 7 9 5 5 18 18 2 2 21 13 1 2 17 18 5 17 1 19 18 13\nsplit_gain=3.03267 2.0277 3.75429 2.03836 1.98276 1.70762 2.08407 1.81644 1.42435 1.46415 1.41214 2.92918 4.52438 2.58746 1.95214 2.63311 2.41133 1.95104 1.8085 2.00618 1.62021 2.1443 2.09301 1.80483 2.28146 2.2648 1.54742 3.25827 2.39188 2.11755\nthreshold=3.2718994203396332 5.5511151231257837e-16 1.5000000000000002 30.514518737792972 1.4978661367769956 1.4978661367769956 3.1132812500000004 -0.36630902128874782 0.69839536661192347 3.0201273556387074 2.5244140625000004 1.0000000180025095e-35 2.3505859375000004 3.8173828125000004 4.297262267176281 4.2121001622835372 -31.266788482666012 22.129880905151371 2.5000000000000004 -0.42002175472665648 0.35607680265790181 11.076014518737795 0.95146732080301388 2.0126758453675748 1.7006835937500002 0.67078623592565922 2.9477339485081675 259.50000000000006 1.4978661367769956 -0.076599057954334346\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 -3 10 -2 -5 7 -7 -4 -10 11 12 14 17 15 16 -1 -14 -16 -20 22 26 -18 25 -25 -23 27 28 -22 -28\nright_child=4 2 8 5 -6 6 -8 -9 9 -11 -12 -13 13 -15 18 -17 20 -19 19 -21 21 23 -24 24 -26 -27 29 -29 -30 -31\nleaf_value=-0.0097086771836150075 -0.020028018404100279 0.0067939455639775123 -0.024069771323213422 0.014476347600262354 0.010085997580020135 -0.021158902193606114 0.018007355758768526 0.0021016678215507248 -0.01393942805025737 0.0076853593751444278 0.0078658218915356008 0.022278292944619488 -0.027096923260690041 0.0099806033141523077 -0.025310278057327839 0.023969439368391465 0.026434980448982428 0.0036423673645566259 -0.020945006420988309 0.0067214700810734607 -0.011280693257891279 0.00013868787756468202 -0.013890956026658002 -0.0097412059627477593 0.01798509601583196 -0.021784765192726478 0.010103421042216629 -0.011431547391913266 0.010468775167530607 -0.012089545052440569\nleaf_weight=14.5854757598222 8.4411071378119669 12.163426643512709 6.9052349637559614 20.580712537857835 1.8237649144620025 2.8769026476923054 4.621672450697587 13.046351432644997 6.6374209376226627 3.661380769644893 24.175248124567247 4.2476613088288095 9.4351776715752731 2.1979997342073139 3.4953481216236915 3.8260615081409233 3.9866019926920435 1.9946065107346829 2.394598077270639 6.7752843601288673 4.8448234599127344 8.0840699319423965 0.8225261995863774 4.2513543014601947 4.5581991353094509 7.8914216267967268 40.450239377234539 13.575203635212345 21.233386972873404 3.4284285307949167\nleaf_count=9185 69381 1904 5676 721 542 2376 119 8265 243 243 16885 841 23458 1093 31775 3731 9858 569 17933 20596 247 306 6699 2451 488 379 616 1766 7807 115\ninternal_value=0.000985479 0.00164784 -0.00564692 0.00261061 -0.0142573 0.00857272 0.00226138 -0.00310285 -0.0143328 -0.00625172 0.0012554 0.000227212 -0.000502771 -0.0164027 0.00110246 0.00220841 0.00137532 -0.0218712 -0.00970471 -0.00185238 0.00287693 0.00202923 0.0183339 -0.00557018 0.00497578 -0.0113362 0.00432329 5.05585e-06 0.00616077 0.00813864\ninternal_weight=267.012 256.747 29.3675 227.379 10.2649 41.1256 20.5449 15.9233 17.204 10.2988 186.254 162.078 157.831 13.6278 144.203 131.538 127.712 11.4298 12.6652 9.16988 113.126 108.317 4.80913 24.785 8.80955 15.9755 83.5321 39.6534 26.0782 43.8787\ninternal_count=246268 176345 8066 168279 69923 11481 10760 10641 6162 486 156798 139913 139072 25120 113952 43648 39917 24027 70304 38529 30732 14175 16557 3624 2939 685 10551 9820 8054 731\nis_linear=0\nshrinkage=0.03\n\n\nTree=266\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 2 1 19 17 2 2 2 4 18 18 1 18 17 17 0 1 1 6 10 17 18 18 17 0 2 18 1 19 17\nsplit_gain=1.79114 0.996978 0.846186 1.43777 1.51118 0.764818 1.75778 1.88329 1.30715 1.25596 2.21631 1.7933 1.94687 1.98146 1.40417 1.29025 2.121 1.58354 1.33407 1.34232 1.31934 1.27452 2.12819 1.59271 1.49133 3.15914 2.8078 3.20951 2.03214 1.90775\nthreshold=5.1405116420104324 -55.708036422729485 4.0016611463566001 253.50000000000003 1.0434576633023316 -28.340197563171383 -15.657813072204588 -17.667204856872555 2.6340332031250004 4.8242976514536702 3.7494348669884658 -0.10824782141703854 3.5972184255501678 2.5380284121401901 0.32133416703193302 0.57379665772197053 2.1468275719512682 2.9477339485081675 0.89599609375000011 1.2424533248940002 3.8863394171308765 3.5114340430413216 3.4498615536424366 0.32406367513646889 1.2225960941547209 -2.5585746765136714 2.0126758453675748 1.2755077334848104 260.50000000000006 0.72612140026470018\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 5 -4 -5 8 7 -7 -3 10 11 14 13 21 -8 16 18 -18 19 -12 -17 22 24 -24 25 -13 27 -26 -27 -28\nright_child=-2 2 3 4 -6 6 9 -9 -10 -11 15 12 -14 -15 -16 20 17 -19 -20 -21 -22 -23 23 -25 26 28 29 -29 -30 -31\nleaf_value=-0.014105708993548894 -0.018491759659221171 0.0066666982200938073 0.0068376048561804318 0.0015358468124407332 -0.020532841106275134 -0.0052114855325920192 -0.014161052490079611 -0.028729299452063151 -0.012470695544646349 0.0067088206703615526 -0.028586202851233387 -0.021529815127451606 0.012853565485669334 -0.016541164695648841 0.017746803697743539 -0.029749766208620285 0.017979701733839119 -0.016687177931675675 0.0012945130597800832 0.0099450935631275675 0.0076990444615068271 -0.019075017205253283 0.030777121579970763 -0.011543478268238202 -0.029003173662817478 0.0017923844859741929 0.00014918433644813098 0.0003446306454732339 -0.019935881382509256 0.022494969845396919\nleaf_weight=3.7005050704731266 4.0300720398417402 21.053651656599758 32.09020097753455 9.2200559953926113 2.9314091752748928 18.533133149862355 0.71052529045027668 2.8564240832165515 2.8174235668932957 18.107003314353562 3.9326309263794865 7.9014594771640487 7.4799811847492483 6.9089239887953218 6.6221961448169395 3.7252073039089728 3.5131888972255183 0.73802034100845038 2.9218114138968758 0.45966683763981553 0.59951194841698896 2.3981451765957891 2.1887985845637554 0.48581521449818854 2.5900480558957488 27.451974528119997 5.8278872971477522 50.62825227065332 3.488457487541381 7.4342992908259475\nleaf_count=40655 49417 28866 724 110 33 21922 1564 4068 14630 26753 26131 2132 2038 1826 5962 6919 539 123 470 36 433 883 163 420 1056 5521 751 1138 542 511\ninternal_value=-0.000760064 -0.000414517 -0.000162116 0.00356378 -0.00494443 -0.000955072 -0.00160343 -0.00949501 0.00399738 -0.000536538 -0.001462 -0.000152652 -0.00104439 -0.00203176 0.0133266 -0.0116949 -0.00551269 0.00992974 -0.0146031 -0.0234573 -0.0240875 -0.000983369 -0.000410313 0.0220406 -0.00118238 -0.00577611 0.00154738 -0.00159227 -0.00120488 0.0133744\ninternal_weight=263.347 259.317 255.616 44.2417 12.1515 211.374 187.503 21.3896 23.8711 166.114 148.007 132.117 124.784 117.304 7.33272 15.89 11.5653 4.25121 7.31411 4.3923 4.32472 110.395 107.997 2.67461 105.322 38.8419 66.4805 53.2183 30.9404 13.2622\ninternal_count=246336 196919 156264 867 143 155397 111901 25990 43496 85911 59158 24507 16981 14943 7526 34651 27299 662 26637 26167 7352 13117 12234 583 11651 8195 3456 2194 6063 1262\nis_linear=0\nshrinkage=0.03\n\n\nTree=267\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 13 3 1 12 2 2 2 2 2 21 2 5 5 21 18 0 18 2 2 2 19 2 2 7 18 7 2 21 5\nsplit_gain=1.76323 3.10029 2.68455 3.79081 2.22447 2.1246 1.77163 2.97464 2.46679 1.74989 1.80987 1.72867 1.82477 2.8872 1.84065 1.58057 2.52066 2.30715 1.89496 1.86747 1.84237 2.08979 1.95949 1.88846 2.90726 1.71065 2.15347 1.64417 1.61712 1.51438\nthreshold=0.47260169401963864 -0.47254882959329203 0.40160889672495398 2.5144622974930591 0.90934753668501156 -20.765821456909176 -1.3165187835693357 -3.0241031646728511 3.8993310928344731 -4.708568572998046 7.5000000000000009 22.129880905151371 2.3505859375000004 3.1562500000000004 1.5000000000000002 4.0342014792848504 0.76735450420087303 4.2413008733233104 9.8937015533447283 -10.615592002868651 -12.11164617538452 262.50000000000006 -22.378742218017575 6.3334450721740732 1.2426757812500002 0.89587973461402759 1.2426757812500002 -29.210144996643063 5.5000000000000009 1.4980468750000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 9 -4 5 -3 7 -7 -8 -1 -11 12 15 -14 -15 16 18 -17 -2 -19 21 -18 27 25 -25 28 -27 -23 -22 -10\nright_child=11 4 3 -5 -6 6 8 -9 29 10 -12 -13 13 14 -16 17 20 19 -20 -21 23 22 -24 24 -26 26 -28 -29 -30 -31\nleaf_value=0.00018649304515224615 -0.028671124438861637 -0.016619273971313263 -0.028550110359438004 0.008492336210086146 0.018647283721013767 -0.00253376193018828 -0.023826217686911927 0.021790086659899011 -0.012186029746107649 0.021467530567646435 -0.013270019803891329 0.0079174674085691829 -0.032223107521650765 -0.025682105108026557 0.0060967214487779075 0.027186932160832681 0.0014302559863013248 0.0087067023417393332 0.0087211503140839416 -0.018583300032180419 0.011529241478812742 -0.017646198161096516 -0.033484532083158831 -0.0099639257896763211 0.017957028119680914 0.0155503883965769 -0.016614783556410332 0.0088311643354037905 -0.018947169449362176 -0.0011691317793377028\nleaf_weight=7.5032255772322332 3.4596735183640694 13.302904066820302 3.3173978946631433 5.0802428413244343 2.4897853496659073 22.246098284501276 5.9330144531558053 4.6318065184481716 15.360495651726067 11.590274396375092 0.84359827823686906 34.795563902789581 2.8407504218794193 1.0676647108411979 6.9429769560629211 4.0162396258165236 18.367080844123851 8.0386788343275786 1.1478468199456981 2.0169333968783603 1.9334768358967291 3.2605921902577393 2.5739569592551543 12.061219421189888 3.5359807419299605 21.485888006794998 1.2365998254390422 3.3043732100632042 2.5198322781361639 36.601023521227319\nleaf_count=41066 622 92791 3150 187 117 30739 7873 2649 29317 10817 2554 1997 572 245 566 376 382 1205 150 1098 34 87 75 455 43 480 33 47 21 16538\ninternal_value=-0.000131945 -0.00262935 0.0059921 -0.00762178 -0.00509357 -0.00587839 -0.00401877 0.00228918 -0.00693643 0.0118165 0.0182983 0.00226136 0.000230553 -0.0108373 -0.000518927 0.0016861 -9.65902e-05 0.01055 -0.0194618 0.00204715 0.00142822 -0.00453038 -0.0154694 0.00527598 -0.00293352 0.0099246 0.0131733 -0.00491352 -0.00602755 -0.00459447\ninternal_weight=263.505 128.9 28.3347 8.39764 100.565 98.0753 84.7724 26.8779 57.8945 19.9371 12.4339 134.605 99.8098 10.8514 8.01064 88.9584 74.8865 14.0719 4.60752 10.0556 70.279 27.506 9.13892 42.773 15.5972 27.1758 22.7225 6.56497 4.45331 51.9615\ninternal_count=246286 237798 57774 3337 180024 179907 87116 33388 53728 54437 13371 8488 6491 1383 811 5108 2429 2679 772 2303 1657 591 209 1066 498 568 513 134 55 45855\nis_linear=0\nshrinkage=0.03\n\n\nTree=268\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 11 1 1 1 0 2 12 6 4 3 2 2 1 5 2 3 21 3 21 2 2 4 2 0 1 2 19 3 4\nsplit_gain=2.37996 1.81328 1.35731 1.9511 2.76543 1.93451 1.49787 1.27374 1.29865 1.27239 1.57711 1.1935 3.35699 2.05766 1.66744 1.474 1.35817 1.18928 1.65692 1.10471 1.15445 1.88636 1.72752 1.86196 1.05754 2.52125 1.61202 2.35601 1.82983 1.62197\nthreshold=1.3170161875856263 0.45066394906892915 0.45969269984116679 2.9477339485081675 2.6981674883069502 1.5447487307370646 30.514518737792972 0.88192242588002812 8.1289062500000018 3.3325195312500004 0.65908129556435835 31.708583831787113 1.0000000180025095e-35 1.3779212206518803 1.4980468750000002 -37.459321975708001 0.51817819236196982 5.5000000000000009 0.25432107015297428 2.5000000000000004 -15.657813072204588 5.8049912452697763 1.0693359375000002 21.425971984863285 0.54700948377946179 -0.041910702480567281 8.8705892562866229 260.50000000000006 0.62297257604742995 2.6340332031250004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=7 2 5 4 11 -2 -6 8 24 10 19 12 14 -13 15 -4 17 18 -17 -5 -21 -22 23 -23 25 -1 29 28 -28 -26\nright_child=1 -3 3 9 6 -7 -8 -9 -10 -11 -12 13 -14 -15 -16 16 -18 -19 -20 20 21 22 -24 -25 26 -27 27 -29 -30 -31\nleaf_value=-0.017366483838862593 0.030671033828305341 -0.0083705911730365141 0.021221549551369564 -0.0091577591856000888 -0.028407648719946409 -0.0042546098837947764 0.0057124797815013752 0.011938608975272361 -0.025042945177165286 0.018544229032006598 0.017776840128274504 -0.0154193791717104 -0.015780601786025781 0.013774485433303084 0.018444662790193477 0.015471941416991507 -0.021084550763112966 0.018604341848709374 -0.021443808600375036 -0.0054657270354149749 0.014977870810435497 -0.0095251620392462297 -0.0095097983207026015 0.0245185633985897 -0.0046184210560917946 0.0035269279051604414 0.004854014132996483 -0.022429514561019822 -0.015816671923967806 -0.016296284896393849\nleaf_weight=4.8990145554406004 2.9627819298366367 9.8890355444018478 1.0056349533260798 5.1021407443622584 3.6185006984760539 1.8901671089079171 1.1533995536083237 4.3641820590710259 1.3761068043638705 7.0669286009506314 6.6137556264875448 1.7906345549140485 8.8658119405054077 10.816438635079065 5.36480504494466 0.90887218776606671 2.6558380322530866 1.5372267374132205 1.7868909057651761 6.9738131797639609 13.451711657340644 1.6666236234595979 7.421899450419005 2.7299956089846091 52.323880840637457 42.92781339979274 34.120647006004333 2.6975878714363253 3.3495342747592067 12.975896615473287\nleaf_count=112518 1181 349 38 135 84 945 52 165 144 61 54 502 860 526 203 94 41 73 133 79 106 26 106 42 32358 71591 17091 804 396 5083\ninternal_value=-0.000178142 0.0033028 0.00458734 0.00363702 -0.00135786 0.0186651 -0.0205997 -0.00248859 -0.00296974 0.00745573 0.00530035 0.00178832 -0.00241448 0.00879951 0.00700271 -0.00197237 -0.00761911 0.003244 -0.00817899 0.00268488 0.0047779 0.00770528 -0.00109501 0.0122399 -0.00261057 0.00107492 -0.00427625 0.000455353 0.00264311 -0.00714976\ninternal_weight=264.308 105.273 95.3839 90.5309 39.5041 4.85295 4.7719 159.035 154.67 51.0269 43.9599 34.7322 22.1251 12.6071 13.2593 7.89446 6.88883 4.23299 2.69576 37.3462 32.244 25.2702 11.8185 4.39662 153.294 47.8268 105.468 40.1678 37.4702 65.2998\ninternal_count=245840 5690 5341 3215 2606 2126 136 240150 239985 609 548 2470 1442 1028 582 379 341 300 227 494 359 280 174 68 239841 184109 55732 18291 17487 37441\nis_linear=0\nshrinkage=0.03\n\n\nTree=269\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 2 18 1 6 1 3 14 14 7 3 3 5 1 2 18 12 12 14 2 2 0 2 7 21 19 0 1 14 3\nsplit_gain=1.47492 1.56235 2.02998 1.65017 2.34112 1.48075 2.92693 1.33476 1.24795 2.63689 1.60233 2.00604 1.29636 1.95502 1.7858 1.55871 1.18297 1.1734 1.51616 1.50022 1.6949 1.95286 1.58904 1.47116 2.29886 1.86123 1.83101 2.68597 2.12592 1.66489\nthreshold=0.53650715206675215 -26.006598472595211 2.249904835165133 1.075454090076706 6.8105468750000009 1.4475910192202701 0.32220753877234559 0.023535737397887727 0.050734622491290134 0.89599609375000011 0.28904431203898151 0.3261289997719386 2.1386718750000004 1.0240454605967615 7.3050365447998056 3.4811217321331038 0.28697820620240239 0.34440402210285487 0.81584600691354858 18.240050315856937 34.417465209960945 0.92717806344988862 35.92213439941407 0.89599609375000011 2.5000000000000004 254.50000000000003 0.60985949876859957 2.9477339485081675 0.74140086107412606 0.65908129556435835\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 4 7 6 -5 -3 -9 10 -10 -12 15 14 -14 -11 -4 19 -19 23 -21 -22 -23 25 29 -2 -27 28 -28 -25\nright_child=17 3 16 5 -6 -7 -8 8 9 12 11 -13 13 -15 -16 -17 -18 18 -20 20 21 22 -24 24 -26 26 27 -29 -30 -31\nleaf_value=0.0027335937915188954 -0.013813327988186078 -0.0091013818549658451 -0.020301048500986185 0.0022663615936918185 -0.024177960772110242 -0.0011407356516949794 -0.031676083435146503 0.020363346832171884 -0.021374632245750241 0.028304724654591953 0.023335398686093967 -0.01680212673856037 0.0068933165627497392 0.021703628455869893 -0.014590180562463688 -0.010998225786478038 0.0035925906991852012 -0.022244571563755467 0.0021453621968079827 0.018233526847175108 -0.024501752645554275 -0.014920534919155017 0.012021325845593466 -0.026619880141016242 0.0085267569658241532 0.018039239676907556 -0.032176358705137124 0.0092572376059011363 0.00081270087385854094 0.0088936454080032479\nleaf_weight=7.6998343291702467 8.9271013355373743 3.5961141328426143 9.9064988857938925 4.6053137260759049 1.3955241535841185 104.85349872485949 3.3118198243355437 5.5520014889989389 3.6503799310254434 4.2361099401125584 1.4758359532924035 1.1109897792518948 9.8792116565103942 4.1067803532973803 4.1216644235196318 0.54428497132823839 1.9213531289406649 2.5585938152798899 6.0240264098029002 11.85111615343658 1.1171576849565088 1.5698732817545522 8.6902491719192749 2.2120144636946852 30.888025442944127 3.9387522996421493 2.6974191957924614 6.5934183537319768 3.2605488830886316 1.2601622875954492\nleaf_count=642 1113 8953 110465 4280 125 10925 2860 2481 59150 1868 2983 4866 14189 276 5030 9986 2308 88 264 792 257 20 267 214 792 365 215 94 199 19\ninternal_value=0.000290641 -0.00134629 -0.00911629 -0.000298012 0.00489076 -0.00215019 -0.0138921 0.00653931 0.00837093 0.0054848 -0.0106525 0.00621873 0.0101442 0.00581461 1.7855e-05 0.0227013 -0.0164422 0.00334549 -0.00668818 0.00445022 0.0107557 0.00212257 0.00694005 0.00186338 0.00583987 -0.00352981 0.00250876 -0.0033364 -0.0166007 -0.0146243\ninternal_weight=263.556 171.967 19.5277 152.44 39.6689 112.771 7.91713 38.2734 34.6773 29.1253 6.23721 2.58683 22.8881 18.1077 14.0009 4.78039 11.8279 91.5885 8.58262 83.0058 23.2284 11.3773 10.2601 59.7774 34.3602 25.4172 16.4901 12.5514 5.95797 3.47218\ninternal_count=246086 241387 113415 127972 109907 18065 7140 109782 100829 98348 66999 7849 31349 19495 19219 11854 112773 4699 352 4347 1336 544 287 3011 1025 1986 873 508 414 233\nis_linear=0\nshrinkage=0.03\n\n\nTree=270\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 1 0 1 1 2 18 0 2 1 2 1 3 5 3 0 2 2 5 3 2 21 0 2 0 0 17 1 0 2\nsplit_gain=3.18594 3.35206 2.32008 1.85515 1.83135 1.94978 1.80482 3.8765 1.99679 3.05029 2.90807 2.06793 2.22913 1.81601 1.65975 1.61796 2.4951 1.70962 1.59887 1.53283 1.5115 1.43468 2.0532 1.41775 2.64496 2.01698 3.14286 2.14181 2.7368 1.7431\nthreshold=21.425971984863285 0.15052965826911111 0.60985949876859957 0.6180242047922454 1.3779212206518803 35.92213439941407 4.1510089048756029 0.52258841453381233 -10.210010528564451 4.0016611463566001 -18.836229324340817 1.0021760634374313 0.58795853567030554 2.1386718750000004 0.2708708835714248 1.1210156985015984 12.256900787353517 8.8705892562866229 1.7006835937500002 0.25176753112663547 2.9882335662841801 5.5000000000000009 1.6765212893675572 8.8705892562866229 0.82186659671441242 0.96046990436194013 1.0434576633023316 2.9477339485081675 1.3771184689138856 11.076014518737795\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 18 -4 5 -5 8 -8 9 10 -3 13 21 -10 -6 17 -17 -14 -2 -12 -9 23 -23 24 -13 26 -26 -27 -29 -25\nright_child=2 6 3 4 14 -7 7 20 11 -11 19 12 15 -15 -16 16 -18 -19 -20 -21 -22 22 -24 29 25 27 -28 28 -30 -31\nleaf_value=-0.020135977937926347 0.00534052464809847 -0.0065791739655014182 0.018030605603383525 0.0052509565081437278 -0.0090975087544392184 -0.026696883680817941 -0.00016058293125604289 0.026922333718607812 0.025664902686762376 0.0041263615059851056 0.003714996538338879 -0.028477664129624963 -0.02088538445505389 -0.0034505964650532713 0.0081215076532349419 0.015193726305624695 -0.0089940515169588935 0.0073110429425322365 -0.023447885341819259 -0.029737776859661766 -0.011764323994336095 0.010458946401942041 -0.010073183406905993 0.012929323763284621 0.016603860678722823 -0.0231081488938897 -0.026458197281397269 -0.017958389870097433 0.015397774547928678 -0.0072216156573175795\nleaf_weight=7.8273914011234629 2.4643820856238667 29.258899927164858 10.325010882967147 2.9740876279932982 4.8102809964616418 2.3540264032296818 32.945959803999131 6.4879428928880989 3.9890756330518107 14.659700853706452 1.1868989469256828 5.5589141729697102 2.816490246885226 2.7963793378190758 46.036732557083724 15.565180464720472 4.2650867860065764 3.1593208898193552 3.1308216843381729 8.0899048216085685 0.46817354508402775 10.37643774944536 6.173058408770884 4.9038394776471241 5.4146064107062584 6.7718919705948775 1.0420181590384343 3.1372947137424489 3.7616467671468854 10.180180148141742\nleaf_count=139548 12342 9195 7977 982 1039 1426 48876 1970 4989 251 1736 1281 248 906 1716 158 69 125 5136 1513 377 827 99 320 298 821 221 107 49 1322\ninternal_value=-4.46738e-05 -0.00207524 0.00528509 0.00688402 0.00455575 -0.0108523 -0.00119754 0.00437312 -0.00277036 -0.00731889 -0.0116671 -2.83499e-05 -0.00139697 0.014764 0.00634849 0.00578348 0.0098094 -0.00706726 -0.011881 -0.0255127 0.0231672 -0.00467343 0.00266276 -0.00764379 -0.0117992 -0.00606298 0.00703792 -0.0123084 -0.000124116 -0.000273956\ninternal_weight=262.932 190.836 72.0953 66.5001 56.1751 5.32811 183.009 39.9021 143.107 53.1954 38.5357 89.9114 83.126 6.78545 50.847 25.8061 19.8303 5.97581 5.5952 9.2768 6.95612 57.3199 16.5495 40.7704 25.6864 20.1275 6.45662 13.6708 6.89894 15.084\ninternal_count=245924 215306 30618 13140 5163 2408 75758 51223 24535 12695 12444 11840 5945 5895 2755 600 227 373 17478 3249 2347 5345 926 4419 2777 1496 519 977 156 1642\nis_linear=0\nshrinkage=0.03\n\n\nTree=271\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 13 0 18 0 0 13 5 5 4 7 19 5 18 0 10 18 14 13 18 17 11 17 13 17 0 7 13 17 14\nsplit_gain=2.39031 1.53638 1.83982 2.80377 1.51755 1.34453 1.31918 1.37069 1.37676 1.31837 2.85558 1.43861 1.28935 1.85342 1.27194 3.48739 2.78621 4.31051 3.22773 2.71815 2.6459 2.31096 1.91613 1.88233 1.91259 1.92455 1.59546 1.45574 1.26505 1.58976\nthreshold=1.3771184689138856 -0.67231277955111934 0.88405423821904405 5.0271592211193568 0.97942481616843191 1.1495945197163422 -0.37639895965301279 2.2500000000000004 4.5058593750000009 3.9514160156250004 1.2426757812500002 247.50000000000003 1.7006835937500002 1.8688348091416842 0.34029801008043176 1.0000000180025095e-35 4.1820210059610323 0.2247238965590572 -0.56995612852957833 4.2696729980286863 0.32133416703193302 0.66750229771629133 1.2133651393498066 -0.54370230766509287 0.69259390652607344 0.65747518837589958 1.2426757812500002 -0.43755180013342404 1.1976019298503242 0.043928974430191954\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 3 -1 -4 -6 7 -2 -9 10 11 12 13 -8 15 20 18 -18 22 26 -3 -22 -16 -20 25 -25 -19 -21 -26 -30\nright_child=6 14 4 -5 5 -7 9 8 -10 -11 -12 -13 -14 -15 16 -17 17 19 23 27 21 -23 -24 24 28 -27 -28 -29 29 -31\nleaf_value=-0.017559290499080598 0.010852450466983634 0.0043961812394517439 0.02500085406181695 0.013569093963462812 -0.010378326931524193 0.011827048597287011 0.0054570088150576145 -0.016666510114262994 0.010651054412157548 0.008727976961199287 -0.021105095377958499 -0.00528002283032081 0.016630613525974336 -0.021800274282602668 -0.030015788562843173 0.0086387254420792203 -0.024302845838587106 0.036091234041143359 0.014254312850806907 -0.016658775904891353 -0.026483133893489365 0.010086345873241458 0.0046822708494318839 -0.029756292151304136 0.0052116637273546679 -0.0056850869730779765 -0.001958163281562249 0.0093539219772194124 0.0099641142904827503 -0.0071287219379528059\nleaf_weight=5.1424911697274087 32.524749891065767 4.01092153142418 4.4027431821523342 3.5833913096805645 2.8995525233535755 5.8584570692114539 13.700167770997043 2.8187836085344307 2.0412398069165647 15.099354998625129 4.7131033472069239 15.053760682576465 6.5230878493239279 1.6532089476677345 4.9914784938074481 6.5533658187932806 2.0751167955291434 4.6091535891470121 5.1617827235013456 1.4569489548277363 11.345328301535174 1.2511753281978588 1.5618774137149101 2.3654733658430578 27.511263269935284 37.126738824542031 1.0842401213502624 8.7815594681538283 5.2425019771110355 22.728416281780312\nleaf_count=13127 1022 18452 649 10201 924 582 974 210 36 408 337 640 373 719 2511 3576 5323 3790 1270 9248 98995 126 1121 5696 8132 11187 573 22339 2572 20653\ninternal_value=0.000555955 -0.0015695 0.00568246 -0.00469524 0.0124088 0.00416524 0.00436508 0.00866417 -0.00533332 0.00143164 -0.00135655 0.00165376 0.00644732 0.00136842 -0.00267394 -0.00897969 -0.0014406 0.00916462 -0.00329903 0.0147016 -0.0160278 -0.0223761 -0.0222211 -0.00185321 -0.00286805 -0.00782614 0.0296352 0.00449937 0.000737203 -0.00369171\ninternal_weight=263.871 169.744 21.8866 8.72588 13.1608 8.75801 94.1275 37.3848 4.86002 56.7427 41.6433 36.9302 21.8765 15.3534 147.857 23.1608 124.697 18.007 106.69 15.9319 16.6074 12.5965 6.55336 100.136 94.9744 39.4922 5.69339 10.2385 55.4822 27.9709\ninternal_count=245766 241047 25483 23328 2155 1506 4719 1268 246 3451 3043 2706 2066 1693 215564 121149 94415 41273 53142 35950 117573 99121 3632 49510 48240 16883 4363 31587 31357 23225\nis_linear=0\nshrinkage=0.03\n\n\nTree=272\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 3 0 2 2 19 2 19 14 7 3 3 6 5 2 14 14 3 13 2 19 17 6 6 2 13 13 17 5 3\nsplit_gain=1.73201 1.64702 1.70048 1.44831 1.85565 2.78695 1.114 1.47666 1.18868 1.39707 1.43485 2.57613 1.30366 2.34404 1.07717 1.34377 1.53019 1.26101 1.07588 1.12133 1.03218 0.977853 0.967946 2.09134 2.20299 2.02183 2.44474 1.10041 1.04001 1.13267\nthreshold=22.129880905151371 0.10244777304966225 0.17863154480046692 24.399772644042972 28.11904335021973 250.50000000000003 34.417465209960945 261.50000000000006 0.070321542204431389 1.2426757812500002 0.62297257604742995 0.50207005315878861 2.9179687500000004 3.8388671875000004 16.994721412658695 0.8507468585983845 0.37880621680625948 0.53650715206675215 -0.59662603815460169 35.92213439941407 219.50000000000003 2.8667313866239854 1.0000000180025095e-35 1.8691406250000002 15.156654357910158 -0.076599057954334346 -0.0081313880310213676 2.775573723725536 4.2558593750000009 0.45898141303402712\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 2 -1 -2 5 -5 20 8 -8 18 11 12 13 -11 22 16 17 -16 -10 -20 -6 -21 27 24 25 -24 -27 28 -3 -30\nright_child=3 14 -4 4 6 -7 7 -9 9 10 -12 -13 -14 -15 15 -17 -18 -19 19 21 -22 -23 23 -25 -26 26 -28 -29 29 -31\nleaf_value=0.0013869541678101624 0.016807962353894965 0.0013997026046778465 -0.023800260795806712 -0.025003158741043554 -0.012957968596671358 0.0093832117277812439 -0.0053492802779334693 -0.020440242044759426 -0.012456364242510688 0.0013717179304151852 -0.017304631495230951 0.017339540182880558 0.01414517116820071 -0.035682404147547843 -0.013449445793991677 0.0082997358492635284 -0.024495178284507833 0.014177451887584796 -0.011763277171166308 0.019076004224135593 0.013724147253348109 -0.012203904325770794 -0.028447005810091561 0.0038643080393459248 0.014911879850627523 0.019287434821413128 -0.022758919696211754 -0.015661311430445433 0.018639256122478975 -0.015570127416688107\nleaf_weight=4.180540903640404 7.3239112788021421 146.55870886943597 4.3865716904671217 3.7926596360297244 0.64850659902981433 3.0737709316140354 8.8901977861322603 1.4167107231696707 0.63810390443415599 3.4838695482699205 2.376353561950963 5.9639210617624485 1.0091122804988115 1.6529913304057116 2.0408436124170053 2.5092752767683342 4.3825095853251241 1.9098836775592642 0.61261148978008506 12.128491857280206 14.611438922641296 0.30761924415287545 5.9153603471349916 10.907689156519266 1.2497706266337889 1.7936894494748687 1.2451632958186589 2.287981361830929 4.5048447730609942 0.25565256456320629\nleaf_count=42246 2806 89142 37624 3427 566 420 5083 312 1301 1633 146 154 44 135 1086 1352 2756 26 512 3981 4219 4905 10098 10412 98 523 731 18926 1273 32\ninternal_value=0.000937337 -0.000510698 -0.0126492 0.00502285 0.00334898 -0.010374 0.00525987 0.00240711 0.00379449 0.00676432 0.000143958 0.00462673 -0.00840967 -0.0144234 0.00011452 -0.00864364 -0.0140882 7.21257e-05 0.0132845 0.0153403 0.011863 0.0174516 0.000704121 -0.00517249 -0.0143152 -0.0194828 0.000689704 0.00154003 0.00190123 0.0144181\ninternal_weight=262.059 194.128 8.56711 67.9303 60.6064 6.86643 53.7399 38.48 37.0633 28.1731 14.4862 12.1099 6.14597 5.13686 185.561 10.8425 8.33324 3.95073 13.6868 13.0487 15.2599 12.4361 174.719 21.1117 10.204 8.95421 3.03885 153.607 151.319 4.7605\ninternal_count=245969 216325 79870 29644 26838 3847 22991 18206 17894 12811 2112 1966 1812 1768 136455 5220 3868 1112 10699 9398 4785 8886 131235 21862 11450 11352 1254 109373 90447 1305\nis_linear=0\nshrinkage=0.03\n\n\nTree=273\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 7 9 2 1 14 2 7 5 14 0 13 21 5 5 1 9 2 5 1 1 14 7 1 1 5 1 19 5\nsplit_gain=1.45758 3.99179 2.0785 2.64625 2.14093 2.93137 3.55548 1.79871 2.45673 2.55962 4.04262 3.527 2.49514 2.31261 1.8 1.78584 3.29435 1.75676 1.73171 1.56367 1.67513 1.57065 1.46692 1.42687 1.41546 1.70561 2.83622 1.59017 1.37409 1.35714\nthreshold=0.030057938850762859 1.5000000000000002 3.1132812500000004 1.0000000180025095e-35 14.537487030029299 3.3033179733375269 0.050734622491290134 -13.633526802062987 1.2426757812500002 1.4980468750000002 0.45315309081690569 1.1495945197163422 -0.31691436769765968 2.5000000000000004 2.1386718750000004 3.9609375000000004 1.5661550609387962 1.0000000180025095e-35 8.3473243713378924 2.0131835937500004 2.9477339485081675 2.054932859494814 0.82879072691359035 2.1386718750000004 2.5144622974930591 2.1468275719512682 1.8691406250000002 2.9477339485081675 247.50000000000003 2.8623046875000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 17 3 4 5 7 -7 15 9 10 11 -9 -10 -11 -14 16 -1 -2 -8 21 -21 -18 23 -3 25 26 -15 -26 -16 -25\nright_child=1 22 -4 -5 -6 6 18 8 12 13 -12 -13 14 24 28 -17 19 -19 -20 20 -22 -23 -24 29 27 -27 -28 -29 -30 -31\nleaf_value=-0.022722433716866719 0.012055323129569926 -0.030886063222407903 0.0097071337575378083 0.016145742673569427 0.0036071264031514371 -0.029270613145706628 0.0058457625107511601 0.010279756182939455 -0.035803802775675957 -0.021526338636971148 -0.024632802485624886 -0.024370527968065141 0.010248172424768562 0.017863467234878964 -0.02725858637031402 0.022973850361188205 0.015372317214778296 -0.013922731330048128 -0.015435169596396542 -0.027006980257782826 0.0063876516935506035 -0.0007048871853131476 -0.0045790589328751248 0.015660550357227392 -0.02122797241464584 0.028231692964766252 -0.022496783700837446 0.01312603054072654 0.0035355760872929679 -0.015388118784208839\nleaf_weight=5.2717752544908754 9.9565752337742897 5.6832678418467726 20.650544706614426 7.6951932100180889 66.566671701124847 1.9934392913200998 47.477673374742153 8.5420906652930189 5.0331605510340296 1.6118419308878618 10.306140066180488 2.6579054425933455 2.449209319386453 2.9180363464032535 3.0087390589505958 1.5538540072094509 6.8749767528875116 2.0848683899026801 2.7530406284931805 2.2793847850662123 1.7638756493233811 17.633889903745057 7.9417315417186378 0.96008205731845953 1.6565325673764155 3.7357274666355806 1.6173023229346197 1.3292046870628837 1.6249541572764199 2.8331790392525136\nleaf_count=117931 1767 3554 8286 3053 39632 51 1060 11071 5415 5959 21446 533 2052 2289 4567 3425 2218 431 80 1336 90 2212 2527 26 60 101 2735 31 883 1242\ninternal_value=4.3641e-05 -0.00612346 0.000860163 -5.88604e-05 -0.000757282 -0.00293454 0.00258981 -0.00644725 -0.010264 -0.00584578 -0.011977 0.00073322 -0.0213977 0.00452556 -0.00799759 -0.00128727 -0.00302952 0.00683477 0.00440088 0.00125237 -0.0140607 0.00423076 -0.0149663 -0.0224033 0.00970782 0.0156192 0.00200689 -0.006478 -0.0177489 -0.00590192\ninternal_weight=258.465 29.4597 229.005 208.355 200.659 134.093 52.2242 81.8686 46.4908 34.3748 21.5061 11.2 12.1161 12.8686 7.0829 35.3778 33.8239 12.0414 50.2307 28.5521 4.04326 24.5089 17.4183 9.47653 11.2568 8.27107 4.53534 2.98574 4.63369 3.79326\ninternal_count=246063 9547 236516 228230 225177 185545 1191 184354 57142 44225 33050 11604 12917 11175 7502 127212 123787 2198 1140 5856 1426 4430 7349 4822 5216 5125 5024 91 5450 1268\nis_linear=0\nshrinkage=0.03\n\n\nTree=274\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 17 0 3 1 7 5 19 4 10 4 4 18 13 18 17 17 1 17 1 18 1 17 13 7 17 13 2 2 2\nsplit_gain=1.55324 2.32651 2.19224 2.67757 2.96073 2.58358 2.82429 3.39749 2.81254 2.25363 2.16147 1.96656 2.90164 1.74483 1.72679 1.7026 1.62686 1.65765 2.47658 1.82 4.73681 1.52367 1.50148 1.3041 1.25732 3.70561 2.46099 2.40003 1.46179 1.45346\nthreshold=1.1495945197163422 0.12980400503841663 0.97942481616843191 0.19435047505660144 0.87590939928379707 1.7006835937500002 3.7011718750000004 234.50000000000003 3.2895507812500004 1.4978661367769956 1.4160156250000002 2.4313964843750004 3.0677824455408698 -0.39543742724656211 2.9174053685313028 0.29595754201755942 0.058793162820948631 0.39417715631504779 1.0329219259280322 1.4475910192202701 2.249904835165133 1.4475910192202701 0.45436599119170595 -0.34166011882750691 0.89599609375000011 0.65522739201539382 -0.032724337246802453 28.11904335021973 31.708583831787113 -3.0241031646728511\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 10 3 9 23 8 -7 -8 11 14 -1 21 -13 -10 -3 -16 -2 18 22 20 -19 -6 -18 -5 26 29 27 -23 -29 -26\nright_child=16 2 -4 4 5 6 7 -9 13 -11 -12 12 -14 -15 15 -17 17 19 -20 -21 -22 24 -24 -25 25 -27 -28 28 -30 -31\nleaf_value=0.022463708803165026 -0.014435740850799326 0.0064171138593961717 -0.012748407240285219 0.016351186209031626 -0.020894111513340006 -0.03180445264867543 -0.02977134084163777 0.0067448435682554292 -0.01315783632626802 0.0060348225913004847 0.00040875881983388286 0.0050131853160499257 -0.029182571607795436 0.02614173876593413 0.0035907127600680153 -0.029059822766450655 -0.0035659117504903788 -0.026420227730560689 -0.013279911506330235 0.0030103172447364856 0.012813702610753621 -0.014861830812780084 0.030476610000682212 -0.0061820206574728434 0.0027263694838368952 0.017515016277924528 0.013530893407719421 0.023404141323945842 -0.014118541732050981 -0.023863502602198834\nleaf_weight=4.87545541978196 4.1415613106482843 1.5438112523549978 20.27247221134644 11.358942714727617 1.6666195550982355 4.0601384596161543 2.1763892424869464 7.3639296645276318 0.60683946246058429 3.5875055897140546 12.64368258908854 3.500705298988577 4.1422849655174332 4.4006544931651197 1.4668646087596915 8.0099202064406807 1.2281254847889602 5.4494803282400408 1.4123638344801746 108.60113129168531 4.0421669157105953 11.471202267573512 4.5891865678713657 2.2856480804703656 3.6071063212780246 11.658204595848927 5.0979559032305177 2.0170264466487438 0.57569940926214247 2.4446772973860815\nleaf_count=5164 325 1767 6138 24742 8138 2010 457 632 443 1117 11275 153 2918 510 13254 133719 870 1638 2204 3001 945 5862 615 15858 385 787 764 75 208 79\ninternal_value=-0.000172996 -0.00246891 -0.00405064 -0.00204012 0.000215312 -0.00250296 -0.0136357 -0.00361962 0.000659499 -0.0133879 0.00742807 -0.00181472 -0.0147521 0.0199817 -0.0196866 -0.0241457 0.00214835 0.00280635 0.01584 0.00188211 -0.0100831 0.00105924 0.0238208 0.0124122 0.00257704 0.0081276 -0.00270162 -0.00909328 0.0134617 -0.00987571\ninternal_weight=260.298 130.834 113.315 93.0421 78.434 64.7894 13.6005 9.54032 51.189 14.6081 17.5191 46.1815 7.64299 5.00749 11.0206 9.47678 129.464 125.322 7.22968 118.093 9.49165 38.5385 5.81731 13.6446 36.8719 17.71 19.1619 14.0639 2.59273 6.05178\ninternal_count=246053 236455 220016 213878 64021 23421 3099 1089 20322 149857 16439 19369 3071 953 148740 146973 9598 9273 3689 5584 2583 16298 1485 40600 8160 1251 6909 6145 283 464\nis_linear=0\nshrinkage=0.03\n\n\nTree=275\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 12 0 18 3 17 7 7 17 7 18 18 7 21 7 19 18 1 17 1 19 3 6 3 17 7 9 21 19\nsplit_gain=1.59413 2.18156 1.31021 1.09916 1.40944 1.86083 1.54324 1.49064 1.47581 1.43364 1.32372 3.31082 1.79392 1.64309 1.54716 1.45243 1.28655 1.17117 1.88718 1.26278 1.1493 1.92107 1.82354 1.69221 1.65653 1.50013 1.38629 1.14878 1.88677 1.77957\nthreshold=0.030057938850762859 1.5000000000000002 0.32887916944499429 1.4460303622695712 2.0126758453675748 0.36709515926886871 0.28904399529622732 1.0000000180025095e-35 1.2426757812500002 0.96280603246018803 3.1132812500000004 5.2755571088971331 5.2176457913436565 3.2763671875000004 3.5000000000000004 2.6738281250000004 225.00000000000003 5.6401308701359305 1.7090960825073716 0.84061941967369014 1.7090960825073716 260.50000000000006 0.45898141303402712 1.0000000180025095e-35 0.53650715206675215 0.95146732080301388 1.2426757812500002 1.0000000180025095e-35 3.5000000000000004 233.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=3 2 -2 10 5 6 -5 16 -6 -8 11 12 15 14 -12 27 -7 18 19 -15 21 -9 -22 24 26 -26 -24 29 -29 -1\nright_child=1 -3 -4 4 8 7 9 20 -10 -11 13 -13 -14 17 -16 -17 -18 -19 -20 -21 22 -23 23 -25 25 -27 -28 28 -30 -31\nleaf_value=-0.015952902361867582 -0.011336817936199815 -0.01244616103518602 0.0086278377411238612 -0.024993565065944145 0.022505017302243489 0.013378510000186061 0.0074072582289501348 -0.026816567793414414 -0.004637862211612765 -0.014790451218805251 0.024187991923110032 -0.02251504496720028 0.021086904947544657 0.002754838750736655 -0.013166454738494059 -0.016909439743737566 -0.024593040870657147 0.013879613961180634 0.012404107259832559 -0.022303674532906794 0.018288400807852458 0.015617387446338424 -0.015255026709247225 0.020963517710652593 0.010372816522116683 -0.0063708226040274461 0.0080118894577881003 -0.012130850219957473 0.025887166702885323 0.00045757061780967866\nleaf_weight=5.2478722580835315 3.0987884453659289 17.004385612888598 9.9993807792217222 2.3087471106700823 5.9489143561654592 0.41419020472676438 6.0050680818676465 1.3665227892888645 2.0296274826005174 3.1863518266691244 5.3238703696927159 5.0435530200835492 2.5949555437506424 2.5858124906985704 0.53109296699335562 3.6238974977931084 1.6516895945424037 3.4812283937258135 3.0759833976771915 3.8018570988817828 10.368239076226017 0.77613566257059563 6.5106006287969667 3.774523446802049 16.971412132028494 5.811961173079907 2.5841768104583025 1.0302947964823936 2.617049708723016 122.77281548608329\nleaf_count=23750 1603 7331 597 352 721 17 411 105 167 1010 1768 61552 4490 193 1172 1089 166 4484 74 523 125 12 86 23 136 73 32 406 389 133299\ninternal_value=0.000734934 -0.0056475 0.00343546 0.00158581 0.00470591 0.00303544 -0.00741807 0.00552688 0.0158527 -0.00098413 0.000220339 -0.000781508 0.000169836 0.00749191 0.0193732 -0.000383137 -0.0150995 0.0011141 -0.00445942 -0.0131599 0.00670074 -0.0113673 0.00776581 0.00429024 0.00173848 0.00601329 -0.00850523 0.000195745 0.0148485 -0.000333566\ninternal_weight=261.541 30.1026 13.0982 231.438 69.7082 61.7296 11.5002 50.2295 7.97854 9.19142 161.73 142.93 137.887 18.7998 5.85496 135.292 2.06588 12.9449 9.46365 6.38767 48.1636 2.14266 46.0209 35.6527 31.8782 22.7834 9.09478 131.668 3.64734 128.021\ninternal_count=246156 9531 2200 236625 3436 2548 1773 775 888 1421 233189 224975 163423 8214 2940 158933 183 5274 790 716 592 117 475 350 327 209 118 157844 795 157049\nis_linear=0\nshrinkage=0.03\n\n\nTree=276\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 12 14 17 18 18 2 1 3 17 2 2 2 14 2 17 3 10 1 18 18 18 12 2 7 2 14 2 0 18\nsplit_gain=1.64561 1.408 1.6841 1.36126 2.44176 2.20651 1.83166 1.73417 1.61982 1.87522 1.47039 1.32908 1.55301 2.25338 2.20861 2.00017 2.48748 1.89846 4.23683 4.61714 2.50079 2.22726 1.77895 1.77015 1.54959 1.42711 2.19057 1.563 2.11238 1.40827\nthreshold=1.3771184689138856 0.63107571507209059 0.81584600691354858 0.95146732080301388 0.89587973461402759 1.7005986908310777 14.537487030029299 1.6109634896136267 0.42638606281916264 0.41124616060114544 62.792827606201179 20.132114410400394 -1.3165187835693357 0.25401570275144508 -3.0241031646728511 0.29595754201755942 0.1291727638644857 1.0000000180025095e-35 1.5661550609387962 1.4978661367769956 4.2121001622835372 4.4829865921051741 0.025581085251720429 -14.450672149658201 1.0000000180025095e-35 -16.491968154907223 0.70437716445572607 -30.420168876647946 1.1851094193663554 2.249904835165133\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=11 3 -3 8 7 -6 -7 -5 9 -2 -8 12 14 15 17 16 -14 18 19 -1 25 -22 -21 -23 -13 27 -27 29 -29 -20\nright_child=1 2 -4 4 5 6 10 -9 -10 -11 -12 24 13 -15 -16 -17 -18 -19 20 22 21 23 -24 -25 -26 26 -28 28 -30 -31\nleaf_value=0.021698435899468186 -0.012820424295055553 -0.026810319919712198 0.001382714476810367 -0.015507855098922923 -0.016182475964383866 0.012234967908283548 -0.026252415462002288 0.015469694703617377 0.0090620089958183773 0.0068642465834759265 0.0089692147062056596 -0.019488389596311478 0.022745644065969468 -0.0048021823041976353 0.013678399380987605 -0.028152915360963524 -0.02264389605995102 0.010205667859630022 0.0021351527642700452 -0.032035950052828312 0.021414533583679236 -0.011496219375343349 0.00048919005256775063 0.012566909421707233 0.0040655301909758379 -0.020574370105305769 0.0037430949564266252 0.011062122182354667 -0.012690204235573091 -0.016201343813977819\nleaf_weight=1.2950702213911416 6.9468151285627746 2.1795138277229844 4.5355768576264381 1.1491978384174233 10.926895526161845 5.9375566908156525 1.7160429475036383 7.807893319914001 41.17867997338908 8.6917639588682487 1.1076141912931232 1.7236388901529256 1.3547660908629651 34.943550742677132 6.6922348739146278 6.9970335171790907 1.0245567381557519 7.6154157420801303 6.3952787041489483 8.7373547510446468 6.0898524918495687 5.2002825360714242 1.756702442217885 3.9503236749078505 33.425280105396155 9.0008304724551618 4.5948853049903837 12.1231224239682 3.6154568239071514 7.5089668352422905\nleaf_count=211 986 85 152 282 873 423 298 118 527 602 365 10930 284 30364 3767 6888 484 4470 114 148916 899 3484 6850 683 19006 765 409 903 102 1287\ninternal_value=-0.000920049 0.00227179 -0.0100569 0.00334561 -0.00194294 -0.00775929 0.00332763 0.010337 0.0059994 -0.00212487 -0.0137039 -0.00271646 -0.00412034 -0.00860616 -0.00169656 -0.0209537 0.00228354 -0.00317432 -0.00474725 -0.0203921 -0.00130349 0.00879017 -0.0270224 -0.000893497 0.00240333 -0.00497952 -0.0126625 -0.00115747 0.00517359 -0.00819006\ninternal_weight=256.222 92.1776 6.71509 85.4625 28.6452 19.6881 8.76121 8.95709 56.8173 15.6386 2.82366 164.045 128.896 44.3199 84.5758 9.37636 2.37932 77.8835 70.2681 11.7891 58.479 15.2405 10.4941 9.15061 35.1489 43.2385 13.5957 29.6428 15.7386 13.9042\ninternal_count=245527 4711 237 4474 2359 1959 1086 400 2115 1588 663 240816 210880 38020 172860 7656 768 169093 164623 155977 8646 5066 155766 4167 29936 3580 1174 2406 1005 1401\nis_linear=0\nshrinkage=0.03\n\n\nTree=277\nnum_leaves=31\nnum_cat=0\nsplit_feature=13 4 5 4 19 22 17 2 21 13 14 5 19 17 5 5 3 1 13 1 14 18 3 3 3 1 14 13 7 3\nsplit_gain=1.21845 2.49425 1.85733 1.76007 2.52037 2.49247 1.63131 1.58918 1.56735 1.61361 1.49218 1.35218 1.93848 1.66333 1.60395 1.75247 1.33023 1.66127 2.23592 1.52701 1.50206 1.46926 1.71006 2.20986 1.54563 2.17783 1.38839 2.46166 2.24384 1.6805\nthreshold=-0.20472349212580346 3.4389648437500004 2.8037109375000004 1.9406738281250002 238.50000000000003 0.93750000000000011 1.8070006533037593 -5.9238033294677725 3.5000000000000004 -0.055817889458273311 1.0000000180025095e-35 3.3837890625000004 244.50000000000003 2.1810207941665634 3.8388671875000004 4.9179687500000009 0.10244777304966225 0.45969269984116679 -0.35333865714255164 2.9477339485081675 0.84146630540292722 3.4498615536424366 0.37963534769016083 0.62297257604742995 0.28287969384316608 1.6575619461558879 0.65891777326083356 -0.6866788914798736 1.7006835937500002 0.38657275885365588\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=16 3 -3 4 10 -5 8 -7 9 -6 -2 12 13 -4 -13 -16 -1 18 20 21 -18 24 -23 26 -19 -26 28 -28 -24 -30\nright_child=1 2 11 5 6 7 -8 -9 -10 -11 -12 14 -14 -15 15 -17 17 19 -20 -21 -22 22 23 -25 25 -27 27 -29 29 -31\nleaf_value=-0.017519370531393075 0.001299069322950598 0.017380831277290613 -0.029353912464773396 -0.020571161086275849 -0.0058479782057375236 -0.017777961286360923 -0.016961564128915425 0.0054229302117931319 -0.005557629177459977 0.01308542901650671 -0.031737464637427694 0.015150317764126311 0.0083422195768261473 0.0037376901980352229 -0.0098145137023774101 0.0081597223350709756 0.020219141705159253 0.0085424643139026465 -0.017192341264264673 0.005460430626911799 -0.012729951525048241 -0.0079770763589197147 0.021166345610969445 -0.014447190293144325 -0.016087868913993181 -0.0017874067408025906 0.012822067787644938 -0.022358170840944241 0.017970231048492494 -0.023682956030856444\nleaf_weight=1.9874277476635587 1.762586341305165 6.5939914134600022 3.2756362700175705 12.364282929129786 4.737507363974319 2.7075787933096658 2.8033766333899157 8.4102257372579725 9.8456150845337316 15.332389002856987 2.5671021699609469 7.3631819977641184 2.9675672069935786 1.7081411582548183 8.668671991718897 8.8681817837997396 10.4006772173235 7.7134055028302546 1.1580433380570418 60.485113198183171 0.72334058547267055 6.4064491175848302 12.431013572395392 2.4133728633487399 11.553110853423275 41.876376044217977 3.7182823825164668 1.8850725386625935 0.5005952373794561 1.3420471991466909\nleaf_count=46961 3166 4234 2284 17111 6741 4174 12774 1287 4861 4825 9883 5536 3379 2566 3454 984 26893 3342 25578 1425 18495 22802 2845 654 2987 2012 1898 2817 13 306\ninternal_value=0.000798606 -0.0018025 0.00400632 -0.00559147 -0.00142569 -0.0118373 0.0016423 -0.00122813 0.00387791 0.00855075 -0.0205682 0.000887616 -0.0091926 -0.0192829 0.00433718 -0.000775304 0.00237989 0.0027287 0.01291 0.00181703 0.0172034 -0.000669021 0.00482821 0.00869305 -0.00328527 -0.00511837 0.0120601 -0.000606784 0.0167468 -0.0100261\ninternal_weight=264.57 99.976 39.4454 60.5307 37.0486 23.4821 32.7189 11.1178 29.9155 20.0699 4.32969 32.8514 7.95134 4.98378 24.9 17.5369 164.594 162.607 12.2821 150.325 11.124 89.8397 28.6968 22.2904 61.1429 53.4295 19.877 5.60335 14.2737 1.84264\ninternal_count=246287 87259 22437 64822 42250 22572 29201 5461 16427 11566 13049 18203 8229 4850 9974 4438 159028 112067 70966 41101 45388 39676 31335 8533 8341 4999 7879 4715 3164 319\nis_linear=0\nshrinkage=0.03\n\n\nTree=278\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 17 18 18 18 19 22 17 0 5 3 12 14 17 2 2 2 3 18 2 2 18 14 14 13 5 18 4 14 17\nsplit_gain=2.60829 1.73263 1.61794 3.47686 1.99008 3.40054 2.56215 2.41903 2.15525 1.81553 1.68484 1.54886 2.01475 1.40413 1.51751 1.68199 2.22994 1.5997 1.38072 1.35513 1.71429 1.34746 1.19234 1.53122 2.12088 1.33354 1.38781 1.58841 1.24432 1.23371\nthreshold=1.1495945197163422 0.048554954793221271 2.8029010331479989 2.9174053685313028 3.1132683346437333 254.50000000000003 1.0000000180025095e-35 1.5316304627735702 -0.25794343014581789 1.2426757812500002 0.58795853567030554 0.63107571507209059 0.81584600691354858 1.2286103330956608 -5.1300392150878897 -1.0000000180025095e-35 2.5338792800903325 0.48555671035752257 3.860674306308975 3.4383306503295903 9.8937015533447283 2.3502401828962087 0.97222286375224221 0.65891777326083356 -0.34166011882750691 2.1386718750000004 3.4498615536424366 1.9753417968750002 0.22768215226658742 0.71600374927799881\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 9 -4 5 -5 7 8 -6 21 -11 -3 -13 14 -10 17 -17 -16 -15 -18 -21 -1 23 24 25 26 -8 -28 -26 -29\nright_child=1 11 3 4 6 -7 22 -9 13 10 -12 12 -14 18 15 16 19 -19 -20 20 -22 -23 -24 -25 28 -27 27 29 -30 -31\nleaf_value=-0.0070429583239170054 -0.01718570068611696 0.0039211581180967105 0.014902094013889948 -0.032602001983840362 -0.016315896193851986 0.0048732584284841753 -0.017206139388400946 -0.0081307822636605322 -0.0046477840979169604 -0.0093977951461357134 0.0094530286084383384 -0.025175888788443235 0.0029206475777425254 -0.0090830114301406351 0.026912048737195819 -0.025412169188702653 0.025610908040964008 -0.012382599161439335 0.023487686993247953 -0.019691067178856395 0.007516654944544383 -0.02943679394431745 0.015244914686022583 -0.024344682274459909 -0.0041957634447478435 -0.018738171676935544 -0.0035515429746714872 0.026775979628525026 -0.027243646690117754 -0.0063513221269829055\nleaf_weight=5.5257518586981877 2.9526572964887228 116.72197290630129 9.1565119311145491 4.2216218464794784 2.8391301540966003 3.3469721946114532 0.97168101519355943 15.82939538606473 10.416732802511882 12.644261105309626 5.2840968850068739 2.93686065712245 5.0813109999871813 0.85439633916769153 4.9054057204226664 1.1770681688885378 1.8986383115235401 0.50684759741852747 6.0249062061851077 1.6034832731663744 9.6235106108883866 4.8868352369803461 0.77858176783820354 5.2634623527658837 4.3523313444526952 1.2778660005756695 3.7103654254138387 3.9273029007804325 4.0641836220569303 0.85435264556969048\nleaf_count=1470 289 8920 1765 2969 13757 831 783 68937 28632 2434 72 100 182 1257 3640 2016 625 127 7438 3593 12269 2802 1952 24338 10929 7673 10267 1756 22770 1597\ninternal_value=-0.000585271 0.0024269 -0.00363429 -0.00175737 -0.00362934 -0.0173951 -0.00214445 0.00147119 0.00539099 -0.0098181 -0.00363593 0.00302412 -0.00902096 0.00744157 0.00432697 0.00906439 0.00310407 0.021875 0.0188031 0.0072793 0.0023381 -0.0192113 -0.00988442 -0.0112537 -0.00662788 0.00228408 0.00664217 0.0108962 -0.0170364 0.0207822\ninternal_weight=253.638 127.693 125.946 97.6047 88.4482 7.56859 80.8796 55.6795 39.8501 28.3409 17.9284 124.74 8.01817 37.011 30.1317 19.715 14.3027 5.41225 6.8793 13.1256 11.227 10.4126 25.2001 24.4215 19.1581 10.7416 9.4637 8.49202 8.41651 4.78166\ninternal_count=246190 9491 236699 229921 228156 3800 224356 142291 73354 6778 2506 9202 282 59597 50902 22270 18503 3767 8695 16487 15862 4272 82065 80113 55775 22076 14403 13620 33699 3353\nis_linear=0\nshrinkage=0.03\n\n\nTree=279\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 13 0 1 7 4 4 21 7 1 4 0 14 1 14 14 17 0 11 1 17 17 5 2 0 21 2 2 2 1\nsplit_gain=1.54214 1.10554 2.0324 2.37377 2.29016 2.08285 1.74741 2.70149 2.59127 2.28407 2.23995 1.90021 1.7114 1.51339 1.49836 1.19341 1.15392 1.14609 1.09104 1.04903 1.15775 1.03625 1.0313 1.73109 1.03015 2.79517 1.39169 3.51128 1.13713 1.05905\nthreshold=-0.79535375801325292 -0.43125657894431851 0.38644940889434637 0.35607680265790181 3.4804687500000004 1.9406738281250002 2.6030273437500004 6.5000000000000009 1.2426757812500002 0.98142480879535021 2.2978515625000004 1.6765212893675572 0.12667907975775003 -0.58235837231666587 0.032916404154818342 0.060350501553964049 0.10726386778582141 0.84505259813102007 1.0000000180025095e-35 1.1267314367267358 0.068176441062887241 0.077783200051907503 1.7006835937500002 8.8705892562866229 0.97942481616843191 1.5000000000000002 -9.8266754150390607 -22.80348968505859 -41.62170219421386 1.7622051334857016\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 2 4 5 -2 17 7 9 12 13 19 -9 -8 -3 -10 -16 -14 -4 -18 20 -11 -21 -23 -24 28 -26 27 -27 -5 -30\nright_child=1 6 3 24 -6 -7 8 11 14 10 -12 -13 16 -15 15 -17 18 -19 -20 21 -22 22 23 -25 25 26 -28 -29 29 -31\nleaf_value=-0.021167964911923319 -0.022668437699424854 0.0077883833543888944 -0.0070304084213343851 -0.011963017527765841 0.0096001535577591862 -0.0060200796247910802 -0.0042620792383771273 -0.0038168861937420756 0.0096133846222288839 0.026212343199121208 -0.013005493749879565 -0.031890519990196542 -0.010838593393205009 -0.025662450201552962 -0.019932429045066417 -0.00097827157021441145 0.023640631202102771 0.036242446258512008 -0.0065052832884819776 -0.011595216416196546 -0.01037313144485584 0.00092293324853434514 -0.0085875134042401133 0.025090336153092446 -0.019868569804108501 0.0055767152133411584 0.007557986684313703 -0.019415172992565968 0.0010034354005320609 0.014623147593254197\nleaf_weight=1.8355749994695418 4.2442878307664378 1.0774515233053668 0.29518285368999331 1.4358543787246869 2.4598261824326073 1.3644174258756718 3.273781219712645 4.6163116271673044 10.899979575282428 1.915531986621317 9.435299396123126 3.6943691432516061 0.50954907910056491 4.2425432291466363 2.227026991241245 42.575819923904561 10.241705410052017 3.58978658045055 0.77758406564532911 4.4124394314703741 0.45273100285703538 60.682736170958833 1.4704128892508368 3.0249493968892693 4.1578115098018307 15.098326939625851 28.465219832331059 6.3971953763057163 7.7369123296572324 15.273640242509826\nleaf_count=44617 29826 8155 8753 2003 816 4657 1495 10059 3015 90 1631 224 478 60725 1222 33154 6737 1858 411 792 2302 10774 593 278 529 470 1494 344 6780 1350\ninternal_value=0.000843786 0.00108332 0.00372948 0.00503431 -0.0111194 0.0228831 -0.000377411 -0.00301921 0.00317554 -0.00135413 -5.0901e-06 -0.0183813 0.0138977 -0.0187273 0.000131345 -0.00233504 0.0189831 0.0321884 0.0211491 0.00185443 0.0182152 0.0010475 0.00206566 0.0145164 0.00356911 0.00120427 0.00331345 -0.00239346 0.00855125 0.0102763\ninternal_weight=257.884 256.049 90.5185 83.8143 6.70411 5.24939 165.53 95.0248 70.5054 86.7141 81.3941 8.31068 14.8026 5.31999 55.7028 44.8028 11.5288 3.88497 11.0193 71.9588 2.36826 69.5905 65.1781 4.49536 78.565 54.1186 49.9607 21.4955 24.4464 23.0106\ninternal_count=245632 201015 58880 28238 30642 15268 142135 95623 46512 85340 16460 10283 9121 68880 37391 34376 7626 10611 7148 14829 2392 12437 11645 871 12970 2837 2308 814 10133 8130\nis_linear=0\nshrinkage=0.03\n\n\nTree=280\nnum_leaves=31\nnum_cat=0\nsplit_feature=0 18 1 19 6 1 21 2 13 16 17 5 6 6 2 2 1 2 3 18 2 13 5 14 3 0 5 0 13 2\nsplit_gain=2.0202 2.22525 3.14272 2.15334 2.21361 1.55691 2.39876 2.26523 1.95405 2.3026 1.62537 1.49832 1.40073 1.31616 1.29163 1.47878 2.71657 2.4928 1.44675 2.2777 2.25924 1.23406 1.15462 1.15359 1.78212 1.63665 1.53895 1.41452 1.37142 1.1477\nthreshold=1.4460303622695712 2.0126758453675748 1.8912989126705502 260.50000000000006 3.3837890625000004 2.054932859494814 2.5000000000000004 16.375147819519047 -0.34166011882750691 1.0000000180025095e-35 0.13727533581160231 5.3203125000000009 1.7006835937500002 2.5244140625000004 -18.054428100585934 12.256900787353517 0.15052965826911111 20.132114410400394 0.40160889672495398 4.8242976514536702 10.479034423828127 -0.0029195251563053355 1.7006835937500002 0.6618576200563534 0.41780940659066851 0.54195883642662512 4.1367187500000009 0.87173191741369926 -0.76787006514439582 7.3050365447998056\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 2 3 4 -2 6 11 8 23 -10 13 14 22 -7 -1 16 -16 -17 -18 20 -20 -19 29 24 26 -26 -8 -27 -25 -3\nright_child=1 12 -4 -5 -6 10 7 -9 9 -11 -12 -13 -14 -15 15 17 18 21 19 -21 -22 -23 -24 28 25 27 -28 -29 -30 -31\nleaf_value=-0.010944702670993418 -0.026020596946590025 -0.014122267352856897 0.0046598436459212994 0.011416661000987278 0.010278606391763132 -0.01705062592091541 -0.023034046956524331 0.0066406693233064295 -0.028090347910324841 0.00099067065306221208 -0.0026979599068519661 -0.015054917654877462 -0.013894873439551175 0.013451106280767977 -0.018661150646686782 -0.022953934268339249 0.02295601984170511 0.011274064107536386 -0.027035954478894845 0.021110301392217565 0.014874165240850024 -0.0076354189576000641 0.025091353469319202 0.0045648393210869914 -0.011680705461832076 0.027053158492567383 0.011119838431207948 -0.0058759943435111968 -0.018730290302861646 0.01502982278599705\nleaf_weight=2.2840650695339368 6.5465270555719126 1.2377707569903544 59.63189537137805 1.9104881328312329 1.3136199337895957 10.989196371569047 2.1276641786232653 12.027270802771453 6.3560559478036982 3.5751048066151725 78.535155278685579 1.8618872033981677 0.57442854123655784 0.68588962487410832 1.2348756627130821 1.9622649038416309 12.763917131930841 12.064151083275874 1.9416536799294584 2.5324091144411796 1.0939165563213462 3.2998267984508489 7.0083411797022563 3.1932144591172573 1.1966260835065439 4.00667961075942 1.1528262364146256 1.1267682959805823 4.8352671219056429 1.682745021902974\nleaf_count=46428 1691 157 947 117 51 915 22384 14868 52933 2626 6674 774 32 18 20414 6987 11305 9409 932 545 98 6835 351 7759 747 371 734 183 28417 415\ninternal_value=0.000182515 0.0040907 0.00203372 -0.0129385 -0.0194787 -0.00165853 0.00136356 -0.00385637 -0.00850777 -0.0184886 -0.00434522 0.00637207 0.0163242 -0.0143389 0.00773976 0.00915483 0.0146082 0.00253513 0.0176987 0.00398678 -0.0119897 0.00699433 0.019184 -0.00219792 0.00463574 0.0132729 -0.0112379 0.0200458 -0.00998561 0.00222389\ninternal_weight=250.753 79.9058 69.4025 9.77064 7.86015 170.847 80.6364 39.5975 27.5702 9.93116 90.2102 41.039 10.5033 11.6751 39.1771 36.893 19.5668 17.3262 18.3319 5.56798 3.03557 15.364 9.92886 17.639 9.61056 6.33007 3.28049 5.13345 8.02848 2.92052\ninternal_count=246117 3761 2806 1859 1742 242356 234749 131022 116154 55559 7607 103727 955 933 102953 56525 33294 23231 12880 1575 1030 16244 923 60595 24419 1301 23118 554 36176 572\nis_linear=0\nshrinkage=0.03\n\n\nTree=281\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 12 21 4 3 1 0 21 18 9 18 2 0 18 3 21 0 17 1 22 19 1 17 2 2 1 3 18 0 17\nsplit_gain=1.63819 2.67313 2.77445 2.01719 1.78416 1.77539 1.70245 1.61675 2.13064 2.96441 1.54777 2.40953 2.20485 2.07494 3.09732 2.62767 1.72104 1.81815 2.32903 1.93174 1.67695 1.6726 1.55292 1.50802 3.24663 1.37375 1.30179 1.82903 1.25491 1.56699\nthreshold=0.41124616060114544 8.8817841970012543e-16 1.5000000000000002 2.1186523437500004 0.51817819236196982 1.6575619461558879 0.34392793676569194 3.5000000000000004 3.1132683346437333 1.0000000180025095e-35 5.2067464199591091 -20.765821456909176 0.77696153631098397 3.9796379800581989 0.21010136461729736 2.5000000000000004 0.87173191741369926 1.2133651393498066 0.35607680265790181 1.0000000180025095e-35 262.50000000000006 1.6109634896136267 0.69259390652607344 -4.708568572998046 -19.19599628448486 1.4475910192202701 0.43673077644757391 2.1383330595080281 0.17863154480046692 2.1810207941665634\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=7 4 3 -3 10 -5 -6 23 26 -10 12 -12 13 20 -15 -16 22 18 -18 21 -2 -19 -14 24 -1 -20 27 -9 29 -13\nright_child=1 2 -4 5 6 -7 -8 8 9 -11 11 28 16 14 15 -17 17 19 25 -21 -22 -23 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=0.0051469530409115199 -0.027872770626821239 0.020896357562681808 -0.017827332752530536 0.0077639098158582415 -0.019263608641495001 -0.020218003554474687 0.0075692842027098662 -0.02195028110628492 -0.029347566550579914 0.0082097836121147183 -0.01854986818003437 0.024989237232822353 -0.012579588922608315 -0.02434642613077893 0.013985680213805196 -0.013699474958658376 0.022882771249265601 -0.025378259514686082 -0.016161051040771517 0.0057531395959583599 0.0061132982853265138 -0.0041676098504196081 0.020061406643967408 0.0059500366099605601 -0.026504481019898076 0.002882304687837484 -0.00026011460731762993 0.0038080999817945776 -0.014147805836274436 -0.005041861860336061\nleaf_weight=6.8144133078482145 5.5684262347163012 2.8588024220423467 11.358537125291546 3.4725786825142109 1.2711993834946032 2.8908701697282595 59.966484392548409 5.8752381245976704 6.3609271946365862 2.0480860872799553 1.2814293062677951 6.783104535132221 0.8968082682313242 3.326309146625106 8.2371335408648729 3.7017718127348749 3.9819738732867327 5.1026550394740324 2.8155821452226055 6.4123027425362116 1.2361889020980923 8.4957969379356637 6.0347686333992669 20.614273563405334 3.7854227586598199 31.204209663974325 24.852220223474433 3.4756074650410937 0.409940970822391 1.5837114558915457\nleaf_count=14375 15319 594 5804 1334 1167 111 2717 719 22291 431 45278 5945 942 61502 11859 10515 2754 6169 1914 2431 1121 999 4422 9624 4538 1168 503 1599 1187 6725\ninternal_value=-0.000140524 0.0014117 -0.00850876 0.00358962 0.00274671 -0.00596744 0.00671171 -0.00387036 -0.0076033 -0.0202998 0.000202462 0.0107793 -0.00112872 -0.00910124 -0.00249332 0.00500628 0.00167749 -0.000112614 0.00372188 -0.00722513 -0.0217064 -0.0133199 0.0147847 0.0013035 -0.00746689 0.00088974 -0.00399381 -0.0129332 0.0165208 0.0193734\ninternal_weight=252.717 178.891 20.5808 9.22225 158.31 6.36345 61.2377 73.8262 42.6121 8.40901 97.0721 10.0582 87.0139 22.0698 15.2652 11.9389 64.9441 58.0125 38.0018 20.0108 6.80462 13.5985 6.93158 31.2141 10.5998 34.0198 34.2031 9.35085 8.77676 8.36682\ninternal_count=246057 191977 7843 2039 184134 1445 3884 54080 25543 22722 180250 59135 121115 100316 83876 22374 20799 15435 5836 9599 16440 7168 5364 28537 18913 3082 2821 2318 13857 12670\nis_linear=0\nshrinkage=0.03\n\n\nTree=282\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 18 12 4 17 3 14 5 4 13 13 10 14 4 1 0 18 21 18 4 4 3 3 13 13 13 18 14 19\nsplit_gain=2.71349 3.3427 1.76951 1.75446 1.73222 1.36085 1.28671 1.10433 2.33055 2.48094 1.64247 1.58497 1.5778 3.21324 1.66087 2.03061 1.39231 4.65564 2.03805 1.47944 1.61504 2.52469 1.75095 1.60275 1.99911 1.54336 2.78348 1.46752 1.46496 1.26449\nthreshold=0.030057938850762859 1.5000000000000002 2.1383330595080281 0.76965903234489019 2.1186523437500004 0.70050815440916869 0.47260169401963864 0.6254473112327732 2.1386718750000004 3.1782226562500004 -0.43125657894431851 -0.05277688766252419 1.0000000180025095e-35 0.38774031279192267 2.2978515625000004 2.5144622974930591 0.49902597412172423 4.1820210059610323 3.5000000000000004 2.4414009612931857 1.0693359375000002 1.4716796875000002 0.44825707527455844 0.23327427011668259 -0.30805170713240548 -0.49896153351186062 -0.32019508363135163 1.7005986908310777 0.56086626140189244 255.50000000000003\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=7 4 -3 -4 -2 -6 -5 8 9 16 -11 -9 13 -10 -15 -16 -1 19 -19 20 22 -22 27 -23 -25 -21 -27 -18 -24 -29\nright_child=1 2 3 6 5 -7 -8 11 12 10 -12 -13 -14 14 15 -17 17 18 -20 25 21 23 28 24 -26 26 -28 29 -30 -31\nleaf_value=-0.010073407474169441 0.016968342000842022 -0.025830303089867512 -0.017770647235058036 0.017834890165058376 0.0087753251836797282 -0.014756740680081312 -0.0082208515848702177 0.0028014103744621499 -0.0030145590349017889 -0.013708260358515476 0.027674398795522772 0.018614253698731993 0.0056516656053005945 0.011834459331129947 -0.035549628767116835 -0.0035661410209593728 0.0038614237404401338 0.0331697783791884 -0.0091387839226178755 0.010967004389019123 -0.019656948222734002 -0.019323019932399751 -0.0087318778432411825 -0.0052537229742964453 0.0094773017086253949 -0.024407059946762351 -0.0016158298848587269 0.028622546437608472 0.012075340436110396 -0.010146114447083373\nleaf_weight=8.339696786110947 4.4282672023123286 8.0403414619431732 4.8642923087528498 2.4892498082772363 3.0028253972277525 4.149392804864533 2.4922877550707199 73.246773700726322 24.34979391093583 0.41382000361511462 3.9540461298929577 5.2569320523242586 7.4049046456689203 0.5644284309428248 5.7515747922014935 2.7589675767003428 5.3891570963637623 4.7016045403645412 0.79737479324347749 3.3870744372021617 4.5883306312812229 1.9378945700514112 6.2725794462094182 12.755403824448875 20.286918337824318 6.026108873727428 15.88560669139957 5.9494508605143892 4.239316512044752 0.35418761336450177\nleaf_count=50509 717 311 6892 123 456 1057 83 95770 24697 202 965 12723 1690 86 16766 122 735 2138 1453 1170 900 840 167 257 341 5039 18221 1111 54 153\ninternal_value=0.000101213 -0.00830284 -0.0161074 -0.00651785 0.00407349 -0.0050124 0.00560405 0.00123565 -0.000310567 0.0020525 0.0219304 0.00408112 -0.00630709 -0.00904364 -0.0233159 -0.0266463 0.000985347 0.00207863 0.0265727 0.000247579 0.00274739 -0.000915563 0.0090329 0.00202073 0.00380337 -0.0057287 -0.00867551 0.0165324 -2.05652e-05 0.0253533\ninternal_weight=254.079 29.4667 17.8862 9.84583 11.5805 7.15222 4.98154 224.612 146.108 105.279 4.36787 78.5037 40.8297 33.4248 9.07497 8.51054 100.911 92.571 5.49898 87.072 61.7732 39.5685 22.2047 34.9802 33.0423 25.2988 21.9117 11.6928 10.5119 6.30364\ninternal_count=245748 9639 7409 7098 2230 1513 206 236109 127616 84255 1167 108493 43361 41671 16974 16888 83088 32579 3591 28988 4558 2338 2220 1438 598 24430 23260 1999 221 1264\nis_linear=0\nshrinkage=0.03\n\n\nTree=283\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 14 1 7 19 14 19 2 2 5 2 19 17 17 1 7 7 7 5 7 2 2 7 19 9 13 5 17 13\nsplit_gain=1.61396 2.69736 1.60635 1.19265 1.11947 1.78681 1.11025 1.15386 0.985459 2.77326 1.64926 1.52614 5.15704 1.39644 1.39205 1.48048 1.38435 1.88651 4.70109 2.69032 1.66706 1.91752 2.12763 1.68648 1.19936 1.1853 1.78892 1.47855 2.24852 2.06432\nthreshold=0.030057938850762859 1.5000000000000002 0.82879072691359035 2.5144622974930591 2.0131835937500004 245.50000000000003 0.85380319876708544 244.50000000000003 29.286581993103031 34.417465209960945 1.2426757812500002 24.399772644042972 245.50000000000003 0.32133416703193302 0.50811570171766973 1.4866206336930319 4.0859375000000009 2.5244140625000004 1.8691406250000002 4.1015625000000009 2.6738281250000004 -14.08206272125244 11.680273056030275 3.2382812500000004 222.00000000000003 1.0000000180025095e-35 -0.34166011882750691 1.7006835937500002 0.068176441062887241 -0.66255419986965325\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=8 3 4 -2 -3 -6 -4 -8 11 10 -10 16 -13 -12 -11 24 17 18 25 -20 -19 -22 23 -23 -16 27 -27 -1 -29 -30\nright_child=1 2 6 -5 5 -7 7 -9 9 14 13 12 -14 -15 15 -17 -18 20 19 -21 21 22 -24 -25 -26 26 -28 28 29 -31\nleaf_value=0.0028228973827532132 -0.0032397445984118601 -0.029321710760301434 0.013067531229912551 0.015127399101614666 -0.017881273762431645 0.015861524456223082 0.0059602496894036391 -0.017434939504815984 0.027127884909680568 -0.010698142528801854 0.023310701146151476 -0.033099085725294859 0.0091964708730006266 -0.0048743497140078607 0.0030830970966794829 0.0082399422873696004 -0.01759837932673735 0.022679782514329453 -0.02998033773177973 0.0083052740366846914 -0.016117791101902198 -0.0028804855604125544 -0.012656242323282657 0.020213821091875549 -0.020063763989225082 0.028852871812311519 -0.0094408162993279022 -0.025122498964021817 -0.018005600129675746 0.0011380705685616051\nleaf_weight=97.483701797136803 6.5729701183340143 5.2514672163756035 1.9311572465455222 4.6963267830451514 3.1748333840753276 1.2257046894269477 2.4423092377641069 3.908308763394075 5.8540326871267876 6.7702146243259476 1.2997610638126387 3.9385623439935085 4.7658881397234154 4.2770464960921686 4.0189688504783048 17.717139713949564 2.595675385758681 5.1011743277001385 5.9384522878191595 1.684245031717239 2.5506357861522746 3.7228194240415133 2.6821564954323103 8.6034294724290685 2.6334768776481496 2.1651264695825607 1.0803581440544565 3.0444337699814232 4.9078503857637079 35.065488717313656\nleaf_count=105670 1997 3562 455 188 515 815 607 1474 2266 2372 170 2895 1836 1260 5480 1461 563 6502 23227 556 18970 1667 526 737 8627 300 302 2008 6150 42872\ninternal_value=0.000302606 -0.00621698 -0.0131726 0.00502361 -0.0209024 -0.00728599 -0.00288264 -0.00885083 0.00116153 0.00522132 0.017201 0.000205975 -0.0113799 0.00424018 0.000425455 0.00381563 0.000837804 0.00121176 -1.94741e-05 -0.0215389 0.00908955 0.00413207 0.00836534 0.013546 -0.00750443 0.00126259 0.0168845 0.000784966 -0.00379246 -0.00159443\ninternal_weight=257.104 29.2031 17.9338 11.2693 9.65201 4.40054 8.28178 6.35062 227.901 42.5706 11.4308 185.33 8.70445 5.57681 31.1398 24.3696 176.626 174.03 151.37 7.6227 22.6602 17.559 15.0084 12.3262 6.65245 143.747 3.24548 140.501 43.0178 39.9733\ninternal_count=246030 9613 7428 2185 4892 1330 2536 2081 236417 21636 3696 214781 4731 1430 17940 15568 210050 209487 181085 23783 28402 21900 2930 2404 14107 157302 602 156700 51030 49022\nis_linear=0\nshrinkage=0.03\n\n\nTree=284\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 18 11 19 5 3 17 4 7 13 7 18 18 3 13 7 5 2 13 18 0 0 4 19 2 17 13 18 18\nsplit_gain=2.1979 3.77163 1.62867 2.04825 1.93066 1.55081 1.72489 1.25768 1.15643 3.72675 2.81984 1.80897 2.66776 2.11397 1.95897 1.79388 2.05941 2.13706 1.68063 1.38936 1.58108 2.27181 1.65318 1.62509 2.92043 2.99818 1.96539 1.91344 1.51705 2.04769\nthreshold=0.50308954086190749 1.5000000000000002 2.6735537653587351 0.66750229771629133 262.50000000000006 5.3300781250000009 0.34948566548243237 5.817026225953116 5.5515136718750009 1.4980468750000002 -0.74650584208841086 2.6738281250000004 5.2755571088971331 5.5664324555418956 0.20268037713056589 -0.1384351843311358 2.3505859375000004 2.8037109375000004 -13.633526802062987 -1.0000000180025095e-35 3.1984648276080736 0.17863154480046692 0.77696153631098397 1.6286621093750002 231.50000000000003 -18.054428100585934 0.96280603246018803 -0.058978139479729179 2.4414009612931857 2.8029010331479989\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 -2 4 -4 -3 7 -7 8 9 19 -11 12 -10 14 -13 -12 17 -17 -15 20 22 -22 -1 24 -24 28 -27 -25 -26 -30\nright_child=1 2 3 -5 -6 6 -8 -9 11 10 15 13 -14 18 -16 16 -18 -19 -20 -21 21 -23 23 27 25 26 -28 -29 29 -31\nleaf_value=0.024145940080026827 0.0089304415416879886 -0.028295716240115888 -0.02709192906976662 0.0059011903631866688 0.0059983676555527216 0.00074781441322778626 -0.029082491960799848 -0.018891314200823912 0.020801346154480156 0.018645045882109839 -0.024433413948577537 -0.027470718658937614 -0.017463081392377053 -0.011631463412172652 -0.00011114855372187392 0.0022243571508687517 0.013434790696381699 -0.029159838225895417 0.023333803402608318 0.0093361626606803579 0.013478505312717026 -0.0081752967852075538 0.027664036008503264 0.0057004928474068241 0.010891168950988133 -0.016664281813742587 -0.00058632459805936347 -0.012382331096006128 -0.027400601163101473 0.0096256391889091436\nleaf_weight=1.9952909989073067 6.599547958966471 8.8688824498676713 1.3793363311258406 5.0259329607306427 1.3707787254825223 3.4896514261741904 2.3939709633341408 1.940504535858081 9.5374187594617279 1.6013947002470583 8.9883079661698311 1.8025785583573739 1.1582605672094235 0.95216376969580174 14.810739925424057 3.5858801653810097 2.7429105301470145 2.969279869072011 4.1705248994257982 16.866920896699924 4.168008721824207 29.348150572433298 1.6729764729752776 66.477660389532275 11.619235792790727 16.597427578682762 11.319459038217927 4.779059823064471 1.635353641851679 1.9763980421881795\nleaf_count=502 297 275 608 322 27 1677 150 42154 990 1854 13884 721 13378 13274 1058 2870 1045 1738 1116 16262 23322 92760 768 4995 693 3849 3347 500 557 930\ninternal_value=-0.00073535 -0.00932417 -0.0166579 -0.00390271 -0.0235796 0.000174785 -0.0138514 0.000605582 0.00086136 -5.10688e-05 -0.0120072 0.00600322 0.0155188 0.000844269 -0.00447083 -0.015519 -0.00536583 -0.0139696 0.0159965 0.00142299 0.000487093 -0.00516999 0.00212284 0.00154128 -0.00289494 -0.00467959 -0.0103909 0.00433313 0.00578386 -0.00944555\ninternal_weight=251.844 23.2445 16.6449 6.40527 10.2397 228.6 5.88362 222.716 220.775 188.344 19.8878 32.4317 10.6957 21.736 16.6133 18.2864 9.29807 6.55516 5.12269 168.456 151.589 33.5162 118.073 116.078 44.8209 43.1479 27.9169 71.2567 15.231 3.61175\ninternal_count=245923 1529 1232 930 302 244394 1827 242567 200413 169876 21391 30537 14368 16169 1779 19537 5653 4608 14390 148485 132223 116082 16141 15639 10144 9376 7196 5495 2180 1487\nis_linear=0\nshrinkage=0.03\n\n\nTree=285\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 2 2 4 4 3 7 4 3 5 19 17 14 6 17 14 5 17 19 4 17 17 2 1 1 14 0 5 7 14\nsplit_gain=3.61059 2.60917 1.79034 1.77182 3.09142 2.69118 3.09791 3.53872 2.29518 2.04198 1.80839 1.77698 2.96585 1.55046 1.48055 1.65625 1.35572 1.66763 1.3249 2.10368 1.29547 1.28414 1.26789 1.23502 1.19859 3.7758 1.37724 1.18119 1.16684 1.53269\nthreshold=8.8817841970012543e-16 29.286581993103031 74.891498565673842 1.0693359375000002 1.6286621093750002 0.31840721315068282 2.3505859375000004 2.1433105468750004 0.23327427011668259 5.3300781250000009 264.50000000000006 1.0587051740042546 0.80526343422453606 9.3476562500000018 1.2783113269342083 0.7674609687686843 1.2426757812500002 4.1669220475094004 249.50000000000003 1.9753417968750002 1.0796935653640076 0.011353531392579346 -31.266788482666012 3.3033179733375269 2.9477339485081675 0.42478442718135506 1.6765212893675572 3.4804687500000004 0.89599609375000011 0.55141373229880208\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 3 18 10 16 6 7 8 -6 28 11 12 22 -8 -3 -16 17 -5 27 -20 -10 -18 -1 -12 26 -26 -24 -2 29 -7\nright_child=2 14 -4 4 5 9 13 -9 20 -11 23 -13 -14 -15 15 -17 21 -19 19 -21 -22 -23 24 -25 25 -27 -28 -29 -30 -31\nleaf_value=-0.0050350488516249714 0.0074009826908558087 0.012471873942793872 -0.026303671412754637 -0.02458326561957614 -0.016148438511182691 -0.01555404991163953 0.0065650099094705901 -0.028183676263503082 0.024478042677316503 -0.02387180272319241 -0.022722862837385886 -0.0033173915034699643 -0.014090780424763685 -0.024421671808914195 0.0048176121084228711 -0.02085289145217125 0.024921572737830602 0.012036594510639976 -0.00079906755906622722 -0.027603694155275006 -0.011938083317284099 -0.013192521825587214 0.023554683327237164 0.010481874243060959 -0.016739419193751374 0.017629149000633126 -0.0088404575684128232 -0.0073248960032621041 0.0043052284003893106 0.0046981123046192924\nleaf_weight=2.4732167185452516 8.2668797164059331 29.214919484142456 3.4929645898027948 7.7846658941563174 2.5721710700385101 5.6211990038812001 11.151200267584127 8.6590296712001003 2.5077119916269344 1.6026330651429805 2.2747086856964525 10.156560378749786 3.0617625941669635 0.66237488277965972 14.832038320834103 1.6419709645907108 0.5944128089561207 0.67669483576446499 5.5170450128267685 4.1548887188261006 0.50318648656377707 0.70035886979732709 13.701750771637876 0.76597388228401531 3.2198093386396058 8.5059047241520584 0.781613970219042 9.4119029059029327 86.166584409839402 6.119060314148669\nleaf_count=5464 1835 6367 102 23366 18404 1165 33470 60093 1829 120 3239 25585 7242 6 9003 6310 20 5016 2361 3752 1408 4875 10493 10 96 104 76 2434 8589 3014\ninternal_value=0.000879842 0.00220393 -0.00858459 0.000563645 -0.00115675 0.000164198 -0.00823563 -0.0171889 0.00155426 0.00242367 0.00565628 0.00736019 0.0107735 0.00305729 0.00852486 0.00121208 -0.0165925 -0.020692 -0.00547371 -0.0138214 0.0169331 0.00754029 0.013813 -0.0138343 0.0157113 0.00761795 0.0213484 -0.000411274 0.00309111 -0.00545831\ninternal_weight=256.795 225.952 30.8437 180.263 135.321 125.565 26.0557 14.2421 5.58307 99.5095 44.9413 41.9006 31.7441 11.8136 45.6889 16.474 9.75613 8.46136 27.3507 9.67193 3.0109 1.29477 28.6823 3.04068 26.2091 11.7257 14.4834 17.6788 97.9068 11.7403\ninternal_count=245848 235364 10484 213684 161375 128098 115210 81734 21641 12888 52309 49060 23475 33476 21680 15313 33277 28382 10382 6113 3237 4895 16233 3249 10769 200 10569 4269 12768 4179\nis_linear=0\nshrinkage=0.03\n\n\nTree=286\nnum_leaves=31\nnum_cat=0\nsplit_feature=11 21 14 7 2 17 14 5 1 17 0 1 13 17 0 17 7 1 2 3 7 14 2 2 5 2 16 3 7 10\nsplit_gain=2.29741 2.99801 3.22986 2.50898 1.97361 2.36905 1.90271 1.86983 1.34547 3.19686 2.30204 2.29714 2.64589 2.79891 2.97146 2.17338 1.98743 1.84504 1.81056 1.58688 1.44323 1.65402 1.39229 1.3607 1.32862 1.68114 1.5734 1.45115 1.2567 1.16158\nthreshold=8.8817841970012543e-16 1.5000000000000002 0.81021705844416803 2.0131835937500004 29.286581993103031 0.11283322958339835 0.0095533965833058416 2.5244140625000004 1.1267314367267358 0.95146732080301388 -0.080203155274325963 1.5661550609387962 -1.0000000180025095e-35 0.068176441062887241 1.3771184689138856 1.5185086528259144 2.6015625000000004 1.9634439907166827 -22.378742218017575 0.37248792739977959 1.0000000180025095e-35 0.39236630927281513 -25.182852745056149 -24.354135513305661 1.2426757812500002 47.592866897583015 1.0000000180025095e-35 0.42638606281916264 0.89599609375000011 0.89587973461402759\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 -2 3 -3 6 7 -1 -6 9 10 -8 20 13 14 -13 -11 -17 28 19 -15 -10 -22 -12 -21 25 -7 27 -26 -14 -23\nright_child=1 2 -4 -5 5 24 8 -9 11 15 22 12 17 18 -16 16 -18 -19 -20 23 21 29 -24 -25 26 -27 -28 -29 -30 -31\nleaf_value=-0.01836535731451025 0.0035980925374370669 -0.035487243995886285 -0.0005906091579115228 -0.0038908480645847986 -0.0021110945218486884 0.017266985789256024 -0.017679191035559438 0.033001317363910827 0.0070466254208476908 -0.027904475003537054 -0.0096337053141852431 -0.032722289856491817 0.03743537982745921 -0.013423876926472879 0.0040395526016886339 -0.015023370163149682 0.012253855323209618 0.00044577776008189126 -0.002313042135587104 0.0047443551883024375 -0.0010616420952308664 -0.031496033784209322 0.018819208793739838 0.021539094326827001 -0.014832840629274736 -0.0047470200476049194 0.012851404377013214 0.0013218093764110598 -0.0013247467047286881 0.0044548655304784482\nleaf_weight=4.2278198039469705 12.189285773603844 6.5561629041270084 8.0282927926580196 3.6145279853957391 1.7521619375471562 9.4226577671755063 1.1607051554157717 3.7667781581822055 1.9230340804667765 2.2044877826456837 1.104548605766756 3.9298191454795406 2.5810897545497933 2.6854276040978666 2.9751827474974561 3.0710413064890076 4.8814474849739895 3.5512136607485445 86.031973161198721 28.734535562775363 3.3761567114843256 4.9612159086437941 15.357429072651314 4.3532219851367691 6.5874593949460012 3.9394344060611948 5.2452122905506249 14.279362833102821 0.86893441718029851 0.67184553671177116\nleaf_count=6895 2526 3486 3023 1651 671 5469 15283 106 3157 26750 28073 628 136 2626 40 66363 13933 281 6894 2151 1920 5111 33282 122 7616 6763 488 631 80 82\ninternal_value=-0.000171007 -0.0077729 -0.0151797 -0.0256118 0.000891924 0.00643429 -0.000531961 0.02324 2.52798e-06 0.00593648 0.0133901 -0.00115417 -4.42625e-05 -0.00107674 -0.0183753 -0.00703684 0.00121862 0.016699 4.5547e-05 0.00562644 -0.013774 -0.0184782 0.0164677 0.00752102 0.00356852 0.0108982 -0.000445832 -0.00422316 0.0295692 -0.027183\ninternal_weight=254.032 30.3883 18.199 10.1707 223.644 44.9931 178.651 5.51894 174.423 27.7797 17.6227 146.644 135.711 128.71 6.905 10.157 7.95249 7.00124 121.805 35.7732 10.9323 9.00922 16.462 33.0878 39.4741 13.3621 26.112 20.8668 3.45002 5.63306\ninternal_count=246237 10686 8160 5137 235551 21744 213807 777 206912 183684 76638 23228 12958 12461 668 107046 80296 497 11793 4899 10270 7113 61355 2273 20967 12232 8735 8247 216 5193\nis_linear=0\nshrinkage=0.03\n\n\nTree=287\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 14 7 4 14 5 18 1 1 5 4 14 13 5 4 4 19 7 14 18 14 19 19 18 6 1 5 5 19\nsplit_gain=2.25755 2.99406 2.9059 3.59605 1.39845 1.22451 2.63141 2.81064 1.89221 1.3425 1.31603 2.01045 1.07694 1.04163 1.02669 1.84039 1.84081 1.88869 1.76157 1.5322 1.3663 1.33662 1.87881 1.31591 1.22819 1.21735 1.02602 1.00588 1.71607 1.00292\nthreshold=0.083740223399813532 1.5000000000000002 0.82879072691359035 2.1386718750000004 2.1186523437500004 0.72685412594425991 1.2426757812500002 1.8688348091416842 0.44640776988282155 2.9477339485081675 4.7402343750000009 12.005371093750002 0.85380319876708544 1.0000000180025095e-35 5.2558593750000009 5.6752929687500009 6.0415039062500009 235.50000000000003 1.4980468750000002 0.30268741420124606 4.516324041782946 0.032916404154818342 231.50000000000003 231.50000000000003 1.4978661367769956 4.2421875000000009 4.0016611463566001 3.1982421875000004 2.2500000000000004 257.50000000000006\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 4 3 -3 13 14 7 9 -8 29 -10 -12 -4 -2 15 18 -17 23 21 24 -19 -1 -23 -18 -20 -16 -9 28 -21 -7\nright_child=1 2 12 -5 -6 6 8 26 10 -11 11 -13 -14 -15 25 16 17 20 19 27 -22 22 -24 -25 -26 -27 -28 -29 -30 -31\nleaf_value=-0.011376809018660006 -0.011595331538790882 -0.034911106477082489 0.016882553959456356 0.0019631542449661827 -0.003969334760248246 -0.022567716621828885 -0.014844365528812251 -0.025642351775165415 0.013117361754466992 0.015342978361780326 -0.02574828647931296 0.013554145112895929 -0.0062368585113590017 0.021160360300341907 -0.023421640836730536 0.023866387145374139 0.0019924818977704577 0.0035250550166429574 0.012947145713697436 -0.0023899473551117984 -0.020468233723647936 0.017911627469323902 0.00047156665364335996 0.021964005013424617 -0.0081664074758861678 0.0064903935224365243 0.0031793084436472011 -0.0027216166958196901 -0.03578200917642984 0.0035042917271412609\nleaf_weight=6.9629856640028711 0.40072130525789074 7.1253697302945529 1.5233376472692146 3.2218967553660605 6.577297013831469 1.0574839429116289 1.8176275926141148 4.4104398762723624 31.502062333890251 4.5124583189608529 1.2799249155959835 1.5038224314666875 5.4464967629841112 3.7937090212653857 2.4119707115596607 4.3403811833495665 7.1186420059699476 4.0614534484104743 2.9988228963629826 2.2894856491204774 2.9972750161479667 4.8424430514762653 116.19946027556131 4.3853544656578833 5.5271563508230201 1.3572763118989994 1.1427965619659515 3.1304598141277893 3.2165841793045615 3.1400329008065455\nleaf_count=10590 506 3316 394 838 1145 793 39638 34520 11565 57 316 100 1874 162 659 3162 1339 275 44 1578 19084 7294 91093 401 6288 193 71 2849 5975 173\ninternal_value=0.000373064 -0.00748964 -0.0149032 -0.0242682 0.00468223 0.00139608 0.0054603 -0.00519361 0.00969582 0.00542254 0.0113806 -0.00654562 0.000300445 0.0164012 0.000180148 0.000566338 0.00828096 0.00360302 -0.000699127 -0.00944314 -0.00793863 0.000537357 0.00131033 0.0107534 -0.000160621 -0.0135481 -0.0201311 -0.0176392 -0.0243987 -0.00614232\ninternal_weight=250.295 28.0888 17.3171 10.3473 10.7717 222.206 50.3666 14.2632 36.1034 8.70998 34.2858 2.78375 6.96983 4.19443 171.84 168.071 22.9031 18.5627 145.167 17.1625 7.05873 128.005 121.042 11.504 8.52598 3.76925 5.55324 8.63653 5.50607 4.19752\ninternal_count=246292 8235 6422 4154 1813 238057 87233 35614 51619 1023 11981 416 2268 668 150824 149972 24261 21099 125711 16734 19359 108977 98387 1740 6332 852 34591 10402 7553 966\nis_linear=0\nshrinkage=0.03\n\n\nTree=288\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 14 5 3 3 0 11 0 2 2 18 1 18 18 0 1 1 3 21 18 1 1 1 18 18 18 2 3 2\nsplit_gain=2.47321 3.15837 2.06155 1.44913 1.12388 3.16317 1.87894 1.07323 1.40182 0.987099 2.69079 1.73646 1.30953 0.928596 1.10765 2.08438 1.98633 2.58655 2.67836 1.92321 2.04371 2.7719 1.75948 1.94614 1.74994 2.4005 1.68211 2.19163 1.5419 1.47624\nthreshold=0.24649637830053225 1.5000000000000002 0.82879072691359035 1.4980468750000002 0.65908129556435835 0.62297257604742995 1.8987378677022575 0.60365083802632757 1.4460303622695712 -1.3165187835693357 -1.7340650558471677 1.4978661367769956 4.0016611463566001 7.4818373127308169 5.0271592211193568 0.31303196667935901 1.075454090076706 1.5661550609387962 0.37963534769016083 1.5000000000000002 3.1132683346437333 4.0016611463566001 1.6109634896136267 1.7622051334857016 3.7012257604091228 4.8242976514536702 2.1383330595080281 -4.708568572998046 0.22136347059745312 -24.354135513305661\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 3 -3 -2 5 13 9 -5 -9 10 11 -7 -13 14 15 -1 24 18 -18 20 -19 -22 -21 -24 26 -26 -17 -28 -25 -16\nright_child=1 2 -4 7 -6 6 -8 8 -10 -11 -12 12 -14 -15 29 16 17 19 -20 22 21 -23 23 28 25 -27 27 -29 -30 -31\nleaf_value=-0.019033265680949131 0.016757466867417942 -0.024524493893751706 -0.0033291939740388355 -0.01906377216563248 0.007828109819571466 0.0046550119637758947 0.011845780648513654 0.014276994573319922 -0.017256702552986614 -0.027126928719102644 0.021194967398352405 -0.032923944949208313 0.0047101445319699111 -0.018570412995845418 -0.013343815389121246 -0.00011594605255021954 0.00013568503844851019 -0.012791912685683861 -0.026900164469185195 0.023208452610704838 0.011496223507123593 -0.020701856694075985 -0.014307173960998808 -0.017161515853444814 -0.019050633387372534 0.017246952883731043 -0.01294824947700844 0.020998418917855081 0.003946744130471364 0.011595932082097932\nleaf_weight=3.5968289693984721 4.0429844423365484 9.4617993550291448 7.4166544690712035 1.8011512379748635 19.363894110802903 2.0100397225469351 1.6485488103062387 2.8590362201284738 0.92883505986537773 4.56503612694996 1.274126383854308 3.2718372675881255 0.74999969068448979 1.0702895417254001 1.6019778865935759 6.8931112033076323 6.9604565026137362 14.942409391153889 5.2502536067315759 2.7794143201294892 8.7174457128478462 2.2208019962417884 5.0143106450295809 2.2438465074328624 3.4086656947244185 1.6754385839388417 1.2606197262845196 13.94926646512601 95.791281863167114 15.685885333258879\nleaf_count=29878 407 2617 1492 446 863 19 29 100 38 546 17 249 19 35003 32266 4110 6801 1026 2416 403 2052 67 1310 787 60756 5120 11305 13180 5043 27430\ninternal_value=7.38798e-05 -0.0084213 -0.0159181 0.00497716 0.00110334 0.000438325 -0.0130706 -0.0047944 0.00455528 -0.0171823 -0.00845066 -0.0168364 -0.0263654 0.00144679 0.00165403 0.000913158 0.0014406 1.42035e-05 -0.0126453 0.0012743 -0.00591119 0.00377253 0.0030704 0.00230941 0.00872314 -0.00622026 0.0122809 0.0175596 0.00329521 0.00868161\ninternal_weight=252.456 26.5105 16.8785 9.63201 225.946 206.582 13.5196 5.58902 3.78787 11.871 7.306 6.03188 4.02184 193.062 191.992 174.704 171.107 143.92 12.2107 131.71 25.8807 10.9382 105.829 103.049 27.1871 5.0841 22.103 15.2099 98.0351 17.2879\ninternal_count=245795 5100 4109 991 240695 239832 879 584 138 850 304 287 268 238953 203950 144254 114376 19905 9217 10688 3145 2119 7543 7140 94471 65876 28595 24485 5830 59696\nis_linear=0\nshrinkage=0.03\n\n\nTree=289\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 0 18 21 0 6 17 18 18 1 4 2 17 17 17 0 18 17 1 18 14 7 4 9 19 18 2 17 17 18\nsplit_gain=1.16481 1.77621 1.49873 3.58808 2.78456 2.39199 2.35311 1.67974 2.74776 2.06336 1.83668 2.08417 2.91575 2.36827 1.90234 2.53501 2.36926 1.70188 1.65295 1.57509 1.48681 1.68843 1.4788 1.43819 1.38915 1.699 1.37587 2.75558 1.68862 1.49538\nthreshold=3.3033179733375269 1.2661233611507157 2.0126758453675748 7.5000000000000009 1.3771184689138856 2.1386718750000004 0.068176441062887241 2.8029010331479989 2.3502401828962087 2.1468275719512682 7.3568274974823007 -5.9238033294677725 0.61464819824310435 0.84061941967369014 0.92594565335601631 0.64273501842566294 6.2567088148180039 1.5700365906568221 2.2520445226790948 3.2384861814448418 0.49906193250331299 1.2426757812500002 1.6572265625000002 1.0000000180025095e-35 241.50000000000003 2.1383330595080281 31.708583831787113 1.5538318668855886 0.25672585397575204 1.2424533248940002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 24 3 26 6 -6 -4 8 9 -8 11 12 23 -14 -13 16 -16 -10 -5 -17 21 -15 -11 -9 -2 -26 27 28 29 -1\nright_child=1 -3 4 18 5 -7 7 10 17 22 -12 14 13 20 15 19 -18 -19 -20 -21 -22 -23 -24 -25 25 -27 -28 -29 -30 -31\nleaf_value=0.00018759698997007399 -0.014389568463107018 0.0080992647903676222 0.015501354183705811 -0.0027304638763309521 0.019348102634178986 -0.020885587000547472 -0.013295678202256229 -0.026123865417362505 -0.02601879430984675 -0.0069815266804563432 -0.0095543445144587527 0.012578461383962403 0.01408111318860799 0.010997104012097355 -0.019140331774578627 -0.0087994478959147159 0.016111270420482285 0.0019794701459518028 -0.036125470823674788 0.014346998040167199 -0.021082516753286439 -0.019632708592060004 0.020514003342829951 0.0089107927114853176 -0.014727645074109069 0.0046191748901644007 0.0052308363303969563 -0.027019458532248696 0.001902578188558234 -0.025122684225567019\nleaf_weight=3.8714275162201366 4.8186451873807519 43.50992020463309 7.3007491532887903 2.1967045610817868 10.448905983141682 0.67964884685352522 3.8003045616021627 5.0593531920302963 7.1418793838022321 2.0196301008545561 11.688105852738316 22.349759460945911 7.2894237054 4.3848137821372557 6.7852240444043179 2.8322510893195831 1.378341882021118 2.3941860839092897 3.2620824148179963 9.0333126260279624 3.260054495566493 1.3242843041127512 4.3381673449184746 0.76196254255108886 3.8537464448018026 25.080436043674126 17.524782426825368 3.4238838340643261 26.334254164098638 3.5558342135409475\nleaf_count=179 283 583 8845 554 1283 21 2125 41105 3356 93 10713 23115 12122 23765 34468 2894 1051 1870 69 6738 47224 17438 60 522 93 978 1509 1025 1831 451\ninternal_value=0.00112144 0.00416233 -0.000241477 -0.00403578 0.00177409 0.0153715 0.000173473 -0.00114215 -0.00872328 0.00220488 0.000906243 0.00292001 -0.0044236 0.00234649 0.00675989 -2.22282e-05 -0.0120798 -0.0194687 -0.0251902 0.00856754 -0.00764632 0.00202492 0.0120179 -0.0209041 -0.000999687 0.00163645 -0.00151079 -0.00474177 -0.00177022 -0.0134731\ninternal_weight=251.702 77.2627 174.439 60.169 114.27 11.1286 103.142 95.8411 19.6942 10.1581 76.1469 64.4588 22.0799 16.2586 42.3789 20.0291 8.16357 9.53607 5.45879 11.8656 8.96915 5.7091 6.3578 5.82132 33.7528 28.9342 54.7102 37.1854 33.7615 7.42726\ninternal_count=246363 1937 244426 5618 238808 1304 237504 228659 7504 2278 221155 210442 142176 100549 68266 45151 35519 5226 623 9632 88427 41203 153 41627 1354 1071 4995 3486 2461 630\nis_linear=0\nshrinkage=0.03\n\n\nTree=290\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 14 6 4 17 1 1 3 18 18 21 1 7 17 19 1 0 0 3 0 14 18 19 5 4 5 18 18 1\nsplit_gain=2.21217 2.18105 1.89122 1.81388 1.39495 1.30693 1.91908 1.60501 1.72361 1.52839 3.69609 2.38082 1.88228 1.33815 1.25024 0.96715 0.918801 0.883061 3.48001 1.55829 1.70173 2.12328 1.65601 1.5115 3.99382 1.44366 1.75417 1.45975 1.37805 1.39468\nthreshold=8.8817841970012543e-16 1.5000000000000002 0.82879072691359035 4.7578125000000009 6.2924274206161508 0.19767305700411134 1.6109634896136267 0.60367420583093068 0.28287969384316608 1.2424533248940002 2.1383330595080281 1.5000000000000002 1.9634439907166827 1.2426757812500002 0.12494407617242172 249.50000000000003 1.5275240303550033 1.1495945197163422 0.97942481616843191 0.37248792739977959 0.53713645201195115 0.28824823456061044 2.8617925509761908 246.50000000000003 2.1386718750000004 3.2895507812500004 5.0781250000000009 3.1132683346437333 4.5271940351011493 1.9634439907166827\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=5 4 3 -3 15 6 7 -1 -9 -8 -11 -12 -13 14 -14 -2 -10 18 19 20 21 -7 -22 25 28 27 -27 -21 29 -25\nright_child=1 2 -4 -5 -6 17 9 8 16 10 11 12 13 -15 -16 -17 -18 -19 -20 23 22 -23 -24 24 -26 26 -28 -29 -30 -31\nleaf_value=-0.011698326755102585 0.016408157521521832 -0.025620562318563998 -0.0017617562813188662 0.0077776883253963553 -0.012319185435488747 0.0011148912508119465 0.0032773514734025376 0.025506150357736911 -0.010572566830098258 -0.028238371621392484 -0.022153651217034737 -0.021423645027070972 0.021461637697255655 -0.011127988255491597 -0.0024568137205093995 -0.0036132078235893059 0.013145991893492604 0.0047214882546375101 -0.01103039192296955 0.01882732685075594 -0.016508323433573266 -0.022220445408441983 0.0096544673531891983 0.020547785089514905 -0.023486738323518098 0.020702621152673538 -0.015249052041683189 -0.0083717515217028911 -0.010975854983671086 0.0038322475001810832\nleaf_weight=2.2055404390497966 5.1334156829632338 8.8664839312594967 7.9499276635737024 1.249765295225983 2.7237881175993826 7.4514793640840953 8.3922231464239321 3.871993595914887 2.520460648050971 6.6338406897411941 2.9091769854080658 1.1228035698027259 5.509237556560036 1.5195884916829516 2.5474548849888379 2.8851571499953934 1.5263247640714324 93.861485289217399 17.59488265710479 1.6341065595497664 1.7104082758681887 5.4847357602258739 11.104219212202507 6.0221714204135521 3.6717409931575267 12.779910624590681 0.63019561145847625 4.45397143171067 2.5664539314439017 14.204541959055858\nleaf_count=22231 842 5083 2660 218 929 34760 121 3306 1940 218 613 581 523 108 241 691 173 7554 5594 157 2218 109757 33025 2046 1061 2172 100 3753 2283 1094\ninternal_value=0.000622771 -0.00705598 -0.0131675 -0.021166 0.00346831 0.00164673 -0.003299 0.00743929 0.0134835 -0.00721868 -0.0115197 -0.00199443 0.00491212 0.00973362 0.0144626 0.00960243 -0.000794419 0.00270547 0.000557857 0.00351361 -0.00233846 -0.0096634 0.00522023 0.00677221 0.00195481 0.0128965 0.0180721 0.000555001 0.00686805 0.00954258\ninternal_weight=250.737 28.8085 18.0662 10.1162 10.7424 221.929 38.7586 10.1243 7.91878 28.6343 20.2421 13.6083 10.6991 9.57628 8.05669 8.01857 4.04679 183.17 89.3088 71.7139 25.7508 12.9362 12.8146 45.9631 26.4649 19.4982 13.4101 6.08808 22.7932 20.2267\ninternal_count=246052 10423 7961 5301 2462 235629 30055 27650 5419 2405 2284 2066 1453 872 764 1533 2113 205574 198020 192426 179760 144517 35243 12666 6484 6182 2272 3910 5423 3140\nis_linear=0\nshrinkage=0.03\n\n\nTree=291\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 11 14 21 5 2 5 9 7 4 4 0 1 1 19 17 2 21 1 13 4 17 5 14 1 17 13 7 2 14\nsplit_gain=2.51152 1.49697 1.60856 2.15327 2.21621 1.44401 1.34508 1.10577 2.82781 1.55458 2.95093 2.38176 2.10457 2.35436 1.66981 2.13696 1.93085 1.83109 1.91263 1.62475 2.10081 1.65203 1.55616 1.4998 1.98715 1.44664 1.43498 1.98421 2.78578 1.27407\nthreshold=-0.90051303354301027 0.11347059467310934 0.81021705844416803 1.5000000000000002 2.7412109375000004 -39.283819198608391 2.8623046875000004 1.0000000180025095e-35 2.3505859375000004 2.2153320312500004 5.7263183593750009 1.8987378677022575 1.6109634896136267 1.8912989126705502 263.50000000000006 0.72612140026470018 4.3805527687072763 2.5000000000000004 2.2520445226790948 -0.48625410865680269 0.84667968750000011 1.971174727056735 4.9804687500000009 0.6486830462841916 3.3033179733375269 0.21173336644813048 -0.4958236943477588 1.2426757812500002 -21.560756683349606 0.8507468585983845\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=-1 5 3 -3 -5 -2 -6 9 -9 19 11 12 -11 25 16 -16 26 29 -19 -7 -21 23 -12 -22 -25 -14 -15 -28 -29 -18\nright_child=1 2 -4 4 6 7 -8 8 -10 10 22 -13 13 14 15 -17 17 18 -20 20 21 -23 -24 24 -26 -27 27 28 -30 -31\nleaf_value=-0.027556500199071961 -0.0082728071484879425 0.00090693710002189812 0.0026557924498569608 -0.032640455157811735 0.017015804513122217 0.0091866674824167115 -0.012933683902238256 0.025699555669079357 -4.6698135604910594e-05 -0.021189323106099205 0.010929934704136642 0.0095456266965004803 -0.017841751790500859 -0.031098863888186525 -0.010161856370339502 0.021240200997520109 -0.020787341330073206 -0.021538441091360081 0.0092087262865858666 -0.018547249830305874 0.0051729519206966286 -0.014594518703561953 -0.0095365416538408711 -0.025384289951801906 0.0063660182033162 0.013378178481061655 0.00095448816239980907 0.0070534245256750963 -0.036486288484723767 0.0074935364546751329\nleaf_weight=1.8111726571945599 11.200696073209711 7.0678467297310617 12.068403366392998 4.7404837688726484 1.1210380002099216 36.044852194789087 2.7571135082707769 5.2557824217082603 9.9305809392536712 7.0944456982133763 19.390302630226248 8.2538174443652661 0.68805125063590633 4.2581488605701407 2.9870605538599184 2.9536898059759551 5.0423450446414817 1.2148632945218185 9.838689579223681 3.4525137403809643 62.823104506871744 3.7020275082560081 3.1978373440517762 2.5470243026203709 2.9163095169351436 7.719531874256063 6.4226285532813581 1.5398755306678138 3.0834905597339457 1.3121462147955751\nleaf_count=35084 61069 995 2861 2876 30 20106 625 898 1293 36528 6072 230 195 297 51 36 316 174 307 25189 19740 18006 555 10988 123 825 458 172 267 134\ninternal_value=0.000647609 0.000960134 -0.00548951 -0.0115397 -0.0207799 0.00178427 -0.00256289 0.00235285 0.00990095 0.00172226 -0.00132612 -0.00468191 -0.00698409 -0.0044463 -0.00757133 0.00626175 -0.0101947 -0.00305151 0.00432286 0.00403748 0.00148926 0.00269115 0.00775101 0.00382537 -0.0100733 0.00919829 -0.0176344 -0.0102932 -0.0233093 -0.0147227\ninternal_weight=252.436 250.625 27.7549 15.6865 8.61864 222.87 3.87815 211.669 15.1864 196.483 84.9969 62.4088 54.155 47.0605 38.6529 5.94075 32.7122 17.408 11.0536 111.486 75.441 71.9885 22.5881 68.2864 5.46333 8.40758 15.3041 11.046 4.62337 6.35449\ninternal_count=246500 211416 7387 4526 3531 204029 655 142960 2191 140769 46617 39990 39760 3232 2212 87 2125 931 481 94152 74046 48857 6627 30851 11111 1020 1194 897 439 450\nis_linear=0\nshrinkage=0.03\n\n\nTree=292\nnum_leaves=31\nnum_cat=0\nsplit_feature=1 2 18 18 18 18 1 2 22 3 5 19 18 2 4 5 3 14 5 7 9 2 2 18 18 4 18 18 14 5\nsplit_gain=1.77892 1.68103 1.66141 1.65757 4.25047 1.89945 1.62802 1.56696 1.84094 1.72386 1.55627 1.83686 1.481 1.30481 1.5339 1.42625 1.31875 1.17573 1.37881 1.20334 1.17277 1.61003 1.14993 1.32084 1.10836 3.24054 1.55774 1.23197 2.20915 1.19699\nthreshold=2.5144622974930591 -33.804542541503899 5.0271592211193568 4.2696729980286863 4.2121001622835372 3.9981586158983733 -0.041910702480567281 -21.560756683349606 1.0000000180025095e-35 0.37963534769016083 4.3378906250000009 264.50000000000006 3.0201273556387074 -11.719409942626951 2.1433105468750004 3.8173828125000004 0.29586597258655462 0.50603332636230081 4.7402343750000009 2.6738281250000004 1.0000000180025095e-35 -8.3227787017822248 -17.275529861450192 0.89587973461402759 3.3842466058243152 1.4716796875000002 3.6762205501217919 3.0201273556387074 0.79704415596844569 1.4980468750000002\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 -1 3 4 5 -3 -4 8 9 -2 11 -5 -10 14 22 -16 20 19 -19 -8 21 -15 -9 -24 27 26 -26 28 29 -18\nright_child=7 2 6 10 -6 -7 17 13 12 -11 -12 -13 -14 16 15 -17 24 18 -20 -21 -22 -23 23 -25 25 -27 -28 -29 -30 -31\nleaf_value=-0.018341224305553704 -0.0081788258369399104 -0.0025187623740469071 -0.014830151289189422 -0.027341107241885647 0.026665268618148216 -0.024467617081448463 -0.014118727316127278 0.0083685689998551707 0.0083236016160636167 0.028418348614138115 0.0036079493302843056 0.0085752262560450827 -0.0084146737360386147 0.010702743130623065 -0.027754481719102317 0.0095221837607873689 0.000672910595652937 0.019924817783241951 -0.0085995190668543987 0.011412031593763687 0.013049110661065521 -0.021740884252810632 0.015315415683011939 -0.018089596189781999 -0.01725396014562048 -0.024135922171466533 0.016713043336716044 0.017135969476744221 -0.0097413269726856974 0.010046295708029602\nleaf_weight=4.9493051829597325 0.96850933961377705 84.276054507999731 1.8367970803121547 5.859686713266397 3.4269124524902841 2.7296100185417336 2.4014894140900074 5.0073483301512161 17.176361015852311 4.8903165439432996 2.7684594623460725 1.0801166694652691 5.5479095383711856 1.1417695256541263 3.4346439823648316 0.63431002257857461 23.295858394820243 8.387445470885039 1.2354628347152345 2.2660651081987453 0.88134731631726015 4.7100597254116101 0.5728566672187293 3.3901729571298347 0.7196877599671988 3.8152039805568165 4.0226215979828339 5.335579425947798 9.1575942430063133 25.251311185755185\nleaf_count=94478 275 43539 22127 50424 2962 9856 5223 143 283 337 418 1629 755 38 170 14 465 9599 460 439 31 584 5 223 77 312 248 194 353 334\ninternal_value=-0.00021574 -0.00276751 -0.00195382 -0.00340236 -0.00205616 -0.00347756 0.00671426 0.00236448 0.00836583 0.0220581 -0.0145792 -0.0213751 0.00405468 0.000416855 -0.00801845 -0.0212113 0.00190441 0.0102722 0.0157979 -0.00189716 -0.0099143 -0.0147725 -0.000506961 -0.0111479 0.00313711 -0.00648155 0.00945123 0.00452364 0.00308542 0.00565274\ninternal_weight=241.171 121.217 116.268 100.141 90.4326 87.0057 16.1273 119.953 28.5831 5.85883 9.70826 6.9398 22.7243 91.3704 13.0393 4.06895 78.331 14.2905 9.62291 4.66755 6.73318 5.85183 8.97038 3.96303 71.5979 8.55751 4.74231 63.0403 57.7048 48.5472\ninternal_count=245995 241154 146676 108828 56357 53395 37848 4841 1650 612 52471 52053 1038 3191 555 184 2636 15721 10059 5662 653 622 371 228 1983 637 325 1346 1152 799\nis_linear=0\nshrinkage=0.03\n\n\nTree=293\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 5 7 4 2 2 16 1 21 13 3 5 4 2 3 7 7 7 4 21 5 5 2 1 13 14 2 3 2 21\nsplit_gain=1.60978 1.87055 3.07202 1.45338 1.33139 1.43281 1.90271 1.98505 1.20235 1.25607 1.07194 1.12294 3.04426 2.10239 1.81663 4.05627 2.58019 2.18256 3.41871 2.07193 1.87939 2.03896 2.01309 1.87155 1.83161 1.6413 1.58765 2.42951 1.6899 1.78064\nthreshold=0.030057938850762859 1.4980468750000002 1.2426757812500002 3.9857867956161503 28.11904335021973 34.417465209960945 1.0000000180025095e-35 1.9634439907166827 3.5000000000000004 1.0000000180025095e-35 0.65908129556435835 1.2426757812500002 1.1459960937500002 8.8705892562866229 0.31840721315068282 2.3505859375000004 0.89599609375000011 1.4980468750000002 3.2580566406250004 6.5000000000000009 3.1982421875000004 2.2500000000000004 -23.956354141235348 1.6575619461558879 -0.39543742724656211 0.72273155396328503 -10.988907814025877 0.29242024582541543 -26.006598472595211 4.5000000000000009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 8 -3 -4 10 -6 7 -7 9 -2 11 12 25 -14 15 16 23 18 19 -16 21 -19 -23 -13 -20 26 28 -28 -1 -30\nright_child=1 2 3 -5 5 6 -8 -9 -10 -11 -12 14 13 -15 17 -17 -18 20 24 -21 -22 22 -24 -25 -26 -27 27 -29 29 -31\nleaf_value=-0.0014608705911257399 -0.014763951362604147 -0.031444445028497324 0.013226094265185664 -0.0085749262150073852 0.013861557120178454 -0.012477842200755242 0.01472717043011337 0.0045345213713376444 -0.009647351806502464 0.015668239191753395 0.0074029674211055349 -0.017190465552305047 -0.020084862635909848 0.0054872162617804771 0.00016053265231769605 0.0060030793174603468 -0.028424200998059031 0.0047594989548197233 0.0037801856948743739 0.014553659040293631 0.0020241240707214374 0.0073657526037890012 -0.034402648973889415 0.014729123134118317 0.034100099156794114 -0.010648900848130294 0.013322456418614737 -0.0093854751782938527 0.022652499255902209 -0.0091315674913684721\nleaf_weight=6.8336886772407288 0.65934962698071498 4.1534252769715208 2.5630333812970312 11.747920513116638 13.150065266476473 8.8269102061600559 7.2897734675598249 15.682495093676151 2.890389931812166 5.8850161398204364 15.720162539915689 2.0949743779843022 12.348047018118164 3.2237853088637101 40.201692531969684 12.29730163159101 6.8311236449293631 2.4921097110177497 3.0397082380266056 10.606749698533575 13.362750761122923 0.93571369530150139 3.7163111575659968 2.548989058773941 4.6165677110554943 8.6191435101368636 5.0793325116830266 13.16620279867959 7.9459938197144293 1.356592996586073\nleaf_count=15558 1418 2032 834 3433 4676 16326 731 920 1562 281 665 31353 21623 4214 6612 34105 50544 441 622 710 2430 388 1150 854 166 20107 12463 2545 5630 1267\ninternal_value=0.000491641 -0.0061728 -0.0115202 -0.00406163 0.00135315 0.00590337 0.0022901 -0.00184134 0.00439354 0.0110516 0.000171192 -0.000574803 -0.0038546 -0.0147791 0.00131409 -0.00580078 -0.0179176 0.0035016 0.00642955 0.00338815 -0.00475675 -0.0161312 -0.0261845 -0.000164876 0.0238898 0.000347436 0.00332716 -0.0027001 0.00990214 0.017581\ninternal_weight=249.885 27.8991 18.4644 14.311 221.986 44.9492 31.7992 24.5094 9.43476 6.54437 177.037 161.317 58.5728 15.5718 102.744 23.7724 11.4751 78.9716 58.4647 50.8084 20.5069 7.14413 4.65202 4.64396 7.65628 43.001 34.3818 18.2455 16.1363 9.30259\ninternal_count=245660 9560 6299 4267 236100 22653 17977 17246 3261 1699 213447 212782 83407 25837 129375 116856 82751 12519 8110 7322 4409 1979 1538 32207 788 57570 37463 15008 22455 6897\nis_linear=0\nshrinkage=0.03\n\n\nTree=294\nnum_leaves=31\nnum_cat=0\nsplit_feature=17 8 12 21 4 7 13 4 11 14 3 4 9 21 14 14 14 5 5 14 13 5 6 4 13 7 2 2 17 2\nsplit_gain=2.01529 2.02093 1.51635 2.78302 2.10093 2.77678 2.25339 2.134 2.03527 1.98898 1.96226 1.71829 1.60972 1.23348 1.97388 1.33109 1.31429 1.20831 2.56412 1.45118 1.63906 1.25692 1.92305 1.98896 1.19221 1.18236 1.16773 2.38715 2.3729 1.95706\nthreshold=3.2718994203396332 1.0000000180025095e-35 5.5511151231257837e-16 1.5000000000000002 3.2580566406250004 1.2426757812500002 -0.39543742724656211 2.3505859375000004 0.60365083802632757 0.11320561501600769 0.23120698015535426 1.0693359375000002 1.0000000180025095e-35 3.5000000000000004 0.85195221821350431 0.59283710889839769 0.32282440459636236 2.1386718750000004 3.4169921875000004 0.016582161273737187 -0.74650584208841086 4.4355468750000009 2.0131835937500004 6.6144968271255502 -1.0000000180025095e-35 2.5244140625000004 -1.7340650558471677 -17.667204856872555 0.09506205666563787 5.8049912452697763\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=2 -2 4 -4 7 6 -6 10 -5 -8 11 24 26 14 16 -15 -9 -7 19 -19 -21 -20 23 -23 -1 -22 27 28 -12 -28\nright_child=1 -3 3 8 5 17 9 13 -10 -11 12 -13 -14 15 -16 -17 -18 18 21 20 25 22 -24 -25 -26 -27 29 -29 -30 -31\nleaf_value=-0.009926665513680967 -0.027049204133447174 0.0019006474842976895 0.0057215340601532187 -0.028015763948231246 -0.0067594532299438221 0.017524850298613719 -0.0090106530278225953 -0.0038788181369020219 -0.0056064863090544642 0.036300475751717652 0.020964847187238497 -0.022218102907038163 -0.023362052347773096 0.0046099182511166303 0.0092242238278434797 -0.013145963453272503 -0.027765630902520298 0.0078456508332808844 0.011852897707320557 0.011381541121405346 -0.021867473461493477 0.0059537150386882887 0.010564371786395684 -0.0229316896792803 0.014504335635080631 0.0032612325685228436 0.013660516349346312 -0.0083487275755307622 0.00084468607151912361 0.0019078314332710818\nleaf_weight=2.9786881264644762 3.8151607803015102 3.7273712331294178 10.737504862984112 4.7650488228111412 1.4631398238339555 4.1222975782443418 0.6160644885351293 3.4605323742724199 11.843557718074861 4.8725670790132938 5.4447538812401026 4.7111989033812733 1.1415901337713865 9.5382582699764935 1.9158683636454714 4.9697150058291291 6.0361322883356934 3.2465146454358083 17.799644808398714 1.426784445217774 7.8719759306202661 3.3098139071140036 6.8070235640811916 3.7479395171999954 2.2876552933480507 1.7872502903863958 18.524313056951314 19.538779992692184 29.594794227826526 37.92557956704502\nleaf_count=26978 65826 4084 1901 5347 1695 945 219 3149 772 1299 967 38217 51 4302 3357 4724 7744 676 13652 1543 15605 917 927 2080 3885 3040 3594 6739 10769 11134\ninternal_value=0.000117318 -0.014195 0.000640461 -0.00586899 0.00153247 0.00636876 0.0228837 -0.000361844 -0.0132617 0.0306097 0.00131327 -0.0108132 0.00247793 -0.00801117 -0.0149661 -0.00192786 -0.0202602 0.00367522 0.00208752 -0.00808612 -0.0130148 0.00679907 -4.99552e-05 -0.0103278 0.00130701 -0.0173481 0.00294979 -0.000189218 0.0044661 0.00593518\ninternal_weight=240.028 7.54253 232.485 27.3461 205.139 57.071 6.95177 148.068 16.6086 5.48863 122.147 9.97754 112.17 25.9205 11.4125 14.508 9.49666 50.1192 45.9969 14.3325 11.086 31.6644 13.8648 7.05775 5.26634 9.65923 111.028 54.5783 35.0395 56.4499\ninternal_count=246138 69910 176228 8020 168208 42598 3213 125610 6119 1518 102334 69080 33254 23276 14250 9026 10893 39385 38440 20864 20188 17576 3924 2997 30863 18645 33203 18475 11736 14728\nis_linear=0\nshrinkage=0.03\n\n\nTree=295\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 2 6 21 18 3 2 21 18 9 18 3 4 3 7 19 2 1 1 8 3 19 17 17 12 21 18 18 1 18\nsplit_gain=1.58445 2.08616 2.03958 2.23208 2.69004 2.03904 2.50854 1.72315 1.93275 1.51608 1.37999 1.29916 3.10892 1.91788 1.82626 1.74912 1.31437 1.37978 1.99405 2.4044 2.61131 1.28982 1.24165 1.1226 1.06112 1.39581 1.32213 1.1856 1.24295 3.11861\nthreshold=-2.5585746765136714 -17.667204856872555 1.8691406250000002 1.5000000000000002 0.89587973461402759 0.53650715206675215 -24.354135513305661 6.5000000000000009 6.5213569628644246 1.0000000180025095e-35 1.8688348091416842 0.27986503791205852 2.0893554687500004 0.23756184665784369 1.2426757812500002 254.50000000000003 -1.3165187835693357 0.6180242047922454 1.4475910192202701 1.0000000180025095e-35 0.3261289997719386 252.50000000000003 1.0864924032472716 0.048554954793221271 0.65514425837065626 2.5000000000000004 3.0201273556387074 1.8688348091416842 2.6981674883069502 3.6242520361853052\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 5 3 -3 -5 6 -1 8 -6 21 -9 12 15 14 22 -2 -13 -18 19 20 -19 -8 -14 -7 27 -26 -27 -20 29 -29\nright_child=11 2 -4 4 7 23 9 10 -10 -11 -12 16 13 -15 -16 -17 17 18 24 -21 -22 -23 -24 -25 25 26 -28 28 -30 -31\nleaf_value=-0.011317602304640236 -0.024861011923151658 0.0025885868762808633 0.007373771946010836 0.0072473014438649442 -0.026932375580594962 0.024141172024662132 0.015551440039411425 0.009329552829903975 0.0081030234180234431 -0.016146775426847584 -0.016542164142090289 0.015615506758088087 0.025406057275660331 -0.017952499601931411 -0.001162863621189669 0.0079308912411735887 0.015780420132196341 0.0073473905395277737 0.0071514627261389302 0.0090411032383582897 -0.03007820225322462 -0.0056995074041524332 -0.0030482097438556182 0.0045322940064050311 0.0058130080818953114 -0.025005488250457255 0.0079986227815908219 -0.013740977125764374 0.0055703124002247431 0.0087981926390312323\nleaf_weight=17.758818082282435 6.2758199708361273 7.8154141149020679 5.6387437867659846 3.5166168139239771 14.669049757690489 2.1436241848532545 8.0212240192456026 2.8061999789433676 1.1162128490366274 1.5065117873456308 2.6980989039057306 6.3953052477480687 4.4130146793826901 2.4259925596967937 10.499328042910404 1.3237796036748362 5.6285938277630967 1.8923380619046364 46.903463424325309 3.1838339337009529 4.7807108592138547 2.9897151364601777 1.587769522901169 21.476396007432413 3.6269522291840977 3.157572660362348 0.94113479380030174 12.659255960449629 18.974439576604709 8.3985282623530093\nleaf_count=117304 35629 12616 1331 104 16360 33 11523 140 629 482 5786 767 2794 1451 9317 2551 5510 1422 875 249 4472 4191 5219 1025 134 75 24 1443 757 1940\ninternal_value=5.99045e-06 -0.00304304 -0.00830396 -0.0111526 -0.0154145 0.000774863 -0.00432351 -0.0193149 -0.0241198 0.00570237 -0.00394576 0.0019775 -0.00391258 0.00274805 0.00664357 -0.0188909 0.00334 0.00249318 0.00163486 -0.0105735 -0.0198907 0.00978721 0.0185173 0.00722016 0.00300333 -0.00705267 -0.0173447 0.00396702 0.000152622 -0.00476029\ninternal_weight=235.224 92.1566 38.2603 32.6216 24.8062 53.8963 30.2763 21.2896 15.7853 12.5175 5.5043 143.068 26.5257 18.9261 16.5001 7.5996 116.542 110.147 104.518 9.85688 6.67305 11.0109 6.00078 23.62 94.6613 7.72566 4.09871 86.9357 40.0322 21.0578\ninternal_count=246153 171524 36966 35635 23019 134558 133500 22915 16989 16196 5926 74629 56961 18781 17330 38180 17668 16901 11391 6143 5894 15714 8013 1058 5248 233 99 5015 4140 3383\nis_linear=0\nshrinkage=0.03\n\n\nTree=296\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 1 9 3 3 18 4 1 21 4 13 18 0 21 3 4 0 0 4 13 7 1 1 9 18 18 18 0 18 13\nsplit_gain=0.992276 2.27011 2.3971 1.23079 2.08736 1.19108 1.0952 1.66114 1.8289 1.18713 1.80931 1.45134 1.7522 1.30425 1.06503 1.47064 0.869755 0.905988 1.71235 0.854933 0.780429 0.760484 2.85288 1.6901 1.68509 1.81917 1.35753 1.73947 1.12824 1.58961\nthreshold=0.31033658837561479 1.6575619461558879 1.0000000180025095e-35 0.23756184665784369 0.23327427011668259 4.9870196042435371 1.0693359375000002 0.29645739473604366 2.5000000000000004 3.2895507812500004 -0.54370230766509287 2.1383330595080281 1.4460303622695712 3.5000000000000004 0.3261289997719386 4.9990234375000009 0.010815665348252143 0.34392793676569194 1.0693359375000002 -1.0000000180025095e-35 2.7412109375000004 1.4475910192202701 1.075454090076706 1.0000000180025095e-35 4.1820210059610323 4.2696729980286863 3.6242520361853052 1.4460303622695712 2.249904835165133 -0.055817889458273311\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=1 6 3 4 5 -3 7 19 -9 10 -8 13 -13 -11 15 -2 -16 18 -18 -1 -5 22 26 -24 28 -26 27 -19 -23 -30\nright_child=14 2 -4 20 -6 -7 9 8 -10 11 -12 12 -14 -15 16 -17 17 21 -20 -21 -22 24 23 -25 25 -27 -28 -29 29 -31\nleaf_value=-0.018262391607690075 0.018667194206024772 -0.01960025389043887 0.014951645184585374 -0.024806633940452016 0.020585455292373264 0.0061913185488618626 0.011998763126137525 0.026848572624796008 -0.0099307610447319609 0.00406830520561062 -0.018680617842008227 0.0013953304599373774 0.024811435384607925 -0.026434003507531386 0.013037252856076788 -0.015025322035463666 0.0092230134182617763 0.014773647209167949 -0.020245419992974047 0.0076842507085665085 0.0042122888543314699 0.0031218385608678518 -0.030434527881211794 0.0075767745731397523 0.025586580999986525 0.0030631108373062746 -0.014400473055955201 -0.018575168893059407 -0.0055178139885567485 0.0063077237047855948\nleaf_weight=1.3715075345844527 5.3607631569939249 4.4880980183895653 1.9748445381292192 7.7155454484459352 1.2758478418425054 1.7237663382729804 1.5959542694837985 4.3243316326894501 1.0074250912732337 1.9714362395461678 5.37850158884706 13.775774271912328 2.6812003818085932 1.6493838627857256 4.6913808677911666 0.64192843595037619 2.1202290582316428 7.1812356037548835 3.657318217693728 1.4036949356933215 0.62898110704554699 91.846810687615289 3.468495165891567 0.9301291187875903 4.4841115994049678 8.6274827346326202 1.7978990936064745 0.81242710303922638 37.581795353580219 12.997826336353681\nleaf_count=33307 3432 1816 98 2389 136 661 14485 6401 6766 230 78196 63717 79 163 3237 334 1045 1981 5161 6175 32 1834 3011 32 295 1884 4363 179 3805 663\ninternal_value=0.000911132 -0.00267833 -0.0110978 -0.0150424 -0.00516689 -0.0125771 0.00177488 0.0107804 0.0194283 -0.00121196 -0.011036 0.00256231 0.00641308 -0.01254 0.00194104 0.013433 0.0014862 0.00110532 -0.00966595 -0.00657946 -0.0224026 0.00152644 -0.00490204 -0.0224821 0.0021405 0.0120333 0.00434687 0.00969578 0.00115223 -0.00241554\ninternal_weight=239.166 52.9663 17.8071 15.8322 7.48771 6.21186 35.1592 8.10696 5.33176 27.0523 6.97446 20.0778 16.457 3.62082 186.2 6.00269 180.197 175.506 5.77755 2.7752 8.34453 169.728 14.1902 4.39862 155.538 13.1116 9.79156 7.99366 142.426 50.5796\ninternal_count=245907 214651 5132 5034 2613 2477 209519 52649 13167 156870 92681 64189 63796 393 31256 3766 27490 24253 6206 39482 2421 18047 9566 3043 8481 2179 6523 2160 6302 4468\nis_linear=0\nshrinkage=0.03\n\n\nTree=297\nnum_leaves=31\nnum_cat=0\nsplit_feature=3 0 1 3 4 19 0 3 14 4 19 12 21 13 3 3 9 13 14 4 0 19 5 19 17 4 19 4 5 21\nsplit_gain=1.30571 1.26591 1.51383 1.13454 1.11986 1.99833 2.7223 2.03315 2.47274 1.08279 0.919773 0.990395 1.08522 0.878518 1.96138 1.55184 1.41683 1.09791 1.1481 1.33116 1.35986 1.07124 0.995715 1.48548 0.959742 0.962156 1.7279 2.10907 1.71577 1.41955\nthreshold=0.53650715206675215 1.6765212893675572 2.3713805701640611 0.62297257604742995 2.2978515625000004 249.50000000000003 0.64273501842566294 0.65908129556435835 0.54439270847295596 2.3220214843750004 238.50000000000003 0.74643410817732136 2.5000000000000004 -0.65529293317224002 0.40911314460322756 0.32220753877234559 1.0000000180025095e-35 -0.48625410865680269 0.8483493944757371 1.9406738281250002 0.76735450420087303 262.50000000000006 2.9179687500000004 262.50000000000006 5.817026225953116 2.8684082031250004 240.50000000000003 2.3220214843750004 2.1386718750000004 3.5000000000000004\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=13 4 -3 -4 10 6 -6 8 -7 -5 -2 -12 -13 14 15 16 -1 18 19 20 -15 -22 23 -21 25 26 -19 29 -27 -28\nright_child=1 2 3 9 5 7 -8 -9 -10 -11 11 12 -14 17 -16 -17 -18 24 -20 22 21 -23 -24 -25 -26 28 27 -29 -30 -31\nleaf_value=-0.023973881714650924 -0.016639273179624384 -0.012557264623769835 0.018551386801275035 -0.0055771424332729481 -0.022018455578953926 2.247845675785815e-06 0.0096980756202192236 0.0035959954222946951 -0.033408820577481922 0.016392113445375552 0.0069542712709099574 0.011558074200449233 -0.01989299171120313 -0.0057280946291590439 -0.022591348927364002 0.013870901022622091 0.0077526040284735436 -0.015105031288079373 -0.010140393850867589 -0.024539324563925279 0.025455129076215718 -0.0067401829676476746 0.0076432804823761457 0.0069550250447867664 -0.019229157368310112 0.016933121343494756 0.004825642989507494 -0.013566759403604814 1.6048501717869578e-05 -0.0048607261221176703\nleaf_weight=1.9200511947837224 0.80022821689317325 1.5920272955045245 10.544882676913405 4.6270277932053423 1.9948886409638387 3.2385140790720452 12.530259433571699 4.5771655211719917 4.1856702217628481 2.1807304474059492 34.928168488857409 0.73236405802890603 1.7749051894061265 1.5736583799716806 4.2867515281304804 3.4488770344498789 1.5727280040574669 8.6830281747866973 2.6682790859575212 1.7686432003498551 6.1008146939282142 0.65067217715341019 6.3546586681205781 1.9625740628891888 1.7010353152926132 5.1664502678426052 32.556377856636686 9.8406846533956109 43.187618947168261 21.940036885271667\nleaf_count=21765 723 108 97 51 808 73 478 75 257 20 1966 30 43 7092 2257 2887 1068 21821 8412 4121 977 471 4228 1906 36710 2331 46503 13807 43067 22066\ninternal_value=0.000534187 0.00353212 0.0101471 0.0128009 0.00147218 -0.00319356 0.00420445 -0.011782 -0.0205643 0.0026584 0.004708 0.00558536 -0.0100301 -0.0010931 -0.00881427 0.00146699 -0.0111422 -0.000435106 0.00573226 0.00843655 0.0166093 0.0218831 0.000800826 -0.010005 -0.00152909 -0.00112592 -0.00328167 -0.00147921 0.00213005 0.000908633\ninternal_weight=239.09 83.7068 18.9447 17.3526 64.7622 26.5265 14.5251 12.0013 7.42418 6.80776 38.2357 37.4354 2.50727 155.383 11.2284 6.94166 3.49278 144.155 21.0793 18.411 8.32515 6.75149 10.0859 3.73122 123.075 121.374 73.0201 64.3371 48.3541 54.4964\ninternal_count=246218 4729 276 168 4453 1691 1286 405 330 71 2762 2039 73 241489 27977 25720 22833 213512 27207 18795 8540 1448 10255 6027 186305 149595 104197 82376 45398 68569\nis_linear=0\nshrinkage=0.03\n\n\nTree=298\nnum_leaves=31\nnum_cat=0\nsplit_feature=2 18 4 14 4 2 11 7 14 3 18 3 7 0 3 6 7 4 3 13 2 2 21 14 2 4 18 4 7 4\nsplit_gain=1.76222 1.22655 1.06123 2.75569 1.78766 1.81531 1.31854 1.16031 1.09709 1.08408 0.984106 0.791916 1.86057 1.26322 1.18016 1.614 2.98701 2.50638 2.23491 1.71511 1.71151 2.61615 1.9154 1.86944 1.53997 2.31576 1.26427 1.83541 1.39786 1.06973\nthreshold=20.132114410400394 3.6242520361853052 6.3540039062500009 0.36880189058923429 3.4389648437500004 68.466045379638686 0.60365083802632757 1.4980468750000002 0.016582161273737187 0.65908129556435835 4.6298509007569537 0.22136347059745312 2.0131835937500004 0.17863154480046692 0.2708708835714248 1.8691406250000002 1.2426757812500002 2.9458007812500004 0.51817819236196982 -0.05277688766252419 -14.86118173599243 2.9882335662841801 2.5000000000000004 0.97076046726095988 11.076014518737795 5.2885742187500009 2.4414009612931857 2.6340332031250004 1.2426757812500002 5.7263183593750009\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=11 2 4 -4 5 7 9 8 -2 -5 -3 12 13 -1 26 16 17 19 -19 -16 22 23 -18 -22 -23 -26 27 -13 -28 -14\nright_child=1 10 3 6 -6 -7 -8 -9 -10 -11 -12 14 29 -15 15 -17 20 18 -20 -21 21 24 -24 -25 25 -27 28 -29 -30 -31\nleaf_value=0.0014603322734672401 -0.010718487212374636 -0.016577216423399016 0.016324045419915167 -0.028383858036237615 0.014944290732932904 -0.017590304699982438 0.0046791547805507903 -0.014659634145233691 0.0079823133273939448 0.0036586489918083069 0.0036405261434976103 0.024849901271240732 0.015072007523775827 -0.025541492384206888 -9.6955486646150917e-05 0.011225356444303846 0.018957733699732322 0.030317603014505687 -0.014553211603726825 -0.011283780453054709 -0.031697656060508228 0.0022077270978622704 -0.011469996479146758 0.011752810231525451 -0.030762021202408216 0.0042820890036300886 -0.020118718312881263 -0.014351831585950003 0.0087653743212961731 -0.0050622211185667697\nleaf_weight=1.9471195995728887 2.2583735894278361 3.7060666791252279 3.0346998579188931 3.5255308257328579 14.998118548160617 2.612130652338009 1.9136727123986927 1.6271825962503488 28.453434018221287 0.9373598233796655 3.4654420990394215 4.0284791950884919 2.2429003897574358 5.1754949068157998 88.865158509483877 6.6133594432741338 2.4408240925092768 3.0072174542404464 0.66505376532040816 13.325779168601057 7.5075922004330122 7.8978563508194366 3.0926230955041092 0.48719155952767312 3.204981606045294 2.4815245952695477 1.5052437643609549 0.66269195268432657 2.8503028567819229 7.8766900582455719\nleaf_count=31835 614 11835 50 200 893 2346 68 215 6471 14 9604 540 9347 85241 23205 1203 960 893 176 1862 2395 1172 2141 49 818 325 12297 108 4854 33701\ninternal_value=-0.000633249 0.00345889 0.00489298 -0.00402139 0.00661855 0.00272953 -0.0146044 0.00484922 0.00631253 -0.0222161 -0.00755759 -0.00228545 -0.00819453 -0.0188889 -0.0015498 -0.00225016 -0.00299913 -0.000684094 0.0208159 -0.00165094 -0.0116925 -0.0153679 0.00279911 -0.0280376 -0.00650022 -0.0171159 0.00840489 0.0177634 -0.00310948 0.000354442\ninternal_weight=232.41 66.532 59.3605 9.41126 49.9492 34.9511 6.37656 32.339 30.7118 4.46289 7.17151 165.878 17.2422 7.12261 148.636 139.589 132.976 105.863 3.67227 102.191 27.1126 21.5791 5.53345 7.99478 13.5844 5.68651 9.04672 4.69117 4.35555 10.1196\ninternal_count=245432 32310 10871 332 10539 9646 282 7300 7085 214 21439 213122 160124 117076 52998 35199 33996 26136 1069 25067 7860 4759 3101 2444 2315 1143 17799 648 17151 43048\nis_linear=0\nshrinkage=0.03\n\n\nTree=299\nnum_leaves=31\nnum_cat=0\nsplit_feature=12 21 1 6 3 18 18 7 18 17 19 9 3 1 0 18 1 3 3 14 1 14 4 4 18 0 14 3 4 1\nsplit_gain=1.15815 1.35331 1.44301 1.28885 1.04658 1.49705 2.67237 2.68778 2.31074 1.61042 1.49653 1.7685 1.72299 1.37849 1.67658 1.3469 1.33703 2.08326 1.64079 2.22619 2.06578 2.06971 1.73144 1.55753 1.53224 2.2055 1.79689 1.52059 1.49606 1.70561\nthreshold=0.030057938850762859 1.5000000000000002 2.5144622974930591 4.7578125000000009 0.19435047505660144 4.1820210059610323 4.2696729980286863 2.6015625000000004 3.7494348669884658 0.67728060322190908 255.50000000000003 1.0000000180025095e-35 0.53650715206675215 -0.56638788419139308 0.76735450420087303 3.5972184255501678 2.9477339485081675 0.56033297298897644 0.3261289997719386 0.36880189058923429 1.8223781763416411 0.8176537162135239 4.2863769531250009 2.8471679687500004 1.2424533248940002 1.5447487307370646 0.24669767028957582 0.40911314460322756 4.3771972656250009 1.8912989126705502\ndecision_type=2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2\nleft_child=4 2 -2 -3 13 8 -7 10 15 -10 11 -8 -12 14 -1 16 18 23 24 22 21 -21 27 -18 25 -6 -26 -20 29 -22\nright_child=1 3 -4 -5 5 6 7 -9 9 -11 12 -13 -14 -15 -16 -17 17 -19 19 20 28 -23 -24 -25 26 -27 -28 -29 -30 -31\nleaf_value=-0.012772240113759201 -0.0063255088644013861 -0.014099359565710708 0.014622040074619485 0.0097823576021506785 0.010583741395037195 0.02439959119864726 -0.016782969484868439 0.01610160753262976 -0.029749879023109178 -0.0032050602810295235 -0.01324717838378031 0.017552370297847807 0.015145443787811399 -0.010550785188081176 0.018632453730010348 0.013368111854021624 -0.004382950484556964 0.0088443519913486476 -0.0076794145134771608 -0.026288367255419016 0.015068357175295092 -0.0004993504814879917 -0.015136332295687917 0.010042302004265716 -0.0094339773810883966 -0.025072380923185259 0.015691279455454624 0.014528480496872999 0.012571457410557121 -0.0085488918325003233\nleaf_weight=1.6227902992749843 6.3615697119466308 14.421928858765684 3.9811742724441501 1.6252797519186959 2.2276737518068357 5.5172195350879205 7.9837611215249371 7.4577156581532327 2.7688589157282868 5.1445090653482453 2.0516285213420806 0.75320862501564523 4.466424790029123 14.095592866545074 2.936919123687062 6.4473274657140669 31.356115705733828 31.575620671792421 3.0100435676035877 8.0397505771497872 2.1504735271937596 4.1151031042395507 2.3032116792164734 7.5226485173916435 2.5691514558249073 2.2103980399024294 10.476336146610036 9.9411668793836707 3.4093312569893888 25.934180768679653\nleaf_count=60686 2009 6991 190 313 226 2151 27287 4095 3229 5316 11984 224 375 99463 3925 2250 797 426 406 2117 146 1688 54 128 832 173 6864 336 64 1298\ninternal_value=-0.00024216 -0.00604195 0.00231595 -0.0112486 0.000518757 0.00119665 0.00744322 0.00246903 6.35951e-05 -0.0147886 -0.00477608 -0.0123243 0.00563523 -0.00607545 0.00716864 0.000921208 0.000287979 0.00324518 -0.00244624 -0.00483083 -0.00823197 -0.018259 0.00480886 -0.00140998 0.00541431 -0.00851983 0.0104238 0.00918657 -0.00391929 -0.00628458\ninternal_weight=234.477 26.39 10.3427 16.0472 208.087 189.432 28.23 22.7127 161.202 7.91337 15.255 8.73697 6.51805 18.6553 4.55971 153.289 146.841 70.4544 76.3868 58.9033 43.6488 12.1549 15.2544 38.8788 17.4836 4.43807 13.0455 12.9512 31.494 28.0847\ninternal_count=246043 9503 2199 7304 236540 72466 46116 43965 26350 8545 39870 27511 12359 164074 64611 17805 15555 1351 14204 6109 5313 3805 796 925 8095 399 7696 742 1508 1444\nis_linear=0\nshrinkage=0.03\n\n\nend of trees\n\nfeature_importances:\nfp=983\nz_gl=875\npop_public=857\nfp_rank_window=799\ngl=713\nz_fp=611\nppm_err=545\nlib_margin=518\nlsynonyms=514\nlib_max_mol=461\nlsid=428\nn_cands=373\nlib_sim=239\nn_query=236\nlib_same=223\nlpmid=158\nlotus=132\nsrc_bit=115\nce_known=101\nnp_bit=74\nin_coconut=44\nin_tier=1\n\nparameters:\n[boosting: gbdt]\n[objective: lambdarank]\n[metric: ndcg]\n[tree_learner: serial]\n[device_type: cpu]\n[data_sample_strategy: bagging]\n[data: ]\n[valid: ]\n[num_iterations: 300]\n[learning_rate: 0.03]\n[num_leaves: 31]\n[num_threads: 0]\n[seed: 0]\n[deterministic: 0]\n[force_col_wise: 0]\n[force_row_wise: 0]\n[histogram_pool_size: -1]\n[max_depth: -1]\n[min_data_in_leaf: 20]\n[min_sum_hessian_in_leaf: 0.001]\n[bagging_fraction: 0.8]\n[pos_bagging_fraction: 1]\n[neg_bagging_fraction: 1]\n[bagging_freq: 1]\n[bagging_seed: 7719]\n[bagging_by_query: 0]\n[feature_fraction: 0.8]\n[feature_fraction_bynode: 1]\n[feature_fraction_seed: 2437]\n[extra_trees: 0]\n[extra_seed: 11797]\n[early_stopping_round: 0]\n[early_stopping_min_delta: 0]\n[first_metric_only: 0]\n[max_delta_step: 0]\n[lambda_l1: 0]\n[lambda_l2: 1]\n[linear_lambda: 0]\n[min_gain_to_split: 0]\n[drop_rate: 0.1]\n[max_drop: 50]\n[skip_drop: 0.5]\n[xgboost_dart_mode: 0]\n[uniform_drop: 0]\n[drop_seed: 21238]\n[top_rate: 0.2]\n[other_rate: 0.1]\n[min_data_per_group: 100]\n[max_cat_threshold: 32]\n[cat_l2: 10]\n[cat_smooth: 10]\n[max_cat_to_onehot: 4]\n[top_k: 20]\n[monotone_constraints: ]\n[monotone_constraints_method: basic]\n[monotone_penalty: 0]\n[feature_contri: ]\n[forcedsplits_filename: ]\n[refit_decay_rate: 0.9]\n[cegb_tradeoff: 1]\n[cegb_penalty_split: 0]\n[cegb_penalty_feature_lazy: ]\n[cegb_penalty_feature_coupled: ]\n[path_smooth: 0]\n[interaction_constraints: ]\n[verbosity: -1]\n[saved_feature_importance_type: 0]\n[use_quantized_grad: 0]\n[num_grad_quant_bins: 4]\n[quant_train_renew_leaf: 0]\n[stochastic_rounding: 1]\n[linear_tree: 0]\n[max_bin: 255]\n[max_bin_by_feature: ]\n[min_data_in_bin: 3]\n[bin_construct_sample_cnt: 200000]\n[data_random_seed: 38]\n[is_enable_sparse: 1]\n[enable_bundle: 1]\n[use_missing: 1]\n[zero_as_missing: 0]\n[feature_pre_filter: 1]\n[pre_partition: 0]\n[two_round: 0]\n[header: 0]\n[label_column: ]\n[weight_column: ]\n[group_column: ]\n[ignore_column: ]\n[categorical_feature: ]\n[forcedbins_filename: ]\n[precise_float_parser: 0]\n[parser_config_file: ]\n[objective_seed: 8855]\n[num_class: 1]\n[is_unbalance: 0]\n[scale_pos_weight: 1]\n[sigmoid: 1]\n[boost_from_average: 1]\n[reg_sqrt: 0]\n[alpha: 0.9]\n[fair_c: 1]\n[poisson_max_delta_step: 0.7]\n[tweedie_variance_power: 1.5]\n[lambdarank_truncation_level: 30]\n[lambdarank_norm: 1]\n[label_gain: 0,1]\n[lambdarank_position_bias_regularization: 0]\n[eval_at: 25]\n[multi_error_top_k: 1]\n[auc_mu_weights: ]\n[num_machines: 1]\n[local_listen_port: 12400]\n[time_out: 120]\n[machine_list_filename: ]\n[machines: ]\n[gpu_platform_id: -1]\n[gpu_device_id: -1]\n[gpu_device_id_list: ]\n[gpu_use_dp: 0]\n[num_gpu: 1]\n\nend of parameters\n\npandas_categorical:null\n"}
for name, text in SOURCES.items():
    open(f'/kaggle/working/b1/{name}', 'w', encoding='utf-8').write(text)
print('sources written', list(SOURCES), flush=True)
# official RDKit (scorer version) when provided by an attached dataset; otherwise keep the image's RDKit (keys may differ for ~1.6% of structures)
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whl = [w for w in glob.glob('/kaggle/input/**/rdkit-2026.3.3-*.whl', recursive=True) if tag in w]
if whl:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-index', '--quiet', whl[0]], capture_output=True, text=True)
    print('rdkit wheel install rc', r.returncode, r.stdout[-300:], r.stderr[-300:], flush=True)
else:
    print('no rdkit 2026.3.3 wheel attached; using the image RDKit', flush=True)
import rdkit; print('rdkit', rdkit.__version__, flush=True)


In [ ]:
PIPELINE_ARGS = ["--k_fpp", "200", "--k_pop", "50", "--w_ice", "0", "--tau", "0.9", "--main", "blend21", "--gl_budget", "14000", "--deadline_s", "30000"]
EMIT = "main"   # 'main' = the pipeline's main list is submission.csv; 'insurance' = the fixed rule + library priority list is submission.csv (separate kernel version per Kaggle submission)
import subprocess, sys, time, json, os, shutil
t0 = time.time()
cmd = [sys.executable, '-X', 'utf8', '/kaggle/working/b1/b1_pipeline.py', '--out', '/kaggle/working'] + PIPELINE_ARGS
print('running', ' '.join(cmd), flush=True)
with open('/kaggle/working/b1_pipeline.log', 'w') as lf:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True); lf.write(line)
    rc = p.wait()
print('pipeline rc', rc, f'{time.time()-t0:.0f}s', flush=True)
assert os.path.exists('/kaggle/working/submission.csv'), 'submission.csv missing'
if EMIT == 'insurance':
    os.replace('/kaggle/working/submission.csv', '/kaggle/working/submission_main.csv'); shutil.copy('/kaggle/working/submission_fixed.csv', '/kaggle/working/submission.csv')
    print('EMIT insurance: submission.csv = fixed rule + library priority (main list kept as submission_main.csv)', flush=True)
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv'); samp = pd.read_csv('/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv')
assert len(sub) == len(samp) and set(sub.molecule_id) == set(samp.molecule_id), (len(sub), len(samp))
assert sub.smiles.notna().all() and sub.smiles.map(lambda s: len(str(s).split(';'))).max() <= 25
print(json.load(open('/kaggle/working/b1_run_manifest.json')))
print('OK', sub.shape, flush=True)
